# Rebuild our quantum circuit submission

**Released under the Apache License 2.0.** Third-party notices and the existing
calibration-rights limitation are retained below.

Google Quantum AI's [Verifiable Quantum Circuit Compilation competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation)
asks us to make three quantum calculations cheaper while preserving their answers.
**invsqrt** computes inverse square-root arithmetic with an interpolated polynomial.
**dqi** decodes an error-correcting code with a fused decoder.
**dfthcsos** loads angle words for a chemistry simulation using five-word XOR batches.

A higher ratio means a lower cost than Google's reference. Our recorded local ratios are
**13.24× / 54.73× / 9.78×** (geometric mean about **19.20572**); the same submission received
**19.20572 on the public leaderboard**. These are historical results, not
a claim that this run reproduces the hidden evaluation's score.

This notebook rebuilds submission-011 from source, compares every output byte by SHA-256,
and runs fresh checks using the competition verifiers. The final file is `submission.zip`.
No earlier notebook output or verifier receipt is used.

## Run this notebook

Attach the competition data and SomeJoint dataset `qcc-our-chemistry-b143-c60-xor2` version 1. The default path is
`/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation`.
No other competition directory is searched or read.
For a local run, set **VQCC_DATA** to the extracted competition package root (the directory
containing `kickmix`, `verifiers`, and `sample_submission`). Set **VQCC_FACTOR** to the explicit local path to `submission.npz`; on Kaggle the default is `/kaggle/input/qcc-our-chemistry-b143-c60-xor2/submission.npz`.
The packaged native library requires **CPython 3.13 on Linux**. NumPy, SciPy, CUDA-enabled PyTorch and a CUDA GPU must be
available; this notebook installs nothing and makes no network request. It prints their
versions and fails if the source, native library, verifier, generated files or archive differ.

The generator source is embedded verbatim from [1d3b1178e6de84acd76c9ac751b0d88f666bfef0](https://github.com/clduab11/quantum-hackathon/tree/1d3b1178e6de84acd76c9ac751b0d88f666bfef0),
a commit on `main`. Every embedded file has its own SHA-256 below. This avoids a runtime
clone and works with internet disabled. The notebook builder extracts these files with
`git show`; its source remains beside this notebook in the repository.

Verification defaults to **full**: 64 invsqrt shots for each of 3,133,452 inputs,
102,400 dqi shots, and 64 dfthcsos shots per address. Budget roughly 30 minutes for
verification plus the build; actual times are printed. The public salt is fixed once;
no seed or salt search occurs. At most two verifier workers and one BLAS thread bound CPU use.

In [1]:
# SPDX-License-Identifier: Apache-2.0
import hashlib
from pathlib import Path, PurePosixPath


def check_hash(path, expected):
    path = Path(path)
    with path.open('rb') as stream:
        actual = hashlib.file_digest(stream, 'sha256').hexdigest()
    print(f'{path.name}: SHA-256 {actual}', flush=True)
    if actual != expected:
        raise ValueError(f'SHA-256 mismatch for {path.name}: {actual} != {expected}')
    return actual


def check_files(root, expected):
    root = Path(root)
    actual = {p.relative_to(root).as_posix() for p in root.rglob('*') if p.is_file()}
    if actual != set(expected):
        raise ValueError(f'File set mismatch: missing={sorted(set(expected)-actual)}, '
                         f'extra={sorted(actual-set(expected))}')
    for name, digest in sorted(expected.items()):
        path = root / name
        if path.is_symlink() or not path.resolve().is_relative_to(root.resolve()):
            raise ValueError(f'Unsafe output path: {name}')
        print(name, end=': ')
        check_hash(path, digest)


def restore_sources(root, sources, hashes):
    if set(sources) != set(hashes):
        raise ValueError('Source inventory mismatch')
    for name, source in sources.items():
        path = PurePosixPath(name)
        if path.is_absolute() or '..' in path.parts or '\\' in name:
            raise ValueError(f'Unsafe source path: {name}')
        if hashlib.sha256(source.encode('utf-8')).hexdigest() != hashes[name]:
            raise ValueError(f'SHA-256 mismatch for vendored source: {name}')
    root = Path(root)
    root.mkdir(parents=True, exist_ok=False)
    for name, source in sorted(sources.items()):
        path = root / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(source.encode('utf-8'))
        print(f'{name}: SHA-256 {hashes[name]}')


def competition_root(override=None, candidates=None):
    if candidates is None:
        candidates = [Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')]
    roots = [Path(override)] if override is not None else candidates
    for root in roots:
        root = Path(root).resolve()
        if all((root / name).is_dir() for name in ('kickmix', 'verifiers', 'sample_submission')):
            return root
    raise FileNotFoundError('No competition root with kickmix/, verifiers/ and sample_submission/; set VQCC_DATA')


def stage_factor(path, root, size, digest):
    import shutil
    path, root = Path(path), Path(root)
    if path.name != 'submission.npz':
        raise ValueError('Factor filename must be submission.npz')
    if path.stat().st_size != size:
        raise ValueError('Factor size mismatch')
    check_hash(path, digest)
    dest = root / 'inputs/somejoint/qcc-our-chemistry-b143-c60-xor2-v1/submission.npz'
    dest.parent.mkdir(parents=True, exist_ok=False)
    shutil.copyfile(path, dest)
    check_hash(dest, digest)
    return dest


def check_archive(path, expected):
    # Call only after every member hash has passed.
    try:
        return check_hash(path, expected)
    except ValueError as error:
        raise ValueError('R2: member-identical, container differs; check zlib versions') from error

In [2]:
# SPDX-License-Identifier: Apache-2.0
CONFIG = {'commit': '1d3b1178e6de84acd76c9ac751b0d88f666bfef0', 'repository': 'https://github.com/clduab11/quantum-hackathon', 'manifest': 'ledger/manifests/submissions/submission-011.json', 'candidate': 'submission-011', 'manifest_commit': '1d3b1178e6de84acd76c9ac751b0d88f666bfef0', 'manifest_sha256': 'fbdfc9dea3d6b7d694f9f602465d8fde942df57643bb31b30e61df3957e9bd8d', 'files': {'dfthcsos/angle_k_0.kmb': 'd0f7e6bf8057b171c22a0cbef041ccecf1334fb17e968fd924286ff70d66d08a', 'dfthcsos/angle_k_1.kmb': '468475d7214894af0616870bdce211799bc2e97682699c97f89d5afbc20824f8', 'dfthcsos/angle_k_10.kmb': '2aa48c950a5d0850ddd544886eb5720f4a92f0b7cf496f182f4b9b9a980a8de7', 'dfthcsos/angle_k_11.kmb': '78c041703bbc43f795ae4ab9224a15e2a6dc97d60c76cf9927226de6c959a466', 'dfthcsos/angle_k_12.kmb': 'e43da6265070311362619ce8c57f780e41f519d0be0590049bba8c2c885c41fd', 'dfthcsos/angle_k_13.kmb': 'fbdac0002c7cd3e66487f966ccac1edf2ac40f628fb7d22af048c1ce66f60219', 'dfthcsos/angle_k_14.kmb': '8b6d5738e981696154031d1f7c71b90fcde08ffd2fea1ae8525b545fd2c6c5c1', 'dfthcsos/angle_k_2.kmb': 'f31de4b7672f853cbecc21be77e30cb6e1ecc837a5d4680ffee77fe7d147bf14', 'dfthcsos/angle_k_3.kmb': '3b175fb0abaa7e20fe22d021c77cb27e16658fd088d6f78bf307286e50319736', 'dfthcsos/angle_k_4.kmb': '2990dbbf6177c166e32426803dd6f2a969659238b580a8500b9359cb12252ab2', 'dfthcsos/angle_k_5.kmb': 'b4d36019132b43661850f984cf8eb723482cfb0af7eeb4cadb0964b6cc103e2a', 'dfthcsos/angle_k_6.kmb': '5c0bd4a2cd9436c925569a33044bac964653709b38a84d1df57e15bb2d60d6c6', 'dfthcsos/angle_k_7.kmb': 'dead802918d2183ecb2d3c96d6ea1df5ba7e5240d3a20cf63dbfc8fa146bb05f', 'dfthcsos/angle_k_8.kmb': 'd2e22516ab580482d755dca2f1f999cb2acb83cffda74df0ef1d7f0531fa3a73', 'dfthcsos/angle_k_9.kmb': '663497ad211daa8803a60e1cfff5d4e760499f971054c2908fe3a59489aa597c', 'dfthcsos/submission.npz': 'aef36d39b83bbbf696e909884218f6c02f067baef14a29de1c41306bc1f6395b', 'dqi.kmb': '7beea291a5de36ef2b4fb50585b2f8d7cce02d3c575b491cf5dcc93e9d9655f6', 'invsqrt.kmb': '17fc223d79d3f1f15ef448389072209972b373403dd0a45352f4fe7eaff541ad'}, 'zip_sha256': 'a710cab21a8bac7ff552944ca48e8afb4e0fe4c98e0be5fd2e86be32b79c01ac', 'zip_bytes': 31514298, 'factor_bytes': 1995236, 'kickmix_sha256': 'c66b3216c4029e0cdfd09054306aa0feadf53ab2f50c1164e37f752874e5005e', 'verifiers': {'dfthcsos': '0775be7a528922367caebe9e4d051d6199811f1aefeaa4b793eaaa4621b8ec9d', 'dqi': 'f14274bc0943fb9769ddb01131be6063191a15009fe1caf1d4bdd9fd25194e65', 'invsqrt': 'ddef520dec6b68d1cf0a6f328ddd36d79eb84570b9192cf3f7acaf073f06733a'}}
SOURCES = {'LICENSE': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2026 Praxen / Chris Dukes\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'NOTICE': 'quantum-hackathon — Praxen\'s entry to the Google Quantum AI "Verifiable Quantum Circuit Compilation" Kaggle competition. Copyright 2026 Praxen / Chris Dukes. Licensed under the Apache License, Version 2.0.\n\nThird-party material in this tree is listed in the files under notices/, one file per area of\nthe tree, read in file-name order. Those files are part of this NOTICE and are distributed with\nit. `uv run python -m ledger.registers notice` prints this NOTICE with every entry in one text.\n', 'ledger/__init__.py': '', 'ledger/archive.py': '"""Submission zips: safe extraction and deterministic builds.\n\n`safe_extract` applies the archive checks from the PRA-171 inventory (§4) before anything reads\nthe files: no duplicate paths, no absolute or `..` paths, no backslashes, no links or special\nfiles.\n\n`build_zip` writes a byte-reproducible zip (sorted entries, fixed timestamps and permissions), so\nG5 can compare the SHA-256 of a rebuilt zip with the one in the manifest. Every candidate zip is\nbuilt with it.\n"""\n\nfrom __future__ import annotations\n\nimport shutil\nimport stat\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nFIXED_DATE_TIME = (1980, 1, 1, 0, 0, 0)\nFILE_MODE = 0o644\n\n\nclass UnsafeArchive(ValueError):\n    """The zip contains a path or entry type that is not allowed."""\n\n\ndef check_members(archive: zipfile.ZipFile) -> None:\n    members = archive.infolist()\n    # Compare normalised paths: "a/b", "./a/b", "a//b" and "a/b/" all extract to the same place.\n    names = [PurePosixPath(m.filename).as_posix() for m in members]\n    if len(set(names)) != len(names):\n        raise UnsafeArchive("zip has duplicate paths")\n    for member in members:\n        path = PurePosixPath(member.filename)\n        if path.is_absolute() or ".." in path.parts or "\\\\" in member.filename:\n            raise UnsafeArchive(f"zip has an unsafe path: {member.filename!r}")\n        mode = member.external_attr >> 16\n        if stat.S_IFMT(mode) not in (0, stat.S_IFREG, stat.S_IFDIR):\n            raise UnsafeArchive(f"zip contains a link or special file: {member.filename!r}")\n\n\ndef safe_extract(zip_path: Path, dest: Path) -> Path:\n    """Check every member, then extract into `dest` (which must not exist yet)."""\n    dest = Path(dest)\n    dest.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(zip_path) as archive:\n        check_members(archive)\n        archive.extractall(dest)\n    return dest\n\n\ndef build_zip(src_dir: Path, dest_zip: Path) -> Path:\n    """Zip every regular file under `src_dir` deterministically."""\n    src_dir = Path(src_dir)\n    files = sorted(p for p in src_dir.rglob("*") if p.is_file())\n    for path in files:\n        if path.is_symlink():\n            raise UnsafeArchive(f"refusing to zip a symlink: {path}")\n    dest_zip = Path(dest_zip)\n    dest_zip.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(dest_zip, "w", compresslevel=6) as archive:\n        for path in files:\n            info = zipfile.ZipInfo(path.relative_to(src_dir).as_posix(), FIXED_DATE_TIME)\n            info.compress_type = zipfile.ZIP_DEFLATED\n            info.external_attr = (stat.S_IFREG | FILE_MODE) << 16\n            info.create_system = 3  # Unix, so the mode bits above are honoured\n            # Streamed, so the 886 MB dqi circuit is never held in memory.\n            large = path.stat().st_size >= zipfile.ZIP64_LIMIT\n            with path.open("rb") as src, archive.open(info, "w", force_zip64=large) as dst:\n                shutil.copyfileobj(src, dst, 1 << 20)\n    return dest_zip\n', 'notices/10-invsqrt.txt': '- The invsqrt generator under tasks/invsqrt/ is a port, not independent clean-room authorship. The module map is `tasks/invsqrt/PROVENANCE.md`. The code parent is "Quantum Circuits | Signed-Digit Memory" v3 by Artem1410 (anazemcev / ViberTeam), https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory?scriptVersionId=361502374 (v3), Apache License 2.0 as declared in that version\'s own source (SPDX header in cell 5, licence text and NOTICE in cell 20; PRA-218). Source snapshot SHA-256 79f6373302cb916a88f65a1b142024d4d2d0b14e41eb58bef3152ebd45ffd0ca, equal to the public version-pinned download. That notebook retains upstream notices for Sumit Chongder, Rudra Bhaskar, Alex Chilton, Kozykorpesh, and Kirill Tymchishin. Those notices are kept in tasks/invsqrt/signed_digit.py. Sumit run 361357149 is v1 (public version metadata, PRA-218); Dmitrii Gluzdov v6 is credited second-hand. Their originals are not loaded. Whether a later Sumit v2 exists upstream stays with PRA-205.\n- The 90-qubit Gray curvature schedule in tasks/invsqrt/schedule90.py is a port of Kozykorpesh, "Quantum AI LB score", V3 run 361677445, https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445 , Apache License 2.0, as that method is used by the attached Apache-2.0 notebooks. The pruned shift is the Dmitrii Gluzdov V6 boundary rule that schedule names (run 361592338). His notebook is not an input. The two Davio reload orders are method parameters from anazemcev, "Quantum Circuits | Source & Exact Compilation" v4, Apache License 2.0, attachment SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb.\n- The affine Toffoli/CSWAP rewrite in tasks/invsqrt/affine.py is a port of Kozykorpesh, "Quantum AI complete-domain affine compilation" v2, Apache License 2.0, attachment SHA-256 81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24, and the measurement-uncompute rule from anazemcev v4. The peephole in tasks/invsqrt/peephole.py is a port of that compilation\'s involution pass.\n- Rotation calibration in tasks/invsqrt/calibrate.py is our scipy linprog search. The slack-maximising drop is credited to LightningV08 and to Mizero Lucky Gall. Mizero v1 has no notebook-wide licence (PRA-194 public-notebooks; PRA-199 item 2) and is reference-only: it is not loaded and it is not a shipping parent. Alex Chilton v8 is the same: credited where the signed-digit parent retains his notice, not loaded, not a licence parent. No calibration file is an input.\n- Coefficient triples are produced by tasks/invsqrt/fit_coefficients.py from the mathematical target (Lawson minimax, then an integer search whose start is the rounded Lawson coefficient). Provenance of that table is `regenerated; dataset-adapted calibration; licence unresolved`. The calibration is adapted from the dataset\'s `code/domain_tables.py` (SHA-256 59cf4d98899433dcc49d9ce512293be20d33da5334211c20e35323555994a4fb); the output equals its `code/coefficients.json` (SHA-256 98bd211f22b158bf05a4d2d934ab966e5e97e54d8307b82e2ca58910c6830763) 256 of 256 rows. Source: kozykappa/quantum-ai-dataset, version 1, datasetVersionId 21136269, https://www.kaggle.com/datasets/kozykappa/quantum-ai-dataset?datasetVersionId=21136269 . Kaggle licence label: "Other (specified in description)". Rights for both files remain unresolved (D-PRA220-01). The Kaggle coefficient dataset version 21136269 and `code/coefficients.json` are not read and are not the optimiser start. Toolbox `fusion/*.npz` and `selected/invsqrt.kmb` are not inputs.\n- `tasks/invsqrt/reference.py` is the competition package\'s reference Newton-Raphson circuit script (`verifiers/invsqrt/nr_reference_circuit.py`), Apache-2.0, Copyright Google Quantum AI authors of that package. Praxen changed the command line (`--out` writes `invsqrt.kmb`). It rebuilds the reference submission (submission-001) for G5 under `configs/gates/reference.yaml` and is not a parent of the Praxen candidate. https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/data\n- The experimental `tasks/invsqrt/public_port/` schedule and interpolation recipe are a port of Dmitrii Gluzdov, "Quantum circuit compilation from source", v7, scriptVersionId 361726742, https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361726742 . The pinned notebook declares Apache-2.0 in cell 50; its UTF-8 SHA-256 is d689cc89f41a6c9130e9f4c3a6b08dfd5d758fcbbb79f527d55cbd3b48e15280. Code cells 20, 22, 23, 29, 34, 40 and 42 supply the deferred repair, interpolated curvature, split arithmetic and virtual low-bit methods. Retained method credits: Sumit Chongder, Rudra Bhaskar, Kozykorpesh, Alex Chilton, Kirill Tymchishin, Artem Nazemtsev, Gu/Kemeny/Yelin/Lukin, Dmitrii Gluzdov, LightningV08, Mizero Lucky Gall and Zongao Bian. Praxen changed package imports, regenerates coefficients by CUDA enumeration of the mathematical target, refines failing groups, and independently recomputes affine rewrites and angles. No saved table, calibration file, rewrite patch, busyaprime binary or other notebook output is an input. The existing fallback generator and its separately disclosed coefficient lineage are unchanged.\n', 'notices/11-invsqrt-study.txt': "- PRA-238 integer carry-accumulator study: independently implemented from the prose carry-save outer-row method in phucthaiv02, QCC Verified Portfolio + DF-THC Retraining Study, https://www.kaggle.com/code/phucthaiv02/qcc-verified-portfolio-df-thc-retraining-study (Apache-2.0 stated in PRA-234 A.5; source SHA-256 6d0e9fd742582f73ae891f03d5991b6948f695b7d5d9a0ee9e3af878d705d5e9). No upstream code, coefficient table, or binary is incorporated. Alex Chilton's cost-anatomy notebook is a discussion reference only. Detailed provenance and limitations: docs/research/PRA-238-invsqrt-design-study.md.\n", 'notices/12-invsqrt-mrmeows-audit.txt': '- PRA-320 source audit (no shipping generator or circuit): mrmeows, qcc-1375-open-circuits v6 and v8, https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits/versions/6, https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits/versions/8, Apache-2.0. Source, documentation and configuration hashes are recorded in ledger/evidence/pra320-source-audit/SOURCE_MAP.json and ADDITIONAL_SOURCE_MAP.json; the upstream licence text is preserved there as LICENSE. The audited arithmetic helper attributes Sumit Chongder, https://www.kaggle.com/code/sumitchongder/notebook2f196e646b, Apache-2.0 as stated in that source. No upstream circuit, proof, certificate or angle replay is a declared input; the published frozen-input replay was not ported into a shipping generator.\n', 'notices/20-dqi.txt': '- `tasks/dqi/reference_circuit.py` is the competition package\'s reference Reed–Solomon decoder (`verifiers/dqi/dqi_reference_circuit.py`), Apache-2.0, Copyright Google Quantum AI authors of that package. https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/data\n- `tasks/dqi/generate.py` is a port of a Berlekamp–Massey + Forney decoder, not an independent clean-room work. The module map is `tasks/dqi/PROVENANCE.md`. Transitive code parent: Gu, Kemeny, Yelin, and Lukin, repository `andigu/uncompute`, commit `64b5a13aa249f1360846657392e6c66817b26d0f`, Apache-2.0 (https://github.com/andigu/uncompute). Those four names are the `pyproject.toml` authors. That repository has no NOTICE file and states no copyright year; none is added here. Its `LICENSE` is copied verbatim to `tasks/dqi/bm_forney/LICENSE` (SHA-256 `cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30`). Used upstream paths: `LICENSE`; `uncompute/classical.py`; `uncompute/gadgets/linear.py`; `tasks/dqi-reed-solomon/contract.json` (field modulus changed from 4331 to 4179); `tasks/dqi-reed-solomon/circuits/discovered/constants.json` keys `bm_windows` and `psi_windows` only; `tasks/dqi-reed-solomon/circuits/discovered/circuit.py`; `tasks/dqi-reed-solomon/circuits/discovered/gen_constants.py` (formula section). Not used: `search.c`, `README.md`, `profile.pdf`, `circuit.ops.xz`, `liveness.json`, and the paper text and figures. Bibliographic citation only: Gu, Kemeny, Yelin, and Lukin, arXiv:2610.22367. Immediate code parent of the kickmix schedule: the identical `src/gen_dqi_v6` embed in Artem1410 (anazemcev / ViberTeam), "Quantum Circuits | Source & Exact Compilation" v4, Apache-2.0, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation (attachment SHA-256 `4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb`) and in Kozykorpesh (kozykappa), "Quantum AI — Complete-Domain Affine Compilation" v2, Apache-2.0, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation (attachment SHA-256 `81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24`). That embed attributes the memory schedule to Dmitrii Gluzdov\'s V6 schedule; his notebook and `femoco_public_words.npz` are not inputs. The shipped borrow route is the notebooks\' affine loan. Multiplication tables are regenerated for modulus 4179. Zenodo 10.5281/zenodo.17301475 is not an input. No k256.dev circuit or numeric target is an input.\n- anazemcev v4 cell-45: Retained opening notice from cell 45 of "Quantum Circuits | Source & Exact Compilation" v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation , attachment SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb: "Quantum Circuits resource labs (2026). Original experiments, explanations and added implementations: anazemcev / ViberTeam. Released under the Apache License, Version 2.0, subject to upstream notices." The directly copied src/gen_dqi_v6 embed supplies tasks/dqi/bm_forney/; tasks/dqi/generate.py adapts its driver and streamed sweep. The existing Gu notice and module map above remain applicable. The following credits are inherited through that embed, not claims that Praxen directly copied those earlier notebooks. Unrelated chemistry, numerical dataset and compiled-input claims in the original NOTICE do not describe inputs to this DQI port.\n- Gluzdov field/adapter: Cell-45 attribution (cell 45 of the hash-pinned anazemcev v4 notebook above), retained for tasks/dqi/bm_forney/gu_formula_derivation.py, port_runtime.py and gu_port.py: "Dmitrii Gluzdov: competition-field formula derivation, Kickmix adapter, record-reading phase sweep and readable source-built portfolio. Apache-2.0. Adapted source V3, run 361529794: https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361529794 . The separately scored V1, run 361509836, returned 6.12847 for submission 57567302. V1 and V3 contain identical embedded generator modules. https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361509836 ." Version identity and scores in this passage are the immediate parent\'s historical attribution, not newly verified measurements. Our field tables are regenerated for modulus 4179; port_runtime.py\'s import and field checks are modified as recorded in the module map.\n- Kozykorpesh symbolic lowering: Inherited through anazemcev v4 cell 45: "Kozykorpesh V3/run361677445, Apache2.0: https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445 ." The applicable DQI credit is "symbolic DQI lowering, and the source context for the portable adapters used in V4." It maps to tasks/dqi/bm_forney/symbolic_runtime.py and its use by tasks/dqi/generate.py. This is the transitive lowering lineage; the identical immediate embed in Kozykorpesh\'s Complete-Domain Affine Compilation v2 remains credited above. The original passage\'s Gray-curvature, signed-Givens, difference-bank and scratch-pebbling credits concern other tasks and are not DQI input claims.\n- Gluzdov V6: Inherited through anazemcev v4 cell 45: "Dmitrii Gluzdov V6/run361592338, Apache2.0: https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361592338 . Compact inverse-norm schedules, template-word loans and streamed sweep, building on the credited Gu decoder and earlier competition-field port." This schedule lineage maps to tasks/dqi/bm_forney/round2_gadgets.py, borrow.py and gu_port.py, and the streamed-sweep integration in tasks/dqi/generate.py. The directly used source is the immediate notebook embed; Gluzdov\'s notebook and FeMoCo numerical dataset are not inputs. Our borrow.py is the modified affine-only extract described in the module map.\n- PRA-236 paired J-fraction port: SomeJoint / Agniv Banerjee (decoder/arithmetic code), anazemcev / Artem1410 / ViberTeam (paired-v6 additions), Phuc Thai (pairing and workspace ideas), and Google Quantum AI (helpers), Apache-2.0 code parents; full URLs, notices, hashes and Praxen modifications in tasks/dqi/jfraction/{NOTICE,PROVENANCE.md,SOURCE_MAP.json}; licence text in tasks/dqi/jfraction/LICENSE. Manwithacat receives method credit for the classical-mask projection; projection.py is new code from its mathematical specification, not copied or transliterated from mask_projection.py, and not a strict clean room. No portfolio outputs or compiled circuit inputs are used.\n- PRA-250: Agniv Banerjee (SomeJoint), quantum-circuits-13-54-local-gm-cost-gains, kernel 140675727 v7, https://www.kaggle.com/code/somejoint/quantum-circuits-13-54-local-gm-cost-gains . Apache-2.0 author grant in cell 34; licence/NOTICE in cells 90/91, retained under tasks/dqi/hankel/. Source SHA-256 3b0df548f40676117fb6f53cd7c9ab0fc5ca29b7ed56f90581921e443e92d7eb. Modified production-only port of Hankel/Bezoutian Forney phase, recursive LDU and cyclic Gold-26 arithmetic. Inherited SomeJoint, Phuc Thai and ViberTeam code uses the existing jfraction package; Manwithacat projection remains a credited method independently implemented by Praxen. See tasks/dqi/hankel/PROVENANCE.md and SOURCE_MAP.json for exact scope.\n- PRA-261 Schur continued-fraction decoder: jday96314/QuantumAI commit   2b9227095bf2c76f9f64a572645d7dce54998298, initial_experiments/6/agent14   and the hash-pinned arithmetic dependency paths in tasks/dqi/schur/source-pins.json.   no licence file; used under the board ruling of 2026-10-26, card `aba6c540`,   extending D-PRA228-01; residual risk to prize eligibility accepted by the board.   Decision D-PRA269-01. Notebook lineage/prose: alexchilton/   toffolis-go-15-05-verified-portfolio-llm-gauge, scriptVersionId 362573669,   changelog v36, source SHA-256 28e0f08476df85323f9d25589f3cf781f13b5df105233193edfe8a5d1cff149c,   cell 8; v33 unavailable anonymously. Copy cells 1/5/6 and all prebuilt circuits   excluded. Missing 16-AND cover replaced by our derived, exhaustive-checked   17-AND cover. Existing Gu arithmetic retains its Apache-2.0 attribution.   Full modifications and source boundaries: tasks/dqi/schur/PROVENANCE.md.\n- PRA-293 fused DQI decoder: Agniv Banerjee (SomeJoint), quantum-circuits-21-34-local-gm-fused-dqi, kernel 140675727 v12, https://www.kaggle.com/code/somejoint/quantum-circuits-21-34-local-gm-fused-dqi . Notebook code Apache-2.0 (credits cell; licence text embedded in cell 6, copied to tasks/dqi/fused/LICENSE). Source SHA-256 18d6760953783bec316ddb46266a1b5ccc50c291fbf47ec35f87c73f4d0cf2ac. Build entry blocks/dqi/team-fused/source/exp/decoder/build.py and its traced closure (53 imported modules, 7 JSON layout files) vendored verbatim under tasks/dqi/fused/source/; every file hash pinned in tasks/dqi/fused/SOURCE_MAP.json. Expected circuit SHA-256 7beea291a5de36ef2b4fb50585b2f8d7cce02d3c575b491cf5dcc93e9d9655f6. Design lineage as stated by the author: rebuilt from a pseudocode description of Alex Chilton\'s 883-qubit Schur decoder (jday96314/QuantumAI, no licence; board card aba6c540, D-PRA269-01, D-PRA273-01); mask projection by mrmeows and Ahmed Berat Ozer after manwithacat; earlier links Dmitrii Gluzdov, SomeJoint, ViberTeam and Phuc Thai. Full credits: tasks/dqi/fused/NOTICE and PROVENANCE.md.\n', 'notices/30-dfthcsos.txt': '- PRA-246 CUDA method port: `tasks/dfthcsos/gpu_fit/` ports selected mathematical definitions from anazemcev / Artem1410 / ViberTeam, "Quantum Circuits — 881Q Paired DQI + R6 DF-THC", v6, https://www.kaggle.com/code/anazemcev/quantum-circuits-881q-paired-dqi-r6-df-thc, Apache-2.0, notebook SHA-256 d4a147523ac07cb2dc5fe4bd5dc9c7da48719e34dfc847402628723b37930682; rebuild source dataset https://www.kaggle.com/datasets/anazemcev/vqcc-viberteam-verified-circuits/versions/2, Apache-2.0. Exact per-file hashes and extracted definitions are in `ledger/evidence/pra246/port-map.json`. Inherited credits: Ahmed Beratozer for the progressive quantisation trainer; Kozykorpesh for the augmented column gauge and signed Givens gauge; Sumit Chongder, Kirill Tym and Artem1410 for inherited compiler lineage (no compiler copied here). CUDA continuous/GN/QAT and exact integer repair retain their published mathematical objectives; QR/barrier/gauge/one-body helpers retain the source NumPy/SciPy execution. Praxen adds the explicit declared/Google intake, fixed seed, exclusive persistence, official evaluator adapter and 11-to-8 continuation. No upstream CLI, circuit, numerical payload or hidden loader is copied.\n- PRA-246 declared ship-lane numerical input: Dmitrii Gluzdov, `dmitriigluzdov/vqcc-femoco-public-factorization`, version 8 (datasetId 12805185), https://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization/versions/8, Apache 2.0 as recorded by John Wick on PRA-245. `femoco_thc_r5.npz`, SHA-256 809a992900937425f01a0593803921873f7bb472d590592c378d3ceb23657df5, 1,871,575 bytes, shape (1,496,384), declared 11-bit words. This is a public factor warm start, not own-source training. Chris\'s accepted card a9b01800-986e-4e7f-b0c2-b288c4591b4e permits it as a declared non-circuit input. The separate Google-only run uses only the previously pinned Google baseline and fixed integral/shift inputs. The k256 artifact is neither loaded nor shipped.\n- PRA-232 module provenance map: `tasks/dfthcsos/qrom.py` is a Praxen reimplementation of truncated algebraic-normal-form (ANF) loading, banked low-address differences, measurement-based ANF erasure and selective ancestor pebbling. Method-only notebook parents (read from the SHA-256-checked source attachments on PRA-194; never executed): kozykappa, "Quantum AI — Complete-Domain Affine Compilation", v2, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation, Apache-2.0, source SHA-256 81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24 (six traversal wires, `gauge_qrom.py`); anazemcev / ViberTeam, "Quantum Circuits — Source & Exact Compilation", v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation, Apache-2.0, SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb (`banked_lab.py`, exact ANF phase repair); magmacot, "Quantum Circuits — clean workspace reuse", v2, https://www.kaggle.com/code/magmacot/quantum-circuits-clean-workspace-reuse, Apache-2.0, SHA-256 f3f393b5c4958f7288f69f68cd7762a62bfcc697d94d7d19607e67ff7ed20d9a (pebbling at prefix degree 4). The banked method\'s upstream attribution includes Kozykorpesh / kozykappa, https://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom?scriptVersionId=361433617, and Sumit Chongder, https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149, read through Kirill Tymchishin\'s QROM + Tower42 v10; retained as credited by these parents, not independently fetched or executed.\n- PRA-232 precision experiment: ahmedberatozer, "Quantum AI — DF-THC Group Compression", v1, https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression, Apache-2.0 notebook code, source SHA-256 b221781e9bf8c1c8da8c5251c6556b294ed07def5e57f5a1ea038addbf7a57df, motivates testing 10-bit angles. `qrom.quantize_words` uses nearest-even rescaling of Google\'s baseline integer words, not that notebook\'s trained factors or quantization optimizer. No claim is made that its physical result transfers to Google\'s baseline.\n- `tasks/dfthcsos/circuit.py`, `experiment.py` and PRA-232 tests are Praxen orchestration and verification code. The existing `generate.py` supplies Google\'s attributed batch-XOR schedule and pinned baseline input. `circuit_checks.py` calls Google\'s unchanged verifier math/resource model; it reads the circuit register width and reports static cost estimates. The only numerical input is Google\'s Apache-2.0 `sample_submission/dfthcsos/submission.npz`, SHA-256 14d5cc25eb3bf82a9ccb87c811db28d127bef6d28697dd190dc1e875de3bfe69. No `femoco_public_words.npz`, `dfthc_group_32_60_30.npz`, toolbox selected circuits, chemistry directory or k256 artifact was loaded. Signed-gauge factor changes and factor refitting are not implemented here.\n- `tasks/dfthcsos/generate.py` ports the package\'s ordinary-QROM ANGLE builder (`verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py`), Apache-2.0, same authors and URL. The factorization bytes are the package\'s `sample_submission/dfthcsos/submission.npz` and are not committed.\n- PRA-198 uses Google\'s Apache-2.0 starter kit v4, https://www.kaggle.com/datasets/sohier/verifiable-circuit-compilation-starter-kit/versions/4 (package author Guang Hao Low), as separately declared code and input parents. Original downloaded files and notices are retained unchanged; exact baseline, integral, fixed-shift, encoder and verifier pins are in `ledger/manifests/inputs/pra198-google-v4.json`. Outputs are labelled "own-generated, derived from Google baseline".\n- D(w)-penalized structure refit and norm-ranked group pruning: Ahmed Berat Ozer, https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression, v1 described in PRA-194 and v2 source inspected on 2026-10-25, Apache-2.0 (`src/dfthc_train.py`). PRA-198 reimplements these methods with analytic NumPy gradients and SciPy CPU L-BFGS-B. No public factor data, words, derived target, warm start or checkpoint is used.\n- Effective-norm / Gram-gauge and banked / six-wire QROM method context: Kozykappa, Complete-Domain Affine Compilation v2, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation; anazemcev, Source & Exact Compilation v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation; Apache-2.0 per PRA-194 / PRA-199. PRA-198 ports no Gram-gauge or banked-QROM code; its cost model is ordinary QROM. The module provenance map is `tasks/dfthcsos/FACTOR_PIPELINE.md`.\n- PRA-242 module provenance map: `tasks/dfthcsos/FIT_V3.md`. `factor_merge.py`, `factor_gauss_newton.py` and `factor_fit_v3.py` reimplement the source-only fitting method of anazemcev / Artem1410 / ViberTeam, Quantum Circuits — 881Q Paired DQI + R6 DF-THC, notebook v6, https://www.kaggle.com/code/anazemcev/quantum-circuits-881q-paired-dqi-r6-df-thc, Apache-2.0, UTF-8 SHA-256 d4a147523ac07cb2dc5fe4bd5dc9c7da48719e34dfc847402628723b37930682, and its hash-checked ViberTeam rebuild source dataset v2. Retain inherited credit to Ahmed Beratozer (structural group-compression / quantisation methods) and Kozykorpesh / Sumit Chongder / Kirill Tym / Artem1410 (compiler lineage); no loader or compiler is copied. Dmitrii Gluzdov, Quantum Circuit Compilation from Source, v14, https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source, Apache-2.0, SHA-256 c3ba0bdde177b5fa159fc641d065c08870020fd773fa0947675675101eb2f52a, informs only the (1,496,384) shape and stage description: the training optimizer is not public. Google\'s pinned baseline is the only numerical initializer. These are bounded CPU adaptations, not exact public-result replays. `factor_measure.py` is an own acceptance wrapper over the unchanged pinned Google evaluator and existing cost model.\n- PRA-245 declared non-circuit input, shipping factor (D-PRA245-01, D-PRA245-12; D-PRA211-05): Dmitrii Gluzdov, "VQCC FeMoCo Angle Words", Kaggle dataset `dmitriigluzdov/vqcc-femoco-public-factorization` (datasetId 12805185), version 8 (2026-10-26T02:51:21Z, notes "Updated circuit sources (78-qubit inverse square root) and refined factorization"), https://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization/versions/8, licence field as Kaggle shows it: `apache-2.0` ("Apache 2.0"). File `femoco_thc_r5.npz`, 1,871,575 bytes, SHA-256 809a992900937425f01a0593803921873f7bb472d590592c378d3ceb23657df5, (R,B,C) = (1,496,384), 11-bit angle words, used unchanged and never committed. Author\'s provenance statement, verbatim from the dataset description: "The femoco_thc_r5 file was trained by the dataset author: a fit of the shifted two-body tensor with a D penalty, progressive 10/11-bit quantization ordered by sensitivity (after ahmedberatozer), exact integer repair of the words, and a coupled refit of weights and offsets with Cayley column rotations (after Artem1410)." Dataset-level sources field, verbatim (it covers the whole dataset, including the 65×60×30 file we do not use, which D-PRA202-03 still governs): "Public Kaggle notebooks (Apache 2.0): k256.dev (refinement of the organizers’ 65 x 60 x 60 baseline); kozykappa, Quantum circuits | sparse SOS factors and circuit (https://www.kaggle.com/code/kozykappa/quantum-circuits-sparse-sos-factors-and-circuit), ensemble compression and identity offsets; Artem1410, Quantum Circuits | Signed-Digit Memory (https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory), effective-norm tuning. Organizers’ starter kit and verifiers: https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation". Earlier versions of the same file were measured, not shipped: v4 a29cb868d59906bebcda50c1c44c14090d503665700d725027e25016544782fd, v5 a0da3225ba2047f43ae24f31881f2a71500d3f73eda24fadc73e9f67da68e13f, v6 1b97a9aec279618b9cf66f28960b3c85fae98d73dde31cd01febe2bc30352778, v7 7db33b3eb951ac9c332aa4f7df8da106f9a7627e5b17a1c6f537e4f9b5e3084f.\n- PRA-245 declared non-circuit input, measured fallback (D-PRA245-12): Artem1410 (anazemcev) / ViberTeam, "VQCC ViberTeam R6 DF-THC and Verified Circuits", Kaggle dataset `anazemcev/vqcc-viberteam-verified-circuits` (datasetId 12845740), version 2 (2026-10-25T15:10:31Z, notes "v2: add single-group THC (1,496,384) chemistry with our fixed-word joint norm refit (8.0935e8 FULL-PASS), credited copy of Dmitrii Gluzdov femoco_thc_r5.npz"), https://www.kaggle.com/datasets/anazemcev/vqcc-viberteam-verified-circuits/versions/2, licence field as Kaggle shows it: `apache-2.0` ("Apache 2.0"). File `thc_submission.npz`, 1,842,734 bytes, SHA-256 9b98b6284b2a846a6a8aa229f4c5a1da62aeba780b57c7341e49b8c3bfd53576 (no `angle_words`; its words were read for measurement only from the same version\'s `thc_selected_compact.npz`, 1,884,386 bytes, SHA-256 141d368ebd6517d3875a38e4e416b4886adb1efc7bd3df901e69436a96af49a9). Author\'s provenance statement, verbatim from the dataset description: "**Credits:** Dmitrii Gluzdov supplies the single-group factorization **and ANF lookup compiler** (`chemistry_r5.py`, as embedded in Ahmed\'s v4), Apache-2.0. His original public v4 `femoco_thc_r5.npz` is redistributed unchanged as `thc_femoco_thc_r5.npz`; SHA256 `a29cb868d59906bebcda50c1c44c14090d503665700d725027e25016544782fd`. Ahmed Berat Ozer supplies the v4 integration idea and wrapper. Artem Nazemtsev supplies the ANF/measurement-erasure lineage; Kozykorpesh supplies pebbling and signed gauge. ViberTeam K adds the independent complete-H joint W/native-offset refit adapted from F/H. See `thc_NOTICE.txt` and `LICENSE`." We retain that inherited credit (Gluzdov; Ahmed Berat Ozer; Artem Nazemtsev; Kozykorpesh; ViberTeam K; Phuc Thai per `thc_NOTICE.txt`). Their circuits (`thc_angle_k_*.kmb`) are not inputs and are not used (D-PRA211-01).\n- PRA-251: `tasks/dfthcsos/fit_jax/` is a JAX reimplementation of Google\'s Apache-2.0 verifier formulas and half-up Givens-word encoder from starter kit v4 (https://www.kaggle.com/datasets/sohier/verifiable-circuit-compilation-starter-kit/versions/4), with unchanged source hashes in `ledger/manifests/inputs/pra198-google-v4.json`. The device-side assignment solver is our implementation of shortest augmenting paths; SciPy is the independent test oracle. No notebook training code was copied or executed for this loss-only deliverable. JAX is Apache-2.0 (https://github.com/jax-ml/jax); custom identity JVP implements the documented straight-through heuristic. Numerical mirror input: Dmitrii Gluzdov, Apache-2.0 dataset `dmitriigluzdov/vqcc-femoco-public-factorization`, version 8, https://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization/versions/8, `femoco_thc_r5.npz`, SHA-256 809a992900937425f01a0593803921873f7bb472d590592c378d3ceb23657df5, declared non-circuit input under D-PRA211-05, supplied by PRA-245. No competition circuit is used as an input or shipped by the mirror.\n- PRA-294: `tasks/dfthcsos/kernel_basis.py`, `kernel_plan.py`, `streamed_qrom.py` and the plan loop in `c59.py` port mrmeows\' exact four-word compiler: `streamed_native.py` (emitter), `balanced_plan.py` and `streamed_plan.py` (planners), `kernel_planes.py`, `hyper_kernel.py`, `pair_exchange.py` and `nullspace` from `rank_parity.py`, all from `sources/dfthcsos_cost_first_shape_screen/source/` in https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits?datasetVersionNumber=8, dataset licence field `Apache 2.0`. Per-file SHA-256 hashes are in `ledger/manifests/inputs/pra294-c59.json`. Changes: Praxen module split and naming, the two planners share one exact-reconstruction tail, reuse of Praxen\'s delta batching, gauge and family writer. The 19 first-stage seeds and constant stage sizes come from that version\'s `certificates/plan_k.json`; no certificate terms, circuits or audits are inputs, and all outputs are regenerated.\n- PRA-294 declared non-circuit input (D-PRA211-05; recorded, not gated, under R-P2-6): mrmeows, `mrmeows/qcc-1375-open-circuits` version 8, `circuits/dfthcsos/submission.npz` (same bytes as `sources/dfthcsos_8p6121_C59_submission/dfthcsos/submission.npz`), 2,232,244 bytes, SHA-256 8229861b3ee22316e144c900737dfc54f7a30ea88de77ff5391f3375d050fa87, (R,B,C) = (10,129,59), 8-bit words, used unchanged and never committed. Licence field `Apache 2.0`. Lineage for the Adjudicator: Michael Rustad\'s DF-THC R10 merged groups v1 factor (the PRA-280 input, licence field "Other (specified in description)", which credits Ahmed Berat Ozer\'s DF-THC Group Compression), refit by mrmeows and merged from C=60 to C=59 columns with the lower-normalization words (v8 README update of 27 October 2026).\n- PRA-321: `tasks/dfthcsos/full_xor/` and `tasks/dfthcsos/b143.py` port SomeJoint\'s full-XOR five-word compiler. Notebook `somejoint/quantum-circuits-22-67-local-gm-full-xor-chem`, kernel 140675727 version 18, scriptVersionId 363182923, https://www.kaggle.com/code/somejoint/quantum-circuits-22-67-local-gm-full-xor-chem, author Agniv Banerjee. Notebook source SHA-256 b44fc04a99ab0c18afce4c7d8408ddc079dd8a87957e294d2102494d8d9b001a. Notebook code Apache-2.0 (credits cell). Embedded files, with upstream SHA-256, are listed in `ledger/manifests/inputs/pra309-chem-8-143-60.json`. Changes: package layout and import paths. The product-basis search is not in the published source; each circuit\'s plan and schedule are committed in `tasks/dfthcsos/full_xor/plans.json`, taken from dataset `BUILD.json` with circuit hashes, proofs and audits removed. Those dropped fields are not inputs.\n- PRA-321 declared non-circuit input (D-PRA211-05): Agniv Banerjee (SomeJoint), `somejoint/qcc-our-chemistry-b143-c60-xor2` version 1 (2026-10-28T13:11:51.017Z), https://www.kaggle.com/datasets/somejoint/qcc-our-chemistry-b143-c60-xor2, licence field Apache 2.0. File `submission.npz`, 1,995,236 bytes, SHA-256 aef36d39b83bbbf696e909884218f6c02f067baef14a29de1c41306bc1f6395b, (R,B,C) = (8,143,60), 8-bit words, used unchanged and never committed. The published notebook does not include the weight fit, so the file is not regenerated. Their ANGLE circuits are not inputs. The dataset contains the factor and `BUILD.json` only. The full-XOR compiler is the notebook above, not the dataset.\n', 'notices/35-chem-catchup.txt': "- PRA-280: tasks/dfthcsos/paired_algebra.py and paired_qrom.py port the deterministic GF(2) factorization and HMR emitter from mrmeows, https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits?datasetVersionNumber=6 source/chemistry_6p69/reproduction/rank_parity.py and hyper_native.py. projective_gauge.py ports the exact integer sign gauge from source/chemistry_four_address/compile_four_address.py in that dataset. Dataset licence field: Apache 2.0. Full source hashes are recorded in ledger/manifests/inputs/pra280-rustad.json. Changes: three-word API, no synthesis search, no external certificates, no circuit inputs, reuse Praxen's delta batching and family writer. All outputs are regenerated.\n- Declared non-circuit input: Michael Rustad, DF-THC R10 merged groups v1, https://www.kaggle.com/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups?datasetVersionNumber=1 fact/dfthc_r10_b129_c60_brot8.npz. Licence field recorded as found: Other (specified in description). The README attributes factor lineage to Ahmed Berat Ozer's DF-THC Group Compression and competition reference data, and says their terms apply. Input SHA-256 is pinned in the input manifest. Licence is recorded, not gated, under Chris's PRA-273 directive.\n- Method reference: alexchilton/toffolis-go-18-3-schur-dqi-at-883-qubits, v37 / run 362577442. It combines rustad's factor and mrmeows's compiler at three words/circuit. No notebook-code licence stated; portfolio circuits labelled Apache-2.0 in credits. No notebook circuit or private gauge used.\n", 'notices/40-search.txt': "- search/ (dfthcsos OpenEvolve search loop, 2026-10-25): OpenEvolve 0.4.0, Apache-2.0, https://github.com/algorithmicsuperintelligence/openevolve (PyPI `openevolve`), is a pinned dependency in the `search` group of pyproject.toml, not vendored. `search/providers.py` is Praxen code: its Codex client speaks the JSON-RPC protocol printed by `codex app-server generate-json-schema` (Codex CLI 0.158.0); OpenEvolve's unreleased `openevolve/llm/codex_app_server.py` (Apache-2.0, main branch, read 2026-10-25) informed the method names and the read-only, never-approve thread settings, and no code was copied. The island, MAP-Elites and discovery-notes search design follows Gu, Kemeny, Yelin and Lukin, arXiv:2610.22367 (method only; no code or circuit from that work is used).\n- search/ (PRA-315 dense fitness + staged evaluator, 2026-10-28): Deb-layered `combined_score`, MAP-Elites on `log_toffoli`/`log_qubits`, and the four-stage evaluator follow the assembly in the internal research report §6.2–6.4 (constraint-handling rules of Deb & Agrawal; Gu et al. MAP-Elites grid; IBM/ShinkaEvolve hash memo). No third-party code was copied. Task adapters call the committed generators under `tasks/{invsqrt,dfthcsos,dqi}/`.\n- search/ (PRA-334 cascade correctness, 2026-10-29): Reduced stage-2 results drive the stage-2 score; adapter cascade owns and deletes the stage0 build dir; public cascade stages catch exceptions like `search/evaluator.py`; verifier memo keys bind shot counts via `TaskAdapter.reduced_shots` / `full_shots`. No third-party code was copied.\n", 'notices/90-not-included.txt': '- The k256.dev decoder and the Signed-Digit Memory v3 chemistry factor are not included. See `docs/DECISIONS.md` D-PRA177-01.\n', 'tasks/__init__.py': '"""Competition task packages."""\n', 'tasks/dfthcsos/b143.py': '# SPDX-License-Identifier: Apache-2.0\n"""Build PRA-321 chemistry: the pinned (8, 143, 60) factor through the full-XOR J=5 compiler.\n\nExample: python -m tasks.dfthcsos.b143 --data /home/cld-main/work/vqcc-data\n  --out /tmp/pra321-build --dry-run\nRequires NumPy, organizer Kickmix, and the declared factor under --data.\nWrites only a new OUT/dfthcsos directory. Invalid inputs or an occupied output\nfail nonzero. The factor is copied unchanged. Every circuit is regenerated from\nthe committed plan and schedule in full_xor/plans.json (no circuit, certificate\nor .kmb is an input).\n"""\nimport argparse\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport shutil\n\nimport numpy as np\n\nfrom tasks.dfthcsos.full_xor.emitter import build_staged\nfrom tasks.dfthcsos.full_xor.planio import plan_from_dict\nfrom tasks.dfthcsos.full_xor.validation import assert_independent\nfrom tasks.dfthcsos.generate import DEFAULT_DATA, sha256_file\n\nINPUT_PATH = Path(\'inputs/somejoint/qcc-our-chemistry-b143-c60-xor2-v1/submission.npz\')\nINPUT_SHA256 = \'aef36d39b83bbbf696e909884218f6c02f067baef14a29de1c41306bc1f6395b\'\nPLANS_PATH = Path(__file__).resolve().parent / \'full_xor\' / \'plans.json\'\nBATCH_SIZE = 5\nADDRESSES = 1220\n# Circuits the emitter must reproduce. These are checks, not inputs.\nCIRCUIT_SHA256 = (\n    \'d0f7e6bf8057b171c22a0cbef041ccecf1334fb17e968fd924286ff70d66d08a\',\n    \'468475d7214894af0616870bdce211799bc2e97682699c97f89d5afbc20824f8\',\n    \'f31de4b7672f853cbecc21be77e30cb6e1ecc837a5d4680ffee77fe7d147bf14\',\n    \'3b175fb0abaa7e20fe22d021c77cb27e16658fd088d6f78bf307286e50319736\',\n    \'2990dbbf6177c166e32426803dd6f2a969659238b580a8500b9359cb12252ab2\',\n    \'b4d36019132b43661850f984cf8eb723482cfb0af7eeb4cadb0964b6cc103e2a\',\n    \'5c0bd4a2cd9436c925569a33044bac964653709b38a84d1df57e15bb2d60d6c6\',\n    \'dead802918d2183ecb2d3c96d6ea1df5ba7e5240d3a20cf63dbfc8fa146bb05f\',\n    \'d2e22516ab580482d755dca2f1f999cb2acb83cffda74df0ef1d7f0531fa3a73\',\n    \'663497ad211daa8803a60e1cfff5d4e760499f971054c2908fe3a59489aa597c\',\n    \'2aa48c950a5d0850ddd544886eb5720f4a92f0b7cf496f182f4b9b9a980a8de7\',\n    \'78c041703bbc43f795ae4ab9224a15e2a6dc97d60c76cf9927226de6c959a466\',\n    \'e43da6265070311362619ce8c57f780e41f519d0be0590049bba8c2c885c41fd\',\n    \'fbdac0002c7cd3e66487f966ccac1edf2ac40f628fb7d22af048c1ce66f60219\',\n    \'8b6d5738e981696154031d1f7c71b90fcde08ffd2fea1ae8525b545fd2c6c5c1\',\n)\nSUBMISSION_SHA256 = \'c88f058efddd1b273a7161ebdb57740ff3da36452acf337995c19ff67951a414\'\n\n\ndef load_plans():\n    """The 15 committed plans. Raises if a record is not a replayable J=5 plan."""\n    rows = json.loads(PLANS_PATH.read_text())\n    if len(rows) != len(CIRCUIT_SHA256):\n        raise ValueError(f\'{len(rows)} plans but {len(CIRCUIT_SHA256)} circuit checks\')\n    return rows\n\n\ndef batch_deltas(words):\n    """XOR of consecutive five-word batches. Batch 0 is the words themselves."""\n    if words.shape != (ADDRESSES, 75):\n        raise ValueError(f\'angle_words shape {words.shape}, expected {(ADDRESSES, 75)}\')\n    batches = np.asarray(words, np.int64).reshape(ADDRESSES, 15, BATCH_SIZE)\n    deltas = batches.copy()\n    deltas[:, 1:] ^= batches[:, :-1]\n    return deltas\n\n\ndef compile_batch(delta, row):\n    """Emit one circuit from a committed plan and schedule. Raises unless it is exact."""\n    if hashlib.sha256(np.ascontiguousarray(delta).tobytes()).hexdigest() != row[\'delta_sha256\']:\n        raise RuntimeError(f"batch {row[\'batch\']} delta does not match the committed plan")\n    plan = plan_from_dict(row[\'plan\'])\n    assert_independent(plan, delta)\n    circuit, metadata = build_staged(plan, row[\'schedule\'])\n    stream = io.BytesIO()\n    circuit.to_file(stream, format=\'kmb\')\n    blob = stream.getvalue()\n    digest = hashlib.sha256(blob).hexdigest()\n    if digest != CIRCUIT_SHA256[row[\'batch\']]:\n        raise RuntimeError(f"angle_k_{row[\'batch\']} sha256 {digest} is not the recorded circuit")\n    if metadata[\'Q\'] > 128:\n        raise RuntimeError(f"angle_k_{row[\'batch\']} uses {metadata[\'Q\']} qubits")\n    report = dict(batch=row[\'batch\'], toffoli_per_call=metadata[\'T\'], num_qubits=metadata[\'Q\'],\n                  sha256=digest, products=metadata[\'products\'])\n    return circuit, blob, report\n\n\ndef generate(out, *, data=DEFAULT_DATA, dry_run=False):\n    source = Path(data) / INPUT_PATH\n    if sha256_file(source) != INPUT_SHA256:\n        raise ValueError(\'B143 factor SHA-256 mismatch; restore the declared input\')\n    destination = Path(out) / \'dfthcsos\'\n    if destination.exists() and any(destination.iterdir()):\n        raise ValueError(\'output dfthcsos directory must be empty; use a fresh directory\')\n    with np.load(source, allow_pickle=False) as archive:\n        words = np.asarray(archive[\'angle_words\'])\n        brot = int(archive[\'brot\'])\n        edges = archive[\'tree_edges\']\n        payload = {key: archive[key] for key in archive.files}\n    if brot != 8:\n        raise ValueError(f\'brot {brot}, expected 8\')\n    deltas = batch_deltas(words)\n    built = [compile_batch(deltas[:, k], row) for k, row in enumerate(load_plans())]\n    from dfthcsos.scoreinfra.circuits import recover_words_from_circuits, submission_sha256\n    circuits = [circuit for circuit, _, _ in built]\n    recovered, _, _ = recover_words_from_circuits(\n        circuits, num_vectors=len(words), num_angles=75, shots_per_address=1, seed=1901728)\n    if not np.array_equal(recovered, words):\n        raise RuntimeError(\'the circuits do not load the submitted angle words\')\n    digest = submission_sha256(circuits, payload, edges)\n    if digest != SUBMISSION_SHA256:\n        raise RuntimeError(f\'submission digest {digest} is not {SUBMISSION_SHA256}\')\n    report = dict(input_sha256=INPUT_SHA256, brot=brot, batch_size=BATCH_SIZE,\n                  submission_sha256=digest, circuits=[row for _, _, row in built], dry_run=dry_run)\n    if dry_run:\n        return report\n    destination.mkdir(parents=True, exist_ok=True)\n    factor = destination / \'submission.npz\'\n    shutil.copyfile(source, factor)\n    paths = [factor]\n    for _, blob, row in built:\n        path = destination / f"angle_k_{row[\'batch\']}.kmb"\n        path.write_bytes(blob)\n        paths.append(path)\n    report[\'files\'] = {str(path.relative_to(out)): dict(sha256=sha256_file(path), size=path.stat().st_size)\n                       for path in paths}\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--out\', type=Path, required=True)\n    parser.add_argument(\'--data\', type=Path, default=DEFAULT_DATA)\n    parser.add_argument(\'--dry-run\', action=\'store_true\', help=\'Validate and compile without writing files.\')\n    args = parser.parse_args()\n    print(json.dumps(generate(args.out, data=args.data, dry_run=args.dry_run), indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'tasks/dfthcsos/full_xor/__init__.py': '# SPDX-License-Identifier: Apache-2.0\n"""Full-XOR J=5 angle loader ported from SomeJoint (Apache-2.0)."""\n', 'tasks/dfthcsos/full_xor/emitter.py': '# SPDX-License-Identifier: Apache-2.0\n"""Staged arbitrary-target XOR emitter with clean dedicated scratch.\n\nAll del_and calls have their original AND parents live. Its documented HMR+\nconditional-CZ cleanup returns the child to zero; the fixed slot can then be\nreinitialized. Product parity maps are restored before any cache transition.\n"""\nimport kickmix as km\nfrom tasks.dfthcsos.full_xor.paired_anf import bits, Plan\nfrom tasks.dfthcsos.full_xor.scheduler import audit, parent, members\n\ndef require(ok, message):\n    """Explicit check. `assert` disappears under `python -O`; emission checks must not."""\n    if not ok:\n        raise RuntimeError(message)\n\ndef high_masks(addresses):\n    return [h<<4 for h in range(1,((int(addresses)-1)>>4)+1)if h.bit_count()>=2]\n\ndef lifted_plan(plan,addresses=1220):\n    """A defined polynomial-extension fixture, NOT a new factorization table."""\n    if plan.blocks is not None or plan.low_bits!=4:raise ValueError(\'need unblocked low4 plan\')\n    if addresses<plan.addresses:raise ValueError(\'use exact prefix restriction before lifting\')\n    cache=[m for m in range(1,16)if m.bit_count()>=2]+high_masks(addresses)\n    return Plan(int(addresses),plan.batch,plan.brot,4,list(plan.products),dict(plan.pure),list(plan.ranks),cache,parity_scratch=0)\n\ndef build_staged(plan,schedule,addresses=None):\n    audit(plan,schedule)\n    if plan.batch!=5 or plan.brot not in (7,8):raise ValueError(\'five 7/8-bit words required\')\n    brot=plan.brot\n    if not 1<=schedule[\'capacity\']<=11:raise ValueError(\'low cache capacity must be1..11\')\n    A=plan.addresses if addresses is None else int(addresses)\n    if A!=plan.addresses:raise ValueError(\'lift the Plan explicitly before emission\')\n    high=[m for m in plan.cache_masks if m>=16]\n    if (A-1).bit_length()+5*brot+len(high)+schedule[\'capacity\']>128:raise ValueError(\'cache exceeds 128-qubit layout\')\n    from tasks.dfthcsos.full_xor.validation import independent_truth\n    for value in independent_truth(plan):\n        if any(((value>>(brot*j+brot-1))&1)!=((value>>(brot*j+brot-2))&1)for j in range(5)):\n            raise ValueError(\'delta is not canonical sign encoded\')\n    if plan.low_bits!=4 or plan.blocks is not None or plan.parity_scratch!=0:\n        raise ValueError(\'need in-place unblocked low4 plan\')\n    b=km.CircuitBuilder();selection=b.create_quantum_register((A-1).bit_length(),\'selection\')\n    words=[b.create_quantum_register(plan.brot,f\'word_{j}\')for j in range(plan.batch)]\n    targets=[q for word in words for q in word]\n    keep=(1<<(5*brot))-1\n    cache={0:True,**{1<<i:q for i,q in enumerate(selection)}}\n    high_pool=b.alloc_qubits(len(high));low_extra=b.alloc_qubits(schedule[\'capacity\']);low_pool=list(low_extra)\n    for m,q in zip(high,high_pool):\n        top=1<<(m.bit_length()-1)\n        b.init_and(cache[m^top],cache[top],q);cache[m]=q\n    low={};free=list(low_pool);computes=0;erases=0;peak=0;product_count=0;controls_checked=0\n    word_ids={q.id for q in targets}\n    def transition(target_mask):\n        nonlocal computes,erases,peak\n        wanted=set(members(target_mask))\n        for m in sorted(set(low)-wanted,reverse=True):\n            top=1<<(m.bit_length()-1);q=low.pop(m)\n            require(m^top in cache and top in cache, f\'erasing {m} without both parents cached\')\n            require(not any(parent(x)==m for x in low), f\'erasing {m} while a child is still live\')\n            b.del_and(cache[m^top],cache[top],q);del cache[m];free.append(q);erases+=1\n        for m in sorted(wanted-set(low)):\n            top=1<<(m.bit_length()-1)\n            require(m^top in cache and top in cache and free, f\'computing {m} without parents and a free slot\')\n            q=free.pop();b.init_and(cache[m^top],cache[top],q);low[m]=q;cache[m]=q;computes+=1\n            peak=max(peak,len(low))\n    def pure_fanout(mask,fanout):\n        for k in bits(fanout&keep):b.cx(cache[mask],targets[k])\n    def product(p):\n        nonlocal controls_checked,product_count\n        require(not((int(p.right1)|int(p.right2))&1), \'constant product control unsupported\')\n        aa={cache[m].id for m in bits(p.left1)}\n        aa.update(cache[h<<4].id for h in bits(p.right2))\n        bb={cache[m].id for m in bits(p.left2)}\n        bb.update(cache[h<<4].id for h in bits(p.right1))\n        require(aa and bb and aa!=bb, \'product controls are empty or identical\')\n        require(not((aa|bb)&word_ids), \'nonborrowed output used as a control/workspace\')\n        pp=min(aa);use_a=pp in bb;bprime=bb^aa if use_a else bb\n        require(bprime, \'product controls collapsed to one wire\')\n        qq=min(bprime)\n        inverse_ops=[(km.q(t),km.q(pp))for t in sorted(aa-{pp})]\n        inverse_ops.extend((km.q(t),km.q(qq))for t in sorted(bprime-{qq}))\n        if use_a:inverse_ops.append((km.q(pp),km.q(qq)))\n        for control,target in inverse_ops:b.cx(control,target)\n        out=list(bits(p.fanout&keep))\n        require(out, \'product fanout is empty\')\n        pivot=targets[out[0]]\n        require(pivot.id not in(aa|bb), f\'pivot {pivot.id} is also a control\')\n        for k in out[1:]:b.cx(pivot,targets[k])\n        b.ccx(km.q(pp),km.q(qq),pivot)\n        for k in out[1:]:b.cx(pivot,targets[k])\n        for control,target in reversed(inverse_ops):b.cx(control,target)\n        product_count+=1;controls_checked+=len(aa|bb)\n    for event in schedule[\'events\']:\n        transition(event[\'cache\'])\n        if event[\'kind\']==\'products\':\n            for index in event[\'indices\']:product(plan.products[index])\n        else:\n            m=event[\'monomial\'];pure_fanout(m,plan.pure[m])\n    transition(0)\n    require(not low and len(free)==schedule[\'capacity\'],\n            f"low cache not returned: live {sorted(low)} free {len(free)} != {schedule[\'capacity\']}")\n    # Nonlinear low terms already streamed after every product. Constant/linear\n    # low terms and pure-high terms require no low slot and commute with them.\n    for m,fan in plan.pure.items():\n        if m<16 and m.bit_count()>=2:continue\n        pure_fanout(m,fan)\n    b.free(low_extra)\n    for m in reversed(high):\n        top=1<<(m.bit_length()-1)\n        require(m^top in cache and top in cache, f\'releasing high monomial {m} without both parents\')\n        b.del_and(cache[m^top],cache[top],cache[m]);del cache[m]\n    b.free(high_pool)\n    circuit=b.finish_circuit()\n    expected_T=len(high)+computes+len(plan.products)\n    expected_Q=(A-1).bit_length()+plan.batch*plan.brot+len(high)+schedule[\'capacity\']\n    require(computes==erases==schedule[\'low_computations\'] and product_count==len(plan.products),\n            f"emitted counts computes={computes} erases={erases} products={product_count} "\n            f"!= schedule {schedule[\'low_computations\']} products {len(plan.products)}")\n    require(expected_Q<=128, f\'emitted qubit count {expected_Q} exceeds 128\')\n    require(int(circuit.num_qubits)==expected_Q,\n            f\'circuit qubits {int(circuit.num_qubits)} != expected {expected_Q}\')\n    emitted_t=int(circuit.operation_statistics().num_toffoli_like)\n    require(emitted_t==expected_T, f\'circuit toffolis {emitted_t} != expected {expected_T}\')\n    return circuit,dict(addresses=A,high_cache=len(high),low_slots=schedule[\'capacity\'],peak_low_live=peak,\n        low_computations=computes,low_measurement_erasures=erases,products=product_count,\n        T=expected_T,Q=expected_Q,all_measurement_erase_parents_live=True,\n        only_redundant_sign_outputs_borrowed=False,borrowed_top_bits=0,promise_top_two_bits_equal=False,no_public_carry=True,borrowed_not_freed=True,control_wire_supports_checked=controls_checked,\n        emitted_gate_count_checked=True,native_validation_run=False)\n', 'tasks/dfthcsos/full_xor/paired_anf.py': '# SPDX-License-Identifier: Apache-2.0\n"""Exact split-ANF XOR lookups.\n\nEvery claimed compiler equality is over GF(2), on all addresses 0<=a<A.\nThe construction never reads or modifies the target while forming a control,\nso it implements XOR on every initial target state.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport math\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport numpy as np\nimport kickmix as km\n\n# Verifier package (installed beside Kickmix), not tasks.dfthcsos.\nfrom dfthcsos.scoreinfra.circuits import recover_words_from_circuits, circuit_costs\n\n\ndef bits(n: int) -> Iterable[int]:\n    while n:\n        t = n & -n\n        yield t.bit_length() - 1\n        n ^= t\n\n\ndef anf(words: np.ndarray, brot: int) -> list[int]:\n    """One packed output coefficient per monomial, masks <A.\n\n    [0,A) is downward closed in bit-subset order, so in-place Moebius\n    inversion needs no arbitrary unused-address padding.\n    """\n    words = np.asarray(words, dtype=np.int64)\n    a, j = words.shape\n    out = [sum(int(words[i, w]) << (w * brot) for w in range(j)) for i in range(a)]\n    for b in range((a - 1).bit_length()):\n        for mask in range(a):\n            if mask & (1 << b):\n                out[mask] ^= out[mask ^ (1 << b)]\n    return out\n\n\ndef coefficient_polys(coeffs: list[int], output_bits: int) -> list[int]:\n    return [sum(1 << mask for mask, c in enumerate(coeffs) if (c >> out) & 1)\n            for out in range(output_bits)]\n\n\ndef output_basis(polys: list[int]) -> tuple[list[int], list[int]]:\n    """Independent output polynomials and masks describing their fanout."""\n    pivots: dict[int, tuple[int, int]] = {}\n    basis: list[int] = []\n    coords: list[int] = []\n    for poly in polys:\n        x, coord = poly, 0\n        while x:\n            p = x.bit_length() - 1\n            if p not in pivots:\n                k = len(basis)\n                basis.append(x)\n                pivots[p] = x, 1 << k\n                coord ^= 1 << k\n                break\n            row, rowcoord = pivots[p]\n            x ^= row\n            coord ^= rowcoord\n        coords.append(coord)\n    fanouts = [sum(1 << j for j, c in enumerate(coords) if (c >> i) & 1)\n               for i in range(len(basis))]\n    for p, c in zip(polys, coords):\n        z = 0\n        for i in bits(c):\n            z ^= basis[i]\n        assert z == p\n    return basis, fanouts\n\n\ndef matrix(poly: int, low_bits: int, addresses: int) -> tuple[list[int], int, int]:\n    """Bilinear matrix columns (nonconstant high monomials), pure halves."""\n    stride = 1 << low_bits\n    high_max = (addresses - 1) >> low_bits\n    columns = [0] * high_max\n    pure_lo, pure_hi = 0, 0\n    for m in bits(poly):\n        l, h = m & (stride - 1), m >> low_bits\n        if h == 0:\n            pure_lo ^= 1 << l\n        elif l == 0:\n            pure_hi ^= 1 << h\n        else:\n            columns[h - 1] ^= 1 << l\n    return columns, pure_lo, pure_hi\n\n\ndef rank_decomposition(columns: list[int]) -> list[tuple[int, int]]:\n    """Exact A=Σ l_i r_i^T, l_i and r_i packed by monomial masks."""\n    pivots: dict[int, tuple[int, int]] = {}\n    lefts: list[int] = []\n    rights: list[int] = []\n    for col_index, original in enumerate(columns, 1):\n        x, coord = original, 0\n        while x:\n            p = x.bit_length() - 1\n            if p not in pivots:\n                k = len(lefts)\n                lefts.append(x)\n                rights.append(0)\n                pivots[p] = x, 1 << k\n                coord ^= 1 << k\n                break\n            row, rowcoord = pivots[p]\n            x ^= row\n            coord ^= rowcoord\n        for k in bits(coord):\n            rights[k] ^= 1 << col_index\n    for j, col in enumerate(columns, 1):\n        rebuilt = 0\n        for l, r in zip(lefts, rights):\n            if (r >> j) & 1:\n                rebuilt ^= l\n        assert rebuilt == col\n    return list(zip(lefts, rights))\n\n\ndef poly_product(a: int, b: int, max_mask: int) -> int:\n    """Boolean polynomial product: x_S*x_T=x_(S union T)."""\n    out = 0\n    for i in bits(a):\n        for j in bits(b):\n            if (i | j) <= max_mask:\n                out ^= 1 << (i | j)\n    return out\n\n\ndef parity_value(poly: int, address: int) -> int:\n    return sum(1 for m in bits(poly) if address & m == m) & 1\n\n\n@dataclass\nclass Product:\n    left1: int\n    right1: int\n    left2: int\n    right2: int\n    fanout: int\n\n\n@dataclass\nclass Plan:\n    addresses: int\n    batch: int\n    brot: int\n    low_bits: int\n    products: list[Product]\n    pure: dict[int, int]\n    ranks: list[int]\n    cache_masks: list[int]\n    blocks: list[\'Plan\'] | None = None\n    parity_scratch: int = 2\n\n    @property\n    def toffoli(self) -> int:\n        if self.blocks is not None:\n            return sum(p.toffoli for p in self.blocks)\n        return len(self.cache_masks) + len(self.products)\n\n    @property\n    def qubits(self) -> int:\n        base = (self.addresses-1).bit_length() + self.batch*self.brot\n        if self.blocks is not None:\n            return max(p.qubits for p in self.blocks)\n        return base + len(self.cache_masks) + (self.parity_scratch if self.products else 0)\n\n    def evaluate(self, address: int) -> int:\n        if self.blocks is not None:\n            result = 0\n            for p in self.blocks:\n                result ^= p.evaluate(address)\n            return result\n        lo, hi = address & ((1 << self.low_bits)-1), address >> self.low_bits\n        result = 0\n        for p in self.products:\n            if ((parity_value(p.left1, lo) ^ parity_value(p.right2, hi)) &\n                (parity_value(p.left2, lo) ^ parity_value(p.right1, hi))):\n                result ^= p.fanout\n        for mask, fanout in self.pure.items():\n            if mask & address == mask:\n                result ^= fanout\n        return result\n\n\ndef make_plan(delta: np.ndarray, brot: int, low_bits: int, *, mixing_steps: int = 0,\n              share_odd: bool = False,\n              in_place_parity: bool = False,\n              seed: int = 0, poly_override: list[int] | None = None) -> Plan:\n    delta = np.asarray(delta, dtype=np.int64)\n    a, j = delta.shape\n    polys = coefficient_polys(anf(delta, brot), j*brot) if poly_override is None else poly_override\n    funcs, fanouts = output_basis(polys)\n    if mixing_steps:\n        funcs, fanouts = mix_output_basis(funcs, fanouts, low_bits, a, mixing_steps, seed)\n    products: list[Product] = []\n    if share_odd:\n        # For two full row-rank odd matrices, cancel distinct low rows with\n        # a shared rank-two bilinear update. Both row ranks fall by >=1.\n        # Its product-of-affine-forms pure-half terms are also added to the\n        # residual polynomials, so all identities remain exact.\n        low_rank = (1 << low_bits)-1\n        high_rank = (a-1)>>low_bits\n        full_rank = min(low_rank,high_rank)\n        eligible = [i for i,p in enumerate(funcs)\n                    if len(rank_decomposition(matrix(p,low_bits,a)[0])) == full_rank]\n        if full_rank >= 3 and full_rank & 1:\n            for i,jj in zip(eligible[::2],eligible[1::2]):\n                ca=matrix(funcs[i],low_bits,a)[0]\n                cb=matrix(funcs[jj],low_bits,a)[0]\n                if low_rank <= high_rank:\n                    left1,left2=1<<1,1<<2\n                    right1=sum(1 << (h+1) for h,col in enumerate(ca) if (col >> 1)&1)\n                    right2=sum(1 << (h+1) for h,col in enumerate(cb) if (col >> 2)&1)\n                else:\n                    # Same operation on the transpose when the high domain\n                    # is the smaller, full-rank side (e.g. A1108 split6/5).\n                    left1,left2=ca[0],cb[1]\n                    right1,right2=1<<1,1<<2\n                product=Product(left1,right1,left2,right2,fanouts[i]^fanouts[jj])\n                products.append(product)\n                full_poly=0\n                for l in bits(left1):\n                    for h in bits(right1):full_poly ^= 1 << (l | (h << low_bits))\n                for l in bits(left2):\n                    for h in bits(right2):full_poly ^= 1 << (l | (h << low_bits))\n                full_poly ^= poly_product(left1,left2,(1<<low_bits)-1)\n                for h in bits(poly_product(right1,right2,(a-1)>>low_bits)):\n                    full_poly ^= 1 << (h << low_bits)\n                funcs[i] ^= full_poly\n                funcs[jj] ^= full_poly\n    pure: dict[int, int] = {}\n    ranks: list[int] = []\n    lowmax, highmax = (1 << low_bits)-1, (a-1) >> low_bits\n    def add_pure(p: int, fanout: int, shift: int) -> None:\n        for m in bits(p):\n            full = m << shift\n            if full < a:\n                pure[full] = pure.get(full, 0) ^ fanout\n    for poly, fanout in zip(funcs, fanouts):\n        columns, lo, hi = matrix(poly, low_bits, a)\n        factors = rank_decomposition(columns)\n        ranks.append(len(factors))\n        for t in range(0, len(factors), 2):\n            l1, r1 = factors[t]\n            l2, r2 = factors[t+1] if t+1 < len(factors) else (0, 0)\n            products.append(Product(l1,r1,l2,r2,fanout))\n            lo ^= poly_product(l1,l2,lowmax)\n            hi ^= poly_product(r1,r2,highmax)\n        add_pure(lo,fanout,0)\n        add_pure(hi,fanout,low_bits)\n    pure = {m:c for m,c in pure.items() if c}\n    used = set(pure)\n    for p in products:\n        used.update(bits(p.left1 | p.left2))\n        used.update(m << low_bits for m in bits(p.right1 | p.right2))\n    # Downward spanning-tree closure makes every needed monomial in one AND.\n    for m in list(used):\n        while m and m.bit_count() >= 2:\n            used.add(m)\n            m ^= 1 << (m.bit_length()-1)\n    cache_masks = sorted(m for m in used if m.bit_count() >= 2)\n    return Plan(a,j,brot,low_bits,products,pure,ranks,cache_masks,\n                parity_scratch=0 if in_place_parity else 2)\n\n\ndef mix_output_basis(funcs: list[int], fanouts: list[int], low_bits: int,\n                     addresses: int, steps: int, seed: int) -> tuple[list[int],list[int]]:\n    """Monotone elementary GL(m,2) search, exact and reversible."""\n    funcs, fanouts = funcs[:], fanouts[:]\n    cost_cache: dict[int,int] = {}\n    def cost(poly: int) -> int:\n        if poly not in cost_cache:\n            cost_cache[poly] = (len(rank_decomposition(matrix(poly,low_bits,addresses)[0]))+1)//2\n        return cost_cache[poly]\n    rng = np.random.default_rng(seed)\n    for _ in range(steps):\n        i,j = map(int,rng.choice(len(funcs),2,replace=False))\n        new = funcs[i] ^ funcs[j]\n        if cost(new) < cost(funcs[i]):\n            funcs[i] = new\n            fanouts[j] ^= fanouts[i]\n    return funcs,fanouts\n\n\ndef build_circuit(plan: Plan) -> km.Circuit:\n    b = km.CircuitBuilder()\n    selection = b.create_quantum_register((plan.addresses-1).bit_length(), \'selection\')\n    words = [b.create_quantum_register(plan.brot, f\'word_{j}\') for j in range(plan.batch)]\n    targets = [q for word in words for q in word]\n\n    def compile_one(p: Plan) -> None:\n        cache: dict[int,object] = {0:True, **{1<<i:q for i,q in enumerate(selection)}}\n        allocated = b.alloc_qubits(len(p.cache_masks))\n        for mask,q in zip(p.cache_masks,allocated):\n            top = 1 << (mask.bit_length()-1)\n            b.init_and(cache[mask ^ top], cache[top], q)\n            cache[mask] = q\n        parity = b.alloc_qubits(p.parity_scratch if p.products else 0)\n        def xorpoly(poly: int, shift: int, target: object) -> None:\n            for mask in bits(poly):\n                b.cx(cache[mask << shift],target)\n        for prod in p.products:\n            inverse_ops=[]\n            if p.parity_scratch:\n                control1,control2=parity[0],parity[1]\n                xorpoly(prod.left1,0,control1); xorpoly(prod.right2,p.low_bits,control1)\n                xorpoly(prod.left2,0,control2); xorpoly(prod.right1,p.low_bits,control2)\n            else:\n                # Two independent linear forms can be materialized by an\n                # invertible CNOT map on existing cache/address wires.\n                # No clean parity ancilla is needed. Restore every wire\n                # immediately after the arbitrary-target CCX fanout.\n                aa={cache[m].id for m in bits(prod.left1)}\n                aa.update(cache[m << p.low_bits].id for m in bits(prod.right2))\n                bb={cache[m].id for m in bits(prod.left2)}\n                bb.update(cache[m << p.low_bits].id for m in bits(prod.right1))\n                pp=min(aa)\n                use_a=pp in bb\n                bprime=bb^aa if use_a else bb\n                assert bprime, \'dependent product controls should have been simplified\'\n                qq=min(bprime)\n                inverse_ops=[(km.q(t),km.q(pp)) for t in sorted(aa-{pp})]\n                inverse_ops += [(km.q(t),km.q(qq)) for t in sorted(bprime-{qq})]\n                if use_a:inverse_ops.append((km.q(pp),km.q(qq)))\n                for control,target in inverse_ops:b.cx(control,target)\n                control1,control2=km.q(pp),km.q(qq)\n            out = list(bits(prod.fanout))\n            pivot = targets[out[0]]\n            # Conjugated fanout implements one arbitrary-target XOR vector\n            # using a single CCX, without any clean product target.\n            for k in out[1:]: b.cx(pivot,targets[k])\n            b.ccx(control1,control2,pivot)\n            for k in out[1:]: b.cx(pivot,targets[k])\n            if p.parity_scratch:\n                xorpoly(prod.left2,0,control2); xorpoly(prod.right1,p.low_bits,control2)\n                xorpoly(prod.left1,0,control1); xorpoly(prod.right2,p.low_bits,control1)\n            else:\n                for control,target in reversed(inverse_ops):b.cx(control,target)\n        b.free(parity)\n        for mask,fanout in p.pure.items():\n            for k in bits(fanout): b.cx(cache[mask],targets[k])\n        for mask in reversed(p.cache_masks):\n            top = 1 << (mask.bit_length()-1)\n            b.del_and(cache[mask ^ top],cache[top],cache[mask])\n        b.free(allocated)\n    for p in plan.blocks or [plan]: compile_one(p)\n    return b.finish_circuit()\n\n\ndef assert_classical(plan: Plan, delta: np.ndarray) -> None:\n    for a,row in enumerate(delta):\n        expected = sum(int(x) << (j*plan.brot) for j,x in enumerate(row))\n        actual = plan.evaluate(a)\n        assert actual == expected, (a,expected,actual)\n\n\ndef batch_deltas(words: np.ndarray, batch: int) -> list[np.ndarray]:\n    a,n=words.shape\n    padded=np.zeros((a,math.ceil(n/batch)*batch),dtype=np.int64)\n    padded[:,:n]=words\n    tables=padded.reshape(a,-1,batch).transpose(1,0,2).copy()\n    tables[1:] ^= tables[:-1].copy()\n    return list(tables)\n\n\ndef signed_gauge(words: np.ndarray, tree_edges: np.ndarray, brot: int) -> np.ndarray:\n    """Exact projector-preserving signed-angle gauge, all arithmetic integral.\n\n    Traverse backwards, propagating endpoint diagonal signs towards the\n    initially-zero vertices. The only surviving initial sign is a global\n    sign on the root. Independent per-angle folding is NOT sufficient.\n    """\n    out=np.asarray(words,dtype=np.int64).copy()\n    edges=np.asarray(tree_edges,dtype=np.int64)\n    assert len(edges)==out.shape[1]\n    signs=np.zeros((out.shape[0],out.shape[1]+1),dtype=bool)\n    period,half,quarter=1<<brot,1<<(brot-1),1<<(brot-2)\n    for k in reversed(range(len(edges))):\n        p,q=map(int,edges[k])\n        effective=np.where(signs[:,p]^signs[:,q],-out[:,k],out[:,k])%period\n        folded=(effective+quarter)%half-quarter\n        toggle=(((effective-folded)//half)&1).astype(bool)\n        signs[:,p]^=toggle;signs[:,q]^=toggle\n        out[:,k]=folded%period\n    return out\n\n\ndef sample_proxy(addresses: int=1366, columns: int=75, *,\n                 preserve_projectors: bool=True) -> np.ndarray:\n    archive=np.load(Path(__file__).resolve().parents[4]/\'sample_submission/dfthcsos/submission.npz\')\n    source=archive[\'angle_words\']\n    # Quantize 12-bit words to eight bits, then fold to [-pi/2,pi/2).\n    # This is a structure proxy, NOT a claimed accurate factorization.\n    source=source[:addresses,:]\n    q=np.rint(source/16).astype(np.int64)\n    if preserve_projectors:\n        q=signed_gauge(q%256,archive[\'tree_edges\'],8)\n    else:\n        q=(((q+64)%128)-64)%256\n    return q[:,:columns]\n\n\ndef run_smoke() -> None:\n    rng=np.random.default_rng(104729)\n    for a,j,br,nangle,lo in [(19,2,4,5,3),(37,3,8,7,3),(130,3,8,8,4)]:\n        words=rng.integers(0,1<<br,(a,nangle),dtype=np.int64)\n        plans=[make_plan(d,br,lo) for d in batch_deltas(words,j)]\n        for p,d in zip(plans,batch_deltas(words,j)): assert_classical(p,d)\n        circuits=[build_circuit(p) for p in plans]\n        recovered,layout,toffoli=recover_words_from_circuits(circuits,num_vectors=a,num_angles=nangle,seed=104729)\n        assert np.array_equal(recovered,words)\n        assert toffoli == 2*sum(p.toffoli for p in plans)\n        assert [p.qubits for p in plans] == [int(c.num_qubits) for c in circuits]\n        print(dict(test=\'smoke\',A=a,J=j,brot=br,angles=nangle,plans=[dict(T=p.toffoli,Q=p.qubits,ranks=p.ranks) for p in plans],recover_T=toffoli),flush=True)\n\n\nif __name__ == \'__main__\':\n    run_smoke()\n', 'tasks/dfthcsos/full_xor/planio.py': '# SPDX-License-Identifier: Apache-2.0\n"""Exact Plan JSON and fixed-schedule carry replay; NO synthesis on replay."""\nfrom dataclasses import asdict\nfrom tasks.dfthcsos.full_xor.paired_anf import Plan,Product,build_circuit\n\ndef plan_dict(plan):return asdict(plan)\ndef plan_from_dict(d):\n required={\'addresses\',\'batch\',\'brot\',\'low_bits\',\'products\',\'pure\',\'ranks\',\'cache_masks\',\'blocks\',\'parity_scratch\'}\n if set(d)!=required:raise ValueError(\'bad saved Plan fields\')\n ints=(\'addresses\',\'batch\',\'brot\',\'low_bits\',\'parity_scratch\')\n if any(type(d[k])is not int for k in ints):raise ValueError(\'noninteger saved Plan routing\')\n if d[\'addresses\']<1 or(d[\'batch\'],d[\'brot\'],d[\'low_bits\'])!=(5,8,4)or d[\'parity_scratch\']not in(0,2):raise ValueError(\'unsupported carry Plan layout\')\n pure={}\n for k,v in d[\'pure\'].items():\n  kk=int(k)\n  if str(kk)!=str(k)or kk in pure or type(v)is not int or not 0<=kk<d[\'addresses\']or not 0<v<(1<<40):raise ValueError(\'ambiguous/out-of-range pure-polynomial key/value\')\n  pure[kk]=v\n products=[]\n for p in d[\'products\']:\n  if set(p)!={\'left1\',\'right1\',\'left2\',\'right2\',\'fanout\'}or any(type(v)is not int or v<0 for v in p.values()):raise ValueError(\'bad saved Product\')\n  if p[\'left1\'].bit_length()>16 or p[\'left2\'].bit_length()>16 or p[\'right1\'].bit_length()>((d[\'addresses\']-1)>>4)+1 or p[\'right2\'].bit_length()>((d[\'addresses\']-1)>>4)+1 or not 0<p[\'fanout\']<(1<<40):raise ValueError(\'product outside declared wire coordinates\')\n  products.append(Product(**p))\n cache=d[\'cache_masks\']\n if any(type(m)is not int or not 0<m<d[\'addresses\']or m.bit_count()<2 for m in cache)or cache!=sorted(set(cache)):raise ValueError(\'bad cache order/masks\')\n if any((m^(1<<(m.bit_length()-1))).bit_count()>=2 and(m^(1<<(m.bit_length()-1)))not in cache for m in cache):raise ValueError(\'cache is not parent closed\')\n if any(type(r)is not int or r<0 for r in d[\'ranks\']):raise ValueError(\'bad saved ranks\')\n return Plan(*(d[k]for k in(\'addresses\',\'batch\',\'brot\',\'low_bits\')),products,pure,list(d[\'ranks\']),list(d[\'cache_masks\']),\n  blocks=None if d[\'blocks\']is None else[plan_from_dict(x)for x in d[\'blocks\']],parity_scratch=d[\'parity_scratch\'])\n', 'tasks/dfthcsos/full_xor/plans.json': '[{"batch":0,"delta_sha256":"b65ce7fa990d90c371764ad661b562b3d6ea7ecab752c4d15e75b8155d5af7fb","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":8,"right1":33469352312422583938084,"left2":168,"right2":45142521030353087912380,"fanout":1014420459785},{"left1":8,"right1":43632814624952149590352,"left2":130,"right2":31361248291698359516190,"fanout":949237788985},{"left1":8192,"right1":2040231323978978838412,"left2":8330,"right2":69243530327220087529020,"fanout":1009769112545},{"left1":8232,"right1":115548432672803218715112,"left2":8234,"right2":85051380390800496535948,"fanout":163697902641},{"left1":32,"right1":59580156522424982321716,"left2":8362,"right2":66203423438020902509178,"fanout":923181040849},{"left1":32,"right1":56475648155582453168206,"left2":168,"right2":69216738579177074529576,"fanout":1034046797560},{"left1":160,"right1":58159153873771891738268,"left2":8192,"right2":34597825913395187296330,"fanout":885669895456},{"left1":32,"right1":3971014521784705632666,"left2":160,"right2":25181580656227386732896,"fanout":4934276665},{"left1":136,"right1":67860588437108124276168,"left2":8328,"right2":35023604806614563095450,"fanout":1012726616603},{"left1":2,"right1":148094343252414894149090,"left2":8330,"right2":60154309767154497028588,"fanout":1085683326497},{"left1":8,"right1":37789023554177354904264,"left2":42,"right2":22519253571133269763080,"fanout":244257722075},{"left1":2,"right1":147430780009836934436090,"left2":130,"right2":35356402525403183777182,"fanout":141094041837},{"left1":8,"right1":68095703496846988670036,"left2":8320,"right2":53172751894537926773114,"fanout":98263842248},{"left1":32,"right1":39116817870667834694032,"left2":8234,"right2":46206428168623864167108,"fanout":132336305174},{"left1":2,"right1":35214862358954327804664,"left2":128,"right2":53039593794025337563070,"fanout":84226334},{"left1":2,"right1":111368552948779510624178,"left2":138,"right2":37958342312614582616710,"fanout":128246793477},{"left1":128,"right1":91104965619379135099258,"left2":136,"right2":106707339656601195786942,"fanout":12277702371},{"left1":2,"right1":93658902636318169059866,"left2":160,"right2":19396722710113000664078,"fanout":1077081403452},{"left1":2,"right1":16119512294107029077754,"left2":128,"right2":45912941396567106659326,"fanout":690015804},{"left1":32,"right1":12762587550540231465316,"left2":8352,"right2":58380730367086673664996,"fanout":219029950229},{"left1":8,"right1":53029304894601580317724,"left2":8232,"right2":27510074675914457873838,"fanout":828256211662},{"left1":8,"right1":34025431712812394321902,"left2":8224,"right2":43109669160750754511620,"fanout":48234042132},{"left1":2,"right1":24887299435329179725606,"left2":8224,"right2":38037051637398104946168,"fanout":992470710044},{"left1":8192,"right1":69075643903615794476794,"left2":8202,"right2":5296151730216479045792,"fanout":153896012798},{"left1":8,"right1":34482832527851966877306,"left2":8194,"right2":18352607635469172442376,"fanout":89573355458},{"left1":160,"right1":127293328200851257723340,"left2":162,"right2":99295941907492926482090,"fanout":1018607816251},{"left1":32,"right1":27155246465454492072656,"left2":8224,"right2":73254128275086595896974,"fanout":37746572235},{"left1":8,"right1":121244880441216427151540,"left2":8232,"right2":37543267900458056064594,"fanout":193962844448},{"left1":10,"right1":115264472351678147938938,"left2":32,"right2":8813786219004511004310,"fanout":167992635955},{"left1":8192,"right1":53453268062368305431140,"left2":8224,"right2":49581829313365310121972,"fanout":752226551},{"left1":8192,"right1":35258040701611554943564,"left2":8202,"right2":71355196486907371576446,"fanout":937357601338},{"left1":2,"right1":48650630532860740117680,"left2":136,"right2":4330812972722855713408,"fanout":205205802278},{"left1":8192,"right1":70363362030689663114558,"left2":8320,"right2":54957623698781889104680,"fanout":72966653121},{"left1":2,"right1":139670213412656497260360,"left2":10,"right2":93980632775366423013584,"fanout":572904745},{"left1":32,"right1":63953267378963478329230,"left2":8224,"right2":8275563551914564799794,"fanout":283569639},{"left1":32,"right1":33908474347178288989512,"left2":40,"right2":55538172716070055993800,"fanout":68872232444},{"left1":128,"right1":34123270927724526226336,"left2":130,"right2":28449645569312057921704,"fanout":4746977844},{"left1":8192,"right1":40711827645150172166782,"left2":8226,"right2":21024563454572132250378,"fanout":236980464858},{"left1":34,"right1":22450653122498291346910,"left2":42,"right2":9656146255428493097276,"fanout":16292000021},{"left1":128,"right1":39390133335603363381620,"left2":160,"right2":29838284824466109826740,"fanout":3357209311},{"left1":32,"right1":2475645123442216638298,"left2":8320,"right2":69155079018299501072664,"fanout":974675386916},{"left1":10,"right1":85827087708435007193430,"left2":32,"right2":59006443327176783691642,"fanout":894157593135},{"left1":168,"right1":4368509349247900116528,"left2":170,"right2":45054973071087521694876,"fanout":868104455988},{"left1":136,"right1":54889856570652682803750,"left2":138,"right2":10228464139902733866896,"fanout":970124029201},{"left1":32,"right1":26990071602356311649436,"left2":8192,"right2":74495856496720592288656,"fanout":72088736785},{"left1":34,"right1":31197677660643618869722,"left2":162,"right2":20650402599114486914714,"fanout":35167652141},{"left1":32,"right1":5058256517559273790494,"left2":8192,"right2":38994222341141853616466,"fanout":4498318812},{"left1":10,"right1":10127751360152361582660,"left2":42,"right2":53914285429009749785410,"fanout":59997866217},{"left1":8,"right1":41857782005951320041066,"left2":128,"right2":65401725588227648886604,"fanout":17193437142},{"left1":32,"right1":38528978827191264130056,"left2":128,"right2":24886216962434927940958,"fanout":68817382202},{"left1":10,"right1":88222245589740066840794,"left2":32,"right2":13839229416527705789492,"fanout":922593641409},{"left1":34,"right1":85372616056334695196980,"left2":42,"right2":140467187440042559119312,"fanout":16743338294},{"left1":2,"right1":101025563706433259611124,"left2":8200,"right2":45447279425213161735720,"fanout":1030855720653},{"left1":8192,"right1":42682495554278415201066,"left2":8200,"right2":18194366691066213981436,"fanout":34442302254},{"left1":10,"right1":132795751849966630952378,"left2":8202,"right2":72750253398373208669834,"fanout":39159527992},{"left1":8,"right1":59657113028945231814662,"left2":8200,"right2":18540111428596696980848,"fanout":3306026761},{"left1":2,"right1":109771275166923299038828,"left2":42,"right2":61434546716737386414392,"fanout":132909613070},{"left1":128,"right1":114898493285630540051692,"left2":138,"right2":86977418288717657413358,"fanout":270613297151},{"left1":128,"right1":24462846727071773457846,"left2":160,"right2":38659220062089322416484,"fanout":138344517415},{"left1":2,"right1":119392330721981240032664,"left2":8320,"right2":61163908785202115060508,"fanout":256841466844},{"left1":8320,"right1":15922957907451754736382,"left2":8322,"right2":73229472700614692291824,"fanout":1074260010949},{"left1":2,"right1":27939733055442730092144,"left2":8226,"right2":23168769492730337306322,"fanout":1008409446931},{"left1":2,"right1":59980332579837691403098,"left2":130,"right2":65756507932506128880306,"fanout":72140903737},{"left1":8,"right1":16065164627459374117130,"left2":8200,"right2":19624534692974808972990,"fanout":4416520934},{"left1":8,"right1":56272911739451286652474,"left2":8192,"right2":41029793294763543964874,"fanout":319611708},{"left1":128,"right1":8111842271140669010564,"left2":8192,"right2":67051876601238009599110,"fanout":34545217746},{"left1":32,"right1":119387995063127239137062,"left2":34,"right2":93382621880544915525538,"fanout":281663707},{"left1":2,"right1":32990303337779054017972,"left2":8200,"right2":17398841777481581428334,"fanout":1021198595829},{"left1":8,"right1":114763530775777849838562,"left2":128,"right2":52413767173145446196704,"fanout":34665731116},{"left1":32,"right1":65666197453358549131674,"left2":128,"right2":60992232150769458211902,"fanout":17533296578},{"left1":128,"right1":64609755054445783695958,"left2":8192,"right2":29507506481073734775616,"fanout":828743346180},{"left1":2,"right1":41065133217870930815500,"left2":40,"right2":45127669786403406934906,"fanout":228068024319},{"left1":8320,"right1":54324621982352342579716,"left2":8322,"right2":51158936740316981354104,"fanout":1034563292115},{"left1":128,"right1":63746362891384948612070,"left2":8192,"right2":49999443936873714254786,"fanout":12804172326},{"left1":34,"right1":22317137803239929317162,"left2":8192,"right2":8393372223543708005362,"fanout":1008395341883},{"left1":16,"right1":41747320510642794364418,"left2":512,"right2":58711152524921124914456,"fanout":251249663},{"left1":16,"right1":97384653277496356754972,"left2":3092,"right2":48774256020200021418268,"fanout":1026661990923},{"left1":20,"right1":138869125226775607636578,"left2":512,"right2":29901971710022852236892,"fanout":133157411104},{"left1":16,"right1":132506811099590410289942,"left2":3088,"right2":15547493307759480101824,"fanout":833257683209},{"left1":516,"right1":66454783507410922717834,"left2":1024,"right2":55180187717318484650884,"fanout":137185523992},{"left1":16,"right1":51515321058807249343500,"left2":1552,"right2":36599910251735172714402,"fanout":871847171322},{"left1":1552,"right1":62122846419600505314862,"left2":2048,"right2":11947856862365476262022,"fanout":885215183309},{"left1":512,"right1":69306753816494143039022,"left2":1024,"right2":58039524246887259301410,"fanout":43365304809},{"left1":512,"right1":60817348039259507338734,"left2":1536,"right2":35747000371950686545816,"fanout":3325232156},{"left1":1552,"right1":28668120201847629239904,"left2":3600,"right2":16471830601436796339324,"fanout":849915074602},{"left1":2576,"right1":98135792086507416468514,"left2":2580,"right2":96920692572312937514120,"fanout":202822192590},{"left1":2048,"right1":54305112273922114536934,"left2":3588,"right2":26056688446655691681818,"fanout":85298512918},{"left1":4,"right1":131926594597348215833650,"left2":3092,"right2":18072598145120263158434,"fanout":18017536821},{"left1":512,"right1":24438182902044530622330,"left2":2048,"right2":18129756906262409171226,"fanout":824684186358},{"left1":16,"right1":72056039556831883694070,"left2":1536,"right2":57039383762512545586906,"fanout":888133437992},{"left1":528,"right1":25792010246224161793186,"left2":1552,"right2":34256308879055376654866,"fanout":1082244059866},{"left1":4,"right1":92537622036706697365878,"left2":1536,"right2":37772163330101915439538,"fanout":1009129424652},{"left1":16,"right1":27017396790624143490122,"left2":528,"right2":60912084326428434353810,"fanout":825253357768},{"left1":16,"right1":118324155394039639882994,"left2":3588,"right2":62630381460074948815868,"fanout":240734582262},{"left1":512,"right1":28086628740537752881366,"left2":1536,"right2":61329162649347579311654,"fanout":68802192927},{"left1":512,"right1":150840491076473716912482,"left2":528,"right2":94310197510807325637448,"fanout":4844544028},{"left1":16,"right1":54852969236337911033232,"left2":512,"right2":39740653106152406256104,"fanout":11874395905},{"left1":20,"right1":39356078500131423957824,"left2":512,"right2":1177874415677496465980,"fanout":227820552715},{"left1":16,"right1":41004653383060397575682,"left2":2560,"right2":24868211397029613389928,"fanout":1044519582431},{"left1":4,"right1":6988115503453651161638,"left2":2068,"right2":61461319358535957136372,"fanout":1039126427604},{"left1":4,"right1":105425837655381723503718,"left2":1540,"right2":58757814213214872670356,"fanout":22400725550},{"left1":1536,"right1":35498558568386274732438,"left2":3584,"right2":32258151704015240008472,"fanout":33719503552},{"left1":1024,"right1":69525019798037416964412,"left2":2052,"right2":27995244834520547394468,"fanout":828241687339},{"left1":2048,"right1":12388333689902320057638,"left2":3072,"right2":62965567613872805723484,"fanout":17782608418},{"left1":1536,"right1":20930369735600350100694,"left2":2048,"right2":28963987458959788150918,"fanout":76970784985},{"left1":4,"right1":49107948633275027268684,"left2":2560,"right2":73497459605135808085386,"fanout":923749640677},{"left1":1024,"right1":37417174787025131377872,"left2":1028,"right2":59645043630288378311796,"fanout":137622382788},{"left1":1024,"right1":62023882665054526214988,"left2":3072,"right2":1532396313769346749368,"fanout":264815100},{"left1":1024,"right1":51940559617462581682574,"left2":1536,"right2":41815311382836723422800,"fanout":4698528968},{"left1":4,"right1":23207157700362688845854,"left2":16,"right2":1923653982626498025830,"fanout":3319807798},{"left1":16,"right1":121835952987423602828094,"left2":20,"right2":41140633468654776451102,"fanout":4297188043},{"left1":16,"right1":60127941851543005856370,"left2":20,"right2":137487988102403304809448,"fanout":320026150},{"left1":20,"right1":49868827498828760596310,"left2":1044,"right2":48669435347523601862734,"fanout":824847897846},{"left1":4,"right1":45898724743064688695936,"left2":2564,"right2":74364405483577321987146,"fanout":69271502314},{"left1":4,"right1":46220622036034309767662,"left2":1028,"right2":53647385690116337359054,"fanout":133169154},{"left1":2560,"right1":24715992531186450091862,"left2":3584,"right2":63107804623158610815924,"fanout":52415244497},{"left1":16,"right1":124234306205894686304868,"left2":2048,"right2":30122113356987693501786,"fanout":13308380},{"left1":2564,"right1":37718894580672157149514,"left2":3588,"right2":74295900127119383410110,"fanout":1060760399067},{"left1":4,"right1":119357420247503237100992,"left2":2052,"right2":60193113462097213351326,"fanout":137476556815},{"left1":512,"right1":59325958135883869248662,"left2":528,"right2":38110989291938088467584,"fanout":34994854382},{"left1":16,"right1":150310934255283055300936,"left2":20,"right2":142410125706293160806024,"fanout":824648337917},{"left1":512,"right1":43074028445023418207532,"left2":1536,"right2":52548763561140609004730,"fanout":986656240},{"left1":512,"right1":62461659650469516717000,"left2":1040,"right2":65656876297986655558516,"fanout":876874764736},{"left1":2052,"right1":99086872494787039636162,"left2":2068,"right2":79658114457475566208996,"fanout":838522313152},{"left1":512,"right1":32044318045563951655682,"left2":1536,"right2":52485389919567184759628,"fanout":17295800373},{"left1":2560,"right1":85454841882684141312164,"left2":2564,"right2":139890398076165353280302,"fanout":9324775399},{"left1":1040,"right1":34385223485507528401084,"left2":3088,"right2":9125579730298300119112,"fanout":124852095726},{"left1":2048,"right1":71951075493826142338982,"left2":2560,"right2":31225379587450788528892,"fanout":72509022956},{"left1":2048,"right1":40023326432577680373870,"left2":2564,"right2":13056004207752446759360,"fanout":872735824689},{"left1":4,"right1":46948158744479422626250,"left2":2052,"right2":37314367014125825468296,"fanout":4613720565},{"left1":4,"right1":116814007539617157209762,"left2":1024,"right2":28447728903027766229784,"fanout":672059178},{"left1":1040,"right1":62058694453493134856394,"left2":3088,"right2":33413260394701464205540,"fanout":262413155885},{"left1":1024,"right1":9741149058144069606074,"left2":3076,"right2":44083914292489273502164,"fanout":918905157410},{"left1":1040,"right1":14677424773954540365100,"left2":3088,"right2":3367399108174386084698,"fanout":910651214127},{"left1":4,"right1":122796956545504383043682,"left2":2048,"right2":26894621488149706025842,"fanout":8872410858},{"left1":20,"right1":100094482302905770486684,"left2":2048,"right2":29411428523848810043588,"fanout":231391050708},{"left1":20,"right1":34869497081428657976454,"left2":2068,"right2":9770130849806142857604,"fanout":33654377926},{"left1":16,"right1":124835720710254740326696,"left2":20,"right2":1588263086192207449156,"fanout":537533891},{"left1":4,"right1":105521903012112767895738,"left2":2564,"right2":11677168195674061057436,"fanout":175021677767},{"left1":20,"right1":147119560570969542702326,"left2":2068,"right2":30549772175639207929306,"fanout":21656762304},{"left1":1024,"right1":22551560727569849703958,"left2":2048,"right2":33458029267607796698422,"fanout":4312740073},{"left1":1024,"right1":74267455251229306481246,"left2":3072,"right2":34886054000076619430456,"fanout":138062335542},{"left1":1028,"right1":135195544990231644301098,"left2":1044,"right2":133919425477005587065836,"fanout":958813846070},{"left1":1028,"right1":32814129389935480638378,"left2":1040,"right2":48019704400188544210124,"fanout":975795913982},{"left1":2048,"right1":140487835787252712925186,"left2":2068,"right2":117606865760288711368750,"fanout":166460900408},{"left1":4,"right1":62070111322375483935920,"left2":516,"right2":4794618777441732954814,"fanout":8828685837},{"left1":528,"right1":67754970999989811019838,"left2":1552,"right2":63802001326684373491816,"fanout":868458766572},{"left1":16,"right1":43873232722223329534404,"left2":2052,"right2":65654831468614480268850,"fanout":1008717144299},{"left1":4,"right1":46398142026726955712572,"left2":516,"right2":6709603227774632162948,"fanout":3741257426},{"left1":16,"right1":140258827990274480560546,"left2":1040,"right2":57913850415928143318218,"fanout":103220448},{"left1":32768,"right1":64268918771074136643108,"left2":33088,"right2":73244835129199218857008,"fanout":845390091269},{"left1":320,"right1":45968778269117304351060,"left2":4096,"right2":27743430420940088284996,"fanout":224111766063},{"left1":256,"right1":67089243486154000562044,"left2":36864,"right2":58618206823515041473010,"fanout":226977519572},{"left1":16384,"right1":3712382966934224090296,"left2":16640,"right2":20953671163764523852792,"fanout":149737749},{"left1":37120,"right1":17459139614926105552912,"left2":37184,"right2":10798716516305523249208,"fanout":42963379221},{"left1":49152,"right1":54281916359309189771570,"left2":53248,"right2":64771292929520219653364,"fanout":1018443143143},{"left1":16448,"right1":983115070132852892616,"left2":20544,"right2":13229344321035909973940,"fanout":9210749422},{"left1":20480,"right1":54938444134860122491608,"left2":20736,"right2":65224381201571830312582,"fanout":218392050960},{"left1":16384,"right1":17254673165466348148780,"left2":49408,"right2":35194152538487018056776,"fanout":1073456798260},{"left1":33088,"right1":41556092394872094816544,"left2":37184,"right2":45537403031541496543234,"fanout":253469848543},{"left1":256,"right1":66117936301088273444298,"left2":33024,"right2":5238852713707794838312,"fanout":824866685221},{"left1":4096,"right1":48324702197025544337642,"left2":32768,"right2":25702440479602998176416,"fanout":288755768},{"left1":4096,"right1":36497200319748393616934,"left2":49152,"right2":42655970357955695616914,"fanout":209567030586},{"left1":4096,"right1":5437846796700953807548,"left2":36864,"right2":18217384787721376429970,"fanout":141769622055},{"left1":64,"right1":68365371861300549030510,"left2":20800,"right2":27137934914541702338992,"fanout":923286230265},{"left1":64,"right1":59083515623304406619986,"left2":53568,"right2":9797944651797246716730,"fanout":72226179541},{"left1":64,"right1":23208573322225441719986,"left2":36928,"right2":43757946178218806316824,"fanout":1059850425566},{"left1":32768,"right1":64490817492490581869312,"left2":32832,"right2":17586193746034360871522,"fanout":68734402566},{"left1":64,"right1":49501385188940631294050,"left2":36864,"right2":62067594552146575509630,"fanout":64375081169},{"left1":4096,"right1":28744737268046310957632,"left2":16640,"right2":63862344798187639824900,"fanout":215452955192},{"left1":32768,"right1":68967288584955663122932,"left2":36928,"right2":21884166201884663772296,"fanout":151060470571},{"left1":64,"right1":7110012099771417950678,"left2":320,"right2":32758388368637536632042,"fanout":304335634},{"left1":256,"right1":23989607271536001893996,"left2":16384,"right2":35294655920170293419934,"fanout":73298738189},{"left1":16384,"right1":27486975631369387227878,"left2":16640,"right2":74076264173925801224152,"fanout":39494739262},{"left1":16384,"right1":13181603195586543293626,"left2":49408,"right2":73233243858067873394582,"fanout":1056095023106},{"left1":36864,"right1":69400574401013026914768,"left2":36928,"right2":9925368919388265732842,"fanout":1074409031960},{"left1":32832,"right1":18212127405771328625770,"left2":33088,"right2":60335173606884215020886,"fanout":136764393014},{"left1":256,"right1":36814826234491810406450,"left2":32768,"right2":36817142153940901784408,"fanout":8772524824},{"left1":4096,"right1":44800906694188723496848,"left2":16384,"right2":45744222694458618359134,"fanout":34362884122},{"left1":4096,"right1":27682688916912018641118,"left2":16448,"right2":21647688212246246508698,"fanout":1047186441532},{"left1":320,"right1":24684316922241090340714,"left2":33024,"right2":66191048487567212556398,"fanout":232803460902},{"left1":4096,"right1":21148867678268654747308,"left2":32768,"right2":2826567821133254571806,"fanout":556588790},{"left1":32768,"right1":40077954287593875087922,"left2":32832,"right2":14373568079015061215408,"fanout":137535881747},{"left1":4160,"right1":69384232027003416410312,"left2":32768,"right2":8130825274870358848100,"fanout":210051341818},{"left1":4096,"right1":42277269362096900758628,"left2":16384,"right2":21470549490361860832040,"fanout":69017798953},{"left1":16384,"right1":25907431970422470854032,"left2":36928,"right2":35596256673751408442328,"fanout":957091467497},{"left1":16384,"right1":66755763178907052844832,"left2":49152,"right2":61608516468207544078266,"fanout":201647603},{"left1":256,"right1":33444962505207121689456,"left2":37120,"right2":70977565233786969745532,"fanout":1098961592272},{"left1":64,"right1":29863381836173182403766,"left2":49152,"right2":17116834987581841378018,"fanout":155493082327},{"left1":49408,"right1":56013710300556266836076,"left2":49472,"right2":48619538487397321027830,"fanout":47978715086},{"left1":16384,"right1":5540206035116457755008,"left2":20480,"right2":35329178059548410254796,"fanout":828961336627},{"left1":64,"right1":71614501205336258257932,"left2":53568,"right2":29391302812033749263878,"fanout":74022848560},{"left1":36864,"right1":52195710618602286758690,"left2":36928,"right2":48964360554449494928770,"fanout":68665150981},{"left1":64,"right1":28499594855591906551326,"left2":49152,"right2":31671763734100921154602,"fanout":841124227643},{"left1":16384,"right1":40391864643557273539052,"left2":20544,"right2":53348462888234334176268,"fanout":119554624563},{"left1":49216,"right1":43503853753803482063018,"left2":53312,"right2":28299353871224004019758,"fanout":43657540097},{"left1":64,"right1":28375091858060638054986,"left2":16640,"right2":46982332601799438288456,"fanout":145746569959},{"left1":4352,"right1":24051173753754273540150,"left2":20736,"right2":11875328277819464329036,"fanout":974855672774},{"left1":256,"right1":52843281734246093498844,"left2":4416,"right2":47710603684504313517718,"fanout":167958073100},{"left1":32768,"right1":25920934916386686772608,"left2":49152,"right2":26286692471269314864050,"fanout":34460858114},{"left1":256,"right1":16626932692700944172208,"left2":4160,"right2":9621297957102116500970,"fanout":185725797101},{"left1":64,"right1":34048357407786374891910,"left2":32832,"right2":14274878739564927086762,"fanout":824704881204},{"left1":16384,"right1":38249043151408548264330,"left2":49152,"right2":22898783948940819386872,"fanout":3594190042},{"left1":32768,"right1":54918519382291745875306,"left2":49408,"right2":70653507422421555195086,"fanout":1078285900250},{"left1":16384,"right1":25571411065796236231610,"left2":20480,"right2":1687003618587027791802,"fanout":4062520262},{"left1":320,"right1":20977187958737546374336,"left2":4416,"right2":7211247546716254818092,"fanout":879683185443},{"left1":64,"right1":36021133871641978828914,"left2":320,"right2":14591009465662972242822,"fanout":13307179},{"left1":16384,"right1":4218824716838993083646,"left2":16640,"right2":58377299055212850581386,"fanout":8657098232},{"left1":64,"right1":15619457865215271571398,"left2":16448,"right2":60269648520027431759368,"fanout":824716825613},{"left1":64,"right1":50117211942353912355298,"left2":33088,"right2":58958220714577891476224,"fanout":224259486703},{"left1":16384,"right1":69746445410595587958638,"left2":16640,"right2":69520913706937723589064,"fanout":17832932334},{"left1":256,"right1":27092875827834165197750,"left2":4160,"right2":57945881645716687869618,"fanout":914223806737},{"left1":32768,"right1":26337161040814040308238,"left2":33024,"right2":36992797907079640529586,"fanout":34379676664},{"left1":16640,"right1":19984447373330610921968,"left2":16704,"right2":47437575369723638794922,"fanout":858812923356},{"left1":64,"right1":25552044023318621894776,"left2":49216,"right2":38347326336255188584628,"fanout":1098626514681},{"left1":320,"right1":48050551993736701088292,"left2":4416,"right2":8903389133141992276436,"fanout":1025862876662},{"left1":16384,"right1":5082120960505336931662,"left2":20480,"right2":57831648345927638447528,"fanout":138152486},{"left1":64,"right1":42697578512669149705642,"left2":16704,"right2":30045329194680332201696,"fanout":988711295270},{"left1":256,"right1":70091804416522289434742,"left2":4416,"right2":42406982591311134102210,"fanout":69337233861},{"left1":16384,"right1":28228080414051927556978,"left2":49152,"right2":37863345023575286029344,"fanout":26157853192},{"left1":49152,"right1":25948636525809649146516,"left2":49216,"right2":2124479344994686609384,"fanout":1046931119656},{"left1":64,"right1":37190919124676249256768,"left2":32768,"right2":26940222958165862433736,"fanout":34373378858},{"left1":4160,"right1":66176998410805640003858,"left2":20544,"right2":36927262660421110266868,"fanout":168356545256},{"left1":4096,"right1":64792090738112038276968,"left2":4416,"right2":445095226815043419794,"fanout":859828910124},{"left1":16640,"right1":65225770819015363425158,"left2":16704,"right2":34088921117502230056828,"fanout":1010237686789}],"pure":{"1":986899953918,"2":915702472155,"5":918640604177,"7":884613510438,"9":64304242726,"16":42242083618,"32":7603174381,"80":17964340500,"96":123765265381,"144":99384886557,"160":270880872508,"192":975632667403,"224":1086930151620,"240":68887457999,"256":13759888377,"272":25067719446,"288":1026024661241,"304":979319980862,"320":875632575994,"400":146261001187,"432":1048694177252,"464":269628131034,"496":206392193541,"528":63652938705,"544":5132264165,"592":77860962030,"624":137221978094,"640":218137426688,"688":867647897401,"736":110945698771,"752":1013224686083,"848":1012810579941,"864":34441314774,"880":1074059350810,"944":876070035002,"976":992879561675,"1024":1078101739716,"1072":137586274815,"1088":194214235118,"1104":1027099270173,"1136":918153332464,"1152":267258687232,"1168":907209593065,"1184":150638820565,"1200":188964590876,"1216":132592692249,"0":86400253450,"3":1034297465102,"10":60382522413,"13":30732069362,"48":889060322546,"64":240954639135,"128":104002296296,"176":1056681481193,"208":996300693448,"352":211208764431,"512":979318654159,"576":884460763647,"608":171598491846,"656":979031812853,"800":106706897881,"816":1083106911020,"896":841743464488,"960":20833712675,"1008":167960642799,"1040":141131765521,"1056":850200742650,"1120":898220242150,"4":828646222617,"336":893569402105,"368":4013563915,"416":112390686162,"480":98046036213,"672":227679718657,"704":928470328816,"720":1004556322851,"832":17043947732,"912":9567025642,"928":922525051130,"992":197570277600,"12":47045667589,"15":915614416395,"384":1051348172830,"560":1059815685592,"768":949642118386,"11":914610332142,"14":112489921732,"8":957512167412,"112":957657709040,"784":236588251138,"6":983428444893,"448":992459883546},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"phased_canonical_6","capacity":6,"events":[{"kind":"products","indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74],"required":267,"cache":267},{"kind":"pure_low","monomial":3,"required":1,"cache":267},{"kind":"pure_low","monomial":5,"required":2,"cache":267},{"kind":"pure_low","monomial":7,"required":9,"cache":267},{"kind":"pure_low","monomial":13,"required":258,"cache":267},{"kind":"products","indices":[75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149],"required":113,"cache":121},{"kind":"pure_low","monomial":9,"required":16,"cache":121},{"kind":"pure_low","monomial":10,"required":32,"cache":121},{"kind":"pure_low","monomial":11,"required":65,"cache":121},{"kind":"products","indices":[150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224],"required":1677,"cache":1677},{"kind":"pure_low","monomial":6,"required":4,"cache":1677},{"kind":"pure_low","monomial":12,"required":128,"cache":1677},{"kind":"pure_low","monomial":14,"required":516,"cache":1677},{"kind":"pure_low","monomial":15,"required":1033,"cache":1677}],"low_computations":11,"peak_live":6,"min_live_required":6}},{"batch":1,"delta_sha256":"5c90c09206689510be83d3f1325903a154dbec2f41fde54ebac1d2bf264edf3b","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":8,"right1":1833712956941225840852,"left2":40,"right2":43493909421496580369754,"fanout":34541265379},{"left1":2,"right1":42854651502674501498564,"left2":32,"right2":67123980399916785048892,"fanout":9190163755},{"left1":136,"right1":135378348881817957747876,"left2":8192,"right2":74053958825979395861010,"fanout":257662588168},{"left1":2,"right1":47993297067672607848212,"left2":138,"right2":141945498739533884785600,"fanout":986835136258},{"left1":128,"right1":120100177025723958329634,"left2":8194,"right2":60708599889185824044222,"fanout":828462601516},{"left1":8320,"right1":30584395679039169169248,"left2":8322,"right2":22489255771902846717586,"fanout":150393176372},{"left1":8202,"right1":92433463763995961355936,"left2":8330,"right2":145118300598239464420604,"fanout":845635325758},{"left1":8224,"right1":124232941124220918146082,"left2":8226,"right2":142664950735640300526494,"fanout":834080393761},{"left1":32,"right1":33561245188553413422784,"left2":8192,"right2":26525615727129001897826,"fanout":824768071433},{"left1":34,"right1":17036109734410304087788,"left2":8192,"right2":67279369908654667068056,"fanout":1009549045271},{"left1":40,"right1":23051520245532860948272,"left2":8232,"right2":31722200549970181677472,"fanout":94471599843},{"left1":8224,"right1":99379461572630198457724,"left2":8352,"right2":130781011505305440049190,"fanout":872046455613},{"left1":32,"right1":74489755261787149056688,"left2":8352,"right2":33240322223928831160396,"fanout":17483876829},{"left1":8,"right1":5689128002067688545086,"left2":8328,"right2":4040170119910031170788,"fanout":192567911940},{"left1":8192,"right1":44798006531591040327490,"left2":8330,"right2":45491596911517595397734,"fanout":941317242666},{"left1":10,"right1":1200319728791721503290,"left2":42,"right2":30880226558279192662386,"fanout":206396919528},{"left1":8194,"right1":43082319207541230854286,"left2":8226,"right2":5033382609368981138820,"fanout":970709590724},{"left1":8320,"right1":20810701769529656366440,"left2":8352,"right2":10065149375267077138406,"fanout":90780995582},{"left1":32,"right1":49292924659985573777492,"left2":8320,"right2":16756763827164632138188,"fanout":26678213894},{"left1":8,"right1":55503032934875187182438,"left2":8192,"right2":16722954684772038686468,"fanout":68904368113},{"left1":8,"right1":133084959117534494672964,"left2":8202,"right2":33529777683284720099378,"fanout":1031396180249},{"left1":8192,"right1":69019389430726400826394,"left2":8320,"right2":19006198425762990610762,"fanout":568598537},{"left1":34,"right1":19624206097403176215934,"left2":42,"right2":41236514798574068900560,"fanout":77916814314},{"left1":10,"right1":149796633234163739741984,"left2":128,"right2":136765913231666901659264,"fanout":1009967835371},{"left1":136,"right1":35448343468341225464858,"left2":138,"right2":102735297993294424535240,"fanout":218474278186},{"left1":8192,"right1":34287827312020918368540,"left2":8352,"right2":67802112122608847803598,"fanout":218067879152},{"left1":8,"right1":15987213979402300321570,"left2":8192,"right2":44835465718113210518552,"fanout":17464949303},{"left1":8,"right1":39141872737003149133258,"left2":8320,"right2":57709134117953953753670,"fanout":47259108852},{"left1":2,"right1":117386739401565285092344,"left2":10,"right2":62284210238145139906670,"fanout":517082424},{"left1":2,"right1":111903431144585632931918,"left2":34,"right2":1941988513742773672896,"fanout":17230521033},{"left1":8,"right1":116086717357567850946042,"left2":32,"right2":11878583010066008455958,"fanout":723139352},{"left1":32,"right1":118215933154228143291950,"left2":162,"right2":130391931360487852402534,"fanout":144987651127},{"left1":32,"right1":35364003281813195959854,"left2":160,"right2":8307159423561263666562,"fanout":8693073421},{"left1":8192,"right1":52202080207886471129096,"left2":8224,"right2":11427622561792742833060,"fanout":138192954404},{"left1":8,"right1":25043335617471462942694,"left2":8194,"right2":74310509054057277878982,"fanout":846558659786},{"left1":2,"right1":7419522677499555743228,"left2":8194,"right2":39163660577151566774226,"fanout":138123356423},{"left1":40,"right1":90869361326946961201454,"left2":42,"right2":113081178081739357727560,"fanout":875734877478},{"left1":8194,"right1":32258186838410945585704,"left2":8202,"right2":7220060618560566939048,"fanout":995912044759},{"left1":128,"right1":130169255253861201556442,"left2":8354,"right2":15977303592566831656042,"fanout":1010389212432},{"left1":8224,"right1":118066441306418926822218,"left2":8352,"right2":80506489361940651751982,"fanout":153795759580},{"left1":34,"right1":2848734342979781980718,"left2":128,"right2":29046108087694323823214,"fanout":1017539920644},{"left1":8192,"right1":66828186976304468945080,"left2":8320,"right2":55645892882125830566006,"fanout":3271951345},{"left1":128,"right1":54267159531981350271806,"left2":130,"right2":16071081264070696757398,"fanout":824703262429},{"left1":32,"right1":2163007877054069010966,"left2":8192,"right2":7189636225453565615588,"fanout":17951243476},{"left1":128,"right1":82611598634297329870708,"left2":168,"right2":4857108134295780983940,"fanout":30899566830},{"left1":8200,"right1":149625042609253997892802,"left2":8328,"right2":83220880105158952349748,"fanout":153863012120},{"left1":40,"right1":8985457123842879612632,"left2":128,"right2":130200181536613243933284,"fanout":924174855219},{"left1":8,"right1":791902887049693912002,"left2":32,"right2":7073987489344036008180,"fanout":17215262216},{"left1":2,"right1":134604310785420566752468,"left2":162,"right2":37312758014538152371942,"fanout":979618688528},{"left1":40,"right1":30497756153938589930618,"left2":128,"right2":15460287762851733238532,"fanout":206508905241},{"left1":2,"right1":68432954830385327692428,"left2":8,"right2":17763171548300090594780,"fanout":3486521034},{"left1":8192,"right1":49371787097677387177388,"left2":8200,"right2":46699358629779928304142,"fanout":137489532943},{"left1":32,"right1":85810576871425201946696,"left2":168,"right2":132827065186479531550284,"fanout":851458584524},{"left1":128,"right1":118788092688253363644542,"left2":138,"right2":90982785768857203741776,"fanout":68488346907},{"left1":32,"right1":112759605352310135707726,"left2":34,"right2":105853571141298108916450,"fanout":824635629097},{"left1":10,"right1":31181403623805255065284,"left2":138,"right2":39478691078707642935736,"fanout":926823621671},{"left1":8194,"right1":23247566717046794443950,"left2":8226,"right2":47943731325215082919268,"fanout":93583646974},{"left1":128,"right1":19724971881435770756574,"left2":8192,"right2":31106348398638030916068,"fanout":17260097765},{"left1":2,"right1":53298111853077310114266,"left2":8192,"right2":61573607959106703938378,"fanout":35418206428},{"left1":130,"right1":145275110979495782948968,"left2":138,"right2":69673539640189825322084,"fanout":98904912613},{"left1":8192,"right1":55501559497043692505704,"left2":8224,"right2":69690864694581437514432,"fanout":3453157377},{"left1":160,"right1":55949543933855654762038,"left2":168,"right2":60918658133294907057944,"fanout":177001268479},{"left1":128,"right1":31252329011853823731138,"left2":8194,"right2":69397528513073404820132,"fanout":206577791484},{"left1":8,"right1":58569431820180095715590,"left2":8192,"right2":71783848199394136693630,"fanout":13069572909},{"left1":128,"right1":74621066098827634749272,"left2":130,"right2":18336849986619755814094,"fanout":34373639452},{"left1":2,"right1":108955941029720866732470,"left2":128,"right2":145142805438342237506230,"fanout":8652910316},{"left1":40,"right1":13950963386267331613464,"left2":42,"right2":15079072515014193202908,"fanout":150085490159},{"left1":8360,"right1":50682560292378440947494,"left2":8362,"right2":66637861877934817836426,"fanout":939557242109},{"left1":2,"right1":53773874583350461914884,"left2":128,"right2":145534689756444975650524,"fanout":4562549279},{"left1":10,"right1":34899434773838029601018,"left2":32,"right2":57235945651159130507150,"fanout":862770027300},{"left1":2,"right1":121423696068660352959314,"left2":128,"right2":1040976109642320190602,"fanout":17418603236},{"left1":8226,"right1":27079935216483009229142,"left2":8234,"right2":60539430192923282362138,"fanout":1001267513028},{"left1":8,"right1":58398050252721531863388,"left2":32,"right2":47964846541321386422548,"fanout":3370711817},{"left1":160,"right1":128076786233744586388860,"left2":168,"right2":137130153794295671971300,"fanout":175254850355},{"left1":8192,"right1":1198823207472264597006,"left2":8200,"right2":42683949158477025672412,"fanout":103619332},{"left1":512,"right1":670549933456520484156,"left2":2064,"right2":35820778532878567944282,"fanout":16995267029},{"left1":1024,"right1":60027228725147169737202,"left2":3600,"right2":40620836089515047230586,"fanout":919370657561},{"left1":512,"right1":1662719741049178770076,"left2":3072,"right2":52359297369067345680032,"fanout":876995681582},{"left1":4,"right1":146322840761733862412672,"left2":3076,"right2":67436869766695806915812,"fanout":871253944024},{"left1":4,"right1":114945475638565693545566,"left2":516,"right2":20648662912197955202184,"fanout":34374426905},{"left1":2048,"right1":43855278011943975042188,"left2":2576,"right2":56494488286088836243242,"fanout":1073840705287},{"left1":1024,"right1":47634596755946929057598,"left2":1044,"right2":69460037955864221692628,"fanout":30128677846},{"left1":16,"right1":27788389241456372272578,"left2":528,"right2":63942289407261880027684,"fanout":17216191534},{"left1":512,"right1":49078205808008598082514,"left2":528,"right2":48388643996563707756386,"fanout":8689034523},{"left1":4,"right1":133016487392041328649198,"left2":1536,"right2":2417483216672368856628,"fanout":975222743299},{"left1":16,"right1":28287122041829846924534,"left2":1024,"right2":52740397623590600555112,"fanout":4666764080},{"left1":1024,"right1":28241010457109296125282,"left2":3072,"right2":49120773347587511224250,"fanout":69085878065},{"left1":512,"right1":46385826684438742552548,"left2":3092,"right2":40831278125824370443084,"fanout":254044406491},{"left1":512,"right1":45139600157991506013538,"left2":1536,"right2":37562810025634907466310,"fanout":188408865},{"left1":4,"right1":85952748406210724893318,"left2":2564,"right2":10882174623005571040178,"fanout":1001654064419},{"left1":1536,"right1":77799518732732663829464,"left2":3584,"right2":112429721820522148003432,"fanout":928347778077},{"left1":20,"right1":68896586001044036812642,"left2":2048,"right2":109139282539260464462596,"fanout":883819994407},{"left1":512,"right1":65434326009336789310308,"left2":2048,"right2":41677169358530946218178,"fanout":137525537293},{"left1":1536,"right1":110960589897968790555934,"left2":1540,"right2":146367840252927452782120,"fanout":179834131197},{"left1":512,"right1":57930279967969449854014,"left2":528,"right2":56975557048488723494526,"fanout":3776189140},{"left1":4,"right1":133096099190577789211044,"left2":3604,"right2":49943184268132087136836,"fanout":254305058363},{"left1":1024,"right1":66168729743648605588128,"left2":1040,"right2":15137160210114408427944,"fanout":69020288495},{"left1":1024,"right1":66198515923087341104970,"left2":2048,"right2":46062106595796486620588,"fanout":52902171},{"left1":1024,"right1":53299945367668580864970,"left2":2048,"right2":31529400607076750351198,"fanout":824636458282},{"left1":516,"right1":43606501941795109972406,"left2":2564,"right2":32499800045638108889434,"fanout":175123454455},{"left1":3072,"right1":23676907741990936002640,"left2":3076,"right2":71310533387164350924556,"fanout":991769927704},{"left1":516,"right1":698431592119689714098,"left2":532,"right2":52171458734154780064892,"fanout":1091557011249},{"left1":16,"right1":6081184392299907756238,"left2":2576,"right2":70927925274115295651748,"fanout":98969259990},{"left1":16,"right1":57417781971239693824478,"left2":532,"right2":28027012545994334329496,"fanout":1004354996688},{"left1":1536,"right1":92631331167446397296018,"left2":1540,"right2":77261750345568594612954,"fanout":262852048579},{"left1":4,"right1":130839607314924497775150,"left2":1024,"right2":47749941212420194021446,"fanout":9213775856},{"left1":1024,"right1":6506560216736868813158,"left2":1536,"right2":66269571787319019922224,"fanout":8862246112},{"left1":2048,"right1":77068138863533388822054,"left2":2576,"right2":38827643535221248234538,"fanout":236981526289},{"left1":1024,"right1":14557053530388802978602,"left2":2052,"right2":139825826598479782725852,"fanout":1017642957769},{"left1":1024,"right1":37323233246246510036340,"left2":2048,"right2":46340534607279364928482,"fanout":141013730498},{"left1":16,"right1":17434263142683526342884,"left2":20,"right2":16194282521275441914894,"fanout":25871648570},{"left1":4,"right1":89963977080127346201594,"left2":528,"right2":13001276313471314426476,"fanout":55200110785},{"left1":2052,"right1":78796502926973599681784,"left2":2068,"right2":67358123781795763707032,"fanout":980275760157},{"left1":4,"right1":68658110929126324913636,"left2":516,"right2":24172422001167752136268,"fanout":824771468811},{"left1":4,"right1":2717513054112224843340,"left2":1024,"right2":7046630966966489525554,"fanout":34380591117},{"left1":4,"right1":149802467684808774235084,"left2":16,"right2":5713977126913829022632,"fanout":68837565985},{"left1":512,"right1":70479421355302500087188,"left2":2052,"right2":80129925108254258657394,"fanout":1025642649851},{"left1":16,"right1":7596796233264841930932,"left2":2064,"right2":10559617295725094499852,"fanout":17261801174},{"left1":4,"right1":35810497424520148915322,"left2":516,"right2":35625058636188423623274,"fanout":137440778985},{"left1":512,"right1":64072166310523763637198,"left2":2052,"right2":34803509653509838645314,"fanout":103447914693},{"left1":512,"right1":34368914327426146103196,"left2":528,"right2":21657891953676612190668,"fanout":138329769754},{"left1":4,"right1":40851867990406636733332,"left2":1024,"right2":3549697438200032331992,"fanout":68773821164},{"left1":1040,"right1":52827108931463121037846,"left2":2048,"right2":149155422412306867956024,"fanout":854632417008},{"left1":512,"right1":43260563704144144346748,"left2":2564,"right2":269761655283782806158,"fanout":927816085259},{"left1":2052,"right1":71801623973710073998032,"left2":2068,"right2":68477258715667104204638,"fanout":909673545773},{"left1":1028,"right1":64595707649280907668694,"left2":3076,"right2":19487148720840295008814,"fanout":845820717064},{"left1":1024,"right1":111264359501875796187152,"left2":3076,"right2":127938781242808324151022,"fanout":206910533918},{"left1":16,"right1":96405964618703916979914,"left2":2064,"right2":142136090621219860555462,"fanout":136645421},{"left1":20,"right1":6128980760165077592298,"left2":2068,"right2":57444599059283901431316,"fanout":932929597235},{"left1":4,"right1":76707709442108612850010,"left2":20,"right2":54785130686071101919416,"fanout":43001720313},{"left1":512,"right1":20857652407249434986878,"left2":516,"right2":73939800339029108648706,"fanout":68756049605},{"left1":2048,"right1":79787913409609675626370,"left2":3584,"right2":59338321775169535308310,"fanout":236861515257},{"left1":16,"right1":75159816014696289877014,"left2":512,"right2":768634552594889235596,"fanout":152440523},{"left1":2048,"right1":96345553416433709024632,"left2":3588,"right2":27297401118813237821706,"fanout":21606829608},{"left1":2048,"right1":147206261983928658502968,"left2":3072,"right2":58490243471771745092556,"fanout":7815810753},{"left1":1024,"right1":9345798119260787099914,"left2":1040,"right2":46197315228062470773468,"fanout":8790737968},{"left1":2048,"right1":17497491228401111329666,"left2":2576,"right2":71461826137270769857284,"fanout":159537940690},{"left1":1024,"right1":17829497002377258897044,"left2":1536,"right2":71157352624669542906690,"fanout":18000631103},{"left1":2048,"right1":143340709544189420957922,"left2":3584,"right2":46255922965134877381034,"fanout":850625822502},{"left1":2064,"right1":5433137040447602978310,"left2":2068,"right2":61243403842451795270852,"fanout":855486180871},{"left1":2048,"right1":147836384222327009099880,"left2":2068,"right2":7420981034246961267632,"fanout":1039282010339},{"left1":1024,"right1":34239284860794438646172,"left2":2052,"right2":109644077923648823689672,"fanout":232216993554},{"left1":1024,"right1":28199324230263489552688,"left2":3072,"right2":10795848149905384977448,"fanout":17261067271},{"left1":16,"right1":43869872669510316533908,"left2":1536,"right2":30744324498543110618268,"fanout":1044283730391},{"left1":1040,"right1":52067785766839892244108,"left2":1044,"right2":6433158515013497150450,"fanout":1017619213329},{"left1":16,"right1":60419653361047597038256,"left2":528,"right2":22499843115526263879292,"fanout":5187189216},{"left1":3072,"right1":1086357387125018570106,"left2":3088,"right2":35387463176564972358830,"fanout":1029751433955},{"left1":16,"right1":27484070826649355593344,"left2":3088,"right2":62991007328688513827346,"fanout":188641891132},{"left1":2052,"right1":3056333740039073914488,"left2":3076,"right2":70857356740635394959440,"fanout":236656509669},{"left1":20,"right1":39161803072312429016846,"left2":512,"right2":75230984691671678577062,"fanout":1056710312664},{"left1":16640,"right1":34747472411595188994884,"left2":49408,"right2":18996797757941211982962,"fanout":958145160245},{"left1":16384,"right1":58345883105925850622942,"left2":53568,"right2":38832962089032570725928,"fanout":1008362845232},{"left1":20480,"right1":21064326115814518419518,"left2":53248,"right2":26300725024294708588656,"fanout":124791305490},{"left1":32768,"right1":74284079774206024776978,"left2":37120,"right2":67111463345560803285880,"fanout":265801682648},{"left1":16448,"right1":20714650535567011156144,"left2":16704,"right2":48194133071342795037176,"fanout":202739282428},{"left1":64,"right1":57233877915105219190264,"left2":4416,"right2":44412837089798317649556,"fanout":881375832047},{"left1":33024,"right1":10597553519320527791778,"left2":49408,"right2":4039516421885909365784,"fanout":20869943074},{"left1":53504,"right1":60717265984032836586948,"left2":53568,"right2":2969278786902608287750,"fanout":897428162567},{"left1":49216,"right1":68783188214869988840676,"left2":53312,"right2":16480090532697425709650,"fanout":872131326146},{"left1":4096,"right1":31821475943705107394794,"left2":20480,"right2":51390442635023399738656,"fanout":137520944416},{"left1":20480,"right1":12771824489123450337590,"left2":20736,"right2":6515192012110951296928,"fanout":910200860140},{"left1":4160,"right1":13038183968161816212942,"left2":20544,"right2":29666644419228154207832,"fanout":868352343250},{"left1":4096,"right1":65936345048651492176608,"left2":36928,"right2":41123083541024501000834,"fanout":1077231891965},{"left1":64,"right1":37653594087952687316022,"left2":16640,"right2":67781340959786292157748,"fanout":134067439642},{"left1":16384,"right1":21819494642132791062044,"left2":20544,"right2":67241643525567975029010,"fanout":154566973231},{"left1":64,"right1":20036713803934575305160,"left2":256,"right2":55514256160843583666158,"fanout":34495519937},{"left1":4160,"right1":40125445073625742194196,"left2":4416,"right2":44567256841736575507704,"fanout":824847643455},{"left1":256,"right1":23404518880753069917976,"left2":4096,"right2":39060091213467000412016,"fanout":34379863504},{"left1":320,"right1":69169035121211760575766,"left2":4096,"right2":52750351674283173596068,"fanout":1017120695358},{"left1":32768,"right1":53720696034422722228906,"left2":36864,"right2":25441851747006486920032,"fanout":314575305},{"left1":32832,"right1":44954265014714637774132,"left2":49216,"right2":8143291745563927336802,"fanout":1035588152611},{"left1":4096,"right1":52075828940431633505840,"left2":16448,"right2":41460727127325706261572,"fanout":838336711213},{"left1":256,"right1":59519312853046159312080,"left2":33088,"right2":61180924418812835387122,"fanout":846844076285},{"left1":4096,"right1":21772946921898779075174,"left2":36864,"right2":11107419562268744215784,"fanout":133556207},{"left1":4096,"right1":21639015585451497254320,"left2":32768,"right2":71767588621145458113958,"fanout":824700702925},{"left1":256,"right1":61763672541933142117976,"left2":32768,"right2":26160391750696659474834,"fanout":29874164},{"left1":33024,"right1":67230479492024567537002,"left2":49408,"right2":57995428718852880671356,"fanout":1026646993932},{"left1":4096,"right1":49239835788274046216220,"left2":16448,"right2":64635034074634685789650,"fanout":850804855314},{"left1":4096,"right1":35028543779435467847108,"left2":20736,"right2":46865501237012090339426,"fanout":931221080038},{"left1":64,"right1":63198942073959129287984,"left2":320,"right2":30238121535316132108206,"fanout":137911460840},{"left1":32768,"right1":36242510025600161205986,"left2":36928,"right2":30306610641938874308090,"fanout":147081931580},{"left1":64,"right1":48044179962541298807444,"left2":320,"right2":44749194858196247926930,"fanout":818476498},{"left1":64,"right1":44972405006115680636354,"left2":49216,"right2":28541079208136068697904,"fanout":120000273},{"left1":64,"right1":70421639899713985189264,"left2":32768,"right2":987376504663841187798,"fanout":68805987899},{"left1":256,"right1":25405800568178949599320,"left2":49152,"right2":37489568363546826029472,"fanout":910183491280},{"left1":4160,"right1":59973584807866669322402,"left2":4416,"right2":20393231703963723899700,"fanout":832607042771},{"left1":4096,"right1":41469610403428226590846,"left2":20736,"right2":33913327314495384744684,"fanout":226623365925},{"left1":256,"right1":23691917350497453182804,"left2":4096,"right2":17303343919283759985240,"fanout":4563981839},{"left1":16448,"right1":5306489595663671023696,"left2":20544,"right2":36307214507533790638732,"fanout":120660958727},{"left1":16384,"right1":68091713738525199370228,"left2":16448,"right2":69137427988445598389532,"fanout":34993526334},{"left1":64,"right1":25083509126079288957228,"left2":16448,"right2":69091499883265577534006,"fanout":3269184234},{"left1":4160,"right1":718910834924584564328,"left2":32768,"right2":72888523044181456177176,"fanout":858256768298},{"left1":32768,"right1":23752543760307156293226,"left2":33024,"right2":65830773375242643807266,"fanout":16119943456},{"left1":32768,"right1":50431140607063567730116,"left2":33024,"right2":12882810305765633026416,"fanout":37583308279},{"left1":64,"right1":31515060964271197672562,"left2":32768,"right2":29320526195613993292582,"fanout":8640019512},{"left1":16384,"right1":39493230347846432877652,"left2":16640,"right2":51394064446007209991444,"fanout":824883990312},{"left1":64,"right1":16169367070918385688396,"left2":16384,"right2":26137602015000364214496,"fanout":4396361528},{"left1":256,"right1":58681562014388712892038,"left2":32768,"right2":64805177001171709645798,"fanout":720623056},{"left1":4096,"right1":16606918491361397500210,"left2":20480,"right2":65478584094498765037026,"fanout":824754438122},{"left1":64,"right1":29925019574384600221692,"left2":37120,"right2":44280360241277662368504,"fanout":132184850640},{"left1":4160,"right1":38650621773785810969440,"left2":20544,"right2":8509690084711870287330,"fanout":969591942618},{"left1":64,"right1":52350687326868220700244,"left2":16384,"right2":18815325336608117727338,"fanout":149414381},{"left1":33024,"right1":4523112284881341981366,"left2":37120,"right2":16662717804308774916122,"fanout":918889178608},{"left1":36864,"right1":11877336344624160397180,"left2":36928,"right2":60031831934225900890338,"fanout":90819053333},{"left1":16384,"right1":74096019653270055953182,"left2":53248,"right2":8028287754380878661250,"fanout":1018547128302},{"left1":16384,"right1":24281184947262299138516,"left2":16640,"right2":73717504064202165760544,"fanout":137675931647},{"left1":64,"right1":34624074373800993685428,"left2":33024,"right2":5382333814240173843722,"fanout":184902948102},{"left1":64,"right1":8150641766815736703230,"left2":16384,"right2":66956623269637534519482,"fanout":17834832855},{"left1":32768,"right1":3995315320507547349050,"left2":32832,"right2":57342176718480103386632,"fanout":17216831182},{"left1":36864,"right1":42220895046416392408624,"left2":53248,"right2":2112311790847298422116,"fanout":1022689464117},{"left1":16384,"right1":15118987714794351842214,"left2":36864,"right2":64336909820588753182594,"fanout":1069046364171},{"left1":16384,"right1":7137486648455722780638,"left2":20480,"right2":1564553296443513302002,"fanout":657512765},{"left1":4096,"right1":52253305274348022743734,"left2":4352,"right2":48678764670540373113326,"fanout":199021513},{"left1":32768,"right1":47037904798229801705760,"left2":33024,"right2":67347434244061471346800,"fanout":405603125},{"left1":32768,"right1":29655103940136801403530,"left2":49152,"right2":29825780831228597427758,"fanout":305198534},{"left1":16640,"right1":65855938091203329323466,"left2":49408,"right2":64781422007230617653502,"fanout":868023542272},{"left1":4096,"right1":4012631529228791553784,"left2":36864,"right2":21141346099824958764332,"fanout":37594667473},{"left1":16448,"right1":28961785634465024599924,"left2":49216,"right2":54698932324931980182396,"fanout":42276742361},{"left1":320,"right1":22177484917307555452080,"left2":16704,"right2":25742530126859894903510,"fanout":21238503693},{"left1":16448,"right1":44407692439173994203622,"left2":32768,"right2":64546385207031976486160,"fanout":214984626131},{"left1":64,"right1":36272631469673250876930,"left2":4160,"right2":35722083686701006999436,"fanout":114490914},{"left1":256,"right1":73224203496649794760736,"left2":16448,"right2":75262241386489459745760,"fanout":1042955186642},{"left1":32768,"right1":24248549986200387982898,"left2":53504,"right2":74996040362652648792174,"fanout":1087565268435},{"left1":32768,"right1":29576429362036614651800,"left2":36928,"right2":15048947352452577382484,"fanout":1017133598986},{"left1":256,"right1":33337096940106687096596,"left2":16640,"right2":32984905545573409937756,"fanout":68924084284}],"pure":{"0":175503630089,"3":189283104740,"6":850751653570,"10":1017149263889,"11":56591898068,"13":141951095822,"16":133496166956,"80":214027983324,"176":872400622358,"192":123497951725,"224":1089895341814,"240":236992131823,"256":192992056854,"272":227212469994,"304":164251447833,"320":98396083745,"336":885765773354,"368":863320946947,"544":197938044624,"560":236474255147,"608":948917626410,"656":875433943097,"704":880351893468,"720":893668490690,"736":43906948067,"752":842886220554,"816":1039954601264,"848":176111866099,"864":997124197656,"880":1095083880692,"912":33440931024,"960":896824627194,"976":863183960027,"1024":1065701747917,"1040":850870596860,"1056":244977095372,"1072":901203155704,"1120":192740120258,"1152":93452109070,"1168":60998230573,"1216":1038931774006,"1":1013145273069,"7":881389138206,"8":176665525286,"9":1095247002145,"14":975071466689,"15":1061493142814,"96":928215322422,"128":257311298275,"144":51452786491,"160":924355473872,"352":957743704079,"384":926741555713,"448":1000190103089,"480":13322888456,"512":201311392990,"624":1017853780494,"672":167922685679,"768":937025654505,"800":1095951766260,"992":258738812442,"1008":47210162981,"1184":1030292815920,"2":222429719262,"12":1031429497385,"32":1001431756777,"112":928566220057,"288":155171044650,"496":1010021420242,"688":866879871462,"832":162433528350,"928":1038657048782,"944":1027435069698,"1136":1065994040325,"4":51255246808,"5":880817806864,"208":885165250041,"416":1099090494009,"528":1013347112162,"896":870842030796,"1104":1047750505771,"48":116903696884,"64":851179412015,"400":18049413875,"432":928368363016,"464":274656838418,"576":1053176360444,"592":871493610982,"784":983182475519,"1088":34490298365,"1200":72206918425,"640":936707481039},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"phased_canonical_6","capacity":6,"events":[{"kind":"products","indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74],"required":267,"cache":267},{"kind":"pure_low","monomial":3,"required":1,"cache":267},{"kind":"pure_low","monomial":5,"required":2,"cache":267},{"kind":"pure_low","monomial":7,"required":9,"cache":267},{"kind":"pure_low","monomial":13,"required":258,"cache":267},{"kind":"products","indices":[75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149],"required":113,"cache":121},{"kind":"pure_low","monomial":9,"required":16,"cache":121},{"kind":"pure_low","monomial":10,"required":32,"cache":121},{"kind":"pure_low","monomial":11,"required":65,"cache":121},{"kind":"products","indices":[150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224],"required":1677,"cache":1677},{"kind":"pure_low","monomial":6,"required":4,"cache":1677},{"kind":"pure_low","monomial":12,"required":128,"cache":1677},{"kind":"pure_low","monomial":14,"required":516,"cache":1677},{"kind":"pure_low","monomial":15,"required":1033,"cache":1677}],"low_computations":11,"peak_live":6,"min_live_required":6}},{"batch":2,"delta_sha256":"72d8ddb15223d40ef74d40f6daaff199444a82a5baafda649aed20a4fa658c47","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":8,"right1":91337010511311367156812,"left2":170,"right2":71436964839682257081406,"fanout":1078896557575},{"left1":2,"right1":102126168731437253754372,"left2":128,"right2":71900804918235469912982,"fanout":34510073365},{"left1":8,"right1":75258629276362633380568,"left2":8362,"right2":5875414408960556460816,"fanout":911307511050},{"left1":32,"right1":41948899714926698446698,"left2":128,"right2":19950016841296579349662,"fanout":137502836525},{"left1":34,"right1":70112968821638258102960,"left2":128,"right2":65028014718571903270832,"fanout":1043697821411},{"left1":32,"right1":10028278066271845378036,"left2":8352,"right2":65237770073619438915056,"fanout":129029890858},{"left1":2,"right1":135844266378777350182088,"left2":8194,"right2":40472086970475997390292,"fanout":16158764512},{"left1":8320,"right1":21631170552287059837016,"left2":8322,"right2":7440504742944523563038,"fanout":956906284590},{"left1":8194,"right1":17866308063490156450902,"left2":8322,"right2":11699989443315770966078,"fanout":158143349954},{"left1":128,"right1":62566125676872550201384,"left2":168,"right2":61078640361942729040442,"fanout":889982014718},{"left1":32,"right1":9509306045200248748344,"left2":34,"right2":4507369190387432689700,"fanout":35115187707},{"left1":2,"right1":121276241185778465691864,"left2":34,"right2":7386275578087314332052,"fanout":18084741058},{"left1":8192,"right1":95398847443307273528416,"left2":8194,"right2":119753857120622372839636,"fanout":21527070264},{"left1":40,"right1":58539332192347929489868,"left2":42,"right2":44423767062496647567496,"fanout":837857561644},{"left1":32,"right1":63551778684954306699616,"left2":8224,"right2":25525881937213538859550,"fanout":824703183114},{"left1":8,"right1":86402075487057790879844,"left2":34,"right2":7723013881823900739332,"fanout":932124816077},{"left1":8192,"right1":73052348905366313584290,"left2":8224,"right2":63460964136987551116386,"fanout":73231367974},{"left1":2,"right1":33782299552677515618838,"left2":8192,"right2":28057248578095000957430,"fanout":827874213649},{"left1":128,"right1":4011997413347612056136,"left2":160,"right2":50463289395570464554294,"fanout":4458927073},{"left1":128,"right1":32854486402288777115472,"left2":160,"right2":21350871198293597500126,"fanout":68972116694},{"left1":8200,"right1":6860153570592668455094,"left2":8328,"right2":3821928459612630610310,"fanout":261386670329},{"left1":128,"right1":14569379868283274248158,"left2":8320,"right2":41930567031485195861022,"fanout":141217375012},{"left1":40,"right1":27885317517566404835942,"left2":8192,"right2":73963495431692542396988,"fanout":42245295411},{"left1":10,"right1":136626978958079125494220,"left2":8192,"right2":59872979604616741546064,"fanout":863272844592},{"left1":32,"right1":32204680678294910534686,"left2":34,"right2":11844957046865053769944,"fanout":69372481806},{"left1":2,"right1":42316296652749527701128,"left2":8194,"right2":69296731509186725677578,"fanout":719058960},{"left1":8,"right1":76346755737255748237814,"left2":130,"right2":38167937737133620529092,"fanout":894239127059},{"left1":34,"right1":8773172485971369701070,"left2":42,"right2":45915626617478926124464,"fanout":910737005598},{"left1":8,"right1":139098858234454461875132,"left2":8192,"right2":51094080770592400630096,"fanout":304287244},{"left1":128,"right1":3336910372796367803168,"left2":162,"right2":66158795663268513739366,"fanout":863084273451},{"left1":32,"right1":65164762135348347132090,"left2":168,"right2":52899751021936687223818,"fanout":254120548355},{"left1":8,"right1":44172029410458466359172,"left2":8202,"right2":13968065372267580862414,"fanout":1099510116109},{"left1":32,"right1":68803450670552260044842,"left2":40,"right2":8326517549999763036984,"fanout":34362476268},{"left1":8,"right1":130633709373145006553322,"left2":8194,"right2":70912721124908269541648,"fanout":965579237108},{"left1":8,"right1":97079058736248002476200,"left2":8232,"right2":34422112133657725501138,"fanout":961602261219},{"left1":40,"right1":29613507365122724858010,"left2":8232,"right2":30209609425834035110432,"fanout":270617214190},{"left1":128,"right1":70721967729509722352470,"left2":138,"right2":4180320183988713435022,"fanout":1074278701857},{"left1":2,"right1":126298020792487288323504,"left2":128,"right2":38996193295706131034754,"fanout":137790477760},{"left1":136,"right1":131212600597408307652240,"left2":138,"right2":138905191521642872491404,"fanout":1061261986093},{"left1":162,"right1":96857646802040035972690,"left2":170,"right2":106068218464396910781016,"fanout":1018059877894},{"left1":8,"right1":49835037984532372692574,"left2":8200,"right2":65205626526610978236330,"fanout":34463741944},{"left1":8,"right1":123614674360967945512020,"left2":130,"right2":10698099829817634752850,"fanout":64142888730},{"left1":2,"right1":56968687870137925352018,"left2":8320,"right2":19542414644129129567940,"fanout":226576814854},{"left1":8,"right1":862164196628004318164,"left2":8330,"right2":31952824384107192666280,"fanout":13071271443},{"left1":2,"right1":24046936809131189409426,"left2":130,"right2":26922563134223938317704,"fanout":17313830146},{"left1":32,"right1":38960106455472580918344,"left2":8192,"right2":41968078218235662381530,"fanout":145059922210},{"left1":136,"right1":18813107279780207723080,"left2":138,"right2":4527577433749022476504,"fanout":198521134841},{"left1":128,"right1":58462088628957438450898,"left2":8328,"right2":39938804002932952174764,"fanout":984433889269},{"left1":136,"right1":70310543517465939666522,"left2":8328,"right2":28842628918492852747764,"fanout":132921755633},{"left1":2,"right1":31901800440134967202414,"left2":8,"right2":132445502766479400352652,"fanout":34395651118},{"left1":8,"right1":133288211135442779119594,"left2":8234,"right2":16159808292770245313136,"fanout":1001116601555},{"left1":8192,"right1":53538538169339631289262,"left2":8226,"right2":17199356617477788974480,"fanout":73706111535},{"left1":2,"right1":6763766035583695830656,"left2":130,"right2":40731596828502505761628,"fanout":3710518244},{"left1":128,"right1":32477947395766734833216,"left2":8192,"right2":32421075006188855323662,"fanout":824668652816},{"left1":32,"right1":14894718139155757723752,"left2":8226,"right2":10306703594504384271144,"fanout":185556780993},{"left1":34,"right1":106409494625848558560150,"left2":42,"right2":90343679837885174701816,"fanout":55664372685},{"left1":136,"right1":42650171689901462626218,"left2":168,"right2":26134550709076638271640,"fanout":850136930816},{"left1":8,"right1":20203551664799629679052,"left2":8320,"right2":37236438475174400161278,"fanout":1022148859896},{"left1":8,"right1":134980831927101835719066,"left2":32,"right2":65179994287202355880294,"fanout":3385985840},{"left1":128,"right1":63797434310330015762322,"left2":160,"right2":72271935105108988967012,"fanout":17232230604},{"left1":2,"right1":19415552646417765887320,"left2":136,"right2":73767530345249432801320,"fanout":1047700644588},{"left1":32,"right1":56474950982652312650838,"left2":8232,"right2":21878793114985118852042,"fanout":846843609310},{"left1":8,"right1":43632101592825618614354,"left2":8320,"right2":46556400517991276156598,"fanout":166811989190},{"left1":2,"right1":67516661762471854743514,"left2":136,"right2":12614075354212743168158,"fanout":935360080584},{"left1":128,"right1":44497035099428527893760,"left2":168,"right2":34636697518640483629022,"fanout":971639686710},{"left1":32,"right1":34951967985653066680660,"left2":8224,"right2":35045747011971432168744,"fanout":400349149},{"left1":8,"right1":58514797615181055246052,"left2":160,"right2":34636513105059515039494,"fanout":134198324971},{"left1":32,"right1":94164894968585507284422,"left2":34,"right2":77883269053455998503148,"fanout":138193930735},{"left1":8,"right1":79266723121347173038110,"left2":8320,"right2":34222854709906102424398,"fanout":854047662529},{"left1":40,"right1":32663981357652282967440,"left2":168,"right2":2106245592453070382728,"fanout":150557099533},{"left1":8,"right1":114974938258137379145708,"left2":8330,"right2":1697683461970762587888,"fanout":170930143464},{"left1":32,"right1":37821621931963942813974,"left2":136,"right2":21316731721508118072778,"fanout":262057769229},{"left1":2,"right1":41713044731708659453694,"left2":8192,"right2":53851706132405602410438,"fanout":38808443613},{"left1":8192,"right1":88704331147131391751724,"left2":8200,"right2":138414850430260134347830,"fanout":235530181},{"left1":8,"right1":140961887735844471532020,"left2":8194,"right2":37866192940155182688510,"fanout":222563986197},{"left1":3072,"right1":51786885618208207553098,"left2":3584,"right2":74033609729639302721112,"fanout":911385502001},{"left1":1024,"right1":3001787528590405824108,"left2":3076,"right2":15435542087722325266744,"fanout":829046785045},{"left1":512,"right1":72143986641258565342430,"left2":1044,"right2":24257040133652872043482,"fanout":884877180211},{"left1":1024,"right1":3973622757195208082554,"left2":2052,"right2":61429244809255546079492,"fanout":205103617475},{"left1":20,"right1":42068935709039783775842,"left2":1024,"right2":69187295065535440131046,"fanout":1025846726637},{"left1":20,"right1":70822108833732658737898,"left2":2048,"right2":57244176828872814636874,"fanout":982540092641},{"left1":528,"right1":47052750980120450656962,"left2":532,"right2":70633531310456677355646,"fanout":81569500989},{"left1":1040,"right1":14103520870702177761748,"left2":1552,"right2":14140706689977748433944,"fanout":1004969796578},{"left1":1024,"right1":3086146987500875998350,"left2":1540,"right2":20989228620649258669170,"fanout":150875997919},{"left1":1040,"right1":61117223257498918101434,"left2":1552,"right2":38610477610082639588496,"fanout":120026825482},{"left1":2048,"right1":68860272083971125404836,"left2":3072,"right2":58592504343391856439532,"fanout":3542473524},{"left1":512,"right1":46889405941081319136518,"left2":1536,"right2":34391178859629678040652,"fanout":17596297460},{"left1":512,"right1":113841874647327643558,"left2":2048,"right2":69935319825796467216076,"fanout":17753171710},{"left1":1028,"right1":990983560748142671580,"left2":1044,"right2":41832719961258461087456,"fanout":850651572686},{"left1":512,"right1":55273544604108644589708,"left2":1024,"right2":12979439152737983698382,"fanout":825123157563},{"left1":1024,"right1":74135030123448558033938,"left2":1536,"right2":2846593777691937056606,"fanout":53793315},{"left1":4,"right1":101188187965888298596974,"left2":2048,"right2":69859644821794562619312,"fanout":17244868059},{"left1":20,"right1":38280436751674042857692,"left2":1024,"right2":27414229883623093673226,"fanout":1052497868501},{"left1":16,"right1":43279608015195568369728,"left2":20,"right2":65480525630837255815748,"fanout":3285064386},{"left1":4,"right1":111768606431715251432248,"left2":2560,"right2":31151530197657478252596,"fanout":210657671889},{"left1":16,"right1":62217238210699270216334,"left2":512,"right2":165401778758479919820,"fanout":135870934},{"left1":16,"right1":62907487042459604763988,"left2":516,"right2":69905147739104127083122,"fanout":119338563554},{"left1":528,"right1":42637183611369356480556,"left2":1552,"right2":40813908438541589872498,"fanout":1091507385596},{"left1":2048,"right1":31675310154468650253334,"left2":3072,"right2":67832288321629108878918,"fanout":824986185221},{"left1":2052,"right1":75550201354330178071068,"left2":2068,"right2":62006105819457139989800,"fanout":231579586298},{"left1":4,"right1":124854544045576379920220,"left2":20,"right2":75369858754610738143206,"fanout":137575322153},{"left1":16,"right1":35427900193519303643858,"left2":1536,"right2":19833462672298520877424,"fanout":26343830564},{"left1":4,"right1":37235100886334784552700,"left2":2048,"right2":22461423531242787742630,"fanout":3588045585},{"left1":4,"right1":65275227282133161118686,"left2":16,"right2":43203427713746119850702,"fanout":9324410675},{"left1":4,"right1":39634336859913172859458,"left2":1028,"right2":32881834181439936541894,"fanout":3774019550},{"left1":1024,"right1":40695760472866412244194,"left2":3072,"right2":53855589016450852313440,"fanout":137543600872},{"left1":16,"right1":52292563392511559743678,"left2":2064,"right2":12702366672688850259860,"fanout":8910720550},{"left1":2048,"right1":10375036442777792541620,"left2":2064,"right2":61528836994431257329510,"fanout":4617523241},{"left1":1024,"right1":3822203961929537053838,"left2":2048,"right2":42728099960819728090798,"fanout":566893536},{"left1":4,"right1":15736165715751106783230,"left2":516,"right2":59658767987167735951156,"fanout":824955297561},{"left1":512,"right1":56402037127320938835150,"left2":532,"right2":50366999481962719868324,"fanout":188491704029},{"left1":512,"right1":20769579478402289280790,"left2":2564,"right2":24939540106531045884654,"fanout":919864344019},{"left1":16,"right1":15844061648523442385436,"left2":2068,"right2":10370503147315030889488,"fanout":78367541814},{"left1":512,"right1":26719161274612736066172,"left2":2576,"right2":35724153025901142295064,"fanout":987442502161},{"left1":2064,"right1":19746583825722566037838,"left2":2576,"right2":57155208950676646083864,"fanout":13804177363},{"left1":1028,"right1":30397940942874261023634,"left2":1044,"right2":5891363524180279747992,"fanout":1040205469491},{"left1":1536,"right1":16988219870267556371340,"left2":2048,"right2":57128847412094009930426,"fanout":107662333244},{"left1":16,"right1":7471747176327682565338,"left2":2576,"right2":50270862937938449080702,"fanout":1056801023804},{"left1":2048,"right1":4600199702130670596864,"left2":3072,"right2":34252911407473997884138,"fanout":9125632980},{"left1":16,"right1":74511941828219384512752,"left2":20,"right2":56328630005729121708170,"fanout":100325067},{"left1":16,"right1":109286958497880934476326,"left2":20,"right2":94119100178374774183976,"fanout":4345638888},{"left1":4,"right1":96213251089840213248450,"left2":1028,"right2":18272479211415888016478,"fanout":8837716781},{"left1":1536,"right1":12893320068759820566860,"left2":1540,"right2":65539109961521897420716,"fanout":876678606306},{"left1":1024,"right1":9177108825415727746936,"left2":3072,"right2":72394358680119896581672,"fanout":34742210502},{"left1":4,"right1":138655879701249102782818,"left2":16,"right2":5520577423862729527210,"fanout":17212784444},{"left1":20,"right1":57194298146237190991778,"left2":512,"right2":5047327829524540824200,"fanout":974821649717},{"left1":2048,"right1":31620390785715739355274,"left2":2064,"right2":3266009581652645716818,"fanout":69003583697},{"left1":1536,"right1":17095688874592595557352,"left2":1552,"right2":54419551627361759467056,"fanout":142476175624},{"left1":512,"right1":40554914707071437440596,"left2":3604,"right2":37797249510226583410302,"fanout":192516842279},{"left1":1024,"right1":47433413354967530301622,"left2":3584,"right2":17688890006996627751488,"fanout":261001511981},{"left1":512,"right1":64071842813889727098120,"left2":1552,"right2":64229437106376488862378,"fanout":64657963293},{"left1":4,"right1":109712453118389318282504,"left2":512,"right2":70181086459768426526992,"fanout":585891065},{"left1":4,"right1":17012462082505702625350,"left2":1044,"right2":66607643713239867051600,"fanout":1035957303819},{"left1":2048,"right1":130090162515811211689352,"left2":2052,"right2":76125282722149763819346,"fanout":34493512921},{"left1":2048,"right1":30343727653745139783564,"left2":2064,"right2":4342712641554402264674,"fanout":556521980},{"left1":1024,"right1":55321986926103750955584,"left2":3584,"right2":41841081169577584485116,"fanout":91131670475},{"left1":16,"right1":58109011514246813045712,"left2":2560,"right2":13209003218490451786456,"fanout":146132841991},{"left1":4,"right1":5337108098463980661166,"left2":2564,"right2":202585216868765536308,"fanout":20539320884},{"left1":512,"right1":75554637210627441013740,"left2":1536,"right2":28959367189577609299712,"fanout":3242143253},{"left1":2048,"right1":46310466411898627533166,"left2":3584,"right2":6792748302736517966510,"fanout":132083997635},{"left1":512,"right1":55363857136153953272938,"left2":3584,"right2":19681716297520566477222,"fanout":69205631500},{"left1":2048,"right1":31652294933222387168868,"left2":3584,"right2":71595040544224109233588,"fanout":147020839378},{"left1":512,"right1":45312071526656417922256,"left2":1540,"right2":48148324779292061922670,"fanout":145272460580},{"left1":512,"right1":49013636543487352670002,"left2":2052,"right2":26596267906346123141752,"fanout":889780102449},{"left1":20,"right1":58528318598398335880926,"left2":2068,"right2":72922719537210235327186,"fanout":1005946281187},{"left1":512,"right1":76317383047054356182638,"left2":516,"right2":88303375495020708034436,"fanout":7579961634},{"left1":1024,"right1":13121520709148428221750,"left2":1040,"right2":72465047837611564141866,"fanout":4511116087},{"left1":16,"right1":31730611264060251134942,"left2":2576,"right2":43871104338328933428564,"fanout":141915850441},{"left1":4,"right1":97865584586042460596428,"left2":20,"right2":39429820139521524065750,"fanout":316865258},{"left1":4,"right1":146131759951276717402236,"left2":2564,"right2":52913164669489569016720,"fanout":1065069625915},{"left1":64,"right1":72478380410598877464966,"left2":49152,"right2":1092319092227480046322,"fanout":158258700839},{"left1":20480,"right1":56240550347445486845516,"left2":20544,"right2":29949641012128156354974,"fanout":922813341710},{"left1":4096,"right1":59187836932051750471870,"left2":32768,"right2":44134240920095733648826,"fanout":34897711634},{"left1":4096,"right1":70596074645490335269164,"left2":36864,"right2":59060508554689851607156,"fanout":18133422057},{"left1":33024,"right1":43935649288158456425264,"left2":33088,"right2":48173084526717298306252,"fanout":1017170635838},{"left1":4096,"right1":11085522138487081957500,"left2":33024,"right2":34896853623850340005514,"fanout":1073470300920},{"left1":37184,"right1":58279003288705399502412,"left2":53568,"right2":7983750253447284691698,"fanout":218085856273},{"left1":32768,"right1":49227410792668068441968,"left2":49408,"right2":32998457223933084819346,"fanout":116786404592},{"left1":16640,"right1":70621750661944059013346,"left2":49408,"right2":6995680639390620375062,"fanout":940127342066},{"left1":32768,"right1":27524935023973046592902,"left2":49152,"right2":21725010330510577001472,"fanout":77765667373},{"left1":256,"right1":20676823261980439249770,"left2":49408,"right2":30877096106754809385086,"fanout":832635283188},{"left1":16384,"right1":10875527541358836166578,"left2":16640,"right2":47009897357689449085146,"fanout":17180202468},{"left1":256,"right1":31240063714911633652942,"left2":20736,"right2":6826899732493414863450,"fanout":1078906265378},{"left1":320,"right1":43731316875270945109152,"left2":16704,"right2":6039215823012851366436,"fanout":966216856030},{"left1":64,"right1":43126410832342339145422,"left2":16448,"right2":60279678288627758970074,"fanout":34915031800},{"left1":4160,"right1":41527263832900959091944,"left2":4416,"right2":6041550670373345866588,"fanout":124332942282},{"left1":4096,"right1":51321344174116203590414,"left2":36864,"right2":732838678084413521040,"fanout":4632467768},{"left1":64,"right1":12670115492957926256204,"left2":4416,"right2":43521180591422432926562,"fanout":1009066715075},{"left1":256,"right1":22843391255937113546926,"left2":36864,"right2":32708846333240632126404,"fanout":150293258202},{"left1":320,"right1":4041520459608058497172,"left2":16384,"right2":70874350534423025351482,"fanout":863807259146},{"left1":4096,"right1":71071656841459351216600,"left2":32768,"right2":8850853728557719903440,"fanout":138195879894},{"left1":32768,"right1":41568175352153165855518,"left2":33088,"right2":39376148867937596249256,"fanout":51858847486},{"left1":64,"right1":6291996685071434877124,"left2":36928,"right2":8891607225621781056290,"fanout":149452945116},{"left1":64,"right1":55677829444383343669214,"left2":16448,"right2":56703388175002459317296,"fanout":5168886999},{"left1":49152,"right1":74018085773630575258728,"left2":49216,"right2":21337626254005635470640,"fanout":944543499486},{"left1":64,"right1":46081219557322963388246,"left2":16384,"right2":13690854532027490156808,"fanout":825204542681},{"left1":320,"right1":51023597376828532036722,"left2":16384,"right2":46293019002823582215318,"fanout":239576358100},{"left1":256,"right1":33183485627230221594524,"left2":16640,"right2":57860146290195631419544,"fanout":137471322933},{"left1":64,"right1":33184952674731952213378,"left2":16640,"right2":21916837867441881584880,"fanout":69152864022},{"left1":64,"right1":55433023367446115125098,"left2":4096,"right2":51784908458678124407532,"fanout":17214593988},{"left1":256,"right1":55028280888975108175330,"left2":32768,"right2":38221305039780266119084,"fanout":4327343642},{"left1":64,"right1":42985561656251832414138,"left2":32832,"right2":40799191336899287555904,"fanout":38673924041},{"left1":256,"right1":16554553702543589790810,"left2":33024,"right2":20406807770279623374730,"fanout":707215351},{"left1":256,"right1":68471283249737394497264,"left2":49472,"right2":48979729082281009280614,"fanout":837519989727},{"left1":32768,"right1":53228338655201820777656,"left2":33088,"right2":33395285751289973618804,"fanout":862552257477},{"left1":4096,"right1":26578265549843026232896,"left2":20736,"right2":65198615612586655082046,"fanout":192213819437},{"left1":49152,"right1":19930919130393298090750,"left2":53248,"right2":48932919286316472853464,"fanout":1073441352404},{"left1":32768,"right1":3419516305132306341640,"left2":49152,"right2":68645750855561841278500,"fanout":17701128738},{"left1":256,"right1":64255819136649867281006,"left2":32768,"right2":48196494751477295855138,"fanout":824921235920},{"left1":256,"right1":39849192914162367683196,"left2":16640,"right2":39620709971189184444146,"fanout":4341748006},{"left1":4096,"right1":6934098552924899035834,"left2":36864,"right2":8728798671355514128912,"fanout":3408145967},{"left1":4160,"right1":52288470998609214960278,"left2":16384,"right2":7901353079241763696216,"fanout":106438784533},{"left1":32768,"right1":33680046387407667024674,"left2":32832,"right2":41964300389160121591450,"fanout":21673733592},{"left1":32832,"right1":19237496650048692592814,"left2":33088,"right2":19564670910867821171594,"fanout":1083004428490},{"left1":16384,"right1":73861502159509788190848,"left2":20480,"right2":18886357816037733294198,"fanout":5018735670},{"left1":4096,"right1":26995219255759817574086,"left2":4416,"right2":28352109658751414922052,"fanout":914393467605},{"left1":64,"right1":31353556428702016723500,"left2":4096,"right2":34841069417147721020656,"fanout":2815530},{"left1":256,"right1":6311066296201324777920,"left2":37120,"right2":12586520854258712642226,"fanout":834025828658},{"left1":64,"right1":36510594084133120838540,"left2":36864,"right2":64332649047993145516052,"fanout":206342586922},{"left1":16384,"right1":38443968834916674144924,"left2":32768,"right2":2421881151219051604494,"fanout":34543241193},{"left1":16384,"right1":34160539715910148311590,"left2":49152,"right2":25489675151903161577834,"fanout":4933941302},{"left1":256,"right1":54022473115947801035020,"left2":49152,"right2":67968665153443913416922,"fanout":37901764293},{"left1":4096,"right1":36703944936231290298554,"left2":49152,"right2":63488744528118087636580,"fanout":193573953596},{"left1":4096,"right1":18072328839824246834638,"left2":49216,"right2":17062837471278078056322,"fanout":872620475665},{"left1":16448,"right1":17112542045626148707004,"left2":20544,"right2":12234450651104696171800,"fanout":958738144470},{"left1":4096,"right1":5359036598790303618848,"left2":4416,"right2":9480957166483819862338,"fanout":13842765361},{"left1":4096,"right1":7838323895355716895974,"left2":20480,"right2":43776568426647830132842,"fanout":4043035703},{"left1":4160,"right1":27525286995043112470818,"left2":4416,"right2":248396786263589448792,"fanout":227969394691},{"left1":20480,"right1":60952303171703400342266,"left2":20736,"right2":23855128460594742704292,"fanout":240733989641},{"left1":256,"right1":2288315789192967086236,"left2":32832,"right2":35490320463036578960208,"fanout":193829474249},{"left1":64,"right1":68582706262578705548504,"left2":36928,"right2":27723184015467069590516,"fanout":1035359551486},{"left1":256,"right1":42866076602611403838244,"left2":37120,"right2":60119385187215525845056,"fanout":897027392985},{"left1":64,"right1":9473512256557796879938,"left2":49152,"right2":19542927989099551483756,"fanout":74050631364},{"left1":4096,"right1":31589592562579617418266,"left2":36864,"right2":7163720721378596563026,"fanout":825069459007},{"left1":64,"right1":68055226142780613179244,"left2":16640,"right2":2454928686176158639634,"fanout":1018055884604},{"left1":4352,"right1":71651086740858879527930,"left2":37120,"right2":47419855748763169926378,"fanout":16957634245},{"left1":4096,"right1":15104285185617588599674,"left2":4352,"right2":4532384604776457707350,"fanout":68902437651},{"left1":64,"right1":58231775718081969612462,"left2":49152,"right2":51667966831914829852266,"fanout":1095232508967},{"left1":4160,"right1":2957376355092062151462,"left2":4416,"right2":58586062819544378080128,"fanout":825388251873},{"left1":64,"right1":35820925878263860328166,"left2":16704,"right2":17458476628029662944564,"fanout":919807210269},{"left1":64,"right1":67672671346008189088028,"left2":16384,"right2":69420719371724206457868,"fanout":137737538256},{"left1":4160,"right1":27190584664390774285296,"left2":4416,"right2":40194981510674758064268,"fanout":158495268088},{"left1":256,"right1":60668017664238037407720,"left2":33088,"right2":7295050320087246490504,"fanout":85786036740},{"left1":320,"right1":39736775084878210514112,"left2":4096,"right2":67834883473252876170586,"fanout":952610525149},{"left1":256,"right1":58724841965275575047032,"left2":16384,"right2":74867871316478634566242,"fanout":8807765753}],"pure":{"1":38214384324,"2":914625390360,"3":172136580625,"5":145557822149,"7":249526811926,"8":253722747375,"12":833521697743,"14":962727830017,"16":915564853809,"32":1065825528310,"48":924391766295,"112":185302568197,"128":56357811227,"144":1068440292291,"208":1001181274569,"272":1061665555142,"288":923346730189,"352":1077112740086,"368":125144713460,"384":159446981080,"400":94623051278,"416":1098904491516,"432":880013662945,"448":12110721300,"464":266941829135,"496":166432861189,"560":943819978001,"656":1069245529618,"688":85917051658,"704":992519719690,"720":898284860479,"736":151282645282,"768":249745425346,"816":881404201204,"848":829451865058,"864":837554651175,"880":217982963516,"944":867328705520,"992":1094459473345,"1008":25514995905,"1024":1026199252503,"1040":876008761861,"1056":68432941019,"1120":265587048695,"1136":241558162199,"1152":1079011589075,"1184":928434083329,"1200":944340849092,"1216":854563760186,"0":150894612778,"4":86403908595,"6":1009602076175,"13":150472501509,"64":42528947957,"96":235767341320,"176":967339330522,"304":21172067629,"512":897714417947,"528":85946860244,"576":952628295733,"608":948596833997,"672":909959303411,"752":1026534789110,"784":209579544012,"800":1031174821118,"832":1001462250294,"896":855386504507,"912":235617256912,"960":876860210930,"1088":1073974013247,"11":266257421344,"15":9409146568,"80":227050009659,"160":112610303442,"224":205574424842,"240":1009818674226,"256":60932153290,"320":828539018462,"544":261658452030,"624":231004768804,"976":43787346222,"1072":26069948893,"1104":880638367734,"1168":39259537958,"9":51015448829,"10":979169254107,"192":1094530945734,"592":991148520180,"928":132621393098,"480":928415212342,"336":834129102652,"640":228322245071},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"phased_canonical_6","capacity":6,"events":[{"kind":"products","indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74],"required":267,"cache":267},{"kind":"pure_low","monomial":3,"required":1,"cache":267},{"kind":"pure_low","monomial":5,"required":2,"cache":267},{"kind":"pure_low","monomial":7,"required":9,"cache":267},{"kind":"pure_low","monomial":13,"required":258,"cache":267},{"kind":"products","indices":[75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149],"required":113,"cache":121},{"kind":"pure_low","monomial":9,"required":16,"cache":121},{"kind":"pure_low","monomial":10,"required":32,"cache":121},{"kind":"pure_low","monomial":11,"required":65,"cache":121},{"kind":"products","indices":[150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224],"required":1677,"cache":1677},{"kind":"pure_low","monomial":6,"required":4,"cache":1677},{"kind":"pure_low","monomial":12,"required":128,"cache":1677},{"kind":"pure_low","monomial":14,"required":516,"cache":1677},{"kind":"pure_low","monomial":15,"required":1033,"cache":1677}],"low_computations":11,"peak_live":6,"min_live_required":6}},{"batch":3,"delta_sha256":"eb8cdde37a831f7b8f38091880d300cb2d88bb4d3899704ffbf93fda51f35458","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":2,"right1":92665325725217365370828,"left2":32,"right2":5168830154712255644744,"fanout":137537962222},{"left1":2,"right1":1328723220580850579104,"left2":8200,"right2":23509817712973827855514,"fanout":245790088957},{"left1":130,"right1":17442891059203149185188,"left2":162,"right2":31719496799448582430402,"fanout":128228663034},{"left1":8192,"right1":46440828170831244546350,"left2":8330,"right2":67922203857457284234694,"fanout":271089024041},{"left1":8194,"right1":116792874144800423614760,"left2":8202,"right2":135804334671482293367274,"fanout":853928064483},{"left1":128,"right1":59796102633055107131688,"left2":130,"right2":25511572690987407355478,"fanout":825006564606},{"left1":168,"right1":5485330245378175606096,"left2":170,"right2":27337535903686060232866,"fanout":86370552854},{"left1":8,"right1":26033119247660298615284,"left2":8354,"right2":3334363603374870650512,"fanout":836698249726},{"left1":8352,"right1":47408642188231550551938,"left2":8360,"right2":13116144466164610053362,"fanout":1039917446874},{"left1":2,"right1":871284739977632030128,"left2":42,"right2":71778462633397319905778,"fanout":1039553204441},{"left1":32,"right1":38476462401622243387722,"left2":8200,"right2":26502997752344526788362,"fanout":82172447787},{"left1":128,"right1":39617995063963747147588,"left2":138,"right2":55961120794088231679870,"fanout":37784324151},{"left1":130,"right1":62051654489343291306828,"left2":8322,"right2":24766469811699892015326,"fanout":1038808505383},{"left1":2,"right1":54479958279453996723680,"left2":130,"right2":37545661865113657916352,"fanout":4733004858},{"left1":8,"right1":24043245088254205812972,"left2":128,"right2":46731674740528573124526,"fanout":17381783241},{"left1":130,"right1":16578159768739343715348,"left2":138,"right2":20071449276794896183930,"fanout":16474688973},{"left1":2,"right1":117509441730303092347858,"left2":8192,"right2":37438158422959699966122,"fanout":4342859474},{"left1":128,"right1":63508962434407939046708,"left2":138,"right2":1514909940614884627632,"fanout":871830913031},{"left1":32,"right1":51158191339200496268116,"left2":42,"right2":54741902929357974229488,"fanout":871910946296},{"left1":8192,"right1":55945702196994922612852,"left2":8320,"right2":9979771819784734046148,"fanout":34424149227},{"left1":40,"right1":38060702392333288962724,"left2":8232,"right2":66010953152543411040208,"fanout":872496578062},{"left1":8200,"right1":92802220082616900455812,"left2":8202,"right2":95384286864802699405286,"fanout":875872649525},{"left1":160,"right1":37912947830821004929220,"left2":8192,"right2":26103254559601780618796,"fanout":85929437900},{"left1":2,"right1":99820483597757156192718,"left2":130,"right2":15174941133389432974358,"fanout":68901391102},{"left1":8226,"right1":4323728091695571479178,"left2":8234,"right2":18808470763373431451160,"fanout":885178504153},{"left1":128,"right1":21541190235447263517164,"left2":8192,"right2":46639636063253887737636,"fanout":140679850442},{"left1":32,"right1":53493887712130579597688,"left2":162,"right2":51017265263649079210276,"fanout":202233680080},{"left1":8192,"right1":11349331887188966021208,"left2":8322,"right2":32612247480739276573640,"fanout":77004652788},{"left1":8,"right1":39693325227379200749128,"left2":10,"right2":90427871031199112027770,"fanout":4308797912},{"left1":8192,"right1":48503110510669862443992,"left2":8352,"right2":56153279325998128877828,"fanout":860050230290},{"left1":8,"right1":49821924071983027412634,"left2":10,"right2":143729604703425552465930,"fanout":583732492},{"left1":10,"right1":42706585646255592399608,"left2":8202,"right2":21832520385098902644702,"fanout":108265154337},{"left1":128,"right1":39711456473700929248946,"left2":8192,"right2":59618716168177484922714,"fanout":827922984153},{"left1":8224,"right1":149337653435613054244242,"left2":8232,"right2":109061908710286582872174,"fanout":56675648055},{"left1":128,"right1":40821731402828227903534,"left2":8328,"right2":72577665621507674938862,"fanout":1021199250189},{"left1":2,"right1":141930472595892301991662,"left2":8322,"right2":73332091373936755047368,"fanout":155040873211},{"left1":8,"right1":148214729028982913561158,"left2":130,"right2":31809128387377783374978,"fanout":34426439994},{"left1":8,"right1":1594947186534488731356,"left2":136,"right2":28071414965659189283526,"fanout":69471620319},{"left1":32,"right1":5815778644177008607398,"left2":160,"right2":37990392867151478133868,"fanout":42293386558},{"left1":8192,"right1":49684032595424408790976,"left2":8194,"right2":2722804580224416769510,"fanout":824680521179},{"left1":130,"right1":29629104075139258510658,"left2":8322,"right2":49642190985868481170288,"fanout":1095699189205},{"left1":8192,"right1":72277005416731860258304,"left2":8224,"right2":59867037716890275599330,"fanout":4497145351},{"left1":8,"right1":66804851000748437748754,"left2":136,"right2":40854507646553799958924,"fanout":29557196},{"left1":2,"right1":115351331982334726449550,"left2":34,"right2":28353976940440236500924,"fanout":824787791662},{"left1":128,"right1":74593728385653010235328,"left2":8320,"right2":20847628778662495818452,"fanout":371009577},{"left1":160,"right1":3102432573866904190676,"left2":8352,"right2":51753482448990334223944,"fanout":150287538901},{"left1":128,"right1":38034821466242476905774,"left2":8192,"right2":5031369987110556589130,"fanout":17267874284},{"left1":32,"right1":142202929887854903422990,"left2":40,"right2":90314366216876866338782,"fanout":3540508678},{"left1":8320,"right1":107389261496299697698002,"left2":8322,"right2":126371562900235540766776,"fanout":163395026743},{"left1":8224,"right1":63824080376576401365044,"left2":8232,"right2":4175883899620569702482,"fanout":39544550602},{"left1":8,"right1":49341410694737512006504,"left2":40,"right2":25868090295124146709556,"fanout":68733884686},{"left1":42,"right1":26203302297183697235112,"left2":170,"right2":28534539372596824967918,"fanout":1001585629974},{"left1":2,"right1":19239532893599441166940,"left2":8202,"right2":46296374513528293739738,"fanout":81787801127},{"left1":40,"right1":32266866287607827372974,"left2":8192,"right2":5572727943693143083148,"fanout":232512228308},{"left1":130,"right1":66124939672712929480900,"left2":138,"right2":58694326236643771834202,"fanout":1021317019700},{"left1":8,"right1":103973327943651027819620,"left2":168,"right2":27468333478210899019290,"fanout":905986062651},{"left1":8,"right1":44091463314647091343112,"left2":8194,"right2":13568749933034212594660,"fanout":274034983741},{"left1":160,"right1":94860640519273969205748,"left2":162,"right2":120727969146292676554360,"fanout":1035909207761},{"left1":2,"right1":128074919142544700576432,"left2":8226,"right2":51683658776320622792698,"fanout":1055853562418},{"left1":8,"right1":103425452368878528047478,"left2":40,"right2":62506058704089642658382,"fanout":922430437},{"left1":40,"right1":38001738938799465365864,"left2":8224,"right2":70412731383446001160576,"fanout":91081800665},{"left1":10,"right1":5790963112636126065814,"left2":8202,"right2":1886918364413512139960,"fanout":227836425699},{"left1":32,"right1":6205047861392521798734,"left2":8192,"right2":20876169035532498155898,"fanout":17446331116},{"left1":32,"right1":55724500926034536791036,"left2":160,"right2":29005430487187890886144,"fanout":828573888308},{"left1":40,"right1":34220607071936806107724,"left2":168,"right2":4611888710523992963458,"fanout":962491246024},{"left1":8192,"right1":47871131464720348151690,"left2":8224,"right2":15054154084640305543748,"fanout":824981990892},{"left1":8322,"right1":66824856854673522261794,"left2":8330,"right2":40707333934393134681466,"fanout":73972508874},{"left1":130,"right1":39634625328767270440782,"left2":138,"right2":24316713583245334083700,"fanout":202437821888},{"left1":8,"right1":116091890761178434089720,"left2":8226,"right2":8361169778802711454576,"fanout":880030841112},{"left1":160,"right1":82403626749928275505474,"left2":162,"right2":140332688872271116583714,"fanout":93734749705},{"left1":160,"right1":71546585530123646960866,"left2":162,"right2":43640172159243264112606,"fanout":965829264133},{"left1":2,"right1":50778170664244305530686,"left2":8232,"right2":15639029137741971681192,"fanout":970323789035},{"left1":32,"right1":27308655575044978839630,"left2":34,"right2":64100725477501055830442,"fanout":17753693683},{"left1":8,"right1":116774265620882335268120,"left2":168,"right2":74763475071833120621414,"fanout":923282451255},{"left1":32,"right1":7594814185435134936200,"left2":8200,"right2":55876859333511162296616,"fanout":923450407448},{"left1":3600,"right1":142246567378095294947380,"left2":3604,"right2":113246629856297484751756,"fanout":249426532842},{"left1":2048,"right1":36617954573234449547070,"left2":2576,"right2":71413476122586007827626,"fanout":931015547364},{"left1":1024,"right1":25633157452049104221560,"left2":1540,"right2":20317968664611563703850,"fanout":948535240386},{"left1":512,"right1":63961026572553710007848,"left2":3600,"right2":47062246556540667331526,"fanout":850356466662},{"left1":1040,"right1":6152353421435389730962,"left2":1552,"right2":5317725216932092648140,"fanout":1043713164581},{"left1":16,"right1":33043171193535735446036,"left2":1028,"right2":27215329208928272438078,"fanout":829746704163},{"left1":512,"right1":41710383639854430241970,"left2":2576,"right2":43153502886902560843428,"fanout":965343053593},{"left1":512,"right1":27688519037571661714644,"left2":3072,"right2":49565017628920676190036,"fanout":129554921710},{"left1":16,"right1":32757688974224671960302,"left2":3092,"right2":69431469980400453820854,"fanout":1051814923988},{"left1":1024,"right1":29597433332152937123228,"left2":3092,"right2":9821360502413116058866,"fanout":1060267875639},{"left1":512,"right1":64634967122225405646140,"left2":2564,"right2":20043350942227057834472,"fanout":252983434969},{"left1":4,"right1":76809373941270174902720,"left2":2564,"right2":61005102457093080003738,"fanout":872864685042},{"left1":16,"right1":24616013979082934596448,"left2":2048,"right2":10379091034779736848556,"fanout":8825854264},{"left1":512,"right1":43752118528706425494856,"left2":2048,"right2":57317548721954574050914,"fanout":34664665078},{"left1":4,"right1":1258377119752347893098,"left2":2048,"right2":44518196608934597722230,"fanout":941096441},{"left1":2560,"right1":49132139452745620144256,"left2":2564,"right2":30735152820308830005296,"fanout":919473817040},{"left1":528,"right1":100466033652993927954200,"left2":532,"right2":119518105872695233390316,"fanout":988583696181},{"left1":2064,"right1":23612751034270914002458,"left2":2576,"right2":19117461833614869315540,"fanout":931256612633},{"left1":16,"right1":37247785113996719448564,"left2":3072,"right2":60285087282563960086544,"fanout":176142345746},{"left1":512,"right1":75417363087647418243694,"left2":1024,"right2":54480568910843249990360,"fanout":567490268},{"left1":1024,"right1":11251838572860745607332,"left2":3072,"right2":47620607837255015624190,"fanout":72743897907},{"left1":16,"right1":17820274406099883843152,"left2":2052,"right2":72340841082708741164140,"fanout":1000441455117},{"left1":512,"right1":34320798390100688327324,"left2":2048,"right2":39053929588624059249070,"fanout":824932700918},{"left1":20,"right1":41554477300890174157738,"left2":532,"right2":58325554941563842265556,"fanout":248535643900},{"left1":2048,"right1":12788128517535835628358,"left2":2068,"right2":67261036236587255813356,"fanout":902664736203},{"left1":2564,"right1":70103170274346292934148,"left2":2580,"right2":32426208904396166505570,"fanout":901120442874},{"left1":2048,"right1":139377780896365102282352,"left2":2052,"right2":126387645694060778718762,"fanout":51582691},{"left1":2048,"right1":43600423294637094426410,"left2":2052,"right2":65063658840060774489930,"fanout":4579522535},{"left1":2048,"right1":10673958377049682764348,"left2":3072,"right2":45705912894890015962122,"fanout":8088843237},{"left1":2048,"right1":11397644372644243024878,"left2":2064,"right2":52292488010073693146190,"fanout":824865137358},{"left1":2064,"right1":132912725585673580695388,"left2":2068,"right2":118814200556633284080218,"fanout":1090472821035},{"left1":1040,"right1":11468486540311927751872,"left2":1044,"right2":160020866112736013952,"fanout":3707690227},{"left1":16,"right1":53408126599866448276760,"left2":1028,"right2":73056716216507659830966,"fanout":932597256688},{"left1":1024,"right1":11217250398835665681062,"left2":2048,"right2":27075853733112181001656,"fanout":455810088},{"left1":1024,"right1":42991661090094989965204,"left2":1040,"right2":27954720381125191990954,"fanout":17316442924},{"left1":1536,"right1":23227413037843759293440,"left2":3584,"right2":52412729447339696611690,"fanout":1004685613349},{"left1":4,"right1":38027120809636171276468,"left2":1540,"right2":61095092871008548406970,"fanout":72157416243},{"left1":4,"right1":121101782318534535719156,"left2":2048,"right2":45405508014293858015864,"fanout":68800153295},{"left1":16,"right1":8226190454307628692164,"left2":1552,"right2":67601823607310528757164,"fanout":270549701838},{"left1":16,"right1":64180997563344026056068,"left2":1552,"right2":55396351587871862046888,"fanout":872368172853},{"left1":1024,"right1":11079541450212676612672,"left2":2560,"right2":15190692425972893954130,"fanout":25955535641},{"left1":1024,"right1":61547732933731853524424,"left2":1028,"right2":18699376582024703234036,"fanout":51591241711},{"left1":16,"right1":44832631103247606007308,"left2":1040,"right2":22890443615274139880202,"fanout":206158691793},{"left1":16,"right1":22937262718711353132976,"left2":1040,"right2":37033456142840535633474,"fanout":36518646},{"left1":512,"right1":1882975048962375047548,"left2":1536,"right2":23744329242358181341728,"fanout":4311749890},{"left1":1040,"right1":19149192436978863901898,"left2":3088,"right2":73554982038113432604802,"fanout":945934360276},{"left1":4,"right1":68349042717339581695922,"left2":1540,"right2":32240616545567109823480,"fanout":253251697680},{"left1":1024,"right1":46429942135016074526950,"left2":3072,"right2":21615535145357351995448,"fanout":21157120304},{"left1":2052,"right1":70494901521988799057720,"left2":2068,"right2":56591658199399167025402,"fanout":906775824621},{"left1":20,"right1":20189139348543336388672,"left2":532,"right2":59698116967976338346276,"fanout":244060903924},{"left1":2048,"right1":8170595700912605504144,"left2":2064,"right2":31440040690988619881934,"fanout":34880171020},{"left1":16,"right1":17620377122987145679036,"left2":2560,"right2":61253975439256594938998,"fanout":1055726289663},{"left1":512,"right1":57813894774291565322592,"left2":528,"right2":35120410386740621730978,"fanout":68919361493},{"left1":20,"right1":53854732387709995250054,"left2":1024,"right2":733209035657516726168,"fanout":249814696215},{"left1":16,"right1":38272940803136599560278,"left2":528,"right2":73128713671797664075612,"fanout":422175276},{"left1":20,"right1":2547958286962801183192,"left2":1044,"right2":9410485406576142970400,"fanout":252733499421},{"left1":2052,"right1":65906088483155397102040,"left2":3076,"right2":6147482080300926411852,"fanout":72598288331},{"left1":516,"right1":54706014372227244300792,"left2":532,"right2":9163425437162976491026,"fanout":944842353622},{"left1":16,"right1":74077191784702111449230,"left2":2052,"right2":64205068660949162035686,"fanout":184432734463},{"left1":4,"right1":48779003915396244278674,"left2":512,"right2":18871305569885963462860,"fanout":68723541029},{"left1":4,"right1":140556973960968413850088,"left2":1024,"right2":61381877659926738026830,"fanout":551495942},{"left1":4,"right1":55773931694249078130068,"left2":2564,"right2":42959667027011103439044,"fanout":995863313677},{"left1":512,"right1":55889167667090232610466,"left2":1024,"right2":71524735760551487689856,"fanout":137592650501},{"left1":2064,"right1":137404824879583457502412,"left2":2068,"right2":92617177413298824149662,"fanout":958198646756},{"left1":512,"right1":46846207604023272847900,"left2":1024,"right2":33898596689347175914416,"fanout":86046280418},{"left1":4,"right1":53457670507441139396904,"left2":1024,"right2":10323145039226338538128,"fanout":136718859},{"left1":4,"right1":103035854629613306420718,"left2":1044,"right2":65826339083292133759326,"fanout":274745926100},{"left1":512,"right1":57500711075231700697970,"left2":1040,"right2":17147364731823826142530,"fanout":1013565155876},{"left1":20,"right1":16451989419411606483746,"left2":2068,"right2":25802089746927533913448,"fanout":30403141641},{"left1":4,"right1":74238092093721309873626,"left2":1028,"right2":60300762281102521942646,"fanout":824687852588},{"left1":4,"right1":123549683119974827107852,"left2":1536,"right2":24060123827857714268034,"fanout":978919631575},{"left1":16,"right1":59059286805474442373864,"left2":512,"right2":21848390385435087082670,"fanout":824648715828},{"left1":4,"right1":117512042860087366464376,"left2":1044,"right2":38279964157642672320642,"fanout":927246854705},{"left1":4,"right1":138842782507404318664842,"left2":1536,"right2":65588788088982931388756,"fanout":1009351121434},{"left1":512,"right1":64744746603209433858376,"left2":528,"right2":15282738101757388396240,"fanout":8855367146},{"left1":33024,"right1":3447749276542251760640,"left2":37120,"right2":66002176765253503501380,"fanout":245883477025},{"left1":64,"right1":59950406647538221750262,"left2":4352,"right2":41510835174399966210174,"fanout":59257064900},{"left1":256,"right1":25632708095986741585058,"left2":20544,"right2":51556129497126999854120,"fanout":1039267195450},{"left1":33024,"right1":65471391483036843130052,"left2":37120,"right2":68665076357246270239748,"fanout":38249356290},{"left1":32768,"right1":17492863017304227757632,"left2":53568,"right2":61589229523220650505520,"fanout":867973412062},{"left1":256,"right1":33867264928942259269552,"left2":33088,"right2":72232172841644606427266,"fanout":206909875726},{"left1":49152,"right1":45235198851988813557764,"left2":49408,"right2":39683588558256383614632,"fanout":995976474628},{"left1":64,"right1":34565231312348418189400,"left2":4096,"right2":24122963853766002851214,"fanout":774058469},{"left1":32768,"right1":25739038856828507887404,"left2":33088,"right2":19390378250283699175196,"fanout":197924950210},{"left1":4096,"right1":56979011014004229847680,"left2":16640,"right2":39376386051207765378302,"fanout":73012216845},{"left1":256,"right1":21497976822746712533976,"left2":4160,"right2":73080400195286705463662,"fanout":836579742198},{"left1":16448,"right1":53309721159490819841478,"left2":49216,"right2":16145355098032898438310,"fanout":961319203113},{"left1":16384,"right1":49092270592971820130888,"left2":32768,"right2":1772462859481481398690,"fanout":34525857595},{"left1":64,"right1":1617246154304690956254,"left2":49152,"right2":17072040891095056020182,"fanout":214448190},{"left1":33024,"right1":22264075357659046225104,"left2":49408,"right2":32875858499330250652914,"fanout":1086479598884},{"left1":256,"right1":58919928879846825580434,"left2":20480,"right2":22304394363726941736186,"fanout":81489030651},{"left1":64,"right1":19878638350674080639810,"left2":4352,"right2":43551422088452381628968,"fanout":1001684072459},{"left1":64,"right1":53893505053652757853670,"left2":4160,"right2":38455459281786403462132,"fanout":38904413173},{"left1":4096,"right1":38650856673004184419366,"left2":32768,"right2":20844382839677277125246,"fanout":824902692863},{"left1":4352,"right1":41150008289639309132342,"left2":37120,"right2":20042713618702730125392,"fanout":859932850477},{"left1":4096,"right1":16102384036480544759036,"left2":53312,"right2":44783092299406363489356,"fanout":47512173000},{"left1":16384,"right1":12898622846137681427982,"left2":32768,"right2":40386722551900919405594,"fanout":4610247162},{"left1":64,"right1":44658531429709776247550,"left2":32768,"right2":15438253584001435749510,"fanout":38786287924},{"left1":256,"right1":10897378086091130173558,"left2":33024,"right2":45013455475482670274394,"fanout":3236224765},{"left1":4096,"right1":39621941548707505356398,"left2":4160,"right2":55067737960126116307738,"fanout":3505336288},{"left1":16384,"right1":25012453349476179133424,"left2":49408,"right2":46988938122479940435494,"fanout":110996960494},{"left1":256,"right1":58803122267516911037504,"left2":32768,"right2":276456914183432625624,"fanout":137621588795},{"left1":64,"right1":19251414859283170459504,"left2":16384,"right2":18939753361953188901896,"fanout":137995885781},{"left1":32768,"right1":72127725521721460163702,"left2":49152,"right2":55505918560990422103078,"fanout":859510236},{"left1":256,"right1":65610229647933334909948,"left2":4352,"right2":51265450088760081856430,"fanout":68755137029},{"left1":64,"right1":17350454023691939468302,"left2":16448,"right2":60356874313123102192666,"fanout":354631621},{"left1":64,"right1":56813285162701437966624,"left2":16448,"right2":51105747110109211870248,"fanout":69356553972},{"left1":36928,"right1":42263765006617590229066,"left2":37184,"right2":20437593567452346174460,"fanout":957910994927},{"left1":4160,"right1":59996077814762195601320,"left2":4416,"right2":36178130801478190746070,"fanout":1023155052315},{"left1":64,"right1":5870147284040003608054,"left2":4096,"right2":11186522920193838932512,"fanout":20715556},{"left1":256,"right1":15108662199671008575472,"left2":4160,"right2":32483539438283014803392,"fanout":1026615005979},{"left1":33024,"right1":41082956109927675116816,"left2":33088,"right2":49549383599201303506356,"fanout":1087160639450},{"left1":32832,"right1":42185527588248300705542,"left2":49216,"right2":71226431947899501846296,"fanout":945449475081},{"left1":64,"right1":62029835350833856953922,"left2":32832,"right2":61781664125292828835350,"fanout":17229142840},{"left1":4096,"right1":71356034490757569597188,"left2":32768,"right2":49083224199625584835950,"fanout":17619957233},{"left1":256,"right1":31784826077052983434878,"left2":16448,"right2":50225661831353488230810,"fanout":926874938134},{"left1":32768,"right1":4477711015841619326064,"left2":32832,"right2":59535629441509485112538,"fanout":141830533941},{"left1":16384,"right1":18928174297894048710330,"left2":53504,"right2":49506756572135261623936,"fanout":188192140764},{"left1":16384,"right1":5996341545169741995524,"left2":32768,"right2":71150323289063165467452,"fanout":17482839511},{"left1":4160,"right1":43822905152560171438384,"left2":16384,"right2":8609435012660992591614,"fanout":880386898744},{"left1":64,"right1":71203020679508608763970,"left2":16448,"right2":40346693019486740196670,"fanout":3438865346},{"left1":36864,"right1":2008154525669200382132,"left2":53248,"right2":63249841663144946394960,"fanout":158172003293},{"left1":4352,"right1":40470792747555613062914,"left2":32768,"right2":71625356288852287761290,"fanout":1083191458603},{"left1":64,"right1":45407157925063989239258,"left2":256,"right2":48075679738429217659370,"fanout":17533694455},{"left1":4352,"right1":40671075858515402088900,"left2":16384,"right2":61913444224097847688106,"fanout":93517898767},{"left1":16384,"right1":6501437692796478960288,"left2":53248,"right2":62790034029983395031904,"fanout":269574984455},{"left1":64,"right1":65502819465337048132622,"left2":4096,"right2":36670044456603187262304,"fanout":73250118118},{"left1":4352,"right1":64106104355627344232884,"left2":16384,"right2":32974944329112553497232,"fanout":1078422001107},{"left1":256,"right1":12875605159293140902428,"left2":16448,"right2":41206253262026412971002,"fanout":889473736897},{"left1":16384,"right1":30672912147630338970766,"left2":16448,"right2":14187917369831406146824,"fanout":824885502735},{"left1":256,"right1":16102500286705534483430,"left2":33024,"right2":53216076192524026431788,"fanout":824800507331},{"left1":64,"right1":18440691778423658639610,"left2":4160,"right2":10362049081346523117148,"fanout":829210042174},{"left1":4096,"right1":45982416739997900407284,"left2":16448,"right2":40531409755962776293954,"fanout":240717408962},{"left1":32768,"right1":44474782645590717404790,"left2":49408,"right2":21002246879831430770272,"fanout":1026276528065},{"left1":256,"right1":67139933028001456652992,"left2":49472,"right2":29052817505125170500624,"fanout":213775431985},{"left1":256,"right1":67545921511058076588702,"left2":4096,"right2":45318142722878601792044,"fanout":3221956372},{"left1":256,"right1":45729835439278641237082,"left2":16448,"right2":73398398215149615737640,"fanout":18152361166},{"left1":64,"right1":56315422789415929275018,"left2":256,"right2":5056677461204832710950,"fanout":4583522779},{"left1":4352,"right1":22481545747369052451414,"left2":4416,"right2":17808835013302183729054,"fanout":979690260960},{"left1":32768,"right1":62888013003140102783232,"left2":49216,"right2":45576438611230811983764,"fanout":128796146401},{"left1":256,"right1":19355675571867615846930,"left2":37184,"right2":26029431196304219416882,"fanout":46925622256},{"left1":16448,"right1":7347068046690943946332,"left2":20544,"right2":56349723994169748636734,"fanout":1099041868073},{"left1":49152,"right1":10030971841404771410518,"left2":49408,"right2":44244414123709336741118,"fanout":1035251732513},{"left1":32768,"right1":63524474402087731195916,"left2":49152,"right2":54547278215090615075318,"fanout":68754401032},{"left1":4096,"right1":41971990696842544288386,"left2":20480,"right2":55139638321639836414518,"fanout":138163965899},{"left1":256,"right1":72850294312311414784966,"left2":16704,"right2":37040837369272356427978,"fanout":125596942067},{"left1":256,"right1":17090135549744562474096,"left2":4096,"right2":55913849036929671136464,"fanout":116907272},{"left1":4352,"right1":19625914910923511413614,"left2":20736,"right2":29792688032166370786220,"fanout":184681482462},{"left1":4096,"right1":16105237150328618214226,"left2":32768,"right2":45381622962088481021422,"fanout":3421031415},{"left1":256,"right1":49345939874327824175346,"left2":49152,"right2":57962043999506765340668,"fanout":55166950632}],"pure":{"0":47780727771,"2":103009342404,"3":1026562271261,"4":258238827234,"6":106871724240,"8":932727151894,"12":855551056947,"15":884896094521,"32":829885518829,"48":162534846711,"64":841395473089,"112":98380137187,"128":270311486421,"160":911474751981,"176":922881437231,"192":910429009956,"272":1095186905299,"336":1014218867956,"352":194193536050,"368":927595100665,"432":1026830247710,"448":20703661792,"512":197801347523,"528":65344373282,"544":42327010082,"576":227567602183,"592":210909596893,"624":928599826958,"720":1031210274338,"736":99621932242,"800":1074745171991,"816":871025608405,"928":162709235471,"944":74073043428,"960":270215208205,"992":13939361732,"1024":1083071867371,"1040":966852669921,"1056":205692543007,"1072":966413959226,"1136":1078386154227,"1152":894074470689,"1168":1030661987039,"1216":1040089335327,"1":215234195714,"14":119992276206,"80":138095621911,"96":7782019899,"208":116066027720,"240":919493096415,"256":30681846321,"304":1073592066259,"384":120007672285,"416":38051518207,"560":1053104869652,"608":1094849280820,"656":99020037067,"672":60410623287,"704":935429337120,"752":197235706864,"768":868570116611,"832":855735863344,"848":56407175407,"864":42649055494,"896":1070390328786,"976":26476940503,"1008":833677608655,"1120":102711488207,"5":922864501200,"9":185408019737,"10":192470391076,"13":858760413455,"144":884277520955,"320":56636786237,"400":1060574254627,"464":108125933837,"640":115408387552,"688":953179050197,"1088":940026950667,"1104":124054071040,"288":202902617859,"880":235418885650,"912":22198540237,"1200":252498663480,"16":240230330840,"480":1004690528292,"496":59203975915,"1184":115597967315,"7":12565868820,"11":179501341997,"224":923330337841,"784":89875408132},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"phased_canonical_6","capacity":6,"events":[{"kind":"products","indices":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74],"required":267,"cache":267},{"kind":"pure_low","monomial":3,"required":1,"cache":267},{"kind":"pure_low","monomial":5,"required":2,"cache":267},{"kind":"pure_low","monomial":7,"required":9,"cache":267},{"kind":"pure_low","monomial":13,"required":258,"cache":267},{"kind":"products","indices":[75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149],"required":113,"cache":121},{"kind":"pure_low","monomial":9,"required":16,"cache":121},{"kind":"pure_low","monomial":10,"required":32,"cache":121},{"kind":"pure_low","monomial":11,"required":65,"cache":121},{"kind":"products","indices":[150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224],"required":1677,"cache":1677},{"kind":"pure_low","monomial":6,"required":4,"cache":1677},{"kind":"pure_low","monomial":12,"required":128,"cache":1677},{"kind":"pure_low","monomial":14,"required":516,"cache":1677},{"kind":"pure_low","monomial":15,"required":1033,"cache":1677}],"low_computations":11,"peak_live":6,"min_live_required":6}},{"batch":4,"delta_sha256":"254c5bc6f0544d9145f45180c8dc51de6a401a5ec6677278c70d6d447f2a8584","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":32,"right1":73008204310190311812240,"left2":800,"right2":71794864852562725507400,"fanout":1031512402990},{"left1":512,"right1":48185378424104968749588,"left2":4128,"right2":71504532608598656579014,"fanout":184428666382},{"left1":32,"right1":34494837907986219265424,"left2":8272,"right2":15057085080677797803572,"fanout":879731732716},{"left1":32,"right1":61847290298806174861494,"left2":40960,"right2":17576346479665030341370,"fanout":1060065693406},{"left1":8192,"right1":29648534735271228784212,"left2":8194,"right2":71743140212945673435848,"fanout":983045427254},{"left1":8,"right1":113338320777110357263894,"left2":32776,"right2":31664442763929566627114,"fanout":853859765974},{"left1":4128,"right1":128552898992809204798550,"left2":4130,"right2":117491242881931646261658,"fanout":828193430760},{"left1":256,"right1":29210499015392734575594,"left2":258,"right2":21819938087149055994064,"fanout":17431279927},{"left1":4,"right1":47353728434500560313210,"left2":72,"right2":37704114560950837579086,"fanout":226775801586},{"left1":768,"right1":30202760280762000257766,"left2":896,"right2":70029196971046886345438,"fanout":1074214215204},{"left1":32,"right1":72532599143774536029240,"left2":8240,"right2":39311552650867836674914,"fanout":957411937496},{"left1":2048,"right1":73461173731018068737112,"left2":2816,"right2":55775192351491632962870,"fanout":77492577326},{"left1":2,"right1":18602545574367402343660,"left2":18,"right2":3852912227226027143050,"fanout":149818231054},{"left1":16,"right1":74821817820218709120294,"left2":1024,"right2":15934908387060321050186,"fanout":931050171398},{"left1":8192,"right1":23889584998954442003132,"left2":16384,"right2":63627259946495711895392,"fanout":35043735294},{"left1":16,"right1":135833990604466621882416,"left2":20,"right2":119968409580291469465582,"fanout":20803605236},{"left1":4096,"right1":28789888453030858704964,"left2":4352,"right2":6864588858047850536816,"fanout":146767669502},{"left1":8,"right1":2802826434041766008774,"left2":24,"right2":15627475120434798915056,"fanout":98430926876},{"left1":8,"right1":105976732798566221142914,"left2":18432,"right2":12248980772800756076464,"fanout":267330440722},{"left1":4096,"right1":62010503642297047866140,"left2":4160,"right2":69934846247631610732644,"fanout":842552455694},{"left1":64,"right1":62932665215906892987658,"left2":1088,"right2":33392163657093018319074,"fanout":69101349408},{"left1":4096,"right1":66404462413202966188954,"left2":36,"right2":11943457481536806630716,"fanout":146481015338},{"left1":512,"right1":13907135202173450429056,"left2":4098,"right2":31576412022912151208056,"fanout":166568264438},{"left1":4128,"right1":44931637979473068098970,"left2":5152,"right2":55118280388704925652018,"fanout":132285604606},{"left1":16,"right1":5253684505306338009584,"left2":1032,"right2":65618295798218871479332,"fanout":953820310240},{"left1":8,"right1":6034210957617852745102,"left2":49160,"right2":66080242267759198995378,"fanout":957176931880},{"left1":1032,"right1":27437072086866444944082,"left2":5128,"right2":70906648663423653420396,"fanout":910984004102},{"left1":4,"right1":101451122203580390518756,"left2":20,"right2":46356141138424035063846,"fanout":86237055292},{"left1":64,"right1":38797580883677376660888,"left2":32768,"right2":5824035834068594551542,"fanout":828361722924},{"left1":128,"right1":64901898552294504266494,"left2":4096,"right2":21426929021685114561048,"fanout":157876107836},{"left1":2048,"right1":5947785119423834765000,"left2":3080,"right2":33583198090420631630498,"fanout":825308409574},{"left1":16384,"right1":20841599572395671159054,"left2":2052,"right2":68033412703785834990574,"fanout":124502417964},{"left1":1024,"right1":25720938889162205221552,"left2":1408,"right2":43191028575194774400570,"fanout":944941102090},{"left1":4096,"right1":60428458516589293294650,"left2":5120,"right2":45814254423481052187256,"fanout":103530106566},{"left1":8192,"right1":30475052481004171053670,"left2":26624,"right2":18698995724000017080598,"fanout":158376922152},{"left1":32768,"right1":74279928650573692294462,"left2":384,"right2":13384963715935163748922,"fanout":1017370640898},{"left1":64,"right1":24273941972535687237604,"left2":4098,"right2":50001857146580344870052,"fanout":232209707764},{"left1":16,"right1":2447030550254460496234,"left2":18,"right2":19326854477284114337076,"fanout":68907092770},{"left1":32768,"right1":13992651776723925774690,"left2":8208,"right2":31863895367137567403466,"fanout":163916466898},{"left1":32768,"right1":51773786185921735638632,"left2":18432,"right2":57249218916715779994604,"fanout":132989203684},{"left1":64,"right1":11568095035264023478306,"left2":1536,"right2":40485720503672677836992,"fanout":854465496800},{"left1":72,"right1":38746012230818126548696,"left2":32840,"right2":60414677379266712996116,"fanout":197705530590},{"left1":1024,"right1":31937079160851348441160,"left2":1060,"right2":61742013938110248464870,"fanout":112308141268},{"left1":512,"right1":14393564968296663398306,"left2":72,"right2":42477421086609225447286,"fanout":1065622895300},{"left1":130,"right1":37433511962204113770568,"left2":8322,"right2":53448993273251835308620,"fanout":828002847980},{"left1":16384,"right1":42064634402573181516856,"left2":80,"right2":682706292759046804140,"fanout":197636665880},{"left1":2048,"right1":32327746369631849280732,"left2":80,"right2":31529373162591118891072,"fanout":114939402438},{"left1":130,"right1":21406873912428781118934,"left2":642,"right2":26696079181781129729982,"fanout":988495014626},{"left1":2,"right1":125986664236098805674128,"left2":4098,"right2":69009478893618491798138,"fanout":842567516876},{"left1":258,"right1":149099670793549378656790,"left2":262,"right2":77303554310538272417696,"fanout":890017094909},{"left1":32768,"right1":12076942141109324106832,"left2":18432,"right2":54590837138148715820536,"fanout":72023467048},{"left1":32768,"right1":27159563902414780067768,"left2":32776,"right2":63701921177207790085548,"fanout":218690228262},{"left1":64,"right1":15303647944923710012468,"left2":4192,"right2":53057464111036662013224,"fanout":945399327276},{"left1":256,"right1":10919275281675215827704,"left2":272,"right2":23117894717621652715340,"fanout":20767903951},{"left1":256,"right1":57296050451526621012956,"left2":292,"right2":58386518562465550525960,"fanout":905848740588},{"left1":2048,"right1":65821284090107693143070,"left2":3080,"right2":14505800141326474950998,"fanout":1030962471988},{"left1":4,"right1":129108117459182313652160,"left2":49156,"right2":47324771343436881552564,"fanout":825040093726},{"left1":4,"right1":15333260091719352412808,"left2":40964,"right2":34461735219331587195320,"fanout":1025637931744},{"left1":512,"right1":46886541848010385901794,"left2":528,"right2":39499438562694683534832,"fanout":163995636418},{"left1":128,"right1":9504270492998369043958,"left2":16512,"right2":7249839705257479009616,"fanout":241205392956},{"left1":512,"right1":72089599513949312390788,"left2":2052,"right2":12646665337606107950086,"fanout":215399674916},{"left1":4,"right1":50586544050892591895050,"left2":128,"right2":69536555999015698668050,"fanout":894072704526},{"left1":2,"right1":61448098549115807195574,"left2":256,"right2":53191261663414462676616,"fanout":636737476},{"left1":1024,"right1":997035204556427257116,"left2":8192,"right2":1633891909971633165240,"fanout":78363805924},{"left1":256,"right1":6374094482232070243274,"left2":4352,"right2":46402511073121354808340,"fanout":897280827940},{"left1":32,"right1":13807938807907826105264,"left2":128,"right2":36754244407084761261342,"fanout":123785059040},{"left1":32768,"right1":39581161328008073570934,"left2":40960,"right2":24130244002589191266320,"fanout":834266212874},{"left1":256,"right1":78811199084061678314688,"left2":258,"right2":113169838361074788438498,"fanout":94573895445},{"left1":1032,"right1":262662891550805430506,"left2":1036,"right2":35025137130868883115042,"fanout":158023611442},{"left1":16384,"right1":5969097398438122989866,"left2":16640,"right2":61811876786357212541642,"fanout":215167148744},{"left1":16384,"right1":54249163833672904817182,"left2":2080,"right2":28404770603184600124696,"fanout":879867866872},{"left1":4096,"right1":63675814593287877479836,"left2":4864,"right2":50593040816474262588242,"fanout":1057435827400},{"left1":4,"right1":134317887619820503816884,"left2":18432,"right2":33752702728589175462424,"fanout":853660808706},{"left1":256,"right1":40623423910909830502170,"left2":768,"right2":31755441568887973983694,"fanout":180538501182},{"left1":2048,"right1":34993095076340441520252,"left2":4098,"right2":66006926683985394353996,"fanout":1026007160838},{"left1":2048,"right1":46890200277551112393674,"left2":18432,"right2":64996060212156265085776,"fanout":841794290},{"left1":4,"right1":827086530424419587556,"left2":4098,"right2":46111680280158437423500,"fanout":1074693995200},{"left1":128,"right1":56922199676378493612582,"left2":8336,"right2":33121271518131707425002,"fanout":123697502212},{"left1":16,"right1":39677260990024537585350,"left2":4096,"right2":73424782384134522581212,"fanout":60266913514},{"left1":1032,"right1":29324244277190693398788,"left2":1544,"right2":45552722575929483807166,"fanout":885550102736},{"left1":49152,"right1":67468337059622808551368,"left2":51200,"right2":2647250259259633994982,"fanout":132611055678},{"left1":256,"right1":29242942418314714012282,"left2":1024,"right2":67352649195289555568636,"fanout":137575532558},{"left1":2048,"right1":3122027704075319861858,"left2":6176,"right2":69956296408541020247960,"fanout":12428120316},{"left1":8192,"right1":6929411785938839778286,"left2":8224,"right2":8766495705273160195428,"fanout":896655349794},{"left1":1024,"right1":6824109931482150294136,"left2":5120,"right2":62551803102780179311960,"fanout":3323130904},{"left1":24,"right1":16037981194717315094798,"left2":4120,"right2":46226233574939900910076,"fanout":55736273428},{"left1":16,"right1":38839062854543370696792,"left2":20,"right2":54316521535656999362220,"fanout":833391477038},{"left1":4,"right1":9073910014739787001804,"left2":6,"right2":51454966122166043991268,"fanout":68836132104},{"left1":1024,"right1":47255254188312889148690,"left2":5124,"right2":12340316653673040875348,"fanout":945062421228},{"left1":64,"right1":59635461926921667222006,"left2":18432,"right2":8930040981728365933868,"fanout":128999096534},{"left1":256,"right1":4316074672738165303358,"left2":292,"right2":20836615492917568827832,"fanout":1095153278674},{"left1":1024,"right1":23623951667583688675030,"left2":2080,"right2":55278851588472265276616,"fanout":862888459492},{"left1":1024,"right1":59630892409113775114292,"left2":384,"right2":39641689452743341283270,"fanout":172102181404},{"left1":64,"right1":35153585792248204036230,"left2":49152,"right2":27872051028955012134198,"fanout":184230217932},{"left1":16384,"right1":16925919196899537871656,"left2":162,"right2":28075357177253676399954,"fanout":888357709322},{"left1":8,"right1":101810838623267595612428,"left2":2088,"right2":64781458431997554869106,"fanout":249691164380},{"left1":16,"right1":65099954270173432557070,"left2":8208,"right2":59219322274881070583908,"fanout":992018091762},{"left1":64,"right1":34436181366785756917834,"left2":384,"right2":10489968347193494958682,"fanout":1048086639822},{"left1":8,"right1":116184754802748503762626,"left2":49152,"right2":33245138447427726905228,"fanout":957408609288},{"left1":4096,"right1":13063890495047529936092,"left2":80,"right2":40166509885268495537334,"fanout":155238536250},{"left1":4,"right1":104875358842607608057938,"left2":76,"right2":6034199283564697715976,"fanout":192371159238},{"left1":130,"right1":69024038245437452366368,"left2":16514,"right2":41489017399810154232830,"fanout":948801112288},{"left1":512,"right1":37870287404853854640514,"left2":80,"right2":62849957779574160577008,"fanout":124254611172},{"left1":1536,"right1":37309479349604428985288,"left2":1792,"right2":7684401020876476537216,"fanout":884967662620},{"left1":32784,"right1":18936777768234525188382,"left2":34832,"right2":51770523697170247671526,"fanout":854162018514},{"left1":16384,"right1":11021292180608858010598,"left2":16768,"right2":75086726103567056939764,"fanout":974401356812},{"left1":4096,"right1":14465754421841772818884,"left2":4128,"right2":22274839115314339027114,"fanout":837332123386},{"left1":8,"right1":42501935998235038229892,"left2":1096,"right2":65417811500081982903600,"fanout":1077771502318},{"left1":2048,"right1":43188973575562204312072,"left2":1536,"right2":52096813198324899739158,"fanout":1065501377752},{"left1":8,"right1":54547867948191091834110,"left2":80,"right2":46616970952554073363054,"fanout":29575232050},{"left1":130,"right1":786456070492940075340,"left2":162,"right2":66632138358844171983992,"fanout":868472119014},{"left1":4,"right1":147426499764857533646696,"left2":134,"right2":32705675384681378409684,"fanout":146179410650},{"left1":128,"right1":21793852566103611523902,"left2":4098,"right2":42990037941170423668336,"fanout":931789410018},{"left1":2,"right1":112354856090449877733564,"left2":512,"right2":2681606803563780434040,"fanout":155538955278},{"left1":256,"right1":32806015387911138762022,"left2":1792,"right2":63267453022967324392392,"fanout":215755929328},{"left1":16,"right1":9852525429170290049048,"left2":32784,"right2":45174684729080254197366,"fanout":982878666244},{"left1":512,"right1":45102834187680741648230,"left2":8208,"right2":29209608303777343169666,"fanout":876185925656},{"left1":1024,"right1":44602504856691203393960,"left2":1088,"right2":10549469363487390626802,"fanout":103954568898},{"left1":32768,"right1":30339261036228780413424,"left2":32896,"right2":38639685315895334801030,"fanout":836961237546},{"left1":32,"right1":56804704772215234004944,"left2":288,"right2":70755196869335904595812,"fanout":63653729492},{"left1":32,"right1":26757950286516935822394,"left2":49184,"right2":53114265553937097537958,"fanout":922549020180},{"left1":64,"right1":14121658124860825313128,"left2":8256,"right2":51440827671315770977116,"fanout":8772188414},{"left1":256,"right1":1506527594164140658314,"left2":258,"right2":58582401499138648517260,"fanout":4645573414},{"left1":16384,"right1":23726512015115770023192,"left2":16448,"right2":46898038801137165947310,"fanout":103818514448},{"left1":16,"right1":59951231457097334986464,"left2":20,"right2":30432831734521678545008,"fanout":21878095110},{"left1":32,"right1":7565682945769975816278,"left2":1536,"right2":54492614566021175868240,"fanout":55668556992},{"left1":16384,"right1":13169422213881316740850,"left2":24576,"right2":70589918154302783694506,"fanout":26310271030},{"left1":8,"right1":99555593746747137109180,"left2":776,"right2":67577169856976833226146,"fanout":1017071011386},{"left1":32,"right1":39110046690187411172556,"left2":40960,"right2":64205122771360597088060,"fanout":200961485012},{"left1":256,"right1":20654287858833357688530,"left2":258,"right2":40877979760249260631112,"fanout":7903840537},{"left1":2,"right1":68811261359267478866446,"left2":2050,"right2":9097511585183112446248,"fanout":26287670294},{"left1":40960,"right1":14412493269005918025764,"left2":41088,"right2":10313782280260247367966,"fanout":112008240670},{"left1":16384,"right1":8221872291710774025646,"left2":49152,"right2":42179663417172299425862,"fanout":965676771886},{"left1":128,"right1":57094601496377424808932,"left2":136,"right2":67355073832437450312376,"fanout":155122719236},{"left1":49152,"right1":3655533279571266199462,"left2":49156,"right2":44532794202875627788002,"fanout":988416894162},{"left1":64,"right1":49133755288493527328138,"left2":80,"right2":47782927871978213175138,"fanout":1025692796988},{"left1":1024,"right1":10949605281067719201696,"left2":768,"right2":24695107590403016587708,"fanout":974403016922},{"left1":32,"right1":53719750159513285383356,"left2":18432,"right2":60209461631408194677588,"fanout":137652878904},{"left1":128,"right1":43293500761489774161578,"left2":2052,"right2":3545898728489013535256,"fanout":927712988688},{"left1":8192,"right1":4146986697488393825660,"left2":57344,"right2":29168813688746865101834,"fanout":902011955246},{"left1":80,"right1":29230558807636363837706,"left2":2128,"right2":70367119857447959012050,"fanout":61070509618},{"left1":64,"right1":42876390748322010686444,"left2":100,"right2":41659124361224372911206,"fanout":948429313272},{"left1":4,"right1":117834507277750009803280,"left2":16,"right2":40734398616679610631790,"fanout":206509642955},{"left1":2,"right1":37401963222633619479628,"left2":40960,"right2":47013024614443131989028,"fanout":112319851546},{"left1":8192,"right1":8428399473161560346312,"left2":640,"right2":61387990771181699060000,"fanout":129299254982},{"left1":256,"right1":15192067021960196966022,"left2":72,"right2":1379315590136564926718,"fanout":107020210426},{"left1":4,"right1":11245958125368046784642,"left2":388,"right2":33185524305487671207296,"fanout":1030907555524},{"left1":64,"right1":45803790195985455017250,"left2":194,"right2":66348216020983227964156,"fanout":172484782646},{"left1":4,"right1":19314208431241643578928,"left2":8212,"right2":18849390453087419701148,"fanout":252632377354},{"left1":1024,"right1":43380543873410543503152,"left2":4128,"right2":60406507405331684593784,"fanout":94939574816},{"left1":130,"right1":59815457230849305429764,"left2":2178,"right2":16664550435198481997782,"fanout":979703311584},{"left1":512,"right1":21202200557177503964206,"left2":4130,"right2":30119998843518595970386,"fanout":1068643700466},{"left1":512,"right1":12667321674239358142644,"left2":8192,"right2":74941950511466477305084,"fanout":836649535728},{"left1":64,"right1":65090015118717939246290,"left2":832,"right2":18536010892771710960410,"fanout":1091697509088},{"left1":2052,"right1":59230351875683189393818,"left2":40960,"right2":64129367695962844110448,"fanout":43969951790},{"left1":8,"right1":56633608330549974087780,"left2":392,"right2":48065753500502633287652,"fanout":1017047863016},{"left1":2080,"right1":52100979367292704197174,"left2":2592,"right2":53080536322592392690052,"fanout":1040201821388},{"left1":8,"right1":146409609939242355045994,"left2":128,"right2":64874540329990161204166,"fanout":223976804544},{"left1":2,"right1":32333815197745007249896,"left2":10,"right2":66424429574187176829712,"fanout":60232449744},{"left1":32,"right1":29884199492368706147548,"left2":16416,"right2":25178699913027535936384,"fanout":107240622826},{"left1":1024,"right1":41672998131802330166608,"left2":4128,"right2":16120632710040079795702,"fanout":155058501654},{"left1":8192,"right1":18843705217577177445670,"left2":26624,"right2":54598091918435261207174,"fanout":859028641502},{"left1":32768,"right1":52142672017789145252232,"left2":33024,"right2":5041465137172192554524,"fanout":253371009788},{"left1":32,"right1":20233895463224464057562,"left2":1096,"right2":28681113524539136486524,"fanout":9375846412},{"left1":128,"right1":62865815093682110346172,"left2":16384,"right2":26706771703675037095568,"fanout":51653953032},{"left1":72,"right1":72384319752936317191162,"left2":8264,"right2":71344237684819316272068,"fanout":991850841830},{"left1":16384,"right1":38490625171824867648556,"left2":40960,"right2":87626483699143334256,"fanout":971514901746},{"left1":2,"right1":75128277763152981204098,"left2":2050,"right2":33535753263449643427240,"fanout":111839821346},{"left1":4,"right1":70769340658562189046146,"left2":8208,"right2":3020185530582141390224,"fanout":198590473216},{"left1":128,"right1":40658324628097106668802,"left2":130,"right2":49545733623485356809686,"fanout":863223288338},{"left1":32,"right1":17833032861098994935512,"left2":8224,"right2":39548170894571121387762,"fanout":834008041706},{"left1":512,"right1":72056631861943597506146,"left2":2052,"right2":37439199503454433809032,"fanout":72749085236},{"left1":2,"right1":86280005936824094486106,"left2":16,"right2":36064711402676146605596,"fanout":181460799987},{"left1":64,"right1":73975426170024969127590,"left2":41024,"right2":48413265169615679385756,"fanout":1077334772458},{"left1":4,"right1":146465326108871589113020,"left2":24580,"right2":62610909084528034223414,"fanout":1043042149084},{"left1":32,"right1":46791198430245485633002,"left2":160,"right2":13748949764053693190762,"fanout":845540803296},{"left1":1032,"right1":28677180109063671306258,"left2":8208,"right2":8870503810450635832418,"fanout":845386356246},{"left1":8,"right1":113380913629612448669942,"left2":256,"right2":55471822993217732309190,"fanout":965343370258},{"left1":2048,"right1":51716640802369540568368,"left2":3584,"right2":63718803773512292012718,"fanout":956739624972},{"left1":32,"right1":64579222819654350939938,"left2":4130,"right2":34236280557016576580796,"fanout":163680940748},{"left1":2,"right1":80321619112988289250684,"left2":514,"right2":75510975002796472929206,"fanout":1040302082612},{"left1":2048,"right1":35571363684197170090660,"left2":2432,"right2":44492829969459794068510,"fanout":166746317354},{"left1":16,"right1":64925257567000043982954,"left2":256,"right2":47496556544839302752110,"fanout":140982540290},{"left1":4096,"right1":69742561637738516022964,"left2":5120,"right2":27825794572943749294662,"fanout":163963272444},{"left1":4096,"right1":19353329904488686577008,"left2":2052,"right2":31148122980732681339610,"fanout":158189757494},{"left1":1032,"right1":39558447692690399585712,"left2":1544,"right2":40367450578949915053904,"fanout":836915231464},{"left1":8192,"right1":32106345044002611564570,"left2":8228,"right2":25842250036004051597774,"fanout":966249953280},{"left1":4098,"right1":1150663217250454434954,"left2":4106,"right2":25661496631604912772574,"fanout":1095196744244},{"left1":16,"right1":55621158918876335389528,"left2":32768,"right2":53451832121832015220336,"fanout":132922134774},{"left1":128,"right1":74299483691282689132628,"left2":4224,"right2":51204864670625881340468,"fanout":876275092680},{"left1":256,"right1":24680668915824293050352,"left2":32768,"right2":57428144854319063389676,"fanout":25985082622},{"left1":2048,"right1":69140149405368562483120,"left2":8208,"right2":13901796823525930630764,"fanout":252467228876},{"left1":4,"right1":96783402440380197206914,"left2":262,"right2":821290514208747470160,"fanout":932445809615},{"left1":8,"right1":50515197991696123301232,"left2":1544,"right2":10824032647161983823094,"fanout":948597172968},{"left1":2,"right1":112874656862642341598828,"left2":256,"right2":39847611807287984003710,"fanout":867949422604},{"left1":2048,"right1":67483382921093168499532,"left2":3328,"right2":6221351843319520034498,"fanout":163291860164},{"left1":1536,"right1":144205403040841534582686,"left2":1540,"right2":99586587664363487665636,"fanout":209547751624},{"left1":80,"right1":60946331660419180848340,"left2":1104,"right2":72386608080974390890132,"fanout":189798727686},{"left1":4098,"right1":36372789872821566011964,"left2":4114,"right2":28444665668910921075034,"fanout":115196477466},{"left1":16384,"right1":27667587080756954826488,"left2":16464,"right2":14729579513309702381962,"fanout":828192073258},{"left1":16384,"right1":56982469300425440670928,"left2":49152,"right2":9244914793813083240958,"fanout":928732552944},{"left1":16,"right1":13562743519488857069836,"left2":32784,"right2":6470506910031326550602,"fanout":120340886726},{"left1":2,"right1":90389749955877215131984,"left2":514,"right2":51975459180286171484200,"fanout":149989549570}],"pure":{"0":1005357887759,"8":926759831044,"48":150759551708,"96":197768973027,"112":154706241256,"128":42919002058,"144":215249962731,"176":1065634547915,"272":829028838135,"288":184500677100,"304":47560254968,"320":209463074500,"336":894159222731,"352":52462157002,"400":33872144431,"416":1064299082708,"432":94288610582,"448":1025606754081,"464":219731315232,"480":1048054471410,"512":269892464672,"544":936353260520,"576":89940435402,"592":213873837528,"608":858570046260,"624":128629269465,"640":136602245174,"704":30718944963,"720":222922482655,"736":1077281735458,"864":1053158873544,"944":1029935056402,"960":919123193804,"976":953081259502,"1072":192501373714,"1088":939907158054,"1104":50953202991,"1152":231726652218,"1168":931603078203,"1184":1060101278708,"1200":158978796074,"1216":223471921697,"2":837722560785,"3":854161287386,"9":244323655961,"12":871780569335,"14":198242342938,"32":996954685126,"192":166735318327,"208":986919410668,"368":898283221553,"528":825560387287,"560":828914848197,"672":206877281263,"768":261023187426,"784":150377672243,"800":1047884409331,"816":73531181814,"928":231375423764,"1024":33893135047,"1040":253319431646,"1120":829195687395,"1":176442100263,"11":266288378052,"13":833572438770,"15":243808135728,"64":969939877159,"80":227803711726,"160":1043796987083,"256":1064313494012,"656":91085457192,"688":150090280389,"752":1068776998166,"848":1018811435256,"896":85145420508,"912":68049518069,"1008":881052823497,"1056":1072789062460,"1136":150640922663,"5":240551257883,"6":1009750649151,"10":227217246684,"16":239897617386,"224":1087177298997,"7":241405053658,"384":149505757419,"496":18233431357,"832":1021913794331,"880":111751269907,"992":270704296208,"240":253555110680,"4":137607120700},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"three-phase","capacity":8,"product_order":[0,1,6,7,8,9,11,12,13,15,16,17,19,20,21,22,23,24,26,27,29,30,32,33,36,37,40,42,43,46,47,48,49,52,53,54,55,58,60,61,62,64,65,67,68,71,73,74,76,78,79,81,82,84,85,86,87,88,90,91,92,95,97,99,100,102,103,106,107,108,109,110,111,112,113,114,117,119,122,124,125,127,129,130,133,135,136,138,140,141,142,145,146,147,149,150,151,153,155,156,157,158,160,163,167,169,171,172,175,177,178,179,180,181,182,183,184,185,187,189,192,193,194,195,196,197,198,202,2,4,10,44,63,77,83,96,116,121,144,148,152,165,168,170,176,186,191,3,5,14,18,25,28,31,34,35,38,39,41,45,50,51,56,57,59,66,69,70,72,75,80,89,93,94,98,101,104,105,115,118,120,123,126,128,131,132,134,137,139,143,154,159,161,162,164,166,173,174,188,190,199,200,201],"events":[{"kind":"products","cache":255,"indices":[0,1,6,7,8,9,11,12,13,15,16,17,19,20,21,22,23,24,26,27,29,30,32,33,36,37,40,42,43,46,47,48,49,52,53,54,55,58,60,61,62,64,65,67,68,71,73,74,76,78,79,81,82,84,85,86,87,88,90,91,92,95,97,99,100,102,103,106,107,108,109,110,111,112,113,114,117,119,122,124,125,127,129,130,133,135,136,138,140,141,142,145,146,147,149,150,151,153,155,156,157,158,160,163,167,169,171,172,175,177,178,179,180,181,182,183,184,185,187,189,192,193,194,195,196,197,198,202]},{"kind":"pure_low","cache":255,"monomial":3},{"kind":"pure_low","cache":255,"monomial":9},{"kind":"pure_low","cache":255,"monomial":12},{"kind":"pure_low","cache":255,"monomial":11},{"kind":"pure_low","cache":255,"monomial":5},{"kind":"pure_low","cache":255,"monomial":6},{"kind":"pure_low","cache":255,"monomial":10},{"kind":"pure_low","cache":255,"monomial":7},{"kind":"products","cache":383,"indices":[2,4,10,44,63,77,83,96,116,121,144,148,152,165,168,170,176,186,191]},{"kind":"pure_low","cache":383,"monomial":13},{"kind":"products","cache":1871,"indices":[3,5,14,18,25,28,31,34,35,38,39,41,45,50,51,56,57,59,66,69,70,72,75,80,89,93,94,98,101,104,105,115,118,120,123,126,128,131,132,134,137,139,143,154,159,161,162,164,166,173,174,188,190,199,200,201]},{"kind":"pure_low","cache":1871,"monomial":14},{"kind":"pure_low","cache":1871,"monomial":15}],"low_computations":11}},{"batch":5,"delta_sha256":"35e6c827c26850cfff1e8a595c609eed94e68eb41a74a89d610c7dce6ede622a","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":2048,"right1":32568370668406444107106,"left2":160,"right2":41567627217963576923008,"fanout":897449778408},{"left1":16384,"right1":19304833840943546203788,"left2":8704,"right2":55536782722975951292044,"fanout":223473886432},{"left1":128,"right1":129692588542817527525848,"left2":1280,"right2":63242138442339951799778,"fanout":226976128028},{"left1":544,"right1":149780651957638839693130,"left2":4640,"right2":137380792319980883282564,"fanout":945712322620},{"left1":4098,"right1":108211717079014471303770,"left2":4106,"right2":19250451772118477328810,"fanout":175505946876},{"left1":2,"right1":117821068092884787063798,"left2":18,"right2":42146424239393607111854,"fanout":26118390252},{"left1":2,"right1":140137129334847014687438,"left2":6,"right2":134166286007780942268864,"fanout":39311830737},{"left1":4,"right1":5731515365071435840212,"left2":16,"right2":25289731677481788364564,"fanout":159484790806},{"left1":2,"right1":134973356363437594049640,"left2":18,"right2":22306337523482871328696,"fanout":73751783399},{"left1":128,"right1":61102290324032759861102,"left2":2048,"right2":99768885136900741444464,"fanout":52157278412},{"left1":32,"right1":95271742322780207059984,"left2":6144,"right2":36447711686686248043498,"fanout":859341905136},{"left1":128,"right1":7458126142294740572666,"left2":32864,"right2":55986849795871421596018,"fanout":931419180276},{"left1":544,"right1":17888476312206290586722,"left2":32832,"right2":74784333431012431456322,"fanout":894372688940},{"left1":2048,"right1":84330057739539962490282,"left2":8704,"right2":24494243341285332416514,"fanout":948720383228},{"left1":512,"right1":17337364410515872548768,"left2":4096,"right2":138125960014032345071742,"fanout":1051263503576},{"left1":32,"right1":76792400418627157248262,"left2":2060,"right2":20766404754383599679726,"fanout":192736713788},{"left1":8,"right1":32093916226846626284186,"left2":33792,"right2":38841393065158966437342,"fanout":999855283248},{"left1":8,"right1":88221196364614274774426,"left2":34048,"right2":60090834817209011631094,"fanout":223405732868},{"left1":96,"right1":74207042120988171811204,"left2":33792,"right2":61769342017738947438296,"fanout":140676697296},{"left1":1024,"right1":12106133702400094186724,"left2":1280,"right2":20919852591365434110924,"fanout":879798571232},{"left1":8192,"right1":23113331271582715912072,"left2":10244,"right2":72192287866198013603844,"fanout":1034284748020},{"left1":544,"right1":3642799408545230744468,"left2":33312,"right2":60642130691241134699528,"fanout":227502777352},{"left1":16,"right1":29769809495411029082496,"left2":18,"right2":39606436987840220015156,"fanout":22131372002},{"left1":12,"right1":16568278863750891672792,"left2":8224,"right2":25166194591251031577790,"fanout":55377658892},{"left1":8192,"right1":60025874080590082414556,"left2":2060,"right2":53747505388691878847728,"fanout":997183247572},{"left1":34,"right1":72317876679121771027416,"left2":528,"right2":12051478492629052319266,"fanout":72950799584},{"left1":32,"right1":119235591117291288180900,"left2":6,"right2":73588918731299651937502,"fanout":1086492779716},{"left1":1024,"right1":8284171273825029613492,"left2":33792,"right2":5845544121071153128416,"fanout":910884135996},{"left1":512,"right1":58714008481437893513212,"left2":33856,"right2":56333451090833030888374,"fanout":20471623708},{"left1":32,"right1":141288614148839996366086,"left2":20480,"right2":36436387144898997741564,"fanout":35181362368},{"left1":8192,"right1":67858367756715351666744,"left2":28672,"right2":4288174959331894681230,"fanout":175169127672},{"left1":8224,"right1":70608126828295564535154,"left2":8480,"right2":61625404820295098775604,"fanout":175369935912},{"left1":96,"right1":3052803888715072114682,"left2":33856,"right2":36337921403659894275416,"fanout":962613006352},{"left1":16,"right1":100006168716565115557192,"left2":18,"right2":133070744355077431291698,"fanout":859953422612},{"left1":4128,"right1":98608087525450014463388,"left2":20480,"right2":25827100272496719782674,"fanout":175826332700},{"left1":8,"right1":16855464078539211245946,"left2":34048,"right2":26914960732610360154752,"fanout":54842636348},{"left1":2056,"right1":54771474503240040391190,"left2":3080,"right2":71876160636178159394434,"fanout":69607946240},{"left1":136,"right1":136126531395238612710640,"left2":648,"right2":50074725905903238051366,"fanout":982944897260},{"left1":16,"right1":61967078299656561658514,"left2":256,"right2":62220448353421147227730,"fanout":851396724953},{"left1":8,"right1":99154101075173841817496,"left2":528,"right2":52419963521474176135260,"fanout":1083149968620},{"left1":4,"right1":4157240219516741672262,"left2":4096,"right2":128472552327641558638012,"fanout":1052066713608},{"left1":8,"right1":19205899581885354207440,"left2":8720,"right2":46497429434777698301098,"fanout":825641456872},{"left1":96,"right1":67221860377039194186490,"left2":1216,"right2":28827776048813443185692,"fanout":1069260530748},{"left1":96,"right1":35907754072081794880506,"left2":384,"right2":15516390518696072938130,"fanout":69258716204},{"left1":32800,"right1":45253294910348384637820,"left2":33312,"right2":27552518161902330507824,"fanout":877113433144},{"left1":96,"right1":24896865847983132064352,"left2":4130,"right2":56101851488163191687082,"fanout":996716848180},{"left1":32768,"right1":59535944724302340338518,"left2":32896,"right2":23726927972728209151030,"fanout":17730381824},{"left1":96,"right1":17090819696805201802504,"left2":168,"right2":46846907478894926678898,"fanout":106583303404},{"left1":2,"right1":82101564919344522469762,"left2":4,"right2":138183550710314551163076,"fanout":141132369942},{"left1":64,"right1":142058416493625343222346,"left2":2120,"right2":87498908433168132852902,"fanout":17596729576},{"left1":64,"right1":18192147752870471234154,"left2":20480,"right2":23688653407042969737122,"fanout":121269185744},{"left1":128,"right1":105419016650606838466860,"left2":32832,"right2":16220313386982013932740,"fanout":140997875772},{"left1":32,"right1":32183265704544106878620,"left2":24592,"right2":71683853589013408346132,"fanout":1031546876152},{"left1":256,"right1":150094100076198691444882,"left2":258,"right2":123593047039256492597736,"fanout":236495369676},{"left1":40,"right1":103043828116955440438700,"left2":1312,"right2":21125418933451113451052,"fanout":120876972036},{"left1":256,"right1":106578817741219756128558,"left2":384,"right2":121416102868540148299534,"fanout":261070053608},{"left1":1024,"right1":72047149118016407281974,"left2":33024,"right2":53323388598685369511038,"fanout":17798280392},{"left1":1024,"right1":44458407653612549253104,"left2":20480,"right2":72788226334662342385186,"fanout":863170732080},{"left1":2048,"right1":8678807090103523516604,"left2":32832,"right2":61450622982805953225268,"fanout":172616583392},{"left1":2052,"right1":98050677386492072995084,"left2":2060,"right2":39347848639836584150950,"fanout":4242589720},{"left1":8208,"right1":91526447258265574518044,"left2":8216,"right2":141846757737107418016594,"fanout":880414870732},{"left1":128,"right1":72977237324295956559336,"left2":448,"right2":74287277235161144746108,"fanout":244292784376},{"left1":2050,"right1":99412228947679991618476,"left2":2082,"right2":62199795564675321425298,"fanout":965630236708},{"left1":12,"right1":134303435024485807153758,"left2":8204,"right2":17482433438321669326398,"fanout":928063684852},{"left1":4,"right1":68431976483469178128282,"left2":6,"right2":26672325328530048511412,"fanout":76757342471},{"left1":1024,"right1":67539780881846482937048,"left2":16384,"right2":46791342873190439686854,"fanout":90007205904},{"left1":4128,"right1":2806902917017110273674,"left2":12800,"right2":59157498285778056961620,"fanout":1086305658940},{"left1":32,"right1":47092731666880957970032,"left2":2052,"right2":136107788735145586428892,"fanout":845639121976},{"left1":16,"right1":54256690829882954644092,"left2":20,"right2":1901907024448476001082,"fanout":846173887501},{"left1":512,"right1":11936365887420011882134,"left2":32832,"right2":49873532666004205748450,"fanout":21076164644},{"left1":2,"right1":98534606416038540846996,"left2":258,"right2":17670494802271031636540,"fanout":39140791788},{"left1":64,"right1":75333024776844225530914,"left2":25088,"right2":27334597836634332439980,"fanout":913837715668},{"left1":8704,"right1":3466672988601455517348,"left2":8960,"right2":14647066605561223473540,"fanout":894173709520},{"left1":4,"right1":2568453047004451744858,"left2":8192,"right2":22682053098814618125882,"fanout":879679562944},{"left1":128,"right1":83957172663083939901508,"left2":32832,"right2":51585281035188282772020,"fanout":158175864000},{"left1":8192,"right1":69271229415219899247804,"left2":44,"right2":27828526868267739384404,"fanout":21277450468},{"left1":8224,"right1":24967865498641860176246,"left2":20480,"right2":13934812045563857119710,"fanout":1017584295968},{"left1":256,"right1":92811347939735051594154,"left2":260,"right2":105889755969212551222714,"fanout":228056822795},{"left1":8224,"right1":50261213298553727741188,"left2":24608,"right2":23905421087984312988078,"fanout":86784614592},{"left1":33024,"right1":57780671576400246144192,"left2":33152,"right2":62454632710747000468468,"fanout":928195291152},{"left1":1344,"right1":67592422169829738951716,"left2":34048,"right2":17846384535861686144482,"fanout":55580048608},{"left1":4,"right1":144807376683867429585418,"left2":16,"right2":691978880687228535280,"fanout":860037389851},{"left1":2,"right1":143009544374581442009520,"left2":16912,"right2":59637301762608006488736,"fanout":210052114640},{"left1":16,"right1":11528090746331606179664,"left2":256,"right2":9651276079962910288610,"fanout":39727076650},{"left1":16,"right1":10972276060495216535464,"left2":6,"right2":103390379016716714751134,"fanout":93535216857},{"left1":1024,"right1":45496409248652492870868,"left2":5122,"right2":19281834203249117357384,"fanout":72009171172},{"left1":8192,"right1":29646709514265934271160,"left2":24576,"right2":30076454077694784541466,"fanout":828810511420},{"left1":8192,"right1":13427596119362490601964,"left2":16896,"right2":75044797028252575972412,"fanout":52010430504},{"left1":4,"right1":138412842611660788309518,"left2":20,"right2":57663245678649147578600,"fanout":38433385483},{"left1":32,"right1":116726102333074747937094,"left2":1152,"right2":46629736833369001366296,"fanout":35101918208},{"left1":4096,"right1":76252687404810922599110,"left2":12800,"right2":51365753727885156288236,"fanout":962411628772},{"left1":1280,"right1":122223963884072660548444,"left2":5376,"right2":113475262111476610866730,"fanout":137787661528},{"left1":32,"right1":148593219553319501760448,"left2":2060,"right2":68339946922721485184280,"fanout":120341663964},{"left1":8,"right1":52527330512052880446072,"left2":16424,"right2":36956121298735395292162,"fanout":910883828752},{"left1":16384,"right1":24060845077465980967254,"left2":16392,"right2":39637632848777519171764,"fanout":207045788696},{"left1":8224,"right1":37849851017375294188598,"left2":8204,"right2":14713547153716841803724,"fanout":914572501220},{"left1":64,"right1":8608361690481262746308,"left2":1152,"right2":30447461465620850685530,"fanout":894305884204},{"left1":32,"right1":34727893513064690825726,"left2":6144,"right2":108644155489204076508104,"fanout":845721166888},{"left1":1024,"right1":65298300844512839418566,"left2":32864,"right2":40041431472734517271484,"fanout":962881969360},{"left1":4098,"right1":44014091765388799820236,"left2":8224,"right2":41838511657482435148606,"fanout":172485573868},{"left1":256,"right1":5095793068845312412098,"left2":260,"right2":12602652762852716161622,"fanout":167568090907},{"left1":32,"right1":36773796923195232266822,"left2":2082,"right2":89343870675235358556738,"fanout":38469431544},{"left1":128,"right1":80921521789360635914520,"left2":528,"right2":37349361150040848439990,"fanout":193020039400},{"left1":64,"right1":9289661537503611855448,"left2":34048,"right2":7063716149791612067224,"fanout":193020568788},{"left1":96,"right1":14902317260194502253756,"left2":1344,"right2":19944548517444094908240,"fanout":928264806592},{"left1":128,"right1":125312667589983412817378,"left2":12,"right2":100296655127250126176822,"fanout":877113697312},{"left1":32832,"right1":80887751283165205369502,"left2":34880,"right2":98208609047009197452090,"fanout":1082805516516},{"left1":4098,"right1":117652797817161475442024,"left2":4130,"right2":97172878821436162325354,"fanout":223878889512},{"left1":8192,"right1":73377191521886358656556,"left2":16384,"right2":27503359663614859192628,"fanout":224278100160},{"left1":12,"right1":89905516699942943066084,"left2":96,"right2":7948776253505099490966,"fanout":982957964540},{"left1":64,"right1":64221517210608747739720,"left2":4098,"right2":44646412829270104018672,"fanout":141603693620},{"left1":16416,"right1":46716359299584159064836,"left2":24608,"right2":69528112355759626316466,"fanout":4176493608},{"left1":96,"right1":46428277223620296029592,"left2":1280,"right2":21858980637702093797640,"fanout":1014085245132},{"left1":4096,"right1":70721515822749483590464,"left2":6144,"right2":55756235599396653122598,"fanout":240600545484},{"left1":16,"right1":37703622141353131019718,"left2":272,"right2":1452965331134497031268,"fanout":17366382792},{"left1":256,"right1":77728782595083741309160,"left2":258,"right2":88413417411865898061434,"fanout":841528448255},{"left1":2048,"right1":8879620524602233045094,"left2":560,"right2":43563490019455837253472,"fanout":89268743392},{"left1":20480,"right1":35741148148425560351752,"left2":28672,"right2":2993865537010353837490,"fanout":879743785188},{"left1":8704,"right1":42255610684741045553918,"left2":25088,"right2":74553221824312952004334,"fanout":243942347972},{"left1":20480,"right1":3208497585688606412020,"left2":28672,"right2":1307961025411885731858,"fanout":928518554864},{"left1":320,"right1":52415115089162425786298,"left2":352,"right2":22814066039258499788380,"fanout":828272604392},{"left1":128,"right1":140918998348809813139276,"left2":32896,"right2":9632566588462026225866,"fanout":1017105094856},{"left1":32,"right1":95807864814485575455444,"left2":14,"right2":78101713777473616381044,"fanout":72478886936},{"left1":8,"right1":59385485858203394277946,"left2":2056,"right2":12629245136285809858948,"fanout":1048106773728},{"left1":16384,"right1":65002369950955297318114,"left2":544,"right2":7967701365452615517426,"fanout":257846987792},{"left1":128,"right1":52810030093748176992902,"left2":32768,"right2":60448404515946168597046,"fanout":103281335504},{"left1":8192,"right1":94013132478002976118360,"left2":10244,"right2":87819432707581016580528,"fanout":141278111996},{"left1":8704,"right1":10188891247796181381880,"left2":12800,"right2":53593008213292304495742,"fanout":928464772348},{"left1":64,"right1":5410029088948400038176,"left2":2114,"right2":55164587861499985492856,"fanout":3371698176},{"left1":16,"right1":62907408719060911269016,"left2":4112,"right2":54639441557757468968522,"fanout":893489518816},{"left1":2048,"right1":54120383684857101310662,"left2":32864,"right2":52864915715640031018922,"fanout":17248872688},{"left1":1024,"right1":63282443770505346660480,"left2":32832,"right2":16645175505486193317428,"fanout":1065288211456},{"left1":32,"right1":17143564771643233258118,"left2":16432,"right2":67910054967950709816004,"fanout":206911574020},{"left1":8,"right1":140225292824613224059606,"left2":33024,"right2":60571842778200528467214,"fanout":3491022884},{"left1":16,"right1":43596040801074354099780,"left2":20,"right2":22114184212233707638294,"fanout":35065310219},{"left1":32,"right1":69501460634531898013764,"left2":25088,"right2":72370034339016767911878,"fanout":928252744928},{"left1":1024,"right1":21368703644332019525884,"left2":1536,"right2":46297763697782503628686,"fanout":55714313432},{"left1":136,"right1":107537634002988992636372,"left2":2060,"right2":1438265507083326174646,"fanout":217584680},{"left1":4,"right1":100597446139320735208496,"left2":256,"right2":51447548973055868853706,"fanout":3821603610},{"left1":4096,"right1":8567985760610438907030,"left2":48,"right2":55683779125537382025386,"fanout":244143412472},{"left1":2050,"right1":7894183178738766966432,"left2":2058,"right2":118248720600681636299838,"fanout":1031129530620},{"left1":8224,"right1":30831961743225335451710,"left2":560,"right2":32584743066298624352844,"fanout":193140369416},{"left1":1024,"right1":10162635588667648569430,"left2":4096,"right2":30952554510613083727732,"fanout":257765160140},{"left1":16,"right1":571730504473304685590,"left2":20,"right2":61113055217294308169434,"fanout":206611880899},{"left1":16384,"right1":148199581798923213113786,"left2":16392,"right2":112087982781527968736220,"fanout":138391595204},{"left1":2,"right1":60592956264870750156236,"left2":4098,"right2":70128667948087645519876,"fanout":876994952200},{"left1":512,"right1":100965962911558422610,"left2":136,"right2":62898931484152834836580,"fanout":172201396232},{"left1":544,"right1":43099374501644074297100,"left2":2560,"right2":68766247697697845371958,"fanout":949070070792},{"left1":2048,"right1":75393157360192613301916,"left2":2576,"right2":7756711871012289588030,"fanout":18054382812},{"left1":16,"right1":18538591934955593740682,"left2":272,"right2":71948153030693811939578,"fanout":107237798363},{"left1":2,"right1":114435704158284728833926,"left2":256,"right2":3052250893596393464736,"fanout":3592475423},{"left1":1088,"right1":126202150256783545966464,"left2":1092,"right2":85879842423957920626456,"fanout":754509004},{"left1":4,"right1":69350770023588025793812,"left2":6,"right2":4337796433876375932834,"fanout":30454568144},{"left1":32,"right1":96573857091683509473962,"left2":16432,"right2":49022913354086146361996,"fanout":172484261056},{"left1":16,"right1":81739689011068530961784,"left2":18,"right2":112668539952579006249836,"fanout":18140220964},{"left1":2,"right1":23036002381395403899466,"left2":258,"right2":7869165060669657505380,"fanout":984319335617},{"left1":16,"right1":30963496749314124537964,"left2":272,"right2":3057281598470491883224,"fanout":905333825780},{"left1":64,"right1":34025036587545403529060,"left2":1344,"right2":68101035183785210058174,"fanout":1000474421448},{"left1":2,"right1":45522839371958025830662,"left2":6,"right2":111702521167467271900852,"fanout":48032566785},{"left1":32,"right1":128680251279574699029308,"left2":1152,"right2":74154460264579478779852,"fanout":138257903660},{"left1":64,"right1":95052354550623416559952,"left2":196,"right2":142692741460145698736498,"fanout":982880149708},{"left1":8208,"right1":31139229099594231268882,"left2":8720,"right2":48107454508517789858936,"fanout":37583324404},{"left1":528,"right1":94528632037438111485242,"left2":536,"right2":130767785677563177485548,"fanout":876176484372},{"left1":96,"right1":39080789340955350686902,"left2":4162,"right2":51930228210049004938306,"fanout":172271861760},{"left1":64,"right1":72572753550921584639226,"left2":1280,"right2":53088700877779909095580,"fanout":825104540716},{"left1":32832,"right1":19243924075497127973708,"left2":34048,"right2":56748281526940491842908,"fanout":876592547064},{"left1":2,"right1":45087095483679751306656,"left2":1026,"right2":51851914082471348472726,"fanout":209997019388},{"left1":2050,"right1":120186445534299610398772,"left2":2082,"right2":50769872987763096608858,"fanout":72547581160},{"left1":20480,"right1":126626623752629302771384,"left2":20512,"right2":76847530536996957598494,"fanout":825318716668},{"left1":128,"right1":35149363130742242367802,"left2":140,"right2":44026440134920868477320,"fanout":20602685484},{"left1":4,"right1":145012584304845036213928,"left2":256,"right2":54952695944284530189146,"fanout":158440030002},{"left1":2,"right1":145808309476807939301042,"left2":6,"right2":12010827405092311732692,"fanout":967024324877},{"left1":32,"right1":110346402050469839160810,"left2":16912,"right2":59297478574147692954162,"fanout":913824290004},{"left1":2052,"right1":20695019829683223510732,"left2":2084,"right2":44323818147717552509588,"fanout":155169117376},{"left1":1024,"right1":88305793317333726142608,"left2":5120,"right2":140859711756286334354012,"fanout":931001540632},{"left1":32768,"right1":63715971520535761751696,"left2":32784,"right2":32789678030965565333536,"fanout":226575566892},{"left1":4,"right1":22915855289457159008456,"left2":6,"right2":115420942273784569986068,"fanout":81972880398},{"left1":16,"right1":72991169279481802574264,"left2":256,"right2":26916874323311021008914,"fanout":209798498326},{"left1":16,"right1":71978641602345329938796,"left2":152,"right2":11381979030407186682480,"fanout":240735497252},{"left1":33024,"right1":54375086006029904438618,"left2":35072,"right2":36489342374675954931312,"fanout":137977981144}],"pure":{"0":262345717501,"1":867295494932,"4":180068999873,"8":1082852971980,"32":931706826791,"64":172818445590,"128":1026064186904,"192":825639832812,"208":5068630216,"304":1042840023506,"400":928229367761,"432":13787847415,"448":201241186519,"480":167804591394,"496":185018627872,"544":969687697375,"576":26756376378,"608":1061106296522,"672":1034268427786,"720":838562017058,"752":827921803042,"768":907079109596,"784":1013107919568,"800":3386517015,"832":1061565552099,"848":140876367044,"896":1001436034334,"912":893322928099,"960":112488537376,"976":969836519392,"1008":1027249605337,"1040":99688825354,"1056":1077480536356,"1088":137355642141,"1104":1086864960012,"1152":86050468307,"1216":159752256046,"2":86283703800,"48":55680635854,"80":185068288512,"112":20548093654,"144":205686958560,"224":43802741218,"240":1003980320825,"256":232332929337,"272":1086944704559,"320":885128357602,"352":954518746377,"368":134100429526,"384":945230247197,"416":1095906417115,"560":914895205387,"592":223725801727,"624":845321063405,"640":257380654800,"688":59978038064,"704":21792691447,"944":266273041875,"1120":859844974812,"1168":198459068355,"3":215081564382,"5":1078368460528,"9":897041383373,"10":939573316364,"14":140809916612,"15":962877983744,"16":982790160669,"160":871979426319,"288":832800764874,"336":201578462715,"464":202687112505,"512":1010203632369,"528":176109327137,"816":24951049271,"992":265616805433,"1024":997253586221,"7":106516189192,"13":86906825976,"176":115210317600,"656":252649207612,"864":256728826911,"880":133845420859,"928":829050193682,"1136":223707384604,"1200":192414494519,"12":918219995174,"736":197236176612,"6":829451730182,"96":1051273059298,"1072":1082132006095,"1184":68973113086,"11":241260307508},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"three-phase","capacity":8,"product_order":[1,3,4,5,6,7,8,10,13,14,15,20,22,23,24,25,26,29,30,31,33,34,38,39,40,41,45,48,49,50,52,53,59,60,62,63,64,66,67,68,70,71,72,73,75,76,77,78,81,82,83,84,86,87,88,90,92,93,94,95,97,99,100,101,107,108,109,110,111,113,114,115,116,117,118,119,120,122,123,124,126,127,128,129,132,134,135,138,139,140,141,143,144,145,147,148,149,150,152,153,154,155,156,158,161,162,163,167,168,170,171,172,173,176,177,19,36,54,57,65,85,91,104,112,136,142,151,157,164,166,174,0,2,9,11,12,16,17,18,21,27,28,32,35,37,42,43,44,46,47,51,55,56,58,61,69,74,79,80,89,96,98,102,103,105,106,121,125,130,131,133,137,146,159,160,165,169,175,178,179],"events":[{"kind":"products","cache":983,"indices":[1,3,4,5,6,7,8,10,13,14,15,20,22,23,24,25,26,29,30,31,33,34,38,39,40,41,45,48,49,50,52,53,59,60,62,63,64,66,67,68,70,71,72,73,75,76,77,78,81,82,83,84,86,87,88,90,92,93,94,95,97,99,100,101,107,108,109,110,111,113,114,115,116,117,118,119,120,122,123,124,126,127,128,129,132,134,135,138,139,140,141,143,144,145,147,148,149,150,152,153,154,155,156,158,161,162,163,167,168,170,171,172,173,176,177]},{"kind":"pure_low","cache":983,"monomial":3},{"kind":"pure_low","cache":983,"monomial":5},{"kind":"pure_low","cache":983,"monomial":9},{"kind":"pure_low","cache":983,"monomial":14},{"kind":"pure_low","cache":983,"monomial":13},{"kind":"pure_low","cache":983,"monomial":12},{"kind":"pure_low","cache":983,"monomial":6},{"kind":"pure_low","cache":983,"monomial":11},{"kind":"products","cache":759,"indices":[19,36,54,57,65,85,91,104,112,136,142,151,157,164,166,174]},{"kind":"pure_low","cache":759,"monomial":10},{"kind":"products","cache":1151,"indices":[0,2,9,11,12,16,17,18,21,27,28,32,35,37,42,43,44,46,47,51,55,56,58,61,69,74,79,80,89,96,98,102,103,105,106,121,125,130,131,133,137,146,159,160,165,169,175,178,179]},{"kind":"pure_low","cache":1151,"monomial":15},{"kind":"pure_low","cache":1151,"monomial":7}],"low_computations":11}},{"batch":6,"delta_sha256":"343f21c0658f4e4cdcb6f9a94f90284cc47eeb225c54d1358081238cc534344f","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":2,"right1":79234153582954701889970,"left2":260,"right2":12348602934848002576150,"fanout":133142672603},{"left1":512,"right1":19029924697365104968674,"left2":12544,"right2":53420458989992947230126,"fanout":215767642154},{"left1":128,"right1":13250134152317732324442,"left2":16400,"right2":47770093881588494572354,"fanout":206697059370},{"left1":4,"right1":53842572387376290037478,"left2":68,"right2":349581171163184301390,"fanout":879496930868},{"left1":128,"right1":143914649228089654536048,"left2":168,"right2":96746885506464028657376,"fanout":1085937876172},{"left1":8,"right1":124295022701187797969394,"left2":34832,"right2":37554761312909491975812,"fanout":1000291708654},{"left1":34816,"right1":27434898621942092388614,"left2":34880,"right2":26380198118783878910286,"fanout":1065973854954},{"left1":1024,"right1":9286342050964851831080,"left2":16448,"right2":64819356079188531343864,"fanout":128863492846},{"left1":32,"right1":12311299100163975806332,"left2":292,"right2":36175638439280556610962,"fanout":1057369756390},{"left1":512,"right1":75557125685886697136644,"left2":34,"right2":31358002227219782897086,"fanout":38521071632},{"left1":1024,"right1":72646886732068938034498,"left2":33808,"right2":12481791209474532172492,"fanout":253034689248},{"left1":1536,"right1":63823237076440110094872,"left2":1540,"right2":68110318636524579575192,"fanout":89758286850},{"left1":128,"right1":48278734395692810172646,"left2":8320,"right2":68567673469422619776142,"fanout":78149326572},{"left1":8,"right1":77395614084128096034166,"left2":2056,"right2":51580237419915269986322,"fanout":862331393062},{"left1":32768,"right1":30552504067557977067272,"left2":33024,"right2":1186854903818966622252,"fanout":894061127416},{"left1":2,"right1":1943517553724764761994,"left2":70,"right2":19265372998734687930310,"fanout":189599065830},{"left1":32784,"right1":32501688525811299452008,"left2":49168,"right2":43518421694536281939408,"fanout":189262726862},{"left1":8192,"right1":37877581734332133197300,"left2":40,"right2":81066106844398741681416,"fanout":43366758956},{"left1":1024,"right1":69983837184080585354714,"left2":3072,"right2":60479405001381868556140,"fanout":249914336266},{"left1":4096,"right1":118034637304486836154846,"left2":4128,"right2":77088835667055975821158,"fanout":979436750582},{"left1":16384,"right1":14219592396671079656508,"left2":16452,"right2":19490596507953370574684,"fanout":911251866644},{"left1":1152,"right1":11185003249310828133404,"left2":1184,"right2":73774186591008378807148,"fanout":20717162542},{"left1":1024,"right1":42941899891089855535940,"left2":12288,"right2":57970487075812772881894,"fanout":112052199164},{"left1":64,"right1":4964273708481844916004,"left2":192,"right2":46520507750640623236470,"fanout":999803720742},{"left1":34816,"right1":74339703440563064092290,"left2":35072,"right2":62444772035999741765838,"fanout":197937736424},{"left1":4096,"right1":18211023938906649932090,"left2":8192,"right2":4338958180717975814150,"fanout":51991687216},{"left1":4096,"right1":52421097686257587099746,"left2":34,"right2":28610905752968707018632,"fanout":30000862928},{"left1":16,"right1":14315522764050131357692,"left2":32784,"right2":1914296900740847499504,"fanout":207067083788},{"left1":8,"right1":103383466235333227087350,"left2":4096,"right2":7106824382897078153716,"fanout":29912398394},{"left1":256,"right1":12941711748033177128704,"left2":12288,"right2":60463497995991234892268,"fanout":158514466014},{"left1":512,"right1":53671877925653453126902,"left2":16912,"right2":22261371357540505117400,"fanout":982755968014},{"left1":2,"right1":13879016825613110620078,"left2":4,"right2":141343268722749803407214,"fanout":206565264899},{"left1":32768,"right1":67033406559735867063516,"left2":32896,"right2":47680420278879942308380,"fanout":962745614034},{"left1":68,"right1":58769878860954169142466,"left2":12288,"right2":32661000588893207646556,"fanout":888000005350},{"left1":32768,"right1":15715707160522398021282,"left2":68,"right2":23117134459243793862660,"fanout":197940610288},{"left1":512,"right1":2251801335985031082648,"left2":640,"right2":43283454142350731410694,"fanout":86338836726},{"left1":12288,"right1":3118483707387859907168,"left2":12800,"right2":34177548188562850165064,"fanout":1074448616454},{"left1":8,"right1":126052498696054647243914,"left2":8704,"right2":36181417511880387996888,"fanout":55646546486},{"left1":512,"right1":7687110573549256044296,"left2":8448,"right2":61215591267319390029038,"fanout":197917547530},{"left1":2,"right1":100363223145059816546740,"left2":64,"right2":12235369391358919856900,"fanout":8839024322},{"left1":2,"right1":120103035554092941015590,"left2":640,"right2":13342719458839510290712,"fanout":64404772578},{"left1":32,"right1":130742339557640574332316,"left2":128,"right2":58649333218018029805670,"fanout":11892309196},{"left1":2,"right1":144044953298742017875692,"left2":16450,"right2":64695834449944107395720,"fanout":834079085774},{"left1":2056,"right1":8021660167973294934270,"left2":2072,"right2":2388425025042223194606,"fanout":81236198944},{"left1":4,"right1":138411509696147540771740,"left2":256,"right2":25188241137008179944936,"fanout":77845351162},{"left1":4096,"right1":21471205419825197920042,"left2":5120,"right2":3540359038799672433666,"fanout":158480478254},{"left1":8192,"right1":46779242495684974988438,"left2":68,"right2":41415210053812191911282,"fanout":218992143916},{"left1":2048,"right1":20262080105262011681288,"left2":3074,"right2":3759887563554348572998,"fanout":928230852308},{"left1":16,"right1":41357734026390335997642,"left2":20,"right2":9492174332831888560504,"fanout":34897000138},{"left1":12288,"right1":45776152470785431068568,"left2":12800,"right2":27194482358304171863500,"fanout":181376180958},{"left1":32,"right1":138233361222592662271656,"left2":68,"right2":46807037029626388533460,"fanout":43422236900},{"left1":2,"right1":4434081502317076411280,"left2":2058,"right2":43295489030230475830094,"fanout":919271513832},{"left1":2048,"right1":14567389554264434598718,"left2":1026,"right2":57908320986279224625792,"fanout":103128895008},{"left1":8192,"right1":30763647804896323982438,"left2":16400,"right2":37530602056198745626064,"fanout":180691196466},{"left1":2,"right1":55125767070159858685270,"left2":16,"right2":46904741182376235839252,"fanout":43905374760},{"left1":16384,"right1":10804378938455355747620,"left2":16418,"right2":47506205962932859914978,"fanout":137774941942},{"left1":16384,"right1":2893613280446572926296,"left2":2056,"right2":25930736183786487581176,"fanout":197750167594},{"left1":2048,"right1":64760141634485768625762,"left2":2056,"right2":14534919658591928747018,"fanout":902348070936},{"left1":68,"right1":46591307866832743122240,"left2":1092,"right2":37398319444462874133236,"fanout":879980460236},{"left1":34,"right1":142148456493110047863776,"left2":16418,"right2":39840966180156830880244,"fanout":940498547760},{"left1":40,"right1":67840676806452610471058,"left2":56,"right2":51393594469738183263406,"fanout":200891747528},{"left1":4096,"right1":12342679644985627732606,"left2":12800,"right2":37590789839479493956246,"fanout":86137162772},{"left1":2048,"right1":23286064293558758792552,"left2":2056,"right2":22246692208870768266154,"fanout":180626656818},{"left1":16448,"right1":71799319437342170948808,"left2":16704,"right2":12460979944837887421680,"fanout":167137125604},{"left1":256,"right1":42216468823222844236098,"left2":4096,"right2":149209289717372294484,"fanout":841862875160},{"left1":2,"right1":39264257505526520589208,"left2":1026,"right2":32559863747544624495688,"fanout":63955586620},{"left1":640,"right1":54630786855964779686956,"left2":8832,"right2":67589783201810136304888,"fanout":201443963908},{"left1":256,"right1":20068685620943835430278,"left2":32784,"right2":28836958944992690419548,"fanout":138446029020},{"left1":2,"right1":132488115077185249481876,"left2":1152,"right2":48048279861032505755334,"fanout":46711842334},{"left1":32768,"right1":64065385571756595527034,"left2":260,"right2":53935286958374737080218,"fanout":43188675632},{"left1":8192,"right1":73836234517785444464914,"left2":9344,"right2":63322605843544295246120,"fanout":215567492642},{"left1":16,"right1":35398449197350472624984,"left2":4112,"right2":31160180272103886048464,"fanout":9240329950},{"left1":640,"right1":65057568994067887291258,"left2":656,"right2":66738197564687121556778,"fanout":1031533628156},{"left1":256,"right1":126141458902782495081434,"left2":260,"right2":136282738804013756173328,"fanout":151210501840},{"left1":4352,"right1":43684388335121077452490,"left2":4864,"right2":13440132645519750722766,"fanout":115744973516},{"left1":128,"right1":33937637108307462414454,"left2":384,"right2":26214928443464122147776,"fanout":859311184920},{"left1":2,"right1":98876174894609709702810,"left2":258,"right2":23479178410351593500974,"fanout":86319026152},{"left1":2,"right1":56403373491985472063804,"left2":2050,"right2":58490879245689329538980,"fanout":60364951052},{"left1":16384,"right1":23374078052912624725342,"left2":20736,"right2":5644177944735463267124,"fanout":1052099667486},{"left1":40,"right1":10562328107239277912168,"left2":2088,"right2":4786480417395348020062,"fanout":95444156114},{"left1":512,"right1":127657687041774522973288,"left2":552,"right2":78737109653482306268238,"fanout":223957533224},{"left1":1024,"right1":27092008483336879336784,"left2":1152,"right2":70409731406147974149170,"fanout":888034484450},{"left1":512,"right1":55189596231365332340160,"left2":1026,"right2":31555614608792008217344,"fanout":154734046240},{"left1":128,"right1":9432437475266815926218,"left2":2176,"right2":18410984131262886996610,"fanout":29226226236},{"left1":34,"right1":15905402209328904239006,"left2":42,"right2":44262962290077453814690,"fanout":72826668058},{"left1":2,"right1":62805924002674654007468,"left2":4,"right2":74358913785813802643514,"fanout":141770499108},{"left1":1024,"right1":30883952158551000811610,"left2":33792,"right2":21146366912025122136122,"fanout":184349361202},{"left1":64,"right1":12629751913324359338052,"left2":256,"right2":47978714248269089700004,"fanout":163423244518},{"left1":2,"right1":22398004613406236643748,"left2":6,"right2":119437276348174231399160,"fanout":3693200680},{"left1":2048,"right1":27679662268892233822282,"left2":2056,"right2":71964239920082663447782,"fanout":138280363558},{"left1":2,"right1":149054133002166724872124,"left2":34818,"right2":62140142737797473544392,"fanout":902695146216},{"left1":8,"right1":68042101692637324697246,"left2":32784,"right2":72086786912809792083622,"fanout":112512003302},{"left1":160,"right1":51019072679821144586088,"left2":16512,"right2":17948740623522762814434,"fanout":862666898632},{"left1":1024,"right1":9141788398648001927554,"left2":1064,"right2":14674078039790513042776,"fanout":956988921882},{"left1":32768,"right1":70230777643983346994124,"left2":32896,"right2":30231691088481622605922,"fanout":120261107762},{"left1":512,"right1":48284832782670723815070,"left2":4608,"right2":46631854116044173348366,"fanout":17649824456},{"left1":32784,"right1":72570727314634513112062,"left2":32816,"right2":25013343352371520429772,"fanout":1013693940254},{"left1":2,"right1":86752413951134838770088,"left2":4098,"right2":52992496206337166824242,"fanout":854512510156},{"left1":32,"right1":110081793316660600624456,"left2":16480,"right2":48335532589121576096788,"fanout":115899431434},{"left1":16448,"right1":37242334621281634832836,"left2":16464,"right2":47097645951661348058342,"fanout":141401318138},{"left1":640,"right1":113937189529416577106390,"left2":648,"right2":86076103070126475287384,"fanout":1068943805452},{"left1":8,"right1":89813972764460307555890,"left2":136,"right2":19496926062117818509946,"fanout":876993761534},{"left1":4,"right1":48622902209072554166378,"left2":36,"right2":150304996119784251518922,"fanout":236102760960},{"left1":128,"right1":56207470056444443679280,"left2":8192,"right2":36571494584298508266898,"fanout":982576320196},{"left1":16384,"right1":9898214580216729362056,"left2":68,"right2":34177486049079338274352,"fanout":55299337464},{"left1":32768,"right1":129205680452975391704066,"left2":32808,"right2":77030551315823309225362,"fanout":1060536723686},{"left1":1024,"right1":55961908416923025385248,"left2":5376,"right2":48496001026160878625866,"fanout":171879178470},{"left1":4,"right1":101911225553652454395968,"left2":260,"right2":32282800194030103435998,"fanout":962389923307},{"left1":4096,"right1":59339705259623343627908,"left2":16400,"right2":72495296689250377094372,"fanout":200891707442},{"left1":2048,"right1":103193226050864669800042,"left2":2056,"right2":86519076989571890094410,"fanout":851412653106},{"left1":16,"right1":45459012218096950846050,"left2":276,"right2":27738160609481976299666,"fanout":46173110566},{"left1":256,"right1":28278557888068937748246,"left2":320,"right2":27642500756322928014302,"fanout":922285822},{"left1":32768,"right1":9046809369626409192784,"left2":32784,"right2":33066792646663375904440,"fanout":837086015700},{"left1":34816,"right1":133968883507445679164074,"left2":34824,"right2":90151581467265939793226,"fanout":266905726144},{"left1":128,"right1":49986795835870104701408,"left2":1024,"right2":36364023381098527625046,"fanout":200925784818},{"left1":512,"right1":32791050059112499249346,"left2":16400,"right2":8806393934636467733224,"fanout":12552125658},{"left1":16,"right1":22760036162515587248836,"left2":276,"right2":15898037875934171944682,"fanout":239530489842},{"left1":4608,"right1":75356375338693430786892,"left2":12288,"right2":73255569920496890491072,"fanout":163880040978},{"left1":2,"right1":112764004824174142485900,"left2":262,"right2":9225608762868009129216,"fanout":863339468836},{"left1":256,"right1":22445344127999840681050,"left2":2048,"right2":47263304879259502859378,"fanout":193053988880},{"left1":640,"right1":34640706126806428667490,"left2":8832,"right2":55826047837224412982584,"fanout":38187491522},{"left1":64,"right1":51816333982807178070874,"left2":32832,"right2":74519425716066663030028,"fanout":69058361914},{"left1":8704,"right1":21225349001474356192086,"left2":25088,"right2":17777124779387919296566,"fanout":106471627968},{"left1":16,"right1":63700307515698303763590,"left2":20,"right2":29703738652452552784250,"fanout":17787920939},{"left1":512,"right1":26006833088750990979650,"left2":4608,"right2":52703428143148294875224,"fanout":21188761296},{"left1":1026,"right1":5147523942579653550402,"left2":1034,"right2":73726315672884124250192,"fanout":189416871110},{"left1":2048,"right1":73707938023161982490530,"left2":2112,"right2":58578976568810706957910,"fanout":991783034418},{"left1":64,"right1":48364223804828108451594,"left2":260,"right2":61466294489160558126890,"fanout":956907571244},{"left1":32,"right1":59188168351694636154258,"left2":4384,"right2":61625895174737528332646,"fanout":270384683544},{"left1":32768,"right1":64251786551954012420878,"left2":33024,"right2":29310024254473613816734,"fanout":115293482506},{"left1":4100,"right1":1391035871393494304556,"left2":12292,"right2":51936522331458175483048,"fanout":1034885928492},{"left1":256,"right1":1736323187676734414760,"left2":4352,"right2":64206617852213788029726,"fanout":931017004576},{"left1":34816,"right1":63269498259587165961018,"left2":34848,"right2":26791677654472056123634,"fanout":17485986820},{"left1":2,"right1":71108854341936331647716,"left2":256,"right2":34518005579785518657136,"fanout":17480887095},{"left1":4,"right1":47808924252378901246074,"left2":16,"right2":3012157170800591561978,"fanout":894038319381},{"left1":4,"right1":108721305692156729434192,"left2":2056,"right2":55310665776185492444208,"fanout":206979914946},{"left1":64,"right1":15665138199392114017482,"left2":34816,"right2":53811763659527304084796,"fanout":12418861608},{"left1":32768,"right1":4382145028460370746656,"left2":1026,"right2":23508744526415740757374,"fanout":29196292844},{"left1":16384,"right1":32334504180757114394166,"left2":2056,"right2":53833238763965863562824,"fanout":928553163986},{"left1":2,"right1":73957824454313595687334,"left2":16,"right2":74644706894885055047130,"fanout":825675952887},{"left1":4,"right1":134605596064204077752088,"left2":64,"right2":23688453209976309163200,"fanout":1043308353068},{"left1":16384,"right1":44801806439712739368308,"left2":16424,"right2":62010256798313696313112,"fanout":988559900174},{"left1":1024,"right1":13093516541421536239534,"left2":1088,"right2":55720235640874285720130,"fanout":948129373940},{"left1":260,"right1":33156679283003772280066,"left2":8452,"right2":32077810533607966281570,"fanout":862686739158},{"left1":8,"right1":63698941243113639116470,"left2":12288,"right2":10208115781359238466942,"fanout":845340608522},{"left1":1026,"right1":55437620157688282146820,"left2":2056,"right2":14881589884974684066442,"fanout":1066193399860},{"left1":4096,"right1":44984564757925857840902,"left2":4160,"right2":24349522290251536059974,"fanout":876241749188},{"left1":128,"right1":50395858345567839448244,"left2":130,"right2":57961806696491758885578,"fanout":124233458428},{"left1":2048,"right1":49845317496936652498458,"left2":2064,"right2":16991333494749662082076,"fanout":1039517663232},{"left1":2048,"right1":71664484995891481147372,"left2":16400,"right2":55219581585481907309522,"fanout":29330689250},{"left1":4,"right1":134306896612415831284388,"left2":1024,"right2":30292715898807377590282,"fanout":845072103138},{"left1":1152,"right1":65307409454325116981788,"left2":1664,"right2":5372701113788355793064,"fanout":214861750328},{"left1":1024,"right1":69018081147862704074040,"left2":33792,"right2":69544717281607709706778,"fanout":868133949164},{"left1":32,"right1":88390164214837888470950,"left2":16432,"right2":62748839632197926668208,"fanout":46808746496},{"left1":4,"right1":65617540270923263522796,"left2":6,"right2":123130675538988149533890,"fanout":146411869231},{"left1":32,"right1":116717969284001427091246,"left2":1026,"right2":63751675989311174674560,"fanout":26510942742},{"left1":16384,"right1":42487968836482720094306,"left2":25088,"right2":4916780702801537303318,"fanout":1000159714532},{"left1":16448,"right1":71191585826642653266746,"left2":49216,"right2":31584030579429910351870,"fanout":192548487198},{"left1":64,"right1":28973143528195327493854,"left2":32768,"right2":7996721256449412678856,"fanout":120326517494},{"left1":4096,"right1":20333069888443354123114,"left2":16384,"right2":59154105459936960946470,"fanout":63398673434},{"left1":8192,"right1":18391009964535253846408,"left2":9218,"right2":7705945670528700637410,"fanout":189954921014},{"left1":32,"right1":148252398724799950053500,"left2":8704,"right2":46475240604690365343274,"fanout":115238906928},{"left1":512,"right1":31813631242471353040170,"left2":4352,"right2":73018250352522293270190,"fanout":868221768386},{"left1":8,"right1":75990920598450420863594,"left2":16448,"right2":3669495065790333572784,"fanout":966168425722},{"left1":16384,"right1":18304388702129061860404,"left2":18464,"right2":6741292042636864967908,"fanout":55683449366},{"left1":1152,"right1":28680255411382873991092,"left2":33920,"right2":74724128453203181184676,"fanout":1000259780298},{"left1":34,"right1":132909246760065604941794,"left2":546,"right2":32194132018481845827446,"fanout":879812071142},{"left1":2048,"right1":41422416597506988103526,"left2":18432,"right2":41749578508081367019094,"fanout":38366875140},{"left1":4096,"right1":41505847720631509709704,"left2":4352,"right2":59371268693768961603104,"fanout":261737603606},{"left1":1152,"right1":13530350606686638130202,"left2":1664,"right2":27484259284566099921974,"fanout":260932765422},{"left1":8704,"right1":7142044954788295833264,"left2":9728,"right2":28824558351503485965908,"fanout":261023908902},{"left1":1026,"right1":1411319340855196248828,"left2":9218,"right2":22363367921699577057306,"fanout":236135124200},{"left1":32,"right1":33449520757591334105748,"left2":2088,"right2":34048827658992012260122,"fanout":232713546256},{"left1":4,"right1":70853494963276613400168,"left2":256,"right2":59717035761851240173448,"fanout":146110356208},{"left1":8192,"right1":102349206571553067847246,"left2":8226,"right2":102663967163790943759966,"fanout":996882175730},{"left1":8192,"right1":44288741468287877954990,"left2":16400,"right2":22323052348825703230472,"fanout":266334509568},{"left1":8,"right1":36222852893441299796726,"left2":1152,"right2":67637566596262112442076,"fanout":261760486456},{"left1":64,"right1":70721726574215622909336,"left2":8192,"right2":46120382393609573769180,"fanout":888134837256},{"left1":1024,"right1":23443800466458529610590,"left2":640,"right2":63662383354040475901344,"fanout":155257859312},{"left1":16384,"right1":20369813831525134040710,"left2":49168,"right2":33864759815055842997454,"fanout":906140764922},{"left1":4096,"right1":40450539950419736776620,"left2":4608,"right2":74675314176290189177926,"fanout":1031298482422},{"left1":32,"right1":139334482180242883440742,"left2":4352,"right2":59664665603723866596184,"fanout":201833319964},{"left1":32768,"right1":54526764225344668376178,"left2":32784,"right2":74151908136031736057472,"fanout":77916464166},{"left1":1152,"right1":12437803548324787038706,"left2":17536,"right2":46114683354282872749994,"fanout":52345885700},{"left1":34816,"right1":257927082200592830902,"left2":34832,"right2":53126429531848725559556,"fanout":1068993659420},{"left1":16,"right1":1603172046140436894704,"left2":276,"right2":4221772102973702834834,"fanout":51370329040},{"left1":16,"right1":77060982459834106980346,"left2":20,"right2":76530292692356920430826,"fanout":8485597501},{"left1":8192,"right1":48379735416387005413760,"left2":16448,"right2":2922947003124346618562,"fanout":992019537646},{"left1":12288,"right1":40384592145540601552756,"left2":12352,"right2":36593661832699906746274,"fanout":270114019068},{"left1":2,"right1":25113750267975371922022,"left2":262,"right2":58695008148004594640574,"fanout":68015097299},{"left1":256,"right1":19485274867583099088738,"left2":2048,"right2":23346549155462963000980,"fanout":971383378992},{"left1":32784,"right1":117329374702070934754784,"left2":32792,"right2":77682531230299206206922,"fanout":1048290266640},{"left1":128,"right1":33404818901753473404508,"left2":144,"right2":2077603615622965910516,"fanout":98283557412},{"left1":640,"right1":5356723450003427188506,"left2":17024,"right2":63922199452927465345324,"fanout":18065184298},{"left1":16384,"right1":27596314460675122442016,"left2":8704,"right2":63430015347949831520586,"fanout":902761864200},{"left1":1056,"right1":4399233625142411625140,"left2":1120,"right2":28056995851673948921864,"fanout":132141169888},{"left1":4,"right1":132027255816909393622094,"left2":20,"right2":71953921728336057725086,"fanout":21611935013},{"left1":32784,"right1":31945678401092815917118,"left2":33040,"right2":68085922952326090314636,"fanout":897147130886},{"left1":32,"right1":36725268430670735674254,"left2":68,"right2":45824740741843512729642,"fanout":1090970643994},{"left1":640,"right1":41192487264563222712062,"left2":8832,"right2":38378818713513566989470,"fanout":98485985986},{"left1":34816,"right1":40683480791354934099336,"left2":34820,"right2":73397137117233101915434,"fanout":1031679247054},{"left1":32,"right1":142639109398520034804094,"left2":34848,"right2":53081171462446210320362,"fanout":214886527204},{"left1":4,"right1":130733783732398418867422,"left2":20,"right2":42217312202955219213346,"fanout":12585920483}],"pure":{"0":270684715835,"2":141732358649,"4":1047937600303,"8":1014635563793,"32":68938421988,"80":265613330133,"96":94285802557,"112":892619139085,"128":55163358990,"176":957326046013,"192":1009756224744,"208":192386679025,"224":889995670312,"240":274155495709,"320":991230486560,"352":42427539197,"384":141769386439,"416":954523909693,"432":879862556933,"464":828814130397,"480":1022279734,"512":1087265246705,"528":1001783035922,"544":845588931068,"560":141970316761,"608":93989955121,"640":149468412979,"672":69303268803,"688":1000472888565,"704":1013732933366,"784":924053400815,"816":8739492051,"832":846629052691,"848":919861146602,"896":210554187774,"912":1047538183184,"928":30149166097,"944":235219266095,"976":196612318146,"992":996044293071,"1008":1090469492261,"1040":1039969041889,"1056":29726787358,"1072":927257725130,"1088":1018380477217,"1104":970866759930,"1120":249796302281,"1152":888199846430,"1168":901276093899,"1184":9525520590,"1200":897767499031,"1":50701269019,"10":239946428409,"12":184148888354,"14":919260902626,"160":60329412908,"256":103970566109,"272":846599151121,"288":21830230289,"400":1053019003177,"448":77339614470,"496":957542052918,"592":1051474333688,"624":269744344589,"736":245051108587,"768":72345907995,"864":940362757140,"1024":110911483107,"1136":948585480249,"5":996516107789,"6":914523634953,"7":146345353226,"9":235822722031,"48":1027030989617,"64":189450216166,"576":84962515455,"720":1091186328548,"752":159403419700,"880":849361246448,"960":879446383585,"1216":89840687637,"13":155104321070,"336":63772294953,"656":1039331171104,"800":206463179276,"144":911049636056,"368":1027268427750,"304":937011182309,"11":876489012940,"16":846226249263,"3":1061464450250,"15":163614702286},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"three-phase","capacity":8,"product_order":[0,1,3,7,8,9,11,15,17,19,20,22,25,26,28,29,30,31,33,36,37,38,39,42,44,45,46,48,49,50,53,54,55,58,59,60,61,63,64,65,71,73,74,76,78,80,82,84,85,87,88,93,95,97,98,99,102,104,106,107,108,110,111,115,116,117,118,122,123,124,125,127,128,130,131,133,134,139,140,141,142,143,144,146,150,153,154,155,156,159,160,161,162,163,166,168,170,171,173,174,175,177,180,181,185,186,187,188,189,194,195,196,198,202,2,4,12,21,23,35,40,41,66,68,70,72,75,81,92,100,101,103,114,120,147,151,169,176,178,183,192,193,199,5,6,10,13,14,16,18,24,27,32,34,43,47,51,52,56,57,62,67,69,77,79,83,86,89,90,91,94,96,105,109,112,113,119,121,126,129,132,135,136,137,138,145,148,149,152,157,158,164,165,167,172,179,182,184,190,191,197,200,201],"events":[{"kind":"products","cache":951,"indices":[0,1,3,7,8,9,11,15,17,19,20,22,25,26,28,29,30,31,33,36,37,38,39,42,44,45,46,48,49,50,53,54,55,58,59,60,61,63,64,65,71,73,74,76,78,80,82,84,85,87,88,93,95,97,98,99,102,104,106,107,108,110,111,115,116,117,118,122,123,124,125,127,128,130,131,133,134,139,140,141,142,143,144,146,150,153,154,155,156,159,160,161,162,163,166,168,170,171,173,174,175,177,180,181,185,186,187,188,189,194,195,196,198,202]},{"kind":"pure_low","cache":951,"monomial":10},{"kind":"pure_low","cache":951,"monomial":12},{"kind":"pure_low","cache":951,"monomial":14},{"kind":"pure_low","cache":951,"monomial":5},{"kind":"pure_low","cache":951,"monomial":6},{"kind":"pure_low","cache":951,"monomial":9},{"kind":"pure_low","cache":951,"monomial":13},{"kind":"pure_low","cache":951,"monomial":3},{"kind":"products","cache":831,"indices":[2,4,12,21,23,35,40,41,66,68,70,72,75,81,92,100,101,103,114,120,147,151,169,176,178,183,192,193,199]},{"kind":"pure_low","cache":831,"monomial":7},{"kind":"products","cache":1647,"indices":[5,6,10,13,14,16,18,24,27,32,34,43,47,51,52,56,57,62,67,69,77,79,83,86,89,90,91,94,96,105,109,112,113,119,121,126,129,132,135,136,137,138,145,148,149,152,157,158,164,165,167,172,179,182,184,190,191,197,200,201]},{"kind":"pure_low","cache":1647,"monomial":11},{"kind":"pure_low","cache":1647,"monomial":15}],"low_computations":11}},{"batch":7,"delta_sha256":"e0a44a4b255fc19f66edf19a7fd37f5e73a969bad66f95f179b9dcb4de348948","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":32,"right1":35966346535125998948844,"left2":64,"right2":13607064540547457596226,"fanout":261670357016},{"left1":16448,"right1":24627284766116152544912,"left2":16960,"right2":32588516105869297655212,"fanout":928330083568},{"left1":2048,"right1":6946896033776067546412,"left2":16384,"right2":62165639973072342502076,"fanout":206844464136},{"left1":33792,"right1":275727178073333706556,"left2":42,"right2":73673946842161698959000,"fanout":965294480636},{"left1":512,"right1":55135824330511365539718,"left2":16900,"right2":75111264419616761584234,"fanout":223472518152},{"left1":128,"right1":65873499154612536708334,"left2":8208,"right2":69624524061666497123056,"fanout":859007880384},{"left1":2176,"right1":23877811962191984889274,"left2":10368,"right2":21105785709821134013830,"fanout":910802871492},{"left1":32768,"right1":34842560835355784849494,"left2":49216,"right2":47421387496070238731394,"fanout":945778605264},{"left1":32768,"right1":8407110182629786262652,"left2":8448,"right2":70173950076881080654794,"fanout":189584687140},{"left1":8,"right1":51104559212008745869274,"left2":16388,"right2":32295708366558052245234,"fanout":89539273948},{"left1":2,"right1":73644598522055284086746,"left2":16,"right2":54867161314604510670984,"fanout":94418758393},{"left1":128,"right1":74317866787959782588186,"left2":1028,"right2":49362569092058271900804,"fanout":979402760228},{"left1":64,"right1":68724633370702312094792,"left2":2112,"right2":12537726077761733826912,"fanout":979399357644},{"left1":4,"right1":42055528453213380864280,"left2":260,"right2":47595921821192994957524,"fanout":137607308832},{"left1":128,"right1":45286923799350109655968,"left2":8336,"right2":17744145715079015740694,"fanout":945510473784},{"left1":136,"right1":47479634361479340255596,"left2":104,"right2":4798290661035827176608,"fanout":1013692290092},{"left1":4,"right1":125743677462793038784462,"left2":16,"right2":4981010172989435361496,"fanout":850673910496},{"left1":64,"right1":10973989195644638595610,"left2":200,"right2":28745514499756823267908,"fanout":896978925620},{"left1":128,"right1":60216718712745621285846,"left2":16516,"right2":37499733675406072494384,"fanout":241256950788},{"left1":2176,"right1":27682211314563450750100,"left2":33792,"right2":55770797175668911284304,"fanout":1034886058012},{"left1":16384,"right1":23273140238450894039872,"left2":16400,"right2":53184101547791441752926,"fanout":34964042780},{"left1":32,"right1":49733414166295025552494,"left2":16384,"right2":59792160854287584079336,"fanout":863234483436},{"left1":2,"right1":62615357106495407866690,"left2":18,"right2":16540522369827914461124,"fanout":894409650708},{"left1":8,"right1":44461985517177083337230,"left2":264,"right2":20239314666629133061484,"fanout":931873954000},{"left1":32768,"right1":2034886596364663401972,"left2":33796,"right2":32811776607935445247608,"fanout":69269766388},{"left1":16416,"right1":17220249494343550166074,"left2":49152,"right2":60339268266512312830468,"fanout":808204512},{"left1":16,"right1":65554244198039031016774,"left2":8208,"right2":68027218111517531041952,"fanout":980206535880},{"left1":8,"right1":23948365618163496930404,"left2":280,"right2":38629802832831160636970,"fanout":1031131429908},{"left1":8192,"right1":51977455542028542994868,"left2":10368,"right2":24266278100892571729826,"fanout":209382810664},{"left1":512,"right1":43488765717480536699260,"left2":96,"right2":65091199274372953692042,"fanout":914159889420},{"left1":8,"right1":890099555183029934966,"left2":4128,"right2":45395853960247938858162,"fanout":846057494728},{"left1":4,"right1":56732208524448451949538,"left2":20,"right2":23225798539580294478920,"fanout":162489296578},{"left1":33280,"right1":61314695800862231898870,"left2":16452,"right2":34920320236905705495344,"fanout":931876313288},{"left1":2,"right1":84512670622960312166820,"left2":6,"right2":34305724310001438498602,"fanout":111217884446},{"left1":8,"right1":15061957715600864412474,"left2":16448,"right2":59474100536431001719892,"fanout":175571783892},{"left1":128,"right1":21796453498919663090310,"left2":162,"right2":30426844456425879192430,"fanout":1051878238428},{"left1":64,"right1":19981084084575903129458,"left2":2176,"right2":36584155385911953807086,"fanout":949056974020},{"left1":32768,"right1":10717775056406819513976,"left2":1028,"right2":48836579742183947173688,"fanout":1051880858828},{"left1":32,"right1":49255106372780866143542,"left2":64,"right2":36130870504124281478696,"fanout":141479956524},{"left1":64,"right1":65460659046802219794566,"left2":2176,"right2":26686641678065743127206,"fanout":825252433940},{"left1":4,"right1":8311429645932678792992,"left2":256,"right2":35925884496291388605030,"fanout":845824725760},{"left1":32768,"right1":56262579553059305090886,"left2":2304,"right2":42607313904991821928588,"fanout":879679555816},{"left1":2,"right1":83688490772456988014936,"left2":18,"right2":72501655949175720242122,"fanout":209694356001},{"left1":1024,"right1":61184102410201439692788,"left2":32772,"right2":20492638747028418217214,"fanout":189262990552},{"left1":8208,"right1":51572603552847332986804,"left2":8336,"right2":33161915734315707307498,"fanout":158779634920},{"left1":4,"right1":124035811744433696071428,"left2":6,"right2":77382004783958391147922,"fanout":146647990823},{"left1":512,"right1":63070338595925577266598,"left2":34,"right2":31195421004985784727334,"fanout":1031811638492},{"left1":32,"right1":25747180526225165553328,"left2":33824,"right2":16667641445429106599900,"fanout":948130076716},{"left1":384,"right1":106572829387730097802602,"left2":392,"right2":129220285272649141954286,"fanout":137439270948},{"left1":16384,"right1":5098796368460384713680,"left2":16416,"right2":55548276892891276145792,"fanout":979604137992},{"left1":32768,"right1":73318140104426479234722,"left2":4098,"right2":15243120638618193239226,"fanout":123828450340},{"left1":4,"right1":125441033937645700831784,"left2":2052,"right2":32439736476354757594992,"fanout":1047972337884},{"left1":8704,"right1":129089192425880682228888,"left2":8706,"right2":77947268204597811431864,"fanout":86168369172},{"left1":1024,"right1":62410697966560276635140,"left2":2304,"right2":72679218461828686718822,"fanout":1000324785164},{"left1":512,"right1":10489331985985638361280,"left2":4640,"right2":72374721774410838642178,"fanout":124033708080},{"left1":8192,"right1":27655287364703943763036,"left2":4232,"right2":18265236223220918321402,"fanout":893491399728},{"left1":16,"right1":33238395269225897529748,"left2":128,"right2":48739294194550499941074,"fanout":894229030952},{"left1":1024,"right1":74195389648254975350746,"left2":5120,"right2":53259871016488671369510,"fanout":103363962916},{"left1":8192,"right1":66545367194704067573202,"left2":12288,"right2":50911031702948577116694,"fanout":999656008936},{"left1":4096,"right1":31311493168329589497086,"left2":8192,"right2":50084044578822540762394,"fanout":827923168496},{"left1":4,"right1":40501286902218744610046,"left2":32,"right2":24403482110959014674916,"fanout":241337636896},{"left1":4104,"right1":38225868792699041021806,"left2":4136,"right2":59187525178302235835356,"fanout":1069394690080},{"left1":512,"right1":71147266614560756534412,"left2":49156,"right2":20066497193758164355316,"fanout":1034150800608},{"left1":128,"right1":33121443843674532722342,"left2":32900,"right2":5022351544694217896114,"fanout":4174169292},{"left1":128,"right1":19308216890947023925730,"left2":4098,"right2":70517357826123780416854,"fanout":963093260288},{"left1":16384,"right1":75098645708440342565112,"left2":18560,"right2":22638839018804986091056,"fanout":842015242476},{"left1":1024,"right1":71461210420073769620144,"left2":4104,"right2":53021725687133830398244,"fanout":880215596036},{"left1":2,"right1":42803638813722638859188,"left2":4,"right2":32665342565465831052536,"fanout":167235609077},{"left1":4096,"right1":67103375863162769780252,"left2":33792,"right2":64534301665067659767590,"fanout":928454863096},{"left1":8,"right1":26900606858338200482622,"left2":2312,"right2":69834531303261706315612,"fanout":189921285124},{"left1":8,"right1":93131931115171348132430,"left2":4106,"right2":36223089800121852511408,"fanout":175304868092},{"left1":8192,"right1":59083653472861283431722,"left2":2176,"right2":14788529191380673416858,"fanout":893836641340},{"left1":64,"right1":53760465932291763758160,"left2":16384,"right2":12360404159481865628952,"fanout":842487577616},{"left1":32768,"right1":73141237959973647175292,"left2":16448,"right2":73146560398077578090838,"fanout":842417885384},{"left1":32772,"right1":53271750924870459497098,"left2":32836,"right2":14032855801147867879466,"fanout":175422842920},{"left1":32768,"right1":47786015445230761385576,"left2":16388,"right2":74391573344174377469390,"fanout":158445348092},{"left1":128,"right1":58391213849759107345948,"left2":192,"right2":22308987899437397945150,"fanout":138379003960},{"left1":8,"right1":118912015754133194956054,"left2":34,"right2":56507674561304056177482,"fanout":137589429456},{"left1":32768,"right1":12514454678582977806918,"left2":34,"right2":58208929774170160572046,"fanout":893757479104},{"left1":4128,"right1":11530893412664922816690,"left2":4256,"right2":51081385170898964764912,"fanout":914761768996},{"left1":1024,"right1":41045668897041682988310,"left2":392,"right2":7823809104386176291816,"fanout":69460042968},{"left1":2,"right1":21846074877826436011328,"left2":4,"right2":7333324296204905496488,"fanout":832684424137},{"left1":16,"right1":56838608024884597289106,"left2":256,"right2":43591806472396881696332,"fanout":59460890636},{"left1":256,"right1":62679987232719908443002,"left2":392,"right2":27943611565184202187806,"fanout":124435102948},{"left1":512,"right1":42938739480646770439552,"left2":272,"right2":40670526461407030098680,"fanout":155290446900},{"left1":8192,"right1":7761290618173436532294,"left2":12296,"right2":43288380878152865785734,"fanout":825240120516},{"left1":8,"right1":114782649366027804919214,"left2":8448,"right2":73211972467801773413270,"fanout":1017787724820},{"left1":4,"right1":50529530221152039646844,"left2":256,"right2":29073317410819447988288,"fanout":110698880527},{"left1":16,"right1":3469618009341620729140,"left2":272,"right2":60355397459525865519948,"fanout":3471175990},{"left1":512,"right1":6420743605948755482862,"left2":608,"right2":29029001425815072575456,"fanout":261807205584},{"left1":2,"right1":137912307468231354768078,"left2":258,"right2":18778959402481066546312,"fanout":870904024317},{"left1":2,"right1":22681090194000097971916,"left2":256,"right2":14421696388438894844142,"fanout":21237196019},{"left1":2,"right1":66806737878209834394514,"left2":258,"right2":5065730586781884397368,"fanout":910481677091},{"left1":128,"right1":51917993807123782101112,"left2":528,"right2":37928339092009285718662,"fanout":962558177336},{"left1":8,"right1":99618949894390245308446,"left2":2304,"right2":13956823540721840138134,"fanout":828257648868},{"left1":32768,"right1":35841014482598587835622,"left2":4098,"right2":51276022169993995619096,"fanout":931003634928},{"left1":1152,"right1":1923283734221450950372,"left2":3200,"right2":48215008729377222819810,"fanout":34710026252},{"left1":512,"right1":622154123524348594116,"left2":96,"right2":17208401017229890633180,"fanout":965564433628},{"left1":2,"right1":98321399070268427112632,"left2":272,"right2":63540619618595617690364,"fanout":245584295881},{"left1":16,"right1":39246245562549260657350,"left2":256,"right2":24104814412503187777306,"fanout":163044971811},{"left1":2,"right1":17427190262229079558142,"left2":256,"right2":14774536953762466334464,"fanout":97894414581},{"left1":256,"right1":52070468521129653190906,"left2":8448,"right2":63049953173708020011570,"fanout":1068587810832},{"left1":96,"right1":116833873569664809572984,"left2":104,"right2":131841675570863016203534,"fanout":257967535296},{"left1":4096,"right1":38314053280321631814510,"left2":8704,"right2":41724868839448590039610,"fanout":258707082284},{"left1":8192,"right1":71692540487768631661452,"left2":10,"right2":111536408427381079464008,"fanout":3761775852},{"left1":2,"right1":15410577838464860577202,"left2":8704,"right2":15271149877644981459364,"fanout":983493771268},{"left1":4096,"right1":4204638600657875567558,"left2":12800,"right2":74861537381476368029988,"fanout":859073416444},{"left1":2176,"right1":72109169756626095589204,"left2":18560,"right2":55329300003845343182198,"fanout":845517424888},{"left1":512,"right1":37854786844404690605858,"left2":2304,"right2":4602785610830130262714,"fanout":138246364408},{"left1":512,"right1":14092608542203471543546,"left2":546,"right2":66299674692961982164578,"fanout":52011004964},{"left1":128,"right1":1314218800169788190446,"left2":16516,"right2":16921127516722749081914,"fanout":54843195632},{"left1":16448,"right1":27421571536425094802202,"left2":16960,"right2":73665333511356157928468,"fanout":257714557120},{"left1":33792,"right1":43375539898739590526410,"left2":35840,"right2":18595968071422972824924,"fanout":416548920},{"left1":4,"right1":38788906551901143149386,"left2":512,"right2":22718144953445756074392,"fanout":948799591652},{"left1":16,"right1":18784999940993977285408,"left2":18,"right2":16818634513313381238456,"fanout":832973576982},{"left1":32,"right1":60129087734767578565544,"left2":33824,"right2":62361890282737563523238,"fanout":1082613887184},{"left1":64,"right1":2412605116321286526874,"left2":256,"right2":13315261006730801601310,"fanout":55109537852},{"left1":4,"right1":33582498046646237167778,"left2":4096,"right2":406206097236359293256,"fanout":879611658240},{"left1":528,"right1":54114638771535552575742,"left2":536,"right2":64862050214749356686778,"fanout":3356546288},{"left1":4104,"right1":44130496234360589371802,"left2":5128,"right2":14955116197098678211056,"fanout":189192739068},{"left1":1024,"right1":14368973882532134079550,"left2":4096,"right2":1170476003132365784560,"fanout":1034619978772},{"left1":8,"right1":88643248543973951252014,"left2":8208,"right2":9278078172269290731310,"fanout":258586506260},{"left1":32768,"right1":14947825329019371325702,"left2":34832,"right2":23637880420361095945066,"fanout":171866376444},{"left1":32768,"right1":48530896370669197597152,"left2":4130,"right2":50724450882272508557078,"fanout":17259823312},{"left1":4,"right1":7056760591407784244910,"left2":20,"right2":62323584843352829969990,"fanout":170876258282},{"left1":512,"right1":62679611933415285384452,"left2":400,"right2":23969571677452498469800,"fanout":842352614408},{"left1":16,"right1":99978144723007610176214,"left2":20,"right2":77191621573717314607608,"fanout":94291755833},{"left1":128,"right1":22265659220295421156906,"left2":66,"right2":64689681812147632866020,"fanout":261926999060},{"left1":8,"right1":6378502689507700901222,"left2":24,"right2":46170079962421872681628,"fanout":258586981388},{"left1":528,"right1":47867107840442078451656,"left2":33296,"right2":24441029601004322596802,"fanout":172538474536},{"left1":512,"right1":58829577279618348533472,"left2":4610,"right2":29333351227838675227074,"fanout":1083217074204},{"left1":8192,"right1":67483490940965552943820,"left2":4104,"right2":28372745254737750419140,"fanout":224079174856},{"left1":2304,"right1":47985566036264779053048,"left2":2432,"right2":72785133524930596283140,"fanout":72356201532},{"left1":16384,"right1":1815321419753762784162,"left2":16448,"right2":43293400690336078420812,"fanout":982623465508},{"left1":8208,"right1":54042077325971350856046,"left2":8720,"right2":69014116443222849069194,"fanout":1086226495548},{"left1":33280,"right1":71080212087073487378772,"left2":49664,"right2":64357789968150156088312,"fanout":1034898911232},{"left1":4,"right1":44727727465984222932508,"left2":260,"right2":54311123988474653939576,"fanout":962378275290},{"left1":2,"right1":113300302434383655461588,"left2":6,"right2":8259704752265366677474,"fanout":39358235852},{"left1":4096,"right1":15937857754494536799130,"left2":8192,"right2":51406172862243632513984,"fanout":893770280168},{"left1":2176,"right1":25848745766499594277920,"left2":6272,"right2":34557979604767469913486,"fanout":124490146024},{"left1":528,"right1":63861088431880283012404,"left2":8704,"right2":31584553833214345427440,"fanout":38602806324},{"left1":128,"right1":45874811754604662095614,"left2":33796,"right2":1778573639843669343960,"fanout":1051474789604},{"left1":4,"right1":25235060381760492442144,"left2":272,"right2":19743978218480132722300,"fanout":846900878643},{"left1":2048,"right1":38622183049980208618470,"left2":49156,"right2":63617028549210158906052,"fanout":21139603700},{"left1":32,"right1":43552194334941117895972,"left2":16416,"right2":27185532461865611405652,"fanout":89457434648},{"left1":16388,"right1":22727907484298380968638,"left2":16420,"right2":29764862886356588258132,"fanout":192404274232},{"left1":2048,"right1":676996108580353467930,"left2":16384,"right2":52694978408416018689442,"fanout":189179887808},{"left1":8,"right1":40671596270456969977250,"left2":16448,"right2":62385694251107286146334,"fanout":1031411080208},{"left1":4096,"right1":39242868902568869399750,"left2":5120,"right2":19968142737436692066786,"fanout":21072772324},{"left1":16,"right1":37525257387357297727472,"left2":1024,"right2":3102691242913257995296,"fanout":897181087800},{"left1":32768,"right1":48840580113617708943646,"left2":96,"right2":37008019664544162251322,"fanout":980275230940},{"left1":16,"right1":96645623728551035711868,"left2":20,"right2":138563734849988192387568,"fanout":51804966973},{"left1":2048,"right1":30584407287113164589196,"left2":10752,"right2":48137046576529067705656,"fanout":862352317492},{"left1":1024,"right1":45457394759362116286922,"left2":2048,"right2":8780302874150351759390,"fanout":134532140},{"left1":2,"right1":83490926358010580648842,"left2":274,"right2":33160699172336571491412,"fanout":902964446471},{"left1":512,"right1":1601861771250500147278,"left2":2816,"right2":60178908923786042567872,"fanout":226829024276},{"left1":1028,"right1":27940844314856438034468,"left2":1540,"right2":36998397205203333931586,"fanout":962354004184},{"left1":256,"right1":65436239701108805750064,"left2":272,"right2":11950096957365999539144,"fanout":893406672324},{"left1":1024,"right1":6835632243143427649968,"left2":32772,"right2":41027166186593393181438,"fanout":223340400700},{"left1":32768,"right1":5567753158952034584218,"left2":4136,"right2":51901637725140065149002,"fanout":51888312352},{"left1":256,"right1":130713008162981596875018,"left2":258,"right2":109901069944638748394226,"fanout":142384241393},{"left1":16,"right1":59031981801335036563624,"left2":272,"right2":40255830222900460977568,"fanout":832181620521},{"left1":16384,"right1":20279569838728073823344,"left2":16448,"right2":12464086410917712603762,"fanout":209460194520},{"left1":16,"right1":33048449356362792124162,"left2":18,"right2":58802264494375894157030,"fanout":43852047105},{"left1":384,"right1":35201795048763062973258,"left2":33152,"right2":22861869381527863077088,"fanout":155572302072},{"left1":8,"right1":80317144293454226980926,"left2":8216,"right2":51429956968316254413636,"fanout":1051463048392},{"left1":32,"right1":73871835702506066204592,"left2":10,"right2":34718752954695464875596,"fanout":120462034124},{"left1":64,"right1":40000380115074732757800,"left2":49156,"right2":56725415647443588079740,"fanout":1048256235520},{"left1":32768,"right1":74053457428770336165734,"left2":2320,"right2":25182839894650002268750,"fanout":124354610176},{"left1":33792,"right1":66334391532836535369938,"left2":37888,"right2":30051016029173928190708,"fanout":120866526408},{"left1":4,"right1":96262350726541134873162,"left2":260,"right2":39600271287473783755594,"fanout":145187523631},{"left1":16384,"right1":40918508771027217219970,"left2":18440,"right2":64124080571822805841454,"fanout":876512290036},{"left1":8192,"right1":59715221678239547354352,"left2":10368,"right2":71390842205384713243292,"fanout":931014910012},{"left1":1024,"right1":29564476158679616850462,"left2":4232,"right2":5457802605729445945264,"fanout":55380294892},{"left1":2,"right1":111550890872279648663896,"left2":4098,"right2":38652009503107077281816,"fanout":1017048004640},{"left1":32,"right1":24603682868787732874372,"left2":8704,"right2":13056044016472247491834,"fanout":963080142900},{"left1":16388,"right1":23105236096973436006374,"left2":49156,"right2":16329429984809966727884,"fanout":158109322484},{"left1":256,"right1":90216139994982575839132,"left2":264,"right2":76143479052844446809134,"fanout":175559998680},{"left1":96,"right1":40880176030258468528250,"left2":160,"right2":10865592405261551228370,"fanout":911070730432},{"left1":10,"right1":94041345261587649048812,"left2":138,"right2":616377252123036762620,"fanout":825037692960}],"pure":{"2":1046979339327,"32":974168053514,"64":34728326128,"112":141520009760,"160":106433677601,"176":107289124295,"224":81366955752,"240":93748215053,"256":248368734696,"288":124953553938,"304":263046961634,"320":969989491767,"336":232411632845,"368":858353365038,"384":1048891164905,"400":231996705989,"416":932340772159,"432":966766827017,"480":872851638513,"528":871305632548,"576":983530290390,"592":127913308678,"624":1031025276872,"640":875617986556,"672":984354769359,"736":1038844676084,"768":885631928796,"800":235468898056,"816":915620236521,"832":253768439807,"864":219881141307,"896":7599162857,"944":1065334012189,"992":85377812759,"1008":1043995353896,"1056":1090464639220,"1072":26286941221,"1152":91230633679,"1168":845988303348,"1200":996905062901,"0":1052286710726,"1":201783376848,"4":944871890386,"208":846948981749,"464":1030757876529,"512":13017540886,"544":863018608357,"560":889479763200,"656":889059687374,"688":116164984844,"704":876912297174,"752":846850110257,"880":1065991727344,"912":996466292470,"928":20733019413,"960":940902579940,"976":1064177357574,"1024":77680158664,"1040":102203261945,"1088":854479021811,"1120":884948459471,"1136":952542369087,"1216":249076766506,"3":829731897853,"6":751710483,"7":1048187036920,"10":863621624829,"13":154621969464,"14":1065758284012,"15":106786730232,"16":128180024358,"96":80946840061,"128":939772545320,"352":111400715456,"496":67888104,"608":9280566221,"1184":141612826163,"8":888384920785,"11":931944257548,"12":245655723580,"48":167852378587,"144":1095857589768,"272":1060770478563,"448":8955241984,"784":262647725320,"848":149348881132,"9":7667786735,"192":1073267938510,"720":274640861941,"5":162706298073,"80":1077671281602,"1104":51371885341},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"three-phase","capacity":8,"product_order":[3,10,11,13,16,19,22,23,24,27,30,31,33,35,37,40,41,42,43,45,46,47,48,50,51,53,54,56,57,60,61,63,64,66,67,68,69,70,77,78,79,80,81,82,83,84,87,88,90,91,92,93,94,95,96,98,99,100,108,109,112,113,114,115,117,118,119,120,122,123,124,125,126,128,129,130,132,136,137,139,141,142,148,149,151,153,154,155,156,157,158,159,160,161,163,164,166,168,169,170,173,174,177,179,5,6,8,14,26,28,44,52,55,58,59,71,85,86,101,103,104,105,106,121,131,134,138,140,152,165,172,175,0,1,2,4,7,9,12,15,17,18,20,21,25,29,32,34,36,38,39,49,62,65,72,73,74,75,76,89,97,102,107,110,111,116,127,133,135,143,144,145,146,147,150,162,167,171,176,178],"events":[{"kind":"products","cache":1275,"indices":[3,10,11,13,16,19,22,23,24,27,30,31,33,35,37,40,41,42,43,45,46,47,48,50,51,53,54,56,57,60,61,63,64,66,67,68,69,70,77,78,79,80,81,82,83,84,87,88,90,91,92,93,94,95,96,98,99,100,108,109,112,113,114,115,117,118,119,120,122,123,124,125,126,128,129,130,132,136,137,139,141,142,148,149,151,153,154,155,156,157,158,159,160,161,163,164,166,168,169,170,173,174,177,179]},{"kind":"pure_low","cache":1275,"monomial":3},{"kind":"pure_low","cache":1275,"monomial":7},{"kind":"pure_low","cache":1275,"monomial":10},{"kind":"pure_low","cache":1275,"monomial":15},{"kind":"pure_low","cache":1275,"monomial":11},{"kind":"pure_low","cache":1275,"monomial":12},{"kind":"pure_low","cache":1275,"monomial":9},{"kind":"pure_low","cache":1275,"monomial":5},{"kind":"products","cache":1499,"indices":[5,6,8,14,26,28,44,52,55,58,59,71,85,86,101,103,104,105,106,121,131,134,138,140,152,165,172,175]},{"kind":"pure_low","cache":1499,"monomial":13},{"kind":"products","cache":1631,"indices":[0,1,2,4,7,9,12,15,17,18,20,21,25,29,32,34,36,38,39,49,62,65,72,73,74,75,76,89,97,102,107,110,111,116,127,133,135,143,144,145,146,147,150,162,167,171,176,178]},{"kind":"pure_low","cache":1631,"monomial":6},{"kind":"pure_low","cache":1631,"monomial":14}],"low_computations":11}},{"batch":8,"delta_sha256":"016208281f8389925e1b3b6b5e61e91a25befcc8ee3ca8c37c10b66bdeaa7731","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":4096,"right1":14131925928579927964512,"left2":16384,"right2":38202600107773378429424,"fanout":209447690300},{"left1":64,"right1":62746933004718342161684,"left2":16384,"right2":60356494920513393494596,"fanout":146398705682},{"left1":2,"right1":84300570762769200100266,"left2":576,"right2":67869677522336806046496,"fanout":215333997294},{"left1":32,"right1":9693768117610304427678,"left2":1056,"right2":34532517640912932581124,"fanout":141380742338},{"left1":32,"right1":54681642491610315791076,"left2":16384,"right2":35265556546805157295708,"fanout":988279404246},{"left1":1024,"right1":69206484858968887248848,"left2":2048,"right2":28403575899398936518618,"fanout":896689560618},{"left1":32768,"right1":43776803416826393593008,"left2":4128,"right2":21854263875099644951348,"fanout":43286923320},{"left1":2,"right1":112868647059734393079368,"left2":16450,"right2":74025109903622365467112,"fanout":89760204290},{"left1":16,"right1":51715674216032170806008,"left2":2096,"right2":17388458928953332738468,"fanout":1082580278000},{"left1":2048,"right1":72743336902823575945852,"left2":32770,"right2":65638157844534499227688,"fanout":876289856556},{"left1":64,"right1":28921715827689842926416,"left2":4352,"right2":21800791270225477674922,"fanout":72916800742},{"left1":16448,"right1":70437266465669579529810,"left2":16704,"right2":43754077143010555110672,"fanout":46574392840},{"left1":256,"right1":52605495397761427720072,"left2":16640,"right2":38542186612578312924242,"fanout":845386933450},{"left1":1024,"right1":39614502576962992810144,"left2":16384,"right2":48424703360071831437656,"fanout":52426894860},{"left1":144,"right1":20553025884970350056048,"left2":656,"right2":45397973995708804398518,"fanout":974200387592},{"left1":8,"right1":14761788060110437387774,"left2":648,"right2":43282023822303031882528,"fanout":206876887748},{"left1":1024,"right1":39168530142224314213686,"left2":9232,"right2":6875624416813333918010,"fanout":1052066055230},{"left1":256,"right1":9927388158613750831696,"left2":272,"right2":40165368052463329941080,"fanout":893538388798},{"left1":4,"right1":135373110560122448068754,"left2":512,"right2":44533834833410534501564,"fanout":1005760856288},{"left1":32,"right1":12501878856713711350076,"left2":36896,"right2":74076777481952138970180,"fanout":1005461059078},{"left1":8192,"right1":99043964900346896605508,"left2":17408,"right2":53999194254709839961098,"fanout":163981870826},{"left1":32768,"right1":33541043494712144858448,"left2":37120,"right2":22206638829060029075260,"fanout":137676115138},{"left1":32770,"right1":51866849188651520149854,"left2":33024,"right2":4921593553991328440132,"fanout":78051416770},{"left1":128,"right1":67376407904548738669382,"left2":2048,"right2":56307373147444262439974,"fanout":149489110572},{"left1":8,"right1":150829068709857304776260,"left2":33032,"right2":60106066768989731857772,"fanout":914292534006},{"left1":6144,"right1":61556537854874910526396,"left2":6176,"right2":8968587299941579510492,"fanout":991299947034},{"left1":4096,"right1":38677054268016063493528,"left2":2052,"right2":1984790858950476904498,"fanout":34576326392},{"left1":16384,"right1":54245822462141881557208,"left2":20480,"right2":65879616012111212760824,"fanout":859934687772},{"left1":64,"right1":46031487092701539740872,"left2":1026,"right2":70892848901831456651660,"fanout":834231922428},{"left1":2,"right1":5929563275408689231226,"left2":640,"right2":23430761956009442963798,"fanout":227197850324},{"left1":8,"right1":61869175986399368712754,"left2":2052,"right2":52511966032832904624770,"fanout":223409009666},{"left1":128,"right1":68912834745175137782822,"left2":512,"right2":44699306637255070030922,"fanout":841994655936},{"left1":16384,"right1":56324910507925626104964,"left2":4128,"right2":422001962973798300958,"fanout":974133141560},{"left1":12,"right1":34568054556902892912164,"left2":76,"right2":11687053842926065795126,"fanout":827867843614},{"left1":2080,"right1":39893707942368200833046,"left2":2336,"right2":69606782309043382872512,"fanout":871344116770},{"left1":8,"right1":127504072943566388316114,"left2":33032,"right2":74605926387680955918804,"fanout":261405066290},{"left1":4,"right1":26249090638678786181634,"left2":1026,"right2":17956739302114947531386,"fanout":150191085568},{"left1":2,"right1":32605092908420992354938,"left2":6,"right2":91337520923749489368948,"fanout":21898456015},{"left1":32,"right1":48904432984513699802944,"left2":8200,"right2":32500892513077209477334,"fanout":919942779950},{"left1":4096,"right1":61886639917759939615758,"left2":8200,"right2":22338405011166401412324,"fanout":55614039040},{"left1":1024,"right1":21764578388450490226906,"left2":8192,"right2":20234190941869160282302,"fanout":966212723414},{"left1":32768,"right1":50204453602024409174586,"left2":33408,"right2":69592202691659439088692,"fanout":258754925606},{"left1":128,"right1":58885217562603157178518,"left2":4256,"right2":19385077102640192598942,"fanout":157841419466},{"left1":512,"right1":14784078813558571703670,"left2":2048,"right2":52861270445008408462194,"fanout":885584231628},{"left1":256,"right1":70803751035805350713780,"left2":144,"right2":14188279729094440505828,"fanout":120497823434},{"left1":8,"right1":132118751796140572756272,"left2":576,"right2":38429091549968399587772,"fanout":137586934490},{"left1":64,"right1":15233034683258262219536,"left2":2048,"right2":4839678398068841745018,"fanout":827938490052},{"left1":16,"right1":257272191230769003768,"left2":20,"right2":35605762802865188784446,"fanout":859467102941},{"left1":8,"right1":131192081411475886501068,"left2":64,"right2":35792206906185478185594,"fanout":149968271570},{"left1":16,"right1":49875892537079594434022,"left2":18,"right2":69892586584644985955568,"fanout":30934237990},{"left1":32,"right1":74119311452415671620836,"left2":17472,"right2":74448727019995347393852,"fanout":123998052380},{"left1":4096,"right1":26110850565882058769602,"left2":5120,"right2":45096462377459951504324,"fanout":850954302676},{"left1":32770,"right1":107918721960719151987024,"left2":40962,"right2":131490862904366079352914,"fanout":46910939856},{"left1":32,"right1":4449545688149728495004,"left2":8200,"right2":63550686167637501763836,"fanout":854632440848},{"left1":32768,"right1":6341749082635043098666,"left2":8208,"right2":69676875948232145417450,"fanout":910985057338},{"left1":32,"right1":89995631501247880808628,"left2":44,"right2":78396236257144108930512,"fanout":997339961560},{"left1":64,"right1":70075823593260569364664,"left2":2080,"right2":19654068377406068616292,"fanout":953663232566},{"left1":128,"right1":67475886587802409771052,"left2":32896,"right2":37944338224931933932878,"fanout":905916894442},{"left1":8,"right1":78284257401360744455938,"left2":17408,"right2":61865792887460548119288,"fanout":880268156098},{"left1":32768,"right1":49371563826258842938278,"left2":33024,"right2":60054072552223211166634,"fanout":18236762308},{"left1":2048,"right1":68775737009068822868250,"left2":4128,"right2":24173728431498607531524,"fanout":1048357821648},{"left1":17408,"right1":114086047416244618070302,"left2":17416,"right2":109789372889511047009800,"fanout":26375409898},{"left1":8192,"right1":35304627996166585883936,"left2":8256,"right2":49249264985022303170468,"fanout":828574464750},{"left1":17408,"right1":1234748486125677268930,"left2":17440,"right2":8069792624606572200442,"fanout":825674626294},{"left1":33024,"right1":64782190608098651916222,"left2":37120,"right2":52926705437434344434708,"fanout":991983887554},{"left1":256,"right1":58558944972393280143636,"left2":272,"right2":25984068700531548926970,"fanout":17852907759},{"left1":8200,"right1":149049764048881918739180,"left2":12296,"right2":24441356377837104864166,"fanout":21155999452},{"left1":8192,"right1":83233151418747454398008,"left2":33024,"right2":603593344139906625736,"fanout":141399690760},{"left1":16,"right1":39083832083001673083590,"left2":1042,"right2":53340019220521273251350,"fanout":43434560750},{"left1":2048,"right1":147958161869401372162046,"left2":2052,"right2":101432704660274479690912,"fanout":163713303236},{"left1":4,"right1":112608057273218517579946,"left2":1088,"right2":52744579985404814398652,"fanout":868236189906},{"left1":144,"right1":30500757291034556862892,"left2":2192,"right2":24741266794356907389342,"fanout":223622871752},{"left1":33024,"right1":144127214019010170690596,"left2":41216,"right2":78494849010636421923506,"fanout":257902515760},{"left1":64,"right1":44391819769948570009902,"left2":128,"right2":48175328079977443868550,"fanout":111706515498},{"left1":2,"right1":139878987678300808503456,"left2":514,"right2":710421568934185166026,"fanout":842351966936},{"left1":256,"right1":7535136639768798780494,"left2":4128,"right2":16358049648815355481728,"fanout":252867907286},{"left1":16,"right1":139324195288885629739700,"left2":18,"right2":137451632362249439977452,"fanout":60183953096},{"left1":33024,"right1":68197441629306531821702,"left2":33152,"right2":50291305083906620741692,"fanout":851122201092},{"left1":16384,"right1":35155985279483499566244,"left2":16640,"right2":70251759906688020381514,"fanout":26610115134},{"left1":2052,"right1":53074358248078178341186,"left2":18436,"right2":34643057972614950948044,"fanout":948719381534},{"left1":8200,"right1":16459893393459792790788,"left2":576,"right2":30204779733122025320220,"fanout":215151162080},{"left1":1024,"right1":40703617891705574662574,"left2":1056,"right2":50452671331516160186126,"fanout":172808913950},{"left1":4096,"right1":17045316388949893218674,"left2":20480,"right2":45353563862707685998566,"fanout":897560924174},{"left1":512,"right1":11199911790509865069780,"left2":544,"right2":7245628796202318682984,"fanout":85970174988},{"left1":1026,"right1":27385275934577974701130,"left2":17410,"right2":3225056124496745725006,"fanout":34742603282},{"left1":8192,"right1":87710055027990636030874,"left2":2080,"right2":62182886415995502421406,"fanout":1000024505032},{"left1":256,"right1":12699210682787336831372,"left2":896,"right2":35315493356335013988596,"fanout":252445080092},{"left1":12,"right1":68035549063783465489558,"left2":8208,"right2":3771623269295188151260,"fanout":1040000140516},{"left1":12,"right1":35918542447526173802418,"left2":8204,"right2":32226034727880652967626,"fanout":232200398072},{"left1":16384,"right1":19671758615316224785518,"left2":16386,"right2":24060282536665910724920,"fanout":138445993516},{"left1":32768,"right1":20705270768996451949504,"left2":4352,"right2":39825299534760886612000,"fanout":86215628504},{"left1":4,"right1":65964740289060361241100,"left2":32772,"right2":47221713000028923865266,"fanout":146972486206},{"left1":16,"right1":28798985602785035281612,"left2":18,"right2":68712139228148765430276,"fanout":30284656632},{"left1":192,"right1":35457581553064745024984,"left2":640,"right2":74153271142404166590976,"fanout":249358239458},{"left1":32768,"right1":71815750293629000454574,"left2":2080,"right2":39476202763720689577292,"fanout":120810643512},{"left1":64,"right1":33979353689357861438604,"left2":128,"right2":30064532766051492934590,"fanout":80813947964},{"left1":128,"right1":59977993457799919272594,"left2":144,"right2":57212062267130533861664,"fanout":80845921800},{"left1":32768,"right1":47205762418104436211016,"left2":32800,"right2":34430146301104590714506,"fanout":979859470538},{"left1":8,"right1":131907349304093458259940,"left2":1034,"right2":51436268847990363554586,"fanout":115608315598},{"left1":256,"right1":27788409148596739504788,"left2":16386,"right2":72894419787402011553562,"fanout":172773477084},{"left1":8,"right1":77377993138786780300228,"left2":512,"right2":34883133584215036998722,"fanout":893789722156},{"left1":2,"right1":87064145841603888032042,"left2":17410,"right2":66939013563623582742056,"fanout":1043223463624},{"left1":8192,"right1":77012013302699551802146,"left2":10272,"right2":40455183581709028891842,"fanout":266389161716},{"left1":32,"right1":59360088840976683531208,"left2":544,"right2":18632547362908994519460,"fanout":841813858836},{"left1":16,"right1":69351891488726129505832,"left2":16464,"right2":70028634564861172750366,"fanout":931540562464},{"left1":16384,"right1":75295721323630088660054,"left2":18432,"right2":37690201274460100273754,"fanout":81003809508},{"left1":2048,"right1":5330755582939108074756,"left2":10248,"right2":59113876224213801368444,"fanout":953496767686},{"left1":32768,"right1":4194727845482927826618,"left2":288,"right2":31563352361423073393558,"fanout":888691243056},{"left1":256,"right1":20768721403902989319662,"left2":16704,"right2":6826404023217685311598,"fanout":919393324246},{"left1":64,"right1":4527736908743585193658,"left2":2116,"right2":5512876360309037473804,"fanout":1082413558824},{"left1":16384,"right1":70892317015584775826876,"left2":16640,"right2":9294068459431349959584,"fanout":999989628152},{"left1":8192,"right1":78188493108613011529046,"left2":36864,"right2":3058584344962991671654,"fanout":215199829558},{"left1":16,"right1":70884485493234582298004,"left2":656,"right2":69557976952650984658096,"fanout":29328330250},{"left1":4,"right1":20283190991956970925254,"left2":6,"right2":19826904256458344385638,"fanout":13351984686},{"left1":32768,"right1":25664875474463713833660,"left2":40968,"right2":42821449764096472205558,"fanout":824884648498},{"left1":512,"right1":29673545384465954419428,"left2":32768,"right2":49936196339203859783372,"fanout":232310581782},{"left1":16448,"right1":122097778219432999686624,"left2":16450,"right2":130411353311218583955022,"fanout":223641402612},{"left1":256,"right1":1137364771708752583556,"left2":320,"right2":63537670319747162738208,"fanout":25940640986},{"left1":16,"right1":33769833197401860111484,"left2":8196,"right2":133676567596189243216256,"fanout":121202681062},{"left1":4096,"right1":18853252281691369350880,"left2":144,"right2":29068184069759015203032,"fanout":72625871380},{"left1":1024,"right1":64039141705269418098602,"left2":1048,"right2":69971435895211531650730,"fanout":1051711694880},{"left1":1024,"right1":15735303944265320139114,"left2":8192,"right2":148085888874973174195878,"fanout":86640830012},{"left1":8192,"right1":39964126453281066491810,"left2":144,"right2":8727055024190134157660,"fanout":1008895663150},{"left1":16384,"right1":30208293344104623111836,"left2":16400,"right2":16297302495092749684228,"fanout":132689439940},{"left1":32770,"right1":8663261444727463468786,"left2":2080,"right2":24092961561746913012770,"fanout":224245853404},{"left1":64,"right1":15995759991505424399370,"left2":17408,"right2":41425884966084691109612,"fanout":129186283240},{"left1":512,"right1":11094791451447365437848,"left2":32770,"right2":64432698892428492518808,"fanout":885584696040},{"left1":1024,"right1":55758739726215926044580,"left2":3072,"right2":19900149744887331292386,"fanout":854667034660},{"left1":4352,"right1":38327524215172945844222,"left2":36864,"right2":13991466510795147496244,"fanout":157922501676},{"left1":512,"right1":32572242604732117209062,"left2":33282,"right2":75417843793063920342106,"fanout":1005371916320},{"left1":1024,"right1":42421710700283473649956,"left2":8208,"right2":21467166295550951007416,"fanout":166446153778},{"left1":144,"right1":66125716877902924773392,"left2":2192,"right2":862904208871741287316,"fanout":931788228860},{"left1":2,"right1":55901770032410096148912,"left2":16,"right2":13049733996558176255476,"fanout":86020533714},{"left1":1024,"right1":2381596154159194924546,"left2":17408,"right2":49497021913831652854712,"fanout":138458837232},{"left1":128,"right1":69801314518711008103174,"left2":4224,"right2":41768886554623124202380,"fanout":8974248680},{"left1":17408,"right1":74479139949477570682184,"left2":17472,"right2":22688715391873760416898,"fanout":880246192364},{"left1":512,"right1":72282068631304741346046,"left2":8192,"right2":285724568934614090878,"fanout":885180914428},{"left1":4,"right1":106546874908065626566958,"left2":256,"right2":68543838302788247557710,"fanout":50986967},{"left1":4096,"right1":35861354835328307189742,"left2":20482,"right2":66096290186773596119478,"fanout":72658376914},{"left1":4,"right1":134071445069268103634634,"left2":32774,"right2":14544739762615952743468,"fanout":876745997364},{"left1":2,"right1":75106527200083322257824,"left2":128,"right2":30176785645433255839706,"fanout":1031297089590},{"left1":8,"right1":55737273346445924355218,"left2":1032,"right2":15157700685077678632422,"fanout":180737289758},{"left1":144,"right1":71048978404039149522222,"left2":176,"right2":65846323750302522034098,"fanout":871108185336},{"left1":2052,"right1":26914568893109475596142,"left2":2308,"right2":53644959951190102369488,"fanout":845370899156},{"left1":4096,"right1":4024750207134028679496,"left2":4672,"right2":54796420155344629277906,"fanout":124435220216},{"left1":8192,"right1":16127078827140978253974,"left2":6176,"right2":61174178187526042836678,"fanout":914238745656},{"left1":32,"right1":6933180342036906678924,"left2":160,"right2":74179752591526427003692,"fanout":880337886218},{"left1":8,"right1":42524808589618829632160,"left2":128,"right2":9807637931776727152474,"fanout":103662370026},{"left1":128,"right1":5496842209797018741610,"left2":2048,"right2":15477088162830744859786,"fanout":147003341044},{"left1":2,"right1":89762039184225696907762,"left2":16,"right2":5030210126013644374386,"fanout":838154832888},{"left1":32768,"right1":69685564766506375864782,"left2":36864,"right2":49152654910273512020746,"fanout":172808781856},{"left1":8,"right1":11433072410621941662252,"left2":576,"right2":36352287823785324782470,"fanout":103328695032},{"left1":1024,"right1":5434348924159503884336,"left2":12,"right2":350147167104526762442,"fanout":863269947930},{"left1":8,"right1":4084916715927953029876,"left2":32778,"right2":7227086610901424126900,"fanout":1069059734738},{"left1":256,"right1":26148767242006941093048,"left2":512,"right2":71012177480161071543370,"fanout":902147477206},{"left1":4,"right1":76461838951118805579666,"left2":32,"right2":62086806882490478712218,"fanout":1034700458510},{"left1":8208,"right1":3011460269963197297892,"left2":8216,"right2":71772137252813328371830,"fanout":939973023964},{"left1":256,"right1":130644359078373248577174,"left2":260,"right2":113811925811744556984596,"fanout":4848615189},{"left1":12,"right1":91600640140853824621126,"left2":28,"right2":7662745631121383199772,"fanout":210117853892},{"left1":256,"right1":58827344411094441097840,"left2":1282,"right2":55787915058634144365042,"fanout":71976142850},{"left1":32,"right1":73362410767545651367830,"left2":32770,"right2":61197776476437134386966,"fanout":244343567898},{"left1":512,"right1":13237888441728305379020,"left2":576,"right2":25319263276689158185646,"fanout":137521661444},{"left1":576,"right1":79065393986804732792680,"left2":8768,"right2":139255309227017142983488,"fanout":859174916110},{"left1":2,"right1":53153423248954903467908,"left2":6,"right2":107690822813861709289038,"fanout":46490139328},{"left1":4,"right1":3059177920135594378480,"left2":16384,"right2":68989508045589844249390,"fanout":47080672270},{"left1":2052,"right1":30191509219854501576546,"left2":2068,"right2":62582973940021839770800,"fanout":189146865882},{"left1":8,"right1":33355586073434616239346,"left2":16448,"right2":62316521972249384495760,"fanout":862550681332},{"left1":4352,"right1":51351739251673324586780,"left2":6400,"right2":8954170544529050578044,"fanout":1051226805488},{"left1":4096,"right1":72716368920003791503946,"left2":12416,"right2":66531643932474930085218,"fanout":1094458470096},{"left1":32768,"right1":39010671485194880468738,"left2":8200,"right2":140215481401439148947730,"fanout":1085588296740},{"left1":2,"right1":17501565766366942892354,"left2":16,"right2":47057705867848903744264,"fanout":159300635125},{"left1":4,"right1":100934888968165245667348,"left2":592,"right2":42487126067604265468456,"fanout":9097066218},{"left1":2,"right1":113631323418102285618490,"left2":2052,"right2":61220754024060234132938,"fanout":1031845461720},{"left1":4096,"right1":58094308515689835687338,"left2":640,"right2":24110859987425099827690,"fanout":261357449460},{"left1":1024,"right1":60972515231227184821750,"left2":1026,"right2":8894220390524298428144,"fanout":86270869200},{"left1":128,"right1":16345869423274583818348,"left2":8336,"right2":72547643537362692578120,"fanout":902560616510},{"left1":128,"right1":56363153422390574189220,"left2":8320,"right2":35978337085482785895846,"fanout":839137840},{"left1":256,"right1":48816848297359582838692,"left2":1280,"right2":60684042727617771869632,"fanout":1008796566220},{"left1":144,"right1":30529713250608529228358,"left2":208,"right2":14987564363888520106832,"fanout":906116727026},{"left1":4,"right1":142870767559727615194580,"left2":32772,"right2":17922501343434955107084,"fanout":207112028210},{"left1":16,"right1":36190783980994795459506,"left2":20,"right2":3602519252239648381640,"fanout":5283114732},{"left1":8192,"right1":78842664522350378292362,"left2":8832,"right2":63537773718848799537532,"fanout":1017251767844},{"left1":4,"right1":71362412832922825338342,"left2":1028,"right2":46529378343455769206492,"fanout":158091446818},{"left1":6144,"right1":125185842581518678850190,"left2":6152,"right2":134079535926864335296268,"fanout":46375645214},{"left1":8192,"right1":79316321900872036274208,"left2":8204,"right2":140131231154629000438490,"fanout":106414140940},{"left1":17408,"right1":16804075686154118041920,"left2":17664,"right2":62769702224788206297832,"fanout":868323829270},{"left1":4,"right1":51087888097968725522288,"left2":20,"right2":55885887167295231946248,"fanout":7584797654},{"left1":4,"right1":61844013246305464847448,"left2":20,"right2":24750450535134238312492,"fanout":51926134785},{"left1":4,"right1":117818871375906377374400,"left2":1030,"right2":64981662946022268920850,"fanout":115162087980},{"left1":4096,"right1":8091050390439286387340,"left2":4224,"right2":30598930495166063846266,"fanout":3402547774},{"left1":64,"right1":15439890758159589416128,"left2":2080,"right2":58254514618956539526412,"fanout":982554043100},{"left1":8,"right1":103007235804594360045316,"left2":4352,"right2":58533994894509234802498,"fanout":842733729334},{"left1":2,"right1":127961368401443060691928,"left2":256,"right2":7531917969134976519586,"fanout":824786946610},{"left1":4096,"right1":11807969458774091753486,"left2":2052,"right2":44167121626995859612698,"fanout":163289899262},{"left1":64,"right1":57737913728750287898246,"left2":512,"right2":21984203380082590284958,"fanout":60815708384},{"left1":4,"right1":17209822491471409530888,"left2":2052,"right2":30954446604545333227958,"fanout":43689061616},{"left1":16448,"right1":41229061463648386245938,"left2":16464,"right2":8279702166519883819766,"fanout":1082749026010},{"left1":32768,"right1":65768130207155326486964,"left2":2052,"right2":40118126561545386231432,"fanout":914627231936},{"left1":512,"right1":114680985791278652433834,"left2":514,"right2":78204697356093128434028,"fanout":877160436984},{"left1":640,"right1":73936569356087678492624,"left2":2688,"right2":58655299483792720227446,"fanout":115430050024},{"left1":4096,"right1":61115171028930673034312,"left2":4224,"right2":39162890451767997045514,"fanout":945432434232},{"left1":512,"right1":54113002961911997358474,"left2":4352,"right2":47941304224271023333894,"fanout":98046895142},{"left1":2048,"right1":6533991284626987473320,"left2":10256,"right2":2389278820130349534008,"fanout":991244201674}],"pure":{"16":825157601342,"64":198391944763,"80":56051300844,"96":21294342957,"112":1057063517165,"144":171026486755,"160":172203304189,"176":205269625651,"208":17535798231,"240":64976403399,"256":881021694461,"272":29846922737,"304":154634566124,"336":5032644334,"352":1027289175501,"368":63636639801,"400":42412154597,"416":29344672482,"496":3336975053,"544":253316956711,"560":158513171252,"608":1012709053205,"640":146199487508,"848":230885104882,"896":971571646414,"944":849883621125,"976":163963130834,"1008":22430019132,"1088":864061428518,"1120":228675487034,"1168":206545222894,"1184":836649290792,"0":1051700425730,"1":167718811161,"2":91101783283,"3":1039433070021,"4":894359775503,"5":962811528971,"8":34847844908,"10":1052533645554,"12":871775533031,"15":89725131828,"128":893402426608,"192":931686584297,"288":842655255094,"320":1039717421857,"384":141648587826,"448":825387130138,"464":824965255226,"512":1022369125603,"592":876158322172,"624":936688076804,"656":872519693831,"688":270764088872,"752":855521031423,"768":936487748134,"784":209633133330,"800":855172432688,"816":81872499165,"832":910469236210,"1024":63718490904,"1040":136398426401,"1216":944240007642,"7":163694652612,"9":115191637746,"48":1051981365820,"528":898670735156,"576":1095969667797,"672":1095165799889,"736":110760815315,"864":1047720350774,"880":253121202888,"960":853740212974,"992":1074006201566,"1072":21539574747,"1104":1034184554759,"1200":154940336683,"11":982856369700,"14":72346381536,"32":115699470382,"480":889459968235,"704":59090985709,"1056":978443101426,"1152":150491681031,"6":249460939238,"720":33920056527,"1136":879952278805,"224":3643403027,"928":1061409199843,"912":927934762981,"432":992403665437,"13":158357918742},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"three-phase","capacity":8,"product_order":[0,1,3,4,5,7,8,10,11,12,13,16,17,20,25,26,27,28,30,32,33,34,36,37,38,39,40,46,47,48,49,50,51,53,55,56,58,60,61,62,63,65,66,68,69,70,75,76,78,79,81,82,84,85,87,88,89,92,98,99,101,102,104,105,106,108,109,110,113,116,117,118,120,121,123,125,127,130,132,133,135,137,138,141,143,145,149,152,155,156,157,158,159,163,164,165,166,167,170,172,174,177,180,182,183,184,185,186,187,188,190,191,192,193,195,196,202,2,14,15,18,23,29,31,42,43,44,45,71,73,74,80,83,86,93,95,96,100,103,112,119,122,131,134,136,140,142,144,146,147,148,151,154,161,162,168,171,173,175,176,178,181,189,194,198,199,200,201,6,9,19,21,22,24,35,41,52,54,57,59,64,67,72,77,90,91,94,97,107,111,114,115,124,126,128,129,139,150,153,160,169,179,197],"events":[{"kind":"products","cache":999,"indices":[0,1,3,4,5,7,8,10,11,12,13,16,17,20,25,26,27,28,30,32,33,34,36,37,38,39,40,46,47,48,49,50,51,53,55,56,58,60,61,62,63,65,66,68,69,70,75,76,78,79,81,82,84,85,87,88,89,92,98,99,101,102,104,105,106,108,109,110,113,116,117,118,120,121,123,125,127,130,132,133,135,137,138,141,143,145,149,152,155,156,157,158,159,163,164,165,166,167,170,172,174,177,180,182,183,184,185,186,187,188,190,191,192,193,195,196,202]},{"kind":"pure_low","cache":999,"monomial":3},{"kind":"pure_low","cache":999,"monomial":5},{"kind":"pure_low","cache":999,"monomial":10},{"kind":"pure_low","cache":999,"monomial":12},{"kind":"pure_low","cache":999,"monomial":11},{"kind":"pure_low","cache":999,"monomial":14},{"kind":"pure_low","cache":999,"monomial":6},{"kind":"pure_low","cache":999,"monomial":13},{"kind":"products","cache":479,"indices":[2,14,15,18,23,29,31,42,43,44,45,71,73,74,80,83,86,93,95,96,100,103,112,119,122,131,134,136,140,142,144,146,147,148,151,154,161,162,168,171,173,175,176,178,181,189,194,198,199,200,201]},{"kind":"pure_low","cache":479,"monomial":7},{"kind":"pure_low","cache":479,"monomial":9},{"kind":"products","cache":1499,"indices":[6,9,19,21,22,24,35,41,52,54,57,59,64,67,72,77,90,91,94,97,107,111,114,115,124,126,128,129,139,150,153,160,169,179,197]},{"kind":"pure_low","cache":1499,"monomial":15}],"low_computations":11}},{"batch":9,"delta_sha256":"d562b4ece3fc58f33d02d048cf91418d7856bb0c90fcd1e4fee8961b9616e755","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":4104,"right1":117616131781020807597230,"left2":320,"right2":70938117187186587400248,"fanout":927716398296},{"left1":16,"right1":16651977607149474709810,"left2":272,"right2":27066242536110466132340,"fanout":862369751584},{"left1":64,"right1":2784070970490164361528,"left2":17472,"right2":44534447281537551361234,"fanout":86583073840},{"left1":2,"right1":102136821490078890796002,"left2":258,"right2":12099929192003566930146,"fanout":111233135891},{"left1":16392,"right1":42591579876820477702720,"left2":18688,"right2":21360160232345718081406,"fanout":1000059962376},{"left1":4,"right1":120531737299418260564482,"left2":260,"right2":67435354944976668219254,"fanout":34173696991},{"left1":8328,"right1":78019748646166990890086,"left2":24704,"right2":137845269362637075344634,"fanout":914090684440},{"left1":1024,"right1":6304271475974624376802,"left2":1152,"right2":3156127411652248020236,"fanout":845116353568},{"left1":4,"right1":28450631602146719351334,"left2":6,"right2":61697055263756782068056,"fanout":163679835428},{"left1":8192,"right1":63061476303898251776884,"left2":4132,"right2":66191168491365510575660,"fanout":1034014740528},{"left1":16384,"right1":136116022774837408907780,"left2":16432,"right2":57974702817680800438246,"fanout":979254647836},{"left1":256,"right1":43786953354607335401582,"left2":272,"right2":45646313399505001601034,"fanout":138096145207},{"left1":4096,"right1":74450367499070739320556,"left2":4128,"right2":49803839056970341653008,"fanout":209862317264},{"left1":32768,"right1":541904840313442400502,"left2":48,"right2":8504001227483600279896,"fanout":996500058304},{"left1":17408,"right1":1509003683954517684282,"left2":21504,"right2":26205436288922088236462,"fanout":123829803240},{"left1":16,"right1":51264589974085843617454,"left2":20,"right2":34404472280248900618976,"fanout":145944817886},{"left1":16384,"right1":135911822271843468770844,"left2":24578,"right2":73193157414502714927566,"fanout":859678187524},{"left1":17408,"right1":88343131409526852888076,"left2":17412,"right2":98405271620151471968212,"fanout":172686317620},{"left1":256,"right1":46407140769589052718764,"left2":272,"right2":43896747000609831390238,"fanout":34944005389},{"left1":2048,"right1":20189876295701616844206,"left2":24578,"right2":40840239554130128786328,"fanout":52549919948},{"left1":2,"right1":122271525127932892144602,"left2":4,"right2":120557863520709784015372,"fanout":64255894767},{"left1":520,"right1":56205746458767067997262,"left2":2816,"right2":10278525205044169366504,"fanout":55501666364},{"left1":192,"right1":47193455412242940429372,"left2":16576,"right2":61252951347046935270904,"fanout":38187040816},{"left1":8194,"right1":71668314911206494032122,"left2":40962,"right2":2642298393715678090902,"fanout":104033166384},{"left1":4,"right1":91914692656222456355928,"left2":260,"right2":13705938767284071912444,"fanout":72341901036},{"left1":16,"right1":61667555119849038970538,"left2":8210,"right2":72566767499749699568752,"fanout":1048914432244},{"left1":32768,"right1":28347537074643647796238,"left2":4100,"right2":138625036084698797296348,"fanout":842568307936},{"left1":2048,"right1":31424197412551024865720,"left2":4132,"right2":8324674969082391096148,"fanout":158054800408},{"left1":8,"right1":74899325290312083245344,"left2":320,"right2":56276311162402888891356,"fanout":845573056512},{"left1":8,"right1":27320139452606212866610,"left2":36,"right2":29963661629063933372668,"fanout":223753584836},{"left1":16,"right1":59217411156568058294602,"left2":18,"right2":28752217742765234518914,"fanout":141663335375},{"left1":128,"right1":16470266865863876846150,"left2":4228,"right2":2093373404386289639604,"fanout":931806632972},{"left1":64,"right1":8947315241017507333464,"left2":2048,"right2":61444277875081032708796,"fanout":1014229164284},{"left1":8192,"right1":59604118316906973251390,"left2":9216,"right2":3522995992061400904366,"fanout":17650951220},{"left1":32768,"right1":50497258974505213139592,"left2":36896,"right2":72075964377312370268070,"fanout":842153081912},{"left1":64,"right1":37731061443704868324718,"left2":1024,"right2":17508967450083983982960,"fanout":842767007952},{"left1":2112,"right1":91748227479841575289756,"left2":2120,"right2":78794302537419455898438,"fanout":979468563680},{"left1":33280,"right1":104213704861501284123540,"left2":37376,"right2":146345608698302569360314,"fanout":880146696196},{"left1":16,"right1":40784628359485657709380,"left2":256,"right2":25135619853179020513000,"fanout":9123332076},{"left1":8192,"right1":30984063557782991719968,"left2":12,"right2":103496184141485415017628,"fanout":124231363776},{"left1":16,"right1":126845086773355890865854,"left2":18,"right2":120717466384871175872194,"fanout":34816203029},{"left1":32784,"right1":117285469283322784632648,"left2":36880,"right2":77982083940592049133024,"fanout":69337661644},{"left1":4128,"right1":14202446390644317431216,"left2":36896,"right2":59064448195323146695990,"fanout":37594919960},{"left1":4,"right1":60193494948119897785046,"left2":16,"right2":69472267474605792784310,"fanout":26338118611},{"left1":8,"right1":100272019600874039094208,"left2":32816,"right2":5333299470775495641436,"fanout":980195014716},{"left1":16384,"right1":15181621777020822940122,"left2":9218,"right2":9720012803143409381808,"fanout":1086170864856},{"left1":4100,"right1":106265242035939835444206,"left2":4136,"right2":61112622998857573244916,"fanout":1000663687228},{"left1":8192,"right1":62124545075921157215326,"left2":41472,"right2":44487231008754004138160,"fanout":1052081446964},{"left1":256,"right1":7257366896441433437876,"left2":260,"right2":12184401174445499603310,"fanout":155376365360},{"left1":512,"right1":68992279496531031857802,"left2":4096,"right2":36670412376288916860002,"fanout":37648602340},{"left1":2,"right1":66129703281583454451118,"left2":512,"right2":64187118412448096741666,"fanout":51687464160},{"left1":512,"right1":81328207846981680201668,"left2":524,"right2":130402488050682544506476,"fanout":1031600927764},{"left1":1024,"right1":25575514096917310780214,"left2":1088,"right2":68185224424266454850254,"fanout":962758976732},{"left1":8,"right1":62739890459412596579468,"left2":1026,"right2":5311959610920073160486,"fanout":140727288848},{"left1":1024,"right1":49711519301479266782670,"left2":5120,"right2":61447827873362296874982,"fanout":141132307668},{"left1":32,"right1":72772580318869859945662,"left2":128,"right2":44437748417730555699470,"fanout":107107897556},{"left1":1026,"right1":34735027697083342126680,"left2":5122,"right2":12797761777031153741428,"fanout":17329563840},{"left1":4,"right1":2109358417785220482836,"left2":6,"right2":53327487317957142618598,"fanout":966735049674},{"left1":8,"right1":56144905864004761108510,"left2":32816,"right2":69373579804888828144198,"fanout":1000191092980},{"left1":128,"right1":34374695825761387947968,"left2":4108,"right2":27565558899947835409602,"fanout":158322928832},{"left1":8,"right1":147173434733943274223590,"left2":32800,"right2":48381331008132183484798,"fanout":859062405316},{"left1":32,"right1":16308077897429686552250,"left2":4096,"right2":13607668768293869058440,"fanout":1017316908076},{"left1":192,"right1":86303548051488414505056,"left2":16576,"right2":113806059937819245271078,"fanout":189987563712},{"left1":4100,"right1":32497323981860164598422,"left2":9216,"right2":61746490082672576216602,"fanout":69057176608},{"left1":64,"right1":54855896493347085435086,"left2":17472,"right2":46125523452681364397360,"fanout":124366626868},{"left1":64,"right1":62402734053496057658596,"left2":17472,"right2":70041678249767385788728,"fanout":842430734340},{"left1":2,"right1":58184887502982144818642,"left2":256,"right2":72921239699255169519670,"fanout":841625232680},{"left1":16,"right1":83994915608935262869908,"left2":20,"right2":90997518596604603786446,"fanout":824868416318},{"left1":2304,"right1":22726915992356512667964,"left2":10496,"right2":251601542643988669112,"fanout":963082502160},{"left1":32,"right1":61218404359130824010326,"left2":1024,"right2":40355585688854591374028,"fanout":928654307344},{"left1":128,"right1":26778300383775925402964,"left2":320,"right2":41664997572058957727952,"fanout":931340803128},{"left1":4,"right1":32685551996275335362938,"left2":6,"right2":103776154724565155891112,"fanout":213675288874},{"left1":4100,"right1":45438225433657023336640,"left2":6148,"right2":17037651887211923836588,"fanout":223607792872},{"left1":32768,"right1":12802370188243469880622,"left2":32800,"right2":28135368813863266529036,"fanout":825442747440},{"left1":320,"right1":72105925159441944467900,"left2":448,"right2":14406663974441823481470,"fanout":876912063532},{"left1":32768,"right1":75462471665422303864572,"left2":520,"right2":12069412074640739053890,"fanout":141347842252},{"left1":8,"right1":101850135959219718854936,"left2":4136,"right2":29791525661176287545956,"fanout":210399719648},{"left1":4104,"right1":88132168788166449580794,"left2":4608,"right2":36814122593154603983190,"fanout":979322022972},{"left1":16,"right1":23913351679109404236528,"left2":272,"right2":26810178713269175608164,"fanout":22294494427},{"left1":512,"right1":72723463396811946928028,"left2":33280,"right2":25467261111924519841168,"fanout":876458282204},{"left1":32768,"right1":21025358539248486398190,"left2":41088,"right2":45753068018607982274872,"fanout":996985021660},{"left1":4100,"right1":98103123595385637725278,"left2":5124,"right2":20313769652414066695820,"fanout":34441537584},{"left1":2112,"right1":97210108535010519416188,"left2":2120,"right2":95505047155779474535596,"fanout":897060763708},{"left1":8194,"right1":27100391218287835712458,"left2":8322,"right2":51033071957644005769676,"fanout":86436528352},{"left1":8194,"right1":109198141975578326592102,"left2":12290,"right2":99745456524146084752596,"fanout":1052134617124},{"left1":16384,"right1":54374485919998740857430,"left2":36,"right2":30278299509086457504744,"fanout":226842698804},{"left1":20480,"right1":113228092372214620866086,"left2":21504,"right2":60640051248336981117456,"fanout":1017775177968},{"left1":8,"right1":114626585540491160815584,"left2":2568,"right2":26166904353472150563996,"fanout":72627578932},{"left1":2560,"right1":50799221205681289728526,"left2":35328,"right2":48336815476685097935296,"fanout":192618705144},{"left1":16384,"right1":34803144451821522657700,"left2":16704,"right2":24285401332940981642510,"fanout":210063376584},{"left1":64,"right1":40994808027896518657048,"left2":2304,"right2":17126232222494609908882,"fanout":965901874196},{"left1":64,"right1":139283724521176810230330,"left2":4168,"right2":86296505634653923752536,"fanout":124220130536},{"left1":2,"right1":139962425620259320127218,"left2":258,"right2":12610805447376701952806,"fanout":34625232589},{"left1":32784,"right1":27566430745156058749454,"left2":34832,"right2":45364548383237973526600,"fanout":1048322031648},{"left1":4,"right1":3580824356009323261948,"left2":16,"right2":2450286770310165011194,"fanout":42615641555},{"left1":32,"right1":66695163656012270692616,"left2":64,"right2":31011400515695258890428,"fanout":997441468456},{"left1":2048,"right1":31459524649493332419266,"left2":35328,"right2":13706416525353425972530,"fanout":1086224334856},{"left1":2,"right1":54770993814018111321290,"left2":514,"right2":10348993003297280286248,"fanout":72883166460},{"left1":33280,"right1":37131193130689832013546,"left2":33408,"right2":30358862662297622875568,"fanout":1082466507988},{"left1":4,"right1":77224258296624150428438,"left2":16,"right2":5142132645689378315004,"fanout":862956210481},{"left1":20,"right1":10026985470303198678002,"left2":276,"right2":58911151953967370595502,"fanout":969755458023},{"left1":512,"right1":28731939288230868806162,"left2":8192,"right2":50577982182390469809846,"fanout":193072211996},{"left1":16,"right1":1336479414132734132476,"left2":256,"right2":39579298068612123281038,"fanout":20872110315},{"left1":16,"right1":20057859711972999671914,"left2":272,"right2":64401813440669373874980,"fanout":35027019972},{"left1":8192,"right1":15232397507276644502582,"left2":9216,"right2":13669733580741508492284,"fanout":979533831224},{"left1":16384,"right1":74928523630669521101772,"left2":4132,"right2":31666747431298241884510,"fanout":86772290620},{"left1":8320,"right1":102376743842744505711838,"left2":8328,"right2":146385400986666073066482,"fanout":106584872960},{"left1":32,"right1":61749317580477891755092,"left2":2048,"right2":4193730257041037266578,"fanout":189920453672},{"left1":4096,"right1":81728718806051802039918,"left2":25600,"right2":10210485170714789468132,"fanout":223821438184},{"left1":64,"right1":73104288519321269775440,"left2":2048,"right2":48928976836810931109744,"fanout":149166312},{"left1":1026,"right1":17949793463476108473526,"left2":9218,"right2":65847341277706984405560,"fanout":104031648812},{"left1":192,"right1":137466356676322422354320,"left2":200,"right2":83933421235354744096470,"fanout":158121912516},{"left1":32,"right1":74674644656856965467976,"left2":4104,"right2":96564736819356070439912,"fanout":418389000},{"left1":16384,"right1":29973005474228209167700,"left2":2112,"right2":20574152329631171730024,"fanout":1034428931076},{"left1":2,"right1":61848607364529948831146,"left2":2560,"right2":6743740815319571955486,"fanout":103549764800},{"left1":33288,"right1":52828225727192187833742,"left2":35328,"right2":63687172426263588814684,"fanout":931739529440},{"left1":32784,"right1":17071709619759738636922,"left2":32912,"right2":69502086322248385072584,"fanout":879543774248},{"left1":8,"right1":21887898781594552698864,"left2":32792,"right2":27914342041589932635942,"fanout":37798861008},{"left1":256,"right1":21648174391187844098536,"left2":260,"right2":66414383400078337932860,"fanout":150495413196},{"left1":512,"right1":18382594878461846325988,"left2":2048,"right2":10914233549158741090442,"fanout":845050818760},{"left1":4,"right1":56513197336341850812970,"left2":16,"right2":7825828406900275847826,"fanout":107341746738},{"left1":16384,"right1":37596200259011520251342,"left2":16432,"right2":5991621566099561497082,"fanout":1051260411100},{"left1":2,"right1":68234923924284251338748,"left2":20,"right2":58059090716032932968274,"fanout":65410958295},{"left1":16,"right1":387400052444686716540,"left2":1040,"right2":14893632544879559911904,"fanout":37783392308},{"left1":2,"right1":148492075720075106337804,"left2":130,"right2":19755329760690520703666,"fanout":189327736872},{"left1":2,"right1":27714474787317978088246,"left2":16,"right2":17687039561396093422570,"fanout":842749180987},{"left1":8192,"right1":12254252566279308150422,"left2":40976,"right2":20851154604774962579938,"fanout":979603867844},{"left1":16,"right1":49550987615187472482060,"left2":33296,"right2":54571898292948458566884,"fanout":914640926920},{"left1":2,"right1":125150742244453192001174,"left2":258,"right2":54187896942289786062428,"fanout":828243452909},{"left1":16384,"right1":80129872211131753810252,"left2":16576,"right2":65898553272491581207960,"fanout":845910653152},{"left1":32,"right1":41297795906909154054624,"left2":2048,"right2":75394419322241032098348,"fanout":155024678120},{"left1":8192,"right1":37836778934594077242128,"left2":9218,"right2":59849143382440897165668,"fanout":876255428636},{"left1":4,"right1":70662152620953330669164,"left2":20,"right2":22462476501918136695346,"fanout":842367752410},{"left1":512,"right1":9512589729169406921210,"left2":8320,"right2":65923387521854110678960,"fanout":1068924192784},{"left1":128,"right1":40059942257432934566660,"left2":44,"right2":9271147921481396256028,"fanout":1048911558860},{"left1":8320,"right1":5321886203256055195256,"left2":41088,"right2":44945152271350592329238,"fanout":158780372164},{"left1":2048,"right1":24242638382217017919218,"left2":32768,"right2":29760440960602008896916,"fanout":859533275144},{"left1":4096,"right1":136100942136730540074050,"left2":25600,"right2":34646250177134838170790,"fanout":1016916869148},{"left1":128,"right1":65270201436544396762510,"left2":17408,"right2":7545492990827894467920,"fanout":189318094048},{"left1":4,"right1":116695514365823217532238,"left2":260,"right2":29602086160109764466696,"fanout":64425023545},{"left1":8,"right1":53845356615778787945654,"left2":320,"right2":51007646183802809132520,"fanout":54895844388},{"left1":32,"right1":29524978780697558063294,"left2":4100,"right2":60851032701585815633634,"fanout":825171704880},{"left1":16,"right1":18233013711178422306880,"left2":256,"right2":24956117109462040960218,"fanout":206444683575},{"left1":64,"right1":43834111017540375060542,"left2":18432,"right2":41916867826472472413576,"fanout":37850987568},{"left1":2048,"right1":49182910255682786043424,"left2":10248,"right2":38924416529406538776948,"fanout":261606608116},{"left1":32,"right1":31437146235282830275922,"left2":8226,"right2":35205513248616418674522,"fanout":880212757748},{"left1":32,"right1":31723691386138908170202,"left2":32768,"right2":58753590948315690427970,"fanout":138058664152},{"left1":8,"right1":10547742110374498111388,"left2":4132,"right2":33256935501979144206332,"fanout":209996490980},{"left1":4,"right1":141187049791950799050000,"left2":256,"right2":7212466488653381176394,"fanout":840841355511},{"left1":320,"right1":51531745183353236393358,"left2":8512,"right2":20939169536706706798504,"fanout":842768590048},{"left1":1024,"right1":62897172508793420887218,"left2":9216,"right2":25786800825222713490932,"fanout":17797281816},{"left1":16386,"right1":137641626443605965350350,"left2":16394,"right2":150025408801408621450480,"fanout":914156951780},{"left1":512,"right1":31955219576923476119840,"left2":640,"right2":14063842840876889255582,"fanout":1047988589816},{"left1":32784,"right1":125643032846940552631144,"left2":32792,"right2":86070873427932640803440,"fanout":206829837332},{"left1":128,"right1":16953441976300875047370,"left2":9218,"right2":60435421329745277407438,"fanout":876656201928},{"left1":16392,"right1":62525796203068640703326,"left2":8320,"right2":17241820628869798799756,"fanout":223755431140},{"left1":256,"right1":65998452687258904143266,"left2":258,"right2":59732504296808389071262,"fanout":155559983676},{"left1":2,"right1":138985305119081632215022,"left2":66,"right2":65198980842796686492468,"fanout":862765978872},{"left1":12,"right1":111810574259880511983438,"left2":32780,"right2":52707015452583835685486,"fanout":20740320508},{"left1":16392,"right1":146125459422347685235336,"left2":328,"right2":11189092018930637673602,"fanout":1017317227560},{"left1":4104,"right1":44823701689526620904932,"left2":2368,"right2":27350187892320101583154,"fanout":1069315849240},{"left1":4096,"right1":92849988230772262914694,"left2":6400,"right2":27012579473118283704610,"fanout":103965579264},{"left1":2112,"right1":147053742952130393225836,"left2":18496,"right2":126633558744185522652490,"fanout":828862372916},{"left1":192,"right1":17952822447151481260066,"left2":200,"right2":11827622582629611567128,"fanout":86383055068},{"left1":2,"right1":97349875834999918159950,"left2":18,"right2":16518512576046762157588,"fanout":7765825024},{"left1":8194,"right1":21676452167882333753718,"left2":9218,"right2":69239012162700646887960,"fanout":1031264845844},{"left1":528,"right1":48321315513664247686990,"left2":8720,"right2":57315568866781375757534,"fanout":862428656668},{"left1":4,"right1":62264104532033799281914,"left2":32768,"right2":6994411890278612424366,"fanout":845584585784},{"left1":1026,"right1":140253648491477489849128,"left2":17410,"right2":77204697083393776823348,"fanout":89325887688},{"left1":64,"right1":71979684469050545897020,"left2":1088,"right2":29932052327747502931432,"fanout":158580875276},{"left1":2560,"right1":70791143561977738709646,"left2":2568,"right2":23744614471771419422130,"fanout":137722599468},{"left1":2560,"right1":21109414913691101434646,"left2":35328,"right2":14465047774427092964254,"fanout":1013680715796},{"left1":16384,"right1":110182664116865485494748,"left2":320,"right2":60578534466947416689924,"fanout":997039600696},{"left1":512,"right1":69218647655742751135008,"left2":4096,"right2":47780229672996216262484,"fanout":828190497008},{"left1":8,"right1":104944439578928335660000,"left2":33280,"right2":53430637460207613116918,"fanout":72075716836},{"left1":128,"right1":40245108525288406904536,"left2":640,"right2":26255612773111794389014,"fanout":227043446804},{"left1":16384,"right1":90392236077582970278546,"left2":2368,"right2":61535448861408043967442,"fanout":107242641444},{"left1":2,"right1":54753856501284647668888,"left2":258,"right2":55410093934676282573406,"fanout":138307770423},{"left1":1024,"right1":65030301857762989357534,"left2":1088,"right2":19033565911314788102356,"fanout":418959392},{"left1":16384,"right1":137617018909619681266384,"left2":9218,"right2":54208042176781768103222,"fanout":227569315864}],"pure":{"2":231797107156,"4":236527618309,"8":127857121087,"16":94368365064,"96":86652862932,"144":1014151237605,"176":25264971725,"240":961722058231,"288":838119205913,"304":205657161161,"336":107152005895,"368":1081715138827,"384":270683213511,"432":1017775786484,"464":274361012698,"496":1060084002018,"528":983347751697,"560":84994803443,"640":137392277711,"656":931989104126,"672":35349788218,"736":141010522868,"752":919125885415,"784":63633476876,"832":34831143205,"848":205183598355,"912":858554510589,"928":8302556162,"944":59893100542,"960":872283561444,"992":253790459906,"1008":38808591591,"1024":218439866911,"1040":953461895207,"1056":991999434287,"1088":1030821768252,"1152":1000378936820,"1168":884977764344,"1184":39362881251,"1200":993059675330,"0":67868019418,"128":163914066901,"160":85550705608,"208":944039930419,"256":836945825753,"320":974020483824,"352":1014421470257,"416":252467660847,"448":85479133956,"480":1081810619924,"512":154618174678,"544":1014165199643,"576":121165178319,"592":69057970731,"624":881005820936,"720":948266002665,"816":29817304091,"864":60348564731,"880":1065937601553,"896":923335583501,"976":103969529344,"1120":271054412229,"1216":829177794282,"1":13287817476,"5":988698968862,"6":1077989403197,"10":935293626406,"11":845922184196,"14":845718543368,"48":940984628013,"80":68603347213,"112":944371012822,"224":902410015257,"400":219701775306,"608":889847012910,"800":102387479311,"3":51442084876,"7":862887619596,"9":885051960088,"15":103295487196,"64":98345815496,"272":137369893102,"704":93784231406,"1072":860053687045,"1104":863875567073,"1136":29729035018,"32":879492784669,"688":149436886843,"13":172805389512,"192":884106122243,"768":902816907478,"12":218373219134},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"three-phase","capacity":8,"product_order":[0,1,2,3,5,6,7,8,9,10,11,12,14,15,16,17,18,20,22,24,25,28,29,30,31,33,35,38,39,40,43,45,46,48,52,53,54,55,56,57,59,61,62,63,64,65,66,67,69,70,71,74,76,78,81,83,84,85,86,89,91,92,94,95,99,100,102,103,104,105,106,108,110,111,112,118,120,121,122,123,124,125,128,129,131,132,134,137,138,139,140,141,142,145,147,148,149,150,151,154,155,156,157,159,163,164,165,168,169,172,177,178,179,4,19,27,32,36,68,72,82,90,107,109,113,130,143,144,160,161,162,176,13,21,23,26,34,37,41,42,44,47,49,50,51,58,60,73,75,77,79,80,87,88,93,96,97,98,101,114,115,116,117,119,126,127,133,135,136,146,152,153,158,166,167,170,171,173,174,175],"events":[{"kind":"products","cache":943,"indices":[0,1,2,3,5,6,7,8,9,10,11,12,14,15,16,17,18,20,22,24,25,28,29,30,31,33,35,38,39,40,43,45,46,48,52,53,54,55,56,57,59,61,62,63,64,65,66,67,69,70,71,74,76,78,81,83,84,85,86,89,91,92,94,95,99,100,102,103,104,105,106,108,110,111,112,118,120,121,122,123,124,125,128,129,131,132,134,137,138,139,140,141,142,145,147,148,149,150,151,154,155,156,157,159,163,164,165,168,169,172,177,178,179]},{"kind":"pure_low","cache":943,"monomial":5},{"kind":"pure_low","cache":943,"monomial":6},{"kind":"pure_low","cache":943,"monomial":10},{"kind":"pure_low","cache":943,"monomial":14},{"kind":"pure_low","cache":943,"monomial":3},{"kind":"pure_low","cache":943,"monomial":7},{"kind":"pure_low","cache":943,"monomial":13},{"kind":"pure_low","cache":943,"monomial":12},{"kind":"products","cache":975,"indices":[4,19,27,32,36,68,72,82,90,107,109,113,130,143,144,160,161,162,176]},{"kind":"pure_low","cache":975,"monomial":11},{"kind":"products","cache":1499,"indices":[13,21,23,26,34,37,41,42,44,47,49,50,51,58,60,73,75,77,79,80,87,88,93,96,97,98,101,114,115,116,117,119,126,127,133,135,136,146,152,153,158,166,167,170,171,173,174,175]},{"kind":"pure_low","cache":1499,"monomial":9},{"kind":"pure_low","cache":1499,"monomial":15}],"low_computations":11}},{"batch":10,"delta_sha256":"6a48a92d3e8e56bd536592b235f9251cab994d8de0d576e8ef212767fca6a2c4","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":2,"right1":35403488500257736729750,"left2":16384,"right2":73912581579291932890424,"fanout":46824485094},{"left1":16384,"right1":11636579828767841345032,"left2":528,"right2":28392683000800706516132,"fanout":111671046190},{"left1":2,"right1":82002651065063674765278,"left2":16450,"right2":28810503211916677135696,"fanout":1095196928242},{"left1":66,"right1":47042582749361893744248,"left2":2052,"right2":38516198669084109020002,"fanout":1026467046602},{"left1":256,"right1":24638795848750250741236,"left2":260,"right2":68003914938250775417980,"fanout":30197139483},{"left1":16,"right1":73733942357195489067364,"left2":2048,"right2":25844086299716001361120,"fanout":210366680774},{"left1":16384,"right1":24024744850154829080724,"left2":16448,"right2":64518197027693375873112,"fanout":9005753074},{"left1":8,"right1":10336288912918659804110,"left2":16384,"right2":36549826653008602977008,"fanout":80766881804},{"left1":512,"right1":60104847904009909539312,"left2":4104,"right2":57027134088401325535480,"fanout":257966796288},{"left1":4096,"right1":51579659297048006070842,"left2":2052,"right2":51915658365175115406468,"fanout":937110863608},{"left1":16,"right1":7239626215373087457254,"left2":32768,"right2":21714578495505332458952,"fanout":34910697012},{"left1":8,"right1":1477298471439419981224,"left2":66,"right2":36553259738501364792340,"fanout":897630091320},{"left1":2048,"right1":31849433816323595195894,"left2":10240,"right2":32005010397745707211246,"fanout":836916143646},{"left1":4,"right1":140987630405391318482364,"left2":132,"right2":22809254188369022409942,"fanout":89993445088},{"left1":8192,"right1":31920288977354750273446,"left2":8720,"right2":1071828667072501647150,"fanout":931652058650},{"left1":8192,"right1":30871701625976911292818,"left2":260,"right2":13403868230915689745656,"fanout":962121360956},{"left1":256,"right1":72773887240987395319764,"left2":288,"right2":41190050011942274557644,"fanout":25805459140},{"left1":16384,"right1":54028390469771255766836,"left2":16386,"right2":38384034209097801656158,"fanout":72850536508},{"left1":128,"right1":2965735608992086273882,"left2":640,"right2":9063573755441061544134,"fanout":55567197382},{"left1":32896,"right1":96957796397678429294198,"left2":32904,"right2":96771928171280560994218,"fanout":146401133792},{"left1":128,"right1":60383305892188992289624,"left2":32896,"right2":40170144210471467869784,"fanout":80812713170},{"left1":8,"right1":90468545765667523151114,"left2":32778,"right2":43145001546769882589742,"fanout":940076703770},{"left1":4096,"right1":113942185239414728638752,"left2":4108,"right2":120190822297491440235224,"fanout":249523474978},{"left1":8192,"right1":44622738312359132510746,"left2":4104,"right2":73228865279848369562128,"fanout":1049025246458},{"left1":1056,"right1":41534237093978666186582,"left2":9248,"right2":39253775303192922835862,"fanout":29897666596},{"left1":4096,"right1":37357090049982300326360,"left2":6144,"right2":193158632710468936814,"fanout":232511903484},{"left1":4,"right1":33234438667663166502654,"left2":66,"right2":10918437300014159651054,"fanout":992120284688},{"left1":8,"right1":76146623836052686721470,"left2":9224,"right2":65445938511608671375680,"fanout":1023210869976},{"left1":256,"right1":14066503327409824855420,"left2":272,"right2":26087944823230923944150,"fanout":51637515802},{"left1":32770,"right1":61228014057885749372778,"left2":40962,"right2":12754632741961822030222,"fanout":996737682984},{"left1":8,"right1":80748167302672935130814,"left2":32770,"right2":13484822274406757051254,"fanout":266637149194},{"left1":64,"right1":27740918882544047087410,"left2":16448,"right2":36820665644727235783670,"fanout":828105302028},{"left1":2,"right1":116670404130538968944124,"left2":66,"right2":60166859445571352742646,"fanout":896641666278},{"left1":256,"right1":46433412300145608903344,"left2":272,"right2":5280851925338387761326,"fanout":836514867713},{"left1":16386,"right1":7197600442055073169782,"left2":4104,"right2":1139028756815652061432,"fanout":121199450828},{"left1":512,"right1":69929471951498409704550,"left2":4112,"right2":13630238117482233823664,"fanout":3904956140},{"left1":8192,"right1":24021040979380266744316,"left2":144,"right2":61153388016696971329864,"fanout":913925482688},{"left1":4096,"right1":69059178749837067137364,"left2":16448,"right2":67163663749208664111054,"fanout":825085661946},{"left1":4,"right1":128859271079571449767924,"left2":9216,"right2":67535779946382817626118,"fanout":1094143376110},{"left1":2304,"right1":17920393449632809361022,"left2":6400,"right2":50329359348689787362538,"fanout":124331765482},{"left1":32,"right1":66984688402531367172272,"left2":288,"right2":13375622104079268711018,"fanout":112252862466},{"left1":4,"right1":73823601257553079455334,"left2":68,"right2":71923935803509848752770,"fanout":863234303230},{"left1":256,"right1":92908266401873661946680,"left2":260,"right2":91048930072008978920842,"fanout":72239431623},{"left1":1024,"right1":33702379339004146729638,"left2":2052,"right2":26024497775916001064570,"fanout":639157962},{"left1":2052,"right1":42696553721936120524426,"left2":2180,"right2":58291286911766992016068,"fanout":155460183584},{"left1":8448,"right1":140860764705246141719920,"left2":8456,"right2":149990521515856655247920,"fanout":183925399268},{"left1":512,"right1":16849676498090066517988,"left2":2816,"right2":71528216615451182412768,"fanout":224244459034},{"left1":256,"right1":31834584737099363381364,"left2":9472,"right2":17741388842559380455006,"fanout":149455168548},{"left1":4096,"right1":59297899117520824349290,"left2":16416,"right2":44065554007239037051536,"fanout":120461247228},{"left1":32768,"right1":26885417401954779787838,"left2":34816,"right2":41396367696278431181486,"fanout":888906195506},{"left1":1024,"right1":9842485853089253111270,"left2":3072,"right2":23513856554042793766830,"fanout":902932217554},{"left1":2,"right1":74036875675556672650794,"left2":130,"right2":50893468918210012290634,"fanout":940175324396},{"left1":12,"right1":147773831015120981835102,"left2":44,"right2":7614783101155152033716,"fanout":163560691428},{"left1":66,"right1":45144812113592801801652,"left2":4162,"right2":67583263193909896817900,"fanout":922526076144},{"left1":4096,"right1":26136980862104038609166,"left2":4224,"right2":73621697580791640704102,"fanout":38067291368},{"left1":64,"right1":70699630796928278829728,"left2":4608,"right2":58330123153518338073912,"fanout":1006030875876},{"left1":2,"right1":114032433761403471498836,"left2":256,"right2":18345867035208997177674,"fanout":179500934368},{"left1":4112,"right1":7270779154588780392572,"left2":4144,"right2":2714331982642525978300,"fanout":1065635160614},{"left1":2304,"right1":116758044289651329818124,"left2":2312,"right2":106461739052579689333664,"fanout":962645393088},{"left1":8,"right1":94713611047122683311624,"left2":32904,"right2":14739328994694330800616,"fanout":20504778782},{"left1":4096,"right1":694151919906300933850,"left2":20544,"right2":45415564995354561446470,"fanout":64270296074},{"left1":32770,"right1":57798633302489651180956,"left2":32786,"right2":58716929142700799365542,"fanout":1094292274394},{"left1":2,"right1":99150151918616463440724,"left2":128,"right2":51204885531072844277166,"fanout":172002248430},{"left1":4,"right1":4231795216397760575020,"left2":148,"right2":60846365892956965603994,"fanout":862598131906},{"left1":64,"right1":18050127137238876572308,"left2":320,"right2":9250503386312507930212,"fanout":111773941304},{"left1":8,"right1":26759186996824367553050,"left2":16456,"right2":40794874626065183986754,"fanout":267242453756},{"left1":2,"right1":112651722085327249063938,"left2":16384,"right2":31264799506521078427612,"fanout":21276323344},{"left1":64,"right1":315369403807992630180,"left2":1056,"right2":40605846044975365686448,"fanout":604038890},{"left1":32768,"right1":5060597262964290286986,"left2":35072,"right2":9353576902524874997758,"fanout":862630581482},{"left1":32768,"right1":49355241879587751471028,"left2":35072,"right2":18658959896999087607480,"fanout":235855605764},{"left1":32,"right1":10161528184144737143388,"left2":16448,"right2":58989560541189923823984,"fanout":833875346948},{"left1":8,"right1":122308361352451552678440,"left2":2048,"right2":66369148289815333363034,"fanout":825507593976},{"left1":128,"right1":68260607350380791375558,"left2":160,"right2":42920908201966153032894,"fanout":258013185038},{"left1":8192,"right1":26522156363234004997198,"left2":32768,"right2":74185383096886258033610,"fanout":34699087406},{"left1":8192,"right1":77964961601983944299422,"left2":8194,"right2":89577804202956108706082,"fanout":149399413482},{"left1":16,"right1":14521539018085777285294,"left2":18,"right2":54381335087650465708364,"fanout":11877280705},{"left1":32,"right1":31156925456313130637896,"left2":32896,"right2":72590598221357269183002,"fanout":197952465622},{"left1":512,"right1":21613673187218982457228,"left2":16896,"right2":24180441098719193279214,"fanout":834011843284},{"left1":2304,"right1":8865256782877682166586,"left2":3328,"right2":17372750047941443836920,"fanout":885514044950},{"left1":512,"right1":48383160126895371009924,"left2":16896,"right2":29987855758759041162104,"fanout":851157458454},{"left1":4096,"right1":66155413256798201474240,"left2":13312,"right2":24463144537667542509012,"fanout":54763254284},{"left1":8192,"right1":5777484098647500624552,"left2":8224,"right2":1857186529697928548922,"fanout":859534451906},{"left1":1024,"right1":43799213070269250289502,"left2":16416,"right2":29959718544765463139654,"fanout":845923804188},{"left1":16,"right1":27988214301851641559412,"left2":2320,"right2":59407509262566883611808,"fanout":3557215472},{"left1":32770,"right1":39574949460665689095382,"left2":40962,"right2":37139400860708636712342,"fanout":939874724050},{"left1":1056,"right1":37497293577939631352428,"left2":9248,"right2":22228547337110278215436,"fanout":249626101812},{"left1":512,"right1":10584791559076258031700,"left2":656,"right2":56289727768075710451198,"fanout":81101899004},{"left1":16384,"right1":54105251587709554851508,"left2":17408,"right2":1562579126147987661474,"fanout":897045503506},{"left1":4096,"right1":32540969368377226097386,"left2":66,"right2":35859973417785009569610,"fanout":60632990246},{"left1":8192,"right1":40667466010621406712414,"left2":8196,"right2":10765515611616738149088,"fanout":850921264330},{"left1":4,"right1":50564579552469700611240,"left2":260,"right2":69268892388297314960246,"fanout":619496174},{"left1":4,"right1":93312730762921320529736,"left2":32768,"right2":4047201483235782289570,"fanout":919930206244},{"left1":4,"right1":46909972196626260467640,"left2":2048,"right2":72115938044207226591478,"fanout":35144466470},{"left1":32,"right1":54752137475056544728478,"left2":2084,"right2":18248281973632405980984,"fanout":870988789274},{"left1":256,"right1":100322655821246463387806,"left2":260,"right2":139380630861539427894540,"fanout":137575383831},{"left1":32768,"right1":99550653843862347776858,"left2":32780,"right2":82343317372066342746348,"fanout":845452867802},{"left1":2,"right1":66122520296254480234708,"left2":128,"right2":34355810626515454271518,"fanout":837286288412},{"left1":1024,"right1":47269065841444132031470,"left2":528,"right2":45657192104489554154804,"fanout":180861941760},{"left1":4608,"right1":36528905120680280599218,"left2":4240,"right2":74914624688074759862782,"fanout":132976266998},{"left1":4,"right1":90036721918597545532576,"left2":16420,"right2":30431533992842264673354,"fanout":1031711621640},{"left1":32768,"right1":35498170320979764461584,"left2":43008,"right2":49602052055166394419144,"fanout":52392888540},{"left1":32,"right1":69910219809366839425446,"left2":16448,"right2":40316302767496987621976,"fanout":880403595992},{"left1":16,"right1":25030517262234971600394,"left2":272,"right2":63218367661514982700420,"fanout":4879936001},{"left1":1024,"right1":37169580579348691582092,"left2":2056,"right2":41690660700609419141964,"fanout":884777556722},{"left1":32768,"right1":30750768982620680444918,"left2":34816,"right2":64245057667050414226078,"fanout":825016597206},{"left1":4096,"right1":62240283011576969941060,"left2":13312,"right2":60754161380567847155906,"fanout":198307151598},{"left1":512,"right1":11634857403965824178442,"left2":16896,"right2":16891826309224481082434,"fanout":106314075684},{"left1":64,"right1":30694893849618331177392,"left2":80,"right2":44481267190889917420800,"fanout":150125418694},{"left1":64,"right1":46903478886548802561410,"left2":68,"right2":5458846587852839629026,"fanout":60198618326},{"left1":32,"right1":74599596759557719917184,"left2":656,"right2":25529399450916196809874,"fanout":983417235166},{"left1":1024,"right1":53971317745284377646980,"left2":12,"right2":93003016475161825253678,"fanout":1000472579644},{"left1":1024,"right1":2930748340848014614946,"left2":9472,"right2":38439499622799915272442,"fanout":232209522742},{"left1":64,"right1":17298058231259311551090,"left2":2116,"right2":2414151220081254071828,"fanout":1060088367662},{"left1":512,"right1":50114427720062511849352,"left2":8960,"right2":32835501046295026976430,"fanout":979770282180},{"left1":32,"right1":8381452608724596573980,"left2":4624,"right2":52514226833171707330256,"fanout":833338132198},{"left1":8,"right1":143079635710753365414780,"left2":16416,"right2":55242882873378493930788,"fanout":997084498980},{"left1":512,"right1":73678333192526039195578,"left2":17440,"right2":74480941090892750273342,"fanout":104086518316},{"left1":1024,"right1":59645453133005991895864,"left2":2048,"right2":73946331858317963789278,"fanout":825422132256},{"left1":16,"right1":41108886143386472554268,"left2":1056,"right2":60218239631970640872588,"fanout":1005105115358},{"left1":1024,"right1":69809048117259125272248,"left2":32768,"right2":48316685494622438690482,"fanout":103227386580},{"left1":1024,"right1":32730543031892200348156,"left2":16448,"right2":47518026220221894797628,"fanout":210031616006},{"left1":32,"right1":15890812119559353708440,"left2":4608,"right2":2130553016984621449872,"fanout":223789331518},{"left1":144,"right1":22498903551838712991370,"left2":1168,"right2":46641852612480223926048,"fanout":26387739148},{"left1":128,"right1":1504553022706777283976,"left2":32896,"right2":57925385504884764485300,"fanout":911002883326},{"left1":8,"right1":18655047936061274437448,"left2":16384,"right2":36269647823402825005660,"fanout":137610266822},{"left1":1056,"right1":139883320020940113496428,"left2":1064,"right2":145377116797519019192764,"fanout":266996038196},{"left1":2048,"right1":65369949068493880257886,"left2":8448,"right2":3773192487552358554082,"fanout":164196176424},{"left1":2,"right1":103035151369107095464568,"left2":4,"right2":99453912978953079110298,"fanout":26292200475},{"left1":16384,"right1":7051288023529021135934,"left2":17408,"right2":32314789422075864318702,"fanout":825016713922},{"left1":4,"right1":123355918135380369616722,"left2":16420,"right2":73779275042068199323938,"fanout":956886823630},{"left1":2,"right1":58849515928420153171754,"left2":8194,"right2":64641492080224345160932,"fanout":154990212818},{"left1":4,"right1":20474436153383946681072,"left2":66,"right2":75419845718098793622468,"fanout":1008378632752},{"left1":2,"right1":54909268024775513938366,"left2":8450,"right2":66873131784772133484972,"fanout":241124438770},{"left1":512,"right1":48354453369533866507258,"left2":768,"right2":15397277475417483502192,"fanout":149656176352},{"left1":4128,"right1":15578660002022714555824,"left2":20480,"right2":32139911885786026973994,"fanout":146781172276},{"left1":128,"right1":34758160821065267377900,"left2":12,"right2":118307769598251833346682,"fanout":949067322430},{"left1":1056,"right1":22047022010431459414124,"left2":16448,"right2":67695577178493042685558,"fanout":51741598408},{"left1":144,"right1":143030901027684189618470,"left2":146,"right2":141041347956572393447066,"fanout":905936114186},{"left1":4104,"right1":14053779695852057505482,"left2":4120,"right2":61518496957764243453670,"fanout":175739903190},{"left1":16,"right1":19777184334470041684890,"left2":128,"right2":16401771320270990427314,"fanout":42985855544},{"left1":128,"right1":42385501190697420031416,"left2":528,"right2":36539813396785311567810,"fanout":215221322792},{"left1":256,"right1":38367923687914206149616,"left2":272,"right2":7462661949947502331776,"fanout":52039057169},{"left1":512,"right1":47937710362138265031970,"left2":16416,"right2":39449377364298507107744,"fanout":902247415010},{"left1":256,"right1":19602327884613833191710,"left2":4360,"right2":70978208618591820669498,"fanout":25838619876},{"left1":2,"right1":4616308969035776557760,"left2":16384,"right2":3991869667590661135858,"fanout":167150017788},{"left1":256,"right1":42228578028051383339722,"left2":66,"right2":154369452579127634378,"fanout":957408740386},{"left1":32768,"right1":52296268865867745509990,"left2":9216,"right2":25859959818168909828306,"fanout":1031802329814},{"left1":4096,"right1":36898283111246569183376,"left2":5152,"right2":16291015736689708694740,"fanout":189061453532},{"left1":2048,"right1":47480058938049400727190,"left2":2052,"right2":15541966024711444747306,"fanout":971314040322},{"left1":2,"right1":92380265975537232841406,"left2":528,"right2":41198864013161367664848,"fanout":20749804748},{"left1":256,"right1":141863488317744922807906,"left2":260,"right2":150903443577615556860918,"fanout":209481418205},{"left1":2048,"right1":51708464548948103425134,"left2":11264,"right2":5058913288416148747130,"fanout":21442908722},{"left1":64,"right1":44793216753166090256558,"left2":16416,"right2":39259152862022689955518,"fanout":158884310032},{"left1":128,"right1":46202554835529201352158,"left2":32770,"right2":6936278253630555763782,"fanout":181309931558},{"left1":2,"right1":17055092753226042075314,"left2":4104,"right2":33009432404053044590194,"fanout":260966773486},{"left1":4096,"right1":35140154962831133130094,"left2":4624,"right2":41101816845767313106698,"fanout":1091025512496},{"left1":512,"right1":58172401820597346072458,"left2":528,"right2":39606956433792308821186,"fanout":129097143304},{"left1":4096,"right1":11322109564881195630510,"left2":12544,"right2":44193235422522697068670,"fanout":931619681010},{"left1":32896,"right1":70203992675935240594800,"left2":41088,"right2":70625942153249548546750,"fanout":98652712480},{"left1":8448,"right1":1521611459678654003340,"left2":41216,"right2":58131038933579723044584,"fanout":89859044550},{"left1":8,"right1":25229566841776443174980,"left2":536,"right2":59348775014526363159654,"fanout":163849709578},{"left1":32,"right1":54977263327899894977338,"left2":96,"right2":69460137159314826376910,"fanout":825623456788},{"left1":32896,"right1":17182913044351202210550,"left2":33408,"right2":14897995130440023215304,"fanout":166700043288},{"left1":2,"right1":131152003291033047756672,"left2":262,"right2":55851108533618788383280,"fanout":837718769451},{"left1":2048,"right1":78567263913827525623262,"left2":2050,"right2":91414693157660193024794,"fanout":902684477128},{"left1":32896,"right1":67517005933584800127556,"left2":32900,"right2":2462976599225439635086,"fanout":954289239064},{"left1":256,"right1":40741097935432612141976,"left2":272,"right2":59448242602785379059600,"fanout":86602953514},{"left1":4104,"right1":50628109503830291013084,"left2":4168,"right2":29221043410949549601154,"fanout":269579254004},{"left1":64,"right1":27789648202766793198010,"left2":512,"right2":7897947767422289286376,"fanout":51757595334},{"left1":32,"right1":74745333028213181747794,"left2":8480,"right2":28127857850127995719090,"fanout":120564100634},{"left1":4096,"right1":34790165440362210286678,"left2":5152,"right2":71356250638252485717182,"fanout":1014163047638},{"left1":32,"right1":13110238770543625676338,"left2":9248,"right2":14074592127187577792824,"fanout":982946998802},{"left1":32768,"right1":120232254023017410869532,"left2":32780,"right2":122706133473264108379850,"fanout":132609878578},{"left1":1024,"right1":23239302900387472590580,"left2":1152,"right2":61768458781176437328364,"fanout":859109988544},{"left1":16384,"right1":9700590683515850828990,"left2":18688,"right2":63784335514683679434686,"fanout":111749897416},{"left1":8,"right1":104597127031694215272112,"left2":152,"right2":46176657530859219515694,"fanout":871488955966},{"left1":16,"right1":2951017696271688861520,"left2":276,"right2":26483662593196133071442,"fanout":905428717332},{"left1":512,"right1":25097838982994028446260,"left2":33408,"right2":14518442717097801863852,"fanout":919830017224},{"left1":8,"right1":32710962846308699058330,"left2":32776,"right2":34525297891170046840798,"fanout":828408393266},{"left1":8192,"right1":43175316275074464802230,"left2":8448,"right2":67070320667827284905280,"fanout":897113392686},{"left1":64,"right1":34591412119562078174046,"left2":4096,"right2":75273541221028401823906,"fanout":154701927974},{"left1":16,"right1":985934436773444929000,"left2":32786,"right2":11420973075915095616350,"fanout":845673147446},{"left1":12,"right1":22160503139092655706228,"left2":8204,"right2":36556114601396318154938,"fanout":68834614806},{"left1":128,"right1":51964947728786680339576,"left2":2048,"right2":38216509628039245549394,"fanout":12820288058},{"left1":256,"right1":18675622763465957899916,"left2":258,"right2":40123831544196735014562,"fanout":150908817154},{"left1":32896,"right1":12862982895228408083000,"left2":33920,"right2":57791362071104098063294,"fanout":267061222082},{"left1":32768,"right1":25356422132290770297114,"left2":32784,"right2":15131287067219102080890,"fanout":971301325372},{"left1":2,"right1":115054232143281671472604,"left2":4608,"right2":63200167233965151966860,"fanout":962545507334},{"left1":2048,"right1":94233921918990039525920,"left2":2060,"right2":110908569344362357594122,"fanout":944907360510},{"left1":512,"right1":25652760838975537331626,"left2":2564,"right2":67195894335492147979684,"fanout":828673105120},{"left1":16,"right1":34375240614894740354238,"left2":2064,"right2":57883820710601821081522,"fanout":20805453334},{"left1":1024,"right1":63709206251726589035078,"left2":1152,"right2":7988143161872950990916,"fanout":63836267224},{"left1":144,"right1":25159762079470202062248,"left2":4240,"right2":13430821713043260572,"fanout":971382383150},{"left1":16448,"right1":11613372575552544198032,"left2":17472,"right2":20522408707990672802034,"fanout":1085654753340},{"left1":64,"right1":29834414354187098964274,"left2":592,"right2":54828432190861903024440,"fanout":1014129026294},{"left1":16,"right1":13867213168772960323976,"left2":272,"right2":5724879945650858014178,"fanout":162271994324},{"left1":64,"right1":38790436373090927636968,"left2":4176,"right2":52948493150265401417162,"fanout":95074706494},{"left1":8,"right1":104505442505708918673960,"left2":32778,"right2":18815354554950746111264,"fanout":845687428826},{"left1":32,"right1":32615298326307446140140,"left2":66,"right2":1707051432580956064988,"fanout":54932325630},{"left1":528,"right1":28856028775831178176448,"left2":532,"right2":68061682565100986435456,"fanout":129623263962},{"left1":16,"right1":22571636419345904105624,"left2":4104,"right2":32324949810842663779982,"fanout":1077972961818},{"left1":16,"right1":35624840890256078743756,"left2":32768,"right2":63211764709661000103746,"fanout":69528113866},{"left1":4,"right1":125478641475810411129348,"left2":16384,"right2":27319573193178137638060,"fanout":828071008294}],"pure":{"4":1017737971763,"8":957659422705,"80":930952261424,"96":262561654819,"128":1004924880163,"176":9476371155,"192":223542183699,"224":271022039800,"272":42677758419,"288":884665730516,"304":38550649576,"320":1056730243066,"352":936182084404,"368":188605647879,"432":84907075281,"448":265497136081,"464":1096041242313,"480":157972501280,"496":829699834412,"528":76843330837,"560":163124142567,"592":1089880269039,"608":850490896392,"688":263036084240,"704":73665295933,"720":828107970330,"768":961300739380,"800":987459881432,"816":93818989509,"896":911188489239,"912":185756495396,"992":940866529062,"1008":970694772949,"1040":194298920907,"1056":832851266258,"1072":124449966081,"1088":211457665266,"1104":965712149481,"1184":154884894014,"1200":175406644731,"1216":65063027511,"0":132390838726,"2":854443621856,"3":961570014172,"7":64000934086,"9":952960598749,"12":258406276103,"15":937154975756,"16":179921933801,"32":1012654539821,"48":243892498446,"240":974306545362,"336":957091735489,"384":256855241485,"416":120780232406,"544":1078522673638,"576":858643511294,"656":1053288623332,"672":236169343937,"752":970376006898,"784":832688346562,"848":4140040238,"928":1082921129181,"944":1081743898100,"1120":971718843186,"1136":902731458335,"1168":218588505051,"1":832331515919,"5":198539407833,"11":140844604420,"13":928015985884,"14":120296624648,"144":888202249224,"256":538833951,"400":859463861275,"512":1082144193220,"640":104053553195,"736":841107112999,"832":953848087565,"864":862832958174,"960":829785629649,"1024":905530058186,"1152":944808447010,"10":136568434996,"112":253101280231,"208":1005523425284,"880":862671392706,"976":111553075503,"64":888771249376,"624":183673619137,"6":63972046784,"160":1065435454701},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"three-phase","capacity":8,"product_order":[0,1,2,3,4,5,6,7,8,9,11,16,17,22,25,26,28,31,32,33,34,35,37,39,40,41,42,43,46,48,50,52,53,55,56,57,58,60,64,65,66,67,70,71,75,77,78,79,82,83,87,88,90,92,93,94,97,99,101,102,103,106,107,108,110,112,114,115,116,117,118,120,121,124,125,127,128,129,131,133,134,136,138,141,142,143,144,145,147,148,149,150,152,154,155,156,160,161,163,164,166,167,168,170,174,176,180,184,187,188,189,190,193,194,195,196,198,199,200,202,12,13,14,15,18,23,24,27,36,38,44,45,47,51,54,62,63,72,74,80,81,85,86,89,96,98,105,109,111,113,122,126,130,132,135,137,139,140,151,157,169,171,173,175,179,182,183,191,192,10,19,20,21,29,30,49,59,61,68,69,73,76,84,91,95,100,104,119,123,146,153,158,159,162,165,172,177,178,181,185,186,197,201],"events":[{"kind":"products","cache":759,"indices":[0,1,2,3,4,5,6,7,8,9,11,16,17,22,25,26,28,31,32,33,34,35,37,39,40,41,42,43,46,48,50,52,53,55,56,57,58,60,64,65,66,67,70,71,75,77,78,79,82,83,87,88,90,92,93,94,97,99,101,102,103,106,107,108,110,112,114,115,116,117,118,120,121,124,125,127,128,129,131,133,134,136,138,141,142,143,144,145,147,148,149,150,152,154,155,156,160,161,163,164,166,167,168,170,174,176,180,184,187,188,189,190,193,194,195,196,198,199,200,202]},{"kind":"pure_low","cache":759,"monomial":3},{"kind":"pure_low","cache":759,"monomial":9},{"kind":"pure_low","cache":759,"monomial":12},{"kind":"pure_low","cache":759,"monomial":5},{"kind":"pure_low","cache":759,"monomial":11},{"kind":"pure_low","cache":759,"monomial":14},{"kind":"pure_low","cache":759,"monomial":10},{"kind":"pure_low","cache":759,"monomial":6},{"kind":"products","cache":507,"indices":[12,13,14,15,18,23,24,27,36,38,44,45,47,51,54,62,63,72,74,80,81,85,86,89,96,98,105,109,111,113,122,126,130,132,135,137,139,140,151,157,169,171,173,175,179,182,183,191,192]},{"kind":"pure_low","cache":507,"monomial":7},{"kind":"pure_low","cache":507,"monomial":13},{"kind":"products","cache":1403,"indices":[10,19,20,21,29,30,49,59,61,68,69,73,76,84,91,95,100,104,119,123,146,153,158,159,162,165,172,177,178,181,185,186,197,201]},{"kind":"pure_low","cache":1403,"monomial":15}],"low_computations":11}},{"batch":11,"delta_sha256":"74f1fdcf170a7b0eb23f160ad5de7297620673cc81a9d33f6a953eee1ec14653","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":32,"right1":64350701520563497645630,"left2":68,"right2":894021326646782421914,"fanout":1014299373596},{"left1":64,"right1":30981082236341372592340,"left2":18432,"right2":45872890528019592887898,"fanout":962542495976},{"left1":32768,"right1":64215076363889431564118,"left2":514,"right2":49107157598794263578632,"fanout":965374953472},{"left1":64,"right1":65155625844327236149228,"left2":4176,"right2":33784750241614107889186,"fanout":1034084478156},{"left1":16,"right1":59992692550410378423248,"left2":256,"right2":48150913478255658339542,"fanout":73821268471},{"left1":4112,"right1":115700535653662276371008,"left2":12304,"right2":147643082914777511275304,"fanout":120460735684},{"left1":128,"right1":61594643675400435846624,"left2":8192,"right2":94204296123493648048430,"fanout":241459268636},{"left1":2,"right1":17029344055764764294330,"left2":16,"right2":37716476219076445033628,"fanout":158880040992},{"left1":128,"right1":72106009774996084987962,"left2":32898,"right2":49838021802294078833536,"fanout":858993521896},{"left1":32768,"right1":55712784571539605601410,"left2":392,"right2":1598064352924772400678,"fanout":52144906436},{"left1":2,"right1":13903940824452447633042,"left2":16,"right2":39282141865641212777402,"fanout":845389967679},{"left1":4096,"right1":45162979593373335039114,"left2":4104,"right2":71544930890822431083312,"fanout":89724606516},{"left1":16,"right1":4985774352718636314800,"left2":512,"right2":127004552410220306878718,"fanout":157907740892},{"left1":68,"right1":124127347653236141724898,"left2":76,"right2":148773599364231262437700,"fanout":1014230744288},{"left1":4,"right1":111460931965740288855680,"left2":260,"right2":57082049939505306971872,"fanout":4449114056},{"left1":8,"right1":116099849615694351554526,"left2":5120,"right2":60878063293791500992426,"fanout":258302017540},{"left1":256,"right1":61252596450191688665486,"left2":260,"right2":71672816650536036058462,"fanout":141782610140},{"left1":264,"right1":132346971505464396403792,"left2":33032,"right2":136944779682740254633324,"fanout":172403259616},{"left1":4,"right1":82574926049192589144640,"left2":16,"right2":69915423551118197649826,"fanout":829011657250},{"left1":2,"right1":82461705005011666397268,"left2":256,"right2":2105624003742644495980,"fanout":962898691554},{"left1":16512,"right1":85524194799777908980276,"left2":49280,"right2":78558527271688159106916,"fanout":896842860568},{"left1":1024,"right1":57335114010799885563930,"left2":8192,"right2":148137016340026895763036,"fanout":911073399000},{"left1":16,"right1":67993170052816303333560,"left2":8216,"right2":49203459173094737671208,"fanout":241405011000},{"left1":4,"right1":106330026707081238302586,"left2":256,"right2":32345685691165610505080,"fanout":37644982782},{"left1":2080,"right1":8141709980365077179882,"left2":16512,"right2":11754451069385375473584,"fanout":928250658036},{"left1":4,"right1":4404507364665928612182,"left2":6,"right2":51085364004066126538238,"fanout":21002926883},{"left1":2,"right1":129486458910098621284630,"left2":4,"right2":149397467205245972509166,"fanout":11893616086},{"left1":8,"right1":89395242275859956507066,"left2":33282,"right2":67855234377498893242924,"fanout":37863880744},{"left1":4,"right1":92136937787419209162990,"left2":6,"right2":83362102387428874471980,"fanout":24762842907},{"left1":1024,"right1":10794834877637215883786,"left2":1536,"right2":6888978486800500076876,"fanout":949068244196},{"left1":48,"right1":38005265139471572400988,"left2":8240,"right2":13674881249134076552864,"fanout":107254699020},{"left1":8704,"right1":62387133485995867064186,"left2":10752,"right2":12439858338383215568954,"fanout":38133358832},{"left1":8448,"right1":47852946433419599513946,"left2":10240,"right2":93141370435771977847224,"fanout":944963903696},{"left1":2,"right1":75980401000960919637518,"left2":258,"right2":54107210621016105774178,"fanout":72106822114},{"left1":4096,"right1":9892433718240016792602,"left2":8192,"right2":991735382686040859124,"fanout":210331770928},{"left1":2064,"right1":41536639517433907250046,"left2":2072,"right2":21008220685082720650738,"fanout":860013515784},{"left1":2048,"right1":38501840518853522441138,"left2":8706,"right2":30006498281456066720062,"fanout":824903000100},{"left1":16512,"right1":36672388100825190578532,"left2":16576,"right2":66289681460104763914252,"fanout":1048388581396},{"left1":4,"right1":120889409564269047212554,"left2":20,"right2":21400122365449771299894,"fanout":154823872775},{"left1":2,"right1":15213254432132945168440,"left2":258,"right2":43680041046774640939214,"fanout":893674138940},{"left1":4,"right1":100555645304914366630112,"left2":2096,"right2":3123744396112485548288,"fanout":913959085296},{"left1":64,"right1":56118968326508431480228,"left2":1024,"right2":43444228411302728670952,"fanout":38467329084},{"left1":1024,"right1":13253797072534854065794,"left2":2096,"right2":44211721767582907305700,"fanout":72410740796},{"left1":32768,"right1":74983144004211237344680,"left2":33282,"right2":9164993611261521348698,"fanout":928518507772},{"left1":2048,"right1":29461634119336410895114,"left2":1028,"right2":52688870144554630520944,"fanout":206227436748},{"left1":4096,"right1":29167045499339855823826,"left2":1028,"right2":62710998493148722312128,"fanout":175222085852},{"left1":1024,"right1":9661725771204893316174,"left2":48,"right2":6522954794596550156116,"fanout":828059738128},{"left1":514,"right1":82230849712940348455022,"left2":522,"right2":149138603392727255046334,"fanout":1048041552076},{"left1":264,"right1":52532851528952979229980,"left2":8704,"right2":114213338200283316963876,"fanout":1086360964144},{"left1":33280,"right1":117392483269705335972050,"left2":33288,"right2":38624904939796761989374,"fanout":880404135152},{"left1":2080,"right1":36052914632975253116474,"left2":16448,"right2":1221598629232406701356,"fanout":89404261416},{"left1":2,"right1":71319204880601497781010,"left2":6,"right2":43914936039824746002334,"fanout":829165720615},{"left1":2048,"right1":34732813673220449002112,"left2":8200,"right2":137095546657844971585302,"fanout":539279612},{"left1":4112,"right1":36464758680865145135664,"left2":6160,"right2":34310084797245643762254,"fanout":172808009752},{"left1":64,"right1":28585303360314056558228,"left2":16384,"right2":5745455631951659024872,"fanout":1000272306228},{"left1":384,"right1":23412109881127717472486,"left2":33280,"right2":16078081149564187715984,"fanout":120395137256},{"left1":16,"right1":53628529908438549592762,"left2":128,"right2":72869668432246652484434,"fanout":55513717952},{"left1":2,"right1":113831961105357772572448,"left2":128,"right2":46477727396938660132068,"fanout":1000138129648},{"left1":128,"right1":98078569802295606475294,"left2":132,"right2":91251243471186702923630,"fanout":244077036576},{"left1":32768,"right1":18107761711770394756998,"left2":33032,"right2":1110281674058501302992,"fanout":928316922880},{"left1":512,"right1":22533416846332261410728,"left2":768,"right2":26892880313616008166822,"fanout":172339552292},{"left1":4,"right1":112179780605156227153246,"left2":6,"right2":3390177152739892748046,"fanout":140712426252},{"left1":128,"right1":55912423097312810537680,"left2":33280,"right2":143353175683452768144824,"fanout":893625363500},{"left1":68,"right1":2501706759966527841800,"left2":580,"right2":52205429006132260192,"fanout":258573655096},{"left1":8200,"right1":7601423197905815472622,"left2":10248,"right2":61414811703622313574144,"fanout":979590499360},{"left1":4096,"right1":69690940940604363796,"left2":5120,"right2":48649485744045701047618,"fanout":137112768},{"left1":5120,"right1":8716519703194405593874,"left2":7168,"right2":64756123019003550710046,"fanout":965363382496},{"left1":1024,"right1":65637972939497888853206,"left2":3088,"right2":45580603322855006617712,"fanout":157988957220},{"left1":3072,"right1":45600006609149414066242,"left2":10240,"right2":31526617903020673782684,"fanout":1034968685584},{"left1":2,"right1":126857501383634955196678,"left2":258,"right2":59048421932868044511662,"fanout":39004391892},{"left1":256,"right1":37470456748334835828000,"left2":272,"right2":31720844039219180789904,"fanout":3942499108},{"left1":4100,"right1":20912072492795803236554,"left2":6148,"right2":58649483685490976231562,"fanout":189180474620},{"left1":136,"right1":54218467732723751311890,"left2":34880,"right2":20943692352991745273072,"fanout":1068455434452},{"left1":512,"right1":77975573131730534753068,"left2":8704,"right2":8271397633485004747240,"fanout":880269596876},{"left1":32768,"right1":72420071880798198201850,"left2":33032,"right2":2919987043014772700530,"fanout":963082191060},{"left1":32,"right1":33681676866273280477382,"left2":1120,"right2":26159963901892429184354,"fanout":931742664924},{"left1":514,"right1":125798524331921193837042,"left2":16898,"right2":63150219949271117059214,"fanout":1017853301796},{"left1":4,"right1":124773463692788187152322,"left2":20,"right2":2056560504098586099544,"fanout":34791823396},{"left1":2048,"right1":83343758853421709022172,"left2":10242,"right2":132636552007147386533818,"fanout":120473528324},{"left1":16512,"right1":104534175011636182490030,"left2":49280,"right2":76323255461372760420128,"fanout":1048043112460},{"left1":16384,"right1":69835998525533368783750,"left2":16400,"right2":62165054439141916189990,"fanout":120865689628},{"left1":64,"right1":44249931735031234075734,"left2":33344,"right2":70782640853331793191434,"fanout":103898736696},{"left1":16,"right1":57781525686452187119104,"left2":272,"right2":26552392271195736169938,"fanout":269432830},{"left1":16448,"right1":130018904338618040267328,"left2":49216,"right2":117067695544053137698316,"fanout":226897221644},{"left1":2048,"right1":110463520198578936263148,"left2":8200,"right2":144484700406179660290106,"fanout":86584912948},{"left1":8,"right1":56941167400935753549514,"left2":6160,"right2":70385646765916652516248,"fanout":913901415440},{"left1":16384,"right1":27063701962532085554888,"left2":34816,"right2":123464899690101108382920,"fanout":138113513684},{"left1":128,"right1":56378361232015699020898,"left2":512,"right2":7183172886182543018656,"fanout":69605796076},{"left1":1024,"right1":69210902768217733067746,"left2":1538,"right2":30012524565327960584124,"fanout":123963442428},{"left1":2048,"right1":125135780642662197113682,"left2":4112,"right2":11211562429721250245916,"fanout":69661372452},{"left1":8,"right1":79749218412240177313968,"left2":68,"right2":14165544779939917444,"fanout":828139946012},{"left1":5120,"right1":58752907068834175986464,"left2":5152,"right2":11880245500190781873040,"fanout":1013826463796},{"left1":32,"right1":45970872533476571548140,"left2":416,"right2":43070731171420106665146,"fanout":1085699841076},{"left1":2,"right1":136598429434522588238032,"left2":6,"right2":42033092447612634768262,"fanout":4735043304},{"left1":8,"right1":138940232547418656872386,"left2":384,"right2":24354208394836749642516,"fanout":876387250220},{"left1":2080,"right1":10453122408319965534614,"left2":6160,"right2":30924192920135858850162,"fanout":1048188814528},{"left1":2048,"right1":62073682022582467105278,"left2":7168,"right2":64367731165591230073368,"fanout":172818438372},{"left1":2048,"right1":98659445015300601073862,"left2":8706,"right2":86959598375193483066460,"fanout":962544022736},{"left1":128,"right1":52865590076650009126476,"left2":16384,"right2":35240346019713063226018,"fanout":4227923188},{"left1":2,"right1":44459231189558844087066,"left2":256,"right2":55783211791156979744592,"fanout":7636791807},{"left1":8704,"right1":139554743949828322178594,"left2":8736,"right2":65700531162360008709696,"fanout":158860368924},{"left1":4,"right1":126656196149779626737154,"left2":132,"right2":26608312279068051991152,"fanout":243809390636},{"left1":514,"right1":1781234210540400562870,"left2":34816,"right2":53427066435143528065252,"fanout":983494622244},{"left1":16,"right1":66590835878782777580000,"left2":256,"right2":39445867032895596707102,"fanout":832739525628},{"left1":32768,"right1":37101713110501754337932,"left2":384,"right2":63078753615803108753692,"fanout":55232697400},{"left1":1024,"right1":49340067412333278349668,"left2":1288,"right2":74470352161136764323948,"fanout":824633777188},{"left1":16,"right1":35807055655488484935396,"left2":256,"right2":20367343364192282283878,"fanout":859597246502},{"left1":128,"right1":24039142482646220245050,"left2":176,"right2":14629128692047661640020,"fanout":896846794812},{"left1":32768,"right1":41142680006286178665846,"left2":32836,"right2":56891225777235697883924,"fanout":224024915188},{"left1":2048,"right1":68860076817598863107136,"left2":7168,"right2":48292063897069012048298,"fanout":137843499208},{"left1":64,"right1":27517461075842401409534,"left2":2064,"right2":39246815431224159387454,"fanout":1048657013968},{"left1":256,"right1":114527453229050072946738,"left2":258,"right2":90914395624320635273830,"fanout":5202976715},{"left1":16,"right1":52061945478126555731284,"left2":18,"right2":51810889398023834172394,"fanout":138056181054},{"left1":2,"right1":79185169133848412394986,"left2":18,"right2":58740899433163564158932,"fanout":824898892237},{"left1":64,"right1":55490060273791074180506,"left2":68,"right2":7548841968275467605638,"fanout":138327162100},{"left1":1024,"right1":81010360530074325208236,"left2":1028,"right2":133097552243481739478724,"fanout":982809585896},{"left1":32,"right1":47409931927493319885078,"left2":16480,"right2":7147814775089031140904,"fanout":54828254208},{"left1":16448,"right1":56558563243084097377212,"left2":16960,"right2":69285517830619560958556,"fanout":825320859684},{"left1":16,"right1":38255046272161166345578,"left2":20,"right2":67931465645464129623852,"fanout":5087348239},{"left1":48,"right1":56267679407414443863256,"left2":176,"right2":18817944768811557029084,"fanout":1017051145248},{"left1":1024,"right1":25243355371982579581664,"left2":4144,"right2":69466138081470377625738,"fanout":996515000320},{"left1":8,"right1":50837109465693819503408,"left2":4104,"right2":119104133757587844336,"fanout":52414436552},{"left1":514,"right1":105154063584775287683670,"left2":33282,"right2":76950879311295716105972,"fanout":261537725660},{"left1":32768,"right1":62575401473925439980828,"left2":49156,"right2":40888512797860138620366,"fanout":89725399052},{"left1":1028,"right1":39882085079476205200592,"left2":3076,"right2":6766635782305306788238,"fanout":72077805568},{"left1":8200,"right1":42833932057897105909936,"left2":8232,"right2":53547391475103989874200,"fanout":1068992309480},{"left1":16640,"right1":61368402151270077909720,"left2":18688,"right2":15819984939280367412206,"fanout":155169382616},{"left1":16384,"right1":52491590244829988252154,"left2":49664,"right2":13053678897614358822530,"fanout":189730143452},{"left1":32,"right1":26927555764324061869398,"left2":16452,"right2":35021376857401673405590,"fanout":945912868072},{"left1":32,"right1":8723529613736037577204,"left2":4096,"right2":72114522515671942682426,"fanout":210067318976},{"left1":264,"right1":88288440263270961894230,"left2":8456,"right2":117648642769261936708136,"fanout":825251860480},{"left1":16,"right1":71424557520687201889996,"left2":272,"right2":4225723584797592969840,"fanout":829060219406},{"left1":256,"right1":45230770690246745293722,"left2":272,"right2":42304929438844198826020,"fanout":21007181602},{"left1":64,"right1":73669253309661150239722,"left2":4096,"right2":38735276053431278556644,"fanout":103952692472},{"left1":2,"right1":9224422832072634861904,"left2":16,"right2":9538124519080140443252,"fanout":81205645859},{"left1":4,"right1":121016978878778776700188,"left2":260,"right2":30261852887537796010286,"fanout":43018820371},{"left1":512,"right1":95933292922216373009822,"left2":8704,"right2":122153196025036870863228,"fanout":103897692388},{"left1":512,"right1":33643439318385906749838,"left2":16960,"right2":67836715853707352800440,"fanout":141479690240},{"left1":5120,"right1":50951301676293087005006,"left2":7168,"right2":10374235946716996626664,"fanout":845707004132},{"left1":32768,"right1":116881964524124670774486,"left2":33282,"right2":131842745218955311906946,"fanout":483924212},{"left1":68,"right1":131913738555452509858140,"left2":8260,"right2":145484215818987623439712,"fanout":18200184864},{"left1":16,"right1":64504607729413408437292,"left2":1040,"right2":64655103538778204262790,"fanout":996646846672},{"left1":16384,"right1":37769809737825547688632,"left2":24576,"right2":31734147705408473206842,"fanout":1051732265992},{"left1":256,"right1":135800123700012552407398,"left2":258,"right2":138408630176219355438140,"fanout":22009944096},{"left1":2,"right1":100455420008629193819076,"left2":14,"right2":127319426249203406361304,"fanout":120275614920},{"left1":8,"right1":95048069383056263804738,"left2":16392,"right2":57931127952576164142106,"fanout":914173472972},{"left1":2048,"right1":80804204142004003247224,"left2":10752,"right2":135508772547685526590758,"fanout":999789956336},{"left1":64,"right1":100135540626947558730452,"left2":578,"right2":112786574879895972739176,"fanout":34362103012},{"left1":384,"right1":12546388592401452654286,"left2":8576,"right2":8233532438026957696832,"fanout":1065155566652},{"left1":8,"right1":71388860838239078343688,"left2":16384,"right2":13339397141316142364244,"fanout":72074924284},{"left1":32768,"right1":34275093063440089001410,"left2":384,"right2":813590636073316075992,"fanout":897513962728},{"left1":2048,"right1":142419446388244293750998,"left2":1028,"right2":32926754780757302544520,"fanout":103631352004},{"left1":64,"right1":53468964066588042819834,"left2":192,"right2":14975955524548744954500,"fanout":55511667948},{"left1":32,"right1":55101794075980703702288,"left2":16768,"right2":57857006769227853447848,"fanout":244074999036},{"left1":2,"right1":7118968586813695564262,"left2":18,"right2":47892857558489181122948,"fanout":68785664274},{"left1":1024,"right1":51473605554134824582188,"left2":5120,"right2":71506095713997993744058,"fanout":154901159980},{"left1":16384,"right1":101134912018448929020324,"left2":49664,"right2":109157064232827683537656,"fanout":72893606084},{"left1":264,"right1":68530273763490817002682,"left2":328,"right2":16246120912779114160568,"fanout":69540052208},{"left1":1028,"right1":17616403503652054404226,"left2":2080,"right2":19983512654438113136162,"fanout":896644480004},{"left1":256,"right1":9256895103749498219368,"left2":514,"right2":73178558842859902075542,"fanout":258652248276},{"left1":16448,"right1":62014060229877591254994,"left2":49216,"right2":70513177675410732277922,"fanout":4243913760},{"left1":8,"right1":134078932330412535016078,"left2":4120,"right2":39586158332267521929560,"fanout":89941405948},{"left1":4,"right1":39379689404428158382384,"left2":16516,"right2":25542247988840606422192,"fanout":258572550192},{"left1":512,"right1":77965957098297310104044,"left2":4624,"right2":51752226994600954357790,"fanout":17583318028},{"left1":32,"right1":30909337910368842151248,"left2":136,"right2":15843691728112189794292,"fanout":271589420},{"left1":4096,"right1":32485433328303950303334,"left2":4164,"right2":68950703752323198435428,"fanout":210132065512},{"left1":4112,"right1":47321677778106098765180,"left2":2080,"right2":43785111733462959433726,"fanout":828730045488},{"left1":4,"right1":10286853469791543871986,"left2":32774,"right2":98554310347426382113764,"fanout":176028966120},{"left1":8194,"right1":38234369219664034922270,"left2":10242,"right2":32754774756546855893280,"fanout":845840179232},{"left1":8192,"right1":108547635546431154046112,"left2":8320,"right2":75522070755681297282526,"fanout":175626306772},{"left1":16384,"right1":70893137376729746636176,"left2":16392,"right2":66021256230987903818526,"fanout":103430020340},{"left1":1104,"right1":17095218769899517969300,"left2":3088,"right2":2638745959985817360478,"fanout":928720885808},{"left1":32,"right1":43852396230746565515368,"left2":68,"right2":72618092050539493249206,"fanout":140729920724},{"left1":4096,"right1":13885641563280723552716,"left2":4128,"right2":13293442823983734023152,"fanout":876643881172},{"left1":32768,"right1":12749590285587804375232,"left2":384,"right2":39640618046171026534502,"fanout":1034149969128},{"left1":512,"right1":107312208625051511557162,"left2":16896,"right2":7617767871378599532092,"fanout":206429548568},{"left1":256,"right1":24394051017603456284000,"left2":512,"right2":90049736968267361945632,"fanout":979590769700},{"left1":4,"right1":32188640211113625750348,"left2":4096,"right2":37011042020351865937872,"fanout":1048455988416},{"left1":4096,"right1":14093458346459450843604,"left2":4128,"right2":68984897905967107351306,"fanout":982553464056},{"left1":2048,"right1":2075320516572233812092,"left2":392,"right2":56670346898321594806176,"fanout":3570989272}],"pure":{"0":866846884899,"1":868470353095,"2":3425905157,"4":63685207088,"32":69239948852,"80":158612856889,"96":1022423062038,"112":56739746365,"128":953885205197,"160":168189550053,"176":859295249360,"208":953297599548,"240":151260043807,"256":128735307713,"272":1009856205837,"304":1078543525085,"320":845104818124,"352":240589461481,"432":151379308537,"448":240263691284,"480":240247520293,"496":872850926841,"512":167385039085,"528":1057329891026,"560":979003063589,"624":1008967687422,"640":104069388087,"656":214274552789,"672":94268296214,"688":77322852121,"736":868171398124,"784":175053354266,"800":841698039313,"816":979668114709,"832":980158565115,"848":914071016421,"928":162421988918,"960":1094630116359,"992":1056901153792,"1008":137654189007,"1040":43924587468,"1088":171526988526,"1136":147050188345,"1216":837953643795,"48":1065133488687,"192":47465839859,"368":893719679014,"384":179401137439,"544":63617231041,"592":257646533672,"608":961872984535,"720":849970995192,"880":13300283132,"912":78347097634,"1056":202565418788,"1120":863455811086,"1152":209413720256,"1200":1085718410449,"5":953965659677,"6":189094234418,"8":235992250609,"9":997402224144,"10":146770615597,"13":106784420052,"224":99339456538,"288":1082233780790,"464":136804634583,"704":249545608205,"768":958618335518,"896":249880764454,"1024":210886325807,"1072":47664134443,"7":258707033328,"14":192403472632,"16":245668643051,"144":82410028088,"336":847085172212,"752":829009036009,"944":129099221486,"1104":957019709914,"1168":141433699891,"1184":171363191003,"11":1014353683664,"400":227099462693,"416":55563897028,"576":1085858450452,"976":1077882907432,"3":841127942173,"12":932777023259,"15":1083217338368,"864":99106677767,"64":1000177530338},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"three-phase","capacity":8,"product_order":[0,3,4,5,7,10,11,12,13,14,15,16,18,19,21,22,23,25,26,28,29,30,31,32,33,34,35,36,38,39,40,41,42,44,45,46,47,48,51,52,53,60,61,63,64,65,66,67,68,69,70,71,73,75,77,78,82,84,85,88,89,90,91,93,95,96,97,99,100,103,105,106,109,110,111,112,113,114,115,118,120,121,124,125,129,130,131,132,133,134,135,136,138,140,141,143,144,146,147,151,154,155,157,158,159,161,163,165,166,168,171,172,173,176,177,178,1,6,24,37,50,54,56,57,58,76,80,87,92,94,98,101,107,116,117,119,126,128,137,142,145,148,149,152,153,162,164,169,170,175,179,2,8,9,17,20,27,43,49,55,59,62,72,74,79,81,83,86,102,104,108,122,123,127,139,150,156,160,167,174],"events":[{"kind":"products","cache":503,"indices":[0,3,4,5,7,10,11,12,13,14,15,16,18,19,21,22,23,25,26,28,29,30,31,32,33,34,35,36,38,39,40,41,42,44,45,46,47,48,51,52,53,60,61,63,64,65,66,67,68,69,70,71,73,75,77,78,82,84,85,88,89,90,91,93,95,96,97,99,100,103,105,106,109,110,111,112,113,114,115,118,120,121,124,125,129,130,131,132,133,134,135,136,138,140,141,143,144,146,147,151,154,155,157,158,159,161,163,165,166,168,171,172,173,176,177,178]},{"kind":"pure_low","cache":503,"monomial":5},{"kind":"pure_low","cache":503,"monomial":6},{"kind":"pure_low","cache":503,"monomial":9},{"kind":"pure_low","cache":503,"monomial":10},{"kind":"pure_low","cache":503,"monomial":13},{"kind":"pure_low","cache":503,"monomial":11},{"kind":"pure_low","cache":503,"monomial":3},{"kind":"pure_low","cache":503,"monomial":12},{"kind":"products","cache":863,"indices":[1,6,24,37,50,54,56,57,58,76,80,87,92,94,98,101,107,116,117,119,126,128,137,142,145,148,149,152,153,162,164,169,170,175,179]},{"kind":"pure_low","cache":863,"monomial":7},{"kind":"pure_low","cache":863,"monomial":14},{"kind":"products","cache":1631,"indices":[2,8,9,17,20,27,43,49,55,59,62,72,74,79,81,83,86,102,104,108,122,123,127,139,150,156,160,167,174]},{"kind":"pure_low","cache":1631,"monomial":15}],"low_computations":11}},{"batch":12,"delta_sha256":"d8b4a9a8af4be9f07615a918f9eaeeda566ee023f0194edaea64286b37c1b875","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":514,"right1":52755828837021029357242,"left2":2562,"right2":1641841166095299746532,"fanout":1066061407452},{"left1":6144,"right1":55129210767958030665500,"left2":6160,"right2":28481944510747826549874,"fanout":140661097708},{"left1":16,"right1":67130725138136602615744,"left2":656,"right2":48694418786263409069932,"fanout":163481320164},{"left1":33280,"right1":57901885754027857697584,"left2":33284,"right2":30540954572307698549464,"fanout":1051898215668},{"left1":256,"right1":50099903374317418918294,"left2":272,"right2":70751024009772735528140,"fanout":209429082915},{"left1":16,"right1":59110138725509238650368,"left2":4100,"right2":69082036349505312992524,"fanout":181005195276},{"left1":144,"right1":19324034505751279785134,"left2":8336,"right2":17117874284004151678030,"fanout":146064679150},{"left1":2,"right1":92484641828460916810124,"left2":18,"right2":47118946498290398275298,"fanout":829060161599},{"left1":8192,"right1":39300525433909530337974,"left2":8194,"right2":67580490397215467066170,"fanout":1005743432396},{"left1":128,"right1":37949208086445257112876,"left2":640,"right2":67901147544992828917922,"fanout":910614343900},{"left1":16386,"right1":70625284068125015517780,"left2":16450,"right2":9951527048824266857396,"fanout":962322377446},{"left1":32768,"right1":33481454938873769380148,"left2":32864,"right2":73973968287618544436218,"fanout":893958365388},{"left1":512,"right1":1486838047218846282398,"left2":32768,"right2":26321324989100179585456,"fanout":824838646006},{"left1":64,"right1":51080982585565296625720,"left2":9216,"right2":48093243655520741754652,"fanout":1048378356464},{"left1":128,"right1":96317036462270676683550,"left2":130,"right2":122404642191090020889392,"fanout":175559731924},{"left1":96,"right1":68045247177465945100570,"left2":32864,"right2":27018881032891324728650,"fanout":850617435872},{"left1":256,"right1":127324546399457501353220,"left2":768,"right2":25344816640438097641918,"fanout":962442561236},{"left1":1024,"right1":43888833960569429523882,"left2":4100,"right2":63724316248494850687582,"fanout":1017205814980},{"left1":6144,"right1":39409639438007294241666,"left2":14336,"right2":11384414326213385231056,"fanout":86553191126},{"left1":32,"right1":57583859015103056411640,"left2":9248,"right2":17088393569091463072652,"fanout":21409613344},{"left1":1024,"right1":11194294832596792034678,"left2":5124,"right2":34022147333458024496554,"fanout":1066059563018},{"left1":64,"right1":16992715510961052469998,"left2":6208,"right2":59120271904722403715262,"fanout":55670736078},{"left1":4096,"right1":143981192131246514988634,"left2":4108,"right2":112347500854896180715176,"fanout":868358044164},{"left1":8,"right1":106792798926106868668512,"left2":49160,"right2":1463547222432363481206,"fanout":258452103396},{"left1":16384,"right1":52338969397331653208766,"left2":16416,"right2":61225842359096809858106,"fanout":120326328358},{"left1":4,"right1":50086236502655474684590,"left2":36,"right2":18670685008466422496152,"fanout":828693813776},{"left1":12,"right1":19966323479683366205684,"left2":76,"right2":11758095786031397306244,"fanout":132205128398},{"left1":1024,"right1":69188980662448306058756,"left2":17410,"right2":54886801145480022378268,"fanout":269792454178},{"left1":16384,"right1":28062206179574707919684,"left2":49152,"right2":31122270466552567682102,"fanout":86807687684},{"left1":8192,"right1":88862007206609342171558,"left2":8200,"right2":102692939783617545111510,"fanout":149890148876},{"left1":1024,"right1":61795617198017535618816,"left2":25600,"right2":62248674975873926148728,"fanout":1091035076144},{"left1":16,"right1":62518360741592133057938,"left2":112,"right2":41590420411400574582978,"fanout":911005115100},{"left1":8,"right1":142955627093176138194784,"left2":32776,"right2":44810711259490990562032,"fanout":894360493084},{"left1":4096,"right1":3802555425020780056072,"left2":16384,"right2":36154843944702402098388,"fanout":123684057296},{"left1":16,"right1":144530372217142118399198,"left2":272,"right2":139736367128371013278800,"fanout":20566518760},{"left1":256,"right1":71797738131595147863316,"left2":1280,"right2":68913337376955507864172,"fanout":60734493394},{"left1":512,"right1":68204975187224950944056,"left2":96,"right2":38559720764047550876694,"fanout":235635352792},{"left1":64,"right1":41295968821713881853468,"left2":9216,"right2":16258358080061668515626,"fanout":854398731984},{"left1":1024,"right1":67123587898680641872596,"left2":5120,"right2":61105060483674289703966,"fanout":218676262648},{"left1":4,"right1":3122026343075184306850,"left2":1028,"right2":60195409002778769654236,"fanout":834065720354},{"left1":8,"right1":92609874422168455404286,"left2":4096,"right2":35949766301102087755474,"fanout":868236063766},{"left1":8,"right1":26016609950911720317596,"left2":1096,"right2":63153408445152777152352,"fanout":140774536194},{"left1":2,"right1":19234850233013089868772,"left2":258,"right2":73997881202988872099124,"fanout":47763367382},{"left1":32768,"right1":20111072589221043447594,"left2":144,"right2":34036429901322540036840,"fanout":868535558894},{"left1":512,"right1":65312897265112455970500,"left2":24576,"right2":1165463082828433075792,"fanout":1040134182622},{"left1":4,"right1":22842735373580278152726,"left2":268,"right2":43716784158593189520518,"fanout":80845347054},{"left1":144,"right1":26255996293386792382388,"left2":2192,"right2":33708874739155439192736,"fanout":132607770662},{"left1":4,"right1":114069876076254376793592,"left2":260,"right2":121628759562297052574508,"fanout":5137560539},{"left1":128,"right1":22759717634092479987014,"left2":2064,"right2":56887207157801125421526,"fanout":888860625142},{"left1":32768,"right1":74659986484147536581854,"left2":33408,"right2":65275066894067211876376,"fanout":1083386292978},{"left1":2048,"right1":63982507142590935055872,"left2":2192,"right2":4270721970861426738616,"fanout":894160738346},{"left1":4096,"right1":72178908345592326859716,"left2":5120,"right2":66050092773607557057420,"fanout":893589976314},{"left1":4,"right1":34428508224757169765330,"left2":20,"right2":19660684204709012026314,"fanout":757533485},{"left1":16384,"right1":27983000505524442807912,"left2":2064,"right2":72691430445032172229634,"fanout":3828019718},{"left1":1088,"right1":48935849723502415408828,"left2":24576,"right2":55331321310389120962570,"fanout":1026175273182},{"left1":2,"right1":84632709940826955992338,"left2":642,"right2":74552528196344235745154,"fanout":1051947749070},{"left1":512,"right1":27984259964744530269362,"left2":16386,"right2":36343916110244436314238,"fanout":962557515000},{"left1":256,"right1":36323435881077045846846,"left2":2304,"right2":63353348345684504533912,"fanout":175492044838},{"left1":64,"right1":53789136190681882883178,"left2":24640,"right2":38620101982110642540440,"fanout":863158331918},{"left1":4,"right1":11888332432052943339836,"left2":256,"right2":54602679451178002060832,"fanout":216732150},{"left1":640,"right1":59720865829523243525312,"left2":656,"right2":8029241180968986175262,"fanout":957109297170},{"left1":8,"right1":55360563040596358959432,"left2":16,"right2":58023109635065870707788,"fanout":1005542103590},{"left1":12,"right1":104307851977324627521334,"left2":16396,"right2":47635458885912738510996,"fanout":206194655482},{"left1":8,"right1":136276243982138391785054,"left2":8200,"right2":10004710357982774653020,"fanout":871457148124},{"left1":2064,"right1":68383879856732034704368,"left2":2072,"right2":70473931904515776984868,"fanout":210288655416},{"left1":144,"right1":34108619091167600561900,"left2":16528,"right2":17475432336123682577402,"fanout":906018741792},{"left1":2,"right1":27379910332695564353030,"left2":4,"right2":31889112344287848965182,"fanout":828109697741},{"left1":16384,"right1":11599172010738471843164,"left2":1120,"right2":60994611461082328432096,"fanout":927761755698},{"left1":4,"right1":49247989391662574101886,"left2":16,"right2":39837141107717804746976,"fanout":141042392892},{"left1":512,"right1":70085993627827322686654,"left2":16384,"right2":71966023916983502004722,"fanout":833261020686},{"left1":256,"right1":10027846926082709664376,"left2":258,"right2":148536866432352019297666,"fanout":832607023554},{"left1":4,"right1":144451993629806369247028,"left2":1092,"right2":53629485609806987828944,"fanout":123483768022},{"left1":8192,"right1":33616838113259077005420,"left2":1088,"right2":50291382512147107861878,"fanout":962757975042},{"left1":8192,"right1":33252336431824326794018,"left2":8448,"right2":38521684573702246331464,"fanout":868035209732},{"left1":2,"right1":54426742905143173358984,"left2":256,"right2":35144159704805941352976,"fanout":13810331435},{"left1":32768,"right1":125082413383945482319000,"left2":32770,"right2":130283533338769425217572,"fanout":106703228944},{"left1":64,"right1":42621516427620431592244,"left2":33280,"right2":55590036039626677070782,"fanout":982789706788},{"left1":9216,"right1":2783139705027862233484,"left2":13312,"right2":14022425777410704366530,"fanout":17531811902},{"left1":32896,"right1":30408307331956758811018,"left2":33408,"right2":32249175864682460562264,"fanout":911070119996},{"left1":32,"right1":39652402884443507291932,"left2":6144,"right2":50629534789501941197312,"fanout":923232764608},{"left1":12,"right1":15837372686070157999054,"left2":2060,"right2":37666623430447596427956,"fanout":1073910049280},{"left1":4096,"right1":75517852504368403595392,"left2":4384,"right2":50870963342634478446620,"fanout":64257663170},{"left1":2,"right1":109962457594159578235658,"left2":8194,"right2":61275497651556133933464,"fanout":26309038298},{"left1":512,"right1":60253578199207363182340,"left2":96,"right2":24196507053436483209936,"fanout":181062612004},{"left1":2048,"right1":28851349955003425198370,"left2":4096,"right2":25886477656634868206088,"fanout":163613907690},{"left1":128,"right1":21315822478236319806312,"left2":24704,"right2":69828780121550847493750,"fanout":1042651951666},{"left1":1088,"right1":25676884686487565987370,"left2":1120,"right2":13192581526391550285930,"fanout":863034208786},{"left1":32770,"right1":60828847994705330115296,"left2":32834,"right2":37056443008886059236874,"fanout":859647261218},{"left1":32770,"right1":11556833743598454240518,"left2":34818,"right2":70099779740612790158590,"fanout":914707194616},{"left1":8,"right1":147843225511519013290562,"left2":256,"right2":24438389562434839667596,"fanout":200994194440},{"left1":32,"right1":15622163649229507132936,"left2":2688,"right2":64626847082813228860880,"fanout":997351631892},{"left1":8192,"right1":57423670017310629198018,"left2":8224,"right2":62558640929291461432640,"fanout":828709934808},{"left1":264,"right1":115443917126384369499748,"left2":296,"right2":9251885837123352845424,"fanout":988026559516},{"left1":64,"right1":60594897091504143759084,"left2":6144,"right2":59956155850498811794862,"fanout":163648909348},{"left1":1024,"right1":6396782445194814650290,"left2":1312,"right2":48304839355452662537572,"fanout":871544703708},{"left1":12,"right1":90376887563245538940278,"left2":4108,"right2":68979191486013423010678,"fanout":922558806554},{"left1":32770,"right1":74079925276066883910982,"left2":49154,"right2":49389725647730973084068,"fanout":863189784314},{"left1":4,"right1":130183943421026372770230,"left2":256,"right2":134462516858792426846822,"fanout":206744983539},{"left1":2048,"right1":7497498670777469958702,"left2":24576,"right2":66667453573371405549980,"fanout":137608491022},{"left1":4096,"right1":43819461225581653030554,"left2":96,"right2":74731901497207437239878,"fanout":34842883292},{"left1":2,"right1":139817979907233573810490,"left2":4,"right2":140383617136225998848370,"fanout":893758546926},{"left1":16,"right1":10479736426812339546716,"left2":288,"right2":11092026872871871861576,"fanout":18120899814},{"left1":264,"right1":127409592177966584410238,"left2":280,"right2":46500520708727045687444,"fanout":232267724508},{"left1":2048,"right1":7122058980891306485782,"left2":11264,"right2":68046233457988089243496,"fanout":1022822109720},{"left1":512,"right1":61162393097885947327022,"left2":25088,"right2":68421600208007928686594,"fanout":871289647624},{"left1":16,"right1":141458425604467551628546,"left2":272,"right2":147107633192115882153518,"fanout":155454864599},{"left1":64,"right1":61267942498400074486972,"left2":4100,"right2":44021005109046890849314,"fanout":914035719724},{"left1":2,"right1":81627821381596882986504,"left2":8,"right2":23367071965188590069550,"fanout":939524947678},{"left1":4100,"right1":45218760858191923582764,"left2":4116,"right2":25983822051667993671742,"fanout":175873203418},{"left1":4,"right1":107716560768343356051250,"left2":644,"right2":14305023970697480750898,"fanout":240956080354},{"left1":4,"right1":56279782672720819493074,"left2":1024,"right2":37768449560794520059716,"fanout":26054375630},{"left1":16386,"right1":7740695431379297499018,"left2":17410,"right2":50617146561698585792374,"fanout":1077500499168},{"left1":16384,"right1":58305754034013994615196,"left2":12,"right2":137738952900321573086562,"fanout":893400528072},{"left1":4096,"right1":119335327571734823454920,"left2":4360,"right2":123804580439957252674038,"fanout":828171805412},{"left1":4,"right1":79704193111084700285420,"left2":32,"right2":10169097942133004025636,"fanout":828026568914},{"left1":16384,"right1":4299295479138513276480,"left2":16704,"right2":71815841229161979013314,"fanout":1095131121404},{"left1":1024,"right1":74236771261155522400562,"left2":1280,"right2":57404636132559607826538,"fanout":26712000540},{"left1":288,"right1":18980655594946581720532,"left2":33056,"right2":54296074864118974428798,"fanout":157855779364},{"left1":9216,"right1":38301873205891969741044,"left2":9248,"right2":17587640574518238549880,"fanout":845270806210},{"left1":32768,"right1":31826245438622089044286,"left2":2192,"right2":18069318545047758514888,"fanout":997285435614},{"left1":64,"right1":64690040574238120890116,"left2":4100,"right2":36321119163318561249364,"fanout":63756315678},{"left1":4096,"right1":67102146888423503779352,"left2":4104,"right2":71132433306379118617592,"fanout":17932627998},{"left1":12,"right1":142020524391507815980772,"left2":32780,"right2":46735751326056510044578,"fanout":939594356740},{"left1":4,"right1":142141329516109238503410,"left2":264,"right2":3823825347029680993044,"fanout":1083037720126},{"left1":128,"right1":73885197218419039969436,"left2":8192,"right2":32594634754611583959534,"fanout":111941570576},{"left1":4096,"right1":82669756497966746060222,"left2":4352,"right2":150239185905808543354950,"fanout":244592418512},{"left1":264,"right1":103247474618676514329340,"left2":1288,"right2":51984726442075006185748,"fanout":137540541154},{"left1":2048,"right1":5116118178442357796914,"left2":35328,"right2":68501297817122098604706,"fanout":235802248748},{"left1":1024,"right1":52121182365488739820082,"left2":7168,"right2":15892638443332612365904,"fanout":919170120224},{"left1":320,"right1":74546328879685452368686,"left2":352,"right2":3537829287523351536058,"fanout":140963805206},{"left1":144,"right1":51702602274414687604458,"left2":148,"right2":71807264543221641411206,"fanout":146613739766},{"left1":128,"right1":24164620332254096187664,"left2":130,"right2":45736218592452338516754,"fanout":1022235824326},{"left1":2,"right1":56661751364822279004746,"left2":260,"right2":137815677008429212365664,"fanout":99171628538},{"left1":64,"right1":45975356040645134576608,"left2":704,"right2":44427080640553829601918,"fanout":876408522776},{"left1":512,"right1":46535858188338651339190,"left2":264,"right2":36321873492394765037406,"fanout":851289975546},{"left1":9216,"right1":124324879710926573391080,"left2":9224,"right2":101824179459886659959022,"fanout":158746021586},{"left1":64,"right1":66466038795140146695882,"left2":352,"right2":43934378069482814093706,"fanout":249848791562},{"left1":2048,"right1":61403309469796918186602,"left2":33408,"right2":48115605367306418541322,"fanout":919291693778},{"left1":32768,"right1":50465895718214070636484,"left2":288,"right2":38166905477396747629862,"fanout":235820604122},{"left1":4,"right1":31293627743859406994770,"left2":6,"right2":45344808185041881673654,"fanout":8474256609},{"left1":512,"right1":31046930201262610654992,"left2":32768,"right2":56797061230326461475984,"fanout":69393109522},{"left1":2,"right1":134507009005453596235368,"left2":512,"right2":25210741754207000304384,"fanout":137474095340},{"left1":128,"right1":74988451559176439148278,"left2":160,"right2":23740299987448562619512,"fanout":243775977522},{"left1":128,"right1":62535991094876478794612,"left2":16512,"right2":52615142173946960883146,"fanout":837500063436},{"left1":64,"right1":47369308575993193553174,"left2":192,"right2":31414088215715412203002,"fanout":17483840722},{"left1":128,"right1":24590167399217616437120,"left2":2192,"right2":74921884123200932390756,"fanout":226559840256},{"left1":64,"right1":63459401394268328805748,"left2":2064,"right2":55400436128434392175018,"fanout":1091740902448},{"left1":16386,"right1":32532263274330729699028,"left2":49154,"right2":42684010726435439085678,"fanout":35401943606},{"left1":2048,"right1":43387365371776466568208,"left2":2144,"right2":47664965559587084430880,"fanout":206832060986},{"left1":512,"right1":24658144546983813880130,"left2":544,"right2":40199129016054967777550,"fanout":98161972742},{"left1":2048,"right1":33629709909333798169704,"left2":2312,"right2":56811561474829382378518,"fanout":828373864696},{"left1":40,"right1":2802669852242230957650,"left2":32808,"right2":26526925812207632269292,"fanout":210355805698},{"left1":512,"right1":75511334578515880383478,"left2":16384,"right2":73314389979595136647812,"fanout":18019576046},{"left1":32768,"right1":3395236463895312492330,"left2":32896,"right2":71131618450422984172934,"fanout":47112135192},{"left1":2048,"right1":75512829169928740804138,"left2":6148,"right2":65196776083538443224738,"fanout":954032850992},{"left1":4096,"right1":55601969142709830078510,"left2":1088,"right2":20415582709391249208918,"fanout":1043456774372},{"left1":256,"right1":116090838897700575780926,"left2":272,"right2":72932706929095952895030,"fanout":25301950145},{"left1":1024,"right1":272128255141913798150,"left2":4096,"right2":58579507007879767732926,"fanout":94672535244},{"left1":16,"right1":72491387779726867113980,"left2":33280,"right2":19217385099479572496094,"fanout":884946815520},{"left1":2,"right1":49736305573519203027694,"left2":1024,"right2":42548776193054145252306,"fanout":965913615042},{"left1":1024,"right1":24226623172375671747664,"left2":17408,"right2":38179846296838269635484,"fanout":833307023406},{"left1":2,"right1":101195994784454139164644,"left2":18,"right2":6387789686962008562784,"fanout":550974499},{"left1":2048,"right1":47405882818270369871230,"left2":24576,"right2":27263893277682858551610,"fanout":834010925596},{"left1":8,"right1":115406624549983768564914,"left2":640,"right2":38685535842909970132926,"fanout":828593807040},{"left1":32,"right1":10301744943542300122184,"left2":160,"right2":33184950375510303025420,"fanout":52044815064},{"left1":8192,"right1":73906118217669402615718,"left2":8224,"right2":23149679477686965703292,"fanout":226976386052},{"left1":1088,"right1":69636012191015939250366,"left2":5184,"right2":18082112395300588397678,"fanout":141298028784},{"left1":8192,"right1":29441660720113962856726,"left2":144,"right2":37040543298545791157442,"fanout":141109630472},{"left1":16386,"right1":51605084299865906829178,"left2":16514,"right2":35256211160003866156468,"fanout":1043109562370},{"left1":8192,"right1":25284312960850950203332,"left2":6144,"right2":25489210648726922051104,"fanout":124134554356},{"left1":16386,"right1":56355067228849661485834,"left2":16450,"right2":20397144631738571858276,"fanout":103716754684},{"left1":128,"right1":134195906348133215517274,"left2":384,"right2":93548467687240802081472,"fanout":957087752736},{"left1":64,"right1":42712087830530996677362,"left2":6160,"right2":72911165087569721986582,"fanout":112072473818},{"left1":256,"right1":150348335935029451386874,"left2":4096,"right2":5413420635038515212702,"fanout":931252548122},{"left1":2064,"right1":50909935945213014235514,"left2":33280,"right2":37851101231308780157522,"fanout":72042299118},{"left1":2,"right1":84281958746546896215504,"left2":32768,"right2":46413276161035439428920,"fanout":215231154940},{"left1":8192,"right1":70622582406470433163534,"left2":640,"right2":51418975812271146499550,"fanout":1017072513744},{"left1":128,"right1":52986138143216778627406,"left2":32770,"right2":73962797001961642987008,"fanout":974032475390},{"left1":32,"right1":13970048134185248883628,"left2":24578,"right2":43886899568398152638034,"fanout":68721761504},{"left1":32770,"right1":68306663749563775344852,"left2":32786,"right2":48823957228955222593354,"fanout":939592913970},{"left1":16,"right1":42252741894887419770602,"left2":128,"right2":52858675248872025790342,"fanout":902683962598},{"left1":32768,"right1":20243659889703861042622,"left2":49152,"right2":38039570616799762789304,"fanout":206762995906},{"left1":4,"right1":21012259730369669381634,"left2":12,"right2":85985291934281755832772,"fanout":25972631286},{"left1":8192,"right1":60209257536758474202240,"left2":12296,"right2":53734934760193870080176,"fanout":35349202686},{"left1":260,"right1":129040294756484326105496,"left2":276,"right2":63307815636327245884398,"fanout":175137083362},{"left1":2048,"right1":61020623192299640096038,"left2":25600,"right2":15153059406401647772234,"fanout":163243802674},{"left1":2,"right1":40175001563048191071518,"left2":256,"right2":23550896387123641815320,"fanout":37630435030},{"left1":2,"right1":13266882094645112820874,"left2":6,"right2":35609503169414215244200,"fanout":7600850655},{"left1":264,"right1":116716749285804376068672,"left2":4360,"right2":72053096885820065608818,"fanout":17933729012},{"left1":2048,"right1":58548410198489372814514,"left2":16386,"right2":59876861471021359009766,"fanout":29408160966},{"left1":16,"right1":35401500082073974048420,"left2":1088,"right2":74937684733010024482214,"fanout":1025537795274},{"left1":640,"right1":131955419530753007585052,"left2":648,"right2":95094058368347125523082,"fanout":945746541576},{"left1":128,"right1":43041388640409471163848,"left2":2176,"right2":8561032890626282420520,"fanout":43332019718},{"left1":8192,"right1":67705301186165339791922,"left2":8704,"right2":48356552656309780242584,"fanout":893825065172},{"left1":2,"right1":147335718406495602619664,"left2":4,"right2":94821814893358747629620,"fanout":137894090548},{"left1":8192,"right1":44179491376804278727518,"left2":2064,"right2":27852667448096736129042,"fanout":132507946188},{"left1":24576,"right1":35877542520613134709528,"left2":25600,"right2":15988326309672137008554,"fanout":828527611956},{"left1":144,"right1":27002467394121436574382,"left2":16528,"right2":25008879521522689088930,"fanout":980137675506},{"left1":4,"right1":109106559639220971220980,"left2":12,"right2":39315630919464125818546,"fanout":931755071002},{"left1":256,"right1":46287703853499608939452,"left2":288,"right2":66519860887868974068414,"fanout":71955045394},{"left1":512,"right1":12040147425950227331606,"left2":33282,"right2":67084093153081894062176,"fanout":69055414838},{"left1":64,"right1":6305961655991254634732,"left2":32960,"right2":22501800120167472789136,"fanout":63924080346},{"left1":16384,"right1":73317577406905013630918,"left2":4100,"right2":43989265072421773339638,"fanout":845675024440}],"pure":{"0":119688909059,"1":8088047330,"4":231558931744,"16":41996252953,"32":889507937281,"48":877144781320,"64":845317469477,"80":202823697669,"112":1078956122562,"144":983247472136,"160":872932710421,"224":133006242758,"320":99789696242,"352":849417128130,"384":111619999527,"400":78367670984,"416":885233280480,"432":3623481384,"464":243978350043,"496":862446025475,"512":163125469681,"528":98384680242,"544":188980585178,"560":69439656231,"576":176610274049,"608":868486549711,"624":991217707544,"656":871538817021,"720":60299339529,"768":1044480186151,"864":132222730552,"928":949978460963,"960":121034452439,"992":931021666874,"1008":175974189341,"1040":12932294593,"1056":29946350368,"1088":197733516312,"1104":1052199939813,"1136":232245612291,"1152":950042368965,"1184":1087344222504,"1200":43904003090,"2":901839700287,"3":91031334707,"6":1077396162553,"7":20874007286,"9":18005744689,"10":949873738730,"11":1023256755912,"12":897718295499,"13":928555784424,"14":206228613346,"15":991815545048,"128":1017137532651,"176":26208031965,"192":111079901640,"240":68939342547,"256":978348465919,"336":1082111570682,"448":910180664011,"640":60798609452,"752":68313751051,"784":168360016696,"832":1031177112124,"912":1044702113226,"944":180791022123,"976":145273467412,"1024":90664405474,"1120":1017907567144,"96":266576326613,"272":236319677978,"304":1010155794667,"368":119604044314,"480":1010038922777,"592":842734909909,"672":133801715243,"688":1078409376500,"800":73821782214,"880":20752560846,"896":188745960666,"1072":150825592079,"1216":1082330566367,"5":197568225506,"8":270348763852,"288":970663081479,"816":35328877585,"704":884951560136,"736":1048747193291,"848":961032226782,"1168":1070255450092,"208":885430671871},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"three-phase","capacity":8,"product_order":[1,4,5,7,8,10,13,17,18,19,20,21,22,24,25,26,27,29,30,31,33,34,35,37,38,39,40,41,42,45,47,51,52,53,54,57,58,59,61,62,63,64,66,67,68,70,71,72,73,74,77,79,80,81,82,84,86,89,91,92,93,94,95,97,98,99,100,101,102,103,105,106,107,108,110,111,112,113,114,115,116,118,120,121,123,125,126,128,129,132,135,136,139,146,148,150,154,155,156,157,159,160,161,162,165,166,169,170,172,173,178,182,183,184,185,186,187,188,189,190,194,195,196,198,199,202,0,2,6,9,14,16,36,44,46,48,50,55,56,60,65,69,83,85,90,104,109,124,130,131,133,134,141,142,143,144,145,149,152,163,164,167,168,171,176,180,191,192,193,197,3,11,12,15,23,28,32,43,49,75,76,78,87,88,96,117,119,122,127,137,138,140,147,151,153,158,174,175,177,179,181,200,201],"events":[{"kind":"products","cache":999,"indices":[1,4,5,7,8,10,13,17,18,19,20,21,22,24,25,26,27,29,30,31,33,34,35,37,38,39,40,41,42,45,47,51,52,53,54,57,58,59,61,62,63,64,66,67,68,70,71,72,73,74,77,79,80,81,82,84,86,89,91,92,93,94,95,97,98,99,100,101,102,103,105,106,107,108,110,111,112,113,114,115,116,118,120,121,123,125,126,128,129,132,135,136,139,146,148,150,154,155,156,157,159,160,161,162,165,166,169,170,172,173,178,182,183,184,185,186,187,188,189,190,194,195,196,198,199,202]},{"kind":"pure_low","cache":999,"monomial":3},{"kind":"pure_low","cache":999,"monomial":6},{"kind":"pure_low","cache":999,"monomial":10},{"kind":"pure_low","cache":999,"monomial":11},{"kind":"pure_low","cache":999,"monomial":12},{"kind":"pure_low","cache":999,"monomial":13},{"kind":"pure_low","cache":999,"monomial":14},{"kind":"pure_low","cache":999,"monomial":5},{"kind":"products","cache":863,"indices":[0,2,6,9,14,16,36,44,46,48,50,55,56,60,65,69,83,85,90,104,109,124,130,131,133,134,141,142,143,144,145,149,152,163,164,167,168,171,176,180,191,192,193,197]},{"kind":"pure_low","cache":863,"monomial":7},{"kind":"pure_low","cache":863,"monomial":9},{"kind":"products","cache":1631,"indices":[3,11,12,15,23,28,32,43,49,75,76,78,87,88,96,117,119,122,127,137,138,140,147,151,153,158,174,175,177,179,181,200,201]},{"kind":"pure_low","cache":1631,"monomial":15}],"low_computations":11}},{"batch":13,"delta_sha256":"5aac691b7e6d6c4640663e4a6981b2188adfa1fd8340762c2ef0f46a88dfa216","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":8192,"right1":23294155219497863602870,"left2":192,"right2":23418388455648004261098,"fanout":206362702896},{"left1":16,"right1":35528073803890318539802,"left2":18,"right2":40251941751602070349450,"fanout":833710707226},{"left1":64,"right1":40392937563474665007152,"left2":48,"right2":41641569323996379902070,"fanout":90073790708},{"left1":4,"right1":135666698779816824851744,"left2":20,"right2":69775471785532762198254,"fanout":829652003797},{"left1":256,"right1":57427980441534132390262,"left2":260,"right2":56687169432172001247486,"fanout":1004857272878},{"left1":32768,"right1":27035141495067091894068,"left2":34880,"right2":47132973348139069943218,"fanout":927847156756},{"left1":4,"right1":149938299482986127192056,"left2":6,"right2":6034930944890568075450,"fanout":196597580313},{"left1":1024,"right1":10437217169457841267804,"left2":1032,"right2":60784391147151934410852,"fanout":879945633820},{"left1":33280,"right1":61657238462990252846496,"left2":35328,"right2":60530847239498194817818,"fanout":72345241632},{"left1":4096,"right1":126702728853311451814410,"left2":33280,"right2":71089175329488121250828,"fanout":157842991316},{"left1":64,"right1":57356135060389704905116,"left2":1090,"right2":11256809955100833947224,"fanout":210322387980},{"left1":16,"right1":42000755186963857731988,"left2":256,"right2":58014842924648795714122,"fanout":12297099230},{"left1":12,"right1":97060492060239836638542,"left2":4108,"right2":147996819381600077945166,"fanout":876576246796},{"left1":2048,"right1":26140864953327103488162,"left2":16384,"right2":7007469504955925693120,"fanout":3893898300},{"left1":256,"right1":7852928626841052437782,"left2":258,"right2":68120048206327118235390,"fanout":73420163096},{"left1":128,"right1":92311089354675119580844,"left2":44,"right2":37894370925086178168378,"fanout":72947909848},{"left1":32768,"right1":15687283657479314325702,"left2":2304,"right2":6875115264197875871446,"fanout":825049752584},{"left1":128,"right1":33873052066037321990734,"left2":9216,"right2":1413762484545441350718,"fanout":979870424092},{"left1":4,"right1":21256668375798286597212,"left2":4096,"right2":68732086486940101931316,"fanout":896577057840},{"left1":4,"right1":70785990408420799215560,"left2":6,"right2":729018651157859595922,"fanout":966923270859},{"left1":16384,"right1":11468325921700287936904,"left2":16416,"right2":14111283004615140439820,"fanout":175167248600},{"left1":2,"right1":109351913567428086724312,"left2":16,"right2":73247547133927026486286,"fanout":158507541297},{"left1":128,"right1":119151616280958420024740,"left2":24578,"right2":48166292688674347639376,"fanout":192804878540},{"left1":32784,"right1":17942177153765242830784,"left2":33296,"right2":31363933160150934034302,"fanout":914774605880},{"left1":128,"right1":12659182094157637558594,"left2":2432,"right2":9994184434520099081696,"fanout":962612961512},{"left1":1024,"right1":49081245212985355904800,"left2":1152,"right2":36575670116304143854260,"fanout":910814146812},{"left1":8192,"right1":57313070791252745746474,"left2":8512,"right2":22811056710930450147814,"fanout":1048792278040},{"left1":128,"right1":119589075652750138284996,"left2":17536,"right2":4242543303495699390120,"fanout":261001628904},{"left1":16384,"right1":10902213279029786235948,"left2":16704,"right2":49529087744731815292076,"fanout":896979500036},{"left1":4,"right1":15423893223506822920834,"left2":256,"right2":50148502836474951751000,"fanout":838256817946},{"left1":512,"right1":53598483658210950974986,"left2":544,"right2":61248115571276065644666,"fanout":914224368688},{"left1":128,"right1":57057971544860023413978,"left2":16514,"right2":17194060202357581916136,"fanout":141346794740},{"left1":2304,"right1":4277478312682579223586,"left2":35072,"right2":67437250883971194646412,"fanout":965644663848},{"left1":1024,"right1":106649864158420395566028,"left2":5120,"right2":114240584050862534014252,"fanout":948384564436},{"left1":2048,"right1":37395176488827304107052,"left2":32784,"right2":75028170330812213088938,"fanout":209527823552},{"left1":33280,"right1":25010766159727868410486,"left2":33344,"right2":41040397957398728965966,"fanout":38389991440},{"left1":16384,"right1":66159974807359702240950,"left2":48,"right2":18190551082055457855982,"fanout":209864420592},{"left1":4096,"right1":27120442908061441766720,"left2":17408,"right2":8365387617641240869112,"fanout":896856288304},{"left1":4,"right1":149318547708223882895780,"left2":8198,"right2":73033528154700147768794,"fanout":1068844461060},{"left1":4096,"right1":652024380791233308074,"left2":13312,"right2":54927144369443219314090,"fanout":227499099192},{"left1":17408,"right1":1705725452158112160836,"left2":25600,"right2":69732737722166776886772,"fanout":1017035885800},{"left1":32,"right1":55969867892762319943940,"left2":2080,"right2":9031837866337200726450,"fanout":931083332852},{"left1":4096,"right1":75078622460159237659940,"left2":4608,"right2":49816449313776043264080,"fanout":1014095163640},{"left1":64,"right1":70587723991789342731624,"left2":33344,"right2":14907748052011818626900,"fanout":825173002460},{"left1":4128,"right1":134955939526247013476004,"left2":4136,"right2":117736822230454891628638,"fanout":37919182892},{"left1":2,"right1":19618168891889106428182,"left2":18,"right2":25983532032484438396296,"fanout":859192752938},{"left1":4104,"right1":72077170518074071418382,"left2":32784,"right2":45965385859252870726892,"fanout":1000462939384},{"left1":17408,"right1":31440674372582557349182,"left2":25600,"right2":24453188621689230302558,"fanout":1052001188912},{"left1":32768,"right1":2533558591877956829140,"left2":2112,"right2":14550920025431013317926,"fanout":979937846312},{"left1":512,"right1":42182536850378843142108,"left2":32784,"right2":34420375316616177992712,"fanout":965965782020},{"left1":32768,"right1":4026484960575969584586,"left2":33088,"right2":55462633452434291784428,"fanout":1031679048908},{"left1":2,"right1":132431577448060481367470,"left2":32770,"right2":56035649755664793547230,"fanout":226708484304},{"left1":2,"right1":27457553289538755057718,"left2":6,"right2":88892882764619562927134,"fanout":9076931369},{"left1":4,"right1":116142831253288217156476,"left2":260,"right2":73788876964257094197416,"fanout":828545101301},{"left1":8,"right1":25310561644474924854008,"left2":16386,"right2":68031470315581438266402,"fanout":37917045768},{"left1":2,"right1":24491478642785595084752,"left2":16,"right2":50689821502098101119536,"fanout":17779932468},{"left1":8194,"right1":61372598797023710519318,"left2":4100,"right2":126927126018491461038358,"fanout":980127370444},{"left1":8,"right1":24323061343066539467470,"left2":17410,"right2":36100625632195874606076,"fanout":223554623724},{"left1":2,"right1":19140837758427087599426,"left2":258,"right2":75228986800248542601802,"fanout":202184525545},{"left1":1026,"right1":88714713804814610708060,"left2":1154,"right2":150552089729199798758534,"fanout":89539535884},{"left1":32768,"right1":69340145650473783635558,"left2":35328,"right2":70827079088129393290160,"fanout":89862232096},{"left1":128,"right1":130410073014943054074040,"left2":17410,"right2":50167582271168706701190,"fanout":1065772134604},{"left1":1024,"right1":57693964021162662703380,"left2":17408,"right2":61267875552481530258022,"fanout":55244015820},{"left1":512,"right1":71199329085036255279110,"left2":2240,"right2":20140895222609589362574,"fanout":859278464252},{"left1":16,"right1":28108063279227999984258,"left2":256,"right2":57998633038039503262608,"fanout":69236285925},{"left1":1026,"right1":73972292434180456858316,"left2":1034,"right2":10470496552758092388610,"fanout":1034551041028},{"left1":64,"right1":42021337405873806275600,"left2":2112,"right2":8290768835217342899358,"fanout":86640747752},{"left1":32,"right1":49238707161318002820500,"left2":544,"right2":55168342968825123517952,"fanout":1034162061512},{"left1":512,"right1":55899061014203980166632,"left2":544,"right2":18856288166137700693132,"fanout":945431571676},{"left1":8192,"right1":63919951925748628176460,"left2":10240,"right2":69256436843048412228558,"fanout":206239497232},{"left1":8192,"right1":53897737356207885172946,"left2":9218,"right2":39186892759907738049442,"fanout":51822773260},{"left1":4,"right1":72455087236268332768710,"left2":16,"right2":39863593573951943254778,"fanout":4247255557},{"left1":32,"right1":45290303417496118023208,"left2":16384,"right2":40273428753296176987862,"fanout":862363639860},{"left1":8200,"right1":23069929125792999646350,"left2":25600,"right2":21017877703406462717306,"fanout":962422957256},{"left1":256,"right1":55154226133509194556490,"left2":16384,"right2":39291450170371497359544,"fanout":103080004828},{"left1":16,"right1":48352048991444615862876,"left2":2048,"right2":27481126386457031312320,"fanout":828472033512},{"left1":2304,"right1":17265041964871132223816,"left2":18688,"right2":14729150558635877338206,"fanout":896912600292},{"left1":1026,"right1":132203573985258272946418,"left2":1034,"right2":111903025277161907024522,"fanout":72209984512},{"left1":8,"right1":129401817310222439435932,"left2":8194,"right2":20398874154614270042530,"fanout":106984649772},{"left1":512,"right1":40685135175454943146724,"left2":32784,"right2":45753651318967719319572,"fanout":828003115208},{"left1":256,"right1":33013900371636088598866,"left2":272,"right2":15715849425108633294432,"fanout":829632551684},{"left1":2,"right1":34162767220479552593192,"left2":258,"right2":742616627228005729912,"fanout":120278021403},{"left1":64,"right1":57277850626096289829092,"left2":512,"right2":44772632273558100095032,"fanout":124219088912},{"left1":32768,"right1":25915174799327174351592,"left2":32780,"right2":58195274069864121338458,"fanout":138190847220},{"left1":2304,"right1":118538714124432397367130,"left2":6400,"right2":87835601453434801995072,"fanout":999734918364},{"left1":512,"right1":7144730400777136161054,"left2":192,"right2":23356655532985373160132,"fanout":189851251756},{"left1":128,"right1":13837089861409436122204,"left2":24578,"right2":13788334570283376626112,"fanout":106972367044},{"left1":36,"right1":132346970841113394929042,"left2":164,"right2":88139862847960060614472,"fanout":37861987572},{"left1":32768,"right1":35126951321164830259752,"left2":4128,"right2":64470217424578042742092,"fanout":18133606636},{"left1":32,"right1":52435985047718530035112,"left2":2048,"right2":56339528501831138834992,"fanout":141480442080},{"left1":16,"right1":62176971471611220407904,"left2":272,"right2":74158230884088775068930,"fanout":829651077853},{"left1":128,"right1":83779300566227104675598,"left2":9218,"right2":31826558590004976113828,"fanout":962545894460},{"left1":8,"right1":130356253973638330891130,"left2":24578,"right2":17588383001878251818262,"fanout":193006714892},{"left1":8194,"right1":69556124439165266854238,"left2":4104,"right2":76085262530168394178454,"fanout":842014970908},{"left1":256,"right1":70349848578475814469190,"left2":2304,"right2":54471137736281876563980,"fanout":69221628},{"left1":1024,"right1":104930023394872233069942,"left2":1036,"right2":140717005141927332640296,"fanout":72692326440},{"left1":2,"right1":6418821518438074377136,"left2":66,"right2":7484453452261431478340,"fanout":189796499504},{"left1":12,"right1":58446686871680261951206,"left2":4104,"right2":8474585699205365115682,"fanout":210118439156},{"left1":9216,"right1":35172897132038299985910,"left2":9224,"right2":52242166477988362954852,"fanout":896843696352},{"left1":4,"right1":127936729162814495951460,"left2":20,"right2":66344639117657061583138,"fanout":26122448932},{"left1":256,"right1":26296590100842638960024,"left2":32768,"right2":30870549993497586307948,"fanout":896658057412},{"left1":8192,"right1":47033554405730150744630,"left2":10240,"right2":24297689369900028396136,"fanout":911003929620},{"left1":128,"right1":128688833493675218782436,"left2":4132,"right2":37525245685275851801972,"fanout":1034967849160},{"left1":128,"right1":118856435660545760255518,"left2":17410,"right2":30736055376199641806176,"fanout":948986973408},{"left1":32768,"right1":24445428544540435324100,"left2":35328,"right2":33338662360990758824282,"fanout":931217281264},{"left1":8,"right1":121140964914747876944518,"left2":2568,"right2":55954251756998266630010,"fanout":189796450352},{"left1":512,"right1":71373448506821422351648,"left2":576,"right2":39065468961941743211476,"fanout":824636092612},{"left1":2048,"right1":43754666021014745475588,"left2":16384,"right2":72458489412742524424598,"fanout":189919459368},{"left1":32768,"right1":37511355590905713146516,"left2":32816,"right2":5978747622993861669768,"fanout":1017786200264},{"left1":1024,"right1":53141995302902648371360,"left2":1032,"right2":4620567872272794850526,"fanout":1000138658004},{"left1":32792,"right1":41882736317382360922960,"left2":200,"right2":9360871912723367663658,"fanout":825375329508},{"left1":32768,"right1":59860233058194108571972,"left2":32772,"right2":3394872616303929520778,"fanout":120943329472},{"left1":4,"right1":137608503305214287571024,"left2":20,"right2":27572556329374857250800,"fanout":18038840014},{"left1":4,"right1":34538403252374942361970,"left2":32768,"right2":14695904965924449741126,"fanout":243941256436},{"left1":2,"right1":73006006175838578019048,"left2":16,"right2":20687515202164757324766,"fanout":832806116559},{"left1":8192,"right1":32663815366071247029542,"left2":4100,"right2":1103521271609883564712,"fanout":68802327592},{"left1":8194,"right1":35582364690267863047254,"left2":8226,"right2":16825428394536717923268,"fanout":945781786644},{"left1":320,"right1":42230836137060181442924,"left2":8512,"right2":52122233763130842792478,"fanout":34630852648},{"left1":16384,"right1":25072539750472485211206,"left2":4136,"right2":54818519841722471629758,"fanout":172082069696},{"left1":16,"right1":22216503122367422778132,"left2":80,"right2":22310522611073780041100,"fanout":979589714952},{"left1":4,"right1":72355605689890081686906,"left2":16,"right2":2041621657406506184026,"fanout":7612844059},{"left1":8192,"right1":112057436522906750059104,"left2":8196,"right2":143383511807858624754998,"fanout":1000004133084},{"left1":4100,"right1":30728764494921957452252,"left2":4132,"right2":10218292260415495108414,"fanout":877182332952},{"left1":8192,"right1":18002608498987470409658,"left2":8448,"right2":5728738660234422028992,"fanout":945096290556},{"left1":4,"right1":110499478869386661016820,"left2":256,"right2":52795944176499718592090,"fanout":69140543436},{"left1":320,"right1":610207190526339334236,"left2":4416,"right2":44294309319813194567432,"fanout":37929934868},{"left1":8,"right1":6859293093764896383798,"left2":17408,"right2":38021032164999712893084,"fanout":876378126388},{"left1":32,"right1":70253130330062811868352,"left2":96,"right2":35963935648003769305270,"fanout":38455480548},{"left1":2,"right1":97484941269494096854832,"left2":4104,"right2":142287282750072996353154,"fanout":1086358291472},{"left1":512,"right1":52510456979717088720708,"left2":2368,"right2":46905096803418034343064,"fanout":1048644945944},{"left1":8,"right1":100369001170167795511732,"left2":9218,"right2":19312664837180679711258,"fanout":158460018900},{"left1":256,"right1":51968469461796067631868,"left2":8448,"right2":26408062159881991477090,"fanout":206509445372},{"left1":4096,"right1":123581859490885204752088,"left2":8194,"right2":13257585335756578809734,"fanout":141548634120},{"left1":8,"right1":118006271646001085176552,"left2":2368,"right2":17308950793759740271414,"fanout":1013814670548},{"left1":8320,"right1":45659006136863924278626,"left2":10368,"right2":34808790764616011568110,"fanout":1065301587188},{"left1":16,"right1":53430544790872744272620,"left2":272,"right2":25987364631885151798882,"fanout":103953791019},{"left1":2,"right1":5254103554614324874990,"left2":16,"right2":10206354862513313743032,"fanout":210422198045},{"left1":64,"right1":15336280301256663623988,"left2":576,"right2":71519398889757570701316,"fanout":192874034216},{"left1":8320,"right1":71671892664888776173284,"left2":8352,"right2":57352718188776697478056,"fanout":910737594396},{"left1":16392,"right1":136937517041568515654344,"left2":20488,"right2":142700826556525340653656,"fanout":141465485336},{"left1":128,"right1":30216904140150278890968,"left2":2560,"right2":13345382465076177788360,"fanout":34360270888},{"left1":32768,"right1":54076746526663123446912,"left2":448,"right2":37167953919540260821230,"fanout":914360896716},{"left1":8,"right1":83682111081838667225732,"left2":4136,"right2":65868094633169101740070,"fanout":962678020308},{"left1":64,"right1":17532392580623646736338,"left2":4100,"right2":75000709152567767492552,"fanout":86518057988},{"left1":16,"right1":4389467668933771094184,"left2":256,"right2":45919535924800253792518,"fanout":146836806383},{"left1":64,"right1":44244966886276974311650,"left2":48,"right2":56978159365297374157864,"fanout":910815922176},{"left1":128,"right1":111423075196892717380994,"left2":2560,"right2":26930200491696581064772,"fanout":1051260934368},{"left1":4,"right1":5418375089007916188764,"left2":20,"right2":35967554665200829185536,"fanout":86623396629},{"left1":384,"right1":25825194103608397041132,"left2":328,"right2":55418235203689564126124,"fanout":85902497984},{"left1":4,"right1":62334341168369626441670,"left2":16384,"right2":46860103172186827181386,"fanout":124354566204},{"left1":64,"right1":20387842031971195381518,"left2":1024,"right2":5675833290206970189974,"fanout":171801056316},{"left1":4,"right1":88298356627792418718608,"left2":16,"right2":23399620731138305656334,"fanout":142691068873},{"left1":384,"right1":57657950147634995913332,"left2":448,"right2":17362047793489885181944,"fanout":1086495658032},{"left1":4096,"right1":8198673380754704627808,"left2":36880,"right2":22912036502861769170556,"fanout":965309421816},{"left1":16,"right1":74418655359486098593240,"left2":272,"right2":49420118178052630642046,"fanout":841844145414},{"left1":2,"right1":63163941560994950353592,"left2":16384,"right2":55471498845044867722380,"fanout":897596535812},{"left1":2048,"right1":48492549252984371458140,"left2":18432,"right2":25196210624502722850052,"fanout":17384342768},{"left1":4104,"right1":70237983549274721646788,"left2":192,"right2":30886668401809317151562,"fanout":1083085695176},{"left1":4096,"right1":22089891793738649874878,"left2":320,"right2":11484467560173993286044,"fanout":845976180736},{"left1":512,"right1":10003431673247578492710,"left2":560,"right2":12417051138920537918840,"fanout":20872216824},{"left1":32768,"right1":50723179025017536801010,"left2":32780,"right2":34955110211016192859520,"fanout":136372440},{"left1":256,"right1":52953236094707059889796,"left2":33024,"right2":40244015932064446998140,"fanout":192805605608},{"left1":33280,"right1":808195357074674004890,"left2":33312,"right2":16407530148133780850212,"fanout":189798290460},{"left1":32,"right1":25268165794586202054350,"left2":4104,"right2":42809482857483143809718,"fanout":965834441952},{"left1":128,"right1":16172547878707297287322,"left2":16386,"right2":42732529475687330130060,"fanout":175960282336},{"left1":16,"right1":130909679786415967012464,"left2":18,"right2":92966781870365198551376,"fanout":8994620431},{"left1":8192,"right1":67628899955991228435352,"left2":8228,"right2":70139167998678988863484,"fanout":244020727004},{"left1":48,"right1":89511612279357675607870,"left2":56,"right2":90800506130201280238358,"fanout":38402265140},{"left1":16384,"right1":29694866329003635851594,"left2":17408,"right2":44803206229091125873698,"fanout":842430989560},{"left1":8,"right1":90671084194254277619390,"left2":48,"right2":29215171989686902469692,"fanout":52426234048},{"left1":384,"right1":48013846832936988978900,"left2":4480,"right2":55443131135390892764572,"fanout":999789480996},{"left1":2048,"right1":23348000777416861095222,"left2":2080,"right2":45271706996733352686098,"fanout":69336096812},{"left1":8194,"right1":127519953290154509982454,"left2":8320,"right2":93348287776516498458054,"fanout":241460890848},{"left1":2,"right1":8431093251384049515200,"left2":8194,"right2":62362035443394717954346,"fanout":965910988860},{"left1":2560,"right1":84318912630500153943084,"left2":2568,"right2":89335384944927872413182,"fanout":914440110088},{"left1":16384,"right1":9805999363558255997434,"left2":4132,"right2":41962865193719270931792,"fanout":928588021980},{"left1":4100,"right1":69420174332932839536748,"left2":12292,"right2":22476596461605843271182,"fanout":1031598508224},{"left1":16384,"right1":16139101046493659904358,"left2":17408,"right2":46596926482809862125940,"fanout":103550295068},{"left1":128,"right1":119858134475366824097680,"left2":1024,"right2":18355440858986693808088,"fanout":979735999708},{"left1":33280,"right1":12731579735024343638620,"left2":37376,"right2":75355466715305960664804,"fanout":1065289265348}],"pure":{"1":270548207418,"4":89369028574,"8":975531532833,"16":1090302058213,"48":957122162631,"96":893402097430,"144":1021766205144,"192":896592380944,"224":106988763412,"272":978818100197,"368":172248341710,"416":236321228749,"432":110709427671,"464":1087445930269,"544":1023255580725,"576":824720947412,"608":257534273012,"624":55395892754,"656":902131426059,"688":37584961017,"704":982893526070,"720":986781639402,"736":1052704575038,"768":123646829766,"800":81503531528,"816":214376967135,"864":966650424352,"912":923700681425,"960":954022883582,"992":65094743526,"1008":1052718520335,"1104":81365972780,"1152":948569119763,"1168":1096222849068,"1216":90030006517,"0":4127653122,"2":952661455595,"32":257173935833,"64":1018576107832,"80":214983775177,"112":1064968770565,"160":927174296068,"208":897729684751,"288":1044200950320,"304":889944471097,"320":971534891766,"336":993112876856,"480":172755074853,"496":253886596896,"528":914422031552,"592":69740081345,"640":957942924535,"752":1087179595231,"848":124640820986,"896":223557708845,"928":274373022255,"944":1060571961373,"1024":1021329081859,"1040":145424592138,"1056":95529735156,"1088":1026310280207,"1120":181275448119,"1136":116430136125,"1200":1026211564230,"3":919706612491,"9":939973749510,"10":77933901821,"11":1014635485376,"12":224365392092,"176":190022219493,"240":116667321654,"352":106719530477,"400":68739723209,"512":145088960754,"672":80533462221,"832":846981223435,"1072":915163191284,"1184":218845340654,"5":211111255280,"14":155573565480,"15":206775779384,"448":829715582984,"784":1064627877589,"880":967360377832,"13":819470560,"7":931957323004,"384":261757795787,"560":1051799769316,"976":1072680999366,"128":961489341947,"256":196916686106,"6":892399729613},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"three-phase","capacity":8,"product_order":[0,1,2,3,4,6,7,10,11,12,14,15,17,18,19,20,21,22,25,26,27,28,29,31,33,36,37,38,39,40,44,45,47,52,53,54,55,56,57,58,59,61,62,64,65,70,71,72,73,74,77,78,80,81,86,87,90,91,92,93,95,96,97,98,99,102,103,109,112,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,130,131,132,135,136,138,139,142,143,144,145,147,148,149,150,151,152,154,155,157,158,163,164,165,166,167,168,169,170,172,173,175,176,177,178,13,24,41,66,69,75,76,84,89,94,101,107,133,134,156,171,5,8,9,16,23,30,32,34,35,42,43,46,48,49,50,51,60,63,67,68,79,82,83,85,88,100,104,105,106,108,110,111,113,129,137,140,141,146,153,159,160,161,162,174,179],"events":[{"kind":"products","cache":943,"indices":[0,1,2,3,4,6,7,10,11,12,14,15,17,18,19,20,21,22,25,26,27,28,29,31,33,36,37,38,39,40,44,45,47,52,53,54,55,56,57,58,59,61,62,64,65,70,71,72,73,74,77,78,80,81,86,87,90,91,92,93,95,96,97,98,99,102,103,109,112,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,130,131,132,135,136,138,139,142,143,144,145,147,148,149,150,151,152,154,155,157,158,163,164,165,166,167,168,169,170,172,173,175,176,177,178]},{"kind":"pure_low","cache":943,"monomial":3},{"kind":"pure_low","cache":943,"monomial":10},{"kind":"pure_low","cache":943,"monomial":12},{"kind":"pure_low","cache":943,"monomial":5},{"kind":"pure_low","cache":943,"monomial":14},{"kind":"pure_low","cache":943,"monomial":13},{"kind":"pure_low","cache":943,"monomial":7},{"kind":"pure_low","cache":943,"monomial":6},{"kind":"products","cache":975,"indices":[13,24,41,66,69,75,76,84,89,94,101,107,133,134,156,171]},{"kind":"pure_low","cache":975,"monomial":11},{"kind":"products","cache":1247,"indices":[5,8,9,16,23,30,32,34,35,42,43,46,48,49,50,51,60,63,67,68,79,82,83,85,88,100,104,105,106,108,110,111,113,129,137,140,141,146,153,159,160,161,162,174,179]},{"kind":"pure_low","cache":1247,"monomial":9},{"kind":"pure_low","cache":1247,"monomial":15}],"low_computations":11}},{"batch":14,"delta_sha256":"98635d1b554ea3a0f70722b1ecf49c2049d192a78c75c3a04a2cd5bfcce66331","plan":{"addresses":1220,"batch":5,"brot":8,"low_bits":4,"products":[{"left1":256,"right1":54774161367308374800368,"left2":33152,"right2":1339311099357661384438,"fanout":1085871236858},{"left1":128,"right1":20843007912115395456364,"left2":16384,"right2":72045384674960905950736,"fanout":962927446586},{"left1":68,"right1":5403465757823035751230,"left2":196,"right2":6508388495276486408624,"fanout":1082680856068},{"left1":16,"right1":49488037019949146802552,"left2":1056,"right2":24160470580103742308612,"fanout":224212948018},{"left1":1056,"right1":60027860953412010283240,"left2":1312,"right2":54481652837409039018688,"fanout":1048813563098},{"left1":16,"right1":47842190679996304105184,"left2":8210,"right2":34918527108691937336868,"fanout":957075479758},{"left1":128,"right1":35107764199404952234254,"left2":2056,"right2":4313984930834889408742,"fanout":80781570806},{"left1":68,"right1":30917734633696432058334,"left2":2116,"right2":10917197607096408508096,"fanout":1065300064780},{"left1":2048,"right1":25041475605209799954734,"left2":10240,"right2":21001642904378311444868,"fanout":26286889496},{"left1":256,"right1":49341600517823671889786,"left2":33024,"right2":59228986878776127436818,"fanout":965395663572},{"left1":512,"right1":58131799899886933230212,"left2":8194,"right2":59910415515595927845466,"fanout":37963313382},{"left1":4098,"right1":74705529118275012864752,"left2":36864,"right2":36431567220590262185154,"fanout":1082868955898},{"left1":32768,"right1":49861659726193846096716,"left2":33024,"right2":27879285748630923767330,"fanout":209393814732},{"left1":16,"right1":7435964972681317769132,"left2":2052,"right2":111552346426177167986260,"fanout":198543065126},{"left1":4,"right1":130978417668632320137650,"left2":32772,"right2":72060328715870825068076,"fanout":175170788080},{"left1":9216,"right1":68037272277174828069364,"left2":9472,"right2":62608764051189062295026,"fanout":223842189338},{"left1":8194,"right1":29788352052182727700072,"left2":9218,"right2":3753564311518615003364,"fanout":962858462972},{"left1":32,"right1":54531468544652597538160,"left2":608,"right2":54377376556087302301378,"fanout":825241420002},{"left1":1024,"right1":62865424479181775298698,"left2":8194,"right2":10347919381519808817740,"fanout":896957336108},{"left1":8,"right1":25008894658953511722938,"left2":12288,"right2":69815029137145466533260,"fanout":155326748172},{"left1":68,"right1":49651752091126621780838,"left2":36864,"right2":45711165986082305022880,"fanout":149700537048},{"left1":128,"right1":60488245428097327999336,"left2":264,"right2":19441588969889849849968,"fanout":1034059968704},{"left1":16,"right1":52835025706651797501386,"left2":528,"right2":71367514215201063880790,"fanout":879865239262},{"left1":64,"right1":61925468214061945615992,"left2":328,"right2":27215262409263325740064,"fanout":1077164379666},{"left1":16,"right1":58946523885480956335786,"left2":2312,"right2":32952369346027640362858,"fanout":1025964170954},{"left1":4,"right1":107917510454126413019704,"left2":256,"right2":43572055689405664588662,"fanout":34760564023},{"left1":256,"right1":118813833000700406049368,"left2":2312,"right2":98757886385269458961864,"fanout":227502910160},{"left1":8192,"right1":30607530553665090070554,"left2":8196,"right2":48036756729366907168798,"fanout":962781250264},{"left1":16384,"right1":63194744675049950282396,"left2":16448,"right2":48424997208638722676118,"fanout":845922834486},{"left1":8,"right1":67245291085573830235918,"left2":32,"right2":9591474510011299210374,"fanout":98763727398},{"left1":576,"right1":65359626042760243134156,"left2":578,"right2":27985839374954680286892,"fanout":115698044420},{"left1":68,"right1":339982851280800240676,"left2":196,"right2":43857897320783924200170,"fanout":851142565098},{"left1":32768,"right1":30693678890555966616428,"left2":32896,"right2":70348819314035560463714,"fanout":897494996198},{"left1":2056,"right1":19972634971667428037416,"left2":34824,"right2":75016933673095485819242,"fanout":919828432952},{"left1":8192,"right1":33500217860275732359022,"left2":24592,"right2":4751511858991535529176,"fanout":971548576782},{"left1":264,"right1":52432092956737869154618,"left2":268,"right2":22540841039309909140424,"fanout":879444493550},{"left1":1024,"right1":53681312192376778479452,"left2":16400,"right2":31753221215961379190330,"fanout":206897552112},{"left1":128,"right1":1015274482184920671186,"left2":256,"right2":20630050656192353474552,"fanout":128999017984},{"left1":128,"right1":17368629185571615510626,"left2":4224,"right2":42106905321408600234870,"fanout":850876745242},{"left1":8224,"right1":66317573030772127742524,"left2":8288,"right2":67832300385292261249646,"fanout":149903964714},{"left1":64,"right1":12340193724965417193656,"left2":1024,"right2":9295307885103101244314,"fanout":34561209868},{"left1":2,"right1":127655684113659254827572,"left2":256,"right2":30793396700705419570382,"fanout":825607272192},{"left1":128,"right1":8099616592668346576062,"left2":16400,"right2":75340629533042452484028,"fanout":120663106050},{"left1":16400,"right1":57166044503036318410708,"left2":16432,"right2":30369387435282565608846,"fanout":996736561698},{"left1":512,"right1":5868570784409069351574,"left2":2052,"right2":23799018871824553876662,"fanout":1065959175228},{"left1":576,"right1":51886831579243701495556,"left2":32896,"right2":70305914400045407888652,"fanout":845150161942},{"left1":16,"right1":24221506458993115076000,"left2":16384,"right2":5570854045584387620446,"fanout":1040034566666},{"left1":264,"right1":50660026566398160224076,"left2":1288,"right2":68543054430218903131646,"fanout":988594191048},{"left1":4096,"right1":20624431145172670744854,"left2":12320,"right2":10819979776819672639598,"fanout":163444751928},{"left1":1024,"right1":8639163937766173956952,"left2":1056,"right2":39268550239177394484324,"fanout":842564558904},{"left1":16384,"right1":10744408699214383418942,"left2":16416,"right2":71399158216095927452204,"fanout":827938575928},{"left1":512,"right1":42512991897005646919364,"left2":16912,"right2":28942552528411850533612,"fanout":261758844670},{"left1":64,"right1":11083260430393564269134,"left2":272,"right2":72466836719255721685436,"fanout":201005386288},{"left1":272,"right1":40870017561009001436216,"left2":274,"right2":52646590215136346691362,"fanout":181281415449},{"left1":16400,"right1":51750259588951723991432,"left2":16912,"right2":59424725420120623538028,"fanout":954869450},{"left1":256,"right1":2164661254351632193520,"left2":896,"right2":46203619798828866400856,"fanout":240521108722},{"left1":512,"right1":37362296221511443406120,"left2":8736,"right2":28208917383284602011764,"fanout":147035795652},{"left1":2,"right1":94911458228701186377000,"left2":1024,"right2":74992162564525634287314,"fanout":8671987408},{"left1":8192,"right1":9966224197570911097152,"left2":68,"right2":49203956863826731546540,"fanout":1085789105676},{"left1":16,"right1":6488095949342100376650,"left2":17408,"right2":59834958933991059475968,"fanout":114924466182},{"left1":576,"right1":48470795657438302310940,"left2":580,"right2":50761694609420086792462,"fanout":150169202218},{"left1":16384,"right1":15246383165552128272484,"left2":20482,"right2":42657646261485031622982,"fanout":828273387760},{"left1":32768,"right1":52397612157789615864436,"left2":2056,"right2":84244212550262313009012,"fanout":988560542956},{"left1":64,"right1":37724622794467137240340,"left2":16448,"right2":14492793703925601676884,"fanout":880066575876},{"left1":2048,"right1":4248206111487035747960,"left2":272,"right2":58977029350253642558222,"fanout":269793509056},{"left1":32,"right1":19283610540910360578358,"left2":4096,"right2":24729510126651713199816,"fanout":181106171908},{"left1":2048,"right1":23736854517546458374754,"left2":10240,"right2":42559267808433815015882,"fanout":845773937888},{"left1":8192,"right1":45831716439378317532524,"left2":12288,"right2":23090940140631760703182,"fanout":146130528820},{"left1":64,"right1":23843591727458316531004,"left2":12290,"right2":73086826786450838745658,"fanout":957442214426},{"left1":8194,"right1":14972316560499288693386,"left2":9218,"right2":71041555400807692464592,"fanout":868203293224},{"left1":1024,"right1":20223604248882399428208,"left2":576,"right2":43988734670899986445058,"fanout":89188210748},{"left1":17408,"right1":79140221778713389465810,"left2":17412,"right2":150538795648506236887580,"fanout":175372034252},{"left1":256,"right1":23274323408290097639850,"left2":37120,"right2":73032782222466913303910,"fanout":77378885150},{"left1":2,"right1":46455776278735557700858,"left2":16,"right2":19456175241192881388384,"fanout":35269648901},{"left1":2048,"right1":16374722374497744263654,"left2":2688,"right2":59530843178836427579912,"fanout":1013695126078},{"left1":17408,"right1":53618810518618360143304,"left2":17440,"right2":69491620576707321196492,"fanout":86417403130},{"left1":16384,"right1":62166673330938611514804,"left2":49280,"right2":11016285170347971016382,"fanout":1065589731064},{"left1":8,"right1":23481739650483495969400,"left2":576,"right2":51178082921310402301508,"fanout":189397077190},{"left1":4096,"right1":73537905346949192985722,"left2":4608,"right2":20481605803791159110370,"fanout":922582005460},{"left1":4096,"right1":48941199849217223107320,"left2":2056,"right2":53936014075031712152036,"fanout":141031817920},{"left1":16,"right1":13060110745601274763898,"left2":256,"right2":70780289201485015690468,"fanout":153548554755},{"left1":4,"right1":11080496820490544283058,"left2":272,"right2":71862157117758693631948,"fanout":876085844682},{"left1":512,"right1":19677733269695885430244,"left2":33280,"right2":40191184494986160076696,"fanout":17259622444},{"left1":512,"right1":7235055447704270788068,"left2":2564,"right2":35127033315556083139874,"fanout":261221699288},{"left1":2,"right1":111823296528418680590198,"left2":274,"right2":43778004249507327661978,"fanout":979959273239},{"left1":512,"right1":4913089192609021840088,"left2":580,"right2":44074777174029779030990,"fanout":1048423968990},{"left1":32768,"right1":8819855294757285710018,"left2":640,"right2":34520341939021909399372,"fanout":1000092996316},{"left1":16384,"right1":43533801288155753816118,"left2":17440,"right2":70076491111258454891752,"fanout":825150936090},{"left1":32768,"right1":15194443468343951607512,"left2":49168,"right2":70412672705966799194714,"fanout":850517349944},{"left1":32768,"right1":39627158253166670233316,"left2":704,"right2":70181130245795774636010,"fanout":1052248171012},{"left1":64,"right1":33648017327166803357116,"left2":80,"right2":53984338670744956069362,"fanout":38051518154},{"left1":264,"right1":30689701163790546655954,"left2":33032,"right2":8575834431182145754694,"fanout":95410902038},{"left1":2,"right1":41088108693821783849662,"left2":4098,"right2":1114331539525809131106,"fanout":244291138598},{"left1":8,"right1":118469858706854482300518,"left2":8192,"right2":63518890685551091818390,"fanout":72210066658},{"left1":264,"right1":59786910874802424743024,"left2":266,"right2":57302546132505505303120,"fanout":43033305122},{"left1":4096,"right1":23317018923655796922492,"left2":4104,"right2":32886472410537778649790,"fanout":962074559550},{"left1":36864,"right1":12170786332507189365480,"left2":38912,"right2":2119561010356014462306,"fanout":94638059056},{"left1":576,"right1":35150925730050866576228,"left2":8768,"right2":70342043468193780800140,"fanout":885772786402},{"left1":2,"right1":57863959412617121165720,"left2":36866,"right2":55772735025720980475892,"fanout":1005358552796},{"left1":8,"right1":15062477848721775393444,"left2":32778,"right2":5007375087614230070770,"fanout":175301202448},{"left1":2,"right1":128911116179830740953058,"left2":6,"right2":49942452528189734517404,"fanout":21961188046},{"left1":8,"right1":20994829705012780287408,"left2":4104,"right2":31477921296521966451944,"fanout":149601579556},{"left1":2,"right1":17762882626242855281650,"left2":1056,"right2":6573957646122427460316,"fanout":1051845121070},{"left1":16,"right1":38697213600450710429632,"left2":272,"right2":23917300129420484946570,"fanout":180540207810},{"left1":256,"right1":61156723456541737286002,"left2":260,"right2":2949540897863664251826,"fanout":824848662768},{"left1":17408,"right1":77756255228545818517300,"left2":19456,"right2":113759816417397726109818,"fanout":962690553554},{"left1":4,"right1":22027206144022283309090,"left2":576,"right2":31889822833866113862676,"fanout":9042667230},{"left1":4096,"right1":75384784344317885707300,"left2":4160,"right2":30812154157562384814902,"fanout":172114493652},{"left1":8,"right1":20939456492990802298122,"left2":64,"right2":23631517782012595542680,"fanout":129085653188},{"left1":16,"right1":18609680194503096661946,"left2":36880,"right2":6476097486812458985068,"fanout":1051308387044},{"left1":16416,"right1":44657378590852991707236,"left2":16672,"right2":1083497400225611082226,"fanout":261673189910},{"left1":32,"right1":20260554127865625541904,"left2":16384,"right2":71678635695171706086050,"fanout":46174881016},{"left1":8,"right1":118225273857993935351670,"left2":1024,"right2":2958613364077828283666,"fanout":89158190836},{"left1":2,"right1":118516410090370705587360,"left2":2058,"right2":8366499819470298879652,"fanout":9343392828},{"left1":8224,"right1":34331050499374758517902,"left2":10272,"right2":49804432786138737501446,"fanout":241189449254},{"left1":2048,"right1":9464196255593775547344,"left2":32896,"right2":32438614016260333906984,"fanout":137473740000},{"left1":512,"right1":11314772821829795512572,"left2":1536,"right2":61279892859419774757746,"fanout":853829104128},{"left1":4096,"right1":45401320923671382631462,"left2":8224,"right2":68683946841117927484848,"fanout":862755493068},{"left1":16,"right1":44184710481091972671150,"left2":32,"right2":42931818548220835736690,"fanout":870873093664},{"left1":2048,"right1":17657525576581587384642,"left2":8194,"right2":63619492629905496593382,"fanout":20436628722},{"left1":2048,"right1":49899942125351412779030,"left2":6146,"right2":20357443658708426687516,"fanout":30032983044},{"left1":256,"right1":62292943806929813051896,"left2":1056,"right2":53987775101347175890672,"fanout":1069224870598},{"left1":16384,"right1":60959873495652936310940,"left2":32768,"right2":62195377641303420829368,"fanout":115876569280},{"left1":16,"right1":59678125658596288360506,"left2":16384,"right2":713979378821088038446,"fanout":260988600868},{"left1":256,"right1":46296377784215892078994,"left2":2056,"right2":16411755873875540124604,"fanout":1005694285506},{"left1":1024,"right1":2281085696838431148658,"left2":8192,"right2":45295517099336094193368,"fanout":146666747450},{"left1":2,"right1":115471434160340882891684,"left2":640,"right2":31287812112915790461992,"fanout":1017351648828},{"left1":64,"right1":11091225261487988530,"left2":4160,"right2":45970374513609453547012,"fanout":824915969782},{"left1":8,"right1":147245763420209363172602,"left2":32768,"right2":10359295310917150326340,"fanout":72646796844},{"left1":2048,"right1":57907757915256221108496,"left2":18448,"right2":10546018210319175763006,"fanout":1074514888910},{"left1":2,"right1":129243642164567820147922,"left2":274,"right2":26554259776271500967648,"fanout":8990489917},{"left1":2,"right1":55127028062193968593830,"left2":8194,"right2":6390793301518852579394,"fanout":244425090078},{"left1":68,"right1":6236236557652237198654,"left2":2116,"right2":51177336819578299878806,"fanout":948986507484},{"left1":32,"right1":44483475639016846666638,"left2":1056,"right2":38828661205558272042546,"fanout":63800214238},{"left1":32,"right1":47882515032429710553900,"left2":16384,"right2":74958091020785987189448,"fanout":149454903838},{"left1":12,"right1":135584521245802387231610,"left2":28,"right2":69232435397281501564436,"fanout":949123210962},{"left1":2056,"right1":61778849886838145412246,"left2":2088,"right2":35075190695177380040524,"fanout":167272265464},{"left1":17408,"right1":63774879916890164812322,"left2":17440,"right2":18865810145062447471880,"fanout":55130595366},{"left1":2048,"right1":40801302157490349326894,"left2":34944,"right2":29185159061221325678208,"fanout":1069395534030},{"left1":2,"right1":75467665411236740026674,"left2":36866,"right2":52166505382410720202996,"fanout":107222855698},{"left1":512,"right1":39470271354739146041058,"left2":528,"right2":43389225528552226451990,"fanout":902079629028},{"left1":1024,"right1":5612117195357637081848,"left2":9248,"right2":63920740442063653509938,"fanout":979757486820},{"left1":36864,"right1":39184355982144733340480,"left2":37376,"right2":15609110011873461761114,"fanout":833540772366},{"left1":16,"right1":41881751764135568849982,"left2":2320,"right2":65203716840273860336438,"fanout":910579909850},{"left1":272,"right1":44600847880421516676446,"left2":276,"right2":31420455087554957702274,"fanout":244581909749},{"left1":2,"right1":50044082503505419413496,"left2":130,"right2":38898095203824269394402,"fanout":90107212850},{"left1":8194,"right1":51592417973843395279628,"left2":8258,"right2":8645809402856193328614,"fanout":172519467746},{"left1":8192,"right1":145093117144645051563592,"left2":8200,"right2":95786997295176735088726,"fanout":973998048280},{"left1":640,"right1":38638294798509120697214,"left2":4736,"right2":16683672276016174453246,"fanout":68754797088},{"left1":32,"right1":23715623074976053254756,"left2":4096,"right2":10571026624258132734278,"fanout":114937175588},{"left1":128,"right1":118477790705177808842104,"left2":130,"right2":123961927980932896397334,"fanout":12162317878},{"left1":8192,"right1":32641646552688247387408,"left2":8704,"right2":75087617616439388335594,"fanout":43523255014},{"left1":4096,"right1":56232420580864645073000,"left2":4224,"right2":69062423233616486859496,"fanout":34697645264},{"left1":4098,"right1":3053468543831056595946,"left2":4162,"right2":18201765688230856494202,"fanout":957389089796},{"left1":2048,"right1":94126665348232202483952,"left2":36992,"right2":7650865793948948354758,"fanout":996582684188},{"left1":640,"right1":138101598960077211699304,"left2":644,"right2":121468739674273455520724,"fanout":103931329736},{"left1":2048,"right1":81064012093999128909144,"left2":10240,"right2":45580112401435852249026,"fanout":146568782046},{"left1":64,"right1":24929973830461168825060,"left2":1056,"right2":63104654450271327333898,"fanout":90073865768},{"left1":4,"right1":660718285086216867018,"left2":272,"right2":32067433901142753998856,"fanout":168394614297},{"left1":272,"right1":59505051576359752396290,"left2":276,"right2":73472828365040199738402,"fanout":894211405530},{"left1":1024,"right1":33973592006209531969748,"left2":1032,"right2":33075881557904734528480,"fanout":870941859876},{"left1":2,"right1":129406782711998019059992,"left2":1056,"right2":49596686773762474080088,"fanout":227332465368},{"left1":32896,"right1":69238296089011360886964,"left2":33152,"right2":19109656428152042649946,"fanout":1034484328986},{"left1":256,"right1":55919020134285234803100,"left2":2052,"right2":63554468183256254133390,"fanout":29105112308},{"left1":16384,"right1":1503840525319565813152,"left2":16400,"right2":42816518984568126877782,"fanout":3671733270},{"left1":32768,"right1":92776361464941636372,"left2":36866,"right2":54130326304937535476098,"fanout":842487696612},{"left1":16384,"right1":19174632919246772640782,"left2":16896,"right2":22269923266499370361872,"fanout":833525703918},{"left1":512,"right1":4845766718118530693590,"left2":1024,"right2":62809894992008214058442,"fanout":94993176614},{"left1":128,"right1":57692030779089834083266,"left2":704,"right2":4141310765792873627704,"fanout":944907889396},{"left1":128,"right1":144465253949401561543148,"left2":2176,"right2":89138759407289370284204,"fanout":894160266810},{"left1":2,"right1":8696487847615407570300,"left2":2054,"right2":62664834740494895855862,"fanout":140708683274},{"left1":16,"right1":55600004532385328268886,"left2":24,"right2":32858866657049260434356,"fanout":902077672978},{"left1":128,"right1":38797716926197898000430,"left2":32896,"right2":23575784605626924335608,"fanout":38353779950},{"left1":128,"right1":34775780168745711935660,"left2":640,"right2":12588154551551225189662,"fanout":836998139962},{"left1":8192,"right1":73797687159214798503100,"left2":8200,"right2":10104472652079671749438,"fanout":38569775144},{"left1":8224,"right1":50983419673452118023294,"left2":9248,"right2":9801143237909515134024,"fanout":29159377950},{"left1":32,"right1":24961843177236823769218,"left2":4098,"right2":52096871801794397609348,"fanout":106872579268},{"left1":4,"right1":70627771948963092658724,"left2":2112,"right2":44863574406875539005210,"fanout":966147099864},{"left1":2052,"right1":58133753515133462506702,"left2":34820,"right2":50215029119044687065210,"fanout":1040392270890},{"left1":8,"right1":71955019936558732242288,"left2":8192,"right2":61746171644610939332822,"fanout":167486880774},{"left1":32768,"right1":74284282023508885567228,"left2":33408,"right2":44474825429851841335322,"fanout":86606753500},{"left1":16384,"right1":49744195364501441161322,"left2":16386,"right2":33434102783061859508664,"fanout":980208108250},{"left1":4,"right1":65175755210642375463104,"left2":1056,"right2":2319738762090418392830,"fanout":910570479860},{"left1":4,"right1":74314238865097864479616,"left2":128,"right2":42675926458623870526106,"fanout":63923483868},{"left1":8,"right1":115188001393830659288906,"left2":40,"right2":21126628247431309544966,"fanout":965778348232},{"left1":4,"right1":62013283913792632316766,"left2":16,"right2":48955208146005118413922,"fanout":34443353114},{"left1":256,"right1":4319275845689472214840,"left2":258,"right2":74670209984929267157806,"fanout":171933115105},{"left1":32,"right1":50956431697202952229514,"left2":272,"right2":63855623409750039515046,"fanout":962073379894},{"left1":68,"right1":8531860915003401178416,"left2":32836,"right2":26444462792453135405272,"fanout":192550202598},{"left1":8192,"right1":13697405591910489813266,"left2":25600,"right2":9908794348617687263796,"fanout":232400147456},{"left1":2048,"right1":146143791296154714671184,"left2":6144,"right2":17613252948772752694680,"fanout":29342613552},{"left1":8192,"right1":72154049938649033888278,"left2":8224,"right2":11043869437578696919010,"fanout":155171084526},{"left1":64,"right1":7000403259202827279094,"left2":4162,"right2":21488913856384131965026,"fanout":192939358226},{"left1":4,"right1":117270428304367433675222,"left2":8,"right2":150900486711303643532170,"fanout":249190297300},{"left1":4,"right1":72897327415894268102276,"left2":256,"right2":21693654651437740937404,"fanout":140690779684},{"left1":17408,"right1":74446505376653991651434,"left2":17472,"right2":34871854591087149575058,"fanout":158313883662},{"left1":8194,"right1":3003930482303590814324,"left2":12290,"right2":16809543423252151821046,"fanout":936886018060},{"left1":16384,"right1":24685283677933166651892,"left2":16512,"right2":16949897694000152465808,"fanout":189082041906},{"left1":64,"right1":6722964456291802519012,"left2":16400,"right2":5510003867265096599866,"fanout":258036864028},{"left1":4,"right1":55516738252310365416970,"left2":16,"right2":15302714875187896123874,"fanout":175826389291},{"left1":4,"right1":73818522373898178088800,"left2":32896,"right2":41692983495792270356398,"fanout":184586340890},{"left1":2052,"right1":9557722804561928308206,"left2":3076,"right2":44681322738130930123014,"fanout":150135556320},{"left1":16384,"right1":1812520102998802390438,"left2":264,"right2":5461127022880874971516,"fanout":103195016406}],"pure":{"0":1078036466175,"2":970962421041,"4":1044095176473,"16":1044634270248,"32":134064965397,"64":1052552927249,"80":922815359764,"112":949536614929,"160":864013719103,"176":215202531605,"208":1077985465070,"224":214395123223,"256":16209493491,"272":124222651928,"288":979367575029,"352":85630375651,"400":132337766345,"432":24913327365,"448":988593910304,"464":1052472169229,"480":56006329117,"496":828609839876,"512":108147321887,"544":244542917117,"592":897785118975,"608":183897866530,"640":13737338107,"656":996548411623,"688":932932142306,"736":271587681598,"752":1053254861850,"768":214846808793,"784":1061177670383,"800":33804509174,"896":150460496873,"912":198370147333,"928":9141492281,"960":145612409647,"1008":855316762623,"1040":253456103398,"1072":911422201358,"1088":1065669957662,"1120":1053108348208,"1168":849566836933,"5":11858091780,"9":46409719330,"10":979302598089,"13":1043645911754,"15":1005909370882,"96":1035221338383,"128":176309136694,"144":841882484456,"192":1073189602040,"240":979317299684,"320":106969966885,"336":838174038489,"368":150859873803,"384":24797368335,"416":1031277897928,"528":193913370423,"576":59454337778,"624":1008745243856,"672":245672897830,"720":112319528502,"816":950041455608,"832":915502863671,"864":205992629192,"944":13186495479,"976":184245810911,"1024":8088908583,"1136":232076684351,"3":59293902344,"7":1052247065130,"11":1060403157738,"12":1035508503569,"48":141914333666,"304":881003728087,"848":949488183848,"880":918957396791,"1104":51405251869,"1184":226975610143,"6":95176025543,"14":1005059842752,"704":218574212311,"992":1031465201969,"1056":20921918163,"1200":1077333387261,"1216":885451257378,"1":46607898680,"8":927413879607,"1152":988429284603,"560":68385043442},"ranks":[],"cache_masks":[3,5,6,7,9,10,11,12,13,14,15,48,80,96,112,144,160,176,192,208,224,240,272,288,304,320,336,352,368,384,400,416,432,448,464,480,496,528,544,560,576,592,608,624,640,656,672,688,704,720,736,752,768,784,800,816,832,848,864,880,896,912,928,944,960,976,992,1008,1040,1056,1072,1088,1104,1120,1136,1152,1168,1184,1200,1216],"blocks":null,"parity_scratch":0},"schedule":{"label":"three-phase","capacity":8,"product_order":[3,4,5,7,8,10,13,15,16,17,18,22,23,24,25,26,27,28,29,30,34,35,36,39,40,41,43,44,46,47,49,50,51,52,53,54,56,57,58,59,60,63,64,66,69,70,71,73,75,77,80,81,83,84,85,87,90,93,94,97,100,102,103,104,105,106,108,110,111,112,113,114,116,118,119,121,123,124,125,129,130,131,132,133,134,135,136,137,140,141,143,144,146,147,151,156,157,158,159,160,161,163,164,166,167,170,171,174,175,177,179,181,182,184,185,186,187,189,191,193,194,195,198,199,201,202,19,48,61,65,67,68,78,79,92,95,101,107,117,120,127,149,153,176,190,192,196,0,1,2,6,9,11,12,14,20,21,31,32,33,37,38,42,45,55,62,72,74,76,82,86,88,89,91,96,98,99,109,115,122,126,128,138,139,142,145,148,150,152,154,155,162,165,168,169,172,173,178,180,183,188,197,200],"events":[{"kind":"products","cache":887,"indices":[3,4,5,7,8,10,13,15,16,17,18,22,23,24,25,26,27,28,29,30,34,35,36,39,40,41,43,44,46,47,49,50,51,52,53,54,56,57,58,59,60,63,64,66,69,70,71,73,75,77,80,81,83,84,85,87,90,93,94,97,100,102,103,104,105,106,108,110,111,112,113,114,116,118,119,121,123,124,125,129,130,131,132,133,134,135,136,137,140,141,143,144,146,147,151,156,157,158,159,160,161,163,164,166,167,170,171,174,175,177,179,181,182,184,185,186,187,189,191,193,194,195,198,199,201,202]},{"kind":"pure_low","cache":887,"monomial":5},{"kind":"pure_low","cache":887,"monomial":9},{"kind":"pure_low","cache":887,"monomial":10},{"kind":"pure_low","cache":887,"monomial":13},{"kind":"pure_low","cache":887,"monomial":3},{"kind":"pure_low","cache":887,"monomial":11},{"kind":"pure_low","cache":887,"monomial":6},{"kind":"pure_low","cache":887,"monomial":14},{"kind":"products","cache":983,"indices":[19,48,61,65,67,68,78,79,92,95,101,107,117,120,127,149,153,176,190,192,196]},{"kind":"pure_low","cache":983,"monomial":12},{"kind":"products","cache":1757,"indices":[0,1,2,6,9,11,12,14,20,21,31,32,33,37,38,42,45,55,62,72,74,76,82,86,88,89,91,96,98,99,109,115,122,126,128,138,139,142,145,148,150,152,154,155,162,165,168,169,172,173,178,180,183,188,197,200]},{"kind":"pure_low","cache":1757,"monomial":15},{"kind":"pure_low","cache":1757,"monomial":7}],"low_computations":11}}]\n', 'tasks/dfthcsos/full_xor/scheduler.py': '# SPDX-License-Identifier: Apache-2.0\n"""Exact parent-live low-monomial cache schedules, no synthesis/native work.\n\nReordering paired XOR products is exact: all controls depend only on address,\nand each target operation is XOR by an address function. Pure-low corrections\nare consolidated in Plan.pure and performed only after every paired product.\nFor each tested order, dynamic programming finds the minimum computation count\nover ALL canonical-parent-closed caches of the requested capacity. Order search\nitself is only a few deterministic grouping heuristics, not global optimality.\n"""\nfrom functools import lru_cache\nfrom dataclasses import dataclass\nimport numpy as np\n\nLOW=[m for m in range(1,16)if m.bit_count()>=2]\nINDEX={m:j for j,m in enumerate(LOW)}\n\ndef parent(m):return m^(1<<(m.bit_length()-1))\n\ndef closure(monomials):\n    result=set()\n    for original in monomials:\n        m=int(original)\n        if not 0<=m<16:raise ValueError(\'not a low4 monomial\')\n        while m.bit_count()>=2:\n            result.add(m);m=parent(m)\n    return sum(1<<INDEX[m]for m in result)\n\ndef members(mask):return [m for j,m in enumerate(LOW)if(mask>>j)&1]\n\ndef closed(mask):return closure(members(mask))==mask\n\ndef product_requirement(product):\n    poly=int(product.left1)|int(product.left2)\n    if poly&1:raise ValueError(\'constant controls not supported by existing emitter\')\n    monomials=[i for i in range(1,16)if(poly>>i)&1]\n    return closure(monomials)\n\n@lru_cache(None)\ndef states_and_cost(capacity):\n    if not 1<=capacity<=11:raise ValueError(\'capacity must be1..11\')\n    states=[m for m in range(1<<11)if m.bit_count()<=capacity and closed(m)]\n    cost=np.array([[(t&~s).bit_count()for t in states]for s in states],dtype=np.int16)\n    return np.array(states,dtype=np.int16),cost\n\ndef requirements(plan):\n    if plan.low_bits!=4 or plan.blocks is not None:raise ValueError(\'need an unblocked low4 plan\')\n    req=[product_requirement(p)for p in plan.products]\n    for m in plan.pure:\n        if not(m<16 or(m&15)==0):raise ValueError(\'Plan.pure contains a mixed monomial\')\n    return req\n\ndef events_for_order(plan,order):\n    events=[]\n    # Consecutive same-required products use one cache state.\n    for index in order:\n        r=product_requirement(plan.products[index])\n        if events and events[-1][\'kind\']==\'products\'and events[-1][\'required\']==r:\n            events[-1][\'indices\'].append(int(index))\n        else:events.append(dict(kind=\'products\',indices=[int(index)],required=r))\n    # All pure-low terms are postponed. Descending order is canonical postorder;\n    # DP can retain ancestors for the later pure corrections without recompute.\n    for m in sorted((m for m,v in plan.pure.items()if m<16 and m.bit_count()>=2 and v),reverse=True):\n        events.append(dict(kind=\'pure_low\',monomial=int(m),required=closure([m])))\n    return events\n\ndef optimize_cache(plan,order,capacity=8,label=\'fixed\'):\n    req=requirements(plan)\n    if any(r.bit_count()>capacity for r in req):raise ValueError(\'one product requires more low products than capacity\')\n    order=list(map(int,order))\n    if sorted(order)!=list(range(len(plan.products))):raise ValueError(\'order is not a product permutation\')\n    events=events_for_order(plan,order);states,cost=states_and_cost(capacity);M=len(states)\n    INF=np.int32(1000000);dp=np.full(M,INF,dtype=np.int32);dp[0]=0;backs=[]\n    for event in events:\n        r=event[\'required\'];dest=np.flatnonzero((states&r)==r)\n        values=dp[:,None]+cost[:,dest];old=np.argmin(values,axis=0)\n        new=np.full(M,INF,dtype=np.int32);new[dest]=values[old,np.arange(len(dest))]\n        back=np.full(M,-1,dtype=np.int16);back[dest]=old;backs.append(back);dp=new\n    end=int(np.argmin(dp));computations=int(dp[end]);chosen=[]\n    for i in range(len(events)-1,-1,-1):\n        chosen.append(int(states[end]));end=int(backs[i][end])\n        if end<0:raise RuntimeError(\'cache DP backpointer is missing\')\n    chosen.reverse()\n    if end!=0:raise RuntimeError(\'cache DP did not return to the empty cache\')\n    out=dict(label=label,capacity=capacity,product_order=order,events=[dict(e,cache=s)for e,s in zip(events,chosen)],\n        low_computations=computations,min_live_required=max([r.bit_count()for r in req]+[0]),\n        peak_live=max([s.bit_count()for s in chosen]+[0]),cache_states_considered=M,\n        optimal_retention_for_this_fixed_order=True,globally_optimal_order_claim=False)\n    audit(plan,out);return out\n\ndef _require(ok, message):\n    """Explicit check. `assert` disappears under `python -O`; a cache audit must not."""\n    if not ok:\n        raise RuntimeError(message)\n\ndef audit(plan,schedule):\n    """Exact lifetime/parent/control audit, independent of native emission."""\n    current=set();computations=0;erases=0;done=[];pure=[];peak=0\n    for e in schedule[\'events\']:\n        target=set(members(e[\'cache\']))\n        _require(closed(e[\'cache\']) and len(target)<=schedule[\'capacity\'],\n                 f"cache {e[\'cache\']} is not closed within capacity {schedule[\'capacity\']}")\n        for m in sorted(current-target,reverse=True):\n            _require(parent(m).bit_count()<2 or parent(m)in current,\n                     f\'erasing {m} while parent {parent(m)} is not live\')\n            _require(not any(parent(child)==m for child in current if child!=m),\n                     f\'erasing {m} while a child is still live\')\n            current.remove(m);erases+=1\n        for m in sorted(target-current):\n            _require(parent(m).bit_count()<2 or parent(m)in current,\n                     f\'computing {m} while parent {parent(m)} is not live\')\n            current.add(m);computations+=1;peak=max(peak,len(current))\n        _require(current==target, f\'cache state {sorted(current)} != target {sorted(target)}\')\n        if e[\'kind\']==\'products\':\n            for i in e[\'indices\']:\n                need=set(members(product_requirement(plan.products[i])))\n                _require(need<=current, f\'product {i} controls {sorted(need-current)} are not cached\')\n                done.append(i)\n        else:\n            _require(e[\'monomial\'] in current, f"pure monomial {e[\'monomial\']} is not cached")\n            pure.append(e[\'monomial\'])\n    for m in sorted(current,reverse=True):\n        _require(parent(m).bit_count()<2 or parent(m)in current,\n                 f\'final erase of {m} while parent {parent(m)} is not live\')\n        current.remove(m);erases+=1\n    _require(not current and sorted(done)==list(range(len(plan.products))),\n             f\'unfinished cache {sorted(current)} or products {sorted(done)}\')\n    expected=sorted(m for m,v in plan.pure.items()if v and m<16 and m.bit_count()>=2)\n    _require(sorted(pure)==expected and computations==schedule[\'low_computations\'] and erases==computations,\n             f\'pure {sorted(pure)} != {expected} or computations {computations}/{erases} \'\n             f"!= {schedule[\'low_computations\']}")\n    _require(peak<=schedule[\'capacity\'], f"peak live {peak} exceeds capacity {schedule[\'capacity\']}")\n    return dict(parent_live_on_every_compute_and_erase=True,all_low_clean_at_end=True,\n                computes=computations,erases=erases,peak_live=peak,pure_low_deferred=True)\n\ndef candidate_orders(plan,capacity=8):\n    req=requirements(plan);groups={}\n    for i,r in enumerate(req):groups.setdefault(r,[]).append(i)\n    orders={\'original\':list(range(len(req))),\n            \'stable_groups\':[i for r in groups for i in groups[r]],\n            \'lex_groups\':[i for r in sorted(groups)for i in groups[r]]}\n    remaining=set(groups);now=0;greedy=[]\n    while remaining:\n        r=min(remaining,key=lambda r:((r&~now).bit_count(),-len(groups[r]),r))\n        greedy.extend(groups[r]);remaining.remove(r);now=r\n    orders[\'nearest_required\']=greedy\n    states,_=states_and_cost(capacity)\n    for ratio in(False,True):\n        remaining=set(groups);now=0;order=[]\n        while remaining:\n            scored=[]\n            for state in states:\n                state=int(state);covered=[r for r in remaining if r&~state==0]\n                count=sum(len(groups[r])for r in covered)\n                if not count:continue\n                new=(state&~now).bit_count()\n                key=((count/(new+1))if ratio else count,-new,-state.bit_count(),-state)\n                scored.append((key,state,covered))\n            _,state,covered=max(scored,key=lambda x:x[0])\n            for r in sorted(covered,key=lambda r:(-len(groups[r]),r)):\n                order.extend(groups[r]);remaining.remove(r)\n            now=state\n        orders[\'coverage_ratio\'if ratio else\'coverage_count\']=order\n    return orders\n\ndef make_schedule(plan,capacity=8):\n    results=[]\n    for label,order in candidate_orders(plan,capacity).items():\n        s=optimize_cache(plan,order,capacity,label);results.append(s)\n    best=min(results,key=lambda s:(s[\'low_computations\'],s[\'peak_live\'],s[\'label\']))\n    return best,[dict(label=s[\'label\'],low_computations=s[\'low_computations\'],peak_live=s[\'peak_live\'])for s in results]\n', 'tasks/dfthcsos/full_xor/validation.py': '# SPDX-License-Identifier: Apache-2.0\n"""Validated independent truth/dirty-family audits, independent of the compiler."""\n\ndef indices(x):\n out=[]\n while x:\n  low=x&-x;out.append(low.bit_length()-1);x^=low\n return out\n\ndef independent_truth(plan):\n """Expand actual product controls with Boolean OR, then zeta-evaluate.\n\n Does not use Plan.evaluate, compiler matrix/rank/pure-product helpers, or\n its ANF routine. Direct equality on the complete valid prefix [0,A).\n """\n A=plan.addresses;coeff=[0]*A\n if plan.blocks is not None:\n  vals=[0]*A\n  for sub in plan.blocks:\n   for a,v in enumerate(independent_truth(sub)):vals[a]^=v\n  return vals\n for mask,fan in plan.pure.items():\n  if mask<A:coeff[mask]^=int(fan)\n for p in plan.products:\n  left=indices(p.left1)+[h<<plan.low_bits for h in indices(p.right2)]\n  right=indices(p.left2)+[h<<plan.low_bits for h in indices(p.right1)]\n  for l in left:\n   for r in right:\n    m=l|r\n    if m<A:coeff[m]^=int(p.fanout)\n for b in range((A-1).bit_length()):\n  for a in range(A):\n   if a&(1<<b):coeff[a]^=coeff[a^(1<<b)]\n return coeff\n\ndef assert_independent(plan,delta):\n """Reject a plan whose truth table is not the packed delta. This is not an assert.\n\n `python -O` removes assert statements. The generator calls this on every\n circuit, so the check has to raise in every interpreter mode and name the\n first address that differs.\n """\n expected=[sum(int(x)<<(j*plan.brot) for j,x in enumerate(row)) for row in delta]\n actual=independent_truth(plan)\n if len(actual)!=len(expected):\n  raise RuntimeError(\n   f\'plan truth table length {len(actual)} differs from delta length {len(expected)}\')\n for address,(got,want) in enumerate(zip(actual,expected)):\n  if got!=want:\n   raise RuntimeError(f\'plan truth table differs from delta at address {address}\')\n', 'tasks/dfthcsos/generate.py': '# SPDX-License-Identifier: Apache-2.0\n# ANGLE circuit builder ported from the competition package\n# verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py\n# (Apache-2.0). Praxen wrapper: --out writes dfthcsos/angle_k_*.kmb and\n# copies the packaged reference factorization.\n#\n# The public 1.44× factor (Signed-Digit Memory v3, NPZ SHA-256\n# c5b4165e7b1dcacf1d287746642414a9ba6086b4279e5ede2bb73e21e6aebd98)\n# is not used. That notebook is Apache-2.0, but it copies the factor\n# from an earlier archive and states that the starting chemistry factors\n# originate with k256.dev, whose generator and licence are not available.\n# R10 forbids shipping that file. This generator rebuilds the package\n# reference (score target 1.0), not the 1.44× lineage.\n\n"""Write the reference dfthcsos submission from the packaged factorization.\n\nThe factorization bytes are the competition package\'s sample ``submission.npz``\n(Apache-2.0), checked by SHA-256 and copied. The ANGLE circuits are rebuilt\nhere from its ``angle_words`` with the packaged ordinary-QROM construction\n(``b_rot`` 12, ``J`` 8). ``.kmb`` files are never committed.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport os\nimport shutil\nfrom collections.abc import Sequence\nfrom math import ceil\nfrom operator import index\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport numpy.typing as npt\n\nDEFAULT_DATA = Path(\n    os.environ.get(\n        "VQCC_DATA",\n        "/home/cld-main/work/vqcc-data/google-verifiable-quantum-circuit-compilation",\n    )\n)\nREFERENCE_NPZ_SHA256 = "14d5cc25eb3bf82a9ccb87c811db28d127bef6d28697dd190dc1e875de3bfe69"\nREFERENCE_B_ROT = 12\nREFERENCE_BATCH = 8\n# Packaged sample circuits. A match means this builder reproduced them.\nREFERENCE_ANGLE_SHA256 = {\n    0: "7bfb3983accc6ff97d6ab41a88deae924f1d1c97cf30e60879d0bff5688a8e25",\n    1: "a04b6d0d57ce851b123af265066aaab2ab33353ea049d26acd534727466d72b6",\n    2: "5c1b8987bc3fa1a4b8607e470f8c8f1392c22e7a7dbcb228f0ce9ec986e18058",\n    3: "16ac67cdcaf3e2c6abdd98c4138ff1ae8a56cfc4ae15622bcdb3d3a3ba1e22cf",\n    4: "d21cdd4c7e94196654aab15ebcabfe10ce8d343151a5e8dc3672e76684b6628d",\n    5: "63b8d4cba36955e6098321f3b2ef3e34401dd276fa629b2e490d043b1743880b",\n    6: "5bf740428e661d035fda860142578f421969e35e0df2e4e024fbf01a7340b378",\n    7: "606009582111a22bd549b795c8d115ac8bbeb124bc8da24a80380eca6d34163d",\n    8: "57c7cc51f5922fff6e95d76e1f723d4c2d029a241da643b95684d6322157a8fe",\n    9: "22cbe2828988ff954fc07fd971a25e6bc04a02c302bf2cc642e4c3535daaad65",\n}\n\nSELECTION_REGISTER = "selection"\n\n\ndef word_register(j: int) -> str:\n    return f"word_{j}"\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1 << 20), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef batch_deltas(angle_words: npt.ArrayLike, batch_size: int) -> npt.NDArray[np.int64]:\n    """``(K, N+R*B, J)`` tables. ``delta[0] = batch[0]`` and later batches are XORs."""\n    words = np.asarray(angle_words)\n    if words.ndim != 2 or not np.issubdtype(words.dtype, np.integer):\n        raise ValueError("angle_words must be a 2-D integer array (N + R*B, N - 1)")\n    if np.any(words < 0):\n        raise ValueError("angle_words must be non-negative")\n    batch_size = index(batch_size)\n    if batch_size < 1:\n        raise ValueError("batch_size must be a positive integer")\n    num_vectors, num_angles = words.shape\n    num_batches = max(1, ceil(num_angles / batch_size))\n    padded = np.zeros((num_vectors, num_batches * batch_size), dtype=np.int64)\n    padded[:, :num_angles] = words\n    batches = padded.reshape(num_vectors, num_batches, batch_size).transpose(1, 0, 2)\n    deltas = batches.copy()\n    deltas[1:] ^= batches[:-1]\n    return deltas\n\n\ndef build_angle_k_circuit(delta: npt.ArrayLike, brot: int):\n    """Ordinary QROM for one ``(A, J)`` XOR-difference table."""\n    import kickmix as km\n\n    delta = np.asarray(delta, dtype=np.int64)\n    brot = index(brot)\n    if not 1 <= brot <= 63:\n        raise ValueError("brot must be an integer between 1 and 63")\n    if delta.ndim != 2 or delta.shape[0] < 2:\n        raise ValueError("delta must have shape (A, J) with A >= 2 addresses")\n    if np.any(delta < 0) or np.any(delta >= np.int64(1) << brot):\n        raise ValueError(f"words must lie in [0, 2**{brot})")\n    num_rows, batch_size = delta.shape\n    num_bits = (num_rows - 1).bit_length()\n    builder = km.CircuitBuilder()\n    selection = builder.create_quantum_register(num_bits, SELECTION_REGISTER)\n    words = [builder.create_quantum_register(brot, word_register(j)) for j in range(batch_size)]\n    ancillas = builder.alloc_qubits(num_bits - 1) if num_bits > 1 else []\n    bits = [selection[i] for i in reversed(range(num_bits))]\n\n    def load(row: int, control: Any) -> None:\n        for j in range(batch_size):\n            for bit in reversed(range(brot)):\n                if (int(delta[row, j]) >> bit) & 1:\n                    builder.cx(control, words[j][bit])\n\n    def node(control: Any, level: int, lo: int, hi: int) -> None:\n        if lo >= num_rows:\n            return\n        if hi <= num_rows and (hi - lo == 1 or np.all(delta[lo:hi] == delta[lo, 0])):\n            load(lo, control)\n            return\n        mid = (lo + hi) // 2\n        if num_rows <= mid:\n            node(control, level + 1, lo, mid)\n            return\n        bit, ancilla = bits[level], ancillas[level - 1]\n        builder.x(bit)\n        builder.init_and(control, bit, ancilla)\n        builder.x(bit)\n        node(ancilla, level + 1, lo, mid)\n        builder.cx(control, ancilla)\n        node(ancilla, level + 1, mid, hi)\n        builder.del_and(control, bit, ancilla)\n\n    half = 1 << (num_bits - 1)\n    builder.x(bits[0])\n    node(bits[0], 1, 0, half)\n    builder.x(bits[0])\n    node(bits[0], 1, half, 2 * half)\n    return builder.finish_circuit()\n\n\ndef build_angle_k_family(angle_words: npt.ArrayLike, brot: int, batch_size: int):\n    return tuple(build_angle_k_circuit(delta, brot) for delta in batch_deltas(angle_words, batch_size))\n\n\ndef load_angle_words(path: Path) -> npt.NDArray[np.int64]:\n    loaded: Any = np.load(path, allow_pickle=False)\n    with loaded as archive:\n        if "angle_words" not in archive.files:\n            raise KeyError(f"{path} has no angle_words entry")\n        words = np.array(archive["angle_words"])\n    if words.ndim != 2 or not np.issubdtype(words.dtype, np.integer):\n        raise ValueError(f"{path}: angle_words must be a 2-D integer array")\n    return words.astype(np.int64)\n\n\ndef reference_npz(data: Path) -> Path:\n    path = data / "sample_submission" / "dfthcsos" / "submission.npz"\n    if not path.is_file():\n        raise FileNotFoundError(f"packaged reference factorization is missing: {path}")\n    digest = sha256_file(path)\n    if digest != REFERENCE_NPZ_SHA256:\n        raise ValueError(f"reference submission.npz SHA-256 is {digest}, expected {REFERENCE_NPZ_SHA256}")\n    return path\n\n\ndef write_family(circuits: Sequence[Any], directory: Path) -> list[Path]:\n    directory.mkdir(parents=True, exist_ok=True)\n    paths = []\n    for k, circuit in enumerate(circuits):\n        path = directory / f"angle_k_{k}.kmb"\n        circuit.to_file(path, format="kmb")\n        paths.append(path)\n    return paths\n\n\ndef generate(out: Path, data: Path = DEFAULT_DATA) -> dict[str, str]:\n    """Write ``<out>/dfthcsos/`` and return relative path → SHA-256."""\n    source = reference_npz(data)\n    destination = out / "dfthcsos"\n    destination.mkdir(parents=True, exist_ok=True)\n    npz_path = destination / "submission.npz"\n    shutil.copyfile(source, npz_path)\n    circuits = build_angle_k_family(load_angle_words(npz_path), REFERENCE_B_ROT, REFERENCE_BATCH)\n    files = {"dfthcsos/submission.npz": sha256_file(npz_path)}\n    for k, path in enumerate(write_family(circuits, destination)):\n        digest = sha256_file(path)\n        files[f"dfthcsos/{path.name}"] = digest\n        expected = REFERENCE_ANGLE_SHA256.get(k)\n        mark = "match" if digest == expected else "DIFFERS"\n        print(f"{path.name}: sha256 {digest} ({mark})")\n    print(f"submission.npz: sha256 {files[\'dfthcsos/submission.npz\']}")\n    return files\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out", type=Path, required=True, help="Submission root. Receives dfthcsos/.")\n    parser.add_argument("--data", type=Path, default=DEFAULT_DATA, help="Competition package root (VQCC_DATA).")\n    args = parser.parse_args()\n    generate(args.out, args.data)\n\n\nif __name__ == "__main__":\n    main()\n', 'tasks/dqi/fused/LICENSE': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'tasks/dqi/fused/SOURCE_MAP.json': '{\n  "source_url": "https://www.kaggle.com/code/somejoint/quantum-circuits-21-34-local-gm-fused-dqi",\n  "author": "Agniv Banerjee (SomeJoint)",\n  "kernel_id": 140675727,\n  "notebook_version": 12,\n  "source_sha256": "18d6760953783bec316ddb46266a1b5ccc50c291fbf47ec35f87c73f4d0cf2ac",\n  "embedded_cell": 6,\n  "build_entry": "blocks/dqi/team-fused/source/exp/decoder/build.py",\n  "expected_circuit_sha256": "7beea291a5de36ef2b4fb50585b2f8d7cce02d3c575b491cf5dcc93e9d9655f6",\n  "closure": "build.py, the 53 Python modules it imports and the 7 JSON files it opens, traced with a sys.audit hook during a full build",\n  "files": [\n    {\n      "file": "source/exp/decoder/build.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/decoder/build.py",\n      "upstream_sha256": "08f423818ff66867f2c91730daf4a98a74b8efb5f79b6cf2552f8f33d80cc645",\n      "port_sha256": "08f423818ff66867f2c91730daf4a98a74b8efb5f79b6cf2552f8f33d80cc645",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/alternating_gauge/even_fold.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/alternating_gauge/even_fold.py",\n      "upstream_sha256": "784cf9b2873b5fa04da4233a528d7ffd8acf9d30fb2e39dc9992d46b1438cd16",\n      "port_sha256": "784cf9b2873b5fa04da4233a528d7ffd8acf9d30fb2e39dc9992d46b1438cd16",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/alternating_gauge/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/alternating_gauge/kernel.py",\n      "upstream_sha256": "d99aa5eaef8a0df416270d8eee0fa55a9847ee7c08c3aa6ee437352e1abdd05a",\n      "port_sha256": "d99aa5eaef8a0df416270d8eee0fa55a9847ee7c08c3aa6ee437352e1abdd05a",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/alternating_gauge/scalar.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/alternating_gauge/scalar.py",\n      "upstream_sha256": "2fb3e5c4ec62df647de519647a57f51c765907ce1b6ecfe2d512035a91523b15",\n      "port_sha256": "2fb3e5c4ec62df647de519647a57f51c765907ce1b6ecfe2d512035a91523b15",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/arithmetic/division.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/arithmetic/division.py",\n      "upstream_sha256": "28567073545c805e32636668cc0bb33f0dbc441a1de0dee2825774b3a33d2832",\n      "port_sha256": "28567073545c805e32636668cc0bb33f0dbc441a1de0dee2825774b3a33d2832",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/arithmetic/field.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/arithmetic/field.py",\n      "upstream_sha256": "9a84208c0690c0c2473f950920b27bacde4bc78f5e8dabf9a83df5a9b362b603",\n      "port_sha256": "9a84208c0690c0c2473f950920b27bacde4bc78f5e8dabf9a83df5a9b362b603",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/arithmetic/lowwidth.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/arithmetic/lowwidth.py",\n      "upstream_sha256": "c536fa2b42393b83aae2bef80eeda22d83a6c799ef555178e94e09f1f5abc22a",\n      "port_sha256": "c536fa2b42393b83aae2bef80eeda22d83a6c799ef555178e94e09f1f5abc22a",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/confluent_extract/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/confluent_extract/kernel.py",\n      "upstream_sha256": "3acfc52dbf5994a85671b975be506b7967d2d075a3e1fc39f7c7cd16a8586ff3",\n      "port_sha256": "3acfc52dbf5994a85671b975be506b7967d2d075a3e1fc39f7c7cd16a8586ff3",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/confluent_extract/model.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/confluent_extract/model.py",\n      "upstream_sha256": "9b8d6fbaad6aec5da87d5247a11276c2fa177883adff9eca030a9bc2f3286387",\n      "port_sha256": "9b8d6fbaad6aec5da87d5247a11276c2fa177883adff9eca030a9bc2f3286387",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/confluent_synthesis/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/confluent_synthesis/kernel.py",\n      "upstream_sha256": "2eaabffce88e50ab3ce4719e764b3ed43c8bd6d8a6acf22a4461574268691c54",\n      "port_sha256": "2eaabffce88e50ab3ce4719e764b3ed43c8bd6d8a6acf22a4461574268691c54",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/convert/convert.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/convert/convert.py",\n      "upstream_sha256": "7afdafa60806df5afbc0af8fd01417a3e09fc15c5dc83fdd28d68079e78df7a0",\n      "port_sha256": "7afdafa60806df5afbc0af8fd01417a3e09fc15c5dc83fdd28d68079e78df7a0",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/convert/hankel_check.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/convert/hankel_check.py",\n      "upstream_sha256": "6d0ac6f55a70418049f46300863eeabb90a7c38b476b84db6e83d395edc3d72c",\n      "port_sha256": "6d0ac6f55a70418049f46300863eeabb90a7c38b476b84db6e83d395edc3d72c",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/convert/kernels.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/convert/kernels.py",\n      "upstream_sha256": "f5efa2f2b008cce86a0b751d400ee75777b77b6872d1875d6fae281cd7f09ef9",\n      "port_sha256": "f5efa2f2b008cce86a0b751d400ee75777b77b6872d1875d6fae281cd7f09ef9",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/convert/repair.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/convert/repair.py",\n      "upstream_sha256": "8beceea9091a9656901290498fbdc0e4373963374b5ee8f7e98d257ece1daeaf",\n      "port_sha256": "8beceea9091a9656901290498fbdc0e4373963374b5ee8f7e98d257ece1daeaf",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/convert_short_hankel/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/convert_short_hankel/kernel.py",\n      "upstream_sha256": "6dd64a123e4bd668bbf30e7d3c6449e738c8bf693f9a31bce1c9c39810ba7ddb",\n      "port_sha256": "6dd64a123e4bd668bbf30e7d3c6449e738c8bf693f9a31bce1c9c39810ba7ddb",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/convert_short_hankel/phase34.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/convert_short_hankel/phase34.py",\n      "upstream_sha256": "620985ba489a03cef51095157af60b7459732b72b68f85a10265f34905ed203f",\n      "port_sha256": "620985ba489a03cef51095157af60b7459732b72b68f85a10265f34905ed203f",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/decoder/common.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/decoder/common.py",\n      "upstream_sha256": "b483a8df5b42be3a3b372633afc05f60ecaf0d9526f96f523c97056167c11109",\n      "port_sha256": "b483a8df5b42be3a3b372633afc05f60ecaf0d9526f96f523c97056167c11109",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/decoder/extract.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/decoder/extract.py",\n      "upstream_sha256": "6597bd52d2bebedae8860acaa4aaeaf980c3d21977781382ba8926833c8f7696",\n      "port_sha256": "6597bd52d2bebedae8860acaa4aaeaf980c3d21977781382ba8926833c8f7696",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/decoder/safe_bridge.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/decoder/safe_bridge.py",\n      "upstream_sha256": "023c81c775967a71ef44c571200bbe3ffb8b3da443fc2816e751856fc3f0d8d9",\n      "port_sha256": "023c81c775967a71ef44c571200bbe3ffb8b3da443fc2816e751856fc3f0d8d9",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/decoder/synthesis.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/decoder/synthesis.py",\n      "upstream_sha256": "fdd7de226b7445007689ce2e50dbabbe77fdbc20e994ec41ac821a6ad2474c49",\n      "port_sha256": "fdd7de226b7445007689ce2e50dbabbe77fdbc20e994ec41ac821a6ad2474c49",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/fold_cached_norm/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/fold_cached_norm/kernel.py",\n      "upstream_sha256": "6a148ebe23eae9194ec0ddb921ff6da1d362565218913170a36691c45dd7c084",\n      "port_sha256": "6a148ebe23eae9194ec0ddb921ff6da1d362565218913170a36691c45dd7c084",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/fold_cached_norm/phase34.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/fold_cached_norm/phase34.py",\n      "upstream_sha256": "974d76939483c061e7b7995600fcdcb05e91529297725acfba41ce5ce808661f",\n      "port_sha256": "974d76939483c061e7b7995600fcdcb05e91529297725acfba41ce5ce808661f",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/fold_joint/divide.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/fold_joint/divide.py",\n      "upstream_sha256": "8456426c171f36a716a39b457a0de5206b5d4db6c8553f1bfc919248ba1de851",\n      "port_sha256": "8456426c171f36a716a39b457a0de5206b5d4db6c8553f1bfc919248ba1de851",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/fold_joint/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/fold_joint/kernel.py",\n      "upstream_sha256": "d4764e763c6ed1ff538b58c1cc628df0c9577300aae5a4d83a5338e0290e018f",\n      "port_sha256": "d4764e763c6ed1ff538b58c1cc628df0c9577300aae5a4d83a5338e0290e018f",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse12_escape/joint.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse12_escape/joint.py",\n      "upstream_sha256": "4bcc5614693d646bd910b9ed4748b278d2ec8dc1584bf8a45d3e465dd757cdc1",\n      "port_sha256": "4bcc5614693d646bd910b9ed4748b278d2ec8dc1584bf8a45d3e465dd757cdc1",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse12_escape/zero1.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse12_escape/zero1.py",\n      "upstream_sha256": "c9274f825d452082d88a98ded25a269aab247626a17032287c26ca1d8a71c724",\n      "port_sha256": "c9274f825d452082d88a98ded25a269aab247626a17032287c26ca1d8a71c724",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse19_joint/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse19_joint/kernel.py",\n      "upstream_sha256": "035152593a105b8178522a6edbf7cda3b709d8a86f51ae2148e71b01a1274913",\n      "port_sha256": "035152593a105b8178522a6edbf7cda3b709d8a86f51ae2148e71b01a1274913",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse_search/baseline.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse_search/baseline.py",\n      "upstream_sha256": "7aecb36f951122861fe0443eee92f1d65a10484fc86f2e535f3db44f5dd553da",\n      "port_sha256": "7aecb36f951122861fe0443eee92f1d65a10484fc86f2e535f3db44f5dd553da",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse_search/inverse20.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse_search/inverse20.py",\n      "upstream_sha256": "e36162777e51361a7ceb47ae6b86a34ce492b23c2fde8083113b03985b98060d",\n      "port_sha256": "e36162777e51361a7ceb47ae6b86a34ce492b23c2fde8083113b03985b98060d",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse_search/inverse20_iso.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse_search/inverse20_iso.py",\n      "upstream_sha256": "1b4ffd1d35b12ca4f327c8885063fb76eef459eeabed1fddae371e2e4bb4f635",\n      "port_sha256": "1b4ffd1d35b12ca4f327c8885063fb76eef459eeabed1fddae371e2e4bb4f635",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse_width/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse_width/kernel.py",\n      "upstream_sha256": "accdd344715e6972ebe22e1679ca872d455a75466f1b1236b83bb636f6466409",\n      "port_sha256": "accdd344715e6972ebe22e1679ca872d455a75466f1b1236b83bb636f6466409",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse_width_subspaces/kernel19.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse_width_subspaces/kernel19.py",\n      "upstream_sha256": "2cb588e60d5fe22ea59d625164e0a0d285cbcd352a80f5bf86f53f76050c9f4b",\n      "port_sha256": "2cb588e60d5fe22ea59d625164e0a0d285cbcd352a80f5bf86f53f76050c9f4b",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/jet24/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/jet24/kernel.py",\n      "upstream_sha256": "f490e6205c678135dc2273d1f02e5ce9540087995f52c37e174b51dce0b4a5f6",\n      "port_sha256": "f490e6205c678135dc2273d1f02e5ce9540087995f52c37e174b51dce0b4a5f6",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/mask_projection/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/mask_projection/kernel.py",\n      "upstream_sha256": "80148ee048d1fd9aaace9637d2e652a7f46dbcb4306913f053437af4a2119088",\n      "port_sha256": "80148ee048d1fd9aaace9637d2e652a7f46dbcb4306913f053437af4a2119088",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/mul99/kernels.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/mul99/kernels.py",\n      "upstream_sha256": "8f93d0a23f465cf72045e02f702b0b40d629b72d1618952a8fc79c22e23ee6ec",\n      "port_sha256": "8f93d0a23f465cf72045e02f702b0b40d629b72d1618952a8fc79c22e23ee6ec",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/mul99/phase36.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/mul99/phase36.py",\n      "upstream_sha256": "d976ab9f6fbb9e2972b9518576b0135b07181d1b6cbfb74a61c094c211e1dccf",\n      "port_sha256": "d976ab9f6fbb9e2972b9518576b0135b07181d1b6cbfb74a61c094c211e1dccf",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/mul99/schedules.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/mul99/schedules.py",\n      "upstream_sha256": "9d16c576747f9d556b857f846c4029131f977cd6eb15d8a37cff4b84b0919c66",\n      "port_sha256": "9d16c576747f9d556b857f846c4029131f977cd6eb15d8a37cff4b84b0919c66",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/noscratch/kernels.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/noscratch/kernels.py",\n      "upstream_sha256": "fa64bb5d5c359c507ac9a488c3f73363d5390bd43a4697daa67df57c322d7370",\n      "port_sha256": "fa64bb5d5c359c507ac9a488c3f73363d5390bd43a4697daa67df57c322d7370",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/noscratch/phase39.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/noscratch/phase39.py",\n      "upstream_sha256": "a47a3ad3ce8787f868b93b40156e3496eead5200b187244b5db13ec0f0c79f1a",\n      "port_sha256": "a47a3ad3ce8787f868b93b40156e3496eead5200b187244b5db13ec0f0c79f1a",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/noscratch/schedules.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/noscratch/schedules.py",\n      "upstream_sha256": "6871354632e828b1b3efed382d84c39bac42e278f5ac700728efc3f1745cfa8d",\n      "port_sha256": "6871354632e828b1b3efed382d84c39bac42e278f5ac700728efc3f1745cfa8d",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/partial_gauge/prefix.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/partial_gauge/prefix.py",\n      "upstream_sha256": "dba10093b9314daf696ebd8fbc85543dbcf73534eb45d2303dc9ce53189b0652",\n      "port_sha256": "dba10093b9314daf696ebd8fbc85543dbcf73534eb45d2303dc9ce53189b0652",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/partial_gauge/scalar.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/partial_gauge/scalar.py",\n      "upstream_sha256": "7b3cb8eece8ccbc42462f13dcab9ef6b253581a06ce4d14906d88a541774de29",\n      "port_sha256": "7b3cb8eece8ccbc42462f13dcab9ef6b253581a06ce4d14906d88a541774de29",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/partial_gauge/synthesis_noflag.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/partial_gauge/synthesis_noflag.py",\n      "upstream_sha256": "cb316d80cdcfa636735156171a2b4d0bb57be22e4c5495806709f224fd0ea8ae",\n      "port_sha256": "cb316d80cdcfa636735156171a2b4d0bb57be22e4c5495806709f224fd0ea8ae",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/partial_gauge/tail.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/partial_gauge/tail.py",\n      "upstream_sha256": "cef30a29c419aef478fbf9c98f4277ab4a1fc63ca52e95d6025d592f16c7aff3",\n      "port_sha256": "cef30a29c419aef478fbf9c98f4277ab4a1fc63ca52e95d6025d592f16c7aff3",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/phase24_tradeoff/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/phase24_tradeoff/kernel.py",\n      "upstream_sha256": "984fc30f4c834b8f3cc1a7276f26f54801c01c8fd53b757e9ba767325059711a",\n      "port_sha256": "984fc30f4c834b8f3cc1a7276f26f54801c01c8fd53b757e9ba767325059711a",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/phase24_tradeoff/phase6small.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/phase24_tradeoff/phase6small.py",\n      "upstream_sha256": "28590db3c6a0843fedc2894bb9899df45a7b99ea4544ce6b7517e739e672fe32",\n      "port_sha256": "28590db3c6a0843fedc2894bb9899df45a7b99ea4544ce6b7517e739e672fe32",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/safe_scalar/converter.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/safe_scalar/converter.py",\n      "upstream_sha256": "771a242b7910c3e59f27bc82c72e8f7b86f9cb201d84ca3712ba7fd7e28a2841",\n      "port_sha256": "771a242b7910c3e59f27bc82c72e8f7b86f9cb201d84ca3712ba7fd7e28a2841",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/safe_scalar/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/safe_scalar/kernel.py",\n      "upstream_sha256": "72fbedc923066741e50cf4372de75ade749844c3554005b2f72ea5594190e84a",\n      "port_sha256": "72fbedc923066741e50cf4372de75ade749844c3554005b2f72ea5594190e84a",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/safe_scalar/phase.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/safe_scalar/phase.py",\n      "upstream_sha256": "30fdcd43d2234c6476344c1608449684199c8f3e8fb403f96aca80ae2379b048",\n      "port_sha256": "30fdcd43d2234c6476344c1608449684199c8f3e8fb403f96aca80ae2379b048",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/scan/scan.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/scan/scan.py",\n      "upstream_sha256": "bab543d7fbaa33666e8c718a40ec47ac2b9a52e06e59de27bbbe45e68c754b55",\n      "port_sha256": "bab543d7fbaa33666e8c718a40ec47ac2b9a52e06e59de27bbbe45e68c754b55",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/tail_alias/divide.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/tail_alias/divide.py",\n      "upstream_sha256": "46d4c7c4c94a06e9e244c030934ab45092ad6793b530c501226e2edb62a3ee5c",\n      "port_sha256": "46d4c7c4c94a06e9e244c030934ab45092ad6793b530c501226e2edb62a3ee5c",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/tail_alias/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/tail_alias/kernel.py",\n      "upstream_sha256": "3452ed2c039c73d3b1f45743f1ddeb40b88ffc32178b4052115f17bf18ba48ff",\n      "port_sha256": "3452ed2c039c73d3b1f45743f1ddeb40b88ffc32178b4052115f17bf18ba48ff",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/zero_padding_synthesis/kernel.py",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/zero_padding_synthesis/kernel.py",\n      "upstream_sha256": "f2b84166a1ef1fb131ead3d907b21625375bd45a24187e74b440ff45647fc185",\n      "port_sha256": "f2b84166a1ef1fb131ead3d907b21625375bd45a24187e74b440ff45647fc185",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/playbook/exp/cf_check/audit.py",\n      "upstream_path": "blocks/dqi/team-fused/source/playbook/exp/cf_check/audit.py",\n      "upstream_sha256": "c3211bbccebce2183b1a24385812dc505d1041ca7244274bbdffa75a95f37b33",\n      "port_sha256": "c3211bbccebce2183b1a24385812dc505d1041ca7244274bbdffa75a95f37b33",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse12_escape/zero_layout.json",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse12_escape/zero_layout.json",\n      "upstream_sha256": "45d1361fcf582282d15cc6fabeab290f8e9fd9dc4c68a3e14ffff86e9f9a537f",\n      "port_sha256": "45d1361fcf582282d15cc6fabeab290f8e9fd9dc4c68a3e14ffff86e9f9a537f",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse19_joint/layout.json",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse19_joint/layout.json",\n      "upstream_sha256": "0adee49170ae597ed5a7df4d4d0df0e495fba49013f6acd9ae6a3f06b9ad4ab4",\n      "port_sha256": "0adee49170ae597ed5a7df4d4d0df0e495fba49013f6acd9ae6a3f06b9ad4ab4",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse_search/optimized.json",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse_search/optimized.json",\n      "upstream_sha256": "b18cfd6025658d02e38adfbe0ea7eca09960bbbdd458abe1ee1d539e4d3f6ca7",\n      "port_sha256": "b18cfd6025658d02e38adfbe0ea7eca09960bbbdd458abe1ee1d539e4d3f6ca7",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse_search/optimized_isotopy2.json",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse_search/optimized_isotopy2.json",\n      "upstream_sha256": "28424aa2f6b0b92b7b3b4f6bc62feda566746feb9a9f21660ed5103420eae606",\n      "port_sha256": "28424aa2f6b0b92b7b3b4f6bc62feda566746feb9a9f21660ed5103420eae606",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse_width/partial5.json",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse_width/partial5.json",\n      "upstream_sha256": "202ddf3a0a62530ac4f4028774e87c729cba6cc85dd465570a59e882a83cf4f7",\n      "port_sha256": "202ddf3a0a62530ac4f4028774e87c729cba6cc85dd465570a59e882a83cf4f7",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/inverse_width_subspaces/inverse19.json",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/inverse_width_subspaces/inverse19.json",\n      "upstream_sha256": "c868b9fc56f05ec89d34375ad086490691237211ecdc99b4a0c0b6e86b468a3f",\n      "port_sha256": "c868b9fc56f05ec89d34375ad086490691237211ecdc99b4a0c0b6e86b468a3f",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "source/exp/phase24_tradeoff/phase6_six_search.json",\n      "upstream_path": "blocks/dqi/team-fused/source/exp/phase24_tradeoff/phase6_six_search.json",\n      "upstream_sha256": "afb39dcf8748b1b1419a5e8db6b5440be74db6687464e6e0a9052a08b168fdb0",\n      "port_sha256": "afb39dcf8748b1b1419a5e8db6b5440be74db6687464e6e0a9052a08b168fdb0",\n      "changes": "none (verbatim)"\n    },\n    {\n      "file": "LICENSE",\n      "upstream_path": "LICENSE",\n      "upstream_sha256": "cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30",\n      "port_sha256": "cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30",\n      "changes": "none (verbatim)"\n    }\n  ]\n}\n', 'tasks/dqi/fused/source/exp/alternating_gauge/even_fold.py': '"""Full product/quotient jointly updates explicit odd continuant r, no shear."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.tail_alias.kernel import *\nfrom exp.mul99.schedules import divide_zero_bridge414\n\ndef full_product(b,head,A,qi,target):\n n=len(target);V,W=base.vandermonde(n)\n for t in range(n):old.rank_affine(b,head,[1,t],A+[qi],V[t][:len(A)+1],target,[W[j][t]for j in range(n)])\n\ndef full_quotient(b,head,A,qi,B,r,kap):\n k=len(B)\n if not k:ar.mul_xor0(b,head[1],qi,r);return\n words=A[1:]+[qi]\n for ac,bc,tc in reversed(base.quotient_terms(k+2,kap)):\n  old.rank_affine(b,head,ac,words,bc,B+[r],tc)\n\ndef absorb_even(b,head,A,B,qi,r,c,kap,reverse=False,transcript=None):\n """r=qnext->qprev in forward, inverse restores qnext; qi disjoint r."""\n k=len(B);out=head+B;target=out+[r]\n if not reverse:\n  value=old.eval_implicit(b,A,qi,c);F=dv.zero_flag(b,value);old.eval_implicit(b,A,qi,c,True)\n  b.cx(F,A[0][0]);spare=[b.alloc_qubits(12),b.alloc_qubits(12)];full=spare+B+[r]\n  linear_words(b,full,base.embedding_matrix(k+3,kap));full_product(b,head,A,qi,full);b.cx(F,A[0][0])\n  hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n  for w,z in zip(head,spare):b.swap(w,z);b.free(z)\n  fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n  return out,A,(hm,fm)\n linear_words(b,target,base.embedding_matrix(k+3,kap),True)\n value=old.eval_implicit(b,A,qi,c);cx(b,head[1],head[0],c);hm,fm=transcript\n divide_zero_bridge414(b,combine(head),value,K=K,extra_zero_mask=fm[0]);b.free(fm)\n cx(b,head[1],head[0],c);b.cz(combine(head),hm);b.free(hm);old.eval_implicit(b,A,qi,c,True)\n full_quotient(b,head,A,qi,B,r,kap)\n return out,A,transcript\n', 'tasks/dqi/fused/source/exp/alternating_gauge/kernel.py': '"""Half-density continuant prefix: odd encoded heads, one rolling odd q.\n\nRaw b0 becomes a reversible physical q0=1 border; r carries its old value.\nThe border is freed after inner folds and r becomes the numerator top word.\nNo persistent quantum zero flag or second leading-coefficient bank is used.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.tail_alias.kernel import *\nfrom exp.alternating_gauge.even_fold import absorb_even\nfrom exp.alternating_gauge.scalar import prefix_reverse141,divide_zero117,multiply_zero144\n\ndef add_tail_a(b,a,q,d):\n ar.linear_inplace(b,q,ar.square_columns(12));ar.mul_xor0(b,a,q,d)\n ar.linear_inplace(b,q,[power(1<<j,2048)for j in range(12)])\n\ndef odd_encode(b,heads,i,r,d=None):\n a,x=heads[i];p=heads[i-2][1]if i>1 else heads[0][1]\n def after():\n  add_tail_a(b,a,r,d);ar.mul_xor0(b,x,r,d)\n return pg.forward(b,x,p,r,after_product=after if d is not None else None)\n\ndef odd_decode(b,heads,i,r,m,d=None):\n a,x=heads[i];p=heads[i-2][1]if i>1 else heads[0][1]\n if d is not None:ar.mul_xor0(b,x,r,d)\n prefix_reverse141(b,x,p,r,m)\n if d is not None:add_tail_a(b,a,r,d)\n\ndef even_step(b,heads,i,r,d=None,reverse=False):\n a,x=heads[i];q=heads[i-1][1]\n if d is not None and reverse:\n  add_tail_a(b,a,q,d);ar.mul_xor0(b,r,q,d)\n ar.mul_xor0(b,x,q,r)\n if d is not None and not reverse:\n  add_tail_a(b,a,q,d);ar.mul_xor0(b,r,q,d)\n\ndef synthesize(b,words):\n h=len(words)//2;assert h>=3 and h%2==1;cs,ks,_,_=old.constants(h)\n heads=[words[2*i:2*i+2]for i in range(h)];d=heads[0][0];border=heads[0][1]\n r=b.alloc_qubits(12);b.cx(border,r);b.cx(border,d);b.cx(r,border);b.x(border[0]);masks=[None]*h\n for i in range(1,h):\n  if i%2:masks[i]=odd_encode(b,heads,i,r,d)\n  else:even_step(b,heads,i,r,d)\n divide_zero117(b,d,r);A=[d];B=[];hist=[]\n for i in reversed(range(1,h)):\n  if i%2:\n   odd_decode(b,heads,i,r,masks[i]);qi=r;p=heads[i-2][1]if i>1 else border\n  else:\n   qi=heads[i-1][1];p=r\n  aa,bb=A,B\n  if i%2:A,B,token=absorb(b,heads[i],A,B,qi,p,None,0,cs[i],ks[i],wide=False)\n  else:A,B,token=absorb_even(b,heads[i],A,B,qi,r,cs[i],ks[i])\n  hist.append((i,aa,bb,qi,p,cs[i],ks[i],token))\n b.x(border[0]);b.free(border);C=B+[r];ot=outer(b,C,A,cs[0],ks[0],wide=True)\n return C,A,dict(h=h,heads=heads,d=d,r=r,border=border,history=hist,A=C,B=A,outer=ot)\n\ndef unsynthesize(b,t):\n h=t[\'h\'];heads=t[\'heads\'];r=t[\'r\'];cs,ks,_,_=old.constants(h)\n outer(b,t[\'A\'],t[\'B\'],cs[0],ks[0],True,t[\'outer\'],True)\n reclaim(b,[t[\'border\']]);b.x(t[\'border\'][0]);masks=[None]*h\n for i,A,B,qi,p,c,kap,token in reversed(t[\'history\']):\n  if i%2:\n   absorb(b,heads[i],A,B,qi,p,None,0,c,kap,True,token,False);masks[i]=odd_encode(b,heads,i,r)\n  else:absorb_even(b,heads[i],A,B,qi,r,c,kap,True,token)\n multiply_zero144(b,t[\'d\'],r)\n for i in reversed(range(1,h)):\n  if i%2:odd_decode(b,heads,i,r,masks[i],t[\'d\'])\n  else:even_step(b,heads,i,r,t[\'d\'],True)\n b.cx(r,t[\'d\']);b.x(t[\'border\'][0]);b.cx(r,t[\'border\']);b.cx(t[\'border\'],r);b.free(r)\n', 'tasks/dqi/fused/source/exp/alternating_gauge/scalar.py': '"""25-private zero-aware scalar context; same117T as promised old divide."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.tail_alias.kernel import *\nfrom exp.inverse19_joint.kernel import inverse6_init19,inverse_zero\nfrom exp.arithmetic.lowwidth import phase_inverse_generic6\nfrom exp.safe_scalar.phase import phase_inverse36_zero\nfrom exp.noscratch.schedules import norm_xor_noscratch\n\n\ndef divide_zero117(b,x,y,before=None,after=None):\n """All-input x/=y or x when y0;117T/25private; hooks see F=[y0]."""\n to,frm,kk=ar.tower(12);N=b.alloc_qubits(6);ar.linear_inplace(b,y,to);norm_xor_noscratch(b,y,N,kk);ar.linear_inplace(b,y,frm)\n u,F=inverse6_init19(b,N,True)\n if before is not None:before(F)\n conj=[ar.field(12).frobenius(1<<i,6)for i in range(12)]\n ar.linear_inplace(b,y,conj);b.cx(F,y[0]);out=b.alloc_qubits(12);ar.mul_xor0(b,x,y,out)\n oldm=b.alloc_bits(12);b.hmr(x,oldm);b.swap(x,out);b.free(out);b.cx(F,y[0]);ar.linear_inplace(b,y,conj)\n ar.linear_inplace(b,x,to);b.cx(F,u[0]);b.cx(F,N[0])\n for off in(0,6):\n  half=x[off:off+6];out=b.alloc_qubits(6);ar.mul_xor0(b,half,u,out)\n  m=b.alloc_bits(6);b.hmr(half,m);b.swap(half,out);b.free(out);ar.phase_product(b,half,N,m);b.free(m)\n b.cx(F,N[0]);b.cx(F,u[0]);ar.linear_inplace(b,x,frm)\n b.cx(F,y[0]);ar.phase_product(b,x,y,oldm);b.cx(F,y[0]);b.free(oldm)\n if after is not None:after(F)\n fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F);b.z(fm[0]);zm=b.alloc_bits(6);b.bit_store(zm,False);b.cx(fm[0],zm[0]);ar.phase_product(b,N,u,zm);b.free(zm);b.free(fm)\n um=b.alloc_bits(6);b.hmr(u,um);b.free(u);phase_inverse_generic6(b,N,um);b.free(um)\n ar.linear_inplace(b,y,to);ar._erase_norm(b,y,N,kk);ar.linear_inplace(b,y,frm);b.free(N)\n\ndef multiply_zero144(b,x,y):\n """Inverse of divide_zero117,144T/25private, no deferred transcript."""\n u,F=inverse_zero(b,y);b.cx(F,y[0]);out=b.alloc_qubits(12);ar.mul_xor0(b,x,y,out)\n m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out);b.cx(F,y[0]);b.cx(F,u[0]);ar.phase_product(b,x,u,m);b.cx(F,u[0]);b.free(m)\n um=b.alloc_bits(12);fm=b.alloc_bits(1);b.hmr(u,um);b.hmr(F,fm[0]);b.free(u);b.free(F)\n phase_inverse36_zero(b,y,um,fm[0]);b.free(um);b.free(fm)\n\ndef prefix_reverse141(b,x,p,q,m):\n """Inverse oldpg.forward:141T/25private, including zero-q swap chart."""\n def before(F):b.cswap(F,x,p);b.cx(p,x);b.ccx(F,p,x)\n def after(F):\n  b.cz(x,m)\n  for j in range(12):b.ccz(F,x[j],m[j]);b.ccz(F,p[j],m[j])\n  b.free(m)\n divide_zero117(b,x,q,before,after)\n', 'tasks/dqi/fused/source/exp/arithmetic/division.py': '"""Width-bounded GF24 scalar multiplication/division reference schedules.\n\nAll24-bit words use explicit a+b*y, y²+y=K. This file is separate from the\nverified primitive API while the scalar-normalization direction is measured.\n"""\nfrom field import *\n\n\ndef mul24_xor126(b,x,y,z,K=None):\n    """z ^=x*y,126Toffolis,1scratch; inputs/target disjoint."""\n    if K is None:K=extension_columns()[2]\n    f=field(12);a=x[:12];c=x[12:];d=y[:12];e=y[12:];za=z[:12];zb=z[12:]\n    # p=a*d contributes to both output halves.\n    b.cx(za,zb);mul_xor(b,a,d,za);b.cx(za,zb)\n    # q=c*e contributes Kq to the first half only.\n    b.gf2_imul(f.invert(K),target=km.array(za),field=f);mul_xor(b,c,e,za);b.gf2_imul(K,target=km.array(za),field=f)\n    # r=(a+c)(d+e) contributes to the second half.\n    b.cx(c,a);b.cx(e,d);mul_xor(b,a,d,zb);b.cx(e,d);b.cx(c,a)\n\n\n\ndef mul24_xor(b,x,y,z,K=None):\n    """Default105T/1scratch product; old126Tcomparison ismul24_xor126."""\n    mul24_xor105(b,x,y,z,K)\n\n\ndef phase_product24(b,x,y,mask,K=None):\n    if K is None:K=extension_columns()[2]\n    a=x[:12];c=x[12:];d=y[:12];e=y[12:];ma=mask[:12];mb=mask[12:]\n    phase_product(b,a,d,ma)\n    mk=b.alloc_bits(12);b.bit_store(mk,False)\n    # Transpose constant multiplication K into the classical phase mask.\n    for i in range(12):\n        for j in bits(field(12).mul(K,1<<i)):b.cx(ma[j],mk[i])\n    phase_product(b,c,e,mk);b.free(mk)\n    phase_product(b,a,e,mb);phase_product(b,c,d,mb);phase_product(b,c,e,mb)\n\n\ndef multiply_forward(b,x,y,K=None):\n    """105T forward x*=y, returns deferred oldx phase mask.\n\n    For y=0 this is correct only on valid subspace x=0. It is not an all-input\n    identity-at-zero permutation; caller must enforce that promise or add flag.\n    """\n    z=b.alloc_qubits(24);mul24_xor(b,x,y,z,K)\n    m=b.alloc_bits(24);b.hmr(x,m);b.swap(x,z);b.free(z);return m\n\n\ndef divide_preserving_factor(b,x,y,K=None):\n    """513T division x/=y preserving y,43workspace; valid x=0 when y=0."""\n    ym=inverse24_inplace_forward(b,y,K)\n    z=b.alloc_qubits(24);mul24_xor(b,x,y,z,K)\n    xm=b.alloc_bits(24);b.hmr(x,xm);b.swap(x,z);b.free(z)\n    inverse24_inplace_reverse(b,y,ym,K)\n    phase_product24(b,x,y,xm,K);b.free(xm)\n\n\ndef multiply_reverse(b,x,y,transcript,K=None):\n    divide_preserving_factor(b,x,y,K)\n    b.cz(x,transcript);b.free(transcript)\n\n\ndef divide_via_norm(b,x,y,K=None,retain_norm=True):\n    """Division x/=y:321T/43workspace or342T/37workspace.\n\n    y=0 requires promisedx=0. This all-valid-state exactness includes0 but does\n    NOT implement identity on arbitraryx at y=0.\n    """\n    import field as ar\n    if K is None:K=extension_columns()[2]\n    b.cx(y[12:],y[:12]);oldmask=multiply_forward(b,x,y,K);b.cx(y[12:],y[:12])\n    N=b.alloc_qubits(12);ar._norm_xor(b,y,N,K)\n    if retain_norm:u,D=ar._inverse_keep_norm(b,N)\n    else:u=inverse(b,N)\n    for offset in (0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor(b,half,u,out)\n        erase_product(b,out,N,half);b.swap(half,out);b.free(out)\n    if retain_norm:\n        to,frm,K12=tower(12);linear_inplace(b,u,to)\n        v=b.alloc_qubits(6);ar._norm_xor(b,u,v,K12);linear_inplace(b,u,frm)\n        ar._erase_inverse_with_norm(b,N,u,v);b.free(u)\n        erase_inverse(b,D,v);b.free(v)\n        linear_inplace(b,N,to);ar._erase_norm(b,N,D,K12);b.free(D);linear_inplace(b,N,frm)\n    else:erase_inverse_releasing(b,N,u)\n    ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n\n\ndef multiply_reverse_via_norm(b,x,y,transcript,K=None,retain_norm=True):\n    divide_via_norm(b,x,y,K,retain_norm=retain_norm)\n    b.cz(x,transcript);b.free(transcript)\n\n\ndef zero_flag(b,x):\n    """Compute [x=0], n-1ANDs; returnoneflag, clearallinternalANDs."""\n    n=len(x);b.x(x);ladder=b.alloc_qubits(n-1)\n    b.init_and(x[0],x[1],ladder[0])\n    for i in range(1,n-1):b.init_and(ladder[i-1],x[i+1],ladder[i])\n    for i in reversed(range(n-2)):\n        if i==0:b.del_and(x[0],x[1],ladder[0])\n        else:b.del_and(ladder[i-1],x[i+1],ladder[i])\n    b.x(x);b.free(ladder[:-1]);return ladder[-1]\n\n\ndef phase_zero(b,y,mask,head=None,headmask=None):\n    """Applyzero(y)*(mask+dot(head,headmask));22T+1CCZ for y24."""\n    n=len(y);b.x(y);ladder=b.alloc_qubits(n-2)\n    b.init_and(y[0],y[1],ladder[0])\n    for i in range(1,n-2):b.init_and(ladder[i-1],y[i+1],ladder[i])\n    b.ccz(ladder[-1],y[-1],mask)\n    if head is not None:\n        q=b.alloc_qubits(1)[0]\n        for xi,mi in zip(head,headmask):b.ccx(mi,xi,q)\n        b.ccz(ladder[-1],y[-1],q)\n        for xi,mi in reversed(list(zip(head,headmask))):b.ccx(mi,xi,q)\n        b.free(q)\n    for i in reversed(range(n-2)):\n        if i==0:b.del_and(y[0],y[1],ladder[0])\n        else:b.del_and(ladder[i-1],y[i+1],ladder[i])\n    b.free(ladder);b.x(y)\n\n\ndef multiply_zero_forward(b,x,y,K=None):\n    """152T multiplication, identityat y0. Deferredhead+zero transcripts."""\n    flag=zero_flag(b,y);z=b.alloc_qubits(24);mul24_xor(b,x,y,z,K);b.ccx(flag,x,z)\n    oldmask=b.alloc_bits(24);b.hmr(x,oldmask);b.swap(x,z);b.free(z)\n    fm=b.alloc_bits(1);b.hmr(flag,fm[0]);b.free(flag)\n    return oldmask,fm[0]\n\n\ndef divide_zero(b,x,y,K=None,extra_zero_mask=None):\n    """379T exact division x/=y, identityat y0,43workspace."""\n    import field as ar\n    if K is None:K=extension_columns()[2]\n    N=b.alloc_qubits(12);ar._norm_xor(b,y,N,K);flag=zero_flag(b,N)\n    b.cx(y[12:],y[:12]);z=b.alloc_qubits(24);mul24_xor(b,x,y,z,K);b.ccx(flag,x,z);b.cx(y[12:],y[:12])\n    oldmask=b.alloc_bits(24);b.hmr(x,oldmask);b.swap(x,z);b.free(z)\n    b.cx(flag,N[0]);fm=b.alloc_bits(1);b.hmr(flag,fm[0]);b.free(flag)\n    u,D=ar._inverse_keep_norm(b,N)\n    for offset in (0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor(b,half,u,out)\n        erase_product(b,out,N,half);b.swap(half,out);b.free(out)\n    to,frm,K12=tower(12);linear_inplace(b,u,to);v=b.alloc_qubits(6);ar._norm_xor(b,u,v,K12);linear_inplace(b,u,frm)\n    ar._erase_inverse_with_norm(b,N,u,v);b.free(u);erase_inverse(b,D,v);b.free(v)\n    linear_inplace(b,N,to);ar._erase_norm(b,N,D,K12);b.free(D);linear_inplace(b,N,frm)\n    nm=b.alloc_bits(12);b.hmr(N,nm);b.free(N);ar._phase_norm(b,y,nm,K)\n    comb=b.alloc_bits(1);b.bit_store(comb,False);b.cx(fm[0],comb[0]);b.cx(nm[0],comb[0])\n    if extra_zero_mask is not None:b.cx(extra_zero_mask,comb[0])\n    phase_product24(b,x,y,oldmask,K);phase_zero(b,y,comb[0],x,oldmask)\n    b.free(oldmask);b.free(fm);b.free(nm);b.free(comb)\n\n\ndef multiply_zero_reverse(b,x,y,transcript,K=None):\n    oldmask,fm=transcript;divide_zero(b,x,y,K,extra_zero_mask=fm)\n    b.cz(x,oldmask);b.free(oldmask);b.free(fm)\n\n\n\ndef erase_inverse_releasing(b,x,y):\n    """Phase-only inverse erasure; measure/free target BEFORE recomputation.\n\n    Same9/45/153T recursion as field.erase_inverse, but permits early reuse\n    of y\'s qubits. This helper owns and frees y.\n    """\n    import field as ar\n    n=len(x)\n    if n==3:\n        mask=b.alloc_bits(3);b.hmr(y,mask);b.free(y)\n        for im,v in enumerate(inverse3_anf()):\n            ix=list(bits(im))\n            for j in bits(v):\n                if len(ix)==1:b.cz(x[ix[0]],mask[j])\n                elif len(ix)==2:b.ccz(x[ix[0]],x[ix[1]],mask[j])\n        b.free(mask);return\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,y,to)\n    mask=b.alloc_bits(n);b.hmr(y,mask);b.free(y)\n    linear_inplace(b,x,to);norm=b.alloc_qubits(k);ar._norm_xor(b,x,norm,K);invnorm=inverse(b,norm)\n    b.cx(x[k:],x[:k]);phase_product(b,x[:k],invnorm,mask[:k]);phase_product(b,x[k:],invnorm,mask[k:]);b.cx(x[k:],x[:k]);b.free(mask)\n    erase_inverse_releasing(b,norm,invnorm)\n    ar._erase_norm(b,x,norm,K);b.free(norm);linear_inplace(b,x,frm)\n', 'tasks/dqi/fused/source/exp/arithmetic/field.py': '"""Low-width GF(2^m) arithmetic, standard Kickmix polynomial basis.\n\nPublic: mul_xor(builder,x,y,z), inverse(builder,x,target=None),\nlinear_xor(builder, x, z, columns), linear_inplace(builder,x,columns),\nphase_product(builder,x,y,mask), erase_linear/erase_product/erase_inverse.\nAll array arguments can be lists or km.array. Arithmetic preserves inputs.\n`inverse` returns a fresh clean output, 0 maps to 0. Default GF2Field basis.\n"""\nfrom functools import lru_cache\nimport kickmix as km\n\n\ndef bits(x):\n    while x:\n        t=x&-x; yield t.bit_length()-1; x^=t\n\n\ndef apply(columns, value):\n    out=0\n    for i in bits(value): out ^= columns[i]\n    return out\n\n\ndef inverse_columns(columns):\n    n=len(columns)\n    rows=[sum(((c>>i)&1)<<j for j,c in enumerate(columns)) | (1<<(n+i)) for i in range(n)]\n    for j in range(n):\n        p=next(i for i in range(j,n) if rows[i]>>j&1)\n        rows[j],rows[p]=rows[p],rows[j]\n        for i in range(n):\n            if i!=j and rows[i]>>j&1: rows[i]^=rows[j]\n    return [sum(((rows[i]>>(n+j))&1)<<i for i in range(n)) for j in range(n)]\n\n\ndef linear_xor(b,x,z,columns):\n    for i,c in enumerate(columns):\n        for j in bits(c): b.cx(x[i],z[j])\n\n\ndef linear_inplace(b,x,columns):\n    """Invertible GF2 map, no ancillary qubits. columns[j] = image(1<<j)."""\n    n=len(x); rows=[sum(((c>>i)&1)<<j for j,c in enumerate(columns)) for i in range(n)]; ops=[]\n    for j in range(n):\n        p=next(i for i in range(j,n) if rows[i]>>j&1)\n        if p!=j: rows[p],rows[j]=rows[j],rows[p]; ops.append((\'swap\',p,j))\n        for i in range(n):\n            if i!=j and rows[i]>>j&1: rows[i]^=rows[j];ops.append((\'cx\',j,i))\n    for op,i,j in reversed(ops): getattr(b,op)(x[i],x[j])\n\n\ndef phase_linear(b,x,mask,columns):\n    for i,c in enumerate(columns):\n        for j in bits(c): b.cz(x[i],mask[j])\n\n\ndef erase_linear(b,x,z,columns):\n    mask=b.alloc_bits(len(z)); b.hmr(z,mask); phase_linear(b,x,mask,columns);b.free(mask)\n\n\n@lru_cache(None)\ndef field(n): return km.GF2Field(n)\n\n\n@lru_cache(None)\ndef tensor(n):\n    """Symmetric multiplication tensor (linear mask, output vector), rank3/6/15/42."""\n    f=field(n)\n    if n==2: masks=[1,2,3]\n    elif n==3: masks=[1,7,4,5,6,3]\n    elif n in (6,12,24):\n        k,d={6:(2,3),12:(3,4),24:(6,4)}[n]\n        sub=field(k); power=(2**n-1)//(2**k-1)\n        gen=f.pow(f.primitive_element,power)\n        def evalpoly(v,p):\n            q=0\n            for i in reversed(range(k+1)):q=f.mul(q,p)^((v>>i)&1)\n            return q\n        embroot=next(f.pow(gen,i) for i in range(1,2**k) if evalpoly(sub.modulus,f.pow(gen,i))==0)\n        embed=[f.pow(embroot,i) for i in range(k)]\n        theta=f.primitive_element\n        cols=[f.mul(embed[i],f.pow(theta,j)) for j in range(d) for i in range(k)]\n        inv=inverse_columns(cols)\n        lowmasks=[m for m,_ in tensor(k)]\n        masks=[]\n        for point in list(range(2*d-2))+[None]:\n            outcols=[]\n            for extbit in range(n):\n                coord=inv[extbit]\n                if point is None: value=(coord>>(k*(d-1)))&((1<<k)-1)\n                else:\n                    value=0\n                    for j in reversed(range(d)): value=sub.mul(value,point)^((coord>>(k*j))&((1<<k)-1))\n                outcols.append(value)\n            for lm in lowmasks:masks.append(sum(((v&lm).bit_count()%2)<<i for i,v in enumerate(outcols)))\n    else: raise ValueError(n)\n    # Each basis-pair equation determines output vector of rank terms.\n    rank=len(masks); piv={}\n    for i in range(n):\n        for j in range(i,n):\n            row=sum(((m>>i&1)&(m>>j&1))<<r for r,m in enumerate(masks)); rhs=f.mul(1<<i,1<<j)\n            while row:\n                p=(row&-row).bit_length()-1\n                if p in piv:row^=piv[p][0];rhs^=piv[p][1]\n                else:piv[p]=(row,rhs);break\n            if row==0: assert rhs==0,(n,i,j,rhs)\n    out=[0]*rank\n    for p,(row,rhs) in sorted(piv.items(),reverse=True):\n        for j in bits(row^(1<<p)):rhs^=out[j]\n        out[p]=rhs\n    assert all(out)\n    return tuple(zip(masks,out))\n\n\ndef parity_into(b,x,mask,t):\n    for i in bits(mask):b.cx(x[i],t)\n\n\ndef mul_xor(b,x,y,z):\n    """Accumulate product, 1 scratch when inputs disjoint; 3 if overlapping.\n\n    Linear forms are computed temporarily in a chosen input pivot, restored\n    after each rank-one term. The output must be disjoint from both inputs.\n    """\n    n=len(x);assert len(y)==len(z)==n\n    overlap=bool(set(map(str,x)) & set(map(str,y)))\n    if overlap:\n        a,c,p=b.alloc_qubits(3)\n        for mask,v in tensor(n):\n            parity_into(b,x,mask,a);parity_into(b,y,mask,c)\n            b.init_and(a,c,p)\n            for i in bits(v):b.cx(p,z[i])\n            b.del_and(a,c,p)\n            parity_into(b,y,mask,c);parity_into(b,x,mask,a)\n        b.free([a,c,p])\n    else:\n        p=b.alloc_qubits(1)[0]\n        for mask,v in tensor(n):\n            ix=list(bits(mask));pivot=ix[0];a=x[pivot];c=y[pivot]\n            for i in ix[1:]:b.cx(x[i],a);b.cx(y[i],c)\n            b.init_and(a,c,p)\n            for i in bits(v):b.cx(p,z[i])\n            b.del_and(a,c,p)\n            for i in reversed(ix[1:]):b.cx(y[i],c);b.cx(x[i],a)\n        b.free(p)\n\n\ndef phase_product(b,x,y,mask):\n    """Exact phase dot(mask,x*y), zero Toffolis, no quantum workspace."""\n    f=field(len(x))\n    for i in range(len(x)):\n        for j in range(len(y)):\n            v=f.mul(1<<i,1<<j)\n            for k in bits(v):b.ccz(x[i],y[j],mask[k])\n\n\ndef erase_product(b,x,y,z):\n    mask=b.alloc_bits(len(z));b.hmr(z,mask);phase_product(b,x,y,mask);b.free(mask)\n\n\n@lru_cache(None)\ndef tower(n):\n    """Coordinates a+b*y, y²+y=K; returns to/from tower and K."""\n    k=n//2;f=field(n);sub=field(k)\n    gen=f.pow(f.primitive_element,(2**n-1)//(2**k-1))\n    def ev(v,p):\n        out=0\n        for i in reversed(range(k+1)):out=f.mul(out,p)^((v>>i)&1)\n        return out\n    root=next(f.pow(gen,i) for i in range(1,2**k) if ev(sub.modulus,f.pow(gen,i))==0)\n    embed=[f.pow(root,i) for i in range(k)]\n    # Any trace-relative-one element has minimal polynomial y²+y+K.\n    y=next(v for v in range(2**n) if f.pow(v,2**k)^v==1)\n    cols=embed+[f.mul(v,y) for v in embed];inv=inverse_columns(cols)\n    kval=apply(inv,f.square(y)^y);assert kval<2**k\n    return tuple(inv),tuple(cols),kval\n\n\ndef square_columns(n):return [field(n).square(1<<i) for i in range(n)]\n\n\ndef _norm_xor(b,x,z,kval):\n    """x already in tower coordinates. Norm = a²+ab+K b²."""\n    k=len(z);f=field(k);a=x[:k];c=x[k:]\n    linear_xor(b,a,z,square_columns(k));linear_xor(b,c,z,[f.mul(kval,v) for v in square_columns(k)])\n    mul_xor(b,a,c,z)\n\n\ndef _phase_norm(b,x,mask,kval):\n    k=len(mask);f=field(k);a=x[:k];c=x[k:]\n    phase_linear(b,a,mask,square_columns(k));phase_linear(b,c,mask,[f.mul(kval,v) for v in square_columns(k)])\n    phase_product(b,a,c,mask)\n\n\ndef _erase_norm(b,x,z,kval):\n    mask=b.alloc_bits(len(z));b.hmr(z,mask);_phase_norm(b,x,mask,kval);b.free(mask)\n\n\n@lru_cache(None)\ndef inverse3_anf():\n    f=field(3);a=[0 if x==0 else f.invert(x) for x in range(8)]\n    for i in range(3):\n        for s in range(8):\n            if s>>i&1:a[s]^=a[s^(1<<i)]\n    assert a[7]==0\n    return tuple(a)\n\n\ndef inverse(b,x,target=None):\n    """Clean out-of-place inverse: I3=3,I6=21,I12=66,I24=192 Toffolis.\n\n    Target must be initially zero when passed. Allocating target externally can\n    increase peak width: omit target for just-in-time recursive allocation.\n    """\n    n=len(x)\n    if n==3:\n        if target is None:target=b.alloc_qubits(3)\n        anc=b.alloc_qubits(1)[0]\n        for mask,v in enumerate(inverse3_anf()):\n            ix=list(bits(mask))\n            if len(ix)==1:\n                for j in bits(v):b.cx(x[ix[0]],target[j])\n            elif len(ix)==2:\n                b.init_and(x[ix[0]],x[ix[1]],anc)\n                for j in bits(v):b.cx(anc,target[j])\n                b.del_and(x[ix[0]],x[ix[1]],anc)\n        b.free(anc);return target\n    k=n//2;to,frm,K=tower(n)\n    linear_inplace(b,x,to)\n    norm=b.alloc_qubits(k);_norm_xor(b,x,norm,K)\n    invnorm=inverse(b,norm)\n    _erase_norm(b,x,norm,K);b.free(norm)\n    if target is None:target=b.alloc_qubits(n)\n    # x conjugate is (a+b,b), modify temporarily in-place.\n    a=x[:k];c=x[k:];b.cx(c,a)\n    mul_xor(b,a,invnorm,target[:k]);mul_xor(b,c,invnorm,target[k:]);b.cx(c,a)\n    # norm(target) = invnorm, including zero.\n    _erase_norm(b,target,invnorm,K);b.free(invnorm)\n    linear_inplace(b,x,frm);linear_inplace(b,target,frm)\n    return target\n\n\ndef inverse_inplace_forward(b,x):\n    """Forward 192T (n24), classical n-bit deferred input phase transcript."""\n    y=inverse(b,x);mask=b.alloc_bits(len(x));b.hmr(x,mask);b.swap(x,y);b.free(y)\n    return mask\n\n\ndef erase_inverse(b,x,y):\n    """Erase y=inv(x) with a closed phase-only inverse recomputation.\n\n    Generic fallback costs Ierase3=0, Ierase6=9, Ierase12=45,\n    Ierase24=153. It is NOT the claimed 30T inverse12 erasure.\n    """\n    n=len(x)\n    if n==3:\n        mask=b.alloc_bits(3);b.hmr(y,mask)\n        for im,v in enumerate(inverse3_anf()):\n            ix=list(bits(im))\n            for j in bits(v):\n                if len(ix)==1:b.cz(x[ix[0]],mask[j])\n                elif len(ix)==2:b.ccz(x[ix[0]],x[ix[1]],mask[j])\n        b.free(mask);return\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to);linear_inplace(b,y,to)\n    norm=b.alloc_qubits(k);_norm_xor(b,x,norm,K);invnorm=inverse(b,norm)\n    mask=b.alloc_bits(n);b.hmr(y,mask)\n    b.cx(x[k:],x[:k]);phase_product(b,x[:k],invnorm,mask[:k]);phase_product(b,x[k:],invnorm,mask[k:]);b.cx(x[k:],x[:k]);b.free(mask)\n    erase_inverse(b,norm,invnorm);b.free(invnorm)\n    _erase_norm(b,x,norm,K);b.free(norm);linear_inplace(b,x,frm)\n\n\ndef inverse_inplace_reverse_fallback(b,x,transcript):\n    """Close forward transcript; exact345T reverse at degree24, width61.\n\n    Kept as a correctness reference, NOT competitive with the target222T.\n    """\n    y=inverse(b,x)\n    b.cz(y,transcript);b.free(transcript)\n    erase_inverse(b,y,x)\n    b.swap(x,y);b.free(y)\n\n\ndef _inverse_keep_norm(b,x):\n    """Initialized inverse with its half-size input norm kept quantum-live.\n\n    Cost unchanged from inverse(). Caller must eventually erase kept norm.\n    """\n    n=len(x);k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    norm=b.alloc_qubits(k);_norm_xor(b,x,norm,K)\n    invnorm=inverse(b,norm)\n    target=b.alloc_qubits(n)\n    b.cx(x[k:],x[:k]);mul_xor(b,x[:k],invnorm,target[:k]);mul_xor(b,x[k:],invnorm,target[k:]);b.cx(x[k:],x[:k])\n    _erase_norm(b,target,invnorm,K);b.free(invnorm)\n    linear_inplace(b,x,frm);linear_inplace(b,target,frm)\n    return target,norm\n\n\ndef _erase_inverse_with_norm(b,x,y,norm_y):\n    """Erase y=inv(x) when norm(y) is retained. Zero Toffolis.\n\n    y=conjugate(x)*norm(y), so the phase fix is bilinear.\n    Both x and y arrive in polynomial basis; norm_y in half-field basis.\n    """\n    n=len(x);k=n//2;to,frm,K=tower(n)\n    linear_inplace(b,x,to);linear_inplace(b,y,to)\n    b.cx(x[k:],x[:k]);erase_product(b,x[:k],norm_y,y[:k]);erase_product(b,x[k:],norm_y,y[k:]);b.cx(x[k:],x[:k])\n    linear_inplace(b,x,frm)\n\n\ndef inverse_inplace_reverse(b,x,transcript):\n    """Close inverse_inplace_forward: degree24 reverse216T, peak67 qubits.\n\n    Retain norm(norm(input))6 during inverse12. Stream the two output halves\n    so retained N12+t12+D6 add at most43 ancillas to input24. The final D\n    cleanup costs norm(t)15 + inverse6 phase9 =24 additional Toffolis.\n    """\n    n=len(x);assert n in (12,24)\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    N=b.alloc_qubits(k);_norm_xor(b,x,N,K)\n    t,D=_inverse_keep_norm(b,N)\n    outb=b.alloc_qubits(k);mul_xor(b,x[k:],t,outb)\n    erase_product(b,outb,N,x[k:])\n    mul_xor(b,x[:k],t,x[k:]);b.cx(outb,x[k:])\n    b.cx(outb,x[k:]);erase_product(b,x[k:],N,x[:k]);b.cx(outb,x[k:])\n    b.swap(x[:k],x[k:]);b.swap(x[k:],outb);b.free(outb)\n    # N=inv(t), and norm(N)=D was retained inside inverse(N).\n    _erase_inverse_with_norm(b,t,N,D);b.free(N)\n    tok,frmk,Kk=tower(k);linear_inplace(b,t,tok)\n    u=b.alloc_qubits(k//2);_norm_xor(b,t,u,Kk)\n    erase_inverse(b,u,D);b.free(D)\n    _erase_norm(b,t,u,Kk);b.free(u);linear_inplace(b,t,frmk)\n    _erase_norm(b,x,t,K);b.free(t)\n    linear_inplace(b,x,frm)\n    b.cz(x,transcript);b.free(transcript)\n\n\n@lru_cache(None)\ndef extension_columns(K=None):\n    """Map base12 pair a+b*y to standard GF24 polynomial basis.\n\n    Uses base GF12 modulus0x1053. K is any GF12 trace-one element, y²+y=K.\n    Omitting K picks the smallest trace-one word, as in BRIEF.md.\n    Return (to_polynomial_columns, from_polynomial_columns, K).\n    """\n    f=field(12)\n    def trace(v):\n        out=0\n        for i in range(12):out^=f.frobenius(v,i)\n        return out\n    if K is None:K=next(v for v in range(4096) if trace(v)==1)\n    if not 0<=K<4096 or trace(K)!=1:raise ValueError(\'K must have absolute trace1\')\n    _,frm,K0=tower(24)\n    c=next(v for v in range(4096) if f.square(v)^v==K^K0)\n    F=field(24);y=frm[12]^apply(frm[:12],c)\n    cols=tuple(frm[:12])+tuple(F.mul(v,y) for v in frm[:12])\n    return cols,tuple(inverse_columns(cols)),K\n\n\ndef inverse24_inplace_forward(b,x,K=None):\n    """192T forward inversion in explicit a+b*y coordinates; returns transcript."""\n    cols,invcols,_=extension_columns(K);linear_inplace(b,x,cols)\n    transcript=inverse_inplace_forward(b,x);linear_inplace(b,x,invcols)\n    return transcript\n\n\ndef inverse24_inplace_reverse(b,x,transcript,K=None):\n    """216T closed reverse,43 workspace, in explicit a+b*y coordinates."""\n    cols,invcols,_=extension_columns(K);linear_inplace(b,x,cols)\n    inverse_inplace_reverse(b,x,transcript);linear_inplace(b,x,invcols)\n\n\ndef _solve_output_vectors(rows,rank):\n    piv={}\n    for row,rhs in rows:\n        while row:\n            p=(row&-row).bit_length()-1\n            if p in piv:row^=piv[p][0];rhs^=piv[p][1]\n            else:piv[p]=(row,rhs);break\n        if row==0:assert rhs==0,\'tensor does not span function\'\n    out=[0]*rank\n    for p,(row,rhs) in sorted(piv.items(),reverse=True):\n        for j in bits(row^(1<<p)):rhs^=out[j]\n        out[p]=rhs\n    return out\n\n\n@lru_cache(None)\ndef cube4_tensor():\n    """Four ANDs plus linear gates for GF16 cube."""\n    f=field(4);pairs=[(i,j) for i in range(4) for j in range(i+1,4)]\n    quads=[sum(((f.pow((1<<i)^(1<<j),3)^f.pow(1<<i,3)^f.pow(1<<j,3))>>k&1)<<p for p,(i,j) in enumerate(pairs)) for k in range(4)]\n    span={0}\n    for q in quads:span|={v^q for v in list(span)}\n    selected=[];piv={}\n    for a in range(1,16):\n        for c in range(a+1,16):\n            q=sum((((a>>i&1)&(c>>j&1))^((a>>j&1)&(c>>i&1)))<<p for p,(i,j) in enumerate(pairs))\n            if q not in span:continue\n            rem=q\n            for p,v in sorted(piv.items()):\n                if rem>>p&1:rem^=v\n            if rem:\n                p=(rem&-rem).bit_length()-1;piv[p]=rem;selected.append((a,c))\n    assert len(selected)==4\n    rows=[]\n    for i,j in pairs:\n        row=sum((((a>>i&1)&(c>>j&1))^((a>>j&1)&(c>>i&1)))<<p for p,(a,c) in enumerate(selected))\n        rhs=f.pow((1<<i)^(1<<j),3)^f.pow(1<<i,3)^f.pow(1<<j,3);rows.append((row,rhs))\n    out=_solve_output_vectors(rows,len(selected))\n    return tuple((a,c,v) for (a,c),v in zip(selected,out))\n\n\n@lru_cache(None)\ndef cube12_tensor():\n    """Seven GF16 cubic evaluations, each four ANDs: rank28 quadratic map."""\n    f=field(12);sub=field(4);gen=f.pow(f.primitive_element,273)\n    def ev(v,p):\n        out=0\n        for i in reversed(range(5)):out=f.mul(out,p)^((v>>i)&1)\n        return out\n    root=next(f.pow(gen,i) for i in range(1,16) if ev(sub.modulus,f.pow(gen,i))==0)\n    embed=[f.pow(root,i) for i in range(4)]\n    coords=inverse_columns([f.mul(v,f.pow(f.primitive_element,j)) for j in range(3) for v in embed])\n    terms=[]\n    for point in range(7):\n        vals=[]\n        for co in coords:\n            v=0\n            for j in reversed(range(3)):v=sub.mul(v,point)^((co>>(4*j))&15)\n            vals.append(v)\n        for a,c,_ in cube4_tensor():\n            A=sum(((v&a).bit_count()%2)<<i for i,v in enumerate(vals));C=sum(((v&c).bit_count()%2)<<i for i,v in enumerate(vals));terms.append((A,C))\n    rows=[]\n    for i in range(12):\n        for j in range(i+1,12):\n            row=sum((((a>>i&1)&(c>>j&1))^((a>>j&1)&(c>>i&1)))<<p for p,(a,c) in enumerate(terms))\n            rhs=f.pow((1<<i)^(1<<j),3)^f.pow(1<<i,3)^f.pow(1<<j,3);rows.append((row,rhs))\n    out=_solve_output_vectors(rows,len(terms));assert all(out)\n    linear=[]\n    for i in range(12):\n        v=f.pow(1<<i,3)\n        for (a,c),o in zip(terms,out):\n            if a>>i&1 and c>>i&1:v^=o\n        linear.append(v)\n    return tuple((a,c,v) for (a,c),v in zip(terms,out)),tuple(linear)\n\n\ndef cube_xor(b,x,z):\n    """GF12 cube accumulator z ^=x³,28Toffolis and3workspace."""\n    assert len(x)==len(z)==12\n    terms,linear=cube12_tensor();linear_xor(b,x,z,linear)\n    a,c,p=b.alloc_qubits(3)\n    for A,C,v in terms:\n        parity_into(b,x,A,a);parity_into(b,x,C,c);b.init_and(a,c,p)\n        for i in bits(v):b.cx(p,z[i])\n        b.del_and(a,c,p);parity_into(b,x,C,c);parity_into(b,x,A,a)\n    b.free([a,c,p])\n\n\n\n@lru_cache(None)\ndef tensor24_extension(K=None):\n    """105rank multiplicationtensor in explicit a+b*y, y²+y=K basis."""\n    cols,invcols,_=extension_columns(K)\n    return tuple((sum(((m&v).bit_count()%2)<<i for i,v in enumerate(cols)),apply(invcols,out)) for m,out in tensor(24))\n\n\ndef mul24_xor105(b,x,y,z,K=None):\n    """Arbitrarytarget GF24 product in explicit quadraticbasis:105T,1scratch.\n\n    No default126T multiplicationcaller ischanged; this is anoptional newAPI.\n    Allthree24-qubit registers mustbedisjoint.\n    """\n    assert len(x)==len(y)==len(z)==24\n    assert not (set(map(str,x))&set(map(str,y)) or set(map(str,x))&set(map(str,z)) or set(map(str,y))&set(map(str,z)))\n    p=b.alloc_qubits(1)[0]\n    for mask,v in tensor24_extension(K):\n        ix=list(bits(mask));pivot=ix[0];a=x[pivot];c=y[pivot]\n        for i in ix[1:]:b.cx(x[i],a);b.cx(y[i],c)\n        b.init_and(a,c,p)\n        for i in bits(v):b.cx(p,z[i])\n        b.del_and(a,c,p)\n        for i in reversed(ix[1:]):b.cx(y[i],c);b.cx(x[i],a)\n    b.free(p)\n\n\ndef mul_xor0(b,x,y,z):\n    """Standardpolynomialbasis arbitrarytarget product withZEROworkspace.\n\n    Disjointx,y,z required. Conjugateeachoutputrankvectorintoonepivot,\n    applyCCXdirectlythere,thenundooutputconjugation. Same42/105Tcounts.\n    """\n    n=len(x);assert len(y)==len(z)==n\n    assert not(set(map(str,x))&set(map(str,y)) or set(map(str,x))&set(map(str,z)) or set(map(str,y))&set(map(str,z)))\n    for mask,v in tensor(n):\n        ix=list(bits(mask));iz=list(bits(v));p=ix[0];t=iz[0]\n        for i in ix[1:]:b.cx(x[i],x[p]);b.cx(y[i],y[p])\n        for j in iz[1:]:b.cx(z[t],z[j])\n        b.ccx(x[p],y[p],z[t])\n        for j in reversed(iz[1:]):b.cx(z[t],z[j])\n        for i in reversed(ix[1:]):b.cx(y[i],y[p]);b.cx(x[i],x[p])\n\n\ndef mul24_xor105_zero(b,x,y,z,K=None):\n    """Explicita+b*y-basis105T multiplication withZEROprivatequbits."""\n    assert len(x)==len(y)==len(z)==24\n    assert not(set(map(str,x))&set(map(str,y)) or set(map(str,x))&set(map(str,z)) or set(map(str,y))&set(map(str,z)))\n    for mask,v in tensor24_extension(K):\n        ix=list(bits(mask));iz=list(bits(v));p=ix[0];t=iz[0]\n        for i in ix[1:]:b.cx(x[i],x[p]);b.cx(y[i],y[p])\n        for j in iz[1:]:b.cx(z[t],z[j])\n        b.ccx(x[p],y[p],z[t])\n        for j in reversed(iz[1:]):b.cx(z[t],z[j])\n        for i in reversed(ix[1:]):b.cx(y[i],y[p]);b.cx(x[i],x[p])\n', 'tasks/dqi/fused/source/exp/arithmetic/lowwidth.py': '"""Streamed towerinversion and25-scratch preserved-factor division candidates."""\nimport field as ar\nfrom field import *\nfrom division import mul24_xor,phase_product24,multiply_forward\n\n\ndef classical_transpose(b,mask,columns):\n    out=b.alloc_bits(len(columns));b.bit_store(out,False)\n    for i,c in enumerate(columns):\n        for j in bits(c):b.cx(mask[j],out[i])\n    return out\n\n\ndef inverse_stream_forward(b,x):\n    """Initialized in-place inverse with deferredoldinputphase, W(n)=n+1.\n\n    T3=3,T6=21,T12=66,T24=192. Transcriptisintheoriginalpolynomialbasis.\n    Allinternalnorm transcriptsarefixedquadratically beforeparentinputchanges.\n    """\n    n=len(x)\n    if n==3:\n        y=inverse(b,x);mask=b.alloc_bits(n);b.hmr(x,mask);b.swap(x,y);b.free(y);return mask\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    N=b.alloc_qubits(k);ar._norm_xor(b,x,N,K)\n    nm=inverse_stream_forward(b,N);ar._phase_norm(b,x,nm,K);b.free(nm)\n    outb=b.alloc_qubits(k);mul_xor(b,x[k:],N,outb)\n    raw=b.alloc_bits(n);b.hmr(x[k:],raw[k:])\n    mul_xor(b,x[:k],N,x[k:]);b.cx(outb,x[k:]);b.hmr(x[:k],raw[:k])\n    b.swap(x[:k],x[k:]);b.swap(x[k:],outb);b.free(outb)\n    ar._erase_norm(b,x,N,K);b.free(N)\n    linear_inplace(b,x,frm)\n    mask=classical_transpose(b,raw,to);b.free(raw);return mask\n\n\ndef phase_product_quantum(b,x,y,mask):\n    """Phase dot(mask,x*y) with QUANTUMmask: rank(n)CCZ,0scratch."""\n    n=len(x);assert len(y)==len(mask)==n\n    for m,v in tensor(n):\n        ix=list(bits(m));iz=list(bits(v));i=ix[0];j=iz[0]\n        for k in ix[1:]:b.cx(x[k],x[i]);b.cx(y[k],y[i])\n        for k in iz[1:]:b.cx(mask[k],mask[j])\n        b.ccz(x[i],y[i],mask[j])\n        for k in reversed(iz[1:]):b.cx(mask[k],mask[j])\n        for k in reversed(ix[1:]):b.cx(y[k],y[i]);b.cx(x[k],x[i])\n\n\ndef phase_inverse_generic6(b,x,mask):\n    """9T inversephase on6inputbits,7scratch; maskclassical6."""\n    to,frm,K=tower(6);linear_inplace(b,x,to)\n    mt=classical_transpose(b,mask,frm)\n    N=b.alloc_qubits(3);ar._norm_xor(b,x,N,K);u=inverse(b,N)\n    b.cx(x[3:],x[:3]);phase_product(b,x[:3],u,mt[:3]);phase_product(b,x[3:],u,mt[3:]);b.cx(x[3:],x[:3])\n    erase_inverse(b,N,u);b.free(u);ar._erase_norm(b,x,N,K);b.free(N)\n    linear_inplace(b,x,frm);b.free(mt)\n\n\ndef phase_inverse_low12(b,x,mask):\n    """60T phaseofinverse12 using13scratch beyondinput12,maskclassical."""\n    to,frm,K=tower(12);linear_inplace(b,x,to)\n    mt=classical_transpose(b,mask,frm)\n    N=b.alloc_qubits(6);ar._norm_xor(b,x,N,K)\n    nm=inverse_stream_forward(b,N);ar._phase_norm(b,x,nm,K);b.free(nm)\n    b.cx(x[6:],x[:6]);phase_product(b,x[:6],N,mt[:6]);phase_product(b,x[6:],N,mt[6:]);b.cx(x[6:],x[:6])\n    um=b.alloc_bits(6);b.hmr(N,um);b.free(N)\n    N=b.alloc_qubits(6);ar._norm_xor(b,x,N,K);phase_inverse_generic6(b,N,um);b.free(um)\n    ar._erase_norm(b,x,N,K);b.free(N);linear_inplace(b,x,frm);b.free(mt)\n\n\ndef fix_scaled_half_masks(b,head,factor,masks,K):\n    """Phase sum_j dot(maskj,headj*norm(factor)):42CCZ,12scratch.\n\n    Thetwoheadhalves shareonequantum12-bittransposedmask, hence42not84CCZ.\n    """\n    L=b.alloc_qubits(12)\n    def coefficient_mask():\n        for i in range(12):\n            for half in range(2):\n                for j in range(12):\n                    for r in bits(field(12).mul(1<<i,1<<j)):\n                        b.ccx(masks[half][r],head[12*half+j],L[i])\n    coefficient_mask()\n    phase_linear(b,factor[:12],L,square_columns(12))\n    phase_linear(b,factor[12:],L,[field(12).mul(K,v) for v in square_columns(12)])\n    phase_product_quantum(b,factor[:12],factor[12:],L)\n    coefficient_mask();b.free(L)\n\n\ndef divide_lowwidth(b,x,y,K=None):\n    """441T division x/=y,preservingy,25workspace; promise x0wheny0."""\n    if K is None:K=extension_columns()[2]\n    b.cx(y[12:],y[:12]);oldmask=multiply_forward(b,x,y,K);b.cx(y[12:],y[:12])\n    u=b.alloc_qubits(12);ar._norm_xor(b,y,u,K)\n    nm=inverse_stream_forward(b,u);ar._phase_norm(b,y,nm,K);b.free(nm)\n    masks=[]\n    for offset in(0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor(b,half,u,out)\n        m=b.alloc_bits(12);b.hmr(half,m);masks.append(m);b.swap(half,out);b.free(out)\n    fix_scaled_half_masks(b,x,y,masks,K)\n    for m in masks:b.free(m)\n    um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    N=b.alloc_qubits(12);ar._norm_xor(b,y,N,K);phase_inverse_low12(b,N,um);b.free(um)\n    ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n\n\ndef phase_inverse36(b,x,mask):\n    """Inverse12phase:36Toffolis,13scratch beyondinput12.\n\n    Norm6cost15; nestednorm3+inverse3cost9; twoGF3trilinearphasescost12CCZ.\n    Theirthirdquantumcontroliscomputedoneparityatatimefromxandclassicalmask,\n    so no6-qubittransposed-maskbank isneeded.\n    """\n    to12,frm12,K12=tower(12);linear_inplace(b,x,to12)\n    mt=classical_transpose(b,mask,frm12)\n    D=b.alloc_qubits(6);ar._norm_xor(b,x,D,K12)\n    to6,frm6,K6=tower(6);linear_inplace(b,D,to6)\n    E=b.alloc_qubits(3);ar._norm_xor(b,D,E,K6);w=inverse(b,E)\n    anc=b.alloc_qubits(1)[0];f6=field(6)\n    def coefficient(h):\n        for j in range(6):\n            for r in bits(f6.mul(h,1<<j)):\n                b.ccx(mt[r],x[j],anc);b.ccx(mt[r],x[j+6],anc);b.ccx(mt[r+6],x[j+6],anc)\n    for segment in(0,1):\n        if segment==0:b.cx(D[3:],D[:3])\n        controls=D[:3] if segment==0 else D[3:]\n        for m,v in tensor(3):\n            h=apply(frm6,v<<(3*segment));ix=list(bits(m));p=ix[0]\n            coefficient(h)\n            for j in ix[1:]:b.cx(controls[j],controls[p]);b.cx(w[j],w[p])\n            b.ccz(controls[p],w[p],anc)\n            for j in reversed(ix[1:]):b.cx(w[j],w[p]);b.cx(controls[j],controls[p])\n            coefficient(h)\n        if segment==0:b.cx(D[3:],D[:3])\n    b.free(anc);erase_inverse(b,E,w);b.free(w)\n    ar._erase_norm(b,D,E,K6);b.free(E);linear_inplace(b,D,frm6)\n    ar._erase_norm(b,x,D,K12);b.free(D);linear_inplace(b,x,frm12);b.free(mt)\n\n\ndef divide_lowwidth375(b,x,y,K=None):\n    """375T preserved-factor division at25scratch,promisedx0wheny0."""\n    if K is None:K=extension_columns()[2]\n    b.cx(y[12:],y[:12]);oldmask=multiply_forward(b,x,y,K);b.cx(y[12:],y[:12])\n    u=b.alloc_qubits(12);ar._norm_xor(b,y,u,K)\n    nm=inverse_stream_forward(b,u);ar._phase_norm(b,y,nm,K);b.free(nm)\n    masks=[]\n    for offset in(0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor(b,half,u,out)\n        m=b.alloc_bits(12);b.hmr(half,m);masks.append(m);b.swap(half,out);b.free(out)\n    um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    N=b.alloc_qubits(12);ar._norm_xor(b,y,N,K)\n    for offset,m in zip((0,12),masks):phase_product(b,x[offset:offset+12],N,m);b.free(m)\n    phase_inverse36(b,N,um);b.free(um)\n    ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n', 'tasks/dqi/fused/source/exp/confluent_extract/kernel.py': '"""Private fixed Hermite map and injectable singleton-first jet adapter.\n\nNo stable decoder modules are edited. Pass the arithmetic teammate\'s jet\nforward/reverse kernels; both operate on two24-bit data registers and return\nor consume classical transcripts, with no retained quantum cache.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.decoder.common import combine,cx,scalepair,linear_words,K\nfrom exp.inverse12_escape.joint import inverse24_forward190,inverse24_reverse209\ntry:\n    from .model import constants,plan\nexcept ImportError:\n    from model import constants,plan\n\ndef initial(b,words,reverse=False):\n    assert len(words)%2==0\n    linear_words(b,words,constants(len(words)//2)[3],reverse)\n\ndef translate(b,words,head,target,c):\n    a,z=words[2*head:2*head+2]\n    b.cx(a,words[target]);cx(b,z,words[target],c);b.cx(z,words[target+1]);b.cx(z,words[target+2])\n\ndef extract(b,words,jet_forward,jet_reverse,reverse=False,transcript=None,\n            inverse_forward=inverse24_forward190,inverse_reverse=inverse24_reverse209):\n    """Return/consume (kind,token) history; output head pairs alias input bank.\n\n    jet_forward(b,u24,d24,K) -> token\n    jet_reverse(b,u24,d24,token,K) restores both data and closes all phases.\n    """\n    h=len(words)//2;history=[]if not reverse else list(transcript)\n    for op in (reversed(plan(h))if reverse else plan(h)):\n        if op[\'kind\']==\'coordinates\':cx(b,words[2*op[\'head\']+1],words[2*op[\'head\']],op[\'c\']);continue\n        if op[\'kind\']==\'local\':\n            i=2*op[\'head\'];j=i+2;x=combine(words[j:j+2])\n            if not reverse:\n                b.cx(words[i+1],words[j]);history.append((\'local\',inverse_forward(b,x,K)))\n                cx(b,words[j+1],words[j],op[\'c\'])\n            else:\n                cx(b,words[j+1],words[j],op[\'c\']);kind,t=history.pop();assert kind==\'local\'\n                inverse_reverse(b,x,t,K);b.cx(words[i+1],words[j])\n            continue\n        j=op[\'target\'];u=combine(words[j:j+2]);d=combine(words[j+2:j+4]);q=op[\'q\']\n        if not reverse:\n            translate(b,words,op[\'head\'],j,op[\'c\']);history.append((\'jet\',jet_forward(b,u,d,K)))\n            scalepair(b,d,q);b.cx(u,d);scalepair(b,u,q)\n        else:\n            from exp.decoder.common import inv\n            scalepair(b,u,inv(q));b.cx(u,d);scalepair(b,d,inv(q))\n            kind,t=history.pop();assert kind==\'jet\';jet_reverse(b,u,d,t,K)\n            translate(b,words,op[\'head\'],j,op[\'c\'])\n    if reverse:assert not history\n    return history\n', 'tasks/dqi/fused/source/exp/confluent_extract/model.py': '"""Singleton-first confluent quadratic-node map and zero-safe jet EXTRACT.\n\nAll polynomial/rational references are classical. No decoder/verifier claim.\n"""\nfrom functools import lru_cache\nfrom pathlib import Path\nimport sys,json,random,time\nROOT=Path(__file__).resolve().parents[2]\nsys.path.insert(0,str(ROOT/\'playbook/exp/cf_check\'))\nfrom audit import Field\nimport audit as cf_audit\nf=Field(12,0x1053);assert f.K==2048\n\ndef eadd(a,b):return a^b\ndef esquare(a):return f.emul(a,a)\ndef ehead(a,b,c):return (a^f.mul(b,c))|(b<<12)\ndef coords(u,c):\n    b=u>>12;return (u&4095)^f.mul(b,c),b\ndef get(bank,j):return bank[j]|(bank[j+1]<<12)\ndef put(bank,j,x):bank[j]=x&4095;bank[j+1]=x>>12\n\ndef synthetic(poly,gamma):\n    assert len(poly)>=2\n    Q=[0]*(len(poly)-1);Q[-1]=poly[-1]\n    for i in reversed(range(len(Q)-1)):Q[i]=poly[i+1]^f.emul(gamma,Q[i+1])\n    return Q,poly[0]^f.emul(gamma,Q[0])\n\n@lru_cache(None)\ndef constants(h=35):\n    assert h%2 and h>=1;d=(h-1)//2\n    groups=[dict(c=0,kappa=f.K,gamma=4096,head=0,word=0,double=False)]\n    headcs=[0]\n    for j in range(1,d+1):\n        c=2*j;groups.append(dict(c=c,kappa=f.mul(c,c)^c^f.K,gamma=c|4096,head=2*j-1,word=4*j-2,double=True));headcs.extend([c,c])\n    kappas=[f.mul(c,c)^c^f.K for c in headcs];P=[1]\n    for kap in kappas:P=f.pmul(P,[kap,1,1])\n    matrix=[]\n    for group in groups:\n        Q,rem=synthetic(P,group[\'gamma\']);assert rem==0\n        matrix.append([q&4095 for q in Q]);matrix.append([q>>12 for q in Q])\n        if group[\'double\']:\n            QQ,rem=synthetic(Q,group[\'gamma\']);assert rem==0\n            QQ=QQ+[0]\n            matrix.append([q&4095 for q in QQ]);matrix.append([q>>12 for q in QQ])\n    assert len(matrix)==2*h and all(len(r)==2*h for r in matrix)\n    return tuple(headcs),tuple(kappas),tuple(P),tuple(tuple(r)for r in matrix),tuple(groups)\n\n@lru_cache(None)\ndef inverse_matrix(h=35):\n    M=constants(h)[3];n=len(M);a=[list(row)+[int(i==j)for j in range(n)]for i,row in enumerate(M)]\n    for j in range(n):\n        p=next(i for i in range(j,n)if a[i][j]);a[j],a[p]=a[p],a[j]\n        iv=f.inv(a[j][j]);a[j]=[f.mul(v,iv)for v in a[j]]\n        for i in range(n):\n            if i!=j and a[i][j]:\n                q=a[i][j];a[i]=[x^f.mul(q,y)for x,y in zip(a[i],a[j])]\n    assert all(a[i][j]==int(i==j)for i in range(n)for j in range(n))\n    return tuple(tuple(row[n:])for row in a)\n\ndef matrix_apply(M,s):\n    out=[]\n    for row in M:\n        v=0\n        for a,x in zip(row,s):v^=f.mul(a,x)\n        out.append(v)\n    return out\n\ndef initial(s,h=35,reverse=False):return matrix_apply(inverse_matrix(h)if reverse else constants(h)[3],s)\n\n@lru_cache(None)\ndef plan(h=35):\n    groups=constants(h)[4];ops=[]\n    for i,g in enumerate(groups):\n        ops.append(dict(kind=\'coordinates\',head=g[\'head\'],c=g[\'c\']))\n        for target in groups[i+1:]:ops.append(dict(kind=\'jet\',head=g[\'head\'],target=target[\'word\'],c=target[\'c\'],q=g[\'kappa\']^target[\'kappa\']))\n        if g[\'double\']:\n            ops.append(dict(kind=\'local\',head=g[\'head\'],c=g[\'c\']))\n            for target in groups[i+1:]:ops.append(dict(kind=\'jet\',head=g[\'head\']+1,target=target[\'word\'],c=target[\'c\'],q=g[\'kappa\']^target[\'kappa\']))\n    assert sum(op[\'kind\']==\'jet\'for op in ops)==((h-1)//2)**2\n    assert sum(op[\'kind\']==\'local\'for op in ops)==(h-1)//2\n    return tuple(ops)\n\ndef J(u,v):\n    z=int(u==0);ui=f.einv(u);return ui,f.emul(v,esquare(ui^z))\n\ndef extract(bank,h=35,reverse=False,events=None):\n    bank=list(bank);assert len(bank)==2*h\n    for op in (reversed(plan(h))if reverse else plan(h)):\n        if op[\'kind\']==\'coordinates\':bank[2*op[\'head\']]^=f.mul(op[\'c\'],bank[2*op[\'head\']+1]);continue\n        if op[\'kind\']==\'local\':\n            i=2*op[\'head\'];j=i+2\n            if not reverse:\n                u=get(bank,j)^bank[i+1]\n                if events is not None and not u:events.append(dict(kind=\'local_zero_inverse\',head=op[\'head\']+1))\n                a,b=coords(f.einv(u),op[\'c\']);bank[j:j+2]=[a,b]\n            else:\n                a,b=bank[j:j+2];put(bank,j,f.einv(ehead(a,b,op[\'c\']))^bank[i+1])\n            continue\n        i=2*op[\'head\'];j=op[\'target\'];a,b=bank[i:i+2];u=get(bank,j);v=get(bank,j+2);q=op[\'q\'];assert q\n        if not reverse:\n            u^=ehead(a,b,op[\'c\']);v^=b\n            if events is not None and not u:events.append(dict(kind=\'jet_zero_value\',head=op[\'head\'],target=j//2,derivative_nonzero=bool(v)))\n            u,v=J(u,v);v=u^f.escale(v,q);u=f.escale(u,q)\n        else:\n            u=f.escale(u,f.inv(q));v=f.escale(v^u,f.inv(q));u,v=J(u,v);u^=ehead(a,b,op[\'c\']);v^=b\n        put(bank,j,u);put(bank,j+2,v)\n    return bank\n\ndef pair_from_residues(roots,weights):\n    D=[1]\n    for root in roots:D=f.pmul(D,[root,1])\n    N=[0]\n    for root,w in zip(roots,weights):\n        Q,rem=f.pdiv(D,[root,1]);assert rem==[0];N=f.add(N,f.scale(Q,w))\n    return N,D\n\ndef instance(h,roots,weights):\n    P=constants(h)[2];syndrome=[0]*(2*h);errors=[]\n    for root,w in zip(roots,weights):\n        pb=f.ev(P,root);assert pb and not pb>>12\n        start=f.div(w,pb);errors.append(f.div(start,root));v=start\n        for j in range(2*h):syndrome[j]^=v;v=f.mul(v,root)\n    N,D=pair_from_residues(roots,weights)\n    return dict(h=h,roots=roots,residues=weights,errors=errors,syndrome=syndrome,N=N,D=D)\n\ndef direct_jets(roots,weights,h=35):\n    out=[]\n    for group in constants(h)[4]:\n        u=v=0\n        for root,w in zip(roots,weights):\n            t=f.einv(group[\'gamma\']^root);u^=f.escale(t,w);v^=f.escale(esquare(t),w)\n        out.extend([u&4095,u>>12])\n        if group[\'double\']:out.extend([v&4095,v>>12])\n    return out\n\ndef polynomial_oracle(N,D,h=35):\n    cs,ks,*_=constants(h);heads=[];term=None\n    for i,(c,kap)in enumerate(zip(cs,ks)):\n        gamma=c|4096;de=f.ev(D,gamma)\n        if not de:return dict(heads=heads,termination=term,pole=i)\n        u=f.emul(f.ev(N,gamma),f.einv(de));a,b=coords(u,c);heads.extend([a,b])\n        H=f.add(N,f.pmul([a,b],D));Q,rem=f.pdiv(H,[kap,1,1]);assert rem==[0]\n        if H==[0]:term=i;heads.extend([0]*(2*h-len(heads)));break\n        N,D=D,Q\n    return dict(heads=heads,termination=term,pole=None)\n\ndef trial(case):\n    h=case[\'h\'];s=case[\'syndrome\'];jet=initial(s,h);assert jet==direct_jets(case[\'roots\'],case[\'residues\'],h)\n    assert initial(jet,h,True)==s\n    events=[];heads=extract(jet,h,events=events);assert extract(heads,h,True)==jet\n    reference=polynomial_oracle(case[\'N\'],case[\'D\'],h)\n    match=heads==reference[\'heads\']if reference[\'pole\']is None else None\n    raw_ratio=None;raw_padding=None;raw_squarefree=None\n    if reference[\'pole\']is None:\n        assert match\n        pairs=[heads[i:i+2]for i in range(0,len(heads),2)]\n        A,B,tail=cf_audit.synth(f,constants(h)[1],pairs)\n        raw_ratio=f.pmul(A,case[\'D\'])==f.pmul(B,case[\'N\']);assert raw_ratio\n        expected=case[\'D\'][:]\n        for i in range(len(case[\'roots\'])+1,h,2):expected=f.pmul(expected,[constants(h)[1][i],1,1])\n        if not len(case[\'roots\'])%2:expected=[0]+expected\n        raw_padding=B==expected;assert raw_padding\n        raw_squarefree=f.gcd(B,f.derivative(B))==[1];assert raw_squarefree\n        assert len(B)==h+1 and B[-1]==1\n    else:assert heads[:len(reference[\'heads\'])]==reference[\'heads\']\n    return dict(h=h,weight=len(case[\'roots\']),oracle_pole=reference[\'pole\'],termination=reference[\'termination\'],\n        heads_match=match,jet_zero_events=sum(e[\'kind\']==\'jet_zero_value\'for e in events),\n        nonzero_derivative_zero_events=sum(e[\'kind\']==\'jet_zero_value\'and e[\'derivative_nonzero\']for e in events),\n        local_zero_inverses=sum(e[\'kind\']==\'local_zero_inverse\'for e in events),roundtrip=True,\n        raw_synthesis_ratio=raw_ratio,raw_synthesis_padding=raw_padding,raw_synthesis_squarefree=raw_squarefree)\n\ndef main(repeats=4):\n    start=time.time();rng=random.Random(202610270901);tests=[];arbitrary=[]\n    for h in[3,9,35]:\n        inverse_matrix(h)\n        for weight in range(h+1):\n            for repeat in range(repeats):\n                roots=rng.sample(range(1,4096),weight);weights=[rng.randrange(1,4096)for _ in roots]\n                tests.append(trial(instance(h,roots,weights)))\n        for j in range(300):\n            bank=[rng.randrange(4096)for _ in range(2*h)]\n            if j%3==0:\n                for group in constants(h)[4][1:]:bank[group[\'word\']]=bank[group[\'word\']+1]=0\n            if j%3==1:bank=[0 if rng.randrange(2)else v for v in bank]\n            assert extract(extract(bank,h),h,True)==bank\n        arbitrary.append(dict(h=h,cases=300,includes_zero_values_and_zero_jets=True))\n        print(\'h\',h,\'done\',flush=True)\n    peer=json.loads((ROOT/\'exp/confluent_synthesis/fixtures.json\').read_text());compared=0\n    for fixture in peer:\n        h=fixture[\'h\'];case=instance(h,fixture[\'roots\'],fixture[\'weights\']);heads=extract(initial(case[\'syndrome\'],h),h)\n        assert heads==[v for head in fixture[\'heads\']for v in head];compared+=1\n        A,B,_=cf_audit.synth(f,constants(h)[1],[heads[i:i+2]for i in range(0,len(heads),2)])\n        assert A==fixture[\'rawA\']and B==fixture[\'rawB\']\n    result=dict(tests=tests,arbitrary=arbitrary,peer_fixture_heads_compared=compared,\n        matrix_ranks={h:2*h for h in[3,9,35]},counts={h:dict(jets=sum(o[\'kind\']==\'jet\'for o in plan(h)),local_inverses=sum(o[\'kind\']==\'local\'for o in plan(h)))for h in[3,9,35]},\n        seconds=time.time()-start,meaning=\'Classical exactness and fixed-linear map; no full decoder/verifier\')\n    print(json.dumps(dict(cases=len(tests),heads_bad=sum(t[\'heads_match\']is False for t in tests),poles=sum(t[\'oracle_pole\']is not None for t in tests),\n        peer_fixture_heads_compared=compared,arbitrary=arbitrary,seconds=result[\'seconds\']),indent=2))\n    Path(__file__).with_name(\'results.json\').write_text(json.dumps(result,indent=2)+\'\\n\')\n\nif __name__==\'__main__\':main()\n', 'tasks/dqi/fused/source/exp/confluent_synthesis/kernel.py': '"""Read-only constants adapter for frozen alternating-gauge synthesis."""\nfrom pathlib import Path\nimport sys\nfrom functools import lru_cache\nfrom types import FunctionType,SimpleNamespace\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.alternating_gauge import kernel as base\nfrom exp.decoder.common import mul,K,pmul\n@lru_cache(None)\ndef constants(h=35):\n assert h>=3 and h%2==1\n cs=[0]+[2*j for j in range(1,(h+1)//2)for _ in range(2)]\n ks=[mul(c,c)^c^K for c in cs];P=[1]\n for k in ks:P=pmul(P,[k,1,1])\n return cs,ks,P,()\nold=SimpleNamespace(**dict(vars(base.old),constants=constants))\nsynthesize=FunctionType(base.synthesize.__code__,dict(base.synthesize.__globals__,old=old),\'synthesize\',base.synthesize.__defaults__,base.synthesize.__closure__)\nunsynthesize=FunctionType(base.unsynthesize.__code__,dict(base.unsynthesize.__globals__,old=old),\'unsynthesize\',base.unsynthesize.__defaults__,base.unsynthesize.__closure__)\n', 'tasks/dqi/fused/source/exp/convert/convert.py': '"""Reversible fixed-generic-chart half-degree CONVERT.\n\nPublic convert(builder,A[35] words,B[35] words,reverse=False).\nA,B hold GF12 polynomial coefficients; B35=1 implicit.\nSafe divisors make roundtrip exact even on degree gaps; forward decode is only\nproved for generic Euclid degrees. Runtime gap2 routing is not yet implemented.\n"""\nfrom contextlib import contextmanager\nimport sys\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nfrom kernels import ar,cx,convolution_xor,hankel\nfrom lowwidth import phase_inverse36\n\ndef zero_flag(b,a):\n    for q in a:b.x(q)\n    nodes=[];left=a[0]\n    for q in a[1:]:\n        out=b.alloc_qubits(1)[0];b.init_and(left,q,out);nodes.append((left,q,out));left=out\n    for x,y,z in reversed(nodes[:-1]):b.del_and(x,y,z);b.free(z)\n    for q in a:b.x(q)\n    return nodes[-1][2]\n\ndef erase_zero_flag(b,a,flag):\n    for q in a:b.x(q)\n    nodes=[];left=a[0]\n    for q in a[1:-1]:\n        out=b.alloc_qubits(1)[0];b.init_and(left,q,out);nodes.append((left,q,out));left=out\n    b.del_and(left,a[-1],flag);b.free(flag)\n    for x,y,z in reversed(nodes):b.del_and(x,y,z);b.free(z)\n    for q in a:b.x(q)\n\n@contextmanager\ndef inverse_context(b,a):\n    flag=zero_flag(b,a);b.cx(flag,a[0])\n    ai=ar.inverse(b,a)\n    yield ai\n    mask=b.alloc_bits(12);b.hmr(ai,mask);b.free(ai)\n    phase_inverse36(b,a,mask);b.free(mask)\n    b.cx(flag,a[0]);erase_zero_flag(b,a,flag)\n\ndef scale(b,x,a,ai,divide=True):\n    out=b.alloc_qubits(12);ar.mul_xor(b,x,ai if divide else a,out)\n    ar.erase_product(b,a if divide else ai,out,x)\n    b.swap(x,out);b.free(out)\n\ndef runops(b,ops,a,ai,undo=False):\n    for op in (reversed(ops) if undo else ops):\n        if op[0]==\'s\':scale(b,op[1],a,ai,op[2]^undo)\n        elif op[0]==\'m\':ar.mul_xor(b,op[1],op[2],op[3])\n        elif op[0]==\'c\':convolution_xor(b,op[1],op[2],op[3],undo)\n        else:raise ValueError(op[0])\n\ndef pre(b,f,g,payloads,undo=False):\n    m=len(f)-1;n=m-1;a=g[-1];q0=f[-2]\n    with inverse_context(b,a) as ai:\n        q1=ai if f[-1] is None else f[-1]\n        ops=[(\'s\',v[-1],True) for v in payloads]\n        ops += [(\'m\',g[i],v[-1],v[i]) for v in payloads for i in range(n)]\n        if f[-1] is not None:ops += [(\'s\',q1,True)]\n        ops += [(\'m\',q1,g[-2],q0),(\'s\',q0,True),(\'c\',g[:-1],[q0,q1],f[:-2])]\n        runops(b,ops,a,ai,undo)\n\ndef post(b,f,g,payloads,undo=False):\n    n=len(g)-1;a=g[-1];q0=f[-2]\n    def dots():\n        for v in payloads:\n            for i in range(n):ar.mul_xor(b,g[i],v[i],v[-1])\n    if f[-1] is None:\n        with inverse_context(b,a) as ai:\n            q1=ai\n            ops=[(\'s\',v[-1],False) for v in payloads]\n            ops += [(\'m\',g[i],v[i],v[-1]) for v in payloads for i in range(n)]\n            ops += [(\'s\',v[-1],True) for v in payloads]\n            ops += [(\'c\',g[:-1],[q0,q1],f[:-2]),(\'s\',q0,False),(\'m\',q1,g[-2],q0)]\n            runops(b,ops,a,ai,undo)\n    else:\n        q1=f[-1]\n        def qscale(reverse):\n            with inverse_context(b,q1) as qi:\n                for v in payloads:scale(b,v[-1],q1,qi,not reverse)\n        def last(reverse):\n            with inverse_context(b,a) as ai:\n                ops=[(\'s\',v[-1],True) for v in payloads]\n                ops += [(\'c\',g[:-1],[q0,q1],f[:-2]),(\'s\',q0,False),(\'m\',q1,g[-2],q0),(\'s\',q1,False)]\n                runops(b,ops,a,ai,reverse)\n        if not undo:qscale(False);dots();last(False)\n        else:last(True);dots();qscale(True)\n\ndef raw_solve(b,f,g,payloads,reverse=False):\n    m=len(f)-1\n    assert len(g)==m and all(len(v)==m for v in payloads)\n    if m==1:\n        divisors=[p for p in (f[-1],g[-1]) if p is not None]\n        for a in (reversed(divisors) if reverse else divisors):\n            with inverse_context(b,a) as ai:\n                for v in payloads:scale(b,v[0],a,ai,not reverse)\n        return\n    low=[v[:-1] for v in payloads]\n    if not reverse:\n        pre(b,f,g,payloads);raw_solve(b,g,f[:-2],low);post(b,f,g,payloads)\n    else:\n        post(b,f,g,payloads,True);raw_solve(b,g,f[:-2],low,True);pre(b,f,g,payloads,True)\n\ndef bez_solve(b,O,E,payloads,reverse=False):\n    def reduce():\n        for i in range(len(O)):ar.mul_xor(b,E[-1],O[i],E[i])\n    reduce();raw_solve(b,O+[None],E[:-1],payloads,reverse);reduce()\n\ndef frob(b,words,power):\n    f=ar.field(12);cols=[f.pow(1<<i,power) for i in range(12)]\n    for x in words:ar.linear_inplace(b,x,cols)\n\ndef convert(b,A,B,reverse=False):\n    assert len(A)==len(B)==35\n    E=B[::2];O=B[1::2];u=A[1::2];w=A[:34:2];top=A[-1]\n    if reverse:hankel(b,B,A,True)\n    frob(b,B,2048);frob(b,A,2048)\n    if reverse:\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n        for i in range(17):ar.mul_xor(b,O[i],u[i],top)\n        bez_solve(b,O,E,[u,w],True)\n        for i in range(17):ar.mul_xor(b,top,O[i],w[i])\n    else:\n        for i in range(17):ar.mul_xor(b,top,O[i],w[i])\n        bez_solve(b,O,E,[u,w])\n        for i in range(17):ar.mul_xor(b,O[i],u[i],top)\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n    frob(b,A,2);frob(b,B,2)\n    if not reverse:hankel(b,B,A)\n\ndef raw_solve_blocked(b,f,g,payloads,block_size=2,reverse=False):\n    """Closed checkpointed scalar schedule grouped into quotient blocks.\n\n    This is the no-speedup control construction: at most2*block_size quotient\n    metadata words remain in already consumed f/g coefficients; no matrix or\n    evaluation bank is materialized. Product fusion is deliberately absent.\n    """\n    frames=[]\n    for _ in range(block_size):\n        if len(f)==2:break\n        frames.append((f,g,payloads))\n        if not reverse:pre(b,f,g,payloads)\n        else:post(b,f,g,payloads,True)\n        m=len(f)-1;f,g=g,f[:-2];payloads=[v[:-1]for v in payloads]\n    if len(f)==2:raw_solve(b,f,g,payloads,reverse)\n    else:raw_solve_blocked(b,f,g,payloads,block_size,reverse)\n    for f,g,payloads in reversed(frames):\n        if not reverse:post(b,f,g,payloads)\n        else:pre(b,f,g,payloads,True)\n', 'tasks/dqi/fused/source/exp/convert/hankel_check.py': '"""Independent GF(2^12) rank-one, in-place Hankel audit; no verifier."""\nimport random, json\nfrom functools import lru_cache\nMOD=0x1053\n\ndef slowmul(a,b):\n    c=0\n    for _ in range(12):\n        if b&1:c^=a\n        b>>=1;a<<=1\n        if a&4096:a^=MOD\n    return c\nEXP=[1]*8190;LOG=[-1]*4096\nfor i in range(1,4095):EXP[i]=slowmul(EXP[i-1],2)\nassert len(set(EXP[:4095]))==4095\nfor i in range(4095):LOG[EXP[i]]=i;EXP[i+4095]=EXP[i]\ndef mul(a,b):return EXP[LOG[a]+LOG[b]] if a and b else 0\ndef inv(a):assert a;return EXP[4095-LOG[a]]\ndef power(a,n):return EXP[(LOG[a]*n)%4095] if a else int(n==0)\ndef dot(a,b):\n    out=0\n    for x,y in zip(a,b):out^=mul(x,y)\n    return out\n\ndef mi(a):\n    n=len(a);a=[r[:]+[int(i==j) for j in range(n)] for i,r in enumerate(a)]\n    for j in range(n):\n        p=next(i for i in range(j,n) if a[i][j]);a[p],a[j]=a[j],a[p]\n        t=inv(a[j][j]);a[j]=[mul(t,v) for v in a[j]]\n        for i in range(n):\n            if i!=j:\n                t=a[i][j];a[i]=[x^mul(t,y) for x,y in zip(a[i],a[j])]\n    return [r[n:] for r in a]\n\n@lru_cache(None)\ndef terms(k,m):\n    n=k+m-1;points=list(range(n));V=[[power(t,j) for j in range(n)] for t in points];W=mi(V)\n    # Cross = sum_t source(reverse(low),V[t]) * coefficient(kernel,W[:,t]) * V[t,:m]\n    return [(V[t][:k][::-1],[W[j][t] for j in range(n)],V[t][:m]) for t in range(n)]\n\ndef cross(q,low,high):\n    count=0\n    for source,coef,target in terms(len(low),len(high)):\n        p=mul(dot(source,low),dot(coef,q));count+=1\n        for i,c in enumerate(target):high[i]^=mul(c,p)\n    return count\n\ndef tri(q,x,reverse=False):\n    n=len(x)\n    if n==1:return 0\n    k=n//2;m=n-k;lo=x[:k];hi=x[k:]\n    if not reverse:\n        count=tri(q[:m],hi);count+=cross(q[1:n],lo,hi);count+=tri(q[:k],lo)\n    else:\n        count=tri(q[:k],lo,True);count+=cross(q[1:n],lo,hi);count+=tri(q[:m],hi,True)\n    x[:]=lo+hi\n    return count\n\ndef main():\n    rng=random.Random(102617147);tests=0;rank_counts={}\n    for n in (2,3,4,8,16,17,35):\n        for _ in range(100):\n            q=[1]+[rng.randrange(4096) for _ in range(n-1)];x=[rng.randrange(4096) for _ in range(n)];orig=x[:]\n            target=[dot(q[:i+1],orig[i::-1]) for i in range(n)]\n            c=tri(q,x);assert x==target\n            cr=tri(q,x,True);assert x==orig and c==cr\n            rank_counts[n]=c;tests+=1\n    # H_ij=B_(i+j+1), coefficients B_35=1, higher 0. H=J*T with q_s=B_(35-s).\n    for _ in range(100):\n        b=[rng.randrange(4096) for _ in range(35)]+[1];x=[rng.randrange(4096) for _ in range(35)];old=x[:]\n        expected=[dot((b[i+j+1] if i+j+1<=35 else 0 for j in range(35)),old) for i in range(35)]\n        c=tri(list(reversed(b[1:])),x);x.reverse();assert x==expected and c==147\n        x.reverse();tri(list(reversed(b[1:])),x,True);assert x==old\n    out={\'GF12_modulus\':hex(MOD),\'triangular_forward_reverse_cases\':tests,\'hankel35_forward_reverse_cases\':100,\'products\':rank_counts,\'H35_forward_T\':147*42,\'H35_pair_T\':294*42,\'extra_words\':0,\'inversions\':0,\'measurement_phase_corrections\':0}\n    print(json.dumps(out,indent=2))\nif __name__==\'__main__\':main()\n', 'tasks/dqi/fused/source/exp/convert/kernels.py': '"""In-place structured GF12 kernels using exp/arithmetic primitives."""\nimport sys\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/\'arithmetic\'))\nimport field as ar\nfrom hankel_check import mul,inv,mi,power\nfrom functools import lru_cache\n\ndef ccols(c):return [mul(c,1<<i) for i in range(12)]\ndef cx(b,x,y,c=1):\n    if c==1:b.cx(x,y)\n    elif c:ar.linear_xor(b,x,y,ccols(c))\ndef scale_const(b,x,c):\n    if c!=1:ar.linear_inplace(b,x,ccols(c))\n\ndef expose(b,words,coeffs):\n    p=next(i for i,c in enumerate(coeffs) if c);c=coeffs[p]\n    scale_const(b,words[p],c)\n    for i,a in enumerate(coeffs):\n        if i!=p:cx(b,words[i],words[p],a)\n    return words[p],p\n\ndef unexpose(b,words,coeffs,p):\n    for i in reversed(range(len(coeffs))):\n        if i!=p:cx(b,words[i],words[p],coeffs[i])\n    scale_const(b,words[p],inv(coeffs[p]))\n\ndef rank_update(b,aw,ac,bw,bc,tw,tc):\n    x,ap=expose(b,aw,ac);y,bp=expose(b,bw,bc)\n    p=next(i for i,c in enumerate(tc) if c);v=tc[p];vi=inv(v)\n    for i,c in enumerate(tc):\n        if i!=p:cx(b,tw[p],tw[i],mul(c,vi))\n    scale_const(b,tw[p],vi)\n    ar.mul_xor(b,x,y,tw[p])\n    scale_const(b,tw[p],v)\n    for i in reversed(range(len(tc))):\n        if i!=p:cx(b,tw[p],tw[i],mul(tc[i],vi))\n    unexpose(b,bw,bc,bp);unexpose(b,aw,ac,ap)\n\n@lru_cache(None)\ndef vandermonde(n):\n    V=[[power(t,j) for j in range(n)] for t in range(n)]\n    return V,mi(V)\n\n@lru_cache(None)\ndef cross_terms(k,m):\n    n=k+m-1;V,W=vandermonde(n)\n    return [(V[t][:k][::-1],[W[j][t] for j in range(n)],V[t][:m]) for t in range(n)]\n\ndef tri(b,q,x,reverse=False):\n    # q=(q1,...,q_(n-1)), q0=1 implicit.\n    n=len(x)\n    if n==1:return\n    k=n//2;m=n-k\n    if not reverse:tri(b,q[:m-1],x[k:])\n    else:tri(b,q[:k-1],x[:k],True)\n    ts=cross_terms(k,m)\n    for ac,bc,tc in (reversed(ts) if reverse else ts):rank_update(b,x[:k],ac,q[:n-1],bc,x[k:],tc)\n    if not reverse:tri(b,q[:k-1],x[:k])\n    else:tri(b,q[:m-1],x[k:],True)\n\ndef hankel(b,B,A,reverse=False):\n    # H_ij=B_(i+j+1), implicit monic B_degree=len(A).\n    assert len(B)==len(A)\n    q=list(reversed(B[1:]))\n    if reverse:\n        for i in range(len(A)//2):b.swap(A[i],A[-1-i])\n        tri(b,q,A,True)\n    else:\n        tri(b,q,A)\n        for i in range(len(A)//2):b.swap(A[i],A[-1-i])\n\ndef convolution_xor(b,a,c,out,reverse=False):\n    # Coefficients 0..len(out)-1 of a*c; supports short output truncation.\n    N=len(a)+len(c)-1;V,W=vandermonde(N)\n    terms=range(N-1,-1,-1) if reverse else range(N)\n    for t in terms:\n        tc=[W[j][t] for j in range(len(out))]\n        if any(tc):rank_update(b,a,V[t][:len(a)],c,V[t][:len(c)],out,tc)\n\ndef convolution_transpose_left_xor(b,source,c,out,reverse=False):\n    # out[j] += sum_i source[i]*c[i-j], transpose of outpoly*c truncated to source.\n    N=len(out)+len(c)-1;V,W=vandermonde(N)\n    terms=range(N-1,-1,-1) if reverse else range(N)\n    for t in terms:\n        ac=[W[i][t] for i in range(len(source))]\n        if any(ac):rank_update(b,source,ac,c,V[t][:len(c)],out,V[t][:len(out)])\n', 'tasks/dqi/fused/source/exp/convert/repair.py': '"""Adaptive one-gap Barnett repair for raw Bezout solve.\n\nSeparate fromdefaultgenericconvert.py. Firstleading-zeroisrepairedbymultiplygbyx\nandprecompanionMx. More/largegapsandrepairwithf(0)=0decodeincorrectlybut\nforward/reversearestilllocallyphase-correctpermutations. No deferredtranscripts.\n"""\nfrom contextlib import contextmanager\nfrom convert import ar,zero_flag,erase_zero_flag,scale,pre,post,raw_solve,frob\nfrom lowwidth import phase_inverse36\nfrom kernels import hankel\n\n@contextmanager\ndef nonzero_inverse(b,a):\n    ai=ar.inverse(b,a)\n    yield ai\n    mask=b.alloc_bits(12);b.hmr(ai,mask);b.free(ai);phase_inverse36(b,a,mask);b.free(mask)\n\n@contextmanager\ndef safe_word(b,a):\n    flag=zero_flag(b,a);b.cx(flag,a[0])\n    yield\n    b.cx(flag,a[0]);erase_zero_flag(b,a,flag)\n\ndef rotate(b,v,F,reverse=False):\n    for j in (range(1,len(v)) if reverse else range(len(v)-1,0,-1)):\n        b.cswap(F,v[j],v[j-1])\n\ndef coefficient_update(b,f,u,target):\n    if f[-1] is None:b.x(f[0][0])\n    else:b.cx(f[-1],f[0])\n    ar.mul_xor(b,f[0],u,target)\n    if f[-1] is None:b.x(f[0][0])\n    else:b.cx(f[-1],f[0])\n\ndef companion(b,f,payloads,F,reverse=False):\n    """Conditionallymultiplyeachpayloadbyx modf, oritsinverse.\n\n    f0 andleadingaresafenedonlyduringthispermutation, soalwaysinvertible.\n    Masku=F*lowest/bbeforeupdates; afterupdatesu=F*lowest/f0.\n    """\n    lead=f[-1]\n    @contextmanager\n    def lead_safe():\n        if lead is None:yield\n        else:\n            with safe_word(b,lead):yield\n    @contextmanager\n    def invctx(a):\n        if a is None:yield None\n        else:\n            with nonzero_inverse(b,a) as ai:yield ai\n    with safe_word(b,f[0]),lead_safe():\n        if not reverse:\n            for v in payloads:rotate(b,v,F)\n        masks=[]\n        divisor=f[0] if reverse else lead\n        with invctx(divisor) as di:\n            for v in payloads:\n                u=b.alloc_qubits(12)\n                for j in range(12):b.init_and(F,v[0][j],u[j])\n                if divisor is not None:scale(b,u,divisor,di,True)\n                if not reverse:\n                    for i in range(1,len(v)):ar.mul_xor(b,f[i],u,v[i])\n                    coefficient_update(b,f,u,v[0])\n                else:\n                    coefficient_update(b,f,u,v[0])\n                    for i in reversed(range(1,len(v))):ar.mul_xor(b,f[i],u,v[i])\n                mask=b.alloc_bits(12);b.hmr(u,mask);b.free(u);masks.append(mask)\n        recovery=lead if reverse else f[0]\n        with invctx(recovery) as ri:\n            for v,mask in zip(payloads,masks):\n                if recovery is None:\n                    for j in range(12):b.ccz(F,v[0][j],mask[j])\n                else:\n                    tmp=b.alloc_qubits(12);ar.mul_xor(b,v[0],ri,tmp)\n                    for j in range(12):b.ccz(F,tmp[j],mask[j])\n                    ar.erase_product(b,v[0],ri,tmp);b.free(tmp)\n                b.free(mask)\n        if reverse:\n            for v in payloads:rotate(b,v,F,True)\n\ndef equal_code(b,H,code):\n    for i,q in enumerate(H):\n        if code>>i&1:b.x(q)\n    flag=zero_flag(b,H)\n    for i,q in enumerate(H):\n        if code>>i&1:b.x(q)\n    return flag\n\ndef erase_equal(b,H,code,flag):\n    for i,q in enumerate(H):\n        if code>>i&1:b.x(q)\n    erase_zero_flag(b,H,flag)\n    for i,q in enumerate(H):\n        if code>>i&1:b.x(q)\n\ndef history_arm(b,H,a,code):\n    hz=zero_flag(b,H);az=zero_flag(b,a);F=b.alloc_qubits(1)[0];b.ccx(hz,az,F)\n    erase_zero_flag(b,a,az);erase_zero_flag(b,H,hz)\n    for i,q in enumerate(H):\n        if code>>i&1:b.cx(F,q)\n    return F\n\ndef history_clear_flag(b,H,code,F):\n    eq=equal_code(b,H,code);b.cx(eq,F);erase_equal(b,H,code,eq);b.free(F)\n\ndef history_finish(b,H,a,code,F):\n    for i,q in enumerate(H):\n        if code>>i&1:b.cx(F,q)\n    hz=zero_flag(b,H);az=zero_flag(b,a);b.ccx(hz,az,F)\n    erase_zero_flag(b,a,az);erase_zero_flag(b,H,hz);b.free(F)\n\ndef repair_solve(b,f,g,payloads,H,reverse=False):\n    m=len(f)-1\n    if m==1:return raw_solve(b,f,g,payloads,reverse)\n    F=history_arm(b,H,g[-1],m)\n    rotate(b,g,F)\n    if not reverse:companion(b,f,payloads,F)\n    history_clear_flag(b,H,m,F)\n    low=[v[:-1]for v in payloads]\n    if not reverse:\n        pre(b,f,g,payloads);repair_solve(b,g,f[:-2],low,H);post(b,f,g,payloads)\n    else:\n        post(b,f,g,payloads,True);repair_solve(b,g,f[:-2],low,H,True);pre(b,f,g,payloads,True)\n    F=equal_code(b,H,m)\n    if reverse:companion(b,f,payloads,F,True)\n    rotate(b,g,F,True)\n    history_finish(b,H,g[-1],m,F)\n\ndef bez_repair(b,O,E,payloads,reverse=False):\n    def reduce():\n        for i in range(len(O)):ar.mul_xor(b,E[-1],O[i],E[i])\n    reduce();H=list(b.alloc_qubits(5));repair_solve(b,O+[None],E[:-1],payloads,H,reverse);b.free(H);reduce()\n\ndef convert_repair(b,A,B,reverse=False):\n    assert len(A)==len(B)==35\n    E=B[::2];O=B[1::2];u=A[1::2];w=A[:34:2];top=A[-1]\n    if reverse:hankel(b,B,A,True)\n    frob(b,B,2048);frob(b,A,2048)\n    if reverse:\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n        for i in range(17):ar.mul_xor(b,O[i],u[i],top)\n        bez_repair(b,O,E,[u,w],True)\n        for i in range(17):ar.mul_xor(b,top,O[i],w[i])\n    else:\n        for i in range(17):ar.mul_xor(b,top,O[i],w[i])\n        bez_repair(b,O,E,[u,w])\n        for i in range(17):ar.mul_xor(b,O[i],u[i],top)\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n    frob(b,A,2);frob(b,B,2)\n    if not reverse:hankel(b,B,A)\n', 'tasks/dqi/fused/source/exp/convert_short_hankel/kernel.py': '"""Drop-in guarded CONVERT with exact short-H boundary.\n\nDefault preserves the stable arbitrary borrowed-history completion. Optional\nmonic_root=True requires the supplied history word initially zero; if absent\nit is allocated zero. B/payloads otherwise arbitrary, including all gaps.\nReverse infers the option from its forward phase token.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.safe_scalar import converter as old\n\n\ndef _root_ops(b,ops,a,ai,undo=False):\n for op in (reversed(ops)if undo else ops):\n  if op[0]==\'copy\':b.x(op[1][0]);b.cx(ai,op[1])\n  else:old.runops(b,[op],a,ai,undo)\n\n\ndef _pre_root(b,f,g,payloads,undo=False,incoming=None,defer=False):\n m=len(f)-1;n=m-1;a=g[-1];q0,q1=f[-2:]\n def action(ai):\n  ops=[(\'s\',v[-1],True)for v in payloads]\n  ops +=[(\'m\',g[i],v[-1],v[i])for v in payloads for i in range(n)]\n  ops +=[(\'copy\',q1),(\'m\',q1,g[-2],q0),(\'s\',q0,True),(\'c\',g[:-1],[q0,q1],f[:-2])]\n  _root_ops(b,ops,a,ai,undo)\n return old.inverse_apply(b,a,action,incoming,defer)\n\n\ndef _post_root(b,f,g,payloads,reverse=False,incoming=None,defer=False):\n n=len(g)-1;a=g[-1];q0,q1=f[-2:]\n def action(ai):\n  # q1=1/(a+[a0]), so division by q1 is multiplication by the\n  # already clipped a in this same context, even on scalar-zero charts.\n  ops=[(\'s\',v[-1],False)for v in payloads]\n  ops +=[(\'m\',g[i],v[i],v[-1])for v in payloads for i in range(n)]\n  ops +=[(\'s\',v[-1],True)for v in payloads]\n  ops +=[(\'c\',g[:-1],[q0,q1],f[:-2]),(\'s\',q0,False),(\'m\',q1,g[-2],q0),(\'copy\',q1)]\n  _root_ops(b,ops,a,ai,reverse)\n return old.inverse_apply(b,a,action,incoming,defer)\n\n\ndef _look_root(b,f,g,payloads,reverse,phase):\n m=len(f)-1;F=old.fresh_zero_flag(b,g[-1]);old.rotate(b,g,F)\n if not reverse:carry=_pre_root(b,f,g,payloads,defer=True)\n else:carry=_post_root(b,f,g,payloads,True,phase[\'a\'].pop(m),True)\n old.canonical_permute(b,f,g,F);old.clear_flag(b,g[-1],F)\n parent=([v[-1]for v in payloads],f[-1],f[-2])\n old.look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,parent,reverse,phase)\n F=old.fresh_zero_flag(b,g[-1]);old.canonical_permute(b,f,g,F,True)\n if not reverse:phase[\'a\'][m]=_post_root(b,f,g,payloads,False,carry,True)\n else:_pre_root(b,f,g,payloads,True,carry)\n old.rotate(b,g,F,True);old.clear_flag(b,g[-1],F)\n\n\ndef _bez_monic(b,O,E,payloads,reverse,history_scratch12,phase):\n def reduce():\n  for i in range(len(O)):old.ar.mul_xor0(b,E[-1],O[i],E[i])\n reduce();lead=list(b.alloc_qubits(12))if history_scratch12 is None else list(history_scratch12)\n b.x(lead[0]);_look_root(b,O+[lead],E[:-1],payloads,reverse,phase);b.x(lead[0])\n if history_scratch12 is None:b.free(lead)\n reduce()\n\n\ndef convert_safe(b,A,B,reverse=False,history_scratch12=None,phase_token=None,*,monic_root=False):\n assert len(A)==len(B)==35\n phase={\'a\':{},\'q\':{},\'_short_monic\':bool(monic_root)}if phase_token is None else phase_token\n assert not reverse or phase_token is not None\n use_monic=phase[\'_short_monic\']\n # Frobenius first on BOTH directions: reverse input is the completed C.\n old.frob(b,B,2048);old.frob(b,A,2048)\n E=B[::2];O=B[1::2];u=A[1::2];w=A[:34:2];top=A[-1]\n def bez(rev):\n  if use_monic:_bez_monic(b,O,E,[u,w],rev,history_scratch12,phase)\n  else:old.bez_lookahead(b,O,E,[u,w],rev,history_scratch12,phase)\n if not reverse:\n  for i in range(17):old.ar.mul_xor0(b,top,O[i],w[i])\n  bez(False)\n  for i in range(17):old.ar.mul_xor0(b,E[i+1],w[i],top)\n  for i in range(17):b.swap(A[2*i],A[2*i+1])\n  old.hankel(b,B[1:],A[:34])\n  for i in reversed(range(34)):b.swap(A[i],A[i+1])\n else:\n  for i in range(34):b.swap(A[i],A[i+1])\n  old.hankel(b,B[1:],A[:34],True)\n  for i in range(17):b.swap(A[2*i],A[2*i+1])\n  for i in range(17):old.ar.mul_xor0(b,E[i+1],w[i],top)\n  bez(True)\n  for i in range(17):old.ar.mul_xor0(b,top,O[i],w[i])\n old.frob(b,A,2);old.frob(b,B,2)\n if reverse:assert not phase[\'a\']and not phase[\'q\']\n return phase if not reverse else None\n', 'tasks/dqi/fused/source/exp/convert_short_hankel/phase34.py': '"""Private34T phase substitution; never mutates stable module globals.\n\nSame API as kernel.convert_safe. Saving2T per ultimate inverse closure;\nphase helper21private fits below the existing39-private converter peak.\n"""\nfrom types import FunctionType,SimpleNamespace\nfrom contextlib import contextmanager\nfrom exp.safe_scalar import converter as stable\nfrom exp.fold_cached_norm.phase34 import phase_inverse34\nfrom exp.convert_short_hankel import kernel as short\n\n\ndef clone(fn,scope,name=None):\n out=FunctionType(fn.__code__,scope,name or fn.__name__,fn.__defaults__,fn.__closure__)\n out.__kwdefaults__=None if fn.__kwdefaults__ is None else dict(fn.__kwdefaults__)\n return out\n\n# Erase/inverse_apply_safe are imported closures with different globals;\n# clone the chain explicitly before rebinding the converter namespace.\n_erase=clone(stable.erase,dict(stable.erase.__globals__,phase_inverse36_zero=phase_inverse34))\n_apply=clone(stable.inverse_apply_safe,dict(stable.inverse_apply_safe.__globals__,erase=_erase))\n_scope=dict(vars(stable),erase=_erase,inverse_apply_safe=_apply)\n_qgen=clone(stable.q_inverse.__wrapped__,_scope,\'q_inverse\')\n_scope[\'q_inverse\']=contextmanager(_qgen)\nfor _name in [\'scale\',\'runops\',\'zero_toggle\',\'inverse_apply\',\'pre\',\'canonical_permute\',\'masked_product\',\'fresh_zero_flag\',\'clear_flag\',\'q_post\',\'post_look\',\'base\',\'look_solve\',\'bez_lookahead\']:\n _scope[_name]=clone(getattr(stable,_name),_scope,_name)\n_private=SimpleNamespace(**_scope)\n\n# Clone short-H and optional monic-root functions against that private graph.\n_short_scope=dict(vars(short),old=_private)\nfor _name in [\'_root_ops\',\'_pre_root\',\'_post_root\',\'_look_root\',\'_bez_monic\',\'convert_safe\']:\n _short_scope[_name]=clone(getattr(short,_name),_short_scope,_name)\nconvert_safe=_short_scope[\'convert_safe\']\n', 'tasks/dqi/fused/source/exp/decoder/build.py': '"""Build the integrated decoder with isolated degree-two-gap lookahead."""\nimport argparse,json,time,hashlib,gc\nfrom common import *\nfrom exp.confluent_extract.kernel import initial,extract as extract_jets\nfrom exp.confluent_extract.model import constants as confluent_constants\nfrom exp.jet24.kernel import jet_forward,jet_reverse\n\ndef extract(b,words,reverse=False,transcript=None):\n    return extract_jets(b,words,jet_forward,jet_reverse,reverse,transcript)\nfrom exp.fold_cached_norm.kernel import synthesize,unsynthesize\nfrom exp.mask_projection.kernel import project,restore,SELECTED\nsys.path.insert(0,str(ROOT/\'exp/convert\'))\nfrom convert import convert\nfrom exp.convert_short_hankel.phase34 import convert_safe as convert_historyfree\nsys.path.insert(0,str(ROOT/\'exp/scan\'))\nfrom scan import scan\n\ndef build(path):\n    start=time.time();b=km.CircuitBuilder();S=[b.alloc_qubits(12) for i in range(70)]\n    E=[b.alloc_bits(12) for j in range(4095)]\n    stages=[];oldT=0\n    def snapshot(name):\n        nonlocal oldT\n        c=b.finish_circuit();T=c.operation_statistics().num_toffoli_like\n        r=dict(name=name,T=T-oldT,cumulative_T=T,Q=c.num_qubits,num_ops=len(c),seconds=time.time()-start)\n        stages.append(r);oldT=T;print(json.dumps(r),flush=True);del c;gc.collect()\n    with b.mark(\'mask_projection\'):mask_token=project(b,S,E)\n    snapshot(\'mask_projection\')\n    with b.mark(\'initial\'):initial(b,S)\n    snapshot(\'initial\')\n    with b.mark(\'extract\'):xh=extract(b,S)\n    snapshot(\'extract\')\n    with b.mark(\'synthesis\'):A,B,yh=synthesize(b,S)\n    snapshot(\'synthesis\')\n    history_word=b.alloc_qubits(12)\n    with b.mark(\'convert_lookahead\'):phase_token=convert_historyfree(b,A,B,history_scratch12=history_word,monic_root=True)\n    snapshot(\'convert_lookahead\')\n    with b.mark(\'scan\'):scan(b,A,B,E,skip_points=SELECTED,node_polynomial=confluent_constants()[2])\n    snapshot(\'scan\')\n    with b.mark(\'unconvert_lookahead\'):convert_historyfree(b,A,B,True,history_scratch12=history_word,phase_token=phase_token)\n    b.free(history_word)\n    snapshot(\'unconvert_lookahead\')\n    with b.mark(\'unsynthesis\'):unsynthesize(b,yh)\n    snapshot(\'unsynthesis\')\n    with b.mark(\'unextract\'):extract(b,S,True,xh)\n    snapshot(\'unextract\')\n    with b.mark(\'uninitial\'):initial(b,S,True)\n    snapshot(\'uninitial\')\n    with b.mark(\'restore_masks\'):restore(b,E,mask_token)\n    snapshot(\'restore_masks\')\n    # Named builder registers cannot be temporarily freed. Attach the exact\n    # contract metadata after construction, retaining the original wire IDs.\n    metadata=[]\n    for i in range(70):\n        metadata.append(f\'REGISTER r{i} "S{i}"\')\n        metadata.extend(f\'APPEND_TO_REGISTER q{12*i+k} r{i}\' for k in range(12))\n    for j in range(4095):\n        metadata.append(f\'REGISTER r{70+j} "E{j}"\')\n        metadata.extend(f\'APPEND_TO_REGISTER b{12*j+k} r{70+j}\' for k in range(12))\n    c=km.Circuit(\'\\n\'.join(metadata))+b.finish_circuit();c.to_file(str(path),format=\'kmb\')\n    digest=hashlib.file_digest(open(path,\'rb\'),\'sha256\').hexdigest()\n    result=dict(path=str(path),sha256=digest,T_static=c.operation_statistics().num_toffoli_like,Q=c.num_qubits,num_ops=len(c),num_bits=c.num_bits,stages=stages,seconds=time.time()-start,scope=\'STATIC compiled counts include mutually exclusive classical branches; confluent Hermite extraction, zero-power retained-norm synthesis, short-Hankel monic-root guarded h2 conversion, classical mask projection; no verifier cost claimed\')\n    path.with_suffix(\'.build.json\').write_text(json.dumps(result,indent=2)+\'\\n\');print(json.dumps(result),flush=True)\n    return result\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'output\',type=Path);args=p.parse_args();build(args.output)\n', 'tasks/dqi/fused/source/exp/decoder/common.py': '"""Lead\'s finite-field and reversible linear utilities (GF12 word bank)."""\nfrom functools import lru_cache\nfrom pathlib import Path\nimport sys\nimport kickmix as km\n\nROOT=Path(__file__).resolve().parents[2]\nsys.path.insert(0,str(ROOT))\nsys.path.insert(0,str(ROOT/\'exp/arithmetic\'))\nimport field as ar\ngf=km.GF2Field(12)\nK=2048\n\ndef mul(a,b):return gf.mul(a,b)\ndef inv(a):return gf.invert(a) if a else 0\ndef power(a,n):return gf.pow(a,n)\ndef xor(xs):\n    out=0\n    for x in xs:out^=x\n    return out\ndef trim(p):\n    p=list(p)\n    while len(p)>1 and not p[-1]:p.pop()\n    return p\ndef add(a,b):return trim([(a[i] if i<len(a) else 0)^(b[i] if i<len(b) else 0) for i in range(max(len(a),len(b)))])\ndef pmul(a,b):\n    c=[0]*(len(a)+len(b)-1)\n    for i,x in enumerate(a):\n        for j,y in enumerate(b):c[i+j]^=mul(x,y)\n    return trim(c)\ndef divmodp(a,b):\n    a=trim(a);out=[0]*max(1,len(a)-len(b)+1)\n    assert b[-1]\n    bi=inv(b[-1])\n    for k in range(len(a)-len(b),-1,-1):\n        t=mul(a[k+len(b)-1],bi);out[k]=t\n        for j,v in enumerate(b):a[k+j]^=mul(t,v)\n    return trim(out),trim(a)\ndef emul(x,y,k=K):\n    a,b=x&4095,x>>12;c,d=y&4095,y>>12;bd=mul(b,d)\n    return (mul(a,c)^mul(k,bd))|((mul(a,d)^mul(b,c)^bd)<<12)\ndef einv(x,k=K):\n    a,b=x&4095,x>>12;t=inv(mul(a,a)^mul(a,b)^mul(k,mul(b,b)))\n    return mul(a^b,t)|(mul(b,t)<<12)\ndef eev(p,x):\n    v=0\n    for a in reversed(p):v=emul(v,x)^a\n    return v\n\n@lru_cache(None)\ndef constcols(c):return tuple(mul(c,1<<i) for i in range(12))\ndef cx(b,x,z,c=1):\n    if c==1:b.cx(x,z)\n    elif c:ar.linear_xor(b,x,z,constcols(c))\ndef scale(b,x,c):\n    if c!=1:ar.linear_inplace(b,x,constcols(c))\ndef scalepair(b,x,c):\n    scale(b,x[:12],c);scale(b,x[12:],c)\ndef combine(words):return km.array([q for w in words for q in w])\ndef wordlist(reg):return [km.array(list(reg)[i:i+12]) for i in range(0,len(reg),12)]\n\n@lru_cache(None)\ndef matrix_ops(rows):\n    """Elementary row operations reducing an invertible GF12 matrix to I."""\n    a=[list(r) for r in rows];n=len(a);ops=[]\n    assert all(len(r)==n for r in a)\n    for j in range(n):\n        p=next(i for i in range(j,n) if a[i][j])\n        if p!=j:a[p],a[j]=a[j],a[p];ops.append((\'swap\',p,j,0))\n        v=inv(a[j][j])\n        if v!=1:a[j]=[mul(x,v) for x in a[j]];ops.append((\'scale\',j,0,v))\n        for i in range(n):\n            if i!=j and a[i][j]:\n                v=a[i][j];a[i]=[x^mul(y,v) for x,y in zip(a[i],a[j])];ops.append((\'add\',j,i,v))\n    return tuple(ops)\n\ndef linear_words(b,words,rows,reverse=False):\n    ops=matrix_ops(tuple(tuple(r) for r in rows))\n    for op,i,j,v in (ops if reverse else reversed(ops)):\n        if op==\'swap\':b.swap(words[i],words[j])\n        elif op==\'scale\':scale(b,words[i],v if reverse else inv(v))\n        else:cx(b,words[i],words[j],v)\n\ndef zero_hold(b,x):\n    """Return a held all-zero ladder, len(x)-1 Toffolis."""\n    b.x(x);lad=b.alloc_qubits(len(x)-1)\n    b.init_and(x[0],x[1],lad[0])\n    for j in range(2,len(x)):b.init_and(lad[j-2],x[j],lad[j-1])\n    b.x(x)\n    return lad\ndef zero_free(b,x,lad):\n    b.x(x)\n    for j in reversed(range(2,len(x))):b.del_and(lad[j-2],x[j],lad[j-1])\n    b.del_and(x[0],x[1],lad[0]);b.free(lad);b.x(x)\n\ndef reclaim(b,regs):\n    """Reclaim specified known-free physical wires, without new wire IDs."""\n    wanted={str(q) for reg in regs for q in reg}\n    available=b.alloc_qubits(b.finish_circuit().num_qubits-b.num_allocated_qubits)\n    have={str(q) for q in available}\n    assert wanted<=have,(wanted-have)\n    b.free(km.array([q for q in available if str(q) not in wanted]))\n\ndef extmul_xor(b,x,y,z):\n    """126T product in explicit a+b*y basis; one scratch bit."""\n    a,c=x[:12],x[12:];d,e=y[:12],y[12:]\n    b.cx(z[:12],z[12:]);ar.mul_xor(b,a,d,z[:12]);b.cx(z[:12],z[12:])\n    scale(b,z[:12],inv(K))\n    ar.mul_xor(b,c,e,z[:12])\n    scale(b,z[:12],K)\n    b.cx(c,a);b.cx(e,d);ar.mul_xor(b,a,d,z[12:]);b.cx(e,d);b.cx(c,a)\n\ndef phase_extmul(b,x,y,mask):\n    for i in range(24):\n        for j in range(24):\n            v=emul(1<<i,1<<j)\n            for k in ar.bits(v):b.ccz(x[i],y[j],mask[k])\n\ndef write(s,qs,vals):\n    import numpy as np\n    vals=np.asarray(vals,dtype=np.uint64)\n    s.write_across_shots(km.array(qs),((vals[None,:]>>np.arange(len(qs),dtype=np.uint64)[:,None])&1).astype(bool))\ndef read(s,qs):\n    import numpy as np\n    return (s.read_across_shots(km.array(qs)).astype(np.uint64)*(np.uint64(1)<<np.arange(len(qs),dtype=np.uint64))[:,None]).sum(axis=0)\n', 'tasks/dqi/fused/source/exp/decoder/extract.py': '"""In-place conjugate-node initial map and triangular EXTRACT."""\nfrom functools import lru_cache\nfrom common import *\nfrom exp.inverse12_escape.joint import inverse24_forward190 as inverse24_forward20,inverse24_reverse209 as inverse24_reverse20\n\n@lru_cache(None)\ndef constants(h=35):\n    cs=list(range(0,2*h,2));ks=[mul(c,c)^c^K for c in cs];P=[1]\n    for k in ks:P=pmul(P,[k,1,1])\n    matrix=[]\n    for c in cs:\n        g=c|(1<<12);Q=[0]*(2*h);Q[-1]=1\n        for j in reversed(range(2*h-1)):Q[j]=P[j+1]^emul(g,Q[j+1])\n        assert P[0]^emul(g,Q[0])==0\n        matrix.append(tuple(v&4095 for v in Q));matrix.append(tuple(v>>12 for v in Q))\n    return cs,ks,P,tuple(matrix)\n\ndef initial(b,words,reverse=False):\n    linear_words(b,words,constants(len(words)//2)[3],reverse=reverse)\n\ndef extract(b,words,reverse=False,transcript=None):\n    h=len(words)//2;cs,ks,_,_=constants(h);bank=[combine(words[2*j:2*j+2]) for j in range(h)]\n    if not reverse:\n        history=[]\n        for i in range(h):\n            cx(b,words[2*i+1],words[2*i],cs[i])\n            for j in range(i+1,h):\n                b.cx(words[2*i],words[2*j]);cx(b,words[2*i+1],words[2*j],cs[j]);b.cx(words[2*i+1],words[2*j+1])\n                history.append(inverse24_forward20(b,bank[j],K))\n                scalepair(b,bank[j],ks[i]^ks[j])\n        return history\n    history=list(transcript)\n    for i in reversed(range(h)):\n        for j in reversed(range(i+1,h)):\n            scalepair(b,bank[j],inv(ks[i]^ks[j]))\n            inverse24_reverse20(b,bank[j],history.pop(),K)\n            b.cx(words[2*i+1],words[2*j+1]);cx(b,words[2*i+1],words[2*j],cs[j]);b.cx(words[2*i],words[2*j])\n        cx(b,words[2*i+1],words[2*i],cs[i])\n    assert not history\n', 'tasks/dqi/fused/source/exp/decoder/safe_bridge.py': '"""Zero-safe division by temporarily replacing a zero factor by one."""\nfrom common import *\nimport division as dv\nfrom exp.mul99.schedules import divide_zero_bridge414\n\ndef divide_zero_bridge(b,x,y,extra_zero_mask=None):\n    return divide_zero_bridge414(b,x,y,K=K,extra_zero_mask=extra_zero_mask)\n\nif __name__==\'__main__\':\n    import numpy as np,json\n    rng=np.random.default_rng(18171026);n=16384\n    xv=rng.integers(0,1<<24,n,dtype=np.uint64);yv=rng.integers(0,1<<24,n,dtype=np.uint64);yv[:4096]=0\n    b=km.CircuitBuilder();x=b.alloc_qubits(24);y=b.alloc_qubits(24)\n    old,fm=dv.multiply_zero_forward(b,x,y,K);mm=b.alloc_bits(24);b.cz(x,mm);b.cz(x[2],x[13])\n    divide_zero_bridge(b,x,y,fm);b.free(fm);b.cz(x,old);b.free(old)\n    c=b.finish_circuit();s=km.Simulator(batch_size=n,seed=188182);ms=rng.integers(0,1<<24,n,dtype=np.uint64)\n    write(s,x,xv);write(s,y,yv);write(s,mm,ms);s.do(c)\n    assert np.array_equal(read(s,x),xv) and np.array_equal(read(s,y),yv)\n    z=[emul(int(a),int(bb)) if bb else int(a) for a,bb in zip(xv,yv)]\n    want=np.array([bool((a&int(m)).bit_count()%2)^bool(((a>>2)&1)&((a>>13)&1)) for a,m in zip(z,ms)])\n    assert np.array_equal(want,s.read_phase_flipped_across_shots())\n    for q in range(48,c.num_qubits):assert not s.read_across_shots(km.q(q)).any(),q\n    r=dict(shots=n,pair_T=c.operation_statistics().num_toffoli_like,division_T=c.operation_statistics().num_toffoli_like-152,scratch=c.num_qubits-48,value_errors=0,phase_errors=0)\n    print(json.dumps(r,indent=2));(ROOT/\'exp/decoder/safe_bridge.json\').write_text(json.dumps(r,indent=2)+\'\\n\')\n', 'tasks/dqi/fused/source/exp/decoder/synthesis.py': '"""Head-absorption synthesis with a complete deferred measurement transcript.\n\nThe rational interpolation chart is exact on valid heads without extension\npoles. Zero evaluation factors are replaced by one in the remainder map, so\narbitrary heads are preserved; that repair can change the decoded rational\nfunction after an extension pole. The top-coefficient promises are separate.\n"""\nfrom common import *\nfrom extract import constants\nimport division as dv\nfrom lowwidth import divide_lowwidth375\nsys.path.insert(0,str(ROOT/\'exp/convert\'))\nfrom kernels import rank_update,vandermonde\nimport kernels as ck\nfrom exp.noscratch.kernels import mul_xor_noscratch\nfrom convert import inverse_context\nfrom safe_bridge import divide_zero_bridge\n\ndef tail_recur(b,heads,p,q,d,reverse=False):\n    for a,z in (reversed(heads) if reverse else heads):\n        if reverse:b.swap(p,q);ar.mul_xor(b,z,q,p)\n        ar.mul_xor(b,p,q,d)\n        b.cx(z,a);ar.linear_inplace(b,q,ar.square_columns(12));ar.mul_xor(b,a,q,d)\n        ar.linear_inplace(b,q,[power(1<<j,2048) for j in range(12)]);b.cx(z,a)\n        if not reverse:ar.mul_xor(b,z,q,p);b.swap(p,q)\n\n@lru_cache(None)\ndef quotient_terms(n,kap):\n    """floor(l*A/q): A0 does not contribute; n rank-one products."""\n    V,W=vandermonde(n);out=[]\n    for t in range(n):\n        polynomial=[0]+[W[j][t] for j in range(n)]\n        quotient,_=divmodp(polynomial,[kap,1,1]);quotient+= [0]*(n-1-len(quotient))\n        out.append(([1,t],V[t][:n-1],quotient))\n    return out\n\ndef quotient_xor(b,head,A,B,kap,reverse=False):\n    assert len(B)==len(A)-1\n    terms=quotient_terms(len(A),kap)\n    for ac,bc,tc in (reversed(terms) if reverse else terms):\n        if any(tc):rank_update(b,head,ac,A[1:],bc,B,tc)\n\ndef full_product_xor(b,head,A,target):\n    n=len(A)+1;V,W=vandermonde(n)\n    for t in range(n):rank_update_noanc(b,head,[1,t],A,V[t][:len(A)],target,[W[j][t] for j in range(n)])\n\ndef rank_update_noanc(b,aw,ac,bw,bc,tw,tc):\n    x,ap=ck.expose(b,aw,ac);y,bp=ck.expose(b,bw,bc)\n    p=next(i for i,c in enumerate(tc) if c);v=tc[p];vi=inv(v)\n    for i,c in enumerate(tc):\n        if i!=p:cx(b,tw[p],tw[i],mul(c,vi))\n    scale(b,tw[p],vi);mul_xor_noscratch(b,x,y,tw[p]);scale(b,tw[p],v)\n    for i in reversed(range(len(tc))):\n        if i!=p:cx(b,tw[p],tw[i],mul(tc[i],vi))\n    ck.unexpose(b,bw,bc,bp);ck.unexpose(b,aw,ac,ap)\n\n@lru_cache(None)\ndef embedding_matrix(n,kap):\n    # Input [remainder0,remainder1,quotient0,...] -> q*quotient+remainder.\n    rows=[[0]*n for _ in range(n)];rows[0][0]=1;rows[1][1]=1\n    for j in range(n-2):\n        for k,c in enumerate([kap,1,1]):rows[j+k][j+2]=c\n    return tuple(tuple(r) for r in rows)\n\n@lru_cache(None)\ndef evaluation_coefficients(n,c):\n    g=c|(1<<12);out=[1]\n    for j in range(1,n):out.append(emul(out[-1],g))\n    return tuple(out)\n\ndef expose_eval(b,A,c,reverse=False):\n    es=evaluation_coefficients(len(A),c)\n    if reverse:\n        for i in reversed(range(2,len(A))):cx(b,A[i],A[1],es[i]>>12)\n        for i in reversed(range(2,len(A))):cx(b,A[i],A[0],es[i]&4095)\n        cx(b,A[1],A[0],c)\n    else:\n        cx(b,A[1],A[0],c)\n        for i in range(2,len(A)):cx(b,A[i],A[0],es[i]&4095)\n        for i in range(2,len(A)):cx(b,A[i],A[1],es[i]>>12)\n    return combine(A[:2])\n\ndef absorb(b,head,A,B,c,kap,reverse=False,transcript=None):\n    out=head+B\n    if not reverse:\n        value=expose_eval(b,A,c);flag=dv.zero_flag(b,value);expose_eval(b,A,c,True)\n        # Zero-safe factor: A(gamma)=0 gets factor1. Adding1 to A changes\n        # l*A only in its remainder, so the quotient map is unchanged.\n        b.cx(flag,A[0][0])\n        spare=[b.alloc_qubits(12),b.alloc_qubits(12)];target=spare+B\n        linear_words(b,target,embedding_matrix(len(target),kap))\n        full_product_xor(b,head,A,target)\n        b.cx(flag,A[0][0])\n        transcript=b.alloc_bits(24);b.hmr(combine(head),transcript)\n        for h,z in zip(head,spare):b.swap(h,z);b.free(z)\n        flagmask=b.alloc_bits(1);b.hmr(flag,flagmask[0]);b.free(flag)\n        return out,A,(transcript,flagmask)\n    linear_words(b,out,embedding_matrix(len(out),kap),True)\n    value=expose_eval(b,A,c)\n    cx(b,head[1],head[0],c)\n    headmask,flagmask=transcript\n    divide_zero_bridge(b,combine(head),value,flagmask[0]);b.free(flagmask)\n    cx(b,head[1],head[0],c)\n    b.cz(combine(head),headmask);b.free(headmask)\n    expose_eval(b,A,c,True)\n    quotient_xor(b,head,A,B,kap,True)\n    return out,A,transcript\n\ndef rotate(b,words,control,reverse=False):\n    # multiply by x using a top-zero cyclic register, controlled by even chart.\n    for i in (range(1,len(words)) if reverse else reversed(range(1,len(words)))):\n        b.cswap(control,words[i-1],words[i])\n\ndef flag_zero(b,q,flag):\n    ladder=zero_hold(b,q);b.cx(ladder[-1],flag);zero_free(b,q,ladder)\n\ndef tail_scale_forward(b,p,q,d):\n    with inverse_context(b,q) as t:\n        out=b.alloc_qubits(12);mul_xor_noscratch(b,d,t,out)\n        mask=b.alloc_bits(12);b.hmr(d,mask);b.swap(d,out);b.free(out)\n    return mask\n\ndef tail_scale_reverse(b,p,q,d,mask):\n    with inverse_context(b,q) as t:\n        out=b.alloc_qubits(12);mul_xor_noscratch(b,d,q,out)\n        newer=b.alloc_bits(12);b.hmr(d,newer);b.swap(d,out);b.free(out)\n        ar.phase_product(b,d,t,newer);b.free(newer)\n    b.cz(d,mask);b.free(mask)\n\ndef synthesize(b,words,keep_leading=True):\n    """Return A,B, transcript. Exactly70datawords after forward completion."""\n    h=len(words)//2;assert h%2==1\n    cs,ks,_,_=constants(h);heads=[words[2*i:2*i+2] for i in range(h)]\n    p,q,d=[b.alloc_qubits(12) for _ in range(3)];b.x(q[0])\n    tail_recur(b,heads,p,q,d)\n    dmask=tail_scale_forward(b,p,q,d)\n    # Uniform tail: when q=0,d=0, [d/q,p]/q is x times the old even\n    # chart [p]/0. Every transfer is polynomial-linear, so the required\n    # padding x is already present; no parity bit or outer shift is needed.\n    A=[d,p];B=[q];history=[]\n    for i in reversed(range(h)):\n        oldA,oldB=A,B\n        A,B,m=absorb(b,heads[i],A,B,cs[i],ks[i])\n        history.append((heads[i],oldA,oldB,cs[i],ks[i],m))\n    # Highest A coefficient=0 and highest B coefficient=1 are determinant\n    # identities. The second A coefficient can be nonzero on a bad tail.\n    extra=[A[h],A[h+1],B[h]];b.x(B[h][0])\n    freed=extra[1:] if keep_leading else extra\n    for reg in freed:b.free(reg)\n    transcript=dict(h=h,heads=heads,p=p,q=q,d=d,dmask=dmask,history=history,A=A,B=B,extra=extra,freed=freed,keep_leading=keep_leading)\n    return A[:h],B[:h],transcript\n\ndef unsynthesize(b,transcript):\n    t=transcript;h=t[\'h\'];A=t[\'A\'];B=t[\'B\']\n    reclaim(b,t[\'freed\'])\n    b.x(B[h][0])\n    for head,oldA,oldB,c,kap,mask in reversed(t[\'history\']):absorb(b,head,oldA,oldB,c,kap,True,mask)\n    p,q,d=t[\'p\'],t[\'q\'],t[\'d\']\n    tail_scale_reverse(b,p,q,d,t[\'dmask\'])\n    tail_recur(b,t[\'heads\'],p,q,d,True);b.x(q[0])\n    b.free(p);b.free(q);b.free(d)\n', 'tasks/dqi/fused/source/exp/fold_cached_norm/kernel.py': '"""Spend spare global width on a retained norm inside quadratic fold division."""\nfrom pathlib import Path\nimport sys\nfrom types import FunctionType\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.mul99.schedules import multiply_forward99,phase_inverse36_pivot\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.noscratch.kernels import mul_xor_noscratch\nfrom exp.inverse12_escape.joint import inverse_initialized\nfrom exp.arithmetic.division import phase_product24\nfrom exp.arithmetic.field import erase_product\nfrom exp.zero_padding_synthesis import kernel as prior\nfrom exp.inverse12_escape.zero1 import inverse_zero\nfrom exp.fold_cached_norm.phase34 import phase_inverse34\nfrom exp.inverse12_escape.zero1 import inverse6_zero20\nfrom exp.phase24_tradeoff.phase6small import phase_inverse6_classical6\nfrom exp.alternating_gauge import scalar as scalar_prior\n\ndef divide325(b,x,y,K=None):\n    """x/=y, preserve y,325 executed T/36 private; y!=0 or x=0 promise.\n\n    Inner synthesis explicitly clips its factor before this call. The norm\n    stays live through both base-field output scales, allowing immediate\n    old-half phase repayment and avoiding its42T reconstruction.\n    """\n    assert K is not None\n    b.cx(y[12:],y[:12]);old=multiply_forward99(b,x,y,K);b.cx(y[12:],y[:12])\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,y,N,K);u=inverse_initialized(b,N)\n    for off in (0,12):\n        half=x[off:off+12];out=b.alloc_qubits(12)\n        mul_xor_noscratch(b,half,u,out);erase_product(b,out,N,half)\n        b.swap(half,out);b.free(out)\n    mask=b.alloc_bits(12);b.hmr(u,mask);b.free(u)\n    phase_inverse36_pivot(b,N,mask);b.free(mask)\n    prior.ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,old,K);b.free(old)\n\ndef bind(fn,**changes):\n    return FunctionType(fn.__code__,dict(fn.__globals__,**changes),fn.__name__,fn.__defaults__,fn.__closure__)\n\ndivide323=bind(divide325,phase_inverse36_pivot=phase_inverse34)\ndivide323.__doc__=\'Unit factor division:323T/36private; retained norm and34T/21private inverse phase.\'\n\ndef divide_zero324(b,x,y,K=None,extra_zero_mask=None,after=None):\n    """Identity-at-zero division,324T/49 private, all HMR phases closed."""\n    assert K is not None\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,y,N,K);u,F=inverse_zero(b,N)\n    b.cx(y[12:],y[:12]);b.cx(F,y[0]);old=multiply_forward99(b,x,y,K)\n    b.cx(F,y[0]);b.cx(y[12:],y[:12]);b.cx(F,u[0])\n    for off in (0,12):\n        half=x[off:off+12];out=b.alloc_qubits(12);mul_xor_noscratch(b,half,u,out)\n        b.cx(F,N[0]);erase_product(b,out,N,half);b.cx(F,N[0])\n        b.swap(half,out);b.free(out)\n    b.cx(F,u[0]);um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    b.cx(F,y[0]);phase_product24(b,x,y,old,K);b.cx(F,y[0]);b.free(old)\n    if after is not None:after(F)\n    fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n    if extra_zero_mask is not None:b.cx(extra_zero_mask,fm[0])\n    phase_inverse34(b,N,um,fm[0]);b.free(um);b.free(fm)\n    prior.ar._erase_norm(b,y,N,K);b.free(N)\n\nabsorb=bind(prior.absorb_odd,divide_lowwidth369=divide323)\nabsorb_even=bind(prior.absorb_even,divide_lowwidth369=divide323)\nouter=bind(prior.outer,divide_zero369=divide_zero324)\n\ndef _inverse6_zero(b,x,with_zero):\n    assert with_zero\n    return inverse6_zero20(b,x)\n\ndivide_zero113=bind(scalar_prior.divide_zero117,inverse6_init19=_inverse6_zero,phase_inverse_generic6=phase_inverse6_classical6)\nmultiply_zero141=bind(scalar_prior.multiply_zero144,inverse_zero=inverse_zero,phase_inverse36_zero=phase_inverse34)\nprefix_reverse137=bind(scalar_prior.prefix_reverse141,divide_zero117=divide_zero113)\nodd_decode=bind(prior.synthesize.__globals__[\'odd_decode\'],prefix_reverse141=prefix_reverse137)\nsynthesize=bind(prior.synthesize,absorb=absorb,absorb_even=absorb_even,outer=outer,odd_decode=odd_decode,divide_zero117=divide_zero113)\nunsynthesize=bind(prior.unsynthesize,absorb=absorb,absorb_even=absorb_even,outer=outer,odd_decode=odd_decode,multiply_zero144=multiply_zero141)\n', 'tasks/dqi/fused/source/exp/fold_cached_norm/phase34.py': '"""Use the held I6 DAG with a classical mask:34T,21 private, optional zero."""\nfrom exp.jet24.kernel import phase_inverse12_quantum64\n\ndef phase_inverse34(b,x,mask,zero_mask=None):\n    return phase_inverse12_quantum64(b,x,mask,zero_mask)\n', 'tasks/dqi/fused/source/exp/fold_joint/divide.py': '"""Zero-aware streamed factor division24, no edits to arithmetic modules."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.mul99.schedules import *\nfrom exp.safe_scalar.phase import phase_inverse36_zero\nfrom exp.noscratch.kernels import inverse3_noscratch\nfrom exp.arithmetic.lowwidth import classical_transpose\n\ndef inverse_stream_zero(b,x):\n    """Overwrite12-bit x with inv0(x); expose zero via bottom norm,68T.\n\n    Same recursive streamed schedule as old66T inverse, with2AND flag only\n    after deepest inverse\'s old-input wires are erased. Peak12+1 scratch.\n    """\n    n=len(x)\n    if n==3:\n        y=inverse3_noscratch(b,x);mask=b.alloc_bits(3);b.hmr(x,mask);b.swap(x,y);b.free(y)\n        b.x(x);u=b.alloc_qubits(1)[0];F=b.alloc_qubits(1)[0]\n        b.init_and(x[0],x[1],u);b.init_and(u,x[2],F);b.del_and(x[0],x[1],u);b.free(u);b.x(x)\n        return mask,F\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    N=b.alloc_qubits(k);norm_xor_noscratch(b,x,N,K)\n    nm,F=inverse_stream_zero(b,N);ar._phase_norm(b,x,nm,K);b.free(nm)\n    outb=b.alloc_qubits(k);mul_xor_noscratch(b,x[k:],N,outb)\n    raw=b.alloc_bits(n);b.hmr(x[k:],raw[k:])\n    mul_xor_noscratch(b,x[:k],N,x[k:]);b.cx(outb,x[k:]);b.hmr(x[:k],raw[:k])\n    b.swap(x[:k],x[k:]);b.swap(x[k:],outb);b.free(outb)\n    ar._erase_norm(b,x,N,K);b.free(N);linear_inplace(b,x,frm)\n    mask=classical_transpose(b,raw,to);b.free(raw);return mask,F\n\ndef divide_zero371(b,x,y,K=None,extra_zero_mask=None,after=None):\n    """x/=y for y!=0, identity when y0. <=371T,37 private, all-input safe.\n\n    Norm\'s zero flag costs2T in streamed inverse, not a23-bit factor test.\n    Its phase is fused into the existing36T inverse-norm phase correction.\n    WIDTH OBSTRUCTION: u12 remains live across a24-bit product target, +F.\n    """\n    if K is None:K=extension_columns()[2]\n    u=b.alloc_qubits(12);norm_xor_noscratch(b,y,u,K)\n    nm,F=inverse_stream_zero(b,u);ar._phase_norm(b,y,nm,K);b.free(nm)\n    b.cx(y[12:],y[:12]);b.cx(F,y[0]);oldmask=multiply_forward99(b,x,y,K);b.cx(F,y[0]);b.cx(y[12:],y[:12])\n    b.cx(F,u[0]);masks=[]\n    for off in(0,12):\n        half=x[off:off+12];out=b.alloc_qubits(12);mul_xor_noscratch(b,half,u,out)\n        mask=b.alloc_bits(12);b.hmr(half,mask);masks.append(mask);b.swap(half,out);b.free(out)\n    b.cx(F,u[0]);um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,y,N,K)\n    b.cx(F,N[0])\n    for off,mask in zip((0,12),masks):phase_product(b,x[off:off+12],N,mask);b.free(mask)\n    b.cx(F,N[0]);b.cx(F,y[0]);phase_product24(b,x,y,oldmask,K);b.cx(F,y[0]);b.free(oldmask)\n    if after is not None:after(F)\n    fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n    if extra_zero_mask is not None:b.cx(extra_zero_mask,fm[0])\n    phase_inverse36_zero(b,N,um,fm[0]);b.free(um);b.free(fm)\n    ar._erase_norm(b,y,N,K);b.free(N)\n', 'tasks/dqi/fused/source/exp/fold_joint/kernel.py': '"""CRT joint remainder/Taylor fold. Live BEST modules are read only.\n\nThe factor flag remains live across quotient restoration, so its existing\ncompletion can be shared without another zero test or controlled word bank.\nThe polynomial and zero-factor completion match BEST for arbitrary heads.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.partial_gauge import synthesis_noflag as old\nfrom exp.mul99.kernels import mul24_xor99\nfrom exp.mul99.schedules import divide_lowwidth369\nfrom exp.fold_joint.divide import divide_zero371\nfrom exp.decoder.common import *\nbase=old.base;pg=old.pg;ck=old.ck;dv=old.dv\n\ndef epow(c,k):\n    ans=1;g=c|4096\n    for _ in range(k):ans=emul(ans,g)\n    return ans\ndef escale(b,R,c):ar.linear_inplace(b,combine(R),[emul(1<<i,c)for i in range(24)])\n\ndef q_times_low(b,B,kap,reverse=False):\n    n=len(B)\n    if not n:return\n    rows=[[kap if i==j else 1 if i-j in(1,2)else 0 for j in range(n)]for i in range(n)]\n    linear_words(b,B,rows,reverse)\n\ndef low_product(b,head,A,B):\n    """l*A mod x^k, k low Acoeffs; rank0/1/k+1 at k0/1/>=2."""\n    k=len(B)\n    if not k:return\n    V,W=base.vandermonde(k+1)\n    for t in range(k+1):\n        old.rank_affine(b,head,[1,t],A[:k],V[t][:k],B,[W[j][t]for j in range(k)])\n\ndef R_to_tops(b,R,B,c,reverse=False):\n    """CRT(E-remainder, lowTaylor)->(topcoeffs, unchangedlowTaylor)."""\n    k=len(B);powers=base.evaluation_coefficients(k,c)\n    if reverse:\n        cx(b,R[1],R[0],c);escale(b,R,epow(c,k))\n        for word,v in zip(B,powers):cx(b,word,R[0],v&4095);cx(b,word,R[1],v>>12)\n    else:\n        for word,v in zip(B,powers):cx(b,word,R[0],v&4095);cx(b,word,R[1],v>>12)\n        escale(b,R,einv(epow(c,k)));cx(b,R[1],R[0],c)\n\ndef divide414_callback(b,x,y,K,extra_zero_mask,after):\n    F=dv.zero_flag(b,y);b.cx(F,y[0]);divide_lowwidth369(b,x,y,K);b.cx(F,y[0])\n    after(F)\n    mask=b.alloc_bits(1);b.hmr(F,mask[0]);b.free(F);b.cx(extra_zero_mask,mask[0])\n    dv.phase_zero(b,y,mask[0]);b.free(mask)\n\ndef absorb(b,head,A,B,qi,p,virtual,pconst,c,kap,reverse=False,transcript=None,wide=False):\n    k=len(B);assert len(A)==k+1;out=B+head\n    if not reverse:\n        value=old.eval_implicit(b,A,qi,c);flag=dv.zero_flag(b,value)\n        spare=[b.alloc_qubits(12),b.alloc_qubits(12)];b.cx(flag,value[0]);cx(b,head[1],head[0],c)\n        mul24_xor99(b,combine(head),value,combine(spare),K)\n        cx(b,head[1],head[0],c);b.cx(flag,value[0]);old.eval_implicit(b,A,qi,c,True)\n        q_times_low(b,B,kap);b.cx(flag,A[0][0]);low_product(b,head,A,B);b.cx(flag,A[0][0])\n        v=epow(c,k+2);old.add_alias(b,p,spare,[v&4095,v>>12],virtual,pconst)\n        R_to_tops(b,spare,B,c)\n        hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n        for h,z in zip(head,spare):b.swap(h,z);b.free(z)\n        fm=b.alloc_bits(1);b.hmr(flag,fm[0]);b.free(flag)\n        return out,A,(hm,fm)\n    R_to_tops(b,head,B,c,True)\n    v=epow(c,k+2);old.add_alias(b,p,head,[v&4095,v>>12],virtual,pconst)\n    value=old.eval_implicit(b,A,qi,c);hm,fm=transcript\n    def after(F):\n        cx(b,head[1],head[0],c);old.eval_implicit(b,A,qi,c,True)\n        b.cx(F,A[0][0]);low_product(b,head,A,B);b.cx(F,A[0][0]);q_times_low(b,B,kap,True)\n        old.eval_implicit(b,A,qi,c);cx(b,head[1],head[0],c)\n    if wide:divide_zero371(b,combine(head),value,K=K,extra_zero_mask=fm[0],after=after)\n    else:divide414_callback(b,combine(head),value,K,fm[0],after)\n    b.free(fm)\n    cx(b,head[1],head[0],c);b.cz(combine(head),hm);b.free(hm)\n    old.eval_implicit(b,A,qi,c,True)\n    return out,A,transcript\n\ndef synthesize(b,words,keep_leading=True,wide=False):\n    h=len(words)//2;assert h>=3 and h%2==1\n    cs,ks,_,_=old.constants(h);heads=[words[2*i:2*i+2]for i in range(h)]\n    d=b.alloc_qubits(12);F=pg.zero_compact(b,heads[0][1]);masks=[]\n    for i in range(h):masks.append(old.prefix_encode(b,heads,i,F,d))\n    old.divide_safe138(b,d,heads[-1][1]);A=[d];B=[];hist=[]\n    for i in reversed(range(h)):\n        old.prefix_decode(b,heads,i,F,masks[i]);qi=heads[i-1][1]if i else None;p,v,pc=old.p_alias(heads,i,F)\n        oldA,oldB=A,B;A,B,t=absorb(b,heads[i],A,B,qi,p,v,pc,cs[i],ks[i],wide=wide)\n        hist.append((i,oldA,oldB,qi,p,v,pc,cs[i],ks[i],t))\n    extra=[A[h]];freed=[]if keep_leading else extra\n    for word in freed:b.free(word)\n    fmask=b.alloc_bits(1);b.hmr(F,fmask[0]);b.free(F)\n    t=dict(h=h,heads=heads,d=d,F=F,history=hist,A=A,B=B,extra=extra,freed=freed,keep_leading=keep_leading,fmask=fmask,wide=wide)\n    return A[:h],B,t\n\ndef unsynthesize(b,t):\n    if t[\'freed\']:reclaim(b,t[\'freed\'])\n    heads=t[\'heads\'];F=t[\'F\'];masks=[None]*t[\'h\']\n    for i,A,B,qi,p,v,pc,c,kap,token in reversed(t[\'history\']):\n        absorb(b,heads[i],A,B,qi,p,v,pc,c,kap,True,token,t[\'wide\']);masks[i]=old.prefix_encode(b,heads,i,F)\n        if i==0:\n            reclaim(b,[[F]]);lad=base.zero_hold(b,heads[0][1]);b.cx(lad[-1],F);base.zero_free(b,heads[0][1],lad)\n            b.cz(F,t[\'fmask\'][0]);b.free(t[\'fmask\'])\n    old.multiply_safe165(b,t[\'d\'],heads[-1][1])\n    for i in reversed(range(t[\'h\'])):old.prefix_decode(b,heads,i,F,masks[i],t[\'d\'])\n    pg.zero_compact_free(b,heads[0][1],F);b.free(t[\'d\'])\n', 'tasks/dqi/fused/source/exp/inverse12_escape/joint.py': '"""Correlated inverse-norm cache lowers the paired I12 reverse to67T/Q33.\nThe initialized inverse is unchanged64T/Q31; no <=63 isolated claim.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse19_joint.kernel import *\nfrom exp.inverse12_escape.zero1 import erase_raw_cache\n\ndef inverse6_keep_inner19(b,x):\n    """Preserve x; return inv0(x) and norm(inv0(x))3,19T/Q19."""\n    to,frm,K=tower(6);linear_inplace(b,x,to);g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,x,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(x)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms)\n    linear_inplace(b,g,LAY[\'raw_columns\']);R,E=g[:4],g[4:];z=list(E)+list(b.alloc_qubits(4));source=list(x)+list(R)+list(w)\n    for j,e in enumerate(LAY[\'outputs\']):ref.form_xor(b,source,e&65535,z[j])\n    for i,(l,r)in enumerate(LAY[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(LAY[\'outputs\']))\n        with ref.expose_pair(b,source,l,r)as(p,q):rankone_xor(b,p,q,z,v)\n    erase_raw_cache(b,x,R,LAY[\'raw_rows\'][:4]);linear_inplace(b,z,LAY[\'output_inverse_columns\'])\n    linear_inplace(b,x,frm);linear_inplace(b,z,frm);return z,w\n\ndef inverse12_reverse67(b,x,mask):\n    assert len(x)==12;to,frm,K=tower(12);linear_inplace(b,x,to)\n    N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,K);t,v=inverse6_keep_inner19(b,N);out=b.alloc_qubits(6)\n    mul_xor_noscratch(b,x[6:],t,out);erase_product(b,out,N,x[6:]);mul_xor_noscratch(b,x[:6],t,x[6:]);b.cx(out,x[6:])\n    b.cx(out,x[6:]);erase_product(b,x[6:],N,x[:6]);b.cx(out,x[6:]);b.swap(x[:6],x[6:]);b.swap(x[6:],out);b.free(out)\n    D=inverse3_noscratch(b,v);ar._erase_inverse_with_norm(b,t,N,D);b.free(N);erase_inverse(b,v,D);b.free(D)\n    to6,frm6,K6=tower(6);linear_inplace(b,t,to6);ar._erase_norm(b,t,v,K6);b.free(v);linear_inplace(b,t,frm6)\n    ar._erase_norm(b,x,t,K);b.free(t);linear_inplace(b,x,frm);b.cz(x,mask);b.free(mask)\n\ninverse12_forward64=inverse_inplace_forward20\n\ndef phase_inverse6_quantum19(b,u,L):\n    """Phase L dot inv0(u), bothquantum6;19T,9private, exactclosedHMR."""\n    to,frm,K=tower(6);linear_inplace(b,u,to);transpose=[sum(((frm[i]>>j)&1)<<i for i in range(6))for j in range(6)];linear_inplace(b,L,transpose)\n    g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,u,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(u)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms);source=prefix+list(w)\n    for j,e in enumerate(basic.DAG19[\'outputs\']):\n        if e&1:b.z(L[j])\n        for r in bits((e&65535)>>1):b.cz(L[j],source[r])\n    for i,(l,r)in enumerate(basic.DAG19[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(basic.DAG19[\'outputs\']));ix=list(bits(v));piv=ix[0]\n        for j in ix[1:]:b.cx(L[j],L[piv])\n        with ref.expose_pair(b,source,l,r)as(p,q):b.ccz(p,q,L[piv])\n        for j in reversed(ix[1:]):b.cx(L[j],L[piv])\n    m=b.alloc_bits(3);b.hmr(w,m);b.free(w);ref.phase_inverse3_virtual(b,prefix,forms,m);b.free(m)\n    for i,(l,r)in reversed(list(enumerate(basic.raw_forms19()))):\n        with ref.expose_pair(b,u,l,r)as(p,q):b.del_and(p,q,g[i])\n    b.free(g);linear_inplace(b,L,inverse_columns(transpose));linear_inplace(b,u,frm)\n\ndef inverse12_keep_inverse_norm64(b,x):\n    """Preserve x12, return t=inv(x),u=norm(t)6,64T;Q31."""\n    to,frm,K=tower(12);linear_inplace(b,x,to);D=b.alloc_qubits(6);norm_xor_noscratch(b,x,D,K);u=inverse6_init19(b,D);ar._erase_norm(b,x,D,K);b.free(D)\n    t=b.alloc_qubits(12);b.cx(x[6:],x[:6]);mul_xor_noscratch(b,x[:6],u,t[:6]);mul_xor_noscratch(b,x[6:],u,t[6:]);b.cx(x[6:],x[:6]);linear_inplace(b,x,frm);linear_inplace(b,t,frm);return t,u\n\ndef erase_norm_via_inverse_cache19(b,t,N,u):\n    """Erase N=inv0(t),u=norm(t)6;19T vsoldDcachecleanup24T.\n\n    Joint phase mask for N is linear in t and classical HMR outcomes; evaluate\n    inverse6 with that quantum mask, then erase u quadratically as norm(t).\n    """\n    to,frm,K=tower(12);linear_inplace(b,t,to);linear_inplace(b,N,to);m=b.alloc_bits(12);b.hmr(N,m);b.free(N);L=b.alloc_qubits(6);f=field(6)\n    def masks():\n        for j in range(6):\n            for i in range(6):\n                for r in bits(f.mul(1<<i,1<<j)):\n                    b.ccx(m[r],t[i],L[j]);b.ccx(m[r],t[i+6],L[j]);b.ccx(m[r+6],t[i+6],L[j])\n    masks();phase_inverse6_quantum19(b,u,L);masks();b.free(L);b.free(m);ar._erase_norm(b,t,u,K);b.free(u);linear_inplace(b,t,frm)\n\ndef inverse24_reverse209(b,x,mask,K=None):\n    cols,back,_=extension_columns(K);linear_inplace(b,x,cols);to,frm,K24=tower(24);linear_inplace(b,x,to)\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,x,N,K24);t,u=inverse12_keep_inverse_norm64(b,N);out=b.alloc_qubits(12)\n    mul_xor_noscratch(b,x[12:],t,out);erase_product(b,out,N,x[12:]);mul_xor_noscratch(b,x[:12],t,x[12:]);b.cx(out,x[12:]);b.cx(out,x[12:]);erase_product(b,x[12:],N,x[:12]);b.cx(out,x[12:]);b.swap(x[:12],x[12:]);b.swap(x[12:],out);b.free(out)\n    erase_norm_via_inverse_cache19(b,t,N,u);ar._erase_norm(b,x,t,K24);b.free(t);linear_inplace(b,x,frm);b.cz(x,mask);b.free(mask);linear_inplace(b,x,back)\n\ninverse24_forward190=basic.inverse24_forward19\n', 'tasks/dqi/fused/source/exp/inverse12_escape/zero1.py': '"""65T/Q31 zero-aware I12 from a one-AND correlated zero output.\nNot an improvement to isolated64T I12 inversion; no live module is changed.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse19_joint.kernel import *\nfrom types import FunctionType\nZLAY=json.loads(Path(__file__).with_name(\'zero_layout.json\').read_text())\n\ndef erase_raw_cache(b,x,cache,rows):\n    m=b.alloc_bits(len(cache));b.hmr(cache,m);b.free(cache)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,x,l,r)as(p,q):\n            for j,row in enumerate(rows):\n                if row>>i&1:b.ccz(p,q,m[j])\n    b.free(m)\n\ndef inverse6_zero20(b,x):\n    to,frm,K=tower(6);linear_inplace(b,x,to);g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,x,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(x)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms)\n    linear_inplace(b,g,LAY[\'raw_columns\']);R,E=g[:4],g[4:];z=list(E)+list(b.alloc_qubits(4));source=list(x)+list(R)+list(w)\n    for j,e in enumerate(LAY[\'outputs\']):ref.form_xor(b,source,e&65535,z[j])\n    for i,(l,r)in enumerate(LAY[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(LAY[\'outputs\']))\n        with ref.expose_pair(b,source,l,r)as(p,q):rankone_xor(b,p,q,z,v)\n    # One norm coordinate lies in the4D live raw-control span. Keep it, erase3.\n    linear_inplace(b,R,ZLAY[\'cache_columns\']);erase_raw_cache(b,x,R[1:],ZLAY[\'raw_cache_rows\'][1:])\n    source=list(x)+[R[0]]+list(w);left=1|(ZLAY[\'left_w\']<<8);right=1^ZLAY[\'input_form\']^(1<<7)^(ZLAY[\'right_w\']<<8)\n    F=b.alloc_qubits(1)[0]\n    with ref.expose_pair(b,source,left,right)as(p,q):b.init_and(p,q,F)\n    erase_raw_cache(b,x,R[:1],ZLAY[\'raw_cache_rows\'][:1]);linear_inplace(b,z,LAY[\'output_inverse_columns\']);ar._erase_norm(b,z,w,K);b.free(w)\n    linear_inplace(b,x,frm);linear_inplace(b,z,frm);return z,F\n\ndef inverse_zero(b,x):\n    to,frm,K=tower(12);linear_inplace(b,x,to);N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,K)\n    t,F=inverse6_zero20(b,N);ar._erase_norm(b,x,N,K);b.free(N)\n    out=b.alloc_qubits(12);b.cx(x[6:],x[:6]);mul_xor_noscratch(b,x[:6],t,out[:6]);mul_xor_noscratch(b,x[6:],t,out[6:]);b.cx(x[6:],x[:6])\n    ar._erase_norm(b,out,t,K);b.free(t);linear_inplace(b,x,frm);linear_inplace(b,out,frm);return out,F\n\nfrom exp.inverse19_joint import kernel as parentkernel\n_old=parentkernel.inverse_apply_safe\ninverse_apply_safe=FunctionType(_old.__code__,globals(),\'inverse_apply_safe\',_old.__defaults__,_old.__closure__)\n', 'tasks/dqi/fused/source/exp/inverse12_escape/zero_layout.json': '{\n  "norm_coordinate": 3,\n  "raw_keep_coeff": 14,\n  "input_form": 58,\n  "left_w": 1,\n  "right_w": 4,\n  "pair_forms": [\n    2,\n    5\n  ],\n  "linear_correction": 3,\n  "cache_rows": [\n    14,\n    1,\n    2,\n    4\n  ],\n  "cache_columns": [\n    2,\n    5,\n    9,\n    1\n  ],\n  "raw_cache_rows": [\n    27,\n    41,\n    5,\n    14\n  ]\n}\n', 'tasks/dqi/fused/source/exp/inverse19_joint/kernel.py': '"""Width-audited19AND inverse substitutions; stable imports remain unchanged."""\nfrom pathlib import Path\nimport sys,json\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse_width_subspaces import kernel19 as basic\nfrom exp.inverse_width_subspaces.kernel19 import *\nfrom exp.safe_scalar import kernel as safe\nfrom exp.safe_scalar.phase import phase_inverse36_zero\nfrom exp.noscratch.schedules import norm_xor_noscratch\nLAY=json.loads(Path(__file__).with_name(\'layout.json\').read_text())\n\ndef inverse6_init19(b,x,with_zero=False):\n    """Fresh inverse6:19T/Q19, or inverse+zero21T/Q19; preserve x.\n\n    Reuse two redundant raw norm directions as output. Four other norm\n    directions remain until all final products are added. Compute zero only\n    AFTER those four wires are erased, using the still-live inverse norm3.\n    """\n    to,frm,K=tower(6);linear_inplace(b,x,to);g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,x,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(x)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms)\n    linear_inplace(b,g,LAY[\'raw_columns\']);R,E=g[:4],g[4:]\n    z=list(E)+list(b.alloc_qubits(4));source=list(x)+list(R)+list(w)\n    for j,e in enumerate(LAY[\'outputs\']):ref.form_xor(b,source,e&65535,z[j])\n    for i,(l,r)in enumerate(LAY[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(LAY[\'outputs\']));assert v\n        with ref.expose_pair(b,source,l,r)as(p,q):rankone_xor(b,p,q,z,v)\n    # Four retained raw products have only quadratic phase corrections.\n    rm=b.alloc_bits(4);b.hmr(R,rm);b.free(R)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,x,l,r)as(p,q):\n            for j,row in enumerate(LAY[\'raw_rows\'][:4]):\n                if row>>i&1:b.ccz(p,q,rm[j])\n    b.free(rm);linear_inplace(b,z,LAY[\'output_inverse_columns\'])\n    F=None\n    if with_zero:\n        b.x(w);u=b.alloc_qubits(1)[0];F=b.alloc_qubits(1)[0]\n        b.init_and(w[0],w[1],u);b.init_and(u,w[2],F);b.del_and(w[0],w[1],u);b.free(u);b.x(w)\n    ar._erase_norm(b,z,w,K);b.free(w);linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n    return(z,F)if with_zero else z\n\ndef inverse6_xor19(b,x,z=None,keep_norm=False):\n    if z is None and not keep_norm:return inverse6_init19(b,x)\n    return basic.inverse6_xor19(b,x,z,keep_norm)\n\n# Local recursive schedules, no mutation of imported globals.\nfrom types import FunctionType\ninverse6_xor20=inverse6_xor19\nfor _name in(\'inverse_initialized\',\'inverse12_keep_norm\',\'inverse_inplace_forward20\',\'inverse_inplace_reverse20\'):\n    _old=getattr(ref,_name);globals()[_name]=FunctionType(_old.__code__,globals(),_name,_old.__defaults__,_old.__closure__)\n\ndef inverse_zero(b,x):\n    """GF12 inverse0 +zero:66T/Q31, preserve input; closure unchanged36T."""\n    to,frm,K=tower(12);linear_inplace(b,x,to);N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,K)\n    t,F=inverse6_init19(b,N,True);ar._erase_norm(b,x,N,K);b.free(N)\n    out=b.alloc_qubits(12);b.cx(x[6:],x[:6]);mul_xor_noscratch(b,x[:6],t,out[:6]);mul_xor_noscratch(b,x[6:],t,out[6:]);b.cx(x[6:],x[:6])\n    ar._erase_norm(b,out,t,K);b.free(t);linear_inplace(b,x,frm);linear_inplace(b,out,frm);return out,F\n\nconsume=safe.consume;erase=safe.erase\n\ndef inverse_apply_safe(b,a,action,incoming=None,defer=False):\n    ai,F=inverse_zero(b,a);consume(b,ai,F,incoming)\n    b.cx(F,a[0]);b.cx(F,ai[0]);action(ai);b.cx(F,ai[0]);b.cx(F,a[0])\n    return erase(b,a,ai,F,defer)\n\nfrom exp.arithmetic.lowwidth import phase_inverse_generic6,phase_product_quantum\n\ndef inverse12_xor73(b,x,z,base6=inverse6_init19):\n    """Arbitrary-target XOR inverse12:73T/Q43, preserve x and target phase.\n\n    Initialized inverse schedules are NOT arbitrary-target oracles: they erase\n    t using norm(output), which is wrong for a preexisting target. Keep N and\n    close t by a9T phase inverse instead.\n    """\n    to,frm,K=tower(12);linear_inplace(b,x,to);linear_inplace(b,z,to)\n    N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,K);t=base6(b,N)\n    b.cx(x[6:],x[:6]);mul_xor_noscratch(b,x[:6],t,z[:6]);mul_xor_noscratch(b,x[6:],t,z[6:]);b.cx(x[6:],x[:6])\n    m=b.alloc_bits(6);b.hmr(t,m);b.free(t);phase_inverse_generic6(b,N,m);b.free(m)\n    ar._erase_norm(b,x,N,K);b.free(N);linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n', 'tasks/dqi/fused/source/exp/inverse19_joint/layout.json': '{\n  "raw_rows": [\n    41,\n    5,\n    14,\n    16,\n    1,\n    2\n  ],\n  "raw_columns": [\n    19,\n    36,\n    6,\n    5,\n    8,\n    1\n  ],\n  "output_columns": [\n    4,\n    8,\n    16,\n    1,\n    32,\n    6\n  ],\n  "output_inverse_columns": [\n    8,\n    33,\n    1,\n    2,\n    4,\n    16\n  ],\n  "gates": [\n    [\n      2,\n      2048\n    ],\n    [\n      48,\n      6144\n    ],\n    [\n      36,\n      4096\n    ],\n    [\n      2244,\n      4408\n    ],\n    [\n      2158,\n      12302\n    ],\n    [\n      2056,\n      8208\n    ],\n    [\n      6256,\n      8310\n    ],\n    [\n      6236,\n      10252\n    ],\n    [\n      4828,\n      9436\n    ],\n    [\n      4160,\n      8196\n    ]\n  ],\n  "outputs": [\n    54332300,\n    39463256,\n    60299450,\n    46469830,\n    11278712,\n    20656990\n  ]\n}\n', 'tasks/dqi/fused/source/exp/inverse_search/baseline.py': '"""21T correlated inverse6 XOR seed, arbitrary target, clean private state."""\nimport sys\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.noscratch.kernels import *\n\ndef inverse6_xor21(b,x,z):\n    assert len(x)==len(z)==6\n    to,frm,K=tower(6);linear_inplace(b,x,to);linear_inplace(b,z,to)\n    N=b.alloc_qubits(3);f=field(3)\n    linear_xor(b,x[:3],N,square_columns(3))\n    linear_xor(b,x[3:],N,[f.mul(K,v)for v in square_columns(3)])\n    mul_xor_noscratch(b,x[:3],x[3:],N)\n    w=inverse3_noscratch(b,N)\n    b.cx(x[3:],x[:3]);mul_xor_noscratch(b,x[:3],w,z[:3]);b.cx(x[3:],x[:3])\n    mul_xor_noscratch(b,x[3:],w,z[3:])\n    erase_inverse(b,N,w);b.free(w);ar._erase_norm(b,x,N,K);b.free(N)\n    linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n', 'tasks/dqi/fused/source/exp/inverse_search/inverse20.py': '"""Explicit 20-AND GF64 inverse XOR circuit with exact measurement cleanup.\n\nAll masks below describe physical linear forms. Prefix source order is x0..5,\nsix raw norm rank-products, then the three inverse-norm bits. Constant is bit0\nof a form mask; physical source i is bit(i+1).\n"""\ntry:\n    from .baseline import *\nexcept ImportError:\n    from baseline import *\nfrom contextlib import contextmanager\nimport json\n\nDAG=json.loads(Path(__file__).with_name(\'optimized.json\').read_text())\n\n@contextmanager\ndef expose_pair(b,registers,left,right):\n    u,v=left>>1,right>>1;p=next(bits(u));ops=[]\n    for j in bits(u^(1<<p)):b.cx(registers[j],registers[p]);ops.append((j,p))\n    if v>>p&1:v^=u^(1<<p)\n    q=next(bits(v&~(1<<p)))\n    for j in bits(v^(1<<q)):b.cx(registers[j],registers[q]);ops.append((j,q))\n    if left&1:b.x(registers[p])\n    if right&1:b.x(registers[q])\n    yield registers[p],registers[q]\n    if right&1:b.x(registers[q])\n    if left&1:b.x(registers[p])\n    for j,k in reversed(ops):b.cx(registers[j],registers[k])\n\ndef form_xor(b,registers,mask,target):\n    if mask&1:b.x(target)\n    for i in bits(mask>>1):b.cx(registers[i],target)\n\ndef norm_forms(K):\n    forms=[0]*3;f=field(3)\n    for i,v in enumerate(square_columns(3)):\n        for j in bits(v):forms[j]^=1<<(1+i)\n        for j in bits(f.mul(K,v)):forms[j]^=1<<(1+3+i)\n    for i,(_,v)in enumerate(tensor(3)):\n        for j in bits(v):forms[j]^=1<<(1+6+i)\n    return forms\n\ndef inverse3_virtual(b,registers,forms):\n    out=b.alloc_qubits(3)\n    for mask,vector in enumerate(inverse3_anf()):\n        if not vector:continue\n        ix=list(bits(mask))\n        if len(ix)==1:\n            for j in bits(vector):form_xor(b,registers,forms[ix[0]],out[j])\n        else:\n            assert len(ix)==2\n            with expose_pair(b,registers,forms[ix[0]],forms[ix[1]])as(p,q):rankone_xor(b,p,q,out,vector)\n    return out\n\ndef phase_inverse3_virtual(b,registers,forms,mask):\n    for monomial,vector in enumerate(inverse3_anf()):\n        if not vector:continue\n        ix=list(bits(monomial))\n        if len(ix)==1:\n            assert not(forms[ix[0]]&1)\n            for i in bits(forms[ix[0]]>>1):\n                for j in bits(vector):b.cz(registers[i],mask[j])\n        else:\n            with expose_pair(b,registers,forms[ix[0]],forms[ix[1]])as(p,q):\n                for j in bits(vector):b.ccz(p,q,mask[j])\n\ndef inverse6_xor20(b,x,z=None,keep_norm=False):\n    """z ^= inverse(x), inv0=0; 20T,21 total Q for input6+output6.\n\n    Arbitrary target is supported. If omitted, allocate clean six-bit target.\n    Exactly six norm ANDs, three inverse3 ANDs, eleven final ANDs; cleanup0T.\n    """\n    assert len(x)==6\n    to,frm,K=tower(6);linear_inplace(b,x,to)\n    if z is not None:linear_inplace(b,z,to)\n    g=b.alloc_qubits(6)\n    for i,(mask,_)in enumerate(tensor(3)):\n        with expose_pair(b,x,mask<<1,mask<<4)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(x)+list(g);forms=norm_forms(K)\n    w=inverse3_virtual(b,prefix,forms);source=prefix+list(w)\n    if z is None:z=b.alloc_qubits(6)\n    for j,expression in enumerate(DAG[\'outputs\']):form_xor(b,source,expression&65535,z[j])\n    for i,(left,right)in enumerate(DAG[\'gates\']):\n        vector=sum(((expression>>(16+i))&1)<<j for j,expression in enumerate(DAG[\'outputs\']))\n        assert vector\n        with expose_pair(b,source,left,right)as(p,q):rankone_xor(b,p,q,z,vector)\n    wm=b.alloc_bits(3);b.hmr(w,wm);b.free(w)\n    phase_inverse3_virtual(b,prefix,forms,wm);b.free(wm)\n    kept=None\n    if keep_norm:\n        kept=b.alloc_qubits(3)\n        for j,form in enumerate(forms):form_xor(b,prefix,form,kept[j])\n    for i,(mask,_)in reversed(list(enumerate(tensor(3)))):\n        with expose_pair(b,x,mask<<1,mask<<4)as(p,q):b.del_and(p,q,g[i])\n    b.free(g);linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n    return (z,kept)if keep_norm else z\n\ndef inverse_initialized(b,x,target=None):\n    """No-scratch recursive initialized inverse using the20T base at n6."""\n    n=len(x)\n    if n==3:return inverse3_noscratch(b,x,target)\n    if n==6:return inverse6_xor20(b,x,target)\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    norm=b.alloc_qubits(k);f=field(k)\n    linear_xor(b,x[:k],norm,square_columns(k))\n    linear_xor(b,x[k:],norm,[f.mul(K,v)for v in square_columns(k)])\n    mul_xor_noscratch(b,x[:k],x[k:],norm)\n    invnorm=inverse_initialized(b,norm)\n    ar._erase_norm(b,x,norm,K);b.free(norm)\n    if target is None:target=b.alloc_qubits(n)\n    b.cx(x[k:],x[:k]);mul_xor_noscratch(b,x[:k],invnorm,target[:k]);mul_xor_noscratch(b,x[k:],invnorm,target[k:]);b.cx(x[k:],x[:k])\n    ar._erase_norm(b,target,invnorm,K);b.free(invnorm)\n    linear_inplace(b,x,frm);linear_inplace(b,target,frm)\n    return target\n\ndef norm_xor(b,x,z,K):\n    k=len(z);f=field(k)\n    linear_xor(b,x[:k],z,square_columns(k))\n    linear_xor(b,x[k:],z,[f.mul(K,v)for v in square_columns(k)])\n    mul_xor_noscratch(b,x[:k],x[k:],z)\n\ndef inverse12_keep_norm(b,x):\n    """65T inverse12 keeping its6-bit norm; used by closed reverse24."""\n    assert len(x)==12\n    to,frm,K=tower(12);linear_inplace(b,x,to)\n    N=b.alloc_qubits(6);norm_xor(b,x,N,K);w=inverse6_xor20(b,N)\n    target=b.alloc_qubits(12);b.cx(x[6:],x[:6])\n    mul_xor_noscratch(b,x[:6],w,target[:6]);mul_xor_noscratch(b,x[6:],w,target[6:]);b.cx(x[6:],x[:6])\n    ar._erase_norm(b,target,w,K);b.free(w)\n    linear_inplace(b,x,frm);linear_inplace(b,target,frm)\n    return target,N\n\ndef inverse_inplace_forward20(b,x):\n    target=inverse_initialized(b,x);mask=b.alloc_bits(len(x))\n    b.hmr(x,mask);b.swap(x,target);b.free(target);return mask\n\ndef inverse_inplace_reverse20(b,x,mask):\n    """Closed reverse12/24 at71/215T,21/42 private qubits."""\n    n=len(x);assert n in(12,24);k=n//2\n    to,frm,K=tower(n);linear_inplace(b,x,to)\n    N=b.alloc_qubits(k);norm_xor(b,x,N,K)\n    t,D=inverse12_keep_norm(b,N)if n==24 else inverse6_xor20(b,N,keep_norm=True)\n    out=b.alloc_qubits(k);mul_xor_noscratch(b,x[k:],t,out)\n    erase_product(b,out,N,x[k:]);mul_xor_noscratch(b,x[:k],t,x[k:]);b.cx(out,x[k:])\n    b.cx(out,x[k:]);erase_product(b,x[k:],N,x[:k]);b.cx(out,x[k:])\n    b.swap(x[:k],x[k:]);b.swap(x[k:],out);b.free(out)\n    ar._erase_inverse_with_norm(b,t,N,D);b.free(N)\n    tok,frmk,Kk=tower(k);linear_inplace(b,t,tok)\n    u=b.alloc_qubits(k//2);norm_xor(b,t,u,Kk);erase_inverse(b,u,D);b.free(D)\n    ar._erase_norm(b,t,u,Kk);b.free(u);linear_inplace(b,t,frmk)\n    ar._erase_norm(b,x,t,K);b.free(t);linear_inplace(b,x,frm)\n    b.cz(x,mask);b.free(mask)\n\ndef inverse24_forward20(b,x,K=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols)\n    mask=inverse_inplace_forward20(b,x);linear_inplace(b,x,frm);return mask\n\ndef inverse24_reverse20(b,x,mask,K=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols)\n    inverse_inplace_reverse20(b,x,mask);linear_inplace(b,x,frm)\n', 'tasks/dqi/fused/source/exp/inverse_search/inverse20_iso.py': '"""Width-preserving20T GF64 inverse using the lambda2 norm isotopy.\n\nInitialized output reuses three redundant norm directions as output wires.\nNo core modules are edited or monkeypatched.\n"""\ntry:\n    from . import inverse20 as base\n    from .inverse20 import *\nexcept ImportError:\n    import inverse20 as base\n    from inverse20 import *\nDAG=json.loads(Path(__file__).with_name(\'optimized_isotopy2.json\').read_text())\nEXTRA_ROWS=[11,52,34]\nNORM_ROWS=[42,49,54]\nassert [(e>>7)&63 for e in DAG[\'outputs\']]==[11,0,52,34,52,52]\nassert all(not((l|r)&(63<<7))for l,r in DAG[\'gates\'])\n\ndef raw_forms():\n    f=field(3);scale=2;inv=f.invert(scale);forms=[]\n    for mask,_ in tensor(3):\n        left=sum(((mask&f.mul(scale,1<<i)).bit_count()&1)<<(1+i)for i in range(3))\n        right=sum(((mask&f.mul(inv,1<<i)).bit_count()&1)<<(4+i)for i in range(3))\n        forms.append((left,right))\n    return forms\n\ndef xw_form(mask):return(mask&127)|((mask>>6)&(7<<7))\n\ndef erase_extras(b,x,E):\n    masks=b.alloc_bits(3);b.hmr(E,masks);b.free(E)\n    for i,(left,right)in enumerate(raw_forms()):\n        with expose_pair(b,x,left,right)as(p,q):\n            for j,row in enumerate(EXTRA_ROWS):\n                if row>>i&1:b.ccz(p,q,masks[j])\n    b.free(masks)\n\ndef inverse6_xor20(b,x,z=None,keep_norm=False):\n    """20T exact inverse6: fresh-output Q15; arbitrary-target Q18.\n\n    keep_norm=True returns(input_norm3) too, initialized Q18. All private\n    erasures are explicit0T quadratic phase fixes. Zero maps to zero.\n    """\n    assert len(x)==6;fresh=z is None\n    to,frm,K=tower(6);linear_inplace(b,x,to)\n    if not fresh:linear_inplace(b,z,to)\n    g=b.alloc_qubits(6)\n    for i,(left,right)in enumerate(raw_forms()):\n        with expose_pair(b,x,left,right)as(p,q):b.init_and(p,q,g[i])\n    rows=NORM_ROWS+EXTRA_ROWS\n    cols=[sum(((r>>j)&1)<<i for i,r in enumerate(rows))for j in range(6)]\n    linear_inplace(b,g,cols);N,E=g[:3],g[3:]\n    for j,form in enumerate(norm_forms(K)):form_xor(b,x,form&127,N[j])\n    if not fresh:\n        b.cx(E[0],z[0]);b.cx(E[1],[z[2],z[4],z[5]]);b.cx(E[2],z[3])\n        erase_extras(b,x,E)\n    w=inverse3_noscratch(b,N)\n    if fresh:\n        if not keep_norm:ar._erase_norm(b,x,N,K);b.free(N)\n        extra=b.alloc_qubits(3)\n        # Fixed physical layout: these data wires are now output coordinates.\n        z=[E[0],extra[0],E[1],E[2],extra[1],extra[2]]\n        b.cx(z[2],[z[4],z[5]])\n    source=list(x)+list(w)\n    for j,e in enumerate(DAG[\'outputs\']):form_xor(b,source,xw_form(e),z[j])\n    for i,(left,right)in enumerate(DAG[\'gates\']):\n        vector=sum(((e>>(16+i))&1)<<j for j,e in enumerate(DAG[\'outputs\']))\n        with expose_pair(b,source,xw_form(left),xw_form(right))as(p,q):rankone_xor(b,p,q,z,vector)\n    if fresh:\n        ar._erase_norm(b,z,w,K);b.free(w)\n    else:\n        erase_inverse(b,N,w);b.free(w)\n        if not keep_norm:ar._erase_norm(b,x,N,K);b.free(N)\n    linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n    return(z,N)if keep_norm else z\n\n# These higher-level schedules are unchanged structurally. Rebind their private\n# globals to this plugin\'s inverse6 implementation; no imported module changes.\nfrom types import FunctionType\nfor _name in (\'inverse_initialized\',\'inverse12_keep_norm\',\'inverse_inplace_forward20\',\n              \'inverse_inplace_reverse20\',\'inverse24_forward20\',\'inverse24_reverse20\'):\n    _original=getattr(base,_name)\n    globals()[_name]=FunctionType(_original.__code__,globals(),_name,_original.__defaults__,_original.__closure__)\n', 'tasks/dqi/fused/source/exp/inverse_search/optimized.json': '{\n  "prefix_AND": 9,\n  "final_AND": 11,\n  "total_AND": 20,\n  "valid_subspaces": 9,\n  "examined_subspaces": 286,\n  "constraints": [\n    3640\n  ],\n  "final_classes": [\n    1,\n    64,\n    130,\n    260,\n    520,\n    1040,\n    2080,\n    2,\n    4,\n    1664,\n    3072\n  ],\n  "gates": [\n    [\n      18,\n      8192\n    ],\n    [\n      16,\n      8192\n    ],\n    [\n      14,\n      57344\n    ],\n    [\n      8,\n      32768\n    ],\n    [\n      10,\n      40960\n    ],\n    [\n      12,\n      49152\n    ],\n    [\n      6,\n      24576\n    ],\n    [\n      126,\n      57344\n    ],\n    [\n      72,\n      32768\n    ],\n    [\n      11240,\n      24088\n    ],\n    [\n      13088,\n      50928\n    ]\n  ],\n  "outputs": [\n    106226362,\n    73483204,\n    98583492,\n    100721338,\n    67257284,\n    93078468\n  ],\n  "prefix_signal_names": [\n    "1",\n    "x0",\n    "x1",\n    "x2",\n    "x3",\n    "x4",\n    "x5",\n    "g0",\n    "g1",\n    "g2",\n    "g3",\n    "g4",\n    "g5",\n    "w0",\n    "w1",\n    "w2"\n  ]\n}\n', 'tasks/dqi/fused/source/exp/inverse_search/optimized_isotopy2.json': '{\n  "prefix_AND": 9,\n  "final_AND": 11,\n  "total_AND": 20,\n  "isotopy_scale": 2,\n  "valid_subspaces": 15,\n  "examined_subspaces": 421,\n  "constraints": [\n    3598\n  ],\n  "final_classes": [\n    1,\n    32,\n    64,\n    520,\n    4095,\n    16,\n    128,\n    256,\n    3024,\n    642,\n    2308\n  ],\n  "gates": [\n    [\n      18,\n      8192\n    ],\n    [\n      54,\n      24576\n    ],\n    [\n      16,\n      8192\n    ],\n    [\n      10,\n      40960\n    ],\n    [\n      4,\n      16384\n    ],\n    [\n      108,\n      49152\n    ],\n    [\n      112,\n      57344\n    ],\n    [\n      64,\n      32768\n    ],\n    [\n      16396,\n      32812\n    ],\n    [\n      16478,\n      40974\n    ],\n    [\n      24584,\n      32816\n    ]\n  ],\n  "outputs": [\n    38430132,\n    2293760,\n    117881380,\n    27586910,\n    132102692,\n    119257636\n  ],\n  "prefix_signal_names": [\n    "1",\n    "x0",\n    "x1",\n    "x2",\n    "x3",\n    "x4",\n    "x5",\n    "g0",\n    "g1",\n    "g2",\n    "g3",\n    "g4",\n    "g5",\n    "w0",\n    "w1",\n    "w2"\n  ]\n}\n', 'tasks/dqi/fused/source/exp/inverse_width/kernel.py': '"""39-private paired GF24 inverse via a three-bit norm cache.\n\nForward191T is unchanged iso20. Reverse220T adds five Toffolis while\nreducing reverse private workspace42->39. Explicit transcript inclinv0.\nNo stable module is edited or monkeypatched.\n"""\nimport sys,json\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse_search import inverse20_iso as iso\nfrom exp.inverse_search import inverse20 as ref\nfrom exp.inverse_search.inverse20_iso import *\nfrom exp.arithmetic.lowwidth import classical_transpose\nPART=json.loads(Path(__file__).with_name(\'partial5.json\').read_text())\nFULL=json.loads((ROOT/\'exp/inverse_search/optimized.json\').read_text())\n\ndef coefficient_xor(b,t,nm,e,anc):\n    """anc ^=dot(nm,conjugate(t)*e), e GF64 public, t tower12."""\n    f6=field(6)\n    for i in range(6):\n        v=f6.mul(e,1<<i)\n        for r in bits(v):\n            b.ccx(nm[r],t[i],anc);b.ccx(nm[r],t[i+6],anc);b.ccx(nm[r+6],t[i+6],anc)\n\ndef phase_partial_inverse6(b,u,t,nm,cm):\n    """14T: quantum-mask upper3(inv(u)), classical-mask lower3(inv(u)).\n\n    u canonicalGF64, t alreadytower12; nm12 classical, cm3classical.\n    Upper/lower refer to the standardGF8 tower6, not polynomialGF64 bits.\n    All u/norm/inverse prefix state and the one parity ancilla are restored.\n    """\n    to,frm,K=tower(6);linear_inplace(b,u,to);g=b.alloc_qubits(6)\n    for i,(mask,_) in enumerate(tensor(3)):\n        with ref.expose_pair(b,u,mask<<1,mask<<4)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(u)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms);source=prefix+list(w)\n    # Retained lower3 cache\'s ordinary classical HMR correction costs0T.\n    for j,e in enumerate(FULL[\'outputs\'][:3]):\n        assert not(e&1)\n        for r in bits(e&65535):b.cz(source[r-1],cm[j])\n    cb=b.alloc_bits(1)[0];b.bit_store(cb,False)\n    for i,(left,right)in enumerate(FULL[\'gates\']):\n        js=[j for j,e in enumerate(FULL[\'outputs\'][:3]) if e>>(16+i)&1]\n        if not js:continue\n        for j in js:b.cx(cm[j],cb)\n        with ref.expose_pair(b,source,left,right)as(p,q):b.ccz(p,q,cb)\n        for j in reversed(js):b.cx(cm[j],cb)\n    b.free(cb)\n    # Exactly five products span the three missing output functions.\n    anc=b.alloc_qubits(1)[0]\n    def coeff(v):coefficient_xor(b,t,nm,apply(frm,v<<3),anc)\n    for r in range(16):\n        v=sum(((e>>r)&1)<<j for j,e in enumerate(PART[\'outputs\']))\n        if not v:continue\n        coeff(v)\n        if r:b.cz(anc,source[r-1])\n        else:b.z(anc)\n        coeff(v)\n    for i,(left,right)in enumerate(PART[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(PART[\'outputs\']))\n        assert v\n        coeff(v)\n        with ref.expose_pair(b,source,left,right)as(p,q):b.ccz(p,q,anc)\n        coeff(v)\n    b.free(anc)\n    wm=b.alloc_bits(3);b.hmr(w,wm);b.free(w);ref.phase_inverse3_virtual(b,prefix,forms,wm);b.free(wm)\n    for i,(mask,_)in reversed(list(enumerate(tensor(3)))):\n        with ref.expose_pair(b,u,mask<<1,mask<<4)as(p,q):b.del_and(p,q,g[i])\n    b.free(g);linear_inplace(b,u,frm)\n\ndef compress_D(b,N,D):\n    """Retain lower3 tower6 coordinates, eraseupper3 with quadratic norm phase."""\n    to6,frm6,K6=tower(6);linear_inplace(b,D,to6);m=b.alloc_bits(3);b.hmr(D[3:],m);b.free(D[3:])\n    mt=b.alloc_bits(6);b.bit_store(mt,False);b.cx(m,mt[3:]);canon=classical_transpose(b,mt,to6);b.free(mt);b.free(m)\n    to12,frm12,K12=tower(12);linear_inplace(b,N,to12);ar._phase_norm(b,N,canon,K12);linear_inplace(b,N,frm12);b.free(canon)\n    return D[:3]\n\ndef erase_N_cache(b,t,N,C):\n    """Joint cleanup29T: norm(t)15 +sharedprefix9 +fivequantumCCZ."""\n    to12,frm12,K12=tower(12);linear_inplace(b,t,to12);linear_inplace(b,N,to12)\n    nm=b.alloc_bits(12);b.hmr(N,nm);b.free(N);frm6=tower(6)[1];f6=field(6)\n    for j in range(3):\n        for i in range(6):\n            for r in bits(f6.mul(frm6[j],1<<i)):\n                b.ccz(C[j],t[i],nm[r]);b.ccz(C[j],t[i+6],nm[r]);b.ccz(C[j],t[i+6],nm[r+6])\n    cm=b.alloc_bits(3);b.hmr(C,cm);b.free(C)\n    u=b.alloc_qubits(6);iso.norm_xor(b,t,u,K12);phase_partial_inverse6(b,u,t,nm,cm);b.free(nm);b.free(cm)\n    ar._erase_norm(b,t,u,K12);b.free(u);linear_inplace(b,t,frm12)\n\ndef inverse_inplace_reverse39(b,x,mask):\n    assert len(x)==24;k=12;to,frm,K=tower(24);linear_inplace(b,x,to)\n    N=b.alloc_qubits(12);iso.norm_xor(b,x,N,K);t,D=iso.inverse12_keep_norm(b,N);C=compress_D(b,N,D)\n    out=b.alloc_qubits(12);mul_xor_noscratch(b,x[12:],t,out)\n    erase_product(b,out,N,x[12:]);mul_xor_noscratch(b,x[:12],t,x[12:]);b.cx(out,x[12:])\n    b.cx(out,x[12:]);erase_product(b,x[12:],N,x[:12]);b.cx(out,x[12:])\n    b.swap(x[:12],x[12:]);b.swap(x[12:],out);b.free(out)\n    erase_N_cache(b,t,N,C);ar._erase_norm(b,x,t,K);b.free(t);linear_inplace(b,x,frm);b.cz(x,mask);b.free(mask)\n\ndef inverse24_forward39(b,x,K=None):return iso.inverse24_forward20(b,x,K)\n\ndef inverse24_reverse39(b,x,mask,K=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols);inverse_inplace_reverse39(b,x,mask);linear_inplace(b,x,frm)\n', 'tasks/dqi/fused/source/exp/inverse_width/partial5.json': '{\n  "prefix_AND": 9,\n  "final_AND": 5,\n  "outputs": [\n    1630906,\n    1131460,\n    458752\n  ],\n  "gates": [\n    [\n      16,\n      8192\n    ],\n    [\n      32,\n      16384\n    ],\n    [\n      80,\n      40960\n    ],\n    [\n      11240,\n      24088\n    ],\n    [\n      13088,\n      50928\n    ]\n  ],\n  "classes": [\n    64,\n    4032,\n    512,\n    1664,\n    3072\n  ],\n  "input_basis": "tower6 GF8 pair",\n  "output_bits": [\n    3,\n    4,\n    5\n  ],\n  "truth_table_values": 64,\n  "truth_table_errors": 0\n}\n', 'tasks/dqi/fused/source/exp/inverse_width_subspaces/inverse19.json': '{\n  "total_AND": 19,\n  "prefix_AND": 9,\n  "final_AND": 10,\n  "scale": 6,\n  "classes": [\n    65,\n    2048,\n    63,\n    1885,\n    1154,\n    324,\n    2208,\n    1552,\n    1690,\n    3839\n  ],\n  "gates": [\n    [\n      2,\n      8192\n    ],\n    [\n      48,\n      24576\n    ],\n    [\n      36,\n      16384\n    ],\n    [\n      13508,\n      17080\n    ],\n    [\n      8302,\n      49166\n    ],\n    [\n      8200,\n      32784\n    ],\n    [\n      24688,\n      32886\n    ],\n    [\n      24668,\n      40972\n    ],\n    [\n      21468,\n      40156\n    ],\n    [\n      16448,\n      32772\n    ]\n  ],\n  "outputs": [\n    29546338,\n    46486470,\n    11297528,\n    54342028,\n    20694494,\n    39494616\n  ],\n  "truth_table_points": 64,\n  "truth_table_errors": 0,\n  "raw_norm_control_rank": 4,\n  "raw_norm_control_basis": [\n    41,\n    5,\n    14,\n    16\n  ],\n  "controls_only_through_norm": false\n}\n', 'tasks/dqi/fused/source/exp/inverse_width_subspaces/kernel19.py': '"""New19-AND GF64 inverse at norm isotopy6; I24 pair190/219T,39private.\n\nNo imported module is changed. The six-bit XOR inverse supports arbitrary\noutput targets; initialized totalQ21 is wider than iso20, but nested I24 peaks\nremain36forward/39reverse because the larger subcall is not the global peak.\n"""\nimport sys,json\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse_search import inverse20 as ref\nfrom exp.inverse_search import inverse20_iso as iso\nfrom exp.inverse_search.inverse20_iso import *\nfrom exp.inverse_width.kernel import compress_D,erase_N_cache\nDAG19=json.loads(Path(__file__).with_name(\'inverse19.json\').read_text())\n\ndef raw_forms19():\n    f=field(3);scale=6;inv=f.invert(scale);out=[]\n    for mask,v in tensor(3):\n        left=sum(((mask&f.mul(scale,1<<i)).bit_count()&1)<<(1+i)for i in range(3))\n        right=sum(((mask&f.mul(inv,1<<i)).bit_count()&1)<<(4+i)for i in range(3));out.append((left,right))\n    return out\n\ndef inverse6_xor19(b,x,z=None,keep_norm=False):\n    assert len(x)==6;to,frm,K=tower(6);linear_inplace(b,x,to)\n    if z is not None:linear_inplace(b,z,to)\n    g=b.alloc_qubits(6)\n    for i,(left,right)in enumerate(raw_forms19()):\n        with ref.expose_pair(b,x,left,right)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(x)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms);source=prefix+list(w)\n    if z is None:z=b.alloc_qubits(6)\n    for j,e in enumerate(DAG19[\'outputs\']):ref.form_xor(b,source,e&65535,z[j])\n    for i,(left,right)in enumerate(DAG19[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(DAG19[\'outputs\']));assert v\n        with ref.expose_pair(b,source,left,right)as(p,q):rankone_xor(b,p,q,z,v)\n    wm=b.alloc_bits(3);b.hmr(w,wm);b.free(w);ref.phase_inverse3_virtual(b,prefix,forms,wm);b.free(wm)\n    kept=None\n    if keep_norm:\n        kept=b.alloc_qubits(3)\n        for j,f in enumerate(forms):ref.form_xor(b,prefix,f,kept[j])\n    for i,(left,right)in reversed(list(enumerate(raw_forms19()))):\n        with ref.expose_pair(b,x,left,right)as(p,q):b.del_and(p,q,g[i])\n    b.free(g);linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n    return (z,kept)if keep_norm else z\n\n# Rebind only this module\'s private copies of the recursive schedules.\nfrom types import FunctionType\ninverse6_xor20=inverse6_xor19\nfor _name in(\'inverse_initialized\',\'inverse12_keep_norm\',\'inverse_inplace_forward20\'):\n    _old=getattr(ref,_name);globals()[_name]=FunctionType(_old.__code__,globals(),_name,_old.__defaults__,_old.__closure__)\n\ndef inverse24_forward19(b,x,K=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols)\n    mask=inverse_inplace_forward20(b,x);linear_inplace(b,x,frm);return mask\n\ndef inverse_inplace_reverse19(b,x,mask):\n    to,frm,K=tower(24);linear_inplace(b,x,to);N=b.alloc_qubits(12);iso.norm_xor(b,x,N,K)\n    t,D=inverse12_keep_norm(b,N);C=compress_D(b,N,D);out=b.alloc_qubits(12);mul_xor_noscratch(b,x[12:],t,out)\n    erase_product(b,out,N,x[12:]);mul_xor_noscratch(b,x[:12],t,x[12:]);b.cx(out,x[12:])\n    b.cx(out,x[12:]);erase_product(b,x[12:],N,x[:12]);b.cx(out,x[12:])\n    b.swap(x[:12],x[12:]);b.swap(x[12:],out);b.free(out)\n    erase_N_cache(b,t,N,C);ar._erase_norm(b,x,t,K);b.free(t);linear_inplace(b,x,frm);b.cz(x,mask);b.free(mask)\n\ndef inverse24_reverse19(b,x,mask,K=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols);inverse_inplace_reverse19(b,x,mask);linear_inplace(b,x,frm)\n', 'tasks/dqi/fused/source/exp/jet24/kernel.py': '"""Completed dual-number inversion reference; allinput, explicitextension basis.\nJ(u,v)=(inv0u,v*(inv0u+[u0])²). Forward290T; closedpair686T/49private.\nNo persistent quantum cache; return only two classical masks.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse12_escape import joint as base\nfrom exp.inverse12_escape import zero1\nfrom exp.inverse12_escape.joint import *\nfrom exp.mul99.kernels import mul_xor99_zero,phase_product_quantum99\nfrom exp.arithmetic.lowwidth import classical_transpose,phase_product_quantum\n\n\ndef phase_zero_virtual(b,prefix,forms,w,mask):\n    """zero(prefix norm) phase,0T while its norm3 andinverse3 forms live."""\n    b.z(mask);source=list(prefix)+list(w);f=field(3)\n    for i in range(3):\n        for j in range(3):\n            if f.mul(1<<i,1<<j)&1:\n                with ref.expose_pair(b,source,forms[i],1<<(1+len(prefix)+j))as(p,q):b.ccz(p,q,mask)\n\n\ndef phase_outputs_classical(b,source,outputs,gates,m):\n    for j,e in enumerate(outputs):\n        if e&1:b.z(m[j])\n        for r in bits((e&65535)>>1):b.cz(source[r],m[j])\n    cb=b.alloc_bits(1)[0];b.bit_store(cb,False)\n    for i,(l,r)in enumerate(gates):\n        js=[j for j,e in enumerate(outputs)if e>>(16+i)&1]\n        for j in js:b.cx(m[j],cb)\n        with ref.expose_pair(b,source,l,r)as(p,q):b.ccz(p,q,cb)\n        for j in reversed(js):b.cx(m[j],cb)\n    b.free(cb)\n\n\ndef phase_inverse12_quantum64(b,x,L,zero_mask=None):\n    """Phase L.inv0(x),quantumL12,64T withpaid-innerprefixretained."""\n    to12,frm12,K12=tower(12);linear_inplace(b,x,to12);T=[sum(((frm12[i]>>j)&1)<<i for i in range(12))for j in range(12)];linear_inplace(b,L,T)\n    D=b.alloc_qubits(6);norm_xor_noscratch(b,x,D,K12);to6,frm6,K6=tower(6);linear_inplace(b,D,to6);g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,D,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(D)+list(g);forms=ref.norm_forms(K6);w=ref.inverse3_virtual(b,prefix,forms);source=prefix+list(w);out=b.alloc_qubits(6)\n    for j,e in enumerate(basic.DAG19[\'outputs\']):ref.form_xor(b,source,e&65535,out[j])\n    for i,(l,r)in enumerate(basic.DAG19[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(basic.DAG19[\'outputs\']))\n        with ref.expose_pair(b,source,l,r)as(p,q):rankone_xor(b,p,q,out,v)\n    linear_inplace(b,out,frm6);b.cx(x[6:],x[:6]);phase_product_quantum(b,x[:6],out,L[:6]);phase_product_quantum(b,x[6:],out,L[6:]);b.cx(x[6:],x[:6])\n    if zero_mask is not None:phase_zero_virtual(b,prefix,forms,w,zero_mask)\n    m=b.alloc_bits(6);b.hmr(out,m);b.free(out);mt=classical_transpose(b,m,frm6);b.free(m);phase_outputs_classical(b,source,basic.DAG19[\'outputs\'],basic.DAG19[\'gates\'],mt);b.free(mt)\n    wm=b.alloc_bits(3);b.hmr(w,wm);b.free(w);ref.phase_inverse3_virtual(b,prefix,forms,wm);b.free(wm)\n    for i,(l,r)in reversed(list(enumerate(basic.raw_forms19()))):\n        with ref.expose_pair(b,D,l,r)as(p,q):b.del_and(p,q,g[i])\n    b.free(g);linear_inplace(b,D,frm6);ar._erase_norm(b,x,D,K12);b.free(D);linear_inplace(b,L,inverse_columns(T));linear_inplace(b,x,frm12)\n\n\ndef phase_inverse24_classical106(b,x,mask,zero_mask=None):\n    """Phase mask.inv0(x)+zero_mask*[x0],106T/45private."""\n    to,frm,K=tower(24);linear_inplace(b,x,to);mt=classical_transpose(b,mask,frm);N=b.alloc_qubits(12);norm_xor_noscratch(b,x,N,K);L=b.alloc_qubits(12);f=field(12)\n    def coeff():\n        for j in range(12):\n            for i in range(12):\n                for r in bits(f.mul(1<<i,1<<j)):\n                    b.ccx(mt[r],x[i],L[j]);b.ccx(mt[r],x[i+12],L[j]);b.ccx(mt[r+12],x[i+12],L[j])\n    coeff();phase_inverse12_quantum64(b,N,L,zero_mask);coeff();b.free(L);ar._erase_norm(b,x,N,K);b.free(N);b.free(mt);linear_inplace(b,x,frm)\n\n\ndef inverse24_initialized_zero191(b,x):\n    to,frm,K=tower(24);linear_inplace(b,x,to);N=b.alloc_qubits(12);norm_xor_noscratch(b,x,N,K);t,F=zero1.inverse_zero(b,N);ar._erase_norm(b,x,N,K);b.free(N)\n    z=b.alloc_qubits(24);b.cx(x[12:],x[:12]);mul_xor_noscratch(b,x[:12],t,z[:12]);mul_xor_noscratch(b,x[12:],t,z[12:]);b.cx(x[12:],x[:12]);ar._erase_norm(b,z,t,K);b.free(t);linear_inplace(b,x,frm);linear_inplace(b,z,frm);return z,F\n\n\ndef safe_square(b,x,F,undo=False):\n    sq=square_columns(24)\n    if undo:linear_inplace(b,x,inverse_columns(sq));b.cx(F,x[0])\n    else:b.cx(F,x[0]);linear_inplace(b,x,sq)\n\n\ndef forward_canonical(b,u,v):\n    z,F=inverse24_initialized_zero191(b,u);out=b.alloc_qubits(24);safe_square(b,z,F);mul_xor99_zero(b,v,z,out);safe_square(b,z,F,True)\n    m=b.alloc_bits(24);b.hmr(v,m);safe_square(b,u,F);phase_product_quantum99(b,out,u,m);safe_square(b,u,F,True);b.free(m);b.swap(v,out);b.free(out)\n    mu=b.alloc_bits(24);fz=b.alloc_bits(1);b.hmr(u,mu);b.swap(u,z);b.free(z);b.hmr(F,fz[0]);b.free(F);return mu,fz\n\n\ndef jet_forward(b,u,v,K=None):\n    """290T/49private;returns classical(mu24,zero1),noquantumcache."""\n    cols,frm,_=extension_columns(K);linear_inplace(b,u,cols);linear_inplace(b,v,cols);token=forward_canonical(b,u,v);linear_inplace(b,u,frm);linear_inplace(b,v,frm);return token\n\n\ndef jet_reverse(b,u,v,token,K=None):\n    """396T/49private closedreverse ofjet_forward;106T inversephase."""\n    cols,frm,_=extension_columns(K);linear_inplace(b,u,cols);linear_inplace(b,v,cols);mu2,fz2=forward_canonical(b,u,v);mu,fz=token;b.cz(u,mu);b.free(mu);b.cx(fz,fz2);b.free(fz);phase_inverse24_classical106(b,u,mu2,fz2[0]);b.free(mu2);b.free(fz2);linear_inplace(b,u,frm);linear_inplace(b,v,frm)\n', 'tasks/dqi/fused/source/exp/mask_projection/kernel.py': '"""Classical mask projection onto the70 syndrome coordinates,0 Toffolis.\n\nInterpolate a degree<=70 polynomial F with F(0)=0 through70 selected masks\nin the trace-dual field basis. Pay Tr(sum S_k F_(k+1)) directly, subtract\nF\'s evaluations from every mask, and omit the70 now-zero scan positions.\nAn additive FFT avoids a dense4095x70 classical CNOT transform.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.decoder.common import *\n\nSELECTED=tuple(range(70))\n\ndef ev(poly,x):\n    y=0\n    for a in reversed(poly):y=mul(y,x)^a\n    return y\n\ndef tr(x):\n    y=0\n    for _ in range(12):y^=x;x=mul(x,x)\n    assert y in(0,1)\n    return y\n\n@lru_cache(None)\ndef trace_columns():\n    return tuple(sum(tr(mul(1<<i,1<<j))<<i for i in range(12))for j in range(12))\n\n@lru_cache(None)\ndef subspace_polynomials():\n    out=[[0,1]]\n    for j in range(12):\n        s=out[-1];d=ev(s,1<<j);assert d\n        z=[0]*(2*len(s)-1)\n        for i,c in enumerate(s):z[2*i]^=mul(c,c);z[i]^=mul(d,c)\n        out.append(z)\n    return out\n\n@lru_cache(None)\ndef novel_basis(active):\n    ss=subspace_polynomials();out=[]\n    for i in range(active):\n        p=[1]\n        for j in range(i.bit_length()):\n            if i>>j&1:p=pmul(p,ss[j])\n        assert len(p)==i+1 and p[-1]==1\n        out.append(tuple(p))\n    return tuple(out)\n\ndef to_novel(b,words,active,reverse=False):\n    basis=novel_basis(active)\n    for i in(range(1,active)if reverse else range(active-1,0,-1)):\n        for j,c in enumerate(basis[i][:-1]):cx(b,words[i],words[j],c)\n\ndef additive_fft(b,words,m=12,offset=0,active=71,reverse=False):\n    """Novel-basis coefficients -> values at offset+span(1,...,2^(m-1)).\n\n    Only the first active coefficients may be nonzero on entry. The inverse\n    restores the unused classical wires tozero. Output order is numeric field\n    representation, not primitive-element exponent order.\n    """\n    assert len(words)==1<<m and 0<active<=len(words)\n    if not m:return\n    h=1<<(m-1);lo,hi=words[:h],words[h:];high=max(0,active-h)\n    nextactive=min(active,h);ss=subspace_polynomials()[m-1]\n    c=ev(ss,offset);d=ev(ss,1<<(m-1));assert d\n    if reverse:\n        additive_fft(b,lo,m-1,offset,nextactive,True)\n        additive_fft(b,hi,m-1,offset^(1<<(m-1)),nextactive,True)\n    for j in range(nextactive):\n        if j>=high:b.cx(lo[j],hi[j])\n        elif not reverse:\n            cx(b,hi[j],lo[j],c);scale(b,hi[j],d);b.cx(lo[j],hi[j])\n        else:\n            b.cx(lo[j],hi[j]);scale(b,hi[j],inv(d));cx(b,hi[j],lo[j],c)\n    if not reverse:\n        additive_fft(b,lo,m-1,offset,nextactive)\n        additive_fft(b,hi,m-1,offset^(1<<(m-1)),nextactive)\n\n@lru_cache(None)\ndef interpolation_matrix():\n    alpha=gf.primitive_element\n    return tuple(tuple(power(power(alpha,j),k+1)for k in range(70))for j in SELECTED)\n\ndef residual_masks(b,E,F):\n    beta=1;T=trace_columns()\n    for mask in E:\n        ar.linear_xor(b,F[beta],mask,T);beta=mul(beta,gf.primitive_element)\n    assert beta==1\n\ndef project(b,S,E):\n    assert len(S)==70 and len(E)==4095\n    F=[b.alloc_bits(12)for _ in range(4096)]\n    b.bit_store(combine(F),False)\n    Ti=ar.inverse_columns(trace_columns())\n    for k,j in enumerate(SELECTED):\n        b.cx(E[j],F[k+1]);ar.linear_inplace(b,F[k+1],Ti)\n    linear_words(b,F[1:71],interpolation_matrix(),reverse=True)\n    for s,f in zip(S,F[1:71]):\n        ar.linear_inplace(b,f,trace_columns());b.cz(s,f);ar.linear_inplace(b,f,Ti)\n    to_novel(b,F,71);additive_fft(b,F)\n    residual_masks(b,E,F)\n    return F\n\ndef restore(b,E,F):\n    residual_masks(b,E,F)\n    # F is classical deterministic working memory, not quantum garbage.\n    for word in F:b.free(word)\n', 'tasks/dqi/fused/source/exp/mul99/kernels.py': '"""GF24 multiplication: GF16 degree6, 11 evaluations times rank9 =99.\n\nSeparate experimental implementation; does not edit the stable arithmetic API.\n"""\nfrom functools import lru_cache\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2]\nsys.path.insert(0,str(ROOT/\'exp/arithmetic\'))\nimport field as ar\nfrom field import *\n\n\ndef extension_tensor(n,k,d,subterms):\n    f=field(n);sub=field(k);gen=f.pow(f.primitive_element,(2**n-1)//(2**k-1))\n    def ev(v,p):\n        out=0\n        for i in reversed(range(k+1)):out=f.mul(out,p)^((v>>i)&1)\n        return out\n    root=next(f.pow(gen,i) for i in range(1,2**k) if ev(sub.modulus,f.pow(gen,i))==0)\n    embed=[f.pow(root,i) for i in range(k)]\n    cols=[f.mul(v,f.pow(f.primitive_element,j)) for j in range(d) for v in embed]\n    inv=inverse_columns(cols);masks=[]\n    for point in list(range(2*d-2))+[None]:\n        values=[]\n        for co in inv:\n            if point is None:value=(co>>(k*(d-1)))&((1<<k)-1)\n            else:\n                value=0\n                for j in reversed(range(d)):value=sub.mul(value,point)^((co>>(k*j))&((1<<k)-1))\n            values.append(value)\n        for mask,_ in subterms:\n            masks.append(sum(((v&mask).bit_count()%2)<<j for j,v in enumerate(values)))\n    equations=[]\n    for i in range(n):\n        for j in range(i,n):\n            row=sum((((m>>i)&1)&((m>>j)&1))<<r for r,m in enumerate(masks))\n            equations.append((row,f.mul(1<<i,1<<j)))\n    vectors=ar._solve_output_vectors(equations,len(masks));assert all(vectors)\n    return tuple(zip(masks,vectors))\n\n\n@lru_cache(None)\ndef tensor4():return extension_tensor(4,2,2,tensor(2))\n\n\n@lru_cache(None)\ndef tensor99():\n    out=extension_tensor(24,4,6,tensor4());assert len(out)==99\n    return out\n\n\ndef multiplier(b,x,y,z,terms):\n    assert not(set(map(str,x))&set(map(str,y)) or set(map(str,x))&set(map(str,z)) or set(map(str,y))&set(map(str,z)))\n    for mask,v in terms:\n        ix=list(bits(mask));iz=list(bits(v));p=ix[0];t=iz[0]\n        for i in ix[1:]:b.cx(x[i],x[p]);b.cx(y[i],y[p])\n        for j in iz[1:]:b.cx(z[t],z[j])\n        b.ccx(x[p],y[p],z[t])\n        for j in reversed(iz[1:]):b.cx(z[t],z[j])\n        for i in reversed(ix[1:]):b.cx(y[i],y[p]);b.cx(x[i],x[p])\n\n\ndef mul_xor99_zero(b,x,y,z):return multiplier(b,x,y,z,tensor99())\n\n\n@lru_cache(None)\ndef tensor99_extension(K=None):\n    cols,invcols,_=extension_columns(K)\n    return tuple((sum(((mask&col).bit_count()%2)<<j for j,col in enumerate(cols)),apply(invcols,v))\n                 for mask,v in tensor99())\n\n\ndef mul24_xor99_zero(b,x,y,z,K=None):return multiplier(b,x,y,z,tensor99_extension(K))\n\n\ndef phase_product_quantum99(b,x,y,mask):\n    for m,v in tensor99():\n        ix=list(bits(m));iz=list(bits(v));p=ix[0];t=iz[0]\n        for i in ix[1:]:b.cx(x[i],x[p]);b.cx(y[i],y[p])\n        for j in iz[1:]:b.cx(mask[j],mask[t])\n        b.ccz(x[p],y[p],mask[t])\n        for j in reversed(iz[1:]):b.cx(mask[j],mask[t])\n        for i in reversed(ix[1:]):b.cx(y[i],y[p]);b.cx(x[i],x[p])\n\n\n# Stable public spellings: no zero-target promise is required.\nmul_xor99=mul_xor99_zero\nmul24_xor99=mul24_xor99_zero\n', 'tasks/dqi/fused/source/exp/mul99/phase36.py': '"""Inverse12 phase with12 scratch and at most36 EXECUTED Toffolis.\n\nThe coefficient mask is classical. A classical first-nonzero pivot exposes\nits parity in an existing input bit, avoiding the13th quantum wire.\n"""\ntry:\n    from .kernels import *\nexcept ImportError:\n    from kernels import *\nsys.path.insert(0,str(ROOT))\nfrom exp.noscratch.kernels import inverse3_noscratch\nfrom lowwidth import classical_transpose\n\n\ndef phase_classical_parity(b,a,c,x,co):\n    """CCZ(a,c,parity(co*x)), zero quantum scratch, <=1 executed CCZ."""\n    first=b.alloc_bits(len(x));b.bit_store(first,False)\n    seen=b.alloc_bits(1)[0];b.bit_store(seen,False)\n    for j in range(len(x)):\n        b.x(seen);b.push_condition(seen)\n        b.bit_store(first[j],True,control=co[j])\n        b.pop_condition();b.x(seen)\n        b.bit_store(seen,True,control=co[j])\n    for j in range(len(x)):\n        b.push_condition(first[j])\n        for k in range(len(x)):\n            if k!=j:b.ccx(co[k],x[k],x[j])\n        b.ccz(a,c,x[j])\n        for k in reversed(range(len(x))):\n            if k!=j:b.ccx(co[k],x[k],x[j])\n        b.pop_condition()\n    b.free(seen);b.free(first)\n\n\ndef phase_inverse36_pivot(b,x,mask):\n    """Phase mask dot inverse0(x): <=36 mean T /12 quantum scratch.\n\n    Static count168T due to12 mutually exclusive classical branches per CCZ.\n    No quantum input controls branch selection; all branches use HMR-mask bits.\n    """\n    assert len(x)==len(mask)==12\n    to12,frm12,K12=tower(12);linear_inplace(b,x,to12)\n    mt=classical_transpose(b,mask,frm12)\n    D=b.alloc_qubits(6);ar._norm_xor(b,x,D,K12)\n    to6,frm6,K6=tower(6);linear_inplace(b,D,to6)\n    E=b.alloc_qubits(3);ar._norm_xor(b,D,E,K6);w=inverse3_noscratch(b,E)\n    f6=field(6)\n    for segment in(0,1):\n        if segment==0:b.cx(D[3:],D[:3])\n        controls=D[:3] if segment==0 else D[3:]\n        for m,v in tensor(3):\n            h=apply(frm6,v<<(3*segment));co=b.alloc_bits(12);b.bit_store(co,False)\n            for j in range(6):\n                for r in bits(f6.mul(h,1<<j)):\n                    b.cx(mt[r],co[j]);b.cx(mt[r],co[j+6]);b.cx(mt[r+6],co[j+6])\n            ix=list(bits(m));p=ix[0]\n            for j in ix[1:]:b.cx(controls[j],controls[p]);b.cx(w[j],w[p])\n            phase_classical_parity(b,controls[p],w[p],x,co)\n            for j in reversed(ix[1:]):b.cx(w[j],w[p]);b.cx(controls[j],controls[p])\n            b.free(co)\n        if segment==0:b.cx(D[3:],D[:3])\n    erase_inverse(b,E,w);b.free(w)\n    ar._erase_norm(b,D,E,K6);b.free(E);linear_inplace(b,D,frm6)\n    ar._erase_norm(b,x,D,K12);b.free(D);linear_inplace(b,x,frm12);b.free(mt)\n', 'tasks/dqi/fused/source/exp/mul99/schedules.py': '"""Explicit compositions; no patching or edits of existing arithmetic modules."""\ntry:\n    from .kernels import *\nexcept ImportError:\n    from kernels import *\nsys.path.insert(0,str(ROOT))\nfrom exp.noscratch.schedules import norm_xor_noscratch,inverse_stream_noscratch\nfrom exp.noscratch.phase39 import phase_inverse39\nfrom exp.noscratch.kernels import mul_xor_noscratch\nfrom division import phase_product24,zero_flag,phase_zero\ntry:\n    from .phase36 import phase_inverse36_pivot\nexcept ImportError:\n    from phase36 import phase_inverse36_pivot\n\n\ndef multiply_forward99(b,x,y,K=None):\n    """x*=y:99T/24scratch, old-x phase deferred. Promise x=0 when y=0."""\n    z=b.alloc_qubits(24);mul24_xor99(b,x,y,z,K)\n    m=b.alloc_bits(24);b.hmr(x,m);b.swap(x,z);b.free(z);return m\n\n\ndef divide_lowwidth372(b,x,y,K=None):\n    """x/=y, preserves y:372T/24scratch. Promise x=0 when y=0."""\n    return _divide_lowwidth(b,x,y,K,phase_inverse39)\n\n\ndef divide_lowwidth369(b,x,y,K=None):\n    """Same division, <=369 EXECUTED T /24scratch; static count501T."""\n    return _divide_lowwidth(b,x,y,K,phase_inverse36_pivot)\n\n\ndef _divide_lowwidth(b,x,y,K,phase_kernel):\n    if K is None:K=extension_columns()[2]\n    b.cx(y[12:],y[:12]);oldmask=multiply_forward99(b,x,y,K);b.cx(y[12:],y[:12])\n    u=b.alloc_qubits(12);norm_xor_noscratch(b,y,u,K)\n    nm=inverse_stream_noscratch(b,u);ar._phase_norm(b,y,nm,K);b.free(nm)\n    masks=[]\n    for offset in(0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor_noscratch(b,half,u,out)\n        m=b.alloc_bits(12);b.hmr(half,m);masks.append(m);b.swap(half,out);b.free(out)\n    um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,y,N,K)\n    for offset,m in zip((0,12),masks):phase_product(b,x[offset:offset+12],N,m);b.free(m)\n    phase_kernel(b,N,um);b.free(um)\n    ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n\n\ndef divide_zero_bridge417(b,x,y,K=None,extra_zero_mask=None):\n    """x/=y if y!=0, else identity:417T/25scratch, all inputs exact."""\n    flag=zero_flag(b,y);b.cx(flag,y[0])\n    divide_lowwidth372(b,x,y,K)\n    b.cx(flag,y[0])\n    mask=b.alloc_bits(1);b.hmr(flag,mask[0]);b.free(flag)\n    if extra_zero_mask is not None:b.cx(extra_zero_mask,mask[0])\n    phase_zero(b,y,mask[0]);b.free(mask)\n\n\ndef divide_zero_bridge414(b,x,y,K=None,extra_zero_mask=None):\n    """All-input exact zero bridge: <=414 EXECUTED T /25scratch."""\n    flag=zero_flag(b,y);b.cx(flag,y[0])\n    divide_lowwidth369(b,x,y,K)\n    b.cx(flag,y[0])\n    mask=b.alloc_bits(1);b.hmr(flag,mask[0]);b.free(flag)\n    if extra_zero_mask is not None:b.cx(extra_zero_mask,mask[0])\n    phase_zero(b,y,mask[0]);b.free(mask)\n\n\ndef multiply_zero_forward99(b,x,y,K=None):\n    """x*=y if y!=0, else identity:146T/25scratch; deferred masks."""\n    flag=zero_flag(b,y)\n    z=b.alloc_qubits(24);mul24_xor99(b,x,y,z,K);b.ccx(flag,x,z)\n    old=b.alloc_bits(24);b.hmr(x,old);b.swap(x,z);b.free(z)\n    mask=b.alloc_bits(1);b.hmr(flag,mask[0]);b.free(flag)\n    return old,mask[0]\n', 'tasks/dqi/fused/source/exp/noscratch/kernels.py': '"""Scratch-free rankone bilinear accumulation; no coremodule edits."""\nimport sys\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT/\'exp/arithmetic\'))\nimport field as ar\nfrom field import *\n\ndef rankone_xor(b,x,y,z,vector):\n    """z^=vector*(xANDy), arbitraryz, zeroancilla, oneCCX.\n\n    x,y arebits; vector is nonzero bitmaskoverz. Inputsdisjointfromtarget.\n    """\n    out=list(bits(vector));p=out[0]\n    for j in out[1:]:b.cx(z[p],z[j])\n    b.ccx(x,y,z[p])\n    for j in reversed(out[1:]):b.cx(z[p],z[j])\n\ndef tensor_xor_noscratch(b,x,y,z,terms):\n    assert len(x)==len(y)==len(z)\n    assert not(set(map(str,x))&set(map(str,y))or set(map(str,x))&set(map(str,z))or set(map(str,y))&set(map(str,z)))\n    for mask,vector in terms:\n        ix=list(bits(mask));p=ix[0]\n        for j in ix[1:]:b.cx(x[j],x[p]);b.cx(y[j],y[p])\n        rankone_xor(b,x[p],y[p],z,vector)\n        for j in reversed(ix[1:]):b.cx(y[j],y[p]);b.cx(x[j],x[p])\n\ndef mul_xor_noscratch(b,x,y,z):\n    """DisjointGF2^n productaccumulate; n2/3/6/12, rank3/6/15/42,0scratch."""\n    tensor_xor_noscratch(b,x,y,z,tensor(len(x)))\n\ndef mul24_xor_noscratch(b,x,y,z,K=None):\n    """ExplicitquadraticGF24 productaccumulate, arbitrarytarget,105T/0scratch."""\n    assert len(x)==24;tensor_xor_noscratch(b,x,y,z,ar.tensor24_extension(K))\n\ndef inverse3_xor_noscratch(b,x,z):\n    """z^=inverseGF8(x), inverse0=0, arbitrarytarget,3T/0scratch."""\n    assert len(x)==len(z)==3 and not(set(map(str,x))&set(map(str,z)))\n    for mask,vector in enumerate(inverse3_anf()):\n        if not vector:continue\n        ix=list(bits(mask))\n        if len(ix)==1:\n            for j in bits(vector):b.cx(x[ix[0]],z[j])\n        elif len(ix)==2:rankone_xor(b,x[ix[0]],x[ix[1]],z,vector)\n        else:raise ValueError((mask,vector))\n\ndef inverse3_noscratch(b,x,target=None):\n    """Freshcleantargetinverse, or XORintogiventarget,3T/0privatequbits."""\n    if target is None:target=b.alloc_qubits(3)\n    inverse3_xor_noscratch(b,x,target);return target\n', 'tasks/dqi/fused/source/exp/noscratch/phase39.py': '"""Inverse GF4096 phase with twelve private qubits, 39 Toffolis.\n\nOnly imports core modules; this plugin never patches or edits them.\n"""\ntry:\n    from .kernels import *\nexcept ImportError:\n    from kernels import *\nfrom lowwidth import classical_transpose\n\n\ndef norm_bit_xor(b,x,z,K,bit=0):\n    """XOR one GF8 norm coordinate into arbitrary single-qubit z, 3T."""\n    assert len(x)==6\n    a,c=x[:3],x[3:];f=field(3)\n    for j,v in enumerate(square_columns(3)):\n        if v>>bit&1:b.cx(a[j],z)\n        if f.mul(K,v)>>bit&1:b.cx(c[j],z)\n    for mask,vector in tensor(3):\n        if not(vector>>bit&1):continue\n        ix=list(bits(mask));p=ix[0]\n        for j in ix[1:]:b.cx(a[j],a[p]);b.cx(c[j],c[p])\n        b.ccx(a[p],c[p],z)\n        for j in reversed(ix[1:]):b.cx(c[j],c[p]);b.cx(a[j],a[p])\n\n\ndef phase_inverse39(b,x,mask):\n    """Apply (-1)^dot(mask,inverse(x)); inv(0)=0, x unchanged.\n\n    E[0] is HMR-erased and its quadratic norm phase repaid before the\n    mask-parity qubit is allocated. Once that qubit is freed, E[0] is\n    reconstructed with 3 CCX, enabling the ordinary inverse erasure.\n    Peak D6 + E2 + w3 + parity1 = 12 private quantum wires.\n    """\n    assert len(x)==len(mask)==12\n    to12,frm12,K12=tower(12);linear_inplace(b,x,to12)\n    mt=classical_transpose(b,mask,frm12)\n    D=b.alloc_qubits(6);ar._norm_xor(b,x,D,K12)\n    to6,frm6,K6=tower(6);linear_inplace(b,D,to6)\n    E=b.alloc_qubits(3);ar._norm_xor(b,D,E,K6)\n    w=inverse3_noscratch(b,E)\n    em=b.alloc_bits(3);b.bit_store(em,False);b.hmr(E[:1],em[:1])\n    ar._phase_norm(b,D,em,K6);b.free(em);b.free(E[:1])\n    anc=b.alloc_qubits(1)[0];f6=field(6)\n    def coefficient(h):\n        for j in range(6):\n            for r in bits(f6.mul(h,1<<j)):\n                b.ccx(mt[r],x[j],anc)\n                b.ccx(mt[r],x[j+6],anc)\n                b.ccx(mt[r+6],x[j+6],anc)\n    for segment in (0,1):\n        if segment==0:b.cx(D[3:],D[:3])\n        controls=D[:3] if segment==0 else D[3:]\n        for m,v in tensor(3):\n            h=apply(frm6,v<<(3*segment));ix=list(bits(m));p=ix[0]\n            coefficient(h)\n            for j in ix[1:]:b.cx(controls[j],controls[p]);b.cx(w[j],w[p])\n            b.ccz(controls[p],w[p],anc)\n            for j in reversed(ix[1:]):b.cx(w[j],w[p]);b.cx(controls[j],controls[p])\n            coefficient(h)\n        if segment==0:b.cx(D[3:],D[:3])\n    b.free(anc)\n    e0=b.alloc_qubits(1);norm_bit_xor(b,D,e0[0],K6)\n    E=[e0[0],E[1],E[2]]\n    erase_inverse(b,E,w);b.free(w)\n    ar._erase_norm(b,D,E,K6);b.free(E);linear_inplace(b,D,frm6)\n    ar._erase_norm(b,x,D,K12);b.free(D)\n    linear_inplace(b,x,frm12);b.free(mt)\n', 'tasks/dqi/fused/source/exp/noscratch/schedules.py': '"""Reference compositions of the plugins; no changes to arithmetic modules."""\ntry:\n    from .phase39 import *\nexcept ImportError:\n    from phase39 import *\nfrom division import phase_product24\n\n\ndef norm_xor_noscratch(b,x,z,K):\n    k=len(z);f=field(k)\n    linear_xor(b,x[:k],z,square_columns(k))\n    linear_xor(b,x[k:],z,[f.mul(K,v)for v in square_columns(k)])\n    mul_xor_noscratch(b,x[:k],x[k:],z)\n\n\ndef inverse_stream_noscratch(b,x):\n    """Same streamed inverse T3/6/12/24=3/21/66/192, n scratch."""\n    n=len(x)\n    if n==3:\n        y=inverse3_noscratch(b,x);mask=b.alloc_bits(n)\n        b.hmr(x,mask);b.swap(x,y);b.free(y);return mask\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    N=b.alloc_qubits(k);norm_xor_noscratch(b,x,N,K)\n    nm=inverse_stream_noscratch(b,N);ar._phase_norm(b,x,nm,K);b.free(nm)\n    outb=b.alloc_qubits(k);mul_xor_noscratch(b,x[k:],N,outb)\n    raw=b.alloc_bits(n);b.hmr(x[k:],raw[k:])\n    mul_xor_noscratch(b,x[:k],N,x[k:]);b.cx(outb,x[k:]);b.hmr(x[:k],raw[:k])\n    b.swap(x[:k],x[k:]);b.swap(x[k:],outb);b.free(outb)\n    ar._erase_norm(b,x,N,K);b.free(N);linear_inplace(b,x,frm)\n    mask=classical_transpose(b,raw,to);b.free(raw);return mask\n\n\ndef multiply_forward_noscratch(b,x,y,K=None):\n    z=b.alloc_qubits(24);mul24_xor_noscratch(b,x,y,z,K)\n    m=b.alloc_bits(24);b.hmr(x,m);b.swap(x,z);b.free(z);return m\n\n\ndef divide_lowwidth378(b,x,y,K=None):\n    """378T/24 scratch, promise x=0 when y=0; preserves y."""\n    if K is None:K=extension_columns()[2]\n    b.cx(y[12:],y[:12]);oldmask=multiply_forward_noscratch(b,x,y,K);b.cx(y[12:],y[:12])\n    u=b.alloc_qubits(12);norm_xor_noscratch(b,y,u,K)\n    nm=inverse_stream_noscratch(b,u);ar._phase_norm(b,y,nm,K);b.free(nm)\n    masks=[]\n    for offset in(0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor_noscratch(b,half,u,out)\n        m=b.alloc_bits(12);b.hmr(half,m);masks.append(m);b.swap(half,out);b.free(out)\n    um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,y,N,K)\n    for offset,m in zip((0,12),masks):phase_product(b,x[offset:offset+12],N,m);b.free(m)\n    phase_inverse39(b,N,um);b.free(um)\n    ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n', 'tasks/dqi/fused/source/exp/partial_gauge/prefix.py': '"""Closed local q-prefix step with the historical swap zero chart.\n\nInputs (b,p,q) -> (q*b+p,p,q) when q!=0, or (p,b,q) whenq=0.\nForward65T, reverse152T using divide117; pair217T,29private peak.\nThe reverse is exact for all (b,p,q), not only adjacent-nonzero continuants.\n"""\ntry:\n    from .scalar import *\nexcept ImportError:\n    from scalar import *\n\ndef zero_hold(b,x):\n    b.x(x);lad=b.alloc_qubits(len(x)-1);b.init_and(x[0],x[1],lad[0])\n    for j in range(2,len(x)):b.init_and(lad[j-2],x[j],lad[j-1])\n    b.x(x);return lad\n\ndef zero_free(b,x,lad):\n    b.x(x)\n    for j in reversed(range(2,len(x))):b.del_and(lad[j-2],x[j],lad[j-1])\n    b.del_and(x[0],x[1],lad[0]);b.free(lad);b.x(x)\n\ndef forward(b,x,p,q,virtual_zero=None,after_product=None):\n    out=b.alloc_qubits(12);mul_xor0(b,x,q,out);b.cx(p,out)\n    if virtual_zero is not None:b.ccx(virtual_zero,p,out)\n    lad=zero_hold(b,q);b.cswap(lad[-1],x,p)\n    m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out)\n    if after_product is not None:after_product()\n    zero_free(b,q,lad);return m\n\ndef reverse(b,x,p,q,m):\n    lad=zero_hold(b,q);f=lad[-1];b.cswap(f,x,p);b.cx(p,x);b.ccx(f,p,x)\n    b.cx(f,q[0]);divide117(b,x,q);b.cx(f,q[0])\n    b.cz(x,m)\n    for j in range(12):b.ccz(f,x[j],m[j]);b.ccz(f,p[j],m[j])\n    b.free(m);zero_free(b,q,lad)\n\ndef zero_compact(b,x):\n    lad=zero_hold(b,x);b.x(x)\n    for j in reversed(range(2,len(x)-1)):b.del_and(lad[j-2],x[j],lad[j-1]);b.free(lad[j-1])\n    b.del_and(x[0],x[1],lad[0]);b.free(lad[0]);b.x(x)\n    return lad[-1]\n\ndef zero_compact_free(b,x,f):\n    b.x(x);lad=b.alloc_qubits(len(x)-2);b.init_and(x[0],x[1],lad[0])\n    for j in range(2,len(x)-1):b.init_and(lad[j-2],x[j],lad[j-1])\n    b.del_and(lad[-1],x[-1],f);b.free(f)\n    for j in reversed(range(2,len(x)-1)):b.del_and(lad[j-2],x[j],lad[j-1])\n    b.del_and(x[0],x[1],lad[0]);b.free(lad);b.x(x)\n\ndef reverse_compact(b,x,p,q,m,virtual_zero=None):\n    """162T reverse,19private; paired with forward gives227T/23private."""\n    f=zero_compact(b,q);b.cswap(f,x,p);b.cx(p,x);b.ccx(f,p,x)\n    if virtual_zero is not None:b.ccx(virtual_zero,p,x)\n    b.cx(f,q[0]);divide117(b,x,q);b.cx(f,q[0]);b.cz(x,m)\n    for j in range(12):b.ccz(f,x[j],m[j]);b.ccz(f,p[j],m[j])\n    b.free(m);zero_compact_free(b,q,f)\n\ndef boundary_forward(b,x,q,F,after_product=None):\n    """i=1 special chart; F=zero(original q1),66T,12private."""\n    b.cx(F,q[0]);out=b.alloc_qubits(12);mul_xor0(b,x,q,out)\n    m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out);b.cx(F,q[0]);b.x(x[0])\n    if after_product is not None:after_product()\n    b.ccx(F,x,q);b.cx(F,q[0]);b.ccx(F,q,x)\n    return m\n\ndef boundary_reverse(b,x,q,F,m):\n    """129T,18private; exact inverse, closes boundary_forward\'s mask."""\n    b.cswap(F,q,x);b.cx(F,q[0]);b.x(x[0]);b.cx(F,x[0])\n    b.cx(F,q[0]);divide117(b,x,q);b.cx(F,q[0]);b.cz(x,m);b.free(m)\n', 'tasks/dqi/fused/source/exp/partial_gauge/scalar.py': '"""Retained-norm GF12 division:117T,18 private, complete local HMR phase."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT/\'exp/arithmetic\'))\nfrom field import *\nimport field as ar\nfrom lowwidth import phase_inverse_generic6\n\ndef divide117(b,x,y):\n    """x/=y, y preserved; y=0 requires x=0. 117T/18 private qubits."""\n    f=field(12);to,frm,K=tower(12);conj=[f.frobenius(1<<i,6) for i in range(12)]\n    linear_inplace(b,y,conj);out=b.alloc_qubits(12);mul_xor0(b,x,y,out)\n    old=b.alloc_bits(12);b.hmr(x,old);b.swap(x,out);b.free(out);linear_inplace(b,y,conj)\n    N=b.alloc_qubits(6);linear_inplace(b,y,to);ar._norm_xor(b,y,N,K);linear_inplace(b,y,frm)\n    u=inverse(b,N);linear_inplace(b,x,to)\n    for off in(0,6):\n        half=x[off:off+6];out=b.alloc_qubits(6);mul_xor0(b,half,u,out)\n        m=b.alloc_bits(6);b.hmr(half,m);b.swap(half,out);b.free(out);phase_product(b,half,N,m);b.free(m)\n    um=b.alloc_bits(6);b.hmr(u,um);b.free(u);phase_inverse_generic6(b,N,um);b.free(um)\n    linear_inplace(b,y,to);ar._erase_norm(b,y,N,K);linear_inplace(b,y,frm);b.free(N)\n    linear_inplace(b,x,frm);phase_product(b,x,y,old);b.free(old)\n', 'tasks/dqi/fused/source/exp/partial_gauge/synthesis_noflag.py': '"""Partial-q-gauge synthesis. Same public A,B,transcript API as decoder.\n\n70 input field words plus one rolling d field word and one boundary flag.\nOnly lower A/B coefficients are stored; implicit high coefficients are aliases\nof earlier q-prefix slots. extra[0] is the physical A[h] zero on valid heads.\nNo edits to the live decoder; local measurement transcript is fully closed.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/\'exp/decoder\'))\nfrom common import *\nfrom extract import constants\nfrom exp.decoder import synthesis as base\nfrom exp.partial_gauge import prefix as pg\nfrom exp.partial_gauge.scalar import divide117\nfrom exp.partial_gauge.tail import divide_safe138,multiply_safe165,controlled_linear_xor\nfrom exp.mul99.schedules import divide_zero_bridge414\nimport division as dv\nck=base.ck\n\ndef xword(b,w,v):\n    for j in ar.bits(v):b.x(w[j])\n\ndef add_alias(b,src,targets,coeffs,virtual_zero=None,constant=0):\n    if src is None:\n        for z,c in zip(targets,coeffs):xword(b,z,mul(constant,c))\n    else:\n        for z,c in zip(targets,coeffs):cx(b,src,z,c)\n        if virtual_zero is not None:controlled_linear_xor(b,virtual_zero,src,targets,coeffs)\n\ndef eval_implicit(b,A,qi,c,reverse=False):\n    if len(A)==1:\n        assert qi is not None\n        return base.expose_eval(b,A+[qi],c,reverse)\n    v=base.evaluation_coefficients(len(A)+1,c)[-1]\n    if reverse:\n        add_alias(b,qi,A[:2],[v&4095,v>>12],constant=1)\n        return base.expose_eval(b,A,c,True)\n    out=base.expose_eval(b,A,c)\n    add_alias(b,qi,A[:2],[v&4095,v>>12],constant=1);return out\n\ndef rank_affine(b,head,ac,words,bc,target,tc,constant=0):\n    if not any(tc):return\n    x,ap=ck.expose(b,head,ac);y,bp=ck.expose(b,words,bc)\n    xword(b,y,constant)\n    p=next(i for i,c in enumerate(tc) if c);v=tc[p];vi=inv(v)\n    for i,c in enumerate(tc):\n        if i!=p:cx(b,target[p],target[i],mul(c,vi))\n    scale(b,target[p],vi);ar.mul_xor0(b,x,y,target[p]);scale(b,target[p],v)\n    for i in reversed(range(len(tc))):\n        if i!=p:cx(b,target[p],target[i],mul(tc[i],vi))\n    xword(b,y,constant);ck.unexpose(b,words,bc,bp);ck.unexpose(b,head,ac,ap)\n\ndef projected_product(b,head,A,qi,target,kap):\n    k=len(A)-1;n=k+3;V,W=base.vandermonde(n)\n    words=A+([qi] if qi is not None else [])\n    for t in range(n):\n        tc=[W[j][t] for j in range(k+2)];top=W[k+2][t]\n        tc[k]^=mul(kap,top);tc[k+1]^=top\n        rank_affine(b,head,[1,t],words,V[t][:len(words)],target,tc,V[t][k+1] if qi is None else 0)\n\ndef truncated_quotient(b,head,A,qi,B,kap):\n    k=len(B)\n    if not k:return\n    words=A[1:]+([qi] if qi is not None else [])\n    for ac,bc,tc in reversed(base.quotient_terms(k+2,kap)):\n        rank_affine(b,head,ac,words,bc[:len(words)],B,tc[:k],bc[k] if qi is None else 0)\n\ndef prefix_encode(b,heads,i,F,d=None):\n    a,x=heads[i]\n    if not i:\n        if d is not None:b.cx(a,d);b.cx(x,d)\n        return None\n    q=heads[i-1][1]\n    def after():\n        ar.linear_inplace(b,q,ar.square_columns(12));ar.mul_xor0(b,a,q,d)\n        ar.linear_inplace(b,q,[power(1<<j,2048) for j in range(12)]);ar.mul_xor0(b,x,q,d)\n    callback=after if d is not None else None\n    if i==1:return pg.boundary_forward(b,x,q,F,callback)\n    return pg.forward(b,x,heads[i-2][1],q,F if i==2 else None,callback)\n\ndef prefix_decode(b,heads,i,F,m,d=None):\n    a,x=heads[i]\n    if not i:\n        if d is not None:b.cx(a,d);b.cx(x,d)\n        return\n    q=heads[i-1][1]\n    if d is not None:\n        ar.mul_xor0(b,x,q,d)\n        if i==1:b.ccx(F,q,d)\n    if i==1:pg.boundary_reverse(b,x,q,F,m)\n    else:pg.reverse_compact(b,x,heads[i-2][1],q,m,F if i==2 else None)\n    if d is not None:\n        ar.linear_inplace(b,q,ar.square_columns(12));ar.mul_xor0(b,a,q,d)\n        ar.linear_inplace(b,q,[power(1<<j,2048) for j in range(12)])\n\ndef p_alias(heads,i,F):\n    return (None,None,0) if i==0 else (None,None,1) if i==1 else (heads[i-2][1],F if i==2 else None,0)\n\ndef absorb(b,head,A,B,qi,p,virtual,pconst,c,kap,reverse=False,transcript=None):\n    k=len(B);assert len(A)==k+1;out=head+B\n    if not reverse:\n        value=eval_implicit(b,A,qi,c);flag=dv.zero_flag(b,value);eval_implicit(b,A,qi,c,True)\n        b.cx(flag,A[0][0]);spare=[b.alloc_qubits(12),b.alloc_qubits(12)];target=spare+B\n        linear_words(b,target,base.embedding_matrix(k+2,kap));projected_product(b,head,A,qi,target,kap)\n        b.cx(flag,A[0][0]);cs=[0]*(k+2);cs[k]=kap;cs[k+1]=1\n        add_alias(b,p,target,cs,virtual,pconst)\n        hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n        for h,z in zip(head,spare):b.swap(h,z);b.free(z)\n        fm=b.alloc_bits(1);b.hmr(flag,fm[0]);b.free(flag)\n        return out,A,(hm,fm)\n    linear_words(b,out,base.embedding_matrix(k+2,kap),True)\n    quo,rem=divmodp([0]*(k+2)+[1],[kap,1,1]);coeffs=[rem[j] if j<len(rem) else 0 for j in range(2)]+[quo[j] if j<len(quo) else 0 for j in range(k)]\n    add_alias(b,p,out,coeffs,virtual,pconst)\n    value=eval_implicit(b,A,qi,c);cx(b,head[1],head[0],c);hm,fm=transcript\n    divide_zero_bridge414(b,combine(head),value,K=K,extra_zero_mask=fm[0]);b.free(fm)\n    cx(b,head[1],head[0],c);b.cz(combine(head),hm);b.free(hm)\n    eval_implicit(b,A,qi,c,True);truncated_quotient(b,head,A,qi,B,kap)\n    return out,A,transcript\n\ndef synthesize(b,words,keep_leading=True):\n    h=len(words)//2;assert h>=3 and h%2==1\n    cs,ks,_,_=constants(h);heads=[words[2*i:2*i+2] for i in range(h)]\n    d=b.alloc_qubits(12);F=pg.zero_compact(b,heads[0][1]);masks=[]\n    for i in range(h):masks.append(prefix_encode(b,heads,i,F,d))\n    divide_safe138(b,d,heads[-1][1]);A=[d];B=[];history=[]\n    for i in reversed(range(h)):\n        prefix_decode(b,heads,i,F,masks[i]);qi=heads[i-1][1] if i else None;p,virtual,pc=p_alias(heads,i,F)\n        oldA,oldB=A,B;A,B,m=absorb(b,heads[i],A,B,qi,p,virtual,pc,cs[i],ks[i]);history.append((i,oldA,oldB,qi,p,virtual,pc,cs[i],ks[i],m))\n    extra=[A[h]];freed=[] if keep_leading else extra\n    for w in freed:b.free(w)\n    fmask=b.alloc_bits(1);b.hmr(F,fmask[0]);b.free(F)\n    t=dict(fmask=fmask,kind=\'partial_q_gauge_noflag\',h=h,heads=heads,d=d,F=F,history=history,A=A,B=B,extra=extra,freed=freed,keep_leading=keep_leading)\n    return A[:h],B,t\n\ndef unsynthesize(b,t):\n    if t[\'freed\']:reclaim(b,t[\'freed\'])\n    heads=t[\'heads\'];F=t[\'F\'];masks=[None]*t[\'h\']\n    for i,A,B,qi,p,virtual,pc,c,kap,m in reversed(t[\'history\']):\n        absorb(b,heads[i],A,B,qi,p,virtual,pc,c,kap,True,m);masks[i]=prefix_encode(b,heads,i,F)\n        if i==0:\n            reclaim(b,[[F]]);lad=base.zero_hold(b,heads[0][1]);b.cx(lad[-1],F);base.zero_free(b,heads[0][1],lad)\n            b.cz(F,t[\'fmask\'][0]);b.free(t[\'fmask\'])\n    multiply_safe165(b,t[\'d\'],heads[-1][1])\n    for i in reversed(range(t[\'h\'])):prefix_decode(b,heads,i,F,masks[i],t[\'d\'])\n    pg.zero_compact_free(b,heads[0][1],F);b.free(t[\'d\'])\n', 'tasks/dqi/fused/source/exp/partial_gauge/tail.py': '"""Closed valid-tail scale pair; q=0 requires d=0 throughout."""\ntry:\n    from .scalar import *\nexcept ImportError:\n    from scalar import *\nfrom lowwidth import phase_inverse36\n\ndef multiply_closed144(b,x,q):\n    """x*=q, 144T/19private, q0 requires x0; all HMR phases closed."""\n    out=b.alloc_qubits(12);mul_xor0(b,x,q,out);m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out)\n    iq=inverse(b,q);phase_product(b,x,iq,m);b.free(m)\n    im=b.alloc_bits(12);b.hmr(iq,im);b.free(iq);phase_inverse36(b,q,im);b.free(im)\n\ndef divide_safe138(b,x,q):\n    """All-input x/=q when q!=0, identity whenq0;138T/20private."""\n    try:from .prefix import zero_compact,zero_compact_free\n    except ImportError:from prefix import zero_compact,zero_compact_free\n    f=zero_compact(b,q);b.cx(f,q[0]);divide117(b,x,q);b.cx(f,q[0]);zero_compact_free(b,q,f)\n\ndef multiply_safe165(b,x,q):\n    """Inverse all-input tail map,165T/20private, no deferred phase."""\n    try:from .prefix import zero_compact,zero_compact_free\n    except ImportError:from prefix import zero_compact,zero_compact_free\n    f=zero_compact(b,q);b.cx(f,q[0]);multiply_closed144(b,x,q);b.cx(f,q[0]);zero_compact_free(b,q,f)\n\ndef controlled_linear_xor(b,F,x,targets,coeffs):\n    """targets[j]^=F*coeffs[j]*x,12T,0scratch; GF12 coefficients classical."""\n    f=field(12);p=next(i for i,v in enumerate(coeffs) if v);v=coeffs[p];vi=f.invert(v)\n    for i,c in enumerate(coeffs):\n        if i!=p:linear_xor(b,targets[p],targets[i],[f.mul(f.mul(c,vi),1<<j) for j in range(12)])\n    linear_inplace(b,targets[p],[f.mul(vi,1<<j) for j in range(12)])\n    b.ccx(F,x,targets[p]);linear_inplace(b,targets[p],[f.mul(v,1<<j) for j in range(12)])\n    for i in reversed(range(len(coeffs))):\n        if i!=p:linear_xor(b,targets[p],targets[i],[f.mul(f.mul(coeffs[i],vi),1<<j) for j in range(12)])\n', 'tasks/dqi/fused/source/exp/phase24_tradeoff/kernel.py': '"""Inverse24 classical-mask phase tradeoff; frozen jet source is read-only."""\nfrom exp.jet24.kernel import *\nfrom exp.mul99.phase36 import phase_classical_parity\n\n\ndef rank(rows):\n    p={}\n    for v in rows:\n        while v:\n            j=v.bit_length()-1\n            if j in p:v^=p[j]\n            else:p[j]=v;break\n    return len(p)\n\n\ndef cols_from_rows(rows):return[sum(((r>>j)&1)<<i for i,r in enumerate(rows))for j in range(len(rows))]\n\n\ndef layout():\n    rows=[]\n    for l,r in LAY[\'gates\']:\n        for v in(l>>1,r>>1):\n            if rank(rows+[v])>len(rows):rows.append(v)\n    assert len(rows)==12\n    pivot=next(j for j in range(6)if rank(rows+[1<<j])==13);rows.append(1<<pivot);cols=cols_from_rows(rows);inv=inverse_columns(cols)\n    def form(e):return(e&1)|sum((((((e>>1)&65535)&inv[j]).bit_count()&1)<<(j+1))for j in range(13))\n    outputs=[(e&~65535)|form(e&65535)for e in LAY[\'outputs\']];gates=[tuple(form(e)for e in g)for g in LAY[\'gates\']]\n    assert all(not(e>>13&1)for g in gates for e in g)\n    v=sum(((e>>13)&1)<<j for j,e in enumerate(outputs));assert rank([1,2,v])==3\n    rest=[]\n    for j in range(2,6):\n        if rank([1,2,v]+rest+[1<<j])>3+len(rest):rest.append(1<<j)\n    B=[1,2]+rest+[v];assert len(B)==6;U=inverse_columns(B)\n    def outputmix(values):return[sum(0 for _ in[])for j in[]]\n    newoutputs=[]\n    for j in range(6):\n        e=0\n        for i in range(6):\n            if U[i]>>j&1:e^=outputs[i]\n        newoutputs.append(e)\n    assert[(e>>13)&1 for e in newoutputs]==[0,0,0,0,0,1]\n    return dict(pivot=pivot,source_columns=cols,source_inverse=inv,output_B=B,outputs=newoutputs,gates=gates)\n\nL30=layout()\n\n\ndef norm_bit_xor(b,N,target,K12,to6,j):\n    """One tower6 component ofnorm12(N),6CCX andnoquantumscratch.\n    N is12towercoordinates; outputcanonical6isthenmappedbyto6.\n    """\n    f=field(6)\n    for i in range(6):\n        if apply(to6,f.square(1<<i))>>j&1:b.cx(N[i],target)\n        if apply(to6,f.mul(K12,f.square(1<<i)))>>j&1:b.cx(N[i+6],target)\n        row=sum(((apply(to6,f.mul(1<<i,1<<k))>>j)&1)<<k for k in range(6));ix=list(bits(row));p=ix[0]\n        for k in ix[1:]:b.cx(N[6+k],N[6+p])\n        b.ccx(N[i],N[6+p],target)\n        for k in reversed(ix[1:]):b.cx(N[6+k],N[6+p])\n\n\ndef virtual_mask_phase(b,x24,mask24,N12,out6,frm12):\n    """30 executedCCZ,0scratch; L12 virtual inx24 andclassicalmask."""\n    f=field(12)\n    for segment in(0,1):\n        if segment==0:b.cx(N12[6:],N12[:6])\n        xx=N12[:6]if segment==0 else N12[6:]\n        for m,v in tensor(6):\n            h=apply(frm12,v<<(6*segment));co=b.alloc_bits(24);b.bit_store(co,False)\n            for i in range(12):\n                for r in bits(f.mul(h,1<<i)):\n                    b.cx(mask24[r],co[i]);b.cx(mask24[r],co[i+12]);b.cx(mask24[r+12],co[i+12])\n            ix=list(bits(m));p=ix[0]\n            for k in ix[1:]:b.cx(xx[k],xx[p]);b.cx(out6[k],out6[p])\n            phase_classical_parity(b,xx[p],out6[p],x24,co)\n            for k in reversed(ix[1:]):b.cx(out6[k],out6[p]);b.cx(xx[k],xx[p])\n            b.free(co)\n        if segment==0:b.cx(N12[6:],N12[:6])\n\n\ndef phase_inverse24_canonical118(b,x,mask,zero_mask=None):\n    """<=118 executedT/30private, inverse0phase; inputcanonical24."""\n    to24,frm24,K24=tower(24);linear_inplace(b,x,to24);mt=classical_transpose(b,mask,frm24)\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,x,N,K24);to12,frm12,K12=tower(12);linear_inplace(b,N,to12)\n    D=b.alloc_qubits(6);norm_xor_noscratch(b,N,D,K12);to6,frm6,K6=tower(6);linear_inplace(b,D,to6);g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,D,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(D)+list(g);forms=ref.norm_forms(K6);w=ref.inverse3_virtual(b,prefix,forms)\n    linear_inplace(b,g,LAY[\'raw_columns\']);R,E=g[:4],g[4:];source=list(D)+list(R)+list(w);linear_inplace(b,source,L30[\'source_columns\']);C=source[:12];z=list(E)+list(b.alloc_qubits(3))+source[12:]\n    for j,e in enumerate(L30[\'outputs\']):ref.form_xor(b,C,e&8191,z[j])\n    for i,(l,r)in enumerate(L30[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(L30[\'outputs\']))\n        with ref.expose_pair(b,C,l,r)as(p,q):rankone_xor(b,p,q,z,v)\n    linear_inplace(b,z,L30[\'output_B\']);linear_inplace(b,z,LAY[\'output_inverse_columns\']);linear_inplace(b,z,frm6)\n    virtual_mask_phase(b,x,mt,N,z,frm12);b.free(mt)\n    # OutputHMR first frees6 before restoring the one originalD coordinate.\n    zm=b.alloc_bits(6);b.hmr(z,zm);b.free(z);ym=classical_transpose(b,zm,frm6);b.free(zm);mm=classical_transpose(b,ym,LAY[\'output_inverse_columns\']);b.free(ym)\n    dj=b.alloc_qubits(1)[0];norm_bit_xor(b,N,dj,K12,to6,L30[\'pivot\']);S=list(C)+[dj];linear_inplace(b,S,L30[\'source_inverse\']);DD=S[:6];RR=S[6:10];ww=S[10:]\n    phase_outputs_classical(b,S,LAY[\'outputs\'],LAY[\'gates\'],mm)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,DD,l,r)as(p,q):\n            for j,row in enumerate(LAY[\'raw_rows\'][4:]):\n                if row>>i&1:b.ccz(p,q,mm[j])\n    b.free(mm)\n    EE=b.alloc_qubits(3);norm_xor_noscratch(b,DD,EE,K6)\n    if zero_mask is not None:\n        b.z(zero_mask)\n        for i in range(3):\n            for j in range(3):\n                if field(3).mul(1<<i,1<<j)&1:b.ccz(EE[i],ww[j],zero_mask)\n    erase_inverse(b,EE,ww);b.free(ww);ar._erase_norm(b,DD,EE,K6);b.free(EE);zero1.erase_raw_cache(b,DD,RR,LAY[\'raw_rows\'][:4]);linear_inplace(b,DD,frm6)\n    dm=b.alloc_bits(6);b.hmr(DD,dm);b.free(DD);ar._phase_norm(b,N,dm,K12);b.free(dm);linear_inplace(b,N,frm12);ar._erase_norm(b,x,N,K24);b.free(N);linear_inplace(b,x,frm24)\n\n\ndef phase_inverse24_extension118(b,x,mask,K=None,zero_mask=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols);mt=classical_transpose(b,mask,frm);phase_inverse24_canonical118(b,x,mt,zero_mask);b.free(mt);linear_inplace(b,x,frm)\n', 'tasks/dqi/fused/source/exp/phase24_tradeoff/phase6_six_search.json': '{\n  "seed": 202610262337,\n  "profiles": [\n    {\n      "hyper": 1,\n      "tested": 7640\n    },\n    {\n      "hyper": 2,\n      "tested": 7784\n    },\n    {\n      "hyper": 3,\n      "tested": 7672\n    },\n    {\n      "hyper": 4,\n      "tested": 7008\n    }\n  ],\n  "shears_tested": 30104,\n  "candidate": {\n    "p": 1321,\n    "q": 372,\n    "w": 29,\n    "output_bit": 5,\n    "terms": [\n      [\n        4\n      ],\n      [\n        6\n      ],\n      [\n        7\n      ],\n      [\n        9\n      ],\n      [\n        0,\n        2\n      ],\n      [\n        0,\n        3\n      ],\n      [\n        0,\n        5\n      ],\n      [\n        0,\n        6\n      ],\n      [\n        0,\n        7\n      ],\n      [\n        0,\n        8\n      ],\n      [\n        0,\n        10\n      ],\n      [\n        2,\n        6\n      ],\n      [\n        2,\n        8\n      ],\n      [\n        3,\n        4\n      ],\n      [\n        3,\n        10\n      ],\n      [\n        4,\n        5\n      ],\n      [\n        4,\n        6\n      ],\n      [\n        4,\n        7\n      ],\n      [\n        4,\n        8\n      ],\n      [\n        5,\n        10\n      ],\n      [\n        6,\n        7\n      ],\n      [\n        6,\n        9\n      ],\n      [\n        6,\n        10\n      ],\n      [\n        7,\n        8\n      ],\n      [\n        7,\n        10\n      ]\n    ],\n    "zero_terms": [\n      [\n        0\n      ],\n      [\n        1\n      ],\n      [\n        2\n      ],\n      [\n        3\n      ],\n      [\n        4\n      ],\n      [\n        9\n      ],\n      [\n        10\n      ],\n      [\n        0,\n        1\n      ],\n      [\n        0,\n        2\n      ],\n      [\n        0,\n        6\n      ],\n      [\n        0,\n        7\n      ],\n      [\n        0,\n        8\n      ],\n      [\n        0,\n        9\n      ],\n      [\n        1,\n        2\n      ],\n      [\n        1,\n        4\n      ],\n      [\n        1,\n        6\n      ],\n      [\n        1,\n        9\n      ],\n      [\n        1,\n        10\n      ],\n      [\n        2,\n        8\n      ],\n      [\n        3,\n        4\n      ],\n      [\n        3,\n        6\n      ],\n      [\n        3,\n        7\n      ],\n      [\n        4,\n        7\n      ],\n      [\n        4,\n        8\n      ],\n      [\n        5,\n        6\n      ],\n      [\n        6,\n        8\n      ],\n      [\n        6,\n        9\n      ],\n      [\n        6,\n        10\n      ],\n      [\n        7,\n        10\n      ],\n      [\n        8,\n        9\n      ]\n    ],\n    "raw_forms": [\n      [\n        12,\n        80\n      ],\n      [\n        4,\n        64\n      ],\n      [\n        2,\n        48\n      ],\n      [\n        8,\n        16\n      ],\n      [\n        10,\n        32\n      ]\n    ],\n    "raw_coefficients": [\n      1,\n      2,\n      8,\n      16,\n      32\n    ],\n    "truth_points": 64,\n    "errors": 0,\n    "prefix_AND": 6,\n    "scratch": 5,\n    "hyper": 4,\n    "attempt": 902\n  },\n  "seconds": 3.8534505367279053\n}\n', 'tasks/dqi/fused/source/exp/phase24_tradeoff/phase6small.py': '"""Classical-mask inverse6 phase:<=6executedT/5scratch; exact64pointSLP.\nClassicalmask-dependent constantfieldscaling maps63nonzero components toone.\n"""\nfrom exp.phase24_tradeoff.kernel import *\nimport json\nP6=json.loads(Path(__file__).with_name(\'phase6_six_search.json\').read_text())[\'candidate\']\n\ndef classical_equal(b,x,value):\n    chain=b.alloc_bits(len(x)+1);b.bit_store(chain,False);b.x(chain[0])\n    for j,q in enumerate(x):\n        if not(value>>j&1):b.x(q)\n        b.push_condition(chain[j]);b.bit_store(chain[j+1],True,control=q);b.pop_condition()\n        if not(value>>j&1):b.x(q)\n    b.free(chain[:-1]);return chain[-1]\n\n\ndef fixed_phase6(b,x,do_inverse=True,zero_mask=None):\n    to,frm,K=tower(6);linear_inplace(b,x,to);ng=len(P6[\'raw_forms\']);g=b.alloc_qubits(ng)\n    for i,(l,r)in enumerate(P6[\'raw_forms\']):\n        with ref.expose_pair(b,x,l,r)as(p,q):b.init_and(p,q,g[i])\n    w=P6[\'w\'];base=[]\n    for j in range(ng):\n        if rank(base+[w,1<<j])==len(base)+2:base.append(1<<j)\n    B=base+[w];assert len(B)==ng;iv=inverse_columns(B);linear_inplace(b,g,iv)\n    def form(v):\n        return(v&63)|sum((((v>>6)&co).bit_count()&1)<<(6+j)for j,co in enumerate(B))\n    p,q=form(P6[\'p\']),form(P6[\'q\']);assert not(((p|q)>>(6+ng-1))&1);source=list(x)+list(g[:ng-1])\n    with ref.expose_pair(b,source,p<<1,q<<1)as(a,c):b.ccx(a,c,g[ng-1])\n    linear_inplace(b,g,B);S=list(x)+list(g)\n    for term in(P6[\'terms\']if do_inverse else[]):\n        if len(term)==1:b.z(S[term[0]])\n        else:b.cz(S[term[0]],S[term[1]])\n    if zero_mask is not None:\n        b.z(zero_mask);b.push_condition(zero_mask)\n        for term in P6[\'zero_terms\']:\n            if len(term)==1:b.z(S[term[0]])\n            else:b.cz(S[term[0]],S[term[1]])\n        b.pop_condition()\n    linear_inplace(b,g,iv);hm=b.alloc_bits(1)[0];b.hmr(g[ng-1],hm)\n    with ref.expose_pair(b,source,p<<1,q<<1)as(a,c):b.ccz(a,c,hm)\n    rows=[sum(((iv[i]>>j)&1)<<i for i in range(ng))for j in range(ng)]\n    for i,(l,r)in enumerate(P6[\'raw_forms\']):\n        if rows[ng-1]>>i&1:\n            with ref.expose_pair(b,x,l,r)as(a,c):b.ccz(a,c,hm)\n    b.free(hm);b.free(g[ng-1]);mm=b.alloc_bits(ng-1);b.hmr(g[:ng-1],mm);b.free(g[:ng-1])\n    for i,(l,r)in enumerate(P6[\'raw_forms\']):\n        with ref.expose_pair(b,x,l,r)as(a,c):\n            for j in range(ng-1):\n                if rows[j]>>i&1:b.ccz(a,c,mm[j])\n    b.free(mm);linear_inplace(b,x,frm)\n\n\ndef mask_factors():\n    f=field(6);out={}\n    for s in range(1,64):\n        si=f.invert(s);m=sum(((f.mul(si,1<<j)>>P6[\'output_bit\'])&1)<<j for j in range(6));assert m not in out;out[m]=s\n    return out\nFACTORS=mask_factors()\n\n\ndef phase_inverse6_classical6(b,x,mask,zero_mask=None):\n    """Exactmask·inv0(x),0→0,<=6executedT/5scratch,arbitraryclassicalmask."""\n    for m,s in sorted(FACTORS.items()):\n        eq=classical_equal(b,mask,m);b.push_condition(eq);co=[field(6).mul(s,1<<j)for j in range(6)];linear_inplace(b,x,co);fixed_phase6(b,x,zero_mask=zero_mask);linear_inplace(b,x,inverse_columns(co));b.pop_condition();b.free(eq)\n\n    if zero_mask is not None:\n        eq=classical_equal(b,mask,0);b.push_condition(eq);b.push_condition(zero_mask);fixed_phase6(b,x,do_inverse=False,zero_mask=zero_mask);b.pop_condition();b.pop_condition();b.free(eq)\n', 'tasks/dqi/fused/source/exp/safe_scalar/converter.py': '# Correct residues on coprime degree profiles with gaps at most two.\n# Zero-clipped scalar permutations preserve restoration on unsupported profiles.\n"""Guarded history-free multi-h2 Bezout lookahead, with borrowed monic-root word.\n\nOn proper coprime half-degree charts with drops1or2, parent gap flags are\nrecomputed from the deliberatelyzero child quotient metadata. No persistent\nquantum history register. Crosspass token is classical and phaseclosed after\nreverse around a diagonal oracle. See RESTORATION.md for the all-input\nrestoration argument; larger gaps may still produce wrong decoded phases.\n"""\nimport sys\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/\'convert\'))\nfrom convert import ar,frob\nfrom repair import rotate\nfrom kernels import hankel,convolution_xor\nfrom lowwidth import phase_inverse36\nfrom contextlib import contextmanager\n\ndef scale(b,x,a,ai,divide=True):\n    out=b.alloc_qubits(12)\n    ar.mul_xor0(b,x,ai if divide else a,out)\n    ar.erase_product(b,a if divide else ai,out,x)\n    b.swap(x,out);b.free(out)\n\ndef runops(b,ops,a,ai,undo=False):\n    for op in (reversed(ops)if undo else ops):\n        if op[0]==\'s\':scale(b,op[1],a,ai,op[2]^undo)\n        elif op[0]==\'m\':ar.mul_xor0(b,op[1],op[2],op[3])\n        elif op[0]==\'c\':convolution_xor(b,op[1],op[2],op[3],undo)\n        else:raise ValueError(op[0])\n\ndef zero_toggle(b,a,target):\n    # Holdthe11ANDladderuntiltheflagcopy, thenuncomputeallforfree.\n    for q in a:b.x(q)\n    ladder=list(b.alloc_qubits(len(a)-1));b.init_and(a[0],a[1],ladder[0])\n    for i in range(1,len(ladder)):b.init_and(ladder[i-1],a[i+1],ladder[i])\n    b.cx(ladder[-1],target)\n    for i in reversed(range(len(ladder))):\n        if i:b.del_and(ladder[i-1],a[i+1],ladder[i])\n        else:b.del_and(a[0],a[1],ladder[0])\n    b.free(ladder)\n    for q in a:b.x(q)\n\nfrom exp.inverse12_escape.zero1 import inverse_zero,consume,erase,inverse_apply_safe\n\n@contextmanager\ndef q_inverse(b,a,m,reverse,phase):\n    ai,F=inverse_zero(b,a)\n    if reverse:consume(b,ai,F,phase[\'q\'].pop(m))\n    b.cx(F,a[0]);b.cx(F,ai[0]);yield ai;b.cx(F,ai[0]);b.cx(F,a[0])\n    token=erase(b,a,ai,F,not reverse)\n    if not reverse:phase[\'q\'][m]=token\n\ndef inverse_apply(b,a,action,incoming=None,defer=False):\n    return inverse_apply_safe(b,a,action,incoming,defer)\n\ndef pre(b,f,g,payloads,undo=False,incoming=None,defer=False):\n    m=len(f)-1;n=m-1;a=g[-1];q0=f[-2];q1=f[-1]\n    def action(ai):\n        ops=[(\'s\',v[-1],True)for v in payloads]\n        ops +=[(\'m\',g[i],v[-1],v[i])for v in payloads for i in range(n)]\n        ops +=[(\'s\',q1,True),(\'m\',q1,g[-2],q0),(\'s\',q0,True),(\'c\',g[:-1],[q0,q1],f[:-2])]\n        runops(b,ops,a,ai,undo)\n    return inverse_apply(b,a,action,incoming,defer)\n\ndef canonical_permute(b,f,g,F,reverse=False):\n    # [G0,F0,G1,F1,...,G_(m-2),F_(m-2),G_(m-1)] leftcyclic.\n    cyc=[]\n    for i in range(len(g)-1):cyc +=[g[i],f[i]]\n    cyc +=[g[-1]]\n    seq=range(len(cyc)-2,-1,-1) if reverse else range(len(cyc)-1)\n    for i in seq:b.cswap(F,cyc[i],cyc[i+1])\n\ndef masked_product(b,F,x,c,target):\n    u=b.alloc_qubits(12)\n    for j in range(12):b.init_and(F,x[j],u[j])\n    ar.mul_xor0(b,u,c,target)\n    mask=b.alloc_bits(12);b.hmr(u,mask);b.free(u)\n    for j in range(12):b.ccz(F,x[j],mask[j])\n    b.free(mask)\n\ndef fresh_zero_flag(b,a):\n    F=b.alloc_qubits(1)[0];zero_toggle(b,a,F);return F\n\ndef clear_flag(b,a,F):zero_toggle(b,a,F);b.free(F)\n\ndef q_post(b,q,payloads,H,m,parent,reverse=False,phase=None):\n    if parent is None:\n        with q_inverse(b,q,m,reverse,phase) as qi:\n            for v in payloads:scale(b,v[-1],q,qi,not reverse)\n        return\n    tops,p,c=parent;F=fresh_zero_flag(b,q)\n    b.cswap(F,q,p)\n    if not reverse:\n        for v,t in zip(payloads,tops):b.cswap(F,v[-1],t)\n    with q_inverse(b,q,m,reverse,phase) as qi:\n        if not reverse:\n            for v in payloads:scale(b,v[-1],q,qi,True)\n            for v,t in zip(payloads,tops):masked_product(b,F,v[-1],c,t)\n        else:\n            for v,t in zip(payloads,tops):masked_product(b,F,v[-1],c,t)\n            for v in payloads:scale(b,v[-1],q,qi,False)\n    if reverse:\n        for v,t in zip(payloads,tops):b.cswap(F,v[-1],t)\n    b.cswap(F,q,p);clear_flag(b,q,F)\n\ndef post_look(b,f,g,payloads,H,parent,reverse=False,incoming=None,defer=False,phase=None):\n    m=len(f)-1;n=m-1;a=g[-1];q0=f[-2];q1=f[-1]\n    def dots():\n        for v in payloads:\n            for i in range(n):ar.mul_xor0(b,g[i],v[i],v[-1])\n    def last(undo,inc=None,delay=False):\n        def action(ai):\n            ops=[(\'s\',v[-1],True)for v in payloads]\n            ops +=[(\'c\',g[:-1],[q0,q1],f[:-2]),(\'s\',q0,False),(\'m\',q1,g[-2],q0),(\'s\',q1,False)]\n            runops(b,ops,a,ai,undo)\n        return inverse_apply(b,a,action,inc,delay)\n    if not reverse:q_post(b,q1,payloads,H,m,parent,phase=phase);dots();return last(False,incoming,defer)\n    else:\n        carry=last(True,incoming,defer);dots();q_post(b,q1,payloads,H,m,parent,True,phase);return carry\n\ndef base(b,f,g,payloads,H,parent,reverse=False,phase=None):\n    a=g[0];q=f[-1]\n    def first(undo,inc=None,delay=False):\n        def action(ai):runops(b,[(\'s\',v[0],True)for v in payloads]+[(\'s\',q,True)],a,ai,undo)\n        return inverse_apply(b,a,action,inc,delay)\n    def last(undo,inc=None,delay=False):\n        def action(ai):runops(b,[(\'s\',v[0],True)for v in payloads]+[(\'s\',q,False)],a,ai,undo)\n        return inverse_apply(b,a,action,inc,delay)\n    if not reverse:\n        carry=first(False,delay=True);q_post(b,q,payloads,H,1,parent,phase=phase);phase[\'a\'][1]=last(False,carry,True)\n    else:\n        carry=last(True,phase[\'a\'].pop(1),True);q_post(b,q,payloads,H,1,parent,True,phase);first(True,carry)\n\ndef look_solve(b,f,g,payloads,H=None,parent=None,reverse=False,phase=None):\n    m=len(f)-1\n    if m==1:return base(b,f,g,payloads,H,parent,reverse,phase)\n    F=fresh_zero_flag(b,g[-1]);rotate(b,g,F)\n    if not reverse:carry=pre(b,f,g,payloads,defer=True)\n    else:carry=post_look(b,f,g,payloads,H,parent,True,incoming=phase[\'a\'].pop(m),defer=True,phase=phase)\n    canonical_permute(b,f,g,F);clear_flag(b,g[-1],F)\n    childparent=([v[-1]for v in payloads],f[-1],f[-2])\n    look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,childparent,reverse,phase)\n    F=fresh_zero_flag(b,g[-1]);canonical_permute(b,f,g,F,True)\n    if not reverse:phase[\'a\'][m]=post_look(b,f,g,payloads,H,parent,incoming=carry,defer=True,phase=phase)\n    else:pre(b,f,g,payloads,True,incoming=carry)\n    rotate(b,g,F,True);clear_flag(b,g[-1],F)\n\ndef bez_lookahead(b,O,E,payloads,reverse=False,history_scratch12=None,phase=None):\n    def reduce():\n        for i in range(len(O)):ar.mul_xor0(b,E[-1],O[i],E[i])\n    reduce()\n    lead=list(b.alloc_qubits(12))if history_scratch12 is None else list(history_scratch12)\n    assert len(lead)==12\n    b.x(lead[0])\n    look_solve(b,O+[lead],E[:-1],payloads,reverse=reverse,phase=phase)\n    b.x(lead[0])\n    if history_scratch12 is None:b.free(lead)\n    reduce()\n\ndef convert_safe(b,A,B,reverse=False,history_scratch12=None,phase_token=None):\n    """Optional12zero wires borrowed as monic rootleadingword; returnedzero."""\n    phase={\'a\':{},\'q\':{}}if phase_token is None else phase_token\n    assert not reverse or phase_token is not None\n    E=B[::2];O=B[1::2];u=A[1::2];w=A[:34:2];top=A[-1]\n    if reverse:hankel(b,B,A,True)\n    frob(b,B,2048);frob(b,A,2048)\n    if reverse:\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n        for i in range(17):ar.mul_xor0(b,O[i],u[i],top)\n        bez_lookahead(b,O,E,[u,w],True,history_scratch12,phase)\n        for i in range(17):ar.mul_xor0(b,top,O[i],w[i])\n    else:\n        for i in range(17):ar.mul_xor0(b,top,O[i],w[i])\n        bez_lookahead(b,O,E,[u,w],False,history_scratch12,phase)\n        for i in range(17):ar.mul_xor0(b,O[i],u[i],top)\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n    frob(b,A,2);frob(b,B,2)\n    if not reverse:hankel(b,B,A)\n\n    if reverse:assert not phase[\'a\']and not phase[\'q\']\n    return phase if not reverse else None\n', 'tasks/dqi/fused/source/exp/safe_scalar/kernel.py': '"""Zero-aware inverse12: expose zero cheaply at the bottom tower norm."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse_search.inverse20_iso import *\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.safe_scalar.phase import phase_inverse36_zero\n\ndef inverse_zero(b,x):\n    """Preserve x, return inverse0(x) and [x=0],67T with clean scratch."""\n    to,frm,K=tower(12);linear_inplace(b,x,to)\n    N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,K)\n    t,D=inverse6_xor20(b,N,keep_norm=True)\n    b.x(D);u=b.alloc_qubits(1)[0];F=b.alloc_qubits(1)[0]\n    b.init_and(D[0],D[1],u);b.init_and(u,D[2],F);b.del_and(D[0],D[1],u);b.free(u);b.x(D)\n    to6,frm6,K6=tower(6);linear_inplace(b,N,to6)\n    ar._erase_norm(b,N,D,K6);b.free(D);linear_inplace(b,N,frm6)\n    ar._erase_norm(b,x,N,K);b.free(N)\n    out=b.alloc_qubits(12);b.cx(x[6:],x[:6])\n    mul_xor_noscratch(b,x[:6],t,out[:6]);mul_xor_noscratch(b,x[6:],t,out[6:]);b.cx(x[6:],x[:6])\n    ar._erase_norm(b,out,t,K);b.free(t)\n    linear_inplace(b,x,frm);linear_inplace(b,out,frm)\n    return out,F\n\ndef consume(b,ai,F,token):\n    if token is not None:\n        m,z=token;b.cz(ai,m);b.cz(F,z[0]);b.free(m);b.free(z)\n\ndef erase(b,a,ai,F,defer):\n    m=b.alloc_bits(12);z=b.alloc_bits(1)\n    b.hmr(ai,m);b.hmr(F,z[0]);b.free(ai);b.free(F)\n    if defer:return m,z\n    phase_inverse36_zero(b,a,m,z[0]);b.free(m);b.free(z)\n\ndef inverse_apply_safe(b,a,action,incoming=None,defer=False):\n    ai,F=inverse_zero(b,a);consume(b,ai,F,incoming)\n    b.cx(F,a[0]);b.cx(F,ai[0]);action(ai);b.cx(F,ai[0]);b.cx(F,a[0])\n    return erase(b,a,ai,F,defer)\n', 'tasks/dqi/fused/source/exp/safe_scalar/phase.py': '"""Inverse12 phase with12 scratch and at most36 EXECUTED Toffolis.\n\nThe coefficient mask is classical. A classical first-nonzero pivot exposes\nits parity in an existing input bit, avoiding the13th quantum wire.\n"""\ntry:\n    from exp.mul99.kernels import *\nexcept ImportError:\n    from exp.mul99.kernels import *\nsys.path.insert(0,str(ROOT))\nfrom exp.noscratch.kernels import inverse3_noscratch\nfrom lowwidth import classical_transpose\n\n\ndef phase_classical_parity(b,a,c,x,co):\n    """CCZ(a,c,parity(co*x)), zero quantum scratch, <=1 executed CCZ."""\n    first=b.alloc_bits(len(x));b.bit_store(first,False)\n    seen=b.alloc_bits(1)[0];b.bit_store(seen,False)\n    for j in range(len(x)):\n        b.x(seen);b.push_condition(seen)\n        b.bit_store(first[j],True,control=co[j])\n        b.pop_condition();b.x(seen)\n        b.bit_store(seen,True,control=co[j])\n    for j in range(len(x)):\n        b.push_condition(first[j])\n        for k in range(len(x)):\n            if k!=j:b.ccx(co[k],x[k],x[j])\n        b.ccz(a,c,x[j])\n        for k in reversed(range(len(x))):\n            if k!=j:b.ccx(co[k],x[k],x[j])\n        b.pop_condition()\n    b.free(seen);b.free(first)\n\n\ndef phase_inverse36_zero(b,x,mask,zero_mask):\n    """Phase mask dot inverse0(x): <=36 mean T /12 quantum scratch.\n\n    Static count168T due to12 mutually exclusive classical branches per CCZ.\n    No quantum input controls branch selection; all branches use HMR-mask bits.\n    """\n    assert len(x)==len(mask)==12\n    to12,frm12,K12=tower(12);linear_inplace(b,x,to12)\n    mt=classical_transpose(b,mask,frm12)\n    D=b.alloc_qubits(6);ar._norm_xor(b,x,D,K12)\n    to6,frm6,K6=tower(6);linear_inplace(b,D,to6)\n    E=b.alloc_qubits(3);ar._norm_xor(b,D,E,K6);w=inverse3_noscratch(b,E)\n    b.z(zero_mask);zm=b.alloc_bits(3);b.bit_store(zm,False);b.cx(zero_mask,zm[0])\n    phase_product(b,E,w,zm);b.free(zm)\n    f6=field(6)\n    for segment in(0,1):\n        if segment==0:b.cx(D[3:],D[:3])\n        controls=D[:3] if segment==0 else D[3:]\n        for m,v in tensor(3):\n            h=apply(frm6,v<<(3*segment));co=b.alloc_bits(12);b.bit_store(co,False)\n            for j in range(6):\n                for r in bits(f6.mul(h,1<<j)):\n                    b.cx(mt[r],co[j]);b.cx(mt[r],co[j+6]);b.cx(mt[r+6],co[j+6])\n            ix=list(bits(m));p=ix[0]\n            for j in ix[1:]:b.cx(controls[j],controls[p]);b.cx(w[j],w[p])\n            phase_classical_parity(b,controls[p],w[p],x,co)\n            for j in reversed(ix[1:]):b.cx(w[j],w[p]);b.cx(controls[j],controls[p])\n            b.free(co)\n        if segment==0:b.cx(D[3:],D[:3])\n    erase_inverse(b,E,w);b.free(w)\n    ar._erase_norm(b,D,E,K6);b.free(E);linear_inplace(b,D,frm6)\n    ar._erase_norm(b,x,D,K12);b.free(D);linear_inplace(b,x,frm12);b.free(mt)\n', 'tasks/dqi/fused/source/exp/scan/scan.py': '"""Exact45045-Toffoli Chien/residue phase scan,11workspacequbits.\n\nA,B:35GF12 words each, degree<35 numerator and monicdegree35 denominator.\nE:4095classical12-bit masks, indexed beta=alpha**j. Bothbanks and E restored.\nA is already residue-converted; phase is dot(Ej,A(beta)/(beta*P(beta)))\nwhen B(beta)=0, and0otherwise. P is the node polynomial from extract.constants.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2]\nsys.path.insert(0,str(ROOT/\'exp/decoder\'))\nfrom common import *\nfrom extract import constants\n\n\ndef scan(b,A,B,E,*,points=4095,skip_points=(),node_polynomial=None):\n    if len(A)!=35 or len(B)!=35 or len(E)<points:raise ValueError(\'need35+35words and one maskperpoint\')\n    if points!=4095:raise ValueError(\'fullcycle4095points required for bank restoration\')\n    alpha=gf.primitive_element;P=constants()[2]if node_polynomial is None else node_polynomial\n    powers=[power(alpha,k) for k in range(35)]\n    skip_points=frozenset(skip_points)\n    assert all(0<=j<4095 for j in skip_points)\n    beta=1;lead=1;leadstep=power(alpha,35);m=b.alloc_bits(12)\n    for j in range(4095):\n        # The original coefficientk now holds Ak*beta**k (likewiseB).\n        if j not in skip_points:\n            for k in range(1,35):b.cx(A[k],A[0]);b.cx(B[k],B[0])\n            b.gf2_iadd(lead,target=B[0])\n            ladder=zero_hold(b,B[0]);flag=ladder[-1]\n            pv=0\n            for v in reversed(P):pv=mul(pv,beta)^v\n            c=inv(mul(beta,pv));assert c\n            # Classically apply transpose of multiplicationc to maskEj.\n            b.bit_store(m,False)\n            for t in range(12):\n                for i in ar.bits(mul(c,1<<t)):b.cx(E[j][i],m[t])\n            b.ccz(flag,A[0],m)\n            zero_free(b,B[0],ladder)\n            b.gf2_iadd(lead,target=B[0])\n            for k in range(1,35):b.cx(A[k],A[0]);b.cx(B[k],B[0])\n        for k in range(1,35):scale(b,A[k],powers[k]);scale(b,B[k],powers[k])\n        beta=mul(beta,alpha);lead=mul(lead,leadstep)\n    assert beta==lead==1\n    b.free(m)\n', 'tasks/dqi/fused/source/exp/tail_alias/divide.py': '"""Zero-aware initialized inverse fits below the product peak:369T safe divide."""\nfrom pathlib import Path\nimport sys\nfrom types import FunctionType\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.fold_joint import divide as prior\nfrom exp.inverse19_joint.kernel import inverse_zero\n\ndef inverse_overwrite_zero66(b,x):\n    """Overwrite inverse0+zero,66T,31 localQ; old input mask deferred."""\n    out,F=inverse_zero(b,x);m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out)\n    return m,F\n\n_globals=dict(prior.divide_zero371.__globals__,inverse_stream_zero=inverse_overwrite_zero66)\n_fn=prior.divide_zero371\ndivide_zero369=FunctionType(_fn.__code__,_globals,\'divide_zero369\',_fn.__defaults__,_fn.__closure__)\ndivide_zero369.__doc__=\'All-input safe GF24 division: <=369 executed T,37 private. Old initialized inverse+flag replaces streamed68T with66T below37-bit product peak.\'\n', 'tasks/dqi/fused/source/exp/tail_alias/kernel.py': '"""Tail d aliases raw a0; quadratic companion action closes outer fold.\n\nNo additional persistent field word. Inputs and returned A/B use70 words.\nAll maps are total: factors zero are replaced by factor1 consistently.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.fold_joint.kernel import *\nfrom exp.tail_alias.divide import divide_zero369\nfrom types import FunctionType\n# Private binding: imported CRT arithmetic modules are never mutated.\n_absorb=absorb\nabsorb=FunctionType(_absorb.__code__,dict(_absorb.__globals__,divide_zero371=divide_zero369),\n    \'absorb\',_absorb.__defaults__,_absorb.__closure__)\n\ndef expose_quotient(b,C,D):\n    """Top C pair -> quotient l of q*C by monic D, h>=3; involution."""\n    b.cx(C[-1],C[-2]);ar.mul_xor0(b,C[-1],D[-1],C[-2])\n\ndef outer(b,C,D,c,kap,reverse=False,transcript=None,wide=True):\n    """C <- q*C mod (D+[D(gamma)=0]), D monic degree h unchanged.\n\n    D lists its h low coefficients, C its h coefficients. Constant completion\n    leaves quotient l unaffected for h>=3. CRT output is lowerC then top pair.\n    """\n    h=len(C);assert len(D)==h and h>=3;head=C[-2:];low=C[:-2]\n    if not reverse:\n        expose_quotient(b,C,D)\n        value=old.eval_implicit(b,D,None,c);F=dv.zero_flag(b,value)\n        spare=[b.alloc_qubits(12),b.alloc_qubits(12)]\n        b.cx(F,value[0]);cx(b,head[1],head[0],c)\n        mul24_xor99(b,combine(head),value,combine(spare),K)\n        cx(b,head[1],head[0],c);b.cx(F,value[0]);old.eval_implicit(b,D,None,c,True)\n        q_times_low(b,low,kap);b.cx(F,D[0][0]);low_product(b,head,D,low);b.cx(F,D[0][0])\n        R_to_tops(b,spare,low,c)\n        hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n        for w,z in zip(head,spare):b.swap(w,z);b.free(z)\n        fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n        return hm,fm\n    R_to_tops(b,head,low,c,True)\n    value=old.eval_implicit(b,D,None,c);hm,fm=transcript\n    def after(F):\n        cx(b,head[1],head[0],c);old.eval_implicit(b,D,None,c,True)\n        b.cx(F,D[0][0]);low_product(b,head,D,low);b.cx(F,D[0][0]);q_times_low(b,low,kap,True)\n        old.eval_implicit(b,D,None,c);cx(b,head[1],head[0],c)\n    if wide:divide_zero369(b,combine(head),value,K=K,extra_zero_mask=fm[0],after=after)\n    else:divide414_callback(b,combine(head),value,K,fm[0],after)\n    b.free(fm);cx(b,head[1],head[0],c);b.cz(combine(head),hm);b.free(hm)\n    old.eval_implicit(b,D,None,c,True);expose_quotient(b,C,D)\n    return transcript\n\ndef synthesize(b,words,wide=True):\n    h=len(words)//2;assert h>=3 and h%2==1\n    cs,ks,_,_=old.constants(h);heads=[words[2*i:2*i+2]for i in range(h)]\n    d=heads[0][0];F=pg.zero_compact(b,heads[0][1]);b.cx(heads[0][1],d);masks=[None]*h\n    for i in range(1,h):masks[i]=old.prefix_encode(b,heads,i,F,d)\n    old.divide_safe138(b,d,heads[-1][1]);A=[d];B=[];hist=[]\n    for i in reversed(range(1,h)):\n        old.prefix_decode(b,heads,i,F,masks[i]);qi=heads[i-1][1];p,v,pc=old.p_alias(heads,i,F)\n        oldA,oldB=A,B;A,B,t=absorb(b,heads[i],A,B,qi,p,v,pc,cs[i],ks[i],wide=wide)\n        hist.append((i,oldA,oldB,qi,p,v,pc,cs[i],ks[i],t))\n    C=B+[heads[0][1]];ot=outer(b,C,A,cs[0],ks[0],wide=wide)\n    fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n    return C,A,dict(h=h,heads=heads,d=d,F=F,history=hist,A=C,B=A,outer=ot,wide=wide,fmask=fm)\n\ndef unsynthesize(b,t):\n    h=t[\'h\'];heads=t[\'heads\'];F=t[\'F\'];cs,ks,_,_=old.constants(h)\n    outer(b,t[\'A\'],t[\'B\'],cs[0],ks[0],True,t[\'outer\'],t[\'wide\'])\n    reclaim(b,[[F]]);lad=base.zero_hold(b,heads[0][1]);b.cx(lad[-1],F);base.zero_free(b,heads[0][1],lad)\n    b.cz(F,t[\'fmask\'][0]);b.free(t[\'fmask\']);masks=[None]*h\n    for i,A,B,qi,p,v,pc,c,kap,token in reversed(t[\'history\']):\n        absorb(b,heads[i],A,B,qi,p,v,pc,c,kap,True,token,t[\'wide\'])\n        masks[i]=old.prefix_encode(b,heads,i,F)\n    old.multiply_safe165(b,t[\'d\'],heads[-1][1])\n    for i in reversed(range(1,h)):old.prefix_decode(b,heads,i,F,masks[i],t[\'d\'])\n    b.cx(heads[0][1],t[\'d\']);pg.zero_compact_free(b,heads[0][1],F)\n', 'tasks/dqi/fused/source/exp/zero_padding_synthesis/kernel.py': '"""One-flag Taylor folds with common x-power padding on valid heads."""\nfrom pathlib import Path\nimport sys\nfrom types import FunctionType\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.tail_alias.kernel import *\nfrom exp.alternating_gauge import kernel as alt\nfrom exp.alternating_gauge.even_fold import full_product,full_quotient\nfrom exp.confluent_synthesis import kernel as ca\n\n\ndef zero_finish(b,value,F,fm):\n m=b.alloc_bits(1);b.hmr(F,m[0]);b.free(F);b.cx(fm[0],m[0]);b.free(fm)\n dv.phase_zero(b,value,m[0]);b.free(m)\n\n\ndef absorb_odd(b,head,A,B,Q,p,virtual,pconst,c,kap,reverse=False,transcript=None,wide=False):\n k=len(B);assert len(A)==k+1 and virtual is None and pconst==0\n out=B+head;v=epow(c,k+2);co=[v&4095,v>>12]\n if not reverse:\n  value=old.eval_implicit(b,A,Q,c);F=dv.zero_flag(b,value)\n  spare=[b.alloc_qubits(12),b.alloc_qubits(12)];b.cx(F,value[0]);cx(b,head[1],head[0],c)\n  mul24_xor99(b,combine(head),value,combine(spare),K)\n  cx(b,head[1],head[0],c);b.cx(F,value[0]);old.eval_implicit(b,A,Q,c,True)\n  q_times_low(b,B,kap);b.cx(F,A[0][0]);low_product(b,head,A,B);b.cx(F,A[0][0])\n  old.add_alias(b,p,spare,co);old.controlled_linear_xor(b,F,p,spare,co);R_to_tops(b,spare,B,c)\n  hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n  for x,z in zip(head,spare):b.swap(x,z);b.free(z)\n  fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n  return out,A,(hm,fm)\n R_to_tops(b,head,B,c,True);old.add_alias(b,p,head,co)\n value=old.eval_implicit(b,A,Q,c);F=dv.zero_flag(b,value);hm,fm=transcript\n old.controlled_linear_xor(b,F,p,head,co);b.cx(F,value[0]);divide_lowwidth369(b,combine(head),value,K);b.cx(F,value[0])\n cx(b,head[1],head[0],c);old.eval_implicit(b,A,Q,c,True)\n b.cx(F,A[0][0]);low_product(b,head,A,B);b.cx(F,A[0][0]);q_times_low(b,B,kap,True)\n value=old.eval_implicit(b,A,Q,c);zero_finish(b,value,F,fm);old.eval_implicit(b,A,Q,c,True)\n b.cz(combine(head),hm);b.free(hm)\n return out,A,transcript\n\n\ndef absorb_even(b,head,A,B,Q,r,c,kap,reverse=False,transcript=None):\n k=len(B);out=head+B;target=out+[r]\n if not reverse:\n  value=old.eval_implicit(b,A,Q,c);F=dv.zero_flag(b,value);old.eval_implicit(b,A,Q,c,True)\n  b.cx(F,A[0][0]);spare=[b.alloc_qubits(12),b.alloc_qubits(12)];full=spare+B+[r]\n  linear_words(b,full,base.embedding_matrix(k+3,kap));full_product(b,head,A,Q,full);b.cx(F,A[0][0])\n  old.controlled_linear_xor(b,F,r,full[-3:-1],[kap,1])\n  hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n  for x,z in zip(head,spare):b.swap(x,z);b.free(z)\n  fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n  return out,A,(hm,fm)\n value=old.eval_implicit(b,A,Q,c);F=dv.zero_flag(b,value);old.eval_implicit(b,A,Q,c,True)\n old.controlled_linear_xor(b,F,r,out[-2:],[kap,1]);linear_words(b,target,base.embedding_matrix(k+3,kap),True)\n value=old.eval_implicit(b,A,Q,c);cx(b,head[1],head[0],c);hm,fm=transcript\n b.cx(F,value[0]);divide_lowwidth369(b,combine(head),value,K);b.cx(F,value[0]);cx(b,head[1],head[0],c)\n b.cz(combine(head),hm);b.free(hm);old.eval_implicit(b,A,Q,c,True)\n full_quotient(b,head,A,Q,B,r,kap)\n value=old.eval_implicit(b,A,Q,c);zero_finish(b,value,F,fm);old.eval_implicit(b,A,Q,c,True)\n return out,A,transcript\n\nsynthesize=FunctionType(alt.synthesize.__code__,dict(alt.synthesize.__globals__,old=ca.old,absorb=absorb_odd,absorb_even=absorb_even),\'synthesize\',alt.synthesize.__defaults__,alt.synthesize.__closure__)\nunsynthesize=FunctionType(alt.unsynthesize.__code__,dict(alt.unsynthesize.__globals__,old=ca.old,absorb=absorb_odd,absorb_even=absorb_even),\'unsynthesize\',alt.unsynthesize.__defaults__,alt.unsynthesize.__closure__)\n', 'tasks/dqi/fused/source/playbook/exp/cf_check/audit.py': '"""Standalone classical finite-field EXTRACT/SYNTHESIZE audit (no circuits/verifier)."""\nimport argparse, json, random, time\nfrom collections import Counter\n\nclass Field:\n    def __init__(self, m, modulus):\n        self.m=m; self.n=1<<m; self.mask=self.n-1\n        self.exp=[]; self.log=[-1]*self.n; x=1\n        for i in range(self.n-1):\n            assert self.log[x]<0\n            self.log[x]=i; self.exp.append(x)\n            x<<=1\n            if x&self.n:x^=modulus\n        assert x==1\n        self.exp*=2\n        self.K=next(x for x in range(1,self.n) if self.trace(x)==1)\n    def mul(self,a,b):return self.exp[self.log[a]+self.log[b]] if a and b else 0\n    def inv(self,a):return self.exp[self.n-1-self.log[a]] if a else 0\n    def div(self,a,b):return self.mul(a,self.inv(b))\n    def trace(self,a):\n        t=0\n        for _ in range(self.m):t^=a;a=self.mul(a,a)\n        return t\n    def emul(self,u,v):\n        a=u&self.mask;b=u>>self.m;c=v&self.mask;d=v>>self.m\n        bd=self.mul(b,d)\n        return (self.mul(a,c)^self.mul(self.K,bd)) | ((self.mul(a,d)^self.mul(b,c)^bd)<<self.m)\n    def einv(self,u):\n        a=u&self.mask;b=u>>self.m\n        ni=self.inv(self.mul(a,a)^self.mul(a,b)^self.mul(self.K,self.mul(b,b)))\n        return self.mul(a^b,ni)|(self.mul(b,ni)<<self.m)\n    def escale(self,u,a):return self.mul(u&self.mask,a)|(self.mul(u>>self.m,a)<<self.m)\n    def ev(self,p,u):\n        out=0\n        for a in reversed(p):out=self.emul(out,u)^a\n        return out\n    def trim(self,p):\n        while len(p)>1 and not p[-1]:p.pop()\n        return p\n    def add(self,a,b):\n        c=[0]*max(len(a),len(b))\n        for i,x in enumerate(a):c[i]^=x\n        for i,x in enumerate(b):c[i]^=x\n        return self.trim(c)\n    def pmul(self,a,b):\n        c=[0]*(len(a)+len(b)-1)\n        for i,x in enumerate(a):\n            for j,y in enumerate(b):c[i+j]^=self.mul(x,y)\n        return self.trim(c)\n    def scale(self,p,c):return self.trim([self.mul(x,c) for x in p])\n    def pdiv(self,a,b):\n        a=a.copy();out=[0]*max(1,len(a)-len(b)+1);bi=self.inv(b[-1])\n        assert b!=[0]\n        while a!=[0] and len(a)>=len(b):\n            d=len(a)-len(b);v=self.mul(a[-1],bi);out[d]=v\n            for i,x in enumerate(b):a[i+d]^=self.mul(v,x)\n            self.trim(a)\n        return self.trim(out),a\n    def gcd(self,a,b):\n        while b!=[0]:a,b=b,self.pdiv(a,b)[1]\n        return self.scale(a,self.inv(a[-1]))\n    def derivative(self,p):return self.trim([p[i] if i%2 else 0 for i in range(1,len(p))] or [0])\n\ndef data(f, cs, betas, ws):\n    kappas=[f.mul(c,c)^c^f.K for c in cs]\n    nodes=[c|f.n for c in cs]\n    D=[1]\n    for b in betas:D=f.pmul(D,[b,1])\n    N=[0]\n    for b,w in zip(betas,ws):\n        quotient,rem=f.pdiv(D,[b,1]);assert rem==[0]\n        N=f.add(N,f.scale(quotient,w))\n    vals=[f.emul(f.ev(N,g),f.einv(f.ev(D,g))) for g in nodes]\n    return kappas,nodes,N,D,vals\n\ndef extract(f,cs,kappas,vals):\n    vals=vals.copy();heads=[];zeros=[]\n    for i,c in enumerate(cs):\n        b=vals[i]>>f.m;a=(vals[i]&f.mask)^f.mul(b,c);heads.append((a,b))\n        for j in range(i+1,len(cs)):\n            u=vals[j]^a^f.mul(b,cs[j])^(b<<f.m)\n            if not u:zeros.append((i,j))\n            vals[j]=f.escale(f.einv(u),kappas[i]^kappas[j])\n    return heads,zeros\n\ndef synth(f,kappas,heads):\n    p,q,d=0,1,0\n    for a,b in heads:p,q,d=q,p^f.mul(b,q),d^f.mul(p,q)^f.mul(a^b,f.mul(q,q))\n    tail=(p,q,d)\n    if q:A,B=[f.div(d,q),p],[q]\n    else:A,B=[p],[0]\n    for (a,b),kappa in reversed(list(zip(heads,kappas))):\n        A,B=f.add(f.pmul([a,b],A),f.pmul([kappa,1,1],B)),A\n    if not q:A,B=[0]+A,[0]+B\n    return f.trim(A),f.trim(B),tail\n\ndef rational_audit(f,kappas,nodes,N,D,heads):\n    """Follow exact rational transforms until the first genuine pole or termination."""\n    first_poles=[];termination=None\n    for i,(a,b) in enumerate(heads):\n        if f.ev(D,nodes[i])==0:return {\'head_pole\':i,\'poles\':first_poles}\n        assert f.ev(N,nodes[i])==f.emul(a^f.mul(b,nodes[i]&f.mask)^(b<<f.m),f.ev(D,nodes[i]))\n        H=f.add(N,f.pmul([a,b],D))\n        if H==[0]:termination=i;break\n        H,rem=f.pdiv(H,[kappas[i],1,1]);assert rem==[0]\n        poles=[j for j in range(i+1,len(nodes)) if f.ev(H,nodes[j])==0]\n        if poles:return {\'first_nonterminal_zero_step\':i,\'poles\':poles,\'residual_degree\':len(H)-1}\n        N,D=D,H\n    return {\'termination\':termination,\'poles\':first_poles}\n\ndef validate(m,modulus,h,repeats,seed):\n    f=Field(m,modulus);rng=random.Random(seed);cs=list(range(0,2*h,2));assert max(cs)<f.n\n    results=Counter();examples=[]\n    for t in range(h+1):\n        for trial in range(repeats):\n            betas=rng.sample(range(1,f.n),t);ws=[rng.randrange(1,f.n) for _ in betas]\n            kappas,nodes,N,D,vals=data(f,cs,betas,ws)\n            heads,zeros=extract(f,cs,kappas,vals);A,B,tail=synth(f,kappas,heads)\n            ra=rational_audit(f,kappas,nodes,N,D,heads)\n            correct=f.pmul(A,D)==f.pmul(B,N)\n            sqfree=f.gcd(B,f.derivative(B))==[1]\n            degree=len(B)-1\n            result={\'t\':t,\'trial\':trial,\'correct\':correct,\'degree\':degree,\'monic\':B[-1]==1,\'squarefree\':sqfree,\'tail\':tail,\'rational\':ra,\'zeros\':len(zeros)}\n            results[\'cases\']+=1;results[\'correct\']+=correct;results[\'monic\']+=B[-1]==1;results[\'degree_h\']+=degree==h;results[\'squarefree\']+=sqfree\n            if \'termination\' in ra and ra[\'termination\'] is not None:\n                term=ra[\'termination\'];results[\'terminated\']+=1\n                assert all(a==b==0 for a,b in heads[term+1:])\n                results[\'later_heads_zero\']+=1\n            if not correct or not sqfree or degree!=h or not result[\'monic\']:\n                if len(examples)<12:examples.append(dict(result,betas=betas,ws=ws,heads=heads,A=A,B=B))\n            if correct:\n                pad,rem=f.pdiv(B,D);assert rem==[0]\n                expected=D.copy()\n                for j in range(t+1,h,2):expected=f.pmul(expected,[kappas[j],1,1])\n                if not t%2:expected=[0]+expected\n                assert expected==B,(\'padding index pattern\',t,B,expected)\n                allowed=[kappas[i] for i in range(h)]\n                if not tail[1]:\n                    pad,rem=f.pdiv(pad,[0,1]);assert rem==[0]\n                for k in allowed:\n                    quotient,rem=f.pdiv(pad,[k,1,1])\n                    if rem==[0]:pad=quotient\n                assert pad==[1],pad\n                assert all(f.ev(B,b)!=0 for b in range(1,f.n) if b not in betas)\n    return {\'m\':m,\'K\':f.K,\'h\':h,\'repeats_per_weight\':repeats,\'counts\':dict(results),\'examples\':examples}\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--m\',type=int,default=12);parser.add_argument(\'--h\',type=int,default=35);parser.add_argument(\'--repeats\',type=int,default=3);parser.add_argument(\'--seed\',type=int,default=1)\n    args=parser.parse_args();mods={3:0b1011,4:0b10011,6:0b1000011,12:0x1053}\n    start=time.time();result=validate(args.m,mods[args.m],args.h,args.repeats,args.seed);result[\'seconds\']=time.time()-start\n    print(json.dumps(result,indent=2))\n', 'tasks/dqi/generate_fused.py': '# SPDX-License-Identifier: Apache-2.0\n# Source port; not independent clean-room authorship. See fused/PROVENANCE.md.\n"""Build SomeJoint\'s fused DQI decoder from committed source; write only OUT/dqi.kmb.\n\nThe vendored files under fused/source/ are byte-identical to the notebook\'s\nembedded source. Their imports rely on the author\'s sys.path layout, so the\nunchanged build.py runs as a script in its own interpreter, the way its author\nruns it. Every vendored file is checked against SOURCE_MAP.json first. No\nnotebook, circuit, data file or network access is an input.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nPORT = Path(__file__).resolve().parent / \'fused\'\nBUILD = PORT / \'source/exp/decoder/build.py\'\n\n\ndef sha256(path):\n    with open(path, \'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef check_pins():\n    """Fail before building if any vendored file differs from its pin."""\n    files = json.loads((PORT / \'SOURCE_MAP.json\').read_text())[\'files\']\n    drift = [item[\'file\'] for item in files if sha256(PORT / item[\'file\']) != item[\'port_sha256\']]\n    if drift:\n        raise SystemExit(f\'vendored source differs from SOURCE_MAP.json: {drift}\')\n    return len(files)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--out\', type=Path, required=True)\n    args = parser.parse_args()\n    started = time.monotonic()\n    pinned = check_pins()\n    with tempfile.TemporaryDirectory(prefix=\'dqi-fused-\') as scratch:\n        built = Path(scratch) / \'dqi.kmb\'\n        # -B keeps __pycache__ out of the source tree; build.py also writes dqi.build.json,\n        # which stays in scratch so OUT holds only the circuit.\n        subprocess.run([sys.executable, \'-B\', str(BUILD), str(built)], cwd=scratch, check=True,\n                       stdout=sys.stderr)\n        record = json.loads(built.with_suffix(\'.build.json\').read_text())\n        args.out.mkdir(parents=True, exist_ok=True)\n        path = args.out / \'dqi.kmb\'\n        shutil.move(built, path)\n    digest = sha256(path)\n    print(json.dumps(dict(variant=\'fused-confluent\', pinned_files=pinned, num_qubits=record[\'Q\'],\n                          static_toffoli_like=record[\'T_static\'], num_ops=record[\'num_ops\'],\n                          num_bits=record[\'num_bits\'], build_seconds=time.monotonic() - started,\n                          sha256=digest, bytes=path.stat().st_size, stages=record[\'stages\']),\n                     indent=2), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'tasks/invsqrt/__init__.py': '"""Invsqrt generator, integer mirror, and ledger hooks."""\n', 'tasks/invsqrt/affine.py': '"""Exact-domain affine rewrite for the invsqrt circuit.\n\nThis file is a port of the method below. It is not independent authorship.\n\nMethod parents (Apache-2.0), reimplemented here:\n- Kozykorpesh, complete-domain affine compilation v2: linear GF(2) replacement\n  of Toffoli and CSWAP, and reuse of qubit ids whose value is identically 0.\n- anazemcev, source & exact compilation v4: measurement uncompute when the\n  Toffoli result is an affine function of the other wires. The kickback is\n  cancelled by one conditioned CZ. The conditional-Z affine-measurement\n  variant is not used.\n\nEvery accepted replacement is checked against the full 3,133,452-input truth\nvector. A sampled linear system only proposes a combination. Missed rewrites\nare allowed. An invalid replacement is not.\n\nPhase recorded here is the unconditional Z_POW and unconditional CZ/Z\ncontribution. Conditioned diagonals are kickback fixups and are not added.\n"""\n\nfrom __future__ import annotations\n\nimport collections\nimport heapq\nimport re\nfrom decimal import Decimal\nfrom fractions import Fraction\n\nimport numpy as np\n\nMAX_X = 3_133_452\nWORDS = (MAX_X + 63) // 64\n_QUBIT = re.compile(r"(?<!\\w)q(\\d+)(?!\\w)")\n_BIT = re.compile(r"(?<!\\w)b(\\d+)(?!\\w)")\n_METADATA = {\n    "REGISTER",\n    "INFO_TAG",\n    "APPEND_TO_REGISTER",\n    "BIT_STORE0",\n    "BIT_STORE1",\n    "BIT_INVERT",\n    "BIT_XOR",\n}\n_DIAGONAL = {"Z", "CZ", "CCZ", "Z_POW", "NEG"}\n\n\ndef _pack_input_bit(xs: np.ndarray, bit: int) -> np.ndarray:\n    flags = ((xs >> bit) & 1).astype(np.uint8)\n    packed = np.packbits(flags, bitorder="little")\n    buffer = np.zeros(WORDS * 8, dtype=np.uint8)\n    buffer[: len(packed)] = packed\n    return np.ascontiguousarray(buffer.view(np.uint64)[:WORDS])\n\n\nclass CompiledCircuit:\n    """Affine-compiled text plus the phase certificate collected on the way."""\n\n    def __init__(self, text, phase, rotations, peak, rewrites, fresh_bits):\n        self.text = text\n        self.phase = phase\n        self.rotations = rotations\n        self.peak = peak\n        self.rewrites = rewrites\n        self.fresh_bits = fresh_bits\n\n\nclass DomainCompiler:\n    def __init__(self, text: str):\n        self.lines = [line for line in text.splitlines() if line]\n        self.protected = set()\n        for line in self.lines:\n            if line.startswith("APPEND_TO_REGISTER"):\n                self.protected.update(int(match) for match in _QUBIT.findall(line))\n        if self.protected != set(range(22)):\n            raise RuntimeError(f"Expected input qubits 0..21, found {sorted(self.protected)}")\n        seen_bits = [int(match) for line in self.lines for match in _BIT.findall(line)]\n        self.fresh_bit = 1 + max(seen_bits, default=-1)\n        self.zero = np.zeros(WORDS, dtype=np.uint64)\n        self.full = np.full(WORDS, np.uint64(2**64 - 1))\n        if MAX_X % 64:\n            self.full[-1] = np.uint64((1 << (MAX_X % 64)) - 1)\n        self.full.setflags(write=False)\n        self.zero.setflags(write=False)\n        xs = np.arange(1, MAX_X + 1, dtype=np.uint64)\n        self.original = [_pack_input_bit(xs, bit) for bit in range(22)]\n        if int(self.original[0][0] & 1) != 1:\n            raise RuntimeError("Input bit packing is not little-endian on x=1")\n        self.values = {qubit: self.original[qubit] for qubit in range(22)}\n        self.mapping = {qubit: qubit for qubit in range(22)}\n        self.free: list[int] = []\n        self.next_id = 22\n        self.peak = 22\n        self.depth = 0\n        self.stack: list[dict] = []\n        self.out: list[str] = []\n        self.rewrites: collections.Counter = collections.Counter()\n        self.phase = np.zeros(MAX_X, dtype=np.uint64)\n        self.rotations: list[dict] = []\n        self.sample = np.unique(\n            np.r_[\n                np.arange(min(8, WORDS)),\n                np.linspace(0, WORDS - 1, 8, dtype=int),\n            ]\n        )\n        self.false_candidates = 0\n\n    def val(self, qubit: int):\n        return self.values.get(qubit, self.zero)\n\n    def canon(self, array: np.ndarray):\n        if array is self.zero or array is self.full:\n            return array\n        if not array.any():\n            return self.zero\n        if np.array_equal(array, self.full):\n            return self.full\n        array.setflags(write=False)\n        return array\n\n    def equal(self, left, right) -> bool:\n        if left is right:\n            return True\n        return bool(np.array_equal(left, right))\n\n    def xor(self, left, right):\n        if left is self.zero:\n            return right\n        if right is self.zero:\n            return left\n        if left is right:\n            return self.zero\n        return self.canon(np.bitwise_xor(left, right))\n\n    def band(self, left, right):\n        if left is self.zero or right is self.zero:\n            return self.zero\n        if left is self.full:\n            return right\n        if right is self.full or left is right:\n            return left\n        return self.canon(np.bitwise_and(left, right))\n\n    def materialize(self, qubit: int) -> int:\n        if qubit not in self.mapping:\n            if self.free:\n                self.mapping[qubit] = heapq.heappop(self.free)\n            else:\n                self.mapping[qubit] = self.next_id\n                self.next_id += 1\n                self.peak = self.next_id\n        return self.mapping[qubit]\n\n    def release(self):\n        if self.depth:\n            return\n        for qubit in list(self.mapping):\n            if qubit in self.protected:\n                continue\n            if self.val(qubit) is self.zero:\n                heapq.heappush(self.free, self.mapping.pop(qubit))\n                self.values.pop(qubit, None)\n\n    def emit(self, line: str):\n        def replace(match):\n            qubit = int(match.group(1))\n            return f"q{self.materialize(qubit)}"\n\n        self.out.append(_QUBIT.sub(replace, line))\n\n    def literal(self, function, excluded: set[int]):\n        if function is self.zero or function is self.full:\n            return None\n        for qubit, value in self.values.items():\n            if qubit not in excluded and self.equal(function, value):\n                return qubit, False\n        complement = self.xor(function, self.full)\n        for qubit, value in self.values.items():\n            if qubit not in excluded and self.equal(complement, value):\n                return qubit, True\n        return None\n\n    def _signature(self, value, extra: list[int]) -> int:\n        acc = 0\n        for index in self.sample:\n            acc = (acc << 64) | int(value[index])\n        for index in extra:\n            acc = (acc << 64) | int(value[index])\n        return acc\n\n    def linear_terms(self, function, excluded: set[int]):\n        """Propose an affine combination on samples and prove it on every x."""\n        if function is self.zero:\n            return [], False\n        if function is self.full:\n            return [], True\n        variables = [\n            qubit\n            for qubit, value in self.values.items()\n            if qubit not in excluded and value is not self.zero and value is not self.full\n        ]\n        columns = [self.full] + [self.values[qubit] for qubit in variables]\n        extra: list[int] = []\n        for _attempt in range(8):\n            basis = {}\n            for index, value in enumerate(columns):\n                vector = self._signature(value, extra)\n                mask = 1 << index\n                while vector:\n                    pivot = vector.bit_length() - 1\n                    if pivot not in basis:\n                        basis[pivot] = (vector, mask)\n                        break\n                    existing, coeff = basis[pivot]\n                    vector ^= existing\n                    mask ^= coeff\n            vector = self._signature(function, extra)\n            mask = 0\n            while vector:\n                pivot = vector.bit_length() - 1\n                if pivot not in basis:\n                    return None\n                existing, coeff = basis[pivot]\n                vector ^= existing\n                mask ^= coeff\n            chosen = [index for index in range(len(columns)) if (mask >> index) & 1]\n            exact = self.zero\n            for index in chosen:\n                exact = self.xor(exact, columns[index])\n            if self.equal(exact, function):\n                return [variables[index - 1] for index in chosen if index], (0 in chosen)\n            self.false_candidates += 1\n            mismatch = np.flatnonzero(np.not_equal(exact, function))\n            if len(mismatch) == 0 or int(mismatch[0]) in extra:\n                return None\n            extra.append(int(mismatch[0]))\n        return None\n\n    def _clifford(self, target: int, terms, constant: bool) -> list[str]:\n        lines = [f"X q{target}"] if constant else []\n        lines.extend(f"CX q{qubit} q{target}" for qubit in terms)\n        return lines\n\n    def _add_phase(self, bits, units: int):\n        if units == 0 or bits is self.zero:\n            return\n        if bits is self.full:\n            self.phase += np.uint64(units)\n            return\n        flat = np.unpackbits(bits.view(np.uint8), bitorder="little")[:MAX_X]\n        self.phase[flat.astype(bool)] += np.uint64(units)\n\n    def _record_rotation(self, logical: int, theta: Fraction, bits):\n        counted = (theta * 4).denominator != 1\n        units = theta * (1 << 63)\n        if units.denominator != 1:\n            raise RuntimeError(f"Rotation {theta} is not on the 2^-64 lattice")\n        flat = np.ones(MAX_X, dtype=np.uint8) if bits is self.full else (\n            np.unpackbits(bits.view(np.uint8), bitorder="little")[:MAX_X].astype(np.uint8)\n        )\n        self.rotations.append(\n            {\n                "line": len(self.out),\n                "logical": logical,\n                "theta": theta,\n                "counted": counted,\n                "units": int(units),\n                "bits": flat,\n            }\n        )\n\n    def _conditioned(self, parts: list[str]) -> bool:\n        return self.depth > 0 or "if" in parts\n\n    def execute(self, line: str):\n        parts = line.split()\n        op = parts[0]\n        if op in _METADATA:\n            self.emit(line)\n            return\n        if op == "PUSH_CONDITION":\n            self.stack.append(dict(self.values))\n            self.depth += 1\n            self.emit(line)\n            return\n        if op == "POP_CONDITION":\n            before = self.stack.pop()\n            self.depth -= 1\n            if self.depth < 0:\n                raise RuntimeError("Unbalanced POP_CONDITION")\n            keys = set(before) | set(self.values)\n            for qubit in keys:\n                if not self.equal(self.val(qubit), before.get(qubit, self.zero)):\n                    raise RuntimeError(f"Condition join changed q{qubit}")\n            self.emit(line)\n            self.release()\n            return\n        qubits = [int(match) for match in _QUBIT.findall(line)]\n        if op in _DIAGONAL and self._conditioned(parts):\n            if op == "Z_POW":\n                raise RuntimeError(f"Conditioned rotation is outside this compiler: {line}")\n            self.emit(line)\n            return\n        if "if" in parts and op not in _DIAGONAL:\n            raise RuntimeError(f"Inline condition on {op} is unsupported: {line}")\n        if op == "CCX":\n            left, right, target = qubits\n            product = self.band(self.val(left), self.val(right))\n            replacement = None\n            reason = None\n            if product is self.zero:\n                replacement, reason = [], "ccx_zero"\n            elif product is self.full:\n                replacement, reason = [f"X q{target}"], "ccx_one"\n            else:\n                found = self.literal(product, {target})\n                if found is not None:\n                    qubit, inverted = found\n                    replacement = self._clifford(target, [qubit], inverted)\n                    reason = "ccx_literal"\n                else:\n                    terms = self.linear_terms(product, {target})\n                    if terms is not None:\n                        variables, constant = terms\n                        replacement = self._clifford(target, variables, constant)\n                        reason = "ccx_affine"\n            if replacement is None and self.depth == 0:\n                result = self.xor(self.val(target), product)\n                terms = None\n                if result is self.zero:\n                    terms = [], False\n                elif result is self.full:\n                    terms = [], True\n                else:\n                    found = self.literal(result, {target})\n                    if found is not None:\n                        qubit, inverted = found\n                        terms = [qubit], inverted\n                    else:\n                        terms = self.linear_terms(result, {target})\n                if terms is not None:\n                    variables, constant = terms\n                    prepare = self._clifford(target, variables, constant)\n                    bit = self.fresh_bit\n                    self.fresh_bit += 1\n                    replacement = prepare + [\n                        f"HMR q{target} b{bit}",\n                        f"CZ q{left} q{right} if b{bit}",\n                    ] + prepare\n                    reason = "ccx_mbu"\n            if replacement is None:\n                self.emit(line)\n            else:\n                self.rewrites[reason] += 1\n                for emitted in replacement:\n                    self.emit(emitted)\n            self.values[target] = self.xor(self.val(target), product)\n            self.release()\n            return\n        if op == "CSWAP":\n            control, left, right = qubits\n            delta = self.xor(self.val(left), self.val(right))\n            product = self.band(self.val(control), delta)\n            replacement = None\n            reason = None\n            if product is self.zero:\n                replacement, reason = [], "cswap_zero"\n            elif self.equal(product, delta):\n                replacement, reason = [f"SWAP q{left} q{right}"], "cswap_swap"\n            else:\n                found = self.literal(product, {left, right})\n                if found is not None:\n                    qubit, inverted = found\n                    replacement = []\n                    if inverted:\n                        replacement += [f"X q{left}", f"X q{right}"]\n                    replacement += [f"CX q{qubit} q{left}", f"CX q{qubit} q{right}"]\n                    reason = "cswap_literal"\n                else:\n                    terms = self.linear_terms(product, {left, right})\n                    if terms is not None:\n                        variables, constant = terms\n                        replacement = []\n                        if constant:\n                            replacement += [f"X q{left}", f"X q{right}"]\n                        for qubit in variables:\n                            replacement += [f"CX q{qubit} q{left}", f"CX q{qubit} q{right}"]\n                        reason = "cswap_affine"\n            if replacement is None:\n                self.emit(line)\n            else:\n                self.rewrites[reason] += 1\n                for emitted in replacement:\n                    self.emit(emitted)\n            self.values[left] = self.xor(self.val(left), product)\n            self.values[right] = self.xor(self.val(right), product)\n            self.release()\n            return\n        if op == "CX":\n            control, target = qubits\n            control_value = self.val(control)\n            if control_value is self.zero:\n                self.rewrites["cx_zero"] += 1\n            elif control_value is self.full:\n                self.rewrites["cx_one"] += 1\n                self.emit(f"X q{target}")\n            else:\n                self.emit(line)\n            self.values[target] = self.xor(self.val(target), control_value)\n            self.release()\n            return\n        if op == "X":\n            for qubit in qubits:\n                self.values[qubit] = self.xor(self.val(qubit), self.full)\n            self.emit(line)\n            self.release()\n            return\n        if op == "SWAP":\n            left, right = qubits\n            self.values[left], self.values[right] = self.val(right), self.val(left)\n            self.emit(line)\n            self.release()\n            return\n        if op == "R":\n            if all(self.val(qubit) is self.zero for qubit in qubits):\n                self.rewrites["r_zero"] += 1\n            else:\n                self.emit(line)\n            for qubit in qubits:\n                self.values[qubit] = self.zero\n            self.release()\n            return\n        if op == "HMR":\n            self.emit(line)\n            for qubit in qubits[:1]:\n                self.values[qubit] = self.zero\n            self.release()\n            return\n        if op == "Z_POW":\n            theta = Fraction(Decimal(parts[2]))\n            bits = self.val(qubits[0])\n            if bits is self.zero:\n                self.rewrites["zpow_zero"] += 1\n            else:\n                self._record_rotation(qubits[0], theta, bits)\n                self.emit(line)\n                units = theta * (1 << 63)\n                self._add_phase(bits, int(units))\n            self.release()\n            return\n        if op in {"Z", "CZ", "CCZ"}:\n            product = self.full\n            for qubit in qubits:\n                product = self.band(product, self.val(qubit))\n            units = 1 << 63\n            if product is self.zero:\n                self.rewrites[f"{op.lower()}_zero"] += 1\n            else:\n                self.emit(line)\n                self._add_phase(product, units)\n            self.release()\n            return\n        if op == "NEG":\n            self.emit(line)\n            self.phase += np.uint64(1 << 63)\n            self.release()\n            return\n        raise RuntimeError(f"Unsupported operation: {line}")\n\n    def compile(self) -> CompiledCircuit:\n        for number, line in enumerate(self.lines, start=1):\n            self.execute(line)\n            if number % 4000 == 0:\n                print(\n                    f"affine {number}/{len(self.lines)} rewrites {sum(self.rewrites.values())} "\n                    f"peak {self.peak} false {self.false_candidates}",\n                    flush=True,\n                )\n        if self.depth or self.stack:\n            raise RuntimeError("Unbalanced condition stack")\n        for qubit in range(22):\n            if not self.equal(self.val(qubit), self.original[qubit]):\n                raise RuntimeError(f"Input q{qubit} was not restored")\n        for qubit, value in self.values.items():\n            if qubit not in self.protected and value is not self.zero:\n                raise RuntimeError(f"Scratch q{qubit} was not restored")\n        ids = list(self.mapping.values())\n        if len(ids) != len(set(ids)):\n            raise RuntimeError("Physical qubit map is not injective")\n        text = "\\n".join(self.out) + "\\n"\n        return CompiledCircuit(\n            text,\n            self.phase,\n            self.rotations,\n            self.peak,\n            dict(self.rewrites),\n            self.fresh_bit,\n        )\n\n\ndef compile_circuit(circuit) -> CompiledCircuit:\n    """Rewrite `circuit` and return text, phase and rotation activations."""\n    return DomainCompiler(str(circuit)).compile()\n', 'tasks/invsqrt/calibrate.py': '"""Drop one invsqrt rotation when every x keeps a positive margin.\n\nThis file is our scipy linprog calibrator. The slack-maximising drop is\ncredited to LightningV08 and to Mizero Lucky Gall. Mizero v1 has no\nnotebook-wide licence grant and is reference-only: it is not loaded, and it\nis not a shipping parent. This file is not a port of that notebook.\n\nThe linear program maximises the worst slack. Rounding is then checked in\nintegers on all 3,133,452 inputs. No verifier salt, calibration file, or\ncircuit binary is an input.\n\nThe activation bits and the starting phase come from the affine compiler.\nOnly unconditioned, exactly representable Z_POW gates are moved. Clifford\nZ_POW gates stay fixed.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom fractions import Fraction\n\nimport numpy as np\nfrom scipy.optimize import linprog\n\nfrom tasks.invsqrt.peephole import optimize\n\nMAX_X = 3_133_452\nUNIT = 1 << 64\nHALF = UNIT // 2\nREFERENCE_COST = 2_076_774.233715282\n\n\ndef reference_target_tolerance(xs: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    """Official invsqrt target and tolerance for beta = 1, precision 24.\n\n    `r = isqrt(floor(2^130 / x))`, target is `r` rounded at bit 0, tolerance\n    is `min(r >> 25, 2^63)`.\n    """\n    roots = [math.isqrt((1 << 130) // int(x)) for x in xs.tolist()]\n    target = np.array([((root + 1) >> 1) & (UNIT - 1) for root in roots], dtype=np.uint64)\n    tolerance = np.array([min(root >> 25, 1 << 63) for root in roots], dtype=np.uint64)\n    return target, tolerance\n\n\ndef model_phase(cfg, tables) -> np.ndarray:\n    """Integer-model phase in units of 2^-64 turns, for every x."""\n    from tasks.invsqrt.signed_digit import integer_model\n\n    xs = np.arange(1, MAX_X + 1, dtype=np.int64)\n    shift = np.uint64(64 - (cfg.fy + 10))\n    phase = np.empty(MAX_X, dtype=np.uint64)\n    for start in range(0, MAX_X, 131072):\n        block = xs[start : start + 131072]\n        values = integer_model(block, cfg, tables).astype(np.uint64)\n        phase[start : start + len(block)] = values << shift\n    return phase\n\n\ndef assert_phase_matches(compiled, cfg, tables) -> dict:\n    """Require the compiled unconditional phase to equal the integer model."""\n    expected = model_phase(cfg, tables)\n    mismatch = np.flatnonzero(compiled.phase != expected)\n    if len(mismatch):\n        index = int(mismatch[0])\n        raise RuntimeError(\n            "Compiled phase does not match the integer model at "\n            f"x={index + 1}: circuit {int(compiled.phase[index])} "\n            f"model {int(expected[index])} mismatches {len(mismatch)}"\n        )\n    return {"phase_match": True, "inputs": MAX_X}\n\n\ndef _matvec(bits: np.ndarray, columns: list[int], delta: np.ndarray) -> np.ndarray:\n    acc = np.zeros(bits.shape[0], dtype=np.float64)\n    for index, column in enumerate(columns):\n        if delta[index] == 0:\n            continue\n        acc += bits[:, column] * float(delta[index])\n    return acc\n\n\ndef _fit(error, tolerance, bits, free, fixed, bound_units, scale=2.0**28):\n    """Maximise slack with cutting planes. `fixed` is the dropped-rotation shift."""\n    residual = (error.astype(np.float64) + fixed) / scale\n    limit = tolerance.astype(np.float64) / scale\n    width = len(free)\n    active = set(np.linspace(0, MAX_X - 1, 4096, dtype=np.int64).tolist())\n    initial = limit - np.abs(residual)\n    active.update(np.argpartition(initial, 4095)[:4096].tolist())\n    delta = np.zeros(width, dtype=np.float64)\n    margin = 0.0\n    converged = False\n    iterations = 0\n    for iterations in range(1, 61):\n        chosen = np.fromiter(sorted(active), dtype=np.int64)\n        block = bits[chosen][:, free].astype(np.float64)\n        matrix = np.vstack(\n            [np.c_[block, np.ones(len(chosen))], np.c_[-block, np.ones(len(chosen))]]\n        )\n        rhs = np.r_[limit[chosen] - residual[chosen], limit[chosen] + residual[chosen]]\n        bounds = [(-bound_units / scale, bound_units / scale)] * width + [(None, None)]\n        result = linprog(\n            np.r_[np.zeros(width), -1.0],\n            A_ub=matrix,\n            b_ub=rhs,\n            bounds=bounds,\n            method="highs",\n        )\n        if not result.success:\n            raise RuntimeError(result.message)\n        delta, margin = result.x[:width], float(result.x[width])\n        corrected = residual + _matvec(bits, free, delta)\n        slack = limit - np.abs(corrected)\n        actual = float(slack.min())\n        print(\n            f"calibration LP {iterations}: rows {len(chosen)} "\n            f"lp {margin * scale:.3f} exhaustive {actual * scale:.3f}",\n            flush=True,\n        )\n        if actual >= margin - 1e-9 * max(1.0, abs(margin)):\n            converged = True\n            break\n        added = set(np.argpartition(slack, 399)[:400].tolist())\n        if added.issubset(active):\n            break\n        active.update(added)\n    return np.rint(delta * scale).astype(np.int64), {\n        "iterations": iterations,\n        "lp_converged": converged,\n        "active_inputs": len(active),\n    }\n\n\ndef _exact_margin(error, tolerance, bits, deltas) -> tuple[int, int]:\n    shift = np.zeros(MAX_X, dtype=np.int64)\n    for index, delta in enumerate(deltas):\n        if int(delta) == 0:\n            continue\n        shift += bits[:, index].astype(np.int64) * np.int64(delta)\n    largest = max(abs(int(error.min())), abs(int(error.max())))\n    if largest + sum(abs(int(delta)) for delta in deltas) >= HALF:\n        raise RuntimeError("Rounded correction can overflow signed 64-bit phase")\n    corrected = error.astype(np.int64) + shift\n    slack = tolerance.astype(np.int64) - np.abs(corrected)\n    worst = int(np.argmin(slack))\n    return int(slack[worst]), worst\n\n\ndef _angle_key(theta: Fraction) -> float:\n    reduced = theta % 2\n    return abs(float(min(reduced, 2 - reduced)))\n\n\ndef _estimate(qubits, expected_toffoli, t_like, rotations, margin) -> dict:\n    if rotations <= 0 or margin <= 0:\n        return {"estimated_expected_cost": math.inf, "ratio": 0.0, "per_rotation_t": None}\n    epsilon = math.pi * (margin / UNIT) / rotations\n    per_rotation = 0.56 * math.log2(1 / epsilon) + 5.3\n    extra = (t_like + rotations * per_rotation) / 4\n    cost = qubits * (expected_toffoli + extra)\n    return {\n        "estimated_expected_cost": cost,\n        "ratio": REFERENCE_COST / cost,\n        "per_rotation_t": per_rotation,\n        "epsilon": epsilon,\n    }\n\n\ndef _rewrite(lines: list[str], rotations: list[dict], deltas: dict[int, int], dropped: set[int]) -> str:\n    from decimal import Decimal, localcontext\n\n    def exponent_text(theta: Fraction) -> str:\n        with localcontext() as ctx:\n            ctx.prec = 80\n            rendered = format(Decimal(theta.numerator) / Decimal(theta.denominator), "f")\n        if "." not in rendered:\n            rendered += ".0"\n        return rendered\n\n    remove = set()\n    out = list(lines)\n    for index, rotation in enumerate(rotations):\n        if index in dropped:\n            remove.add(rotation["line"])\n            continue\n        delta = int(deltas.get(index, 0))\n        if delta == 0:\n            continue\n        theta = rotation["theta"] + Fraction(delta, HALF)\n        qubit = lines[rotation["line"]].split()[1]\n        out[rotation["line"]] = f"Z_POW {qubit} {exponent_text(theta)}"\n    body = "\\n".join(line for index, line in enumerate(out) if index not in remove and line)\n    return body + "\\n"\n\n\ndef calibrate(compiled, resources: dict) -> dict:\n    """Try the original angles and each single counted drop. Keep the cheapest legal one."""\n    rotations = compiled.rotations\n    counted = [index for index, rotation in enumerate(rotations) if rotation["counted"]]\n    if not counted:\n        raise RuntimeError("The compiled circuit has no counted rotation")\n    bits = np.column_stack([rotation["bits"] for rotation in rotations])\n    xs = np.arange(1, MAX_X + 1, dtype=np.int64)\n    _target, tolerance = reference_target_tolerance(xs)\n    # Reuse the packed target. reference_target_tolerance already walked every x.\n    target = _target\n    error = (compiled.phase - target).view(np.int64)\n    order = sorted(counted, key=lambda index: _angle_key(rotations[index]["theta"]))\n    # The public build drops one rotation. Try the original and the smallest\n    # angle first. Search the other single drops only when that drop has no margin.\n    candidates = [(), (order[0],)]\n    best = None\n    trials = []\n    expanded = False\n    for dropped in candidates:\n        fixed = np.zeros(MAX_X, dtype=np.float64)\n        bound = float(1 << 36)\n        for index in dropped:\n            units = rotations[index]["units"]\n            fixed += bits[:, index] * float(-units)\n            bound = max(bound, 4.0 * abs(units))\n        free = [index for index in counted if index not in dropped]\n        rounded, fit_report = _fit(error, tolerance, bits, free, fixed, bound)\n        deltas = np.zeros(len(rotations), dtype=np.int64)\n        for column, index in enumerate(free):\n            deltas[index] = rounded[column]\n        for index in dropped:\n            deltas[index] = -int(rotations[index]["units"])\n        margin, worst = _exact_margin(error, tolerance, bits, deltas)\n        kept = len(counted) - len(dropped)\n        estimate = _estimate(\n            resources["qubits"],\n            resources["expected_toffoli"],\n            resources["t_like"],\n            kept,\n            margin,\n        )\n        trial = {\n            "dropped": [int(index) for index in dropped],\n            "margin_units": margin,\n            "worst_index": worst,\n            "rotations": kept,\n            "fit": fit_report,\n            **estimate,\n        }\n        trials.append({key: trial[key] for key in ("dropped", "margin_units", "rotations", "ratio")})\n        print(\n            f"calibration drop {list(dropped) or \'none\'}: margin {margin} "\n            f"rotations {kept} ratio {estimate[\'ratio\']:.6f}",\n            flush=True,\n        )\n        if margin <= 0:\n            if dropped == (order[0],) and not expanded:\n                expanded = True\n                candidates.extend((index,) for index in order[1:])\n            continue\n        if best is None or estimate["estimated_expected_cost"] < best["estimate"]["estimated_expected_cost"]:\n            best = {"deltas": deltas, "dropped": set(dropped), "trial": trial, "estimate": estimate}\n    if best is None:\n        raise RuntimeError(f"No calibration kept a positive margin: {trials}")\n    lines = compiled.text.splitlines()\n    rewritten = _rewrite(\n        lines,\n        rotations,\n        {index: int(best["deltas"][index]) for index in range(len(rotations))},\n        best["dropped"],\n    )\n    peepholed, cancelled = optimize(rewritten)\n    return {\n        "text": peepholed,\n        "margin_units": best["trial"]["margin_units"],\n        "worst_x": int(best["trial"]["worst_index"]) + 1,\n        "rotations": best["trial"]["rotations"],\n        "dropped_rotation_indexes": best["trial"]["dropped"],\n        "peephole_cancelled": cancelled,\n        "estimate": best["estimate"],\n        "trials": trials,\n    }\n', 'tasks/invsqrt/peephole.py': '"""Cancel adjacent commuting involutions.\n\nThis file is a port of the method below. It is not independent authorship.\n\nMethod parent (Apache-2.0): the peephole used after rotation calibration in\nKozykorpesh, Quantum AI complete-domain affine compilation v2. Diagonal gates\nand classical conditions are barriers. A match must be the same text within\nthe previous 256 live lines and must not cross a read/write conflict.\n"""\n\nfrom __future__ import annotations\n\nimport collections\nimport re\n\n_QUBIT = re.compile(r"(?<!\\w)q(\\d+)")\n_PHASE = {"Z", "CZ", "CCZ", "Z_POW"}\n_GATES = {"X", "CX", "CCX", "SWAP", "CSWAP"}\n\n\ndef _touch(line: str):\n    """Return (read, write) sets, or None when the line is a barrier."""\n    parts = line.split()\n    if not parts or "if" in parts:\n        return None\n    op = parts[0]\n    if op not in _PHASE | _GATES:\n        return None\n    qubits = {int(match) for match in _QUBIT.findall(line)}\n    if op in _PHASE:\n        return qubits, set()\n    if op == "X":\n        return set(), qubits\n    if op in {"CX", "CCX"}:\n        ordered = [int(match) for match in _QUBIT.findall(line)]\n        return set(ordered[:-1]), {ordered[-1]}\n    if op == "SWAP":\n        return qubits, qubits\n    ordered = [int(match) for match in _QUBIT.findall(line)]\n    return qubits, set(ordered[1:])\n\n\ndef optimize(text: str) -> tuple[str, dict]:\n    """Return peephole text and a count of cancelled gate pairs."""\n    kept: list[str | None] = []\n    counts: collections.Counter = collections.Counter()\n    for line in text.splitlines():\n        if not line:\n            continue\n        described = _touch(line)\n        op = line.split()[0]\n        cancel = False\n        if described is not None and op in _GATES:\n            reads, writes = described\n            for index in range(len(kept) - 1, max(-1, len(kept) - 257), -1):\n                if kept[index] is None:\n                    continue\n                previous = kept[index]\n                if previous == line:\n                    kept[index] = None\n                    counts[op] += 2\n                    cancel = True\n                    break\n                earlier = _touch(previous)\n                if earlier is None:\n                    break\n                earlier_reads, earlier_writes = earlier\n                if writes & earlier_reads or earlier_writes & reads:\n                    break\n        if not cancel:\n            kept.append(line)\n    body = "\\n".join(line for line in kept if line is not None)\n    return body + "\\n", dict(counts)\n', 'tasks/invsqrt/public_port/__init__.py': '# SPDX-License-Identifier: Apache-2.0\n"""Gluzdov v7 source route with locally regenerated coefficients."""\n', 'tasks/invsqrt/public_port/arithmetic.py': '# SPDX-License-Identifier: Apache-2.0\n"""Port of Dmitrii Gluzdov\'s Apache-2.0 notebook v7, scriptVersionId 361726742.\n\nSource SHA-256 d689cc89f41a6c9130e9f4c3a6b08dfd5d758fcbbb79f527d55cbd3b48e15280.\nhttps://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361726742\nCredits: Sumit Chongder (centered polynomial), Rudra Bhaskar (split),\nGu/Kemeny/Yelin/Lukin (deferred repairs), Zongao Bian (virtual low bit),\nand the authors named in tasks/invsqrt/PROVENANCE.md.\nLocal changes: package imports; reuse existing arithmetic helpers; remove\nnotebook setup, fitting drivers, saved-data replay and verifier calls.\n"""\n# Parent code cell 23.\nimport numpy as np\nfrom tasks.invsqrt import signed_digit as H\n\n\ndef prepare(cfg, tabs):\n    d = np.arange(1 << H.DB, dtype=np.int64)\n    Rmax = 0\n    spread = 0\n    minimum = {}\n    maxima = {}\n    for key, (k2, k1, k0) in tabs.items():\n        r, _ = H.pipeline(cfg, k2, k1, d)\n        minimum[key] = int(r.min())\n        maxima[key] = int(r.max())\n        Rmax = max(Rmax, int(r.max()), k1)\n        spread = max(spread, int(r.max() - r.min()))\n    for block in range(1, Rmax.bit_length()):\n        bases = {k: v >> block for k, v in minimum.items()}\n        if all((0 <= tabs[k][1] - (v << block) < 1 << block + 1 and maxima[k] - (v << block) < 1 << block + 1 for k, v in bases.items())):\n            break\n    return {\'block\': block, \'rw\': block + 1, \'cw\': Rmax.bit_length() - block, \'coarse\': bases, \'residual\': {k: tabs[k][1] - (v << block) for k, v in bases.items()}}\n\ndef shift(b, flags, word, L, inverse, prune=True):\n    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]\n    occ = []\n    n = L\n    for k, d in stages:\n        occ.append(min(n, L + 2) if prune and k == 4 else n)\n        n = min(n + d, len(word))\n    seq = list(enumerate(stages))\n    seq = seq[::-1] if inverse else seq\n    for j, (k, d) in seq:\n        indices = range(occ[j]) if inverse else reversed(range(occ[j]))\n        for i in indices:\n            if i + d < len(word):\n                zero_boundary = i + d >= L + sum((amount for _, amount in stages[:j]))\n                if zero_boundary:\n                    if inverse:\n                        b.cx(word[i + d], word[i])\n                        b.del_and(flags[k], word[i], word[i + d])\n                    else:\n                        b.init_and(flags[k], word[i], word[i + d])\n                        b.cx(word[i + d], word[i])\n                else:\n                    b.cswap(flags[k], word[i], word[i + d])\n', 'tasks/invsqrt/public_port/calibration.py': '# SPDX-License-Identifier: Apache-2.0\n"""GPU constraint evaluation for the repository\'s rotation-angle fitter.\n\nThe small cutting-plane linear program uses SciPy/HiGHS on the host. All\n3,133,452-row constraint evaluations and violated-row selection use CUDA\nfloat64. Integer verification of the rounded result remains independent.\nNo saved angle or verifier receipt is an input.\n"""\nimport numpy as np\nfrom scipy.optimize import linprog\nfrom tasks.invsqrt.calibrate import (\n    MAX_X, reference_target_tolerance, _angle_key, _exact_margin,\n    _estimate, _rewrite,\n)\nfrom tasks.invsqrt.peephole import optimize\n\ndef _fit(error, tolerance, bits, free, fixed, bound_units, scale=2.0**28):\n    import torch\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA required for rotation fitting\')\n    residual = torch.as_tensor((error.astype(np.float64)+fixed)/scale, device=\'cuda\')\n    limit = torch.as_tensor(tolerance.astype(np.float64)/scale, device=\'cuda\')\n    matrix_all = torch.as_tensor(bits[:, free].astype(np.float64), device=\'cuda\')\n    width = len(free)\n    active = set(np.linspace(0, MAX_X-1, 4096, dtype=np.int64).tolist())\n    active.update(torch.topk(limit-residual.abs(), 4096, largest=False).indices.cpu().tolist())\n    converged = False\n    for iteration in range(1, 61):\n        chosen = sorted(active)\n        block = matrix_all[chosen].cpu().numpy()\n        matrix = np.vstack([np.c_[block, np.ones(len(chosen))], np.c_[-block, np.ones(len(chosen))]])\n        rhs = torch.cat([limit[chosen]-residual[chosen], limit[chosen]+residual[chosen]]).cpu().numpy()\n        result = linprog(np.r_[np.zeros(width), -1.], A_ub=matrix, b_ub=rhs,\n                         bounds=[(-bound_units/scale, bound_units/scale)]*width+[(None, None)], method=\'highs\')\n        if not result.success:\n            raise RuntimeError(result.message)\n        delta, margin = result.x[:width], float(result.x[width])\n        correction = matrix_all @ torch.as_tensor(delta, device=\'cuda\')\n        slack = limit-(residual+correction).abs()\n        actual = float(slack.min())\n        print(f\'GPU calibration LP {iteration}: rows {len(chosen)} lp {margin*scale:.3f} exhaustive {actual*scale:.3f}\', flush=True)\n        if actual >= margin-1e-9*max(1., abs(margin)):\n            converged = True\n            break\n        added = set(torch.topk(slack, 400, largest=False).indices.cpu().tolist())\n        if added.issubset(active):\n            break\n        active.update(added)\n    return np.rint(delta*scale).astype(np.int64), dict(iterations=iteration, lp_converged=converged,\n                                                     active_inputs=len(active), device=\'cuda\')\n\ndef calibrate(compiled, resources: dict) -> dict:\n    """Try the original angles and each single counted drop. Keep the cheapest legal one."""\n    rotations = compiled.rotations\n    counted = [index for index, rotation in enumerate(rotations) if rotation[\'counted\']]\n    if not counted:\n        raise RuntimeError(\'The compiled circuit has no counted rotation\')\n    bits = np.column_stack([rotation[\'bits\'] for rotation in rotations])\n    xs = np.arange(1, MAX_X + 1, dtype=np.int64)\n    _target, tolerance = reference_target_tolerance(xs)\n    target = _target\n    error = (compiled.phase - target).view(np.int64)\n    order = sorted(counted, key=lambda index: _angle_key(rotations[index][\'theta\']))\n    candidates = [(), (order[0],)]\n    best = None\n    trials = []\n    for dropped in candidates:\n        fixed = np.zeros(MAX_X, dtype=np.float64)\n        bound = float(1 << 36)\n        for index in dropped:\n            units = rotations[index][\'units\']\n            fixed += bits[:, index] * float(-units)\n            bound = max(bound, 4.0 * abs(units))\n        free = [index for index in counted if index not in dropped]\n        rounded, fit_report = _fit(error, tolerance, bits, free, fixed, bound)\n        deltas = np.zeros(len(rotations), dtype=np.int64)\n        for column, index in enumerate(free):\n            deltas[index] = rounded[column]\n        for index in dropped:\n            deltas[index] = -int(rotations[index][\'units\'])\n        margin, worst = _exact_margin(error, tolerance, bits, deltas)\n        kept = len(counted) - len(dropped)\n        estimate = _estimate(resources[\'qubits\'], resources[\'expected_toffoli\'], resources[\'t_like\'], kept, margin)\n        trial = {\'dropped\': [int(index) for index in dropped], \'margin_units\': margin, \'worst_index\': worst, \'rotations\': kept, \'fit\': fit_report, **estimate}\n        trials.append({key: trial[key] for key in (\'dropped\', \'margin_units\', \'rotations\', \'ratio\')})\n        print(f\'calibration drop {list(dropped) or \'none\'}: margin {margin} rotations {kept} ratio {estimate[\'ratio\']:.6f}\', flush=True)\n        if margin <= 0:\n            continue\n        if best is None or estimate[\'estimated_expected_cost\'] < best[\'estimate\'][\'estimated_expected_cost\']:\n            best = {\'deltas\': deltas, \'dropped\': set(dropped), \'trial\': trial, \'estimate\': estimate}\n    if best is None:\n        raise RuntimeError(f\'No calibration kept a positive margin: {trials}\')\n    lines = compiled.text.splitlines()\n    rewritten = _rewrite(lines, rotations, {index: int(best[\'deltas\'][index]) for index in range(len(rotations))}, best[\'dropped\'])\n    peepholed, cancelled = optimize(rewritten)\n    return {\'text\': peepholed, \'margin_units\': best[\'trial\'][\'margin_units\'], \'worst_x\': int(best[\'trial\'][\'worst_index\']) + 1, \'rotations\': best[\'trial\'][\'rotations\'], \'dropped_rotation_indexes\': best[\'trial\'][\'dropped\'], \'peephole_cancelled\': cancelled, \'estimate\': best[\'estimate\'], \'trials\': trials}\n', 'tasks/invsqrt/public_port/deferred.py': '# SPDX-License-Identifier: Apache-2.0\n"""Port of Dmitrii Gluzdov\'s Apache-2.0 notebook v7, scriptVersionId 361726742.\n\nSource SHA-256 d689cc89f41a6c9130e9f4c3a6b08dfd5d758fcbbb79f527d55cbd3b48e15280.\nhttps://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361726742\nCredits: Sumit Chongder (centered polynomial), Rudra Bhaskar (split),\nGu/Kemeny/Yelin/Lukin (deferred repairs), Zongao Bian (virtual low bit),\nand the authors named in tasks/invsqrt/PROVENANCE.md.\nLocal changes: package imports; reuse existing arithmetic helpers; remove\nnotebook setup, fitting drivers, saved-data replay and verifier calls.\n"""\n# Parent code cell 20.\nimport kickmix as km\n\n\ndef demolish(b, register):\n    bits = b.alloc_bits(len(register))\n    for q, m in zip(register, bits):\n        b.hmr(q, m)\n    return list(bits)\n\ndef repair_anf(b, address, table, measurements, low_bits=4):\n    """Möbius-transform the truth table and share monomials of each half."""\n    address = list(address)\n    bits = list(measurements)\n    coef = list(table)\n    for i in range(len(address)):\n        for mask in range(len(coef)):\n            if mask & 1 << i:\n                coef[mask] ^= coef[mask ^ 1 << i]\n\n    def monomials(register):\n        nodes = {1 << i: q for i, q in enumerate(register)}\n        undo = []\n        for mask in range(1, 1 << len(register)):\n            if mask in nodes:\n                continue\n            bit = mask & -mask\n            left = nodes[mask ^ bit]\n            right = nodes[bit]\n            target = b.alloc_qubits(1)[0]\n            b.init_and(left, right, target)\n            nodes[mask] = target\n            undo.append((left, right, target))\n        return (nodes, undo)\n    lo, lundo = monomials(address[:low_bits])\n    hi, hundo = monomials(address[low_bits:])\n    one = b.alloc_qubits(1)[0]\n    b.x(one)\n    for mask, word in enumerate(coef):\n        l = mask & (1 << low_bits) - 1\n        h = mask >> low_bits\n        while word:\n            pos = (word & -word).bit_length() - 1\n            b.push_condition(bits[pos])\n            if l and h:\n                b.cz(lo[l], hi[h])\n            else:\n                b.z(lo[l] if l else hi[h] if h else one)\n            b.pop_condition()\n            word &= word - 1\n    b.x(one)\n    b.free(one)\n    for left, right, target in reversed(hundo + lundo):\n        b.del_and(left, right, target)\n        b.free(target)\n    for bit in bits:\n        b.free(bit)\n\n', 'tasks/invsqrt/public_port/fit.py': '# SPDX-License-Identifier: Apache-2.0\n"""GPU enumeration of Gluzdov v7\'s published interpolated-curvature recipe.\n\nParent cell 22 enumerates base/slope and the neighboring linear coefficient.\nCell 34 requires an even anchor for the virtual bit. We enforce that during\nthe same search. All residuals are evaluated on CUDA in int64/float64.\nOnly the small deterministic choice of groups runs on the host.\n"""\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\n\nfrom .model import Config, certificate, interpolation_values, pipeline, rd\n\n\ndef fit_group(e, group, group_bits=3, fraction_bits=3):\n    import torch\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required; CPU-only fitting is not authorized")\n    cfg = Config()\n    size = 1 << group_bits\n    mid = 1 + (group + .5) * size / 128\n    scale = 2 ** ((e-1)/2)\n    bc = round(.375 * scale * mid**-2.5 * 2**cfg.f2)\n    sc = round(1.25 * .375 * scale * mid**-3.5 * 2**cfg.f2 / 128 * 2**fraction_bits)\n    candidates, needed = [], set()\n    for base in range(bc-4, bc+5):\n        for step in range(max(0, sc-3), sc+4):\n            try:\n                values = interpolation_values(base, step, fraction_bits=fraction_bits, group_bits=group_bits)\n            except ValueError:\n                continue\n            candidates.append((base, step, values))\n            needed.update(enumerate(values))\n    cache = {}\n    for low in range(size):\n        a = group*size + low\n        d = torch.arange(16384, device="cuda", dtype=torch.int64)\n        xn = (1 << 21) + (a << 14) + d\n        d = d[d % 2 == 0] if e else d[(xn <= 3133452) | (d % 4 == 0)]\n        target = scale / torch.sqrt(1 + (a*16384+d).double() * 2**-21) * 2**cfg.fy\n        midpoint = 1 + (a*16384+8191.5) * 2**-21\n        k1c = round(.5 * scale * midpoint**-1.5 * 2**cfg.f1)\n        # ±9 includes cell 22\'s ±5 followed by cell 34\'s ±4 refinement.\n        k1 = torch.arange(k1c-9, k1c+10, device="cuda", dtype=torch.int64)[:, None]\n        for k2 in sorted(k for i, k in needed if i == low):\n            r, q = pipeline(cfg, k2, k1, d[None, :])\n            values = target + q\n            center = (values.amin(1) + values.amax(1))/2 + 1.5\n            anchor = 2 * torch.floor(center/2).to(torch.int64)\n            # Deterministic order: k1, then even k0 ascending, as in the source.\n            scores, anchors = [], []\n            for offset in range(-10, 13, 2):\n                k0 = anchor + offset\n                y = ((k0[:, None] - q) >> cfg.gy) << cfg.gy\n                score = ((y-target).abs()/target).amax(1)*2**24\n                score[(y.amax(1) >= 1 << cfg.fy) | (r.amin(1) < 0)] = float(\'inf\')\n                scores.append(score)\n                anchors.append(k0)\n            score_matrix = torch.stack(scores, dim=1)\n            index = int(score_matrix.argmin())\n            row, col = divmod(index, len(scores))\n            cache[low, k2] = (float(score_matrix[row, col]), k2, int(k1[row, 0]), int(anchors[col][row]))\n    choices = []\n    for base, step, values in candidates:\n        rows = [cache[low, k2] for low, k2 in enumerate(values)]\n        choices.append((max(row[0] for row in rows), base, step, rows))\n    score, base, step, rows = min(choices, key=lambda item: item[0])\n    if not math.isfinite(score):\n        raise RuntimeError(f"No legal fit for parity {e}, group {group}")\n    return dict(score=score, base=base, step=step, rows=[row[1:] for row in rows])\n\n\ndef fit_tables(group_bits=3, fraction_bits=3, threshold=.95):\n    """Refine only groups that miss the stated error bound; never fit to a salt."""\n    size = 1 << group_bits\n    tables, encoding, scores, groups = {}, {}, [], []\n\n    def visit(e, group, bits):\n        fit = fit_group(e, group, bits, fraction_bits)\n        print(json.dumps(dict(parity=e, group=group, group_bits=bits, score=fit[\'score\'])), flush=True)\n        if fit[\'score\'] > threshold and bits:\n            visit(e, group*2, bits-1)\n            visit(e, group*2+1, bits-1)\n            return\n        scores.append(fit[\'score\'])\n        groups.append(dict(parity=e, group=group, group_bits=bits, score=fit[\'score\']))\n        for j, row in enumerate(fit[\'rows\']):\n            address = group*(1 << bits)+j\n            # Embed a smaller group\'s interpolation in the same global adder\n            # schedule. The upper address bits are constant inside this group.\n            base = fit[\'base\'] + sum((2*((address >> i)&1)-1)*rd(fit[\'step\'], fraction_bits-i)\n                                     for i in range(bits, group_bits))\n            tables[e, address] = row\n            encoding[e, address] = (base, fit[\'step\'])\n            expected = base - sum((2*((address >> i)&1)-1)*rd(fit[\'step\'], fraction_bits-i)\n                                  for i in range(group_bits))\n            if expected != row[0] or not 0 <= base < 256:\n                raise RuntimeError(\'Refined curvature encoding is invalid\')\n\n    for e in range(2):\n        for group in range(128//size):\n            visit(e, group, group_bits)\n    encoded = [encoding[e,a] for e in range(2) for a in range(128)]\n    return dict(params=[9, 19, 26, 2, [5]], rows=[tables[e,a] for e in range(2) for a in range(128)],\n                interpolation=dict(fraction_bits=fraction_bits, remainder_bits=group_bits,\n                                   step_width=max(1, max(s.bit_length() for _,s in encoded)), rows=encoded),\n                normalized_worst=max(scores), certificate=certificate(Config(), tables),\n                device=\'cuda\', refinement_threshold=threshold, groups=groups)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--group-bits\', type=int, choices=range(4), default=3)\n    parser.add_argument(\'--fraction-bits\', type=int, choices=(2, 3, 4), default=3)\n    parser.add_argument(\'--threshold\', type=float, default=.95)\n    args = parser.parse_args()\n    result = fit_tables(args.group_bits, args.fraction_bits, args.threshold)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(result, indent=2)+\'\\n\')\n    print(json.dumps(result[\'certificate\']), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'tasks/invsqrt/public_port/generate.py': '# SPDX-License-Identifier: Apache-2.0\n"""Regenerate the public-source invsqrt experiment; the fallback is unchanged.\n\nUsage: uv run --group gpu python -m tasks.invsqrt.public_port.generate --out DIR\nNo cached table, circuit, angle file, or rewrite file is an input.\n"""\nimport argparse\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\n\nfrom .model import Config, certificate\n\n\ndef resources(circuit):\n    stats = circuit.operation_statistics()\n    depth = 0\n    discount = 0.\n    for line in str(circuit).splitlines():\n        opcode = line.split(\' \', 1)[0]\n        if opcode == \'PUSH_CONDITION\':\n            depth += 1\n        elif opcode == \'POP_CONDITION\':\n            depth -= 1\n        elif opcode in (\'CCX\', \'CSWAP\') and depth:\n            discount += 1 - 2**-depth\n    return dict(qubits=int(circuit.num_qubits), static_toffoli=int(stats.num_toffoli_like),\n                expected_toffoli=int(stats.num_toffoli_like)-discount,\n                t_like=int(stats.num_t_like), rotations=int(stats.num_z_pow(min_clifford_hierarchy_level=4)))\n\n\ndef build_from_fit(fitted, storage_mode=None):\n    if storage_mode not in (None, \'affine\', \'future\', \'product\'):\n        raise ValueError(f\'Unknown storage mode: {storage_mode}\')\n    cfg = Config()\n    tables = {(e,a): fitted[\'rows\'][128*e+a] for e in range(2) for a in range(128)}\n    # Recompute instead of trusting a passed flag supplied by a caller.\n    proof = certificate(cfg, tables)\n    if not proof[\'passed\']:\n        raise ValueError(f\'G2 integer mirror failed before circuit construction: {proof}\')\n    print(json.dumps(dict(stage=\'integer_mirror\', **proof)), flush=True)\n\n    import kickmix as km\n    from tasks.invsqrt import affine, calibrate\n    from . import schedule, calibration\n\n    raw, split = schedule.build(cfg, tables, group_bits=None, interpolation=fitted[\'interpolation\'],\n                                main_lookup=\'anf\', affine=True, joint_norm=False)\n    print(json.dumps(dict(stage=\'raw\', **resources(raw))), flush=True)\n    events = []\n    if storage_mode is None:\n        compiled = affine.compile_circuit(raw)\n    else:\n        from .storage import compile_circuit\n        compiled, events = compile_circuit(raw, cap=68, products=storage_mode == \'product\',\n                                          future=storage_mode in (\'future\', \'product\'))\n    phase = calibrate.assert_phase_matches(compiled, cfg, tables)\n    before = resources(km.Circuit(compiled.text))\n    print(json.dumps(dict(stage=\'compiled\', phase=phase, **before)), flush=True)\n    calibrated = calibration.calibrate(compiled, before)\n    circuit = km.Circuit(calibrated[\'text\'])\n    final = resources(circuit)\n    estimate = calibrate._estimate(final[\'qubits\'], final[\'expected_toffoli\'], final[\'t_like\'],\n                                   final[\'rotations\'], calibrated[\'margin_units\'])\n    buffer = io.BytesIO()\n    circuit.to_file(buffer, format=\'kmb\')\n    data = buffer.getvalue()\n    report = dict(integer_mirror=proof, phase=phase, split=split, raw=resources(raw),\n                  compiled=before, rewrites=dict(compiled.rewrites), final=final, estimate=estimate,\n                  calibration={k:v for k,v in calibrated.items() if k != \'text\'},\n                  sha256=hashlib.sha256(data).hexdigest(), bytes=len(data),\n                  parent_version=361726742, tables=fitted, full_verification_required=True)\n    if storage_mode is not None:\n        report[\'storage\'] = dict(mode=storage_mode, target_cap=68, events=events)\n    return data, report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--out\', type=Path, required=True)\n    parser.add_argument(\'--circuit-only\', action=\'store_true\',\n                        help=\'Write only the circuit for exact-file G5 comparison; still refit and prove it.\')\n    parser.add_argument(\'--storage\', choices=(\'affine\', \'future\', \'product\'),\n                        help=\'Experimental idle-wire parking; the default floor candidate is unchanged.\')\n    args = parser.parse_args()\n    from .fit import fit_tables\n    fitted = fit_tables()\n    data, report = (build_from_fit(fitted) if args.storage is None\n                    else build_from_fit(fitted, storage_mode=args.storage))\n    args.out.mkdir(parents=True, exist_ok=True)\n    (args.out/\'invsqrt.kmb\').write_bytes(data)\n    if not args.circuit_only:\n        (args.out/\'build-report.json\').write_text(json.dumps(report, indent=2)+\'\\n\')\n    print(json.dumps({k:v for k,v in report.items() if k not in (\'tables\',\'calibration\',\'storage\')}), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'tasks/invsqrt/public_port/model.py': '# SPDX-License-Identifier: Apache-2.0\n"""Integer arithmetic for the Gluzdov v7 virtual-low schedule.\n\nDerived from Sumit Chongder\'s centered signed-digit polynomial as retained\nin Gluzdov v7 (scriptVersionId 361726742), cells 22, 29, 34 and 40.\nNo saved coefficient table, angle file, patch or binary is loaded.\nThis module deliberately has no kickmix dependency: G2 runs before building.\n"""\nfrom dataclasses import dataclass\nimport math\n\nimport numpy as np\n\nMAX_X = 3_133_452\n\n\n@dataclass(frozen=True)\nclass Config:\n    f2: int = 9\n    f1: int = 19\n    fyr: int = 26\n    gy: int = 2\n    drop2: tuple[int, ...] = (5,)\n\n    @property\n    def fy(self):\n        return self.fyr + self.gy\n\n    @property\n    def w2(self):\n        return self.f2 - 1\n\n    @property\n    def sh2(self):\n        return [self.f2 + 22 - i - self.f1 for i in range(14)]\n\n    @property\n    def sh1(self):\n        return [self.f1 + 22 - i - self.fy for i in range(14)]\n\n    def rrow_list(self):\n        return [i for i in reversed(range(14)) if i not in self.drop2 and self.sh2[i] < self.w2]\n\n\ndef rd(value, shift):\n    if shift < 0:\n        return value << -shift\n    return (value >> shift) + (((value >> (shift - 1)) & 1) if shift else 0)\n\n\ndef pipeline(cfg, k2, k1, d):\n    """Broadcast over numpy or torch integer arrays, with identical rounding."""\n    signs = [2 * ((d >> i) & 1) - 1 for i in range(14)]\n    r = d * 0 + k1\n    for i in cfg.rrow_list():\n        r = r - signs[i] * rd(k2, cfg.sh2[i])\n    q = r * 0\n    for i in reversed(range(14)):\n        q = q + signs[i] * rd(r, cfg.sh1[i])\n    return r, q\n\n\ndef interpolation_values(base, step, *, fraction_bits=3, group_bits=3):\n    values = tuple(base - sum((2 * ((low >> i) & 1) - 1) * rd(step, fraction_bits-i)\n                             for i in range(group_bits)) for low in range(1 << group_bits))\n    if min(values) < 0 or max(values) >= 256:\n        raise ValueError("curvature outside the eight-bit register")\n    return values\n\n\ndef integer_model(xs, cfg, tables):\n    xs = np.asarray(xs, dtype=np.int64)\n    if np.any((xs < 1) | (xs > MAX_X)):\n        raise ValueError("x outside the promised input domain")\n    shifts = 22 - np.frexp(xs)[1].astype(np.int64)\n    normalized = xs << shifts\n    e = shifts & 1\n    address = (normalized >> 14) & 127\n    rows = np.array([tables[p, a] for p in range(2) for a in range(128)], dtype=np.int64)\n    coefficients = rows[e * 128 + address]\n    _, q = pipeline(cfg, coefficients[:, 0], coefficients[:, 1], normalized & 16383)\n    y = ((coefficients[:, 2] - q) >> cfg.gy) << cfg.gy\n    return y << (shifts >> 1)\n\n\ndef certificate(cfg, tables):\n    """Check every promised integer against the official integer root formula."""\n    worst_ratio, worst_x, minimum, bad, count = 0.0, None, 1 << 64, None, 0\n    for start in range(1, MAX_X + 1, 131072):\n        xs = np.arange(start, min(start + 131072, MAX_X + 1), dtype=np.int64)\n        phase = integer_model(xs, cfg, tables).astype(np.uint64) << np.uint64(64-cfg.fy-10)\n        roots = [math.isqrt((1 << 130) // int(x)) for x in xs]\n        target = np.array([((r+1) >> 1) & ((1 << 64)-1) for r in roots], dtype=np.uint64)\n        tol = np.array([min(r >> 25, 1 << 63) for r in roots], dtype=np.uint64)\n        error = np.abs((phase-target).view(np.int64)).astype(np.uint64)\n        ratio = error.astype(np.float64) / tol\n        i = int(ratio.argmax())\n        if ratio[i] > worst_ratio:\n            worst_ratio, worst_x = float(ratio[i]), int(xs[i])\n        # Python integers avoid overflow at the exact x=1 tolerance boundary.\n        margin = min(int(t)-int(v) for t, v in zip(tol, error))\n        minimum = min(minimum, margin)\n        failures = np.flatnonzero(error > tol)\n        if len(failures) and bad is None:\n            bad = int(xs[failures[0]])\n        count += len(xs)\n    return dict(passed=bad is None, tested_inputs=count, worst_ratio=worst_ratio,\n                worst_x=worst_x, min_margin_units=minimum, first_bad_x=bad)\n', 'tasks/invsqrt/public_port/schedule.py': '# SPDX-License-Identifier: Apache-2.0\n"""Port of Dmitrii Gluzdov\'s Apache-2.0 notebook v7, scriptVersionId 361726742.\n\nSource SHA-256 d689cc89f41a6c9130e9f4c3a6b08dfd5d758fcbbb79f527d55cbd3b48e15280.\nhttps://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361726742\nCredits: Sumit Chongder (centered polynomial), Rudra Bhaskar (split),\nGu/Kemeny/Yelin/Lukin (deferred repairs), Zongao Bian (virtual low bit),\nand the authors named in tasks/invsqrt/PROVENANCE.md.\nLocal changes: package imports; reuse existing arithmetic helpers; remove\nnotebook setup, fitting drivers, saved-data replay and verifier calls.\n"""\n# Parent code cell 42.\nimport kickmix as km\nfrom fractions import Fraction\nfrom tasks.invsqrt import signed_digit as H\nfrom tasks.invsqrt import signed_digit as A\nfrom . import arithmetic as O, deferred as D, virtual_low\n\n\ndef joint_top_normalization_repair(b, x, m8, m16):\n    """On positive22-bit inputs, f16=g16 and f8=g8 XOR g16,\n where gk tests whether the highest k input bits are zero.\n Thus the joint phase is m8*g8 XOR (m8 XOR m16)*g16.\n A shared16-bit prefix ladder, or an8-bit phase, costs8.5 mean\n Toffolis instead of10 for the two separate repairs.\n """\n    parity = m16\n    b.cx(m8, parity)\n    b.push_condition(parity)\n    top = list(reversed(x[6:22]))\n    b.x(top)\n    anc = b.alloc_qubits(14)\n    b.init_and(top[0], top[1], anc[0])\n    for j in range(1, 14):\n        b.init_and(anc[j - 1], top[j + 1], anc[j])\n    b.push_condition(m8)\n    b.z(anc[6])\n    b.pop_condition()\n    b.cz(anc[13], top[15])\n    for j in reversed(range(1, 14)):\n        b.del_and(anc[j - 1], top[j + 1], anc[j])\n    b.del_and(top[0], top[1], anc[0])\n    b.free(anc)\n    b.x(top)\n    b.pop_condition()\n    b.x(parity)\n    b.push_condition(parity)\n    b.push_condition(m8)\n    top = x[14:22]\n    b.x(top)\n    H.phase_all(b, list(top))\n    b.x(top)\n    b.pop_condition()\n    b.pop_condition()\n\ndef anf_lookup(b, address, target, table):\n    """Compute an XOR lookup with depth-first ANF monomials, at most\n 2**n-n-1 temporary ANDs and n-1 ladder wires. New implementation.\n """\n    address = list(address)\n    coef = list(table)\n    n = len(address)\n    for i in range(n):\n        for mask in range(1 << n):\n            if mask >> i & 1:\n                coef[mask] ^= coef[mask ^ 1 << i]\n    active = {}\n\n    def needed(mask, nextbit):\n        children = [(mask | 1 << j, j + 1, j) for j in range(nextbit, n)]\n        active[mask] = bool(coef[mask])\n        for child, nxt, j in children:\n            active[mask] |= needed(child, nxt)\n        return active[mask]\n    for i in range(n):\n        needed(1 << i, i + 1)\n\n    def xor(value, control=None):\n        for i, q in enumerate(target):\n            if value >> i & 1:\n                if control is None:\n                    b.x(q)\n                else:\n                    b.cx(control, q)\n    xor(coef[0])\n\n    def visit(mask, control, nextbit):\n        xor(coef[mask], control)\n        for j in range(nextbit, n):\n            child = mask | 1 << j\n            if not active[child]:\n                continue\n            q = b.alloc_qubits(1)[0]\n            b.init_and(control, address[j], q)\n            visit(child, q, j + 1)\n            b.del_and(control, address[j], q)\n            b.free(q)\n    for i in range(n):\n        if active[1 << i]:\n            visit(1 << i, address[i], i + 1)\n\ndef adaptive_lookup(b, address, target, table, affine=False):\n    """Unary decoder pruned wherever every remaining output word is equal.\n\n A full M-leaf binary tree costs M-2 temporary ANDs; address wires and\n one ladder wire per active depth suffice. Demolition is standard MBU.\n """\n    address = list(address)\n\n    def xor_word(value, control):\n        for i, q in enumerate(target):\n            if value >> i & 1:\n                if control is None:\n                    b.x(q)\n                else:\n                    b.cx(control, q)\n\n    def walk(bits, rows, parent):\n        if len(set(rows)) == 1:\n            xor_word(rows[0], parent)\n            return\n        assert bits\n        if affine:\n            base = rows[0]\n            columns = [rows[1 << i] ^ base for i in range(len(bits))]\n            test = [base]\n            for col in columns:\n                test += [v ^ col for v in test]\n            if rows == test:\n                xor_word(base, parent)\n                for bit, col in zip(bits, columns):\n                    if not col:\n                        continue\n                    if parent is None:\n                        xor_word(col, bit)\n                    else:\n                        q = b.alloc_qubits(1)[0]\n                        b.init_and(parent, bit, q)\n                        xor_word(col, q)\n                        b.del_and(parent, bit, q)\n                        b.free(q)\n                return\n        bit = bits[-1]\n        half = len(rows) // 2\n        if parent is None:\n            b.x(bit)\n            walk(bits[:-1], rows[:half], bit)\n            b.x(bit)\n            walk(bits[:-1], rows[half:], bit)\n        else:\n            tmp = b.alloc_qubits(1)[0]\n            b.x(bit)\n            b.init_and(parent, bit, tmp)\n            b.x(bit)\n            walk(bits[:-1], rows[:half], tmp)\n            b.cx(parent, tmp)\n            walk(bits[:-1], rows[half:], tmp)\n            b.cx(parent, tmp)\n            b.x(bit)\n            b.del_and(parent, bit, tmp)\n            b.x(bit)\n            b.free(tmp)\n    walk(address, list(table), None)\n\ndef build(c, tabs, group_bits=2, mode=\'mux\', interpolation=None, main_lookup=\'unary\', affine=False, joint_norm=False):\n    lay = H.layout(c, tabs, LB=c.fy - 7, S=c.fy - 8)\n    sp = O.prepare(c, tabs)\n    assert all((v % 2 == 0 for v in lay[\'lo\'].values())), \'virtual low bit requires even anchors\'\n    b = km.CircuitBuilder()\n    x = b.create_quantum_register(22, name=\'x\')\n    flags = b.alloc_qubits(4)\n    f = [x[21]] + list(flags)\n    for k, amount in H.NSTAGES[:-1]:\n        top = x[22 - amount:22]\n        b.x(top)\n        H.and_all_into(b, list(top), f[k])\n        b.x(top)\n        for i in reversed(range(22 - amount)):\n            b.cswap(f[k], x[i], x[i + amount])\n    b.x(x[21])\n    b.left_rotate(x[:21], 1, control=x[21])\n    b.cx(x[21], x[0])\n    res, coarse, low, high = [b.alloc_qubits(n) for n in [sp[\'rw\'], sp[\'cw\'], lay[\'LB\'] - 1, lay[\'whi\']]]\n    full = km.array(list(low) + list(res) + list(high) + list(coarse))\n    rest = km.array(list(low) + list(res) + list(coarse))\n    ft = []\n    rt = []\n    ht = []\n    ct = []\n    for e in range(2):\n        for a in range(128):\n            k = (e, a)\n            common = lay[\'lo\'][k] >> 1 | sp[\'residual\'][k] << lay[\'LB\'] - 1\n            rt.append(common | sp[\'coarse\'][k] << lay[\'LB\'] - 1 + sp[\'rw\'])\n            ft.append(common | lay[\'hi\'][k] << lay[\'LB\'] - 1 + sp[\'rw\'] | sp[\'coarse\'][k] << lay[\'LB\'] - 1 + sp[\'rw\'] + lay[\'whi\'])\n            ht.append(lay[\'hi\'][k])\n        if group_bits is not None:\n            for a in range(0, 128, 1 << group_bits):\n                assert len({tabs[e, j][0] for j in range(a, a + (1 << group_bits))}) == 1\n                ct.append(tabs[e, a][0])\n        else:\n            ct.extend((tabs[e, a][0] for a in range(128)))\n    address = km.array(list(x[14:21]) + [f[0]])\n    ca = km.array(list(x[14 + (group_bits or 0):21]) + [f[0]])\n    delta = x[:14]\n    c2 = None\n    step_masks = []\n\n    def add(a, t, cin=None, sub=False):\n        A.folded_gadd(b, a, t, cin, sub)\n\n    def phase(reg, lowbit):\n        ext = b.alloc_qubits(10)\n        word = km.array(list(reg) + list(ext))\n        O.shift(b, f, word, len(reg), False)\n        for i, q in enumerate(word):\n            ex = lowbit + i - c.fy - 10\n            if ex < 0:\n                b.z_pow(q, Fraction(2) ** (ex + 1))\n        O.shift(b, f, word, len(reg), True)\n        b.free(ext)\n\n    def load():\n        nonlocal c2\n        c2 = b.alloc_qubits(c.w2)\n        if interpolation is not None:\n            sw = interpolation[\'step_width\']\n            step = b.alloc_qubits(sw)\n            uw = interpolation.get(\'update_width\', c.w2)\n            mask = (1 << uw) - 1\n            packed = [base & mask | tabs[a // 128, a % 128][0] & ~mask | s << c.w2 for a, (base, s) in enumerate(interpolation[\'rows\'])]\n            adaptive_lookup(b, address, km.array(list(c2) + list(step)), packed, affine=affine)\n            for i in range(interpolation[\'remainder_bits\']):\n                sh = interpolation[\'fraction_bits\'] - i\n                if sh >= 0:\n                    a = step[sh:]\n                    t = c2[:uw]\n                    cin = step[sh - 1] if sh else None\n                else:\n                    a = step\n                    t = c2[-sh:uw]\n                    cin = None\n                # Arithmetic is modulo 2**uw. Higher addend bits cannot\n                # affect the low suffix; the lookup supplies the final high\n                # curvature bits for this address directly.\n                a = a[:len(t)]\n                b.cx(x[14 + i], t)\n                add(a, t, cin)\n                b.cx(x[14 + i], t)\n            step_masks.extend(D.demolish(b, step))\n            b.free(step)\n        elif group_bits is None:\n            adaptive_lookup(b, ca, c2, ct, affine=affine)\n        else:\n            b.init_lookup(table=ct, address=ca, target=c2)\n\n    def erase():\n        nonlocal c2\n        mm = D.demolish(b, c2)\n        b.free(c2)\n        c2 = None\n        return mm\n\n    def inner(inverse):\n        rows = c.rrow_list()\n        rows = rows[::-1] if inverse else rows\n        for i in rows:\n            sh = c.sh2[i]\n            if sh >= 0:\n                a = c2[sh:]\n                t = res\n                cin = c2[sh - 1] if sh else None\n            else:\n                a = c2\n                t = res[-sh:]\n                cin = None\n            b.cx(delta[i], t)\n            add(a, t, cin, sub=inverse)\n            b.cx(delta[i], t)\n\n    def outer(inverse):\n        derivative = res[:sp[\'block\']] + coarse\n        virtual_low.outer(b, add, derivative, low, delta, c.sh1, inverse)\n    if main_lookup == \'anf\':\n        anf_lookup(b, address, full, ft)\n    else:\n        b.init_lookup(table=ft, address=address, target=full)\n    phase(high, lay[\'S\'])\n    bits = D.demolish(b, high)\n    b.free(high)\n    load()\n    inner(False)\n    cm1 = erase()\n    add(res[sp[\'block\']:], coarse)\n    outer(False)\n    phase(low[c.gy - 1:], c.gy)\n    outer(True)\n    add(res[sp[\'block\']:], coarse, sub=True)\n    load()\n    inner(True)\n    cm2 = erase()\n    bits += D.demolish(b, rest)\n    b.free(rest)\n    bits += cm1 + cm2\n    rtwidth = len(rest)\n    hw = lay[\'whi\']\n    repairtable = [h | r << hw | tabs[e, a][0] << hw + rtwidth | tabs[e, a][0] << hw + rtwidth + c.w2 for e in range(2) for a, (h, r) in enumerate(zip(ht[128 * e:128 * (e + 1)], rt[128 * e:128 * (e + 1)]))]\n    if interpolation is not None:\n        bits += step_masks\n        sw = interpolation[\'step_width\']\n        off = hw + rtwidth + 2 * c.w2\n        for a, (_, step) in enumerate(interpolation[\'rows\']):\n            repairtable[a] |= step << off | step << off + sw\n    D.repair_anf(b, address, repairtable, bits)\n    b.cx(x[21], x[0])\n    b.right_rotate(x[:21], 1, control=x[21])\n    b.x(x[21])\n    pending = None\n    for k, amount in reversed(H.NSTAGES[:-1]):\n        for i in range(22 - amount):\n            b.cswap(f[k], x[i], x[i + amount])\n        top = x[22 - amount:22]\n        m = b.alloc_bits(1)[0]\n        b.hmr(f[k], m)\n        if joint_norm and k == 3:\n            pending = m\n            continue\n        if joint_norm and k == 4:\n            joint_top_normalization_repair(b, x, pending, m)\n            b.free(pending)\n            b.free(m)\n        else:\n            b.push_condition(m)\n            b.x(top)\n            H.phase_all(b, list(top))\n            b.x(top)\n            b.pop_condition()\n            b.free(m)\n    b.free(flags)\n    return (b.finish_circuit(), {k: v for k, v in sp.items() if k not in [\'coarse\', \'residual\']})\n', 'tasks/invsqrt/public_port/virtual_low.py': '# SPDX-License-Identifier: Apache-2.0\n"""Port of Dmitrii Gluzdov\'s Apache-2.0 notebook v7, scriptVersionId 361726742.\n\nSource SHA-256 d689cc89f41a6c9130e9f4c3a6b08dfd5d758fcbbb79f527d55cbd3b48e15280.\nhttps://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361726742\nCredits: Sumit Chongder (centered polynomial), Rudra Bhaskar (split),\nGu/Kemeny/Yelin/Lukin (deferred repairs), Zongao Bian (virtual low bit),\nand the authors named in tasks/invsqrt/PROVENANCE.md.\nLocal changes: package imports; reuse existing arithmetic helpers; remove\nnotebook setup, fitting drivers, saved-data replay and verifier calls.\n"""\n# Parent code cell 40.\n\n\ndef outer(b, add, derivative, low_suffix, delta, shifts, inverse):\n    assert list(reversed(shifts)) == list(range(14))\n    for i in range(14) if inverse else reversed(range(14)):\n        sh = shifts[i]\n        a0 = derivative[sh]\n        cin = derivative[sh - 1] if sh else None\n        a = derivative[sh + 1:]\n        carry = b.alloc_qubits(1)[0]\n        if cin is not None:\n            b.cx(cin, a0)\n        b.init_and(delta[i], a0, carry)\n        if cin is not None:\n            b.cx(cin, carry)\n            b.cx(cin, a0)\n        b.cx(delta[i], low_suffix)\n        add(a, low_suffix, carry, sub=inverse)\n        b.cx(delta[i], low_suffix)\n        if cin is not None:\n            b.cx(cin, a0)\n            b.cx(cin, carry)\n        b.del_and(delta[i], a0, carry)\n        if cin is not None:\n            b.cx(cin, a0)\n        b.free(carry)\n', 'tasks/invsqrt/signed_digit.py': '"""Signed-digit invsqrt circuit body (Apache-2.0 derivative).\n\nParent notebook, v3, source snapshot SHA-256\n79f6373302cb916a88f65a1b142024d4d2d0b14e41eb58bef3152ebd45ffd0ca:\nQuantum Circuits | Signed-Digit Memory, Artem1410 (anazemcev),\nhttps://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory\nParent licence: Apache-2.0. Dataset-adapted calibration rights are unresolved\n(D-PRA220-01); the retained upstream SPDX notices do not settle that scope.\n\nThe selected schedule is quadratic signed-digit Horner evaluation, a 256-entry\nQROM, no Newton iteration, a derivative split, one borrowed normalized carry,\nand the promised-input rewrite. Parameters are the constants under this\ndocstring. `tasks/invsqrt/generate.py` is the entry point. It writes only\n`invsqrt.kmb`. Upstream notices in the comments below are retained.\n"""\n\n# SPDX-License-Identifier: Apache-2.0\n# New derivative-split integration; original upstream notices retained below.\n# Sumit Chongder: centered signed-digit generator and integer pipeline,361357149.\n# Rudra Bhaskar: derivative split, top-carry reuse, static-high checkpoint,361445283.\n# https://www.kaggle.com/code/rudrabhaskar/highest-public-score-direct-polynomial-phasing?scriptVersionId=361445283\n# Alex Chilton: shared prefix and empty-target temporary-AND shifts,361472684.\n# Kirill Tymchishin: exact Rudra/Alex integration used as the95Q source baseline,361491814.\n# https://www.kaggle.com/code/kirilltym/quantum-circuits-qrom-tower42?scriptVersionId=361491814\n# Kozykorpesh: explicit 8,19,27,1 coefficient widths. The integers are regenerated\n# with dataset-adapted calibration; licence unresolved (D-PRA220-01).\n# Our changes: generic exact coefficient widths/left shifts, normalized-input\n# borrowing on this derivative schedule, and the proven original-input rewrite.\n# The derivative split and its precision calibrations are credited upstream.\n\n# SPDX-License-Identifier: Apache-2.0\n# Arithmetic research candidate. Retained upstream attribution:\n# Sumit Chongder: readable centered signed-digit generator and integer pipeline.\n# https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149\n# anazemcev / Artem1410: early high-word phase, folded carry and normalized-input borrowing.\n# https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory?scriptVersionId=361422248\n# Kozykorpesh: exact reachable-domain calibration, Cfg(8,19,27,1).\n# https://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom?scriptVersionId=361456406\n# Coefficient integers are regenerated by tasks/invsqrt/fit_coefficients.py.\n# Provenance: regenerated; dataset-adapted calibration; licence unresolved.\n# Alex Chilton: shared prefix flags and temporary-AND moves into zero destinations.\n# https://www.kaggle.com/code/alexchilton/where-the-toffolis-go-anatomy-a-cheaper-invsqrt?scriptVersionId=361472684\n# Alex v8 has no notebook-wide licence. It is reference-only and is not loaded.\n# The Apache-2.0 parent of this file is anazemcev Signed-Digit Memory v3, which\n# retains the Alex notice. Sumit run 361357149 is v1 (PRA-218).\n# This revision combines credited methods. No upstream gain is claimed as original.\n# Added own exact rewrite: the original input bound proves x[21]&x[20]==0.\n# Linearize the first conjunction in normalization and the final input phase fixup.\n\n# Attribution: Sumit Chongder, Apache-2.0 as published.\n# https://www.kaggle.com/code/sumitchongder/notebook2f196e646b\n# Sumit run 361357149 is v1 (public version metadata, PRA-218). Kirill\'s v9 cites v1.\n# The Sumit sentences above are the parent notebook\'s notice. This tree does\n# not load that notebook. Whether a later v2 exists stays with PRA-205.\n# This isolated lab regenerates the circuit from readable source; no binaries or\n# remote notebook outputs are inputs. Our added exact integer certificate and\n# reporting helpers and memory-schedule adaptations follow the original generator.\n"""Builds an optimized kickmix circuit for the invsqrt benchmark (Benchmark I).\n\n|x>|0> -> exp(2 pi i / sqrt(x)) |x>|0>,   x in [1, 3133452], 22-bit register \'x\'.\n\nDesign (all arithmetic exactly mirrored by the numpy model below):\n  1. Normalize x by single-bit binary stages (16, 8, 4, 2, 1) into M = x_norm / 2^21 in [1, 2);\n     flags f0..f4 hold the shift count s\' in binary.  e = f0 (parity), s = s\' >> 1.\n     1/sqrt(x) = 2^(s-10) * y,  y = 2^((e-1)/2) / sqrt(M)  in (0.5, 1].\n  2. One 256-entry QROM indexed by (7 mantissa bits, e) XORs quadratic coefficients directly into\n     the working registers (no separate coefficient copies).\n  3. Centered signed-digit Horner: delta_c = sum_i (2 d_i - 1) 2^(i-22) over the 14 low bits.\n     R = K1 - C2*delta_c,  y = K0 - R*delta_c.  Each row is an UNcontrolled Gidney addition\n     wrapped in CNOT flips (Y - A = ~(~Y + A)), with a rounding carry-in.\n  4. The y register is split: rows only touch a 21-bit low register L; the constant high part\n     sits in a separate register and the overflow bit is merged in before phasing.\n  5. Binary de-normalization (shift by s), one Z_POW per bit, then exact mirror uncompute.\n     Lookup uncompute is measurement based (del_lookup); normalization flags are cleared by HMR\n     with classically conditioned phase fixups.\nUsage: python signed_digit_lab.py OUT.kmb [--baseline] (requires kickmix and NumPy)\n"""\nimport sys, math, functools, types\nfrom fractions import Fraction\nimport numpy as np\nimport kickmix as km\n\nW = 22; MB = 21; DB = 14; NSEG = 128\nF2, F1, FYR, GY, DROP2 = 8, 19, 27, 1, (5,)     # precision configuration (verified)\n\n# ----------------------------------------------------------------- numpy model / tables\n@functools.lru_cache(None)\ndef fitc(lo, hi, mid, scale, deg=2, iters=80):\n    f = lambda M: scale / np.sqrt(M)\n    d = np.linspace(lo - mid, hi - mid, 3001)\n    A = np.vander(d, deg + 1, increasing=True)\n    w = np.sqrt(mid + d) / scale\n    wt = np.ones_like(d)\n    for _ in range(iters):                       # Lawson iterations -> weighted minimax\n        sw = w * np.sqrt(wt)\n        coef = np.linalg.lstsq(A * sw[:, None], f(mid + d) * sw, rcond=None)[0]\n        err = np.abs((A @ coef - f(mid + d)) * w)\n        wt = wt * err; wt /= wt.sum()\n    return coef\n\nclass Cfg:\n    def __init__(self, f2, f1, fyr, gy, drop2=()):\n        self.f2, self.f1, self.fyr, self.gy = f2, f1, fyr, gy\n        self.fy = fyr + gy\n        self.w2 = f2 - 1\n        self.sh2 = [f2 + 22 - i - f1 for i in range(DB)]\n        self.sh1 = [f1 + 22 - i - self.fy for i in range(DB)]\n        self.drop2 = set(drop2)\n    def rrow_list(self):\n        return [i for i in reversed(range(DB)) if i not in self.drop2 and self.sh2[i] < self.w2]\n\ndef rd(v, s):\n    if s < 0: return v << (-s)\n    r = v >> s\n    if s >= 1: r = r + ((v >> (s - 1)) & 1)\n    return r\n\ndef pipeline(cfg, K2, K1, d):\n    sig = [2 * ((d >> i) & 1) - 1 for i in range(DB)]\n    R = np.full_like(d, K1)\n    for i in cfg.rrow_list():\n        R = R - sig[i] * rd(K2, cfg.sh2[i])\n    Q = np.zeros_like(d)\n    for i in reversed(range(DB)):\n        Q = Q + sig[i] * rd(R, cfg.sh1[i])\n    return R, Q\n\nMAX_X = 3_133_452\n\ndef valid_residuals(e,a):\n    d=np.arange(1<<DB,dtype=np.int64);xn=(1<<MB)+(a<<DB)+d\n    mask=(d%2==0) if e else ((xn<=MAX_X)|(d%4==0))\n    return d[mask]\n\ndef calibrate_segment(cfg,e,a,c_radius=1,b_radius=3):\n    d=valid_residuals(e,a);lo=1+a*2.**(DB-MB);mid=lo+((1<<DB)-1)/2**(MB+1)\n    scale=2**((e-1)/2);co=fitc(lo,lo+2.**(DB-MB),mid,scale)\n    f=scale/np.sqrt(lo+d*2.**-MB);target=f*2**cfg.fy;w=1/f\n    # Start is the rounded Lawson coefficient, not an imported table.\n    K2c=round(co[2]*2**cfg.f2);K1c=round(-co[1]*2**cfg.f1);best=None;quant=1<<cfg.gy\n    for K2 in range(max(0,K2c-c_radius),min((1<<cfg.w2)-1,K2c+c_radius)+1):\n        for K1 in range(K1c-b_radius,K1c+b_radius+1):\n            R,Q=pipeline(cfg,K2,K1,d)\n            if R.min()<0:continue\n            vals=target+Q;center=.5*(vals.min()+vals.max())+.5*(quant-1)\n            k0s=np.arange(math.floor(center)-quant-1,math.ceil(center)+quant+2,dtype=np.int64)\n            # The stored high anchor and the carry must keep the leading one fixed.\n            k0s=k0s[k0s <= (1<<cfg.fy)-1+int(Q.min())]\n            if len(k0s)==0:continue\n            y=((k0s[:,None]-Q)>>cfg.gy)<<cfg.gy\n            scores=np.max(abs(y-target)*w,axis=1)/2**cfg.fy;i=int(scores.argmin())\n            v=float(scores[i]);K0=int(k0s[i])\n            if best is None or v<best[0]:best=(v,K2,K1,K0)\n    if best is None:raise RuntimeError(\'No legal coefficient triple\')\n    return best\n\ndef regenerate_tables_exact(cfg,refine_threshold=.95):\n    """Fit every segment from the mathematical target.\n\n    The integer search is centred on the rounded Lawson coefficient.\n    No coefficient file is read.\n    """\n    tabs={};ratios=[]\n    for e in(0,1):\n        for a in range(128):\n            best=calibrate_segment(cfg,e,a)\n            if best[0]*2**24>refine_threshold:\n                alt=calibrate_segment(cfg,e,a,c_radius=4,b_radius=7)\n                if alt[0]<best[0]:best=alt\n            tabs[(e,a)]=best[1:];ratios.append(best[0]*2**24)\n    return tabs,dict(worst_ratio=max(ratios),worst_key=[int(np.argmax(ratios))//128,int(np.argmax(ratios))%128],reachable_normalized_inputs=sum(len(valid_residuals(e,a)) for e in(0,1) for a in range(128)))\n\ndef build_tables(cfg, search=2):\n    """Regenerate coefficients with the exact-domain radius policy.\n\n    The radius policy is adapted from the Kozykorpesh dataset domain-tables\n    module (SHA-256 59cf4d98899433dcc49d9ce512293be20d33da5334211c20e35323555994a4fb).\n    Its licence is unresolved; the version-pinned URL is in\n    `fit_coefficients.RECIPE["calibration_lineage"]`. The selected build calls\n    this through `fit_coefficients.derived_tables`.\n    """\n    tables, report = regenerate_tables_exact(cfg)\n    return tables, report[\'worst_ratio\'] / 2**24\n\ndef layout(cfg, tabs, LB=21, S=20):\n    d = np.arange(2**DB, dtype=np.int64)\n    assert max(t[0] for t in tabs.values()) < (1 << cfg.w2)\n    Rmax = 0; per = {}; Ymax = 0\n    for key, (K2, K1, K0) in tabs.items():\n        R, Q = pipeline(cfg, K2, K1, d)\n        assert R.min() >= 0\n        Rmax = max(Rmax, int(R.max()), K1)\n        per[key] = (int(Q.min()), int(Q.max()))\n        Ymax = max(Ymax, K0 - per[key][0])\n    w1 = Rmax.bit_length()\n    lo_tab, hi_tab = {}, {}\n    for key, (K2, K1, K0) in tabs.items():\n        qmin, qmax = per[key]\n        base = K0 % (1 << S); j = 0\n        while base + j * (1 << S) - qmax < 0: j += 1\n        k0lo = base + j * (1 << S)\n        assert k0lo - qmin < (1 << LB), \'layout infeasible\'\n        lo_tab[key] = k0lo; hi_tab[key] = (K0 - k0lo) >> S\n        assert hi_tab[key] >= 0\n    whi = max(max(hi_tab.values()).bit_length(), 1)\n    return dict(w1=w1, LB=LB, S=S, whi=whi, lo=lo_tab, hi=hi_tab)\n\n# ----------------------------------------------------------------- circuit pieces\ndef and_all_into(b, controls, target):\n    k = len(controls)\n    if k == 1: b.cx(controls[0], target); return\n    if k == 2: b.ccx(controls[0], controls[1], target); return\n    anc = b.alloc_qubits(k - 2)\n    b.init_and(controls[0], controls[1], anc[0])\n    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])\n    b.ccx(anc[k - 3], controls[k - 1], target)\n    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])\n    b.del_and(controls[0], controls[1], anc[0])\n    b.free(anc)\n\ndef phase_all(b, controls):\n    k = len(controls)\n    if k == 1: b.z(controls[0]); return\n    if k == 2: b.cz(controls[0], controls[1]); return\n    anc = b.alloc_qubits(k - 2)\n    b.init_and(controls[0], controls[1], anc[0])\n    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])\n    b.cz(anc[k - 3], controls[k - 1])\n    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])\n    b.del_and(controls[0], controls[1], anc[0])\n    b.free(anc)\n\nNSTAGES = [(4, 16), (3, 8), (2, 4), (1, 2), (0, 1)]\n\ndef _baseline_normalize(b, x, f):\n    for (k, d) in NSTAGES:\n        top = x[W - d:W]\n        if d == 1:\n            b.cx(x[W - 1], f[0]); b.x(f[0])\n        else:\n            b.x(top); and_all_into(b, list(top), f[k]); b.x(top)\n        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])\n\n\ndef _combined_normalize(b, x, f):\n    """v3 step 1: f4, f3, f2 (the binary leading-zero count div 4) from one prefix-AND chain on the unshifted x.\n    chain[m] = [x[W-m-2:W] == 0] (top m+2 bits zero); p_j = chain[4j-2] = [lz >= 4j].  f4 = p4, f3 = p2 ^ p4,\n    f2 = p1 ^ p2 ^ p3 ^ p4 ^ p5 (nested prefixes, so XOR = bit 2 of lz).  19 ANDs instead of 15 + 7 + 3."""\n    bits = [x[W - 1 - m] for m in range(20)]                  # x21, x20, ..., x2\n    b.x(km.array(bits))\n    chain = b.alloc_qubits(19)\n    b.init_and(bits[0], bits[1], chain[0])\n    for m in range(1, 19): b.init_and(chain[m - 1], bits[m + 1], chain[m])\n    p = {j: chain[4 * j - 2] for j in range(1, 6)}\n    b.cx(p[4], f[4])\n    b.cx(p[2], f[3]); b.cx(p[4], f[3])\n    for j in range(1, 6): b.cx(p[j], f[2])\n    for m in reversed(range(1, 19)): b.del_and(chain[m - 1], bits[m + 1], chain[m])\n    b.del_and(bits[0], bits[1], chain[0])\n    b.free(chain)\n    b.x(km.array(bits))\n    for (k, d) in NSTAGES:\n        top = x[W - d:W]\n        if d == 1:\n            b.cx(x[W - 1], f[0]); b.x(f[0])\n        elif d == 2:\n            b.x(top); b.ccx(top[0], top[1], f[1]); b.x(top)\n        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])\n\n# MAX_X < 3*2**20 proves the original two top bits are never both one.\n# After inversion, their AND equals XOR with one, on every promised input.\ndef normalize(b, x, f):\n    """v3 step 1: f4, f3, f2 (the binary leading-zero count div 4) from one prefix-AND chain on the unshifted x.\n    chain[m] = [x[W-m-2:W] == 0] (top m+2 bits zero); p_j = chain[4j-2] = [lz >= 4j].  f4 = p4, f3 = p2 ^ p4,\n    f2 = p1 ^ p2 ^ p3 ^ p4 ^ p5 (nested prefixes, so XOR = bit 2 of lz).  19 ANDs instead of 15 + 7 + 3."""\n    bits = [x[W - 1 - m] for m in range(20)]                  # x21, x20, ..., x2\n    b.x(km.array(bits))\n    chain = b.alloc_qubits(19)\n    b.x(chain[0]); b.cx(bits[0],chain[0]); b.cx(bits[1],chain[0])\n    for m in range(1, 19): b.init_and(chain[m - 1], bits[m + 1], chain[m])\n    p = {j: chain[4 * j - 2] for j in range(1, 6)}\n    b.cx(p[4], f[4])\n    b.cx(p[2], f[3]); b.cx(p[4], f[3])\n    for j in range(1, 6): b.cx(p[j], f[2])\n    for m in reversed(range(1, 19)): b.del_and(chain[m - 1], bits[m + 1], chain[m])\n    b.cx(bits[1],chain[0]); b.cx(bits[0],chain[0]); b.x(chain[0])\n    b.free(chain)\n    b.x(km.array(bits))\n    for (k, d) in NSTAGES:\n        top = x[W - d:W]\n        if d == 1:\n            b.cx(x[W - 1], f[0]); b.x(f[0])\n        elif d == 2:\n            b.x(top); b.ccx(top[0], top[1], f[1]); b.x(top)\n        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])\n\ndef _baseline_unnormalize(b, x, f):\n    for (k, d) in reversed(NSTAGES):\n        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])\n        top = x[W - d:W]\n        if d == 1:\n            b.x(f[0]); b.cx(x[W - 1], f[0])\n        else:\n            m = b.alloc_bits(1)[0]\n            b.hmr(f[k], m)\n            b.push_condition(m)                     # fixup runs on half the shots on average\n            b.x(top); phase_all(b, list(top)); b.x(top)\n            b.pop_condition()\n            b.free(m)\n\ndef promise_phase_original_top(b,controls):\n    """Phase the original top16-zero predicate using its promised bit relation."""\n    assert len(controls)==16\n    controls=list(reversed(controls))\n    k=len(controls)\n    anc=b.alloc_qubits(k-2)\n    b.x(anc[0]); b.cx(controls[0],anc[0]); b.cx(controls[1],anc[0])\n    for j in range(1,k-2):\n        b.init_and(anc[j-1],controls[j+1],anc[j])\n    b.cz(anc[k-3],controls[k-1])\n    for j in reversed(range(1,k-2)):\n        b.del_and(anc[j-1],controls[j+1],anc[j])\n    b.cx(controls[1],anc[0]); b.cx(controls[0],anc[0]); b.x(anc[0])\n    b.free(anc)\n\ndef unnormalize(b, x, f):\n    for (k, d) in reversed(NSTAGES):\n        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])\n        top = x[W - d:W]\n        if d == 1:\n            b.x(f[0]); b.cx(x[W - 1], f[0])\n        else:\n            m = b.alloc_bits(1)[0]\n            b.hmr(f[k], m)\n            b.push_condition(m)                     # fixup runs on half the shots on average\n            b.x(top)\n            if d==16: promise_phase_original_top(b,list(top))\n            else: phase_all(b,list(top))\n            b.x(top)\n            b.pop_condition()\n            b.free(m)\n\ndef gadd(b, a, t, cin=None, sub=False):\n    """t += a + cin (or t -= a + cin), Gidney adder: len(t)-1 Toffolis, uncompute by HMR."""\n    a = list(a); t = list(t); o = len(a); n = len(t)\n    assert o <= n\n    if sub: b.x(km.array(t))\n    c = b.alloc_qubits(n - 1) if n > 1 else km.array([])\n    def C(i): return cin if i == 0 else c[i - 1]\n    for i in range(0, n - 1):\n        if i == 0 and cin is None:\n            if o: b.init_and(a[0], t[0], C(1))\n        elif i < o:\n            b.cx(C(i), a[i]); b.cx(C(i), t[i]); b.init_and(a[i], t[i], C(i + 1)); b.cx(C(i), C(i + 1))\n        else:\n            b.init_and(C(i), t[i], C(i + 1))\n    if n - 1 < o: b.cx(a[n - 1], t[n - 1])\n    if n > 1 or cin is not None: b.cx(C(n - 1), t[n - 1])\n    for i in reversed(range(0, n - 1)):\n        if i == 0 and cin is None:\n            if o: b.del_and(a[0], t[0], C(1)); b.cx(a[0], t[0])\n        elif i < o:\n            b.cx(C(i), C(i + 1)); b.del_and(a[i], t[i], C(i + 1)); b.cx(C(i), a[i]); b.cx(a[i], t[i])\n        else:\n            b.del_and(C(i), t[i], C(i + 1)); b.cx(C(i), t[i])\n    if n > 1: b.free(c)\n    if sub: b.x(km.array(t))\n\ndef _baseline_shift_stages(b, f, reg, L, inverse):\n    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]\n    occ = []; o = L\n    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))\n    seq = list(enumerate(stages))\n    if not inverse:\n        for idx, (k, d) in seq:\n            for i in reversed(range(0, occ[idx])):\n                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])\n    else:\n        for idx, (k, d) in reversed(seq):\n            for i in range(0, occ[idx]):\n                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])\n\n\ndef shift_stages(b, f, reg, L, inverse):\n    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]\n    occ = []; o = L\n    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))\n    seq = list(enumerate(stages))\n    if not inverse:\n        for idx, (k, d) in seq:\n            for i in reversed(range(0, occ[idx])):\n                if i + d >= len(reg): continue\n                if i + d >= occ[idx]:                       # v2: target is |0> for every f\n                    b.init_and(f[k], reg[i], reg[i + d]); b.cx(reg[i + d], reg[i])\n                else:\n                    b.cswap(f[k], reg[i], reg[i + d])\n    else:\n        for idx, (k, d) in reversed(seq):\n            for i in range(0, occ[idx]):\n                if i + d >= len(reg): continue\n                if i + d >= occ[idx]:\n                    b.cx(reg[i + d], reg[i]); b.del_and(f[k], reg[i], reg[i + d])\n                else:\n                    b.cswap(f[k], reg[i], reg[i + d])\n\ndef build_circuit(cfg, tabs, lay, adder=None):\n    # The default preserves the public generator exactly. An injected adder\n    # supports isolated, explicitly tested memory experiments.\n    if adder is None: adder=gadd\n    w2, w1, LB, S, whi = cfg.w2, lay[\'w1\'], lay[\'LB\'], lay[\'S\'], lay[\'whi\']\n    gy, fy = cfg.gy, cfg.fy\n    b = km.CircuitBuilder()\n    x = b.create_quantum_register(W, name=\'x\')\n    f = b.alloc_qubits(5)\n    c2 = b.alloc_qubits(w2); R = b.alloc_qubits(w1); Lr = b.alloc_qubits(LB); Yh = b.alloc_qubits(whi)\n    tgt = km.array(list(Lr) + list(R) + list(c2) + list(Yh))\n    table = [0] * 256\n    for (e, a), (K2, K1, K0) in tabs.items():\n        table[a + 128 * e] = lay[\'lo\'][(e, a)] | (K1 << LB) | (K2 << (LB + w1)) | (lay[\'hi\'][(e, a)] << (LB + w1 + w2))\n    addr = km.array(list(x[14:21]) + [f[0]])\n    delta = x[0:DB]\n    def rrows(inverse):\n        seq = cfg.rrow_list()\n        if inverse: seq = list(reversed(seq))\n        for i in seq:\n            sh = cfg.sh2[i]\n            if sh >= 0: off = c2[sh:w2]; t = R[0:w1]; cin = c2[sh - 1] if sh >= 1 else None\n            else: off = c2[0:w2]; t = R[-sh:w1]; cin = None\n            b.cx(delta[i], t); adder(b, off, t, cin, sub=inverse); b.cx(delta[i], t)\n    def yrows(inverse):\n        seq = list(reversed(range(DB)))\n        if inverse: seq.reverse()\n        for i in seq:\n            sh = cfg.sh1[i]; assert 0 <= sh < w1\n            t = Lr[0:LB]; cin = R[sh - 1] if sh >= 1 else None\n            b.cx(delta[i], t); adder(b, R[sh:w1], t, cin, sub=inverse); b.cx(delta[i], t)\n    normalize(b, x, f)\n    b.init_lookup(table=table, address=addr, target=tgt)\n    rrows(False); yrows(False)\n    ext = b.alloc_qubits(10)\n    if LB > S: adder(b, Lr[S:LB], Yh)\n    Cx = km.array(list(Lr[gy:S]) + list(Yh) + list(ext))\n    shift_stages(b, f, Cx, S - gy + whi, False)\n    for j in range(len(Cx)):\n        e = gy + j - fy - 10                       # bit weight 2^e turns\n        if e < 0: b.z_pow(Cx[j], Fraction(2)**(e + 1))\n    shift_stages(b, f, Cx, S - gy + whi, True)\n    if LB > S: adder(b, Lr[S:LB], Yh, sub=True)\n    b.free(ext)\n    yrows(True); rrows(True)\n    b.del_lookup(table=table, address=addr, target=tgt)\n    b.free(tgt)\n    unnormalize(b, x, f)\n    b.free(f)\n    return b.finish_circuit()\n\ndef main(out):\n    cfg = Cfg(F2, F1, FYR, GY, DROP2)\n    tabs, worst = build_tables(cfg)\n    print(f\'model worst relative error / 2^-24 = {worst * 2**24:.3f}\')\n    lay = layout(cfg, tabs)\n    c = build_circuit(cfg, tabs, lay)\n    s = c.operation_statistics()\n    print(\'qubits\', c.num_qubits, \'toffoli(static)\', s.num_toffoli_like,\n          \'rotations\', s.num_z_pow(min_clifford_hierarchy_level=4))\n    c.to_file(out, format=\'kmb\')\n    print(\'wrote\', out)\n\n\ndef selected_cfg():\n    return Cfg(F2,F1,FYR,GY,DROP2)\n\n\ndef config_record(cfg):\n    return dict(f2=cfg.f2,f1=cfg.f1,fyr=cfg.fyr,gy=cfg.gy,drop2=sorted(cfg.drop2))\n\n\n\n\ndef load_or_build_tables(path=None, regenerate=False):\n    """Derive the 256 coefficient triples. No dataset is read.\n\n    `tasks/invsqrt/fit_coefficients.py` is the source. `regenerate=True` fits\n    a second time and requires the same integers. A digest check inside the\n    fitter stops a numeric-library change from silently changing the circuit.\n    """\n    import json\n    from pathlib import Path\n    from tasks.invsqrt.fit_coefficients import PROVENANCE, coefficient_rows, derived_tables\n    cfg, tables, worst, _report = derived_tables()\n    rows = coefficient_rows(tables)\n    if regenerate:\n        again, _again_report = regenerate_tables_exact(cfg)\n        assert coefficient_rows(again) == rows, \'A second fit disagreed with the cached coefficients.\'\n    path = Path(path) if path else None\n    if path and path.exists():\n        saved = json.loads(path.read_text())\n        assert saved[\'config\'] == config_record(cfg)\n        assert saved[\'rows\'] == rows, \'Cached coefficient table differs from the derived model.\'\n        assert saved.get(\'provenance\') == PROVENANCE\n    elif path:\n        path.write_text(json.dumps({\n            \'provenance\': PROVENANCE,\n            \'source\': \'tasks/invsqrt/fit_coefficients.py\',\n            \'config\': config_record(cfg),\n            \'normalized_worst_relative_error\': worst,\n            \'rows\': rows,\n        }, indent=2))\n    return cfg, tables, worst\n\n\ndef integer_model(xs,cfg,tabs):\n    """Exact emitted arithmetic in units 2^-(cfg.fy+10) turns, before mod 1."""\n    xs=np.asarray(xs,dtype=np.int64)\n    if not ((xs>=1)&(xs<=3133452)).all():\n        raise ValueError(\'The supported competition domain is x=1..3133452\')\n    # All inputs fit exactly in float64; frexp gives the exact binary exponent.\n    shifts=21-(np.frexp(xs)[1].astype(np.int64)-1)\n    norm=xs << shifts\n    assert ((norm >> 21)==1).all(), \'Normalized top-bit borrowing requires x[21]=1\'\n    parity=shifts&1\n    index=((norm >> 14)&127)+128*parity\n    table=np.asarray([tabs[(e,a)] for e in range(2) for a in range(128)],dtype=np.int64)\n    K2,K1,K0=(table[index,j] for j in range(3))\n    d=norm&((1 << DB)-1)\n    signs=[2*((d >> i)&1)-1 for i in range(DB)]\n    R=K1.copy()\n    for i in cfg.rrow_list():\n        R-=signs[i]*rd(K2,cfg.sh2[i])\n    Q=np.zeros_like(xs)\n    for i in reversed(range(DB)):\n        Q+=signs[i]*rd(R,cfg.sh1[i])\n    Y=((K0-Q) >> cfg.gy) << cfg.gy\n    if (R<0).any() or (Y<0).any():\n        raise OverflowError(\'An unsigned intermediate is negative\')\n    return Y << (shifts >> 1)\n\n\ndef exhaustive_certificate(cfg,tabs,chunk_size=131072):\n    """Every promised basis input, exact integer target and tolerance.\n\n    This is an arithmetic certificate. Ancillas, restored inputs and phase\n    feed-forward require fresh circuit simulation and the official verifier.\n    """\n    output_fraction=cfg.fy+10\n    worst_ratio,worst_x,min_margin=0.,0,1 << 64\n    for start in range(1,3133453,chunk_size):\n        xs=np.arange(start,min(start+chunk_size,3133453),dtype=np.int64)\n        values=integer_model(xs,cfg,tabs).astype(np.uint64)\n        phase=values << np.uint64(64-output_fraction)\n        roots=[math.isqrt((1 << 130)//int(x)) for x in xs]\n        target=np.array([((r+1) >> 1)&((1 << 64)-1) for r in roots],dtype=np.uint64)\n        tolerance=np.array([min(r >> 25,1 << 63) for r in roots],dtype=np.uint64)\n        error=np.abs((phase-target).view(np.int64)).view(np.uint64)\n        ratio=error/tolerance\n        i=int(np.argmax(ratio))\n        if float(ratio[i])>worst_ratio:\n            worst_ratio,worst_x=float(ratio[i]),int(xs[i])\n        if (error>tolerance).any():\n            i=int(np.flatnonzero(error>tolerance)[0])\n            return {\'pass\':False,\'first_bad_x\':int(xs[i]),\'ratio\':float(ratio[i])}\n        min_margin=min(min_margin,int((tolerance-error).min()))\n    return {\'pass\':True,\'tested_inputs\':3133452,\'worst_ratio\':worst_ratio,\n            \'worst_x\':worst_x,\'min_margin_units\':min_margin,\n            \'phase_fraction_bits\':output_fraction}\n\n\ndef conditional_toffoli_discount(circuit):\n    """Mean discount for this emitted family\'s fair HMR conditional branches."""\n    depth=0\n    total=0\n    discount=0.\n    for line in str(circuit).splitlines():\n        opcode=line.split(\' \',1)[0]\n        if opcode==\'PUSH_CONDITION\':depth+=1\n        elif opcode==\'POP_CONDITION\':depth-=1\n        elif opcode in [\'CCX\',\'CSWAP\']:\n            total+=1\n            if depth:\n                assert depth==1,\'Expected one fair measurement bit in a Toffoli branch.\'\n                discount+=.5\n    assert total==int(circuit.operation_statistics().num_toffoli_like)\n    assert depth==0\n    return discount\n\ndef estimated_resources(circuit,certificate):\n    """Model-based cost; full unchanged official verification remains required."""\n    assert certificate[\'pass\'] and certificate[\'min_margin_units\']>0\n    stats=circuit.operation_statistics()\n    rotations=int(stats.num_z_pow(min_clifford_hierarchy_level=4))\n    epsilon=math.pi*(certificate[\'min_margin_units\']/2**64)/rotations\n    per_rotation=0.56*math.log2(1/epsilon)+5.3\n    static_toffoli=int(stats.num_toffoli_like)\n    # Fair flag measurements activate the emitted normalization phase-fixup\n    # Toffolis on half the shots. Read their actual branch counts so the\n    # original schedules discount 11 and the promise rewrite discounts 10.5.\n    normalization_discount=conditional_toffoli_discount(circuit)\n    expected_toffoli=static_toffoli-normalization_discount\n    t_like=int(stats.num_t_like)\n    extra=(t_like+rotations*per_rotation)/4\n    return {\'qubits\':circuit.num_qubits,\'static_toffoli\':static_toffoli,\n            \'expected_mean_toffoli\':expected_toffoli,\'t_like\':t_like,\n            \'rotations\':rotations,\'rotation_epsilon\':epsilon,\n            \'estimated_expected_cost\':circuit.num_qubits*(expected_toffoli+extra),\n            \'static_cost_upper_estimate\':circuit.num_qubits*(static_toffoli+extra),\n            \'full_official_verification_required\':True}\n\n\n\n\n# ---- Our isolated phase scheduling, folded carry and borrowed workspace ----\ndef folded_gadd(b,a,t,cin=None,sub=False,borrowed=()):\n    a,t=list(a),list(t)\n    o,n=len(a),len(t)\n    assert o<=n and n>=1\n    if sub:\n        for q in t:b.x(q)\n    if n==1:\n        if o:b.cx(a[0],t[0])\n        if cin is not None:b.cx(cin,t[0])\n        if sub:\n            for q in t:b.x(q)\n        return\n    borrowed=list(borrowed)[:max(0,n-2)]\n    count=max(0,n-2-len(borrowed))\n    allocated=b.alloc_qubits(count) if count else []\n    c=borrowed+list(allocated)\n    def C(i):return cin if i==0 else c[i-1]\n    for i in range(n-2):\n        if i==0 and cin is None:\n            if o:b.init_and(a[0],t[0],C(1))\n        elif i<o:\n            b.cx(C(i),a[i]);b.cx(C(i),t[i])\n            b.init_and(a[i],t[i],C(i+1));b.cx(C(i),C(i+1))\n        else:\n            b.init_and(C(i),t[i],C(i+1))\n    i=n-2\n    carry=C(i)\n    if carry is None:\n        if i<o:\n            b.ccx(a[i],t[i],t[-1]);b.cx(a[i],t[i])\n    elif i<o:\n        b.cx(carry,a[i]);b.cx(carry,t[i])\n        b.ccx(a[i],t[i],t[-1]);b.cx(carry,t[-1])\n        b.cx(carry,a[i]);b.cx(a[i],t[i])\n    else:\n        b.ccx(carry,t[i],t[-1]);b.cx(carry,t[i])\n    if n-1<o:b.cx(a[n-1],t[n-1])\n    for i in reversed(range(n-2)):\n        if i==0 and cin is None:\n            if o:b.del_and(a[0],t[0],C(1));b.cx(a[0],t[0])\n        elif i<o:\n            b.cx(C(i),C(i+1));b.del_and(a[i],t[i],C(i+1))\n            b.cx(C(i),a[i]);b.cx(a[i],t[i])\n        else:\n            b.del_and(C(i),t[i],C(i+1));b.cx(C(i),t[i])\n    if count:b.free(allocated)\n    if sub:\n        for q in t:b.x(q)\n\n\ndef truth_table_check(max_width=6,borrowed_count=0):\n    class Bits:\n        def __init__(self):self.v=[]\n        def alloc_qubits(self,n):\n            ids=list(range(len(self.v),len(self.v)+n));self.v.extend([0]*n);return ids\n        def x(self,q):self.v[q]^=1\n        def cx(self,a,t):self.v[t]^=self.v[a]\n        def ccx(self,a,b,t):self.v[t]^=self.v[a]&self.v[b]\n        def init_and(self,a,b,t):\n            assert self.v[t]==0\n            self.v[t]=self.v[a]&self.v[b]\n        def del_and(self,a,b,t):\n            assert self.v[t]==(self.v[a]&self.v[b])\n            self.v[t]=0\n        def free(self,reg):assert not any(self.v[q] for q in reg)\n        def put(self,reg,value):\n            for i,q in enumerate(reg):self.v[q]=(value >> i)&1\n        def read(self,reg):return sum(self.v[q] << i for i,q in enumerate(reg))\n    count=0\n    for n in range(1,max_width+1):\n        for m in range(n+1):\n            for a in range(1 << m):\n                for value in range(1 << n):\n                    for cin_value in (None,0,1):\n                        for sub in (False,True):\n                            b=Bits();areg=b.alloc_qubits(m);target=b.alloc_qubits(n)\n                            cin=b.alloc_qubits(1)[0] if cin_value is not None else None\n                            if cin is not None:b.v[cin]=cin_value\n                            b.put(areg,a);b.put(target,value)\n                            borrowed=b.alloc_qubits(borrowed_count)\n                            folded_gadd(b,areg,target,cin,sub,borrowed=borrowed)\n                            delta=a+(cin_value or 0)\n                            expected=(value-delta if sub else value+delta) % (1 << n)\n                            assert b.read(target)==expected,(n,m,a,value,cin_value,sub)\n                            assert b.read(areg)==a\n                            assert cin is None or b.v[cin]==cin_value\n                            assert not any(b.v[q] for q in borrowed)\n                            count+=1\n    return count\n\n\ndef build_split_circuit(cfg,tabs,lay,adder=None,borrow_top=False):\n    if adder is None:adder=gadd\n    w2,w1,LB,S,whi=cfg.w2,lay[\'w1\'],lay[\'LB\'],lay[\'S\'],lay[\'whi\']\n    b=km.CircuitBuilder()\n    x=b.create_quantum_register(W,name=\'x\')\n    flags=b.alloc_qubits(5)\n    normalize(b,x,flags)\n    # Every promised positive input normalizes to x[21]=1. The arithmetic\n    # address uses x[14:21] and delta x[:14], so this wire is idle in the core.\n    if borrow_top:\n        b.x(x[-1])\n        raw_adder=adder\n        def adder(builder,a,t,cin=None,sub=False):\n            return raw_adder(builder,a,t,cin,sub,borrowed=(x[-1],))\n    c2=b.alloc_qubits(w2)\n    R=b.alloc_qubits(w1)\n    Lr=b.alloc_qubits(LB)\n    Yh=b.alloc_qubits(whi)\n    remaining=km.array(list(Lr)+list(R)+list(c2))\n    target=km.array(list(remaining)+list(Yh))\n    table=[0]*256\n    rest_table=[0]*256\n    high_table=[0]*256\n    for (e,a),(K2,K1,K0) in tabs.items():\n        idx=a+128*e\n        rest_table[idx]=lay[\'lo\'][(e,a)] | (K1 << LB) | (K2 << (LB+w1))\n        high_table[idx]=lay[\'hi\'][(e,a)]\n        table[idx]=rest_table[idx] | (high_table[idx] << (LB+w1+w2))\n    address=km.array(list(x[14:21])+[flags[0]])\n    delta=x[:DB]\n    def rrows(inverse):\n        seq=cfg.rrow_list()\n        if inverse:seq=list(reversed(seq))\n        for i in seq:\n            sh=cfg.sh2[i]\n            if sh>=0:off=c2[sh:w2];t=R[:w1];cin=c2[sh-1] if sh>=1 else None\n            else:off=c2[:w2];t=R[-sh:w1];cin=None\n            b.cx(delta[i],t);adder(b,off,t,cin,sub=inverse);b.cx(delta[i],t)\n    def yrows(inverse):\n        seq=list(reversed(range(DB)))\n        if inverse:seq.reverse()\n        for i in seq:\n            sh=cfg.sh1[i]\n            assert 0<=sh<w1\n            cin=R[sh-1] if sh>=1 else None\n            b.cx(delta[i],Lr);adder(b,R[sh:w1],Lr,cin,sub=inverse);b.cx(delta[i],Lr)\n    def phase(reg,base_bit):\n        ext=b.alloc_qubits(10)\n        word=km.array(list(reg)+list(ext))\n        shift_stages(b,flags,word,len(reg),False)\n        for j in range(len(word)):\n            exponent=base_bit+j-cfg.fy-10\n            if exponent<0:b.z_pow(word[j],Fraction(2)**(exponent+1))\n        shift_stages(b,flags,word,len(reg),True)\n        b.free(ext)\n    b.init_lookup(table=table,address=address,target=target)\n    phase(Yh,S)\n    b.del_lookup(table=high_table,address=address,target=Yh)\n    b.free(Yh)\n    rrows(False)\n    yrows(False)\n    phase(Lr[cfg.gy:LB],cfg.gy)\n    yrows(True)\n    rrows(True)\n    b.del_lookup(table=rest_table,address=address,target=remaining)\n    b.free(remaining)\n    if borrow_top:b.x(x[-1])\n    unnormalize(b,x,flags)\n    b.free(flags)\n    return b.finish_circuit()\n\n\n\ndef build_previous_schedule(cfg,tabs,lay=None):\n    """Regenerate 91Q circuit from the explicit model and credited schedules."""\n    if lay is None: lay=layout(cfg,tabs)\n    return build_split_circuit(cfg,tabs,lay,adder=folded_gadd,borrow_top=True)\n\n\n# The derivative-split version of this API is defined below.\n\n# The derivative-split version of this API is defined below.\n\n\n\n# The derivative-split version of this API is defined below.\n\n\n\n# The derivative-split version of this API is defined below.\n\n\ndef cli():\n    import argparse\n    import json\n    from pathlib import Path\n    parser = argparse.ArgumentParser(description=\'Generate the credited signed-digit circuit and memory-schedule adaptation.\')\n    parser.add_argument(\'output\', nargs=\'?\', default=\'invsqrt.kmb\')\n    parser.add_argument(\'--baseline\', action=\'store_true\')\n    args = parser.parse_args()\n    cfg, tabs, _ = load_or_build_tables()\n    certificate = exhaustive_certificate(cfg, tabs)\n    if not certificate[\'pass\']:\n        raise ArithmeticError(certificate)\n    lay = layout(cfg, tabs)\n    circuit = build_memory_baseline(cfg, tabs, lay) if args.baseline else build_selected(cfg, tabs, lay)\n    circuit.to_file(args.output, format=\'kmb\')\n    report = estimated_resources(circuit, certificate)\n    report[\'arithmetic_certificate\'] = certificate\n    report[\'variant\'] = \'sumit_baseline\' if args.baseline else selected_descriptor()[\'variant\']\n    Path(args.output + \'.resources.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    print(json.dumps(report, indent=2))\n\n\n\n\n\ndef _rudra_gadd_top_reused(b, source, target, cin=None, sub=False):\n    """Gidney addition with n-2 clean carries; the last carry targets the MSB."""\n    import kickmix as km\n    a, t = list(source), list(target)\n    n, m = len(t), len(a)\n    assert n >= 1 and m <= n\n    if sub:\n        b.x(km.array(t))\n    carries = b.alloc_qubits(n - 2) if n > 2 else km.array([])\n    for i in range(n - 1):\n        previous = cin if i == 0 else carries[i - 1]\n        last = i == n - 2\n        out = t[-1] if last else carries[i]\n        if previous is None:\n            if i < m:\n                (b.ccx if last else b.init_and)(a[i], t[i], out)\n        elif i < m:\n            b.cx(previous, a[i]); b.cx(previous, t[i])\n            (b.ccx if last else b.init_and)(a[i], t[i], out)\n            b.cx(previous, out)\n        else:\n            (b.ccx if last else b.init_and)(previous, t[i], out)\n    if n - 1 < m:\n        b.cx(a[n - 1], t[n - 1])\n    if n == 1 and cin is not None:\n        b.cx(cin, t[0])\n    for i in reversed(range(n - 1)):\n        previous = cin if i == 0 else carries[i - 1]\n        last = i == n - 2\n        if previous is None:\n            if i < m:\n                if not last:\n                    b.del_and(a[i], t[i], carries[i])\n                b.cx(a[i], t[i])\n        elif i < m:\n            if not last:\n                b.cx(previous, carries[i]); b.del_and(a[i], t[i], carries[i])\n            b.cx(previous, a[i]); b.cx(a[i], t[i])\n        else:\n            if not last:\n                b.del_and(previous, t[i], carries[i])\n            b.cx(previous, t[i])\n    if n > 2:\n        b.free(carries)\n    if sub:\n        b.x(km.array(t))\n\ndef prepare_derivative_split(reference, cfg, tables):\n    d = np.arange(1 << reference.DB, dtype=np.int64)\n    coarse, residual = {}, {}\n    bounds = {\'normalized_addresses_checked\': 0, \'min_residual\': 1 << 30,\n              \'max_residual\': 0, \'max_table_residual\': 0, \'max_coarse\': 0}\n    for key, (k2, k1, _) in tables.items():\n        derivative, _ = reference.pipeline(cfg, k2, k1, d)\n        base = int(derivative.min()) >> 11\n        low = derivative - (base << 11)\n        coarse[key], residual[key] = base, k1 - (base << 11)\n        assert 0 <= residual[key] < 1 << 12\n        assert (low >= 0).all() and (low < 1 << 12).all()\n        assert ((base + (low >> 11)) < 1 << 7).all()\n        joined = (low & 2047) + ((base + (low >> 11)) << 11)\n        assert np.array_equal(joined, derivative)\n        bounds[\'normalized_addresses_checked\'] += len(d)\n        bounds[\'min_residual\'] = min(bounds[\'min_residual\'], int(low.min()))\n        bounds[\'max_residual\'] = max(bounds[\'max_residual\'], int(low.max()))\n        bounds[\'max_table_residual\'] = max(bounds[\'max_table_residual\'], residual[key])\n        bounds[\'max_coarse\'] = max(bounds[\'max_coarse\'], base)\n    assert cfg.w2 >= 6 and len(cfg.rrow_list()) == 8\n    return {\'coarse\': coarse, \'residual\': residual}, bounds\n\ndef _build_derivative_circuit(reference, cfg, tables, layout, split, borrow_top=False):\n    import kickmix as km\n    assert (layout[\'w1\'], layout[\'LB\'], layout[\'S\'], layout[\'whi\']) == (18, 21, 20, 8)\n    b = km.CircuitBuilder()\n    x = b.create_quantum_register(22, name=\'x\')\n    flags = b.alloc_qubits(5)\n    # These coefficients are still zero during normalization; allocate afterward.\n    reference.normalize(b, x, flags)\n    if borrow_top: b.x(x[-1])\n    def adder(builder,a,t,cin=None,sub=False):\n        if borrow_top: return folded_gadd(builder,a,t,cin,sub,borrowed=(x[-1],))\n        return _rudra_gadd_top_reused(builder,a,t,cin,sub)\n    c2, residual, coarse, low, high = [b.alloc_qubits(n) for n in [cfg.w2, 12, 7, 21, 8]]\n    full_target = km.array(list(low) + list(residual) + list(c2) + list(high) + list(coarse))\n    rest_target = km.array(list(low) + list(residual) + list(c2) + list(coarse))\n    full_table, rest_table, high_table = [0] * 256, [0] * 256, [0] * 256\n    for (parity, address), (k2, _, _) in tables.items():\n        key, index = (parity, address), address + 128 * parity\n        common = layout[\'lo\'][key] | (split[\'residual\'][key] << 21) | (k2 << 33)\n        full_table[index] = common | (layout[\'hi\'][key] << (33+cfg.w2)) | (split[\'coarse\'][key] << (41+cfg.w2))\n        rest_table[index] = common | (split[\'coarse\'][key] << (33+cfg.w2))\n        high_table[index] = layout[\'hi\'][key]\n    address = km.array(list(x[14:21]) + [flags[0]])\n    delta = x[:14]\n\n    def phase_register(register, low_bit):\n        ext = b.alloc_qubits(10)\n        word = km.array(list(register) + list(ext))\n        reference.shift_stages(b, flags, word, len(register), False)\n        for i, qubit in enumerate(word):\n            exponent = low_bit + i - cfg.fy - 10\n            if exponent < 0:\n                b.z_pow(qubit, Fraction(2) ** (exponent + 1))\n        reference.shift_stages(b, flags, word, len(register), True)\n        b.free(ext)\n\n    def inner(inverse):\n        rows = cfg.rrow_list()\n        for i in reversed(rows) if inverse else rows:\n            shift = cfg.sh2[i]\n            # A narrower curvature coefficient can require an exact left shift.\n            # Add to the corresponding residual suffix, leaving lower bits intact.\n            if shift>=0:\n                addend=c2[shift:];target=residual\n                carry=c2[shift-1] if shift else None\n            else:\n                addend=c2;target=residual[-shift:];carry=None\n            b.cx(delta[i],target)\n            adder(b,addend,target,carry,sub=inverse)\n            b.cx(delta[i],target)\n\n    def outer(derivative, inverse):\n        for i in range(14) if inverse else reversed(range(14)):\n            shift = cfg.sh1[i]\n            carry = derivative[shift - 1] if shift else None\n            b.cx(delta[i], low)\n            adder(b, derivative[shift:], low, carry, sub=inverse)\n            b.cx(delta[i], low)\n\n    b.init_lookup(table=full_table, address=address, target=full_target)\n    phase_register(high, 20)\n    # High and the shift workspace have been restored. For every X-measurement\n    # mask m, unlookup contributes (-1)^(m.high(address)); its phase correction\n    # contributes the same sign. Their product is +1, preserving our phase.\n    # Address and its parity flag stay live and unchanged until after erasure.\n    b.del_lookup(table=high_table, address=address, target=high)\n    b.free(high)\n    inner(False)\n    adder(b, residual[11:], coarse)\n    derivative = residual[:11] + coarse\n    outer(derivative, False)\n    # Include bit20 here. There is no output join: floor((L+2^20H)/4)\n    # equals floor(L/4)+2^18H, so the two diagonal phases add exactly.\n    phase_register(low[cfg.gy:], cfg.gy)\n    outer(derivative, True)\n    adder(b, residual[11:], coarse, sub=True)\n    inner(True)\n    b.del_lookup(table=rest_table, address=address, target=rest_target)\n    b.free(rest_target)\n    if borrow_top: b.x(x[-1])\n    reference.unnormalize(b, x, flags)\n    b.free(flags)\n    return b.finish_circuit()\n\n\ndef _derivative_reference(promise):\n    return types.SimpleNamespace(**dict(globals(),\n        normalize=normalize if promise else _combined_normalize,\n        unnormalize=unnormalize if promise else _baseline_unnormalize))\n\n\ndef _build_derivative_variant(cfg,tabs,lay,borrow,promise):\n    if lay is None:lay=layout(cfg,tabs)\n    reference=_derivative_reference(promise)\n    split,bounds=prepare_derivative_split(reference,cfg,tabs)\n    return _build_derivative_circuit(reference,cfg,tabs,lay,split,borrow_top=borrow)\n\n\ndef build_selected(cfg,tabs,lay=None):\n    """Regenerate selected92Q derivative split with borrowed carry and domain rewrite."""\n    return _build_derivative_variant(cfg,tabs,lay,borrow=True,promise=True)\n\n\ndef build_memory_baseline(cfg,tabs,lay=None):\n    """Same Kozykorpesh coefficient model, Rudra/Kirill derivative schedule:93Q."""\n    return _build_derivative_variant(cfg,tabs,lay,borrow=False,promise=False)\n\n\ndef build_borrowed_baseline(cfg,tabs,lay=None):\n    """Same derivative schedule with the normalized input bit borrowed:92Q."""\n    return _build_derivative_variant(cfg,tabs,lay,borrow=True,promise=False)\n\n\ndef build_combined_baseline(cfg,tabs,lay=None):\n    """Compatibility alias for the borrowed-carry comparison."""\n    return build_borrowed_baseline(cfg,tabs,lay)\n\n\ndef comparison_report(cfg,tabs,certificate,lay=None):\n    """One fixed coefficient table: derivative schedule, borrow, domain rewrite."""\n    if lay is None:lay=layout(cfg,tabs)\n    return {\n        \'baseline\':estimated_resources(build_memory_baseline(cfg,tabs,lay),certificate),\n        \'borrowed\':estimated_resources(build_borrowed_baseline(cfg,tabs,lay),certificate),\n        \'selected\':estimated_resources(build_selected(cfg,tabs,lay),certificate),\n        \'selected_descriptor\':selected_descriptor()}\n\n\ndef selected_descriptor():\n    from tasks.invsqrt.fit_coefficients import PROVENANCE\n    return {\n        \'variant\':\'kozy_table_rudra_split_alex_schedule_borrow_and_domain\',\n        \'source_author\':\'Sumit Chongder; Rudra Bhaskar; Alex Chilton; Kozykorpesh; Kirill Tymchishin; anazemcev / Artem1410\',\n        \'source_url\':\'https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory\',\n        \'coefficient_source\':\'Lawson minimax and dataset-adapted exact-domain calibration in tasks/invsqrt/fit_coefficients.py\',\n        \'configuration\':{\'f2\':8,\'f1\':19,\'fyr\':27,\'gy\':1,\'drop2\':[5]},\n        \'license_as_published\':\'Apache-2.0 for the anazemcev v3 parent. Alex v8 and Mizero v1 are reference-only and are not this licence.\',\n        \'circuit_authorship\':\'port\',\n        \'coefficient_provenance\':PROVENANCE,\n        \'baseline_definition\':\'Same fixed Kozykorpesh table with the Rudra derivative split and Alex normalization/shifts, before normalized-input carry borrowing:93Q\',\n        \'baseline_canonical_sha256\':\'9e67c00941ff0623cfaf22f133fe7f70740f8b0a163a4ba64c6fdcd1c9446295\',\n        \'borrowed_canonical_sha256\':\'f3f8457058aecdcf2650e00b411234899cb01933b79a1f6519c0bf28b49591bc\',\n        \'parent_92q_sha256\':\'63f70e623dd995f4f87ca13254f46f991a5ae545d4c20c80e8510f5ff0a49bbd\',\n        \'selected_canonical_sha256\':\'75e8a0e2341bb3e8f8ded1ec1d53e24437410c69e887da9daf7d49bfcc28c2dd\',\n        \'historical_kirill95_sha256\':\'497b161bc468654b5b2570cfaa006ce336dec9b8c89f82f048fcca4e6e8f7919\',\n        \'previous_same_table91_sha256\':\'e3db0cc65b4771b640fbcd73d770d2015c232e9f54bd69479786fb5445ca76b9\',\n        \'measured_counts\':{\'qubits\':90,\'static_toffoli\':1448,\'rotations\':44},\n        \'expected_mean_toffoli\':1437.5,\n        \'estimated_cost\':157751.9485933627,\n        \'changes_in_this_revision\':[\n            \'Regenerate 256 coefficient triples with dataset-adapted calibration; licence unresolved\',\n            \'90-qubit Gray curvature schedule, exact-domain affine rewrite, and one calibrated rotation drop\',\n            \'Borrow normalized x[21] as clean adder carry, restoring it before inverse normalization\',\n            \'Retain our exact original-input top-bit mutual-exclusion rewrite\'],\n        \'upstream_contributions\':\'Rudra derivative split/top-carry/static-high phase checkpoint; Alex prefix flags/empty-target shift; Kirill exact integration; Sumit model; Kozykorpesh precision calibration\',\n        \'full_official_verification\':\'Required separately; actual unchanged full-verifier report identifies the selected hash\'}\n\n\n'}
SOURCE_HASHES = {'LICENSE': '40911d195a0c25e3647660174fbd30f95ad80af621bd1088bbf5649c970d4e58', 'NOTICE': '5189abb098bc3c42bb05b0759e00ff15f9d001cda2998f9ea35d1d53ac827f6b', 'ledger/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ledger/archive.py': '1950107fd862cf70a1ef7fc899737b334928ffb8e03ce0c69bd4c82d428a0f39', 'notices/10-invsqrt.txt': '7d35462413d4c7092f40fdafbda2e06af619f91867b59d2bd2d3ab8b9b2b76e7', 'notices/11-invsqrt-study.txt': '69316ab49abc23d5a4dee971fe8ab2349cc34871a03359ee01e87173639017a8', 'notices/12-invsqrt-mrmeows-audit.txt': '0bc23a786312c260b70fe8c2c06a0499baa997cecec72d204f37356e444e022b', 'notices/20-dqi.txt': 'b9009b1d46e3631a74582dde8d7210a44be0d3ac96bc40ed2a76f86a413b8315', 'notices/30-dfthcsos.txt': '3e0f6f85d4ca9ff37d58377f54417a5097dbfa2ff9f8634d2e43b282bad908ef', 'notices/35-chem-catchup.txt': '1001bc1ee85cd63909b4654c5ccb9376b6bb42b23323891ec5026481f7e884e1', 'notices/40-search.txt': '98e6787a0bfef9fc46d7b2367911f8c8ce33de3372b3a86ff89a13bdc425a21b', 'notices/90-not-included.txt': 'b2d0d485e9df1f3f30c498577e7ad6cab898fdd083a5c85f637b1cf1b08ea1f5', 'tasks/__init__.py': 'c51df006aff856bcc60c6e5ba7d8d501e41be78e8ad8235ed06534c643bd45cc', 'tasks/dfthcsos/b143.py': '071af03c0167d1b09f7914d3bf9749a444b1845fd212b2126da7354c3f345131', 'tasks/dfthcsos/full_xor/__init__.py': '5ef466afd5346008e3454c369d782e27094e9675aed4cae051ff0adf35f51be6', 'tasks/dfthcsos/full_xor/emitter.py': 'b7cf804974c2abdbda7a3a247f1dceb8eb91ae1291a8358bf07ea5f60b24a255', 'tasks/dfthcsos/full_xor/paired_anf.py': 'f3bb5656cbca9ba9df82f0cd9f2f098b8fba0592eb709524433169172d05f4cd', 'tasks/dfthcsos/full_xor/planio.py': '8f761c2992dd2162dd339f93e5f0ab873ae55b6a8c1963dc7cb9a128ada9d561', 'tasks/dfthcsos/full_xor/plans.json': 'db6d6b4cb40624d8e9cd23ddc728b0ba14687c65d7db4f5eea067c7b97f69f41', 'tasks/dfthcsos/full_xor/scheduler.py': 'bc8b17e8c5d40710b0b893195813fb81c63fb6f6c6b924da627f84e5e687eba7', 'tasks/dfthcsos/full_xor/validation.py': 'ebcd4b8b6e846f805968c98f458375f09108dd6b066b7ca10fcfac42fb309e8e', 'tasks/dfthcsos/generate.py': '7c133c8e732328869452dbfd83d781147e4169bd33665b7b6c4b5bc33941149a', 'tasks/dqi/fused/LICENSE': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30', 'tasks/dqi/fused/SOURCE_MAP.json': 'a194c1777dcaf5b4a1a4923d86b436d3a8d993f0f246416268c94f18243737e7', 'tasks/dqi/fused/source/exp/alternating_gauge/even_fold.py': '784cf9b2873b5fa04da4233a528d7ffd8acf9d30fb2e39dc9992d46b1438cd16', 'tasks/dqi/fused/source/exp/alternating_gauge/kernel.py': 'd99aa5eaef8a0df416270d8eee0fa55a9847ee7c08c3aa6ee437352e1abdd05a', 'tasks/dqi/fused/source/exp/alternating_gauge/scalar.py': '2fb3e5c4ec62df647de519647a57f51c765907ce1b6ecfe2d512035a91523b15', 'tasks/dqi/fused/source/exp/arithmetic/division.py': '28567073545c805e32636668cc0bb33f0dbc441a1de0dee2825774b3a33d2832', 'tasks/dqi/fused/source/exp/arithmetic/field.py': '9a84208c0690c0c2473f950920b27bacde4bc78f5e8dabf9a83df5a9b362b603', 'tasks/dqi/fused/source/exp/arithmetic/lowwidth.py': 'c536fa2b42393b83aae2bef80eeda22d83a6c799ef555178e94e09f1f5abc22a', 'tasks/dqi/fused/source/exp/confluent_extract/kernel.py': '3acfc52dbf5994a85671b975be506b7967d2d075a3e1fc39f7c7cd16a8586ff3', 'tasks/dqi/fused/source/exp/confluent_extract/model.py': '9b8d6fbaad6aec5da87d5247a11276c2fa177883adff9eca030a9bc2f3286387', 'tasks/dqi/fused/source/exp/confluent_synthesis/kernel.py': '2eaabffce88e50ab3ce4719e764b3ed43c8bd6d8a6acf22a4461574268691c54', 'tasks/dqi/fused/source/exp/convert/convert.py': '7afdafa60806df5afbc0af8fd01417a3e09fc15c5dc83fdd28d68079e78df7a0', 'tasks/dqi/fused/source/exp/convert/hankel_check.py': '6d0ac6f55a70418049f46300863eeabb90a7c38b476b84db6e83d395edc3d72c', 'tasks/dqi/fused/source/exp/convert/kernels.py': 'f5efa2f2b008cce86a0b751d400ee75777b77b6872d1875d6fae281cd7f09ef9', 'tasks/dqi/fused/source/exp/convert/repair.py': '8beceea9091a9656901290498fbdc0e4373963374b5ee8f7e98d257ece1daeaf', 'tasks/dqi/fused/source/exp/convert_short_hankel/kernel.py': '6dd64a123e4bd668bbf30e7d3c6449e738c8bf693f9a31bce1c9c39810ba7ddb', 'tasks/dqi/fused/source/exp/convert_short_hankel/phase34.py': '620985ba489a03cef51095157af60b7459732b72b68f85a10265f34905ed203f', 'tasks/dqi/fused/source/exp/decoder/build.py': '08f423818ff66867f2c91730daf4a98a74b8efb5f79b6cf2552f8f33d80cc645', 'tasks/dqi/fused/source/exp/decoder/common.py': 'b483a8df5b42be3a3b372633afc05f60ecaf0d9526f96f523c97056167c11109', 'tasks/dqi/fused/source/exp/decoder/extract.py': '6597bd52d2bebedae8860acaa4aaeaf980c3d21977781382ba8926833c8f7696', 'tasks/dqi/fused/source/exp/decoder/safe_bridge.py': '023c81c775967a71ef44c571200bbe3ffb8b3da443fc2816e751856fc3f0d8d9', 'tasks/dqi/fused/source/exp/decoder/synthesis.py': 'fdd7de226b7445007689ce2e50dbabbe77fdbc20e994ec41ac821a6ad2474c49', 'tasks/dqi/fused/source/exp/fold_cached_norm/kernel.py': '6a148ebe23eae9194ec0ddb921ff6da1d362565218913170a36691c45dd7c084', 'tasks/dqi/fused/source/exp/fold_cached_norm/phase34.py': '974d76939483c061e7b7995600fcdcb05e91529297725acfba41ce5ce808661f', 'tasks/dqi/fused/source/exp/fold_joint/divide.py': '8456426c171f36a716a39b457a0de5206b5d4db6c8553f1bfc919248ba1de851', 'tasks/dqi/fused/source/exp/fold_joint/kernel.py': 'd4764e763c6ed1ff538b58c1cc628df0c9577300aae5a4d83a5338e0290e018f', 'tasks/dqi/fused/source/exp/inverse12_escape/joint.py': '4bcc5614693d646bd910b9ed4748b278d2ec8dc1584bf8a45d3e465dd757cdc1', 'tasks/dqi/fused/source/exp/inverse12_escape/zero1.py': 'c9274f825d452082d88a98ded25a269aab247626a17032287c26ca1d8a71c724', 'tasks/dqi/fused/source/exp/inverse12_escape/zero_layout.json': '45d1361fcf582282d15cc6fabeab290f8e9fd9dc4c68a3e14ffff86e9f9a537f', 'tasks/dqi/fused/source/exp/inverse19_joint/kernel.py': '035152593a105b8178522a6edbf7cda3b709d8a86f51ae2148e71b01a1274913', 'tasks/dqi/fused/source/exp/inverse19_joint/layout.json': '0adee49170ae597ed5a7df4d4d0df0e495fba49013f6acd9ae6a3f06b9ad4ab4', 'tasks/dqi/fused/source/exp/inverse_search/baseline.py': '7aecb36f951122861fe0443eee92f1d65a10484fc86f2e535f3db44f5dd553da', 'tasks/dqi/fused/source/exp/inverse_search/inverse20.py': 'e36162777e51361a7ceb47ae6b86a34ce492b23c2fde8083113b03985b98060d', 'tasks/dqi/fused/source/exp/inverse_search/inverse20_iso.py': '1b4ffd1d35b12ca4f327c8885063fb76eef459eeabed1fddae371e2e4bb4f635', 'tasks/dqi/fused/source/exp/inverse_search/optimized.json': 'b18cfd6025658d02e38adfbe0ea7eca09960bbbdd458abe1ee1d539e4d3f6ca7', 'tasks/dqi/fused/source/exp/inverse_search/optimized_isotopy2.json': '28424aa2f6b0b92b7b3b4f6bc62feda566746feb9a9f21660ed5103420eae606', 'tasks/dqi/fused/source/exp/inverse_width/kernel.py': 'accdd344715e6972ebe22e1679ca872d455a75466f1b1236b83bb636f6466409', 'tasks/dqi/fused/source/exp/inverse_width/partial5.json': '202ddf3a0a62530ac4f4028774e87c729cba6cc85dd465570a59e882a83cf4f7', 'tasks/dqi/fused/source/exp/inverse_width_subspaces/inverse19.json': 'c868b9fc56f05ec89d34375ad086490691237211ecdc99b4a0c0b6e86b468a3f', 'tasks/dqi/fused/source/exp/inverse_width_subspaces/kernel19.py': '2cb588e60d5fe22ea59d625164e0a0d285cbcd352a80f5bf86f53f76050c9f4b', 'tasks/dqi/fused/source/exp/jet24/kernel.py': 'f490e6205c678135dc2273d1f02e5ce9540087995f52c37e174b51dce0b4a5f6', 'tasks/dqi/fused/source/exp/mask_projection/kernel.py': '80148ee048d1fd9aaace9637d2e652a7f46dbcb4306913f053437af4a2119088', 'tasks/dqi/fused/source/exp/mul99/kernels.py': '8f93d0a23f465cf72045e02f702b0b40d629b72d1618952a8fc79c22e23ee6ec', 'tasks/dqi/fused/source/exp/mul99/phase36.py': 'd976ab9f6fbb9e2972b9518576b0135b07181d1b6cbfb74a61c094c211e1dccf', 'tasks/dqi/fused/source/exp/mul99/schedules.py': '9d16c576747f9d556b857f846c4029131f977cd6eb15d8a37cff4b84b0919c66', 'tasks/dqi/fused/source/exp/noscratch/kernels.py': 'fa64bb5d5c359c507ac9a488c3f73363d5390bd43a4697daa67df57c322d7370', 'tasks/dqi/fused/source/exp/noscratch/phase39.py': 'a47a3ad3ce8787f868b93b40156e3496eead5200b187244b5db13ec0f0c79f1a', 'tasks/dqi/fused/source/exp/noscratch/schedules.py': '6871354632e828b1b3efed382d84c39bac42e278f5ac700728efc3f1745cfa8d', 'tasks/dqi/fused/source/exp/partial_gauge/prefix.py': 'dba10093b9314daf696ebd8fbc85543dbcf73534eb45d2303dc9ce53189b0652', 'tasks/dqi/fused/source/exp/partial_gauge/scalar.py': '7b3cb8eece8ccbc42462f13dcab9ef6b253581a06ce4d14906d88a541774de29', 'tasks/dqi/fused/source/exp/partial_gauge/synthesis_noflag.py': 'cb316d80cdcfa636735156171a2b4d0bb57be22e4c5495806709f224fd0ea8ae', 'tasks/dqi/fused/source/exp/partial_gauge/tail.py': 'cef30a29c419aef478fbf9c98f4277ab4a1fc63ca52e95d6025d592f16c7aff3', 'tasks/dqi/fused/source/exp/phase24_tradeoff/kernel.py': '984fc30f4c834b8f3cc1a7276f26f54801c01c8fd53b757e9ba767325059711a', 'tasks/dqi/fused/source/exp/phase24_tradeoff/phase6_six_search.json': 'afb39dcf8748b1b1419a5e8db6b5440be74db6687464e6e0a9052a08b168fdb0', 'tasks/dqi/fused/source/exp/phase24_tradeoff/phase6small.py': '28590db3c6a0843fedc2894bb9899df45a7b99ea4544ce6b7517e739e672fe32', 'tasks/dqi/fused/source/exp/safe_scalar/converter.py': '771a242b7910c3e59f27bc82c72e8f7b86f9cb201d84ca3712ba7fd7e28a2841', 'tasks/dqi/fused/source/exp/safe_scalar/kernel.py': '72fbedc923066741e50cf4372de75ade749844c3554005b2f72ea5594190e84a', 'tasks/dqi/fused/source/exp/safe_scalar/phase.py': '30fdcd43d2234c6476344c1608449684199c8f3e8fb403f96aca80ae2379b048', 'tasks/dqi/fused/source/exp/scan/scan.py': 'bab543d7fbaa33666e8c718a40ec47ac2b9a52e06e59de27bbbe45e68c754b55', 'tasks/dqi/fused/source/exp/tail_alias/divide.py': '46d4c7c4c94a06e9e244c030934ab45092ad6793b530c501226e2edb62a3ee5c', 'tasks/dqi/fused/source/exp/tail_alias/kernel.py': '3452ed2c039c73d3b1f45743f1ddeb40b88ffc32178b4052115f17bf18ba48ff', 'tasks/dqi/fused/source/exp/zero_padding_synthesis/kernel.py': 'f2b84166a1ef1fb131ead3d907b21625375bd45a24187e74b440ff45647fc185', 'tasks/dqi/fused/source/playbook/exp/cf_check/audit.py': 'c3211bbccebce2183b1a24385812dc505d1041ca7244274bbdffa75a95f37b33', 'tasks/dqi/generate_fused.py': '60cff2f3304103ddfafda568a28f56beb0bf92e3b01b1fc323fd288bac7fdf8a', 'tasks/invsqrt/__init__.py': '93c5f408a63782c2615a7fd8a69d90deb74e149df9e4b81059b5e38e929ad6ff', 'tasks/invsqrt/affine.py': 'aa351a287c507c264a8aa7d172e1c7b9bfc1d0dcb78c8fa61b92996947eba290', 'tasks/invsqrt/calibrate.py': '7fc0c1cac6ff45753c25d98dfa7cd82fb4803332c37fb616af18fabe47cef309', 'tasks/invsqrt/peephole.py': 'd5ed02b0d28392cc637111fc94fb1760c71feda4f8ca7aad6f73944e81fff54c', 'tasks/invsqrt/public_port/__init__.py': 'cc8392437ebbc8e48c8d352d07b07ae99ede4947040de8b680d8b04c06aa0459', 'tasks/invsqrt/public_port/arithmetic.py': '64c3ecf057dd9e2fb9ce3c6188278a4f11df3e992af742f3111fa808f125d2d3', 'tasks/invsqrt/public_port/calibration.py': 'ef3d11d4ab868eaee6a5a0b07e62f86d4934b0be0eb0c5ff0719b62db22b8549', 'tasks/invsqrt/public_port/deferred.py': 'd886ff6434b2fed86c2ff3246b0ee09f00de12a5791561f1a80558a5acd9ca57', 'tasks/invsqrt/public_port/fit.py': 'fea9cdbbe4bd7b336edbfa6f4163a153335cb2c24c8f1c913549025eae059b4e', 'tasks/invsqrt/public_port/generate.py': '21a19c3919a990a8ce44c5cc879c163e4124efe4166a86b92c48153af3c1844b', 'tasks/invsqrt/public_port/model.py': '24997f12449e0c51c14f7cbf08d7f4a73bd3b61859401feca0ce590be93b220b', 'tasks/invsqrt/public_port/schedule.py': '9df94061846f25298f50de0b835ef376f49999c57b24d51b2a7f6edb31e362d3', 'tasks/invsqrt/public_port/virtual_low.py': '36f177a1bfd871e03d2c967f5b51692e487de7a5e639b3b3116a77a090f4838e', 'tasks/invsqrt/signed_digit.py': 'ab08c1f178405cae9872281fbe73ca41c5de160bea4b5ddeece41b55ac708cd5'}

In [3]:
# SPDX-License-Identifier: Apache-2.0
import importlib.metadata
import json
import os
import platform
import subprocess
import sys
import tempfile
import time
import zlib

STARTED = time.monotonic()
DATA = competition_root(os.environ.get('VQCC_DATA'))
MODE = 'full'
WORKERS = min(2, os.cpu_count() or 1)
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '1'
if sys.version_info[:2] != (3, 13) or platform.system() != 'Linux':
    raise RuntimeError('The pinned competition kickmix requires CPython 3.13 on Linux')
for name in ('kickmix', 'verifiers', 'sample_submission'):
    if not (DATA / name).is_dir():
        raise FileNotFoundError(f'VQCC_DATA must contain {name}/')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA-enabled torch and a CUDA GPU are required for invsqrt fitting')
print('CUDA device:', torch.cuda.get_device_name(0), 'torch CUDA:', torch.version.cuda)
print('Source commit:', CONFIG['commit'])
print('Manifest:', CONFIG['manifest'], 'at', CONFIG['manifest_commit'], 'SHA-256', CONFIG['manifest_sha256'])
memory = next(line for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:'))
environment = {'python': sys.version, 'platform': platform.platform(),
               'packages': {p: importlib.metadata.version(p) for p in ('numpy', 'scipy', 'torch')},
               'cpu_count': os.cpu_count(), 'memory': memory,
               'zlib_runtime_version': zlib.ZLIB_RUNTIME_VERSION,
               'workers': WORKERS, 'blas_threads': 1, 'mode': MODE,
               'source_commit': CONFIG['commit'], 'kickmix_sha256': CONFIG['kickmix_sha256']}
print(json.dumps(environment, indent=2))
print('Verification mode:', MODE)
check_hash(DATA / 'kickmix/kickmix.so', CONFIG['kickmix_sha256'])
for task, relative in {'invsqrt': 'invsqrt/verify_submission_invsqrt.py',
                       'dqi': 'dqi/verify_submission_dqi.py', 'dfthcsos': 'dfthcsos/verify.py'}.items():
    check_hash(DATA / 'verifiers' / relative, CONFIG['verifiers'][task])
factor = Path(os.environ.get('VQCC_FACTOR', '/kaggle/input/qcc-our-chemistry-b143-c60-xor2/submission.npz'))
# Validate before creating the isolated input root or running a generator.
if factor.name != 'submission.npz' or factor.stat().st_size != CONFIG['factor_bytes']:
    raise ValueError('Factor filename or size mismatch')
check_hash(factor, CONFIG['files']['dfthcsos/submission.npz'])
# A new directory on each run prevents stale circuits or cached coefficient tables.
WORK = Path(tempfile.mkdtemp(prefix='vqcc-rebuild-', dir=Path.cwd()))
INPUT = WORK / 'input'
stage_factor(factor, INPUT, CONFIG['factor_bytes'], CONFIG['files']['dfthcsos/submission.npz'])
SOURCE = WORK / 'source'
OUTPUT = WORK / 'submission'
OUTPUT.mkdir()
if any(OUTPUT.iterdir()):
    raise ValueError('Submission output directory must start empty')
REPORTS = WORK / 'reports'
REPORTS.mkdir()
for name in ('submission.zip', 'LICENSE', 'NOTICE', 'reports'):
    if (Path.cwd() / name).exists():
        raise FileExistsError(f'Prior output {name} exists; use a fresh working directory')
(REPORTS / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n')
(REPORTS / 'sources.json').write_text(json.dumps(
    {'commit': CONFIG['commit'], 'files': SOURCE_HASHES, 'expected': CONFIG}, indent=2) + '\n')
restore_sources(SOURCE, SOURCES, SOURCE_HASHES)
(Path.cwd() / 'LICENSE').write_text(SOURCES['LICENSE'])
notice = SOURCES['NOTICE'].rstrip() + '\n\n' + ''.join(
    text for name, text in sorted(SOURCES.items()) if name.startswith('notices/'))
(Path.cwd() / 'NOTICE').write_text(notice)
sys.path[:0] = [str(SOURCE), str(DATA / 'kickmix'), str(DATA / 'verifiers')]
import kickmix as km
check_hash(Path(km.__file__), CONFIG['kickmix_sha256'])
ENV = {
    'VQCC_DATA': str(DATA),
    'PYTHONPATH': os.pathsep.join(
        [str(SOURCE), str(DATA / 'kickmix'), str(DATA / 'verifiers')]),
    'OMP_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'NUMEXPR_NUM_THREADS': '1',
}
print('Fresh workspace:', WORK.name)

CUDA device: NVIDIA GeForce RTX 4060 Ti torch CUDA: 12.8
Source commit: 1d3b1178e6de84acd76c9ac751b0d88f666bfef0
Manifest: ledger/manifests/submissions/submission-011.json at 1d3b1178e6de84acd76c9ac751b0d88f666bfef0 SHA-256 fbdfc9dea3d6b7d694f9f602465d8fde942df57643bb31b30e61df3957e9bd8d
{
  "python": "3.13.13 (main, Apr 14 2026, 14:28:56) [Clang 22.1.3 ]",
  "platform": "Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39",
  "packages": {
    "numpy": "2.5.4",
    "scipy": "1.18.1",
    "torch": "2.8.0+cu128"
  },
  "cpu_count": 16,
  "memory": "MemTotal:       28737712 kB",
  "zlib_runtime_version": "1.3.1",
  "workers": 2,
  "blas_threads": 1,
  "mode": "full",
  "source_commit": "1d3b1178e6de84acd76c9ac751b0d88f666bfef0",
  "kickmix_sha256": "c66b3216c4029e0cdfd09054306aa0feadf53ab2f50c1164e37f752874e5005e"
}
Verification mode: full
kickmix.so: SHA-256 c66b3216c4029e0cdfd09054306aa0feadf53ab2f50c1164e37f752874e5005e


verify_submission_invsqrt.py: SHA-256 ddef520dec6b68d1cf0a6f328ddd36d79eb84570b9192cf3f7acaf073f06733a


verify_submission_dqi.py: SHA-256 f14274bc0943fb9769ddb01131be6063191a15009fe1caf1d4bdd9fd25194e65


verify.py: SHA-256 0775be7a528922367caebe9e4d051d6199811f1aefeaa4b793eaaa4621b8ec9d


submission.npz: SHA-256 aef36d39b83bbbf696e909884218f6c02f067baef14a29de1c41306bc1f6395b


submission.npz: SHA-256 aef36d39b83bbbf696e909884218f6c02f067baef14a29de1c41306bc1f6395b


submission.npz: SHA-256 aef36d39b83bbbf696e909884218f6c02f067baef14a29de1c41306bc1f6395b


LICENSE: SHA-256 40911d195a0c25e3647660174fbd30f95ad80af621bd1088bbf5649c970d4e58
NOTICE: SHA-256 5189abb098bc3c42bb05b0759e00ff15f9d001cda2998f9ea35d1d53ac827f6b
ledger/__init__.py: SHA-256 e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
ledger/archive.py: SHA-256 1950107fd862cf70a1ef7fc899737b334928ffb8e03ce0c69bd4c82d428a0f39
notices/10-invsqrt.txt: SHA-256 7d35462413d4c7092f40fdafbda2e06af619f91867b59d2bd2d3ab8b9b2b76e7
notices/11-invsqrt-study.txt: SHA-256 69316ab49abc23d5a4dee971fe8ab2349cc34871a03359ee01e87173639017a8
notices/12-invsqrt-mrmeows-audit.txt: SHA-256 0bc23a786312c260b70fe8c2c06a0499baa997cecec72d204f37356e444e022b
notices/20-dqi.txt: SHA-256 b9009b1d46e3631a74582dde8d7210a44be0d3ac96bc40ed2a76f86a413b8315
notices/30-dfthcsos.txt: SHA-256 3e0f6f85d4ca9ff37d58377f54417a5097dbfa2ff9f8634d2e43b282bad908ef
notices/35-chem-catchup.txt: SHA-256 1001bc1ee85cd63909b4654c5ccb9376b6bb42b23323891ec5026481f7e884e1
notices/40-search.txt: SHA-256 98e6787a0bfef9fc4

tasks/dqi/fused/source/exp/inverse12_escape/zero1.py: SHA-256 c9274f825d452082d88a98ded25a269aab247626a17032287c26ca1d8a71c724
tasks/dqi/fused/source/exp/inverse12_escape/zero_layout.json: SHA-256 45d1361fcf582282d15cc6fabeab290f8e9fd9dc4c68a3e14ffff86e9f9a537f
tasks/dqi/fused/source/exp/inverse19_joint/kernel.py: SHA-256 035152593a105b8178522a6edbf7cda3b709d8a86f51ae2148e71b01a1274913
tasks/dqi/fused/source/exp/inverse19_joint/layout.json: SHA-256 0adee49170ae597ed5a7df4d4d0df0e495fba49013f6acd9ae6a3f06b9ad4ab4
tasks/dqi/fused/source/exp/inverse_search/baseline.py: SHA-256 7aecb36f951122861fe0443eee92f1d65a10484fc86f2e535f3db44f5dd553da
tasks/dqi/fused/source/exp/inverse_search/inverse20.py: SHA-256 e36162777e51361a7ceb47ae6b86a34ce492b23c2fde8083113b03985b98060d
tasks/dqi/fused/source/exp/inverse_search/inverse20_iso.py: SHA-256 1b4ffd1d35b12ca4f327c8885063fb76eef459eeabed1fddae371e2e4bb4f635
tasks/dqi/fused/source/exp/inverse_search/optimized.json: SHA-256 b18cfd6025658d02e38adfbe0e

tasks/dqi/fused/source/exp/tail_alias/divide.py: SHA-256 46d4c7c4c94a06e9e244c030934ab45092ad6793b530c501226e2edb62a3ee5c
tasks/dqi/fused/source/exp/tail_alias/kernel.py: SHA-256 3452ed2c039c73d3b1f45743f1ddeb40b88ffc32178b4052115f17bf18ba48ff
tasks/dqi/fused/source/exp/zero_padding_synthesis/kernel.py: SHA-256 f2b84166a1ef1fb131ead3d907b21625375bd45a24187e74b440ff45647fc185
tasks/dqi/fused/source/playbook/exp/cf_check/audit.py: SHA-256 c3211bbccebce2183b1a24385812dc505d1041ca7244274bbdffa75a95f37b33
tasks/dqi/generate_fused.py: SHA-256 60cff2f3304103ddfafda568a28f56beb0bf92e3b01b1fc323fd288bac7fdf8a
tasks/invsqrt/__init__.py: SHA-256 93c5f408a63782c2615a7fd8a69d90deb74e149df9e4b81059b5e38e929ad6ff
tasks/invsqrt/affine.py: SHA-256 aa351a287c507c264a8aa7d172e1c7b9bfc1d0dcb78c8fa61b92996947eba290
tasks/invsqrt/calibrate.py: SHA-256 7fc0c1cac6ff45753c25d98dfa7cd82fb4803332c37fb616af18fabe47cef309
tasks/invsqrt/peephole.py: SHA-256 d5ed02b0d28392cc637111fc94fb1760c71feda4f8ca7aad6f73944e81

Fresh workspace: vqcc-rebuild-yv9bdam_


## Build all three tasks

The arithmetic circuit adapts Dmitrii Gluzdov's invsqrt work; the decoder ports Agniv Banerjee (SomeJoint)'s fused DQI compiler; the chemistry circuits port Agniv Banerjee (SomeJoint)'s full-XOR compiler using our committed plans. Their exact source names, URLs, licences, and additional parents are in the complete notices below. Our orchestration, deterministic packaging and
byte checks are in this repository. The only copied output is SomeJoint's declared non-circuit
factorization `dfthcsos/submission.npz`; every `.kmb` is newly generated.
The verifiers also use the reference physics data inside the same competition package.

In [4]:
# SPDX-License-Identifier: Apache-2.0
started = time.monotonic()
command = [sys.executable, '-m', 'tasks.invsqrt.public_port.generate', '--out', str(OUTPUT)] + ['--circuit-only']
if 'invsqrt' == 'dfthcsos':
    command += ['--data', str(INPUT)]
subprocess.run(command, cwd=SOURCE, env=ENV, check=True)
print('invsqrt build seconds:', round(time.monotonic() - started, 2))

{"parity": 0, "group": 0, "group_bits": 3, "score": 0.8221353457927265}


{"parity": 0, "group": 1, "group_bits": 3, "score": 0.8943204643510907}


{"parity": 0, "group": 2, "group_bits": 3, "score": 0.9834761903270947}


{"parity": 0, "group": 4, "group_bits": 2, "score": 0.9078376836122096}


{"parity": 0, "group": 5, "group_bits": 2, "score": 0.8626976479072372}


{"parity": 0, "group": 3, "group_bits": 3, "score": 0.9677781455074566}


... (81 outputs left out)

GPU calibration LP 6: rows 10021 lp -36750308.776 exhaustive -36750308.776


calibration drop [18]: margin -36750310 rotations 43 ratio 0.000000


{"integer_mirror": {"passed": true, "tested_inputs": 3133452, "worst_ratio": 0.9499016330834485, "worst_x": 763987, "min_margin_units": 31510146, "first_bad_x": null}, "phase": {"phase_match": true, "inputs": 3133452}, "split": {"block": 11, "rw": 12, "cw": 7}, "raw": {"qubits": 84, "static_toffoli": 1577, "expected_toffoli": 1566.0, "t_like": 1, "rotations": 44}, "compiled": {"qubits": 84, "static_toffoli": 1564, "expected_toffoli": 1553.0, "t_like": 1, "rotations": 44}, "rewrites": {"r_zero": 1165, "ccx_literal": 5, "ccx_zero": 8, "cx_zero": 6}, "final": {"qubits": 84, "static_toffoli": 1564, "expected_toffoli": 1553.0, "t_like": 1, "rotations": 44}, "estimate": {"estimated_expected_cost": 156882.9658090783, "ratio": 13.237729303528413, "per_rotation_t": 28.582214079089074, "epsilon": 3.051847661949807e-13}, "sha256": "17fc223d79d3f1f15ef448389072209972b373403dd0a45352f4fe7eaff541ad", "bytes": 255914, "parent_version": 361726742, "full_verification_required": true}


invsqrt build seconds: 346.56


In [5]:
# SPDX-License-Identifier: Apache-2.0
started = time.monotonic()
command = [sys.executable, '-m', 'tasks.dqi.generate_fused', '--out', str(OUTPUT)] + []
if 'dqi' == 'dfthcsos':
    command += ['--data', str(INPUT)]
subprocess.run(command, cwd=SOURCE, env=ENV, check=True)
print('dqi build seconds:', round(time.monotonic() - started, 2))

{"name": "mask_projection", "T": 0, "cumulative_T": 0, "Q": 840, "num_ops": 2388214, "seconds": 5.7004477977752686}


{"name": "initial", "T": 0, "cumulative_T": 0, "Q": 840, "num_ops": 2719156, "seconds": 6.40031886100769}


{"name": "extract", "T": 87040, "cumulative_T": 87040, "Q": 889, "num_ops": 9255442, "seconds": 22.836629390716553}


{"name": "synthesis", "T": 44654, "cumulative_T": 131694, "Q": 889, "num_ops": 15098735, "seconds": 34.62219214439392}


{"name": "convert_lookahead", "T": 69220, "cumulative_T": 200914, "Q": 889, "num_ops": 18094594, "seconds": 45.00751757621765}


{"name": "scan", "T": 44275, "cumulative_T": 245189, "Q": 889, "num_ops": 41588664, "seconds": 160.26026797294617}


... (3 outputs left out)

{"name": "uninitial", "T": 0, "cumulative_T": 487331, "Q": 889, "num_ops": 59831421, "seconds": 258.89506006240845}


{"name": "restore_masks", "T": 0, "cumulative_T": 487331, "Q": 889, "num_ops": 59925608, "seconds": 262.99149775505066}


{"path": "/tmp/dqi-fused-ynou6wlo/dqi.kmb", "sha256": "7beea291a5de36ef2b4fb50585b2f8d7cce02d3c575b491cf5dcc93e9d9655f6", "T_static": 487331, "Q": 889, "num_ops": 59925608, "num_bits": 107242, "stages": [{"name": "mask_projection", "T": 0, "cumulative_T": 0, "Q": 840, "num_ops": 2388214, "seconds": 5.7004477977752686}, {"name": "initial", "T": 0, "cumulative_T": 0, "Q": 840, "num_ops": 2719156, "seconds": 6.40031886100769}, {"name": "extract", "T": 87040, "cumulative_T": 87040, "Q": 889, "num_ops": 9255442, "seconds": 22.836629390716553}, {"name": "synthesis", "T": 44654, "cumulative_T": 131694, "Q": 889, "num_ops": 15098735, "seconds": 34.62219214439392}, {"name": "convert_lookahead", "T": 69220, "cumulative_T": 200914, "Q": 889, "num_ops": 18094594, "seconds": 45.00751757621765}, {"name": "scan", "T": 44275, "cumulative_T": 245189, "Q": 889, "num_ops": 41588664, "seconds": 160.26026797294617}, {"name": "unconvert_lookahead", "T": 70342, "cumulative_T": 315531, "Q": 889, "num_ops": 44

{
  "variant": "fused-confluent",
  "pinned_files": 62,
  "num_qubits": 889,
  "static_toffoli_like": 487331,
  "num_ops": 59925608,
  "num_bits": 107242,
  "build_seconds": 291.47603033308405,
  "sha256": "7beea291a5de36ef2b4fb50585b2f8d7cce02d3c575b491cf5dcc93e9d9655f6",
  "bytes": 546664050,
  "stages": [
    {
      "name": "mask_projection",
      "T": 0,
      "cumulative_T": 0,
      "Q": 840,
      "num_ops": 2388214,
      "seconds": 5.7004477977752686
    },
    {
      "name": "initial",
      "T": 0,
      "cumulative_T": 0,
      "Q": 840,
      "num_ops": 2719156,
      "seconds": 6.40031886100769
    },
    {
      "name": "extract",
      "T": 87040,
      "cumulative_T": 87040,
      "Q": 889,
      "num_ops": 9255442,
      "seconds": 22.836629390716553
    },
    {
      "name": "synthesis",
      "T": 44654,
      "cumulative_T": 131694,
      "Q": 889,
      "num_ops": 15098735,
      "seconds": 34.62219214439392
    },
    {
      "name": "convert_lookahead",
    

In [6]:
# SPDX-License-Identifier: Apache-2.0
started = time.monotonic()
command = [sys.executable, '-m', 'tasks.dfthcsos.b143', '--out', str(OUTPUT)] + []
if 'dfthcsos' == 'dfthcsos':
    command += ['--data', str(INPUT)]
subprocess.run(command, cwd=SOURCE, env=ENV, check=True)
print('dfthcsos build seconds:', round(time.monotonic() - started, 2))

{
  "input_sha256": "aef36d39b83bbbf696e909884218f6c02f067baef14a29de1c41306bc1f6395b",
  "brot": 8,
  "batch_size": 5,
  "submission_sha256": "c88f058efddd1b273a7161ebdb57740ff3da36452acf337995c19ff67951a414",
  "circuits": [
    {
      "batch": 0,
      "toffoli_per_call": 305,
      "num_qubits": 126,
      "sha256": "d0f7e6bf8057b171c22a0cbef041ccecf1334fb17e968fd924286ff70d66d08a",
      "products": 225
    },
    {
      "batch": 1,
      "toffoli_per_call": 305,
      "num_qubits": 126,
      "sha256": "468475d7214894af0616870bdce211799bc2e97682699c97f89d5afbc20824f8",
      "products": 225
    },
    {
      "batch": 2,
      "toffoli_per_call": 305,
      "num_qubits": 126,
      "sha256": "f31de4b7672f853cbecc21be77e30cb6e1ecc837a5d4680ffee77fe7d147bf14",
      "products": 225
    },
    {
      "batch": 3,
      "toffoli_per_call": 305,
      "num_qubits": 126,
      "sha256": "3b175fb0abaa7e20fe22d021c77cb27e16658fd088d6f78bf307286e50319736",
      "products": 225
    },
 

dfthcsos build seconds: 8.6


In [7]:
# SPDX-License-Identifier: Apache-2.0
check_files(OUTPUT, CONFIG['files'])
print(f"All {len(CONFIG['files'])} generated/copied file hashes match submission-011.")

dfthcsos/angle_k_0.kmb: angle_k_0.kmb: SHA-256 d0f7e6bf8057b171c22a0cbef041ccecf1334fb17e968fd924286ff70d66d08a


dfthcsos/angle_k_1.kmb: angle_k_1.kmb: SHA-256 468475d7214894af0616870bdce211799bc2e97682699c97f89d5afbc20824f8


dfthcsos/angle_k_10.kmb: angle_k_10.kmb: SHA-256 2aa48c950a5d0850ddd544886eb5720f4a92f0b7cf496f182f4b9b9a980a8de7


dfthcsos/angle_k_11.kmb: angle_k_11.kmb: SHA-256 78c041703bbc43f795ae4ab9224a15e2a6dc97d60c76cf9927226de6c959a466


dfthcsos/angle_k_12.kmb: angle_k_12.kmb: SHA-256 e43da6265070311362619ce8c57f780e41f519d0be0590049bba8c2c885c41fd


dfthcsos/angle_k_13.kmb: angle_k_13.kmb: SHA-256 fbdac0002c7cd3e66487f966ccac1edf2ac40f628fb7d22af048c1ce66f60219


... (11 outputs left out)

dqi.kmb: 

dqi.kmb: SHA-256 7beea291a5de36ef2b4fb50585b2f8d7cce02d3c575b491cf5dcc93e9d9655f6


invsqrt.kmb: invsqrt.kmb: SHA-256 17fc223d79d3f1f15ef448389072209972b373403dd0a45352f4fe7eaff541ad


All 18 generated/copied file hashes match submission-011.


## Fresh verifier results

These calls use the competition's unchanged verifiers. Each cell raises on failure.
The run uses the fixed public salt `PRA171_LOCAL_PUBLIC_FIXED_SALT_0001`.
No stored verifier receipts are consulted.

In [8]:
# SPDX-License-Identifier: Apache-2.0
import math
from invsqrt import verify_submission_invsqrt as inv
from dqi import verify_submission_dqi as dqi
from dfthcsos import verify as df

SALT = 'PRA171_LOCAL_PUBLIC_FIXED_SALT_0001'
receipts = {}

def record(task, result, started):
    if result.get('status') != 'PASS' or not math.isfinite(result.get('score', float('nan'))) or result['score'] <= 0:
        raise ValueError(f'{task} verifier failed: {result}')
    receipt = {'task': task, 'mode': MODE, 'salt': SALT,
               'seconds': round(time.monotonic() - started, 2), 'result': result}
    receipts[task] = receipt
    (REPORTS / (task + '.json')).write_text(json.dumps(receipt, indent=2, allow_nan=False) + '\n')
    print(json.dumps(receipt, indent=2, allow_nan=False))

In [9]:
# SPDX-License-Identifier: Apache-2.0
started = time.monotonic()
spec = inv.ProblemSpecInvSqrt.for_n_p(10, shots_per_x=64)
record('invsqrt', inv.verify(km.Circuit.from_file(OUTPUT / 'invsqrt.kmb'), spec,
                            seed=None, salt=SALT, max_workers=WORKERS), started)

{
  "task": "invsqrt",
  "mode": "full",
  "salt": "PRA171_LOCAL_PUBLIC_FIXED_SALT_0001",
  "seconds": 1152.3,
  "result": {
    "status": "PASS",
    "score": 156882.98474606057,
    "toffoli_equivalent": 1867.654580310245,
    "toffoli_count": 1553.0002254402652,
    "max_toffoli_count": 1564,
    "t_count": 1.0,
    "rotation_count": 44.0,
    "rotation_eps": 3.051847661949807e-13,
    "num_qubits": 84,
    "reaction_depth": 214,
    "num_ops": 37753,
    "verified_shots": 200540928,
    "worst_error_over_tolerance": 0.9995943415273103,
    "worst_x": 32,
    "circuit_sha256": "17fc223d79d3f1f15ef448389072209972b373403dd0a45352f4fe7eaff541ad"
  }
}


In [10]:
# SPDX-License-Identifier: Apache-2.0
started = time.monotonic()
spec = dqi.ProblemSpecDQI.benchmark_instance(num_shots=102400)
record('dqi', dqi.verify(km.Circuit.from_file(OUTPUT / 'dqi.kmb'), spec,
                        seed=None, salt=SALT, log2_delta=-128, max_workers=WORKERS), started)

{
  "task": "dqi",
  "mode": "full",
  "salt": "PRA171_LOCAL_PUBLIC_FIXED_SALT_0001",
  "seconds": 384.79,
  "result": {
    "status": "PASS",
    "score": 446550365194.16925,
    "m": 4095,
    "n": 70,
    "degree": 12,
    "r": 2016,
    "verified_shots": 102400,
    "value_errors": 0,
    "phase_errors": 0,
    "empirical_error_rate": 0.0,
    "effective_error_rate": 0.0008660587301657926,
    "log2_delta": -128,
    "repetition_factor": 1.1912604648115832,
    "mean_error_weight": 31.046737331746634,
    "toffoli_count": 474307.7251367188,
    "max_toffoli_count": 487331,
    "reaction_depth": 9513,
    "num_ops": 59925608,
    "shot_operation_counts": {
      "NEG": 18381412,
      "BIT_INVERT": 210609134395,
      "BIT_STORE0": 12591001600,
      "BIT_STORE1": 222261616,
      "X": 26360627200,
      "Z": 14024992449,
      "R": 9718914145,
      "HMR": 17108712545,
      "CX": 5215362295460,
      "CZ": 57134988262,
      "SWAP": 169853389597,
      "CSWAP": 2246246400,
      "

In [11]:
# SPDX-License-Identifier: Apache-2.0
started = time.monotonic()
record('dfthcsos', df.verify(OUTPUT / 'dfthcsos', seed=None, salt=SALT,
                             shots_per_address=64), started)

{
  "task": "dfthcsos",
  "mode": "full",
  "salt": "PRA171_LOCAL_PUBLIC_FIXED_SALT_0001",
  "seconds": 50.72,
  "result": {
    "status": "PASS",
    "score": 343172572.78225887,
    "lambda_eff": 48.76425214813735,
    "toffoli_count": 16366.0,
    "num_qubits": 430,
    "brot": 8,
    "batch_size": 5,
    "num_circuits": 15,
    "reaction_depth": 203,
    "num_ops": 594940,
    "verified_shots": 78080,
    "submission_sha256": "c88f058efddd1b273a7161ebdb57740ff3da36452acf337995c19ff67951a414",
    "resources": {
      "brot": 8,
      "keep_bits": 10,
      "angles_toffoli_per_pass": 4218.0,
      "angles_toffoli": 8436.0,
      "angles_qubits": 128,
      "angles_selection_qubits": 11,
      "angles_persistent_qubits": 40,
      "angles_temporary_qubits": 77,
      "outer_state_prep_toffoli": 369,
      "inner_state_prep_toffoli": 5311,
      "givens_rotations_toffoli": 2250,
      "outer_state_prep_persistent_qubits": 42,
      "outer_state_prep_temporary_qubits": 96,
      "outer

In [12]:
# SPDX-License-Identifier: Apache-2.0
from ledger.archive import build_zip

if set(receipts) != {'invsqrt', 'dqi', 'dfthcsos'}:
    raise RuntimeError('All three fresh verifier results are required')
check_files(OUTPUT, CONFIG['files'])
archive = build_zip(OUTPUT, WORK / 'submission.zip')
check_archive(archive, CONFIG['zip_sha256'])
if archive.stat().st_size != CONFIG['zip_bytes']:
    raise ValueError('Archive size differs from the ledger')
# Publish the local output name only after all checks have passed.
import shutil
shutil.copyfile(archive, Path.cwd() / 'submission.zip')
(REPORTS / 'reproduction.json').write_text(json.dumps(
    {'files': CONFIG['files'], 'zip_sha256': CONFIG['zip_sha256'],
     'zip_bytes': archive.stat().st_size, 'verification_mode': MODE,
     'wall_seconds': round(time.monotonic() - STARTED, 2)}, indent=2) + '\n')
shutil.copytree(REPORTS, Path.cwd() / 'reports')
print('submission.zip:', archive.stat().st_size, 'bytes; SHA-256', CONFIG['zip_sha256'])
print('Verification:', MODE, '; Kaggle hidden-score equivalence is not asserted.')

dfthcsos/angle_k_0.kmb: angle_k_0.kmb: SHA-256 d0f7e6bf8057b171c22a0cbef041ccecf1334fb17e968fd924286ff70d66d08a


dfthcsos/angle_k_1.kmb: angle_k_1.kmb: SHA-256 468475d7214894af0616870bdce211799bc2e97682699c97f89d5afbc20824f8


dfthcsos/angle_k_10.kmb: angle_k_10.kmb: SHA-256 2aa48c950a5d0850ddd544886eb5720f4a92f0b7cf496f182f4b9b9a980a8de7


dfthcsos/angle_k_11.kmb: angle_k_11.kmb: SHA-256 78c041703bbc43f795ae4ab9224a15e2a6dc97d60c76cf9927226de6c959a466


dfthcsos/angle_k_12.kmb: angle_k_12.kmb: SHA-256 e43da6265070311362619ce8c57f780e41f519d0be0590049bba8c2c885c41fd


dfthcsos/angle_k_13.kmb: angle_k_13.kmb: SHA-256 fbdac0002c7cd3e66487f966ccac1edf2ac40f628fb7d22af048c1ce66f60219


... (12 outputs left out)

dqi.kmb: SHA-256 7beea291a5de36ef2b4fb50585b2f8d7cce02d3c575b491cf5dcc93e9d9655f6


invsqrt.kmb: invsqrt.kmb: SHA-256 17fc223d79d3f1f15ef448389072209972b373403dd0a45352f4fe7eaff541ad


submission.zip: SHA-256 a710cab21a8bac7ff552944ca48e8afb4e0fe4c98e0be5fd2e86be32b79c01ac


submission.zip: 31514298 bytes; SHA-256 a710cab21a8bac7ff552944ca48e8afb4e0fe4c98e0be5fd2e86be32b79c01ac
Verification: full ; Kaggle hidden-score equivalence is not asserted.


## Licence and attribution

The notebook's Praxen-authored code is released under the **Apache License, Version 2.0**.
The full licence is included in the source cell and at [https://github.com/clduab11/quantum-hackathon/blob/1d3b1178e6de84acd76c9ac751b0d88f666bfef0/LICENSE](https://github.com/clduab11/quantum-hackathon/blob/1d3b1178e6de84acd76c9ac751b0d88f666bfef0/LICENSE).
Ported code retains its original notices. This statement does not grant new rights to
third-party material. The invsqrt coefficient table is regenerated with dataset-adapted
calibration; **calibration rights remain unresolved**, as recorded in the repository.
The board authorized this bundle while retaining that record. Publication review must
address that existing limitation; this build does not resolve it.

The following are the repository's complete notices, reproduced verbatim at the pinned
commit. Some describe other experiments or reference implementations: those are identified
in the notices and are not claims that this notebook uses those inputs.

- The invsqrt generator under tasks/invsqrt/ is a port, not independent clean-room authorship. The module map is `tasks/invsqrt/PROVENANCE.md`. The code parent is "Quantum Circuits | Signed-Digit Memory" v3 by Artem1410 (anazemcev / ViberTeam), https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory?scriptVersionId=361502374 (v3), Apache License 2.0 as declared in that version's own source (SPDX header in cell 5, licence text and NOTICE in cell 20; PRA-218). Source snapshot SHA-256 79f6373302cb916a88f65a1b142024d4d2d0b14e41eb58bef3152ebd45ffd0ca, equal to the public version-pinned download. That notebook retains upstream notices for Sumit Chongder, Rudra Bhaskar, Alex Chilton, Kozykorpesh, and Kirill Tymchishin. Those notices are kept in tasks/invsqrt/signed_digit.py. Sumit run 361357149 is v1 (public version metadata, PRA-218); Dmitrii Gluzdov v6 is credited second-hand. Their originals are not loaded. Whether a later Sumit v2 exists upstream stays with PRA-205.
- The 90-qubit Gray curvature schedule in tasks/invsqrt/schedule90.py is a port of Kozykorpesh, "Quantum AI LB score", V3 run 361677445, https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445 , Apache License 2.0, as that method is used by the attached Apache-2.0 notebooks. The pruned shift is the Dmitrii Gluzdov V6 boundary rule that schedule names (run 361592338). His notebook is not an input. The two Davio reload orders are method parameters from anazemcev, "Quantum Circuits | Source & Exact Compilation" v4, Apache License 2.0, attachment SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb.
- The affine Toffoli/CSWAP rewrite in tasks/invsqrt/affine.py is a port of Kozykorpesh, "Quantum AI complete-domain affine compilation" v2, Apache License 2.0, attachment SHA-256 81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24, and the measurement-uncompute rule from anazemcev v4. The peephole in tasks/invsqrt/peephole.py is a port of that compilation's involution pass.
- Rotation calibration in tasks/invsqrt/calibrate.py is our scipy linprog search. The slack-maximising drop is credited to LightningV08 and to Mizero Lucky Gall. Mizero v1 has no notebook-wide licence (PRA-194 public-notebooks; PRA-199 item 2) and is reference-only: it is not loaded and it is not a shipping parent. Alex Chilton v8 is the same: credited where the signed-digit parent retains his notice, not loaded, not a licence parent. No calibration file is an input.
- Coefficient triples are produced by tasks/invsqrt/fit_coefficients.py from the mathematical target (Lawson minimax, then an integer search whose start is the rounded Lawson coefficient). Provenance of that table is `regenerated; dataset-adapted calibration; licence unresolved`. The calibration is adapted from the dataset's `code/domain_tables.py` (SHA-256 59cf4d98899433dcc49d9ce512293be20d33da5334211c20e35323555994a4fb); the output equals its `code/coefficients.json` (SHA-256 98bd211f22b158bf05a4d2d934ab966e5e97e54d8307b82e2ca58910c6830763) 256 of 256 rows. Source: kozykappa/quantum-ai-dataset, version 1, datasetVersionId 21136269, https://www.kaggle.com/datasets/kozykappa/quantum-ai-dataset?datasetVersionId=21136269 . Kaggle licence label: "Other (specified in description)". Rights for both files remain unresolved (D-PRA220-01). The Kaggle coefficient dataset version 21136269 and `code/coefficients.json` are not read and are not the optimiser start. Toolbox `fusion/*.npz` and `selected/invsqrt.kmb` are not inputs.
- `tasks/invsqrt/reference.py` is the competition package's reference Newton-Raphson circuit script (`verifiers/invsqrt/nr_reference_circuit.py`), Apache-2.0, Copyright Google Quantum AI authors of that package. Praxen changed the command line (`--out` writes `invsqrt.kmb`). It rebuilds the reference submission (submission-001) for G5 under `configs/gates/reference.yaml` and is not a parent of the Praxen candidate. https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/data
- The experimental `tasks/invsqrt/public_port/` schedule and interpolation recipe are a port of Dmitrii Gluzdov, "Quantum circuit compilation from source", v7, scriptVersionId 361726742, https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361726742 . The pinned notebook declares Apache-2.0 in cell 50; its UTF-8 SHA-256 is d689cc89f41a6c9130e9f4c3a6b08dfd5d758fcbbb79f527d55cbd3b48e15280. Code cells 20, 22, 23, 29, 34, 40 and 42 supply the deferred repair, interpolated curvature, split arithmetic and virtual low-bit methods. Retained method credits: Sumit Chongder, Rudra Bhaskar, Kozykorpesh, Alex Chilton, Kirill Tymchishin, Artem Nazemtsev, Gu/Kemeny/Yelin/Lukin, Dmitrii Gluzdov, LightningV08, Mizero Lucky Gall and Zongao Bian. Praxen changed package imports, regenerates coefficients by CUDA enumeration of the mathematical target, refines failing groups, and independently recomputes affine rewrites and angles. No saved table, calibration file, rewrite patch, busyaprime binary or other notebook output is an input. The existing fallback generator and its separately disclosed coefficient lineage are unchanged.

- `tasks/dqi/reference_circuit.py` is the competition package's reference Reed–Solomon decoder (`verifiers/dqi/dqi_reference_circuit.py`), Apache-2.0, Copyright Google Quantum AI authors of that package. https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/data
- `tasks/dqi/generate.py` is a port of a Berlekamp–Massey + Forney decoder, not an independent clean-room work. The module map is `tasks/dqi/PROVENANCE.md`. Transitive code parent: Gu, Kemeny, Yelin, and Lukin, repository `andigu/uncompute`, commit `64b5a13aa249f1360846657392e6c66817b26d0f`, Apache-2.0 (https://github.com/andigu/uncompute). Those four names are the `pyproject.toml` authors. That repository has no NOTICE file and states no copyright year; none is added here. Its `LICENSE` is copied verbatim to `tasks/dqi/bm_forney/LICENSE` (SHA-256 `cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30`). Used upstream paths: `LICENSE`; `uncompute/classical.py`; `uncompute/gadgets/linear.py`; `tasks/dqi-reed-solomon/contract.json` (field modulus changed from 4331 to 4179); `tasks/dqi-reed-solomon/circuits/discovered/constants.json` keys `bm_windows` and `psi_windows` only; `tasks/dqi-reed-solomon/circuits/discovered/circuit.py`; `tasks/dqi-reed-solomon/circuits/discovered/gen_constants.py` (formula section). Not used: `search.c`, `README.md`, `profile.pdf`, `circuit.ops.xz`, `liveness.json`, and the paper text and figures. Bibliographic citation only: Gu, Kemeny, Yelin, and Lukin, arXiv:2610.22367. Immediate code parent of the kickmix schedule: the identical `src/gen_dqi_v6` embed in Artem1410 (anazemcev / ViberTeam), "Quantum Circuits | Source & Exact Compilation" v4, Apache-2.0, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation (attachment SHA-256 `4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb`) and in Kozykorpesh (kozykappa), "Quantum AI — Complete-Domain Affine Compilation" v2, Apache-2.0, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation (attachment SHA-256 `81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24`). That embed attributes the memory schedule to Dmitrii Gluzdov's V6 schedule; his notebook and `femoco_public_words.npz` are not inputs. The shipped borrow route is the notebooks' affine loan. Multiplication tables are regenerated for modulus 4179. Zenodo 10.5281/zenodo.17301475 is not an input. No k256.dev circuit or numeric target is an input.
- anazemcev v4 cell-45: Retained opening notice from cell 45 of "Quantum Circuits | Source & Exact Compilation" v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation , attachment SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb: "Quantum Circuits resource labs (2026). Original experiments, explanations and added implementations: anazemcev / ViberTeam. Released under the Apache License, Version 2.0, subject to upstream notices." The directly copied src/gen_dqi_v6 embed supplies tasks/dqi/bm_forney/; tasks/dqi/generate.py adapts its driver and streamed sweep. The existing Gu notice and module map above remain applicable. The following credits are inherited through that embed, not claims that Praxen directly copied those earlier notebooks. Unrelated chemistry, numerical dataset and compiled-input claims in the original NOTICE do not describe inputs to this DQI port.
- Gluzdov field/adapter: Cell-45 attribution (cell 45 of the hash-pinned anazemcev v4 notebook above), retained for tasks/dqi/bm_forney/gu_formula_derivation.py, port_runtime.py and gu_port.py: "Dmitrii Gluzdov: competition-field formula derivation, Kickmix adapter, record-reading phase sweep and readable source-built portfolio. Apache-2.0. Adapted source V3, run 361529794: https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361529794 . The separately scored V1, run 361509836, returned 6.12847 for submission 57567302. V1 and V3 contain identical embedded generator modules. https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361509836 ." Version identity and scores in this passage are the immediate parent's historical attribution, not newly verified measurements. Our field tables are regenerated for modulus 4179; port_runtime.py's import and field checks are modified as recorded in the module map.
- Kozykorpesh symbolic lowering: Inherited through anazemcev v4 cell 45: "Kozykorpesh V3/run361677445, Apache2.0: https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445 ." The applicable DQI credit is "symbolic DQI lowering, and the source context for the portable adapters used in V4." It maps to tasks/dqi/bm_forney/symbolic_runtime.py and its use by tasks/dqi/generate.py. This is the transitive lowering lineage; the identical immediate embed in Kozykorpesh's Complete-Domain Affine Compilation v2 remains credited above. The original passage's Gray-curvature, signed-Givens, difference-bank and scratch-pebbling credits concern other tasks and are not DQI input claims.
- Gluzdov V6: Inherited through anazemcev v4 cell 45: "Dmitrii Gluzdov V6/run361592338, Apache2.0: https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361592338 . Compact inverse-norm schedules, template-word loans and streamed sweep, building on the credited Gu decoder and earlier competition-field port." This schedule lineage maps to tasks/dqi/bm_forney/round2_gadgets.py, borrow.py and gu_port.py, and the streamed-sweep integration in tasks/dqi/generate.py. The directly used source is the immediate notebook embed; Gluzdov's notebook and FeMoCo numerical dataset are not inputs. Our borrow.py is the modified affine-only extract described in the module map.
- PRA-236 paired J-fraction port: SomeJoint / Agniv Banerjee (decoder/arithmetic code), anazemcev / Artem1410 / ViberTeam (paired-v6 additions), Phuc Thai (pairing and workspace ideas), and Google Quantum AI (helpers), Apache-2.0 code parents; full URLs, notices, hashes and Praxen modifications in tasks/dqi/jfraction/{NOTICE,PROVENANCE.md,SOURCE_MAP.json}; licence text in tasks/dqi/jfraction/LICENSE. Manwithacat receives method credit for the classical-mask projection; projection.py is new code from its mathematical specification, not copied or transliterated from mask_projection.py, and not a strict clean room. No portfolio outputs or compiled circuit inputs are used.
- PRA-250: Agniv Banerjee (SomeJoint), quantum-circuits-13-54-local-gm-cost-gains, kernel 140675727 v7, https://www.kaggle.com/code/somejoint/quantum-circuits-13-54-local-gm-cost-gains . Apache-2.0 author grant in cell 34; licence/NOTICE in cells 90/91, retained under tasks/dqi/hankel/. Source SHA-256 3b0df548f40676117fb6f53cd7c9ab0fc5ca29b7ed56f90581921e443e92d7eb. Modified production-only port of Hankel/Bezoutian Forney phase, recursive LDU and cyclic Gold-26 arithmetic. Inherited SomeJoint, Phuc Thai and ViberTeam code uses the existing jfraction package; Manwithacat projection remains a credited method independently implemented by Praxen. See tasks/dqi/hankel/PROVENANCE.md and SOURCE_MAP.json for exact scope.
- PRA-261 Schur continued-fraction decoder: jday96314/QuantumAI commit   2b9227095bf2c76f9f64a572645d7dce54998298, initial_experiments/6/agent14   and the hash-pinned arithmetic dependency paths in tasks/dqi/schur/source-pins.json.   no licence file; used under the board ruling of 2026-10-26, card `aba6c540`,   extending D-PRA228-01; residual risk to prize eligibility accepted by the board.   Decision D-PRA269-01. Notebook lineage/prose: alexchilton/   toffolis-go-15-05-verified-portfolio-llm-gauge, scriptVersionId 362573669,   changelog v36, source SHA-256 28e0f08476df85323f9d25589f3cf781f13b5df105233193edfe8a5d1cff149c,   cell 8; v33 unavailable anonymously. Copy cells 1/5/6 and all prebuilt circuits   excluded. Missing 16-AND cover replaced by our derived, exhaustive-checked   17-AND cover. Existing Gu arithmetic retains its Apache-2.0 attribution.   Full modifications and source boundaries: tasks/dqi/schur/PROVENANCE.md.
- PRA-293 fused DQI decoder: Agniv Banerjee (SomeJoint), quantum-circuits-21-34-local-gm-fused-dqi, kernel 140675727 v12, https://www.kaggle.com/code/somejoint/quantum-circuits-21-34-local-gm-fused-dqi . Notebook code Apache-2.0 (credits cell; licence text embedded in cell 6, copied to tasks/dqi/fused/LICENSE). Source SHA-256 18d6760953783bec316ddb46266a1b5ccc50c291fbf47ec35f87c73f4d0cf2ac. Build entry blocks/dqi/team-fused/source/exp/decoder/build.py and its traced closure (53 imported modules, 7 JSON layout files) vendored verbatim under tasks/dqi/fused/source/; every file hash pinned in tasks/dqi/fused/SOURCE_MAP.json. Expected circuit SHA-256 7beea291a5de36ef2b4fb50585b2f8d7cce02d3c575b491cf5dcc93e9d9655f6. Design lineage as stated by the author: rebuilt from a pseudocode description of Alex Chilton's 883-qubit Schur decoder (jday96314/QuantumAI, no licence; board card aba6c540, D-PRA269-01, D-PRA273-01); mask projection by mrmeows and Ahmed Berat Ozer after manwithacat; earlier links Dmitrii Gluzdov, SomeJoint, ViberTeam and Phuc Thai. Full credits: tasks/dqi/fused/NOTICE and PROVENANCE.md.

- PRA-246 CUDA method port: `tasks/dfthcsos/gpu_fit/` ports selected mathematical definitions from anazemcev / Artem1410 / ViberTeam, "Quantum Circuits — 881Q Paired DQI + R6 DF-THC", v6, https://www.kaggle.com/code/anazemcev/quantum-circuits-881q-paired-dqi-r6-df-thc, Apache-2.0, notebook SHA-256 d4a147523ac07cb2dc5fe4bd5dc9c7da48719e34dfc847402628723b37930682; rebuild source dataset https://www.kaggle.com/datasets/anazemcev/vqcc-viberteam-verified-circuits/versions/2, Apache-2.0. Exact per-file hashes and extracted definitions are in `ledger/evidence/pra246/port-map.json`. Inherited credits: Ahmed Beratozer for the progressive quantisation trainer; Kozykorpesh for the augmented column gauge and signed Givens gauge; Sumit Chongder, Kirill Tym and Artem1410 for inherited compiler lineage (no compiler copied here). CUDA continuous/GN/QAT and exact integer repair retain their published mathematical objectives; QR/barrier/gauge/one-body helpers retain the source NumPy/SciPy execution. Praxen adds the explicit declared/Google intake, fixed seed, exclusive persistence, official evaluator adapter and 11-to-8 continuation. No upstream CLI, circuit, numerical payload or hidden loader is copied.
- PRA-246 declared ship-lane numerical input: Dmitrii Gluzdov, `dmitriigluzdov/vqcc-femoco-public-factorization`, version 8 (datasetId 12805185), https://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization/versions/8, Apache 2.0 as recorded by John Wick on PRA-245. `femoco_thc_r5.npz`, SHA-256 809a992900937425f01a0593803921873f7bb472d590592c378d3ceb23657df5, 1,871,575 bytes, shape (1,496,384), declared 11-bit words. This is a public factor warm start, not own-source training. Chris's accepted card a9b01800-986e-4e7f-b0c2-b288c4591b4e permits it as a declared non-circuit input. The separate Google-only run uses only the previously pinned Google baseline and fixed integral/shift inputs. The k256 artifact is neither loaded nor shipped.
- PRA-232 module provenance map: `tasks/dfthcsos/qrom.py` is a Praxen reimplementation of truncated algebraic-normal-form (ANF) loading, banked low-address differences, measurement-based ANF erasure and selective ancestor pebbling. Method-only notebook parents (read from the SHA-256-checked source attachments on PRA-194; never executed): kozykappa, "Quantum AI — Complete-Domain Affine Compilation", v2, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation, Apache-2.0, source SHA-256 81befe1e69ee286271fa2eb8acf4bb760a727853bf966945318d918482b1ab24 (six traversal wires, `gauge_qrom.py`); anazemcev / ViberTeam, "Quantum Circuits — Source & Exact Compilation", v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation, Apache-2.0, SHA-256 4f017407a40e0988583c9645662407046f04019357adac648b90b2b94598acdb (`banked_lab.py`, exact ANF phase repair); magmacot, "Quantum Circuits — clean workspace reuse", v2, https://www.kaggle.com/code/magmacot/quantum-circuits-clean-workspace-reuse, Apache-2.0, SHA-256 f3f393b5c4958f7288f69f68cd7762a62bfcc697d94d7d19607e67ff7ed20d9a (pebbling at prefix degree 4). The banked method's upstream attribution includes Kozykorpesh / kozykappa, https://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom?scriptVersionId=361433617, and Sumit Chongder, https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149, read through Kirill Tymchishin's QROM + Tower42 v10; retained as credited by these parents, not independently fetched or executed.
- PRA-232 precision experiment: ahmedberatozer, "Quantum AI — DF-THC Group Compression", v1, https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression, Apache-2.0 notebook code, source SHA-256 b221781e9bf8c1c8da8c5251c6556b294ed07def5e57f5a1ea038addbf7a57df, motivates testing 10-bit angles. `qrom.quantize_words` uses nearest-even rescaling of Google's baseline integer words, not that notebook's trained factors or quantization optimizer. No claim is made that its physical result transfers to Google's baseline.
- `tasks/dfthcsos/circuit.py`, `experiment.py` and PRA-232 tests are Praxen orchestration and verification code. The existing `generate.py` supplies Google's attributed batch-XOR schedule and pinned baseline input. `circuit_checks.py` calls Google's unchanged verifier math/resource model; it reads the circuit register width and reports static cost estimates. The only numerical input is Google's Apache-2.0 `sample_submission/dfthcsos/submission.npz`, SHA-256 14d5cc25eb3bf82a9ccb87c811db28d127bef6d28697dd190dc1e875de3bfe69. No `femoco_public_words.npz`, `dfthc_group_32_60_30.npz`, toolbox selected circuits, chemistry directory or k256 artifact was loaded. Signed-gauge factor changes and factor refitting are not implemented here.
- `tasks/dfthcsos/generate.py` ports the package's ordinary-QROM ANGLE builder (`verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py`), Apache-2.0, same authors and URL. The factorization bytes are the package's `sample_submission/dfthcsos/submission.npz` and are not committed.
- PRA-198 uses Google's Apache-2.0 starter kit v4, https://www.kaggle.com/datasets/sohier/verifiable-circuit-compilation-starter-kit/versions/4 (package author Guang Hao Low), as separately declared code and input parents. Original downloaded files and notices are retained unchanged; exact baseline, integral, fixed-shift, encoder and verifier pins are in `ledger/manifests/inputs/pra198-google-v4.json`. Outputs are labelled "own-generated, derived from Google baseline".
- D(w)-penalized structure refit and norm-ranked group pruning: Ahmed Berat Ozer, https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression, v1 described in PRA-194 and v2 source inspected on 2026-10-25, Apache-2.0 (`src/dfthc_train.py`). PRA-198 reimplements these methods with analytic NumPy gradients and SciPy CPU L-BFGS-B. No public factor data, words, derived target, warm start or checkpoint is used.
- Effective-norm / Gram-gauge and banked / six-wire QROM method context: Kozykappa, Complete-Domain Affine Compilation v2, https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation; anazemcev, Source & Exact Compilation v4, https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation; Apache-2.0 per PRA-194 / PRA-199. PRA-198 ports no Gram-gauge or banked-QROM code; its cost model is ordinary QROM. The module provenance map is `tasks/dfthcsos/FACTOR_PIPELINE.md`.
- PRA-242 module provenance map: `tasks/dfthcsos/FIT_V3.md`. `factor_merge.py`, `factor_gauss_newton.py` and `factor_fit_v3.py` reimplement the source-only fitting method of anazemcev / Artem1410 / ViberTeam, Quantum Circuits — 881Q Paired DQI + R6 DF-THC, notebook v6, https://www.kaggle.com/code/anazemcev/quantum-circuits-881q-paired-dqi-r6-df-thc, Apache-2.0, UTF-8 SHA-256 d4a147523ac07cb2dc5fe4bd5dc9c7da48719e34dfc847402628723b37930682, and its hash-checked ViberTeam rebuild source dataset v2. Retain inherited credit to Ahmed Beratozer (structural group-compression / quantisation methods) and Kozykorpesh / Sumit Chongder / Kirill Tym / Artem1410 (compiler lineage); no loader or compiler is copied. Dmitrii Gluzdov, Quantum Circuit Compilation from Source, v14, https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source, Apache-2.0, SHA-256 c3ba0bdde177b5fa159fc641d065c08870020fd773fa0947675675101eb2f52a, informs only the (1,496,384) shape and stage description: the training optimizer is not public. Google's pinned baseline is the only numerical initializer. These are bounded CPU adaptations, not exact public-result replays. `factor_measure.py` is an own acceptance wrapper over the unchanged pinned Google evaluator and existing cost model.
- PRA-245 declared non-circuit input, shipping factor (D-PRA245-01, D-PRA245-12; D-PRA211-05): Dmitrii Gluzdov, "VQCC FeMoCo Angle Words", Kaggle dataset `dmitriigluzdov/vqcc-femoco-public-factorization` (datasetId 12805185), version 8 (2026-10-26T02:51:21Z, notes "Updated circuit sources (78-qubit inverse square root) and refined factorization"), https://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization/versions/8, licence field as Kaggle shows it: `apache-2.0` ("Apache 2.0"). File `femoco_thc_r5.npz`, 1,871,575 bytes, SHA-256 809a992900937425f01a0593803921873f7bb472d590592c378d3ceb23657df5, (R,B,C) = (1,496,384), 11-bit angle words, used unchanged and never committed. Author's provenance statement, verbatim from the dataset description: "The femoco_thc_r5 file was trained by the dataset author: a fit of the shifted two-body tensor with a D penalty, progressive 10/11-bit quantization ordered by sensitivity (after ahmedberatozer), exact integer repair of the words, and a coupled refit of weights and offsets with Cayley column rotations (after Artem1410)." Dataset-level sources field, verbatim (it covers the whole dataset, including the 65×60×30 file we do not use, which D-PRA202-03 still governs): "Public Kaggle notebooks (Apache 2.0): k256.dev (refinement of the organizers’ 65 x 60 x 60 baseline); kozykappa, Quantum circuits | sparse SOS factors and circuit (https://www.kaggle.com/code/kozykappa/quantum-circuits-sparse-sos-factors-and-circuit), ensemble compression and identity offsets; Artem1410, Quantum Circuits | Signed-Digit Memory (https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory), effective-norm tuning. Organizers’ starter kit and verifiers: https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation". Earlier versions of the same file were measured, not shipped: v4 a29cb868d59906bebcda50c1c44c14090d503665700d725027e25016544782fd, v5 a0da3225ba2047f43ae24f31881f2a71500d3f73eda24fadc73e9f67da68e13f, v6 1b97a9aec279618b9cf66f28960b3c85fae98d73dde31cd01febe2bc30352778, v7 7db33b3eb951ac9c332aa4f7df8da106f9a7627e5b17a1c6f537e4f9b5e3084f.
- PRA-245 declared non-circuit input, measured fallback (D-PRA245-12): Artem1410 (anazemcev) / ViberTeam, "VQCC ViberTeam R6 DF-THC and Verified Circuits", Kaggle dataset `anazemcev/vqcc-viberteam-verified-circuits` (datasetId 12845740), version 2 (2026-10-25T15:10:31Z, notes "v2: add single-group THC (1,496,384) chemistry with our fixed-word joint norm refit (8.0935e8 FULL-PASS), credited copy of Dmitrii Gluzdov femoco_thc_r5.npz"), https://www.kaggle.com/datasets/anazemcev/vqcc-viberteam-verified-circuits/versions/2, licence field as Kaggle shows it: `apache-2.0` ("Apache 2.0"). File `thc_submission.npz`, 1,842,734 bytes, SHA-256 9b98b6284b2a846a6a8aa229f4c5a1da62aeba780b57c7341e49b8c3bfd53576 (no `angle_words`; its words were read for measurement only from the same version's `thc_selected_compact.npz`, 1,884,386 bytes, SHA-256 141d368ebd6517d3875a38e4e416b4886adb1efc7bd3df901e69436a96af49a9). Author's provenance statement, verbatim from the dataset description: "**Credits:** Dmitrii Gluzdov supplies the single-group factorization **and ANF lookup compiler** (`chemistry_r5.py`, as embedded in Ahmed's v4), Apache-2.0. His original public v4 `femoco_thc_r5.npz` is redistributed unchanged as `thc_femoco_thc_r5.npz`; SHA256 `a29cb868d59906bebcda50c1c44c14090d503665700d725027e25016544782fd`. Ahmed Berat Ozer supplies the v4 integration idea and wrapper. Artem Nazemtsev supplies the ANF/measurement-erasure lineage; Kozykorpesh supplies pebbling and signed gauge. ViberTeam K adds the independent complete-H joint W/native-offset refit adapted from F/H. See `thc_NOTICE.txt` and `LICENSE`." We retain that inherited credit (Gluzdov; Ahmed Berat Ozer; Artem Nazemtsev; Kozykorpesh; ViberTeam K; Phuc Thai per `thc_NOTICE.txt`). Their circuits (`thc_angle_k_*.kmb`) are not inputs and are not used (D-PRA211-01).
- PRA-251: `tasks/dfthcsos/fit_jax/` is a JAX reimplementation of Google's Apache-2.0 verifier formulas and half-up Givens-word encoder from starter kit v4 (https://www.kaggle.com/datasets/sohier/verifiable-circuit-compilation-starter-kit/versions/4), with unchanged source hashes in `ledger/manifests/inputs/pra198-google-v4.json`. The device-side assignment solver is our implementation of shortest augmenting paths; SciPy is the independent test oracle. No notebook training code was copied or executed for this loss-only deliverable. JAX is Apache-2.0 (https://github.com/jax-ml/jax); custom identity JVP implements the documented straight-through heuristic. Numerical mirror input: Dmitrii Gluzdov, Apache-2.0 dataset `dmitriigluzdov/vqcc-femoco-public-factorization`, version 8, https://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization/versions/8, `femoco_thc_r5.npz`, SHA-256 809a992900937425f01a0593803921873f7bb472d590592c378d3ceb23657df5, declared non-circuit input under D-PRA211-05, supplied by PRA-245. No competition circuit is used as an input or shipped by the mirror.
- PRA-294: `tasks/dfthcsos/kernel_basis.py`, `kernel_plan.py`, `streamed_qrom.py` and the plan loop in `c59.py` port mrmeows' exact four-word compiler: `streamed_native.py` (emitter), `balanced_plan.py` and `streamed_plan.py` (planners), `kernel_planes.py`, `hyper_kernel.py`, `pair_exchange.py` and `nullspace` from `rank_parity.py`, all from `sources/dfthcsos_cost_first_shape_screen/source/` in https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits?datasetVersionNumber=8, dataset licence field `Apache 2.0`. Per-file SHA-256 hashes are in `ledger/manifests/inputs/pra294-c59.json`. Changes: Praxen module split and naming, the two planners share one exact-reconstruction tail, reuse of Praxen's delta batching, gauge and family writer. The 19 first-stage seeds and constant stage sizes come from that version's `certificates/plan_k.json`; no certificate terms, circuits or audits are inputs, and all outputs are regenerated.
- PRA-294 declared non-circuit input (D-PRA211-05; recorded, not gated, under R-P2-6): mrmeows, `mrmeows/qcc-1375-open-circuits` version 8, `circuits/dfthcsos/submission.npz` (same bytes as `sources/dfthcsos_8p6121_C59_submission/dfthcsos/submission.npz`), 2,232,244 bytes, SHA-256 8229861b3ee22316e144c900737dfc54f7a30ea88de77ff5391f3375d050fa87, (R,B,C) = (10,129,59), 8-bit words, used unchanged and never committed. Licence field `Apache 2.0`. Lineage for the Adjudicator: Michael Rustad's DF-THC R10 merged groups v1 factor (the PRA-280 input, licence field "Other (specified in description)", which credits Ahmed Berat Ozer's DF-THC Group Compression), refit by mrmeows and merged from C=60 to C=59 columns with the lower-normalization words (v8 README update of 27 October 2026).
- PRA-321: `tasks/dfthcsos/full_xor/` and `tasks/dfthcsos/b143.py` port SomeJoint's full-XOR five-word compiler. Notebook `somejoint/quantum-circuits-22-67-local-gm-full-xor-chem`, kernel 140675727 version 18, scriptVersionId 363182923, https://www.kaggle.com/code/somejoint/quantum-circuits-22-67-local-gm-full-xor-chem, author Agniv Banerjee. Notebook source SHA-256 b44fc04a99ab0c18afce4c7d8408ddc079dd8a87957e294d2102494d8d9b001a. Notebook code Apache-2.0 (credits cell). Embedded files, with upstream SHA-256, are listed in `ledger/manifests/inputs/pra309-chem-8-143-60.json`. Changes: package layout and import paths. The product-basis search is not in the published source; each circuit's plan and schedule are committed in `tasks/dfthcsos/full_xor/plans.json`, taken from dataset `BUILD.json` with circuit hashes, proofs and audits removed. Those dropped fields are not inputs.
- PRA-321 declared non-circuit input (D-PRA211-05): Agniv Banerjee (SomeJoint), `somejoint/qcc-our-chemistry-b143-c60-xor2` version 1 (2026-10-28T13:11:51.017Z), https://www.kaggle.com/datasets/somejoint/qcc-our-chemistry-b143-c60-xor2, licence field Apache 2.0. File `submission.npz`, 1,995,236 bytes, SHA-256 aef36d39b83bbbf696e909884218f6c02f067baef14a29de1c41306bc1f6395b, (R,B,C) = (8,143,60), 8-bit words, used unchanged and never committed. The published notebook does not include the weight fit, so the file is not regenerated. Their ANGLE circuits are not inputs. The dataset contains the factor and `BUILD.json` only. The full-XOR compiler is the notebook above, not the dataset.

- The k256.dev decoder and the Signed-Digit Memory v3 chemistry factor are not included. See `docs/DECISIONS.md` D-PRA177-01.